In [ ]:
# Atlas v3.4 RegimeFlow | Una celda | T4 x2 / Internet OFF | No auto-submit
# Solo dataset oficial obligatorio. Auditoría histórica y fuentes incorporadas como texto. GPU T4 x2 si hay que reconstruir v2.7.
from pathlib import Path
import sys, hashlib, subprocess, os
from IPython.display import FileLink, display
ROOT=Path('/kaggle/working'); CODE=ROOT/'atlas34_code'; CODE.mkdir(parents=True,exist_ok=True)
DATA=Path('/kaggle/input/competitions/hackathon-ia-pucp-2026')
assert all((DATA/n).is_file() for n in ['train.npz','test.npz','static.npz','sample_submission.csv']), 'Adjunta dataset oficial'
SOURCES={}

# ----- Atlas_v3_2.py -----
SOURCES['Atlas_v3_2.py']=(
    'ATLAS_SOURCE=\'"""Atlas v3.1: independent residual physical downscaling."""\\nimport os\\nfor k in (\\\'OPENBLAS_NUM_THREADS\\\',\\\'OMP_NUM_THREADS\\\',\\\'MKL_NUM_THREADS\\\'): os.environ[k]=\\\'4\\\'\\nimport io, zipfile, time, argparse\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nfrom scipy.linalg import solve\\nfrom sklearn.utils.extmath import randomized_svd\\nfrom threadpoolctl import threadpool_limits\\nNORM=np.array([25.360,2.0652,2.6776,2.0876],dtype=\\\'f4\\\')\\nCUTS=[(\\\'Mayo\\\',\\\'2024050900\\\',\\\'2024051600\\\',\\\'2024060100\\\'),(\\\'Junio I\\\',\\\'2024052500\\\',\\\'2024060100\\\',\\\'2024061600\\\'),(\\\'Junio II\\\',\\\'2024060900\\\',\\\'2024061600\\\',\\\'2024070100\\\'),(\\\'Abril\\\',\\\'2024040900\\\',\\\'2024041600\\\',\\\'2024050100\\\')]\\ndef load(root):\\n    root=Path(root)\\n    if root.is_file():\\n        with zipfile.ZipFile(root) as z:\\n            def read(n):\\n                with np.load(io.BytesIO(z.read(n))) as f: return dict(f)\\n            return read(\\\'train.npz\\\'),read(\\\'static.npz\\\')\\n    if not (root/\\\'train.npz\\\').exists():\\n        matches=[p.parent for p in root.rglob(\\\'train.npz\\\') if (p.parent/\\\'static.npz\\\').exists() and (p.parent/\\\'test.npz\\\').exists()]\\n        if len(matches)!=1: raise ValueError(f\\\'Dataset ambiguo: {matches}\\\')\\n        root=matches[0]\\n    def read(n):\\n        with np.load(root/n) as f: return dict(f)\\n    return read(\\\'train.npz\\\'),read(\\\'static.npz\\\')\\ndef windows(x,t):\\n    dates=pd.to_datetime(t,format=\\\'%Y%m%d%H\\\'); lookup={d:i for i,d in enumerate(dates)}\\n    wanted=[[d-pd.Timedelta(hours=k) for k in (2,1,0)] for d in dates]\\n    idx=np.array([[lookup.get(q,i) for q in row] for i,row in enumerate(wanted)])\\n    mask=np.array([[q in lookup for q in row] for row in wanted],dtype=\\\'f4\\\')\\n    return x[idx],mask\\ndef fields(w,lat,lon):\\n    rh,t,u,v=np.moveaxis(w,-1,0); c=t-273.15\\n    gamma=np.log(np.clip(rh,.1,100)/100)+17.625*c/(243.04+c)\\n    dew=243.04*gamma/(17.625-gamma)\\n    e=6.1094*np.exp(17.625*c/(243.04+c))*rh/100\\n    dy=np.deg2rad(float(np.diff(lat).mean()))*6371000\\n    dx=np.deg2rad(float(np.diff(lon).mean()))*6371000*np.cos(np.deg2rad(lat))[None,None,:,None]\\n    def grad(a): return np.gradient(a,axis=-1)/dx,np.gradient(a,axis=-2)/dy\\n    ux,uy=grad(u); vx,vy=grad(v); tx,ty=grad(t); ex,ey=grad(e)\\n    return np.stack([rh,t,u,v,c-dew,e,np.hypot(u,v),(ux+vy)*1e5,(vx-uy)*1e5,(ux-vy)*1e5,(vx+uy)*1e5,-3600*(u*tx+v*ty),-3600*(u*ex+v*ey),np.hypot(tx,ty)*1e5,np.hypot(ex,ey)*1e5],axis=-1).astype(\\\'f4\\\')\\ndef features(w,physics,lat,lon):\\n    a=fields(w,lat,lon) if physics else w\\n    return np.concatenate([a[:,2],a[:,2]-a[:,1],a[:,2]-a[:,0]],axis=-1).reshape(len(w),-1)\\ndef climate(a,h,tr):\\n    return np.stack([a[tr[h[tr]==i]].mean(0) if np.any(h[tr]==i) else a[tr].mean(0) for i in range(24)])\\ndef metric(p,y):\\n    p=p.copy(); p[...,0]=p[...,0].clip(0,100)\\n    rm=np.sqrt(np.mean((p.astype(\\\'f8\\\')-y)**2,axis=(0,1,2)))\\n    return float(1-(rm/NORM).mean()),rm\\n\\nimport math, json, copy\\nfrom concurrent.futures import ThreadPoolExecutor\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\n\\nBASE={\\\'Mayo\\\':.4599006,\\\'Junio I\\\':.4604175,\\\'Junio II\\\':.4650797440837502,\\\'Abril\\\':.5584575773496497}\\n\\ndef geography(s):\\n    lat,lon=np.meshgrid(s[\\\'wrf_lats\\\'],s[\\\'wrf_lons\\\'],indexing=\\\'ij\\\'); dem=s[\\\'dem\\\'].astype(\\\'f4\\\')\\n    dy=np.deg2rad(float(np.diff(s[\\\'wrf_lats\\\']).mean()))*6371000\\n    dx=np.deg2rad(float(np.diff(s[\\\'wrf_lons\\\']).mean()))*6371000*np.cos(np.deg2rad(lat))\\n    sx=np.gradient(dem,axis=1)/dx; sy=np.gradient(dem,axis=0)/dy\\n    from scipy.ndimage import gaussian_filter\\n    relief=[(dem-gaussian_filter(dem,k))/500 for k in (1,3,7)]\\n    curvature=(np.gradient(sx,axis=1)/dx+np.gradient(sy,axis=0)/dy)*1000\\n    # Raw slopes retained separately for physically interpretable wind projection.\\n    static=np.stack([dem/3000,sx,sy,np.hypot(sx,sy),*relief,curvature,(lat-lat.mean()),(lon-lon.mean())]).astype(\\\'f4\\\')\\n    gx=2*(lon-s[\\\'gfs_lons\\\'][0])/(s[\\\'gfs_lons\\\'][-1]-s[\\\'gfs_lons\\\'][0])-1\\n    gy=2*(lat-s[\\\'gfs_lats\\\'][0])/(s[\\\'gfs_lats\\\'][-1]-s[\\\'gfs_lats\\\'][0])-1\\n    grid=np.stack([gx,gy],-1)[None].astype(\\\'f4\\\')\\n    return static[None],grid\\n\\ndef prepare(X,T,s):\\n    w,mask=windows(X,T); d=fields(w,s[\\\'gfs_lats\\\'],s[\\\'gfs_lons\\\'])\\n    x=np.concatenate([d[:,2],d[:,1],d[:,0]],axis=-1).transpose(0,3,1,2).copy()\\n    dates=pd.to_datetime(T,format=\\\'%Y%m%d%H\\\'); h=dates.hour.to_numpy(); day=dates.dayofyear.to_numpy()\\n    a=h*2*np.pi/24; decl=.409*np.sin(2*np.pi*(day-81)/365.25)\\n    # Solar declination and UTC hour; longitude is applied inside the decoder.\\n    t=np.column_stack([np.sin(a),np.cos(a),np.sin(2*a),np.cos(2*a),np.sin(decl),np.cos(decl),mask]).astype(\\\'f4\\\')\\n    return x,t,h\\n\\nclass Block(nn.Module):\\n    def __init__(self,c,d=1):\\n        super().__init__(); self.f=nn.Sequential(nn.Conv2d(c,c,3,padding=d,dilation=d),nn.GroupNorm(8,c),nn.SiLU(),nn.Conv2d(c,c,3,padding=1),nn.GroupNorm(8,c))\\n    def forward(self,x): return F.silu(x+.5*self.f(x))\\n\\nclass ConvLSTM(nn.Module):\\n    def __init__(self,w):\\n        super().__init__(); self.gates=nn.Conv2d(2*w,4*w,3,padding=1)\\n    def forward(self,x,h,c):\\n        i,f,o,g=self.gates(torch.cat([x,h],1)).chunk(4,1)\\n        c=(f+1).sigmoid()*c+i.sigmoid()*g.tanh(); return o.sigmoid()*c.tanh(),c\\n\\nclass TerrainFlow(nn.Module):\\n    def __init__(self,static,grid,lat,lon,mu,sd,width=48,physics=True):\\n        super().__init__(); self.physics=physics; self.width=width\\n        for name,a in [(\\\'geo\\\',static),(\\\'grid\\\',grid),(\\\'latitude\\\',np.deg2rad(lat)[None,None,:,None]),(\\\'longitude\\\',np.deg2rad(lon)[None,None,None,:]),(\\\'mu\\\',mu[None,:4,None,None]),(\\\'sd\\\',sd[None,:4,None,None])]:\\n            self.register_buffer(name,torch.from_numpy(np.asarray(a,dtype=\\\'f4\\\')))\\n        self.stem=nn.Sequential(nn.Conv2d(24,width,3,padding=1),nn.GroupNorm(8,width),nn.SiLU()); self.temporal=ConvLSTM(width); self.context=Block(width,2)\\n        self.pos=nn.Parameter(torch.zeros(1,17*13,width)); nn.init.normal_(self.pos,std=.02)\\n        self.attn=nn.MultiheadAttention(width,4,batch_first=True,dropout=.05); self.norm=nn.LayerNorm(width)\\n        self.geoenc=nn.Sequential(nn.Conv2d(10,24,3,padding=1),nn.SiLU(),nn.Conv2d(24,24,3,padding=1),nn.SiLU())\\n        # Static terrain-conditioned experts, high-resolution local GFS and dynamic physical interactions.\\n        fine=16 if width<=24 else 32\\n        self.fuse=nn.Sequential(nn.Conv2d(width+45+24+6,fine,1),nn.GroupNorm(4,fine),nn.SiLU(),nn.Conv2d(fine,fine,3,padding=1,groups=4),nn.SiLU())\\n        self.thermo=nn.Sequential(nn.Conv2d(fine,fine,1),nn.SiLU(),nn.Conv2d(fine,2,1))\\n        self.wind=nn.Sequential(nn.Conv2d(fine,fine,1),nn.SiLU(),nn.Conv2d(fine,2,1))\\n    def up(self,x):\\n        with torch.autocast(device_type=x.device.type,enabled=False):\\n            return F.grid_sample(x.float(),self.grid.expand(len(x),-1,-1,-1),align_corners=True,padding_mode=\\\'border\\\')\\n    def forward(self,x,t):\\n        n=len(x)\\n        if not self.physics:\\n            x=x.reshape(n,3,15,17,13); x=torch.cat([x[:,:,:4],torch.zeros_like(x[:,:,4:])],2).reshape(n,45,17,13)\\n        h=x.new_zeros(n,self.width,17,13); c=h\\n        for k in (2,1,0):\\n            z=self.stem(torch.cat([x[:,k*15:(k+1)*15],t[:,:,None,None].expand(-1,-1,17,13)],1))\\n            hn,cn=self.temporal(z,h,c); valid=t[:,8-k,None,None,None]; h=valid*hn+(1-valid)*h; c=valid*cn+(1-valid)*c\\n        low=self.context(h); seq=low.flatten(2).transpose(1,2)\\n        z=self.norm(seq+self.pos); a,_=self.attn(z,z,z,need_weights=False); ctx=(seq+a).transpose(1,2).reshape_as(low)\\n        raw=self.up(x); uv=raw[:,2:4]*self.sd[:,2:4]+self.mu[:,2:4]; u,v=uv[:,0:1],uv[:,1:2]\\n        sx,sy=self.geo[:,1:2],self.geo[:,2:3]; slope=torch.sqrt(sx*sx+sy*sy+1e-6)\\n        upslope=(u*sx+v*sy)/slope; along=(-u*sy+v*sx)/slope\\n        sh,ch,sd,cd=[t[:,i:i+1,None,None] for i in (0,1,4,5)]\\n        # Solar hour angle = UTC phase + longitude - pi; neglect equation of time.\\n        cosh=-(ch*torch.cos(self.longitude)-sh*torch.sin(self.longitude))\\n        sinh=-(sh*torch.cos(self.longitude)+ch*torch.sin(self.longitude))\\n        east=-cd*sinh; north=torch.cos(self.latitude)*sd-torch.sin(self.latitude)*cd*cosh\\n        up=torch.sin(self.latitude)*sd+torch.cos(self.latitude)*cd*cosh\\n        incidence=((up-sx*east-sy*north)/torch.sqrt(1+sx*sx+sy*sy)).clamp(0,1)*(up>0)\\n        dyn=torch.cat([upslope/5,along/5,(u*sx+v*sy)/2,up.clamp(0,1).expand(-1,-1,70,35),incidence,torch.sqrt(u*u+v*v+1e-6)/5],1)\\n        if not self.physics: dyn=torch.zeros_like(dyn)\\n        geo=self.geoenc(self.geo).expand(n,-1,-1,-1); h=self.fuse(torch.cat([self.up(ctx),raw,geo,dyn],1))\\n        return torch.cat([self.thermo(h),self.wind(h)],1).permute(0,2,3,1)\\n\\nimport hashlib, threading\\nfrom scipy.linalg import cho_factor, cho_solve\\nfrom sklearn.decomposition import PCA\\nLOCK=threading.Lock(); LOG=None\\n\\ndef log(*args):\\n    with LOCK:\\n        msg=\\\' \\\'.join(map(str,args)); print(msg,flush=True)\\n        if LOG is not None:\\n            with LOG.open(\\\'a\\\') as f: f.write(msg+\\\'\\\\n\\\')\\n\\ndef load_all(root):\\n    root=Path(root); names=[\\\'train.npz\\\',\\\'test.npz\\\',\\\'static.npz\\\',\\\'sample_submission.csv\\\']\\n    if root.is_file():\\n        with zipfile.ZipFile(root) as z:\\n            data=[]\\n            for n in names[:3]:\\n                with np.load(io.BytesIO(z.read(n))) as f: data.append(dict(f))\\n            return (*data,pd.read_csv(io.BytesIO(z.read(names[3])),dtype={\\\'id\\\':str}))\\n    if not all((root/n).exists() for n in names):\\n        roots=[p.parent for p in root.rglob(\\\'train.npz\\\') if all((p.parent/n).exists() for n in names)]\\n        if len(roots)!=1: raise ValueError(f\\\'Dataset ambiguo: {roots}\\\')\\n        root=roots[0]\\n    data=[]\\n    for n in names[:3]:\\n        with np.load(root/n) as f: data.append(dict(f))\\n    return (*data,pd.read_csv(root/names[3],dtype={\\\'id\\\':str}))\\n\\ndef anchor(X,t,h,Y,tr,ev,s,alpha=.08):\\n    """Train-only local ridge, atmospheric EOFs, and hourly anomalies."""\\n    start=time.perf_counter(); n=len(tr); H,W=Y.shape[1:3]\\n    a=X[:,:15]; b=(a-X[:,15:30])*t[:,7,None,None,None]; c=(a-X[:,30:45])*t[:,6,None,None,None]\\n    z=np.concatenate([a,b[:,:7],c[:,:7]],1).transpose(0,2,3,1).copy()\\n    z-=climate(z,h,tr)[h]; z/=np.maximum(z[tr].std((0,1,2)),1e-4); z=np.clip(z,-12,12)\\n    flat=z.reshape(len(z),-1); pca=PCA(n_components=min(24,n-1),svd_solver=\\\'randomized\\\',random_state=17); pca.fit(flat[tr]); glob=pca.transform(flat); glob/=np.maximum(glob[tr].std(0),1e-4)\\n    yc=climate(Y,h,tr); target=((Y[tr]-yc[h[tr]])/NORM).reshape(n,-1,4)\\n    lat,lon=np.meshgrid(s[\\\'wrf_lats\\\'],s[\\\'wrf_lons\\\'],indexing=\\\'ij\\\'); iy=np.abs(lat.ravel()[:,None]-s[\\\'gfs_lats\\\']).argmin(1); ix=np.abs(lon.ravel()[:,None]-s[\\\'gfs_lons\\\']).argmin(1)\\n    groups=iy*13+ix; both=np.r_[tr,ev]; pred=np.empty((len(both),H*W,4),dtype=\\\'f4\\\')\\n    for g in np.unique(groups):\\n        r,c=divmod(int(g),13); pix=np.where(groups==g)[0]\\n        cells=sorted(set((int(np.clip(r+dr,0,16)),int(np.clip(c+dc,0,12))) for dr,dc in [(0,0),(-1,0),(1,0),(0,-1),(0,1)]))\\n        d=np.concatenate([z[both,r,c],*[z[both,rr,cc,:7] for rr,cc in cells if (rr,cc)!=(r,c)],glob[both],t[both,:4],t[both,6:]],1).astype(\\\'f8\\\')\\n        d=(d-d[:n].mean(0))/np.maximum(d[:n].std(0),.05); d=np.column_stack([np.ones(len(d)),d]); A=d[:n]; yy=target[:,pix].reshape(n,-1).astype(\\\'f8\\\')\\n        gram=A.T@A; reg=np.full(len(gram),alpha*n); reg[0]=1e-6; gram.flat[::len(gram)+1]+=reg\\n        beta=cho_solve(cho_factor(gram),A.T@yy); pred[:,pix]=(d@beta).reshape(len(both),len(pix),4)\\n    pred=pred.reshape(-1,H,W,4)*NORM+yc[h[both]]; pred[...,0]=pred[...,0].clip(0,100)\\n    assert np.isfinite(pred).all(); log(\\\'Anchor\\\',len(np.unique(groups)),\\\'regions\\\',round(time.perf_counter()-start,2),\\\'s\\\'); return pred[:n],pred[n:]\\n\\nclass AtlasResidual(TerrainFlow):\\n    def __init__(self,*a,**kw):\\n        super().__init__(*a,**kw)\\n        for layer in [self.thermo[-1],self.wind[-1]]:\\n            nn.init.zeros_(layer.weight); nn.init.zeros_(layer.bias)\\n\\ndef auxiliary(p,y,a):\\n    norm=p.new_tensor(NORM); p0=a+p*norm; y0=a+y*norm\\n    def vapor(x):\\n        c=(x[...,1]-273.15).clamp(-60,55); return 6.1094*torch.exp(17.625*c/(243.04+c))*x[...,0].clamp(0,100)/100\\n    speed=(torch.sqrt(p0[...,2:].square().sum(-1)+1e-5)-torch.sqrt(y0[...,2:].square().sum(-1)+1e-5))/2.5\\n    err=p-y\\n    return .02*speed.square().mean()+.015*((vapor(p0)-vapor(y0))/4).square().mean()+.01*(err.diff(dim=1).square().mean()+err.diff(dim=2).square().mean())\\n\\ndef fit(X,t,Y,tr,ev,at,ae,s,device,seed,epochs,kind,smoke,checkpoint=None):\\n    start=time.perf_counter(); mu=np.tile(X[tr,:15].mean((0,2,3)),3); sd=np.tile(np.maximum(X[tr,:15].std((0,2,3)),1e-4),3); geo,grid=geography(s)\\n    with LOCK:\\n        torch.manual_seed(seed); model=AtlasResidual(geo,grid,s[\\\'wrf_lats\\\'],s[\\\'wrf_lons\\\'],mu,sd,width=16 if smoke else 48,physics=kind!=\\\'RawResidual\\\')\\n    model=model.to(device); ema=copy.deepcopy(model).eval(); ema.requires_grad_(False); rng=np.random.default_rng(seed); batch=16 if smoke else 32; amp=device.type==\\\'cuda\\\'\\n    xa=torch.as_tensor((X[tr]-mu[None,:,None,None])/sd[None,:,None,None],device=device); ta=torch.as_tensor(t[tr],device=device); aa=torch.as_tensor(at,device=device); ya=torch.as_tensor((Y[tr]-at)/NORM,device=device)\\n    opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.001); sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=math.ceil(len(tr)/batch)*epochs,eta_min=2e-5); scaler=torch.amp.GradScaler(\\\'cuda\\\',enabled=amp); updates=0\\n    log(kind,device,seed,\\\'params\\\',sum(p.numel() for p in model.parameters()))\\n    for ep in range(epochs):\\n        model.train(); total=0.; perm=rng.permutation(len(tr))\\n        for off in range(0,len(tr),batch):\\n            ids=torch.as_tensor(perm[off:off+batch],device=device); opt.zero_grad(set_to_none=True)\\n            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp): p=model(xa[ids],ta[ids])\\n            p=p.float(); mse=(p-ya[ids]).square().mean((0,1,2)); loss=.6*torch.sqrt(mse+1e-6).mean()+.4*mse.mean()\\n            if kind==\\\'PhysicalResidual\\\': loss=loss+auxiliary(p,ya[ids],aa[ids])\\n            if kind==\\\'WindResidual\\\':\\n                ww=p.new_tensor([.35,.65]); loss=(ww*(.6*torch.sqrt(mse[2:]+1e-6)+.4*mse[2:])).sum()\\n                err=(p-ya[ids])[...,2:]; loss=loss+.01*(err.diff(dim=1).square().mean()+err.diff(dim=2).square().mean())\\n            if not torch.isfinite(loss): raise FloatingPointError(\\\'Loss no finita\\\')\\n            scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),1.); prev=scaler.get_scale(); scaler.step(opt); scaler.update()\\n            if scaler.get_scale()>=prev:\\n                sched.step(); updates+=1\\n                with torch.no_grad(): torch._foreach_lerp_(list(ema.parameters()),list(model.parameters()),1-min(.995,(1+updates)/(10+updates)))\\n            total+=float(loss.detach())*len(ids)\\n        log(kind,device,seed,f\\\'ep={ep+1}/{epochs} loss={total/len(tr):.6f} lr={opt.param_groups[0]["lr"]:.6f} elapsed={time.perf_counter()-start:.1f}s\\\')\\n    preds=[]; ema.eval()\\n    with torch.inference_mode():\\n        for k in range(0,len(ev),batch):\\n            idx=ev[k:k+batch]; xb=torch.as_tensor((X[idx]-mu[None,:,None,None])/sd[None,:,None,None],device=device); tb=torch.as_tensor(t[idx],device=device)\\n            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp): p=ema(xb,tb)\\n            preds.append(p.float().cpu().numpy())\\n    pred=np.concatenate(preds)*NORM+ae; pred[...,0]=pred[...,0].clip(0,100); assert np.isfinite(pred).all()\\n    if checkpoint is not None: torch.save(dict(state_dict=ema.cpu().state_dict(),mu=mu,sd=sd,kind=kind,seed=seed),checkpoint)\\n    return pred\\n\\ndef ensemble(X,t,Y,tr,ev,at,ae,s,devices,epochs,kind,smoke,out=None):\\n    seeds=[42] if smoke else [42,123]\\n    with ThreadPoolExecutor(max_workers=len(devices)) as pool:\\n        fs=[pool.submit(fit,X,t,Y,tr,ev,at,ae,s,devices[i%len(devices)],seed,epochs,kind,smoke,None if out is None else out/f\\\'{kind}_{seed}.pt\\\') for i,seed in enumerate(seeds)]\\n        return np.mean([f.result() for f in fs],axis=0)\\n\\ndef weights(a,b,y):\\n    d=(b-a).astype(\\\'f8\\\'); r=(y-a).astype(\\\'f8\\\'); axes=(0,1,2); return (.85*np.clip((d*r).sum(axes)/np.maximum((d*d).sum(axes),1e-10),0,1)).astype(\\\'f4\\\')\\n\\ndef blend(a,b,w):\\n    p=a+(b-a)*w; p[...,0]=p[...,0].clip(0,100); return p\\n\\ndef save_csv(template,times,p,dest):\\n    assert p.shape==(len(times),70,35,4)\\n    ids=[f\\\'{t}_{r:02d}_{c:02d}\\\' for t in times for r in range(70) for c in range(35)]\\n    assert len(ids)==646800 and len(template)==646800 and template.id.is_unique and set(ids)==set(template.id)\\n    d=pd.DataFrame(p.reshape(-1,4),columns=[\\\'RH\\\',\\\'T\\\',\\\'U\\\',\\\'V\\\'],index=ids).loc[template.id].reset_index(drop=True); d.insert(0,\\\'id\\\',template.id.to_numpy())\\n    assert np.isfinite(d.iloc[:,1:].to_numpy()).all() and d.RH.between(0,100).all(); tmp=dest.with_suffix(\\\'.tmp\\\'); d.to_csv(tmp,index=False); tmp.replace(dest)\\n    return hashlib.sha256(dest.read_bytes()).hexdigest()\\n\\ndef main(root,out,smoke=False,anchor_only=False):\\n    global LOG\\n    start=time.perf_counter(); out=Path(out); out.mkdir(parents=True,exist_ok=True); LOG=out/\\\'training.log\\\'; LOG.write_text(\\\'\\\')\\n    # Avoid mistaking an older successful run for a new one.\\n    for name in (\\\'submission_atlas_v3_1.csv\\\',\\\'report.json\\\'):\\n        old=out/name\\n        if old.exists(): old.rename(out/(name+\\\'.previous\\\'))\\n    torch.set_num_threads(4 if smoke or anchor_only else 2); train,test,s,template=load_all(root); Y=train[\\\'Y\\\']; T=train[\\\'times\\\'].astype(str); TT=test[\\\'times\\\'].astype(str)\\n    assert s[\\\'vars\\\'].tolist()==[\\\'RH\\\',\\\'T\\\',\\\'U\\\',\\\'V\\\'] and np.all(T[1:]>T[:-1]) and np.all(TT[1:]>TT[:-1]) and T[-1]<TT[0]\\n    assert np.isfinite(Y).all() and np.isfinite(train[\\\'X\\\']).all() and np.isfinite(test[\\\'X\\\']).all()\\n    X,t,h=prepare(train[\\\'X\\\'],T,s); XT,tT,hT=prepare(test[\\\'X\\\'],TT,s)\\n    devices=[torch.device(\\\'cpu\\\')] if smoke or anchor_only else [torch.device(f\\\'cuda:{i}\\\') for i in range(min(2,torch.cuda.device_count()))]\\n    if not devices: raise RuntimeError(\\\'Seleccionar GPU T4 x2\\\')\\n    epochs=1 if smoke else 48; log(\\\'Atlas v3.1\\\',devices,\\\'smoke\\\',smoke,\\\'anchor_only\\\',anchor_only,\\\'epochs\\\',epochs)\\n    rows=[]; models=[\\\'RawResidual\\\',\\\'PhysicsResidual\\\',\\\'PhysicalResidual\\\']; chosen=None; learned=[]\\n    for i,(name,cut,cal,end) in enumerate(CUTS[:1] if smoke else CUTS):\\n        tr=np.where(T<cut)[0]; ca=np.where((T>=cut)&(T<cal)&(h%3==0))[0]; ev=np.where((T>=cal)&(T<end)&(h%3==0))[0]\\n        assert T[tr[-1]]<T[ca[0]] and T[ca[-1]]<T[ev[0]]\\n        if smoke: tr=tr[::13]\\n        both=np.r_[ca,ev]; at,ae=anchor(X,t,h,Y,tr,both,s); a=ae[len(ca):]; score,rm=metric(a,Y[ev]); rows.append(dict(fold=name,model=\\\'Anchor\\\',skill=score,reference=BASE[name],**dict(zip([\\\'RMSE_RH\\\',\\\'RMSE_T\\\',\\\'RMSE_U\\\',\\\'RMSE_V\\\'],rm.tolist())))); log(rows[-1])\\n        if not anchor_only:\\n            for kind in (models if i<2 else [chosen]):\\n                pred=ensemble(X,t,Y,tr,both,at,ae,s,devices,epochs,kind,smoke)\\n                w=weights(ae[:len(ca)],pred[:len(ca)],Y[ca]) if i<2 else np.mean([w for k,w in learned if k==kind],0); q=blend(a,pred[len(ca):],w); score,rm=metric(q,Y[ev]); rows.append(dict(fold=name,model=kind,skill=score,reference=BASE[name],weights=w.tolist(),**dict(zip([\\\'RMSE_RH\\\',\\\'RMSE_T\\\',\\\'RMSE_U\\\',\\\'RMSE_V\\\'],rm.tolist())))); log(rows[-1]); np.savez_compressed(out/f\\\'{name.replace(" ","_")}_{kind}.npz\\\',times=T[ev],pred=q,anchor=a,neural=pred[len(ca):])\\n                if i<2: learned.append((kind,w))\\n            if i==1:\\n                chosen=max(models,key=lambda k:np.mean([r[\\\'skill\\\'] for r in rows if r[\\\'model\\\']==k])); log(\\\'FROZEN BEFORE AUDIT\\\',chosen)\\n        pd.DataFrame(rows).to_csv(out/\\\'validation_metrics.csv\\\',index=False)\\n    report=dict(model=\\\'Atlas v3.1\\\',smoke=smoke,anchor_only=anchor_only,results=rows,recommend_submit=False)\\n    if not anchor_only:\\n        if smoke: chosen=models[0]\\n        gains={r[\\\'fold\\\']:r[\\\'skill\\\']-r[\\\'reference\\\'] for r in rows if r[\\\'model\\\']==chosen}; gate=not smoke and len(gains)==4 and min(gains[\\\'Mayo\\\'],gains[\\\'Junio I\\\'])>0 and np.mean([gains[\\\'Mayo\\\'],gains[\\\'Junio I\\\']])>=.003 and gains[\\\'Junio II\\\']>=.003 and gains[\\\'Abril\\\']>=-.002\\n        report.update(chosen=chosen,gains=gains,recommend_submit=bool(gate)); log(\\\'VALIDATION_GATE\\\',gate,\\\'gains\\\',gains)\\n        if gate or smoke:\\n            Xall=np.concatenate([X,XT]); tall=np.concatenate([t,tT]); hall=np.r_[h,hT]; tr=np.arange(len(T)); ev=len(T)+np.where(hT%3==0)[0]\\n            if smoke: tr=tr[::13]\\n            at,ae=anchor(Xall,tall,hall,Y,tr,ev,s); pred=ensemble(Xall,tall,Y,tr,ev,at,ae,s,devices,epochs,chosen,smoke,out); w=np.mean([w for k,w in learned if k==chosen],0); p=blend(ae,pred,w)\\n            dest=out/(\\\'SMOKE_NO_SUBIR.csv\\\' if smoke else \\\'submission_atlas_v3_1.csv\\\'); sha=save_csv(template,TT[hT%3==0],p,dest); report.update(weights=w.tolist(),sha256=sha,rows=len(template),csv=str(dest)); log(dest,sha)\\n        else: log(\\\'No CSV competitivo. Conservar Ultralisk v2.7.\\\')\\n    report[\\\'seconds\\\']=time.perf_counter()-start; (out/\\\'report.json\\\').write_text(json.dumps(report,indent=2)); log(\\\'RECOMMEND_SUBMIT=\\\'+(\\\'YES\\\' if report[\\\'recommend_submit\\\'] else \\\'NO\\\'))\\n\\n\'\n'
    'ULTRA_SOURCE=\'# Ultralisk-v2.4 | ShadowBytes | ApuCast | Kaggle T4 x2\\nimport os, time, json, copy, hashlib, threading, platform\\nfrom pathlib import Path\\nfrom concurrent.futures import ThreadPoolExecutor\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\nfrom torch import nn\\nfrom torch.nn import functional as F\\nfrom IPython.display import display, FileLink\\nSMOKE=os.environ.get(\\\'ULTRALISK_SMOKE\\\')==\\\'1\\\'  # prueba tecnica, NO subir\\nOUT=Path(os.environ.get(\\\'ULTRALISK_OUT\\\',\\\'/kaggle/working/Ultralisk-v2.4\\\')); OUT.mkdir(parents=True,exist_ok=True)\\nLOG=OUT/\\\'training.log\\\'; LOG.write_text(\\\'\\\'); LOCK=threading.Lock(); START=time.perf_counter()\\nBATCH=32; EPOCHS_REF=25; EPOCHS_NEW=36\\nSEEDS={\\\'CNN\\\':(0,1,2),\\\'ResNet\\\':(0,1,2),\\\'Mutalisk\\\':(42,123),\\\'Viper\\\':(42,123)}\\nCUTS=[(\\\'2024050900\\\',\\\'2024051600\\\',\\\'2024060100\\\'),(\\\'2024052500\\\',\\\'2024060100\\\',\\\'2024061600\\\'),(\\\'2024060900\\\',\\\'2024061600\\\',\\\'2024070100\\\')]\\nVARS=[\\\'RH\\\',\\\'T\\\',\\\'U\\\',\\\'V\\\']; NORM=np.array([25.360,2.0652,2.6776,2.0876])\\n# Valores historicos de versions/v3/backtest.json (solo referencia, no etiquetas).\\nHISTORICAL={1:.42292221906027727,2:.4136267261457105,3:.42723285641743414}\\nMIN_DEV_GAIN=.003; MIN_AUDIT_GAIN=.002\\ndef log(msg):\\n    line=f\\\'[{time.perf_counter()-START:8.1f}s] {msg}\\\'\\n    with LOCK:\\n        print(line,flush=True)\\n        with LOG.open(\\\'a\\\') as f: f.write(line+\\\'\\\\n\\\')\\nif not SMOKE and (torch.cuda.device_count()!=2 or any(\\\'T4\\\' not in torch.cuda.get_device_name(i) for i in range(2))): raise RuntimeError(\\\'Selecciona Accelerator > GPU T4 x2\\\')\\nDEVICES=[torch.device(\\\'cpu\\\')] if SMOKE else [torch.device(\\\'cuda:0\\\'),torch.device(\\\'cuda:1\\\')]\\ntorch.set_num_threads(2); torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True\\ncpu_name=platform.processor()\\nif Path(\\\'/proc/cpuinfo\\\').is_file():\\n    cpu_name=next((l.split(\\\':\\\',1)[1].strip() for l in Path(\\\'/proc/cpuinfo\\\').read_text().splitlines() if l.startswith(\\\'model name\\\')),cpu_name)\\nlog(f\\\'torch={torch.__version__} CPU={cpu_name} cores={os.cpu_count()} torch_threads={torch.get_num_threads()} smoke={SMOKE}\\\')\\nfor d in DEVICES:\\n    if d.type==\\\'cuda\\\': log(f\\\'{d} {torch.cuda.get_device_name(d)} VRAM={torch.cuda.get_device_properties(d).total_memory/2**30:.2f} GiB\\\')\\n# Datos oficiales\\nroot=Path(os.environ.get(\\\'ULTRALISK_DATA\\\',\\\'/kaggle/input\\\')); required=(\\\'train.npz\\\',\\\'test.npz\\\',\\\'static.npz\\\',\\\'sample_submission.csv\\\')\\nfolders=[p.parent for p in root.rglob(\\\'train.npz\\\') if all((p.parent/k).is_file() for k in required)]\\nofficial=[p for p in folders if \\\'hackathon-ia-pucp-2026\\\' in str(p)]\\nif len(official)==1: DATA=official[0]\\nelif len(folders)==1: DATA=folders[0]\\nelse: raise RuntimeError(f\\\'Dataset oficial no identificado: {folders}\\\')\\nwith np.load(DATA/\\\'train.npz\\\') as f: X,Y,TT=f[\\\'X\\\'].astype(\\\'f4\\\'),f[\\\'Y\\\'].astype(\\\'f4\\\'),f[\\\'times\\\'].astype(str)\\nwith np.load(DATA/\\\'test.npz\\\') as f: XT,TE=f[\\\'X\\\'].astype(\\\'f4\\\'),f[\\\'times\\\'].astype(str)\\nwith np.load(DATA/\\\'static.npz\\\') as f:\\n    DEM=f[\\\'dem\\\'].astype(\\\'f4\\\'); glat,glon,wlat,wlon=[f[k] for k in (\\\'gfs_lats\\\',\\\'gfs_lons\\\',\\\'wrf_lats\\\',\\\'wrf_lons\\\')]\\nTEMPLATE=pd.read_csv(DATA/\\\'sample_submission.csv\\\',dtype={\\\'id\\\':str})\\nassert X.shape==(2935,17,13,4) and Y.shape==(2935,70,35,4) and XT.shape==(792,17,13,4) and DEM.shape==(70,35)\\nassert TEMPLATE.shape==(646800,5) and TEMPLATE.columns.tolist()==[\\\'id\\\',*VARS] and TEMPLATE.id.is_unique\\nassert all(np.isfinite(a).all() for a in (X,Y,XT,DEM)) and max(TT)<min(TE)\\nparts=TEMPLATE.id.str.split(\\\'_\\\',expand=True); ids_t=parts[0].map({s:i for i,s in enumerate(TE)}); assert ids_t.notna().all()\\nti=ids_t.to_numpy(dtype=int); ri=parts[1].astype(int).to_numpy(); ci=parts[2].astype(int).to_numpy()\\nassert (ri>=0).all() and (ri<70).all() and (ci>=0).all() and (ci<35).all()\\nassert len(np.unique(ti))==264 and np.all(np.bincount(ti)[np.unique(ti)]==2450)\\nHOURS=np.unique([int(s[-2:]) for s in parts[0].unique()]); H=np.array([int(s[-2:]) for s in TT]); HT=np.array([int(s[-2:]) for s in TE])\\nlog(f\\\'Data={DATA} train={X.shape} test={XT.shape} horas={HOURS.tolist()}\\\')\\ndef windows(x,times):\\n    dates=pd.to_datetime(times,format=\\\'%Y%m%d%H\\\'); lookup={t:i for i,t in enumerate(dates)}\\n    return x[np.array([[lookup.get(t-pd.Timedelta(hours=k),i) for k in (2,1,0)] for i,t in enumerate(dates)])]\\nW,WT=windows(X,TT),windows(XT,TE)\\nlat,lon=np.meshgrid(wlat,wlon,indexing=\\\'ij\\\'); grid=np.stack([(lon-glon[0])/(glon[-1]-glon[0])*2-1,(lat-glat[0])/(glat[-1]-glat[0])*2-1],-1)[None].astype(\\\'f4\\\')\\ndef z(a): return ((a-a.mean())/max(float(a.std()),1e-6)).astype(\\\'f4\\\')\\ndef smooth(a,k):\\n    p=k//2; return F.avg_pool2d(F.pad(torch.from_numpy(a[None,None]),(p,p,p,p),mode=\\\'replicate\\\'),k,stride=1)[0,0].numpy()\\ndy,dx=np.gradient(DEM)\\ngeo=np.stack([z(DEM),z(dx),z(dy),z(np.hypot(dx,dy)),z(DEM-smooth(DEM,3)),z(DEM-smooth(DEM,7)),z(DEM-smooth(DEM,15)),z(lat),z(lon),z(np.gradient(dy,axis=0)+np.gradient(dx,axis=1))])[None].astype(\\\'f4\\\')\\ndef stats(ix):\\n    yy,xx,hh=Y[ix],X[ix],H[ix]\\n    def climate(a): return np.stack([a[hh==h].mean(0) if np.any(hh==h) else a.mean(0) for h in range(24)])\\n    cl=climate(yy)\\n    return dict(mu=xx.mean((0,1,2)),sd=np.maximum(xx.std((0,1,2)),1e-6),clim=cl,gfs_clim=climate(xx),scale=np.maximum((yy-cl[hh]).std((0,1,2)),1e-6))\\ndef score(p,y):\\n    rm=np.sqrt(np.mean(np.square(p.astype(\\\'f8\\\')-y),axis=(0,1,2))); return float(1-(rm/NORM).mean()),rm\\ndef clip(p):\\n    p=p.astype(\\\'f4\\\',copy=False); p[...,0]=p[...,0].clip(0,100); assert np.isfinite(p).all(); return p\\ndef tensor(a,d): return torch.as_tensor(a,dtype=torch.float32,device=d)\\ndef sync(d):\\n    if d.type==\\\'cuda\\\': torch.cuda.synchronize(d)\\n# Arquitecturas\\nclass Residual(nn.Module):\\n    def __init__(self,a,b,dilation=1):\\n        super().__init__(); self.c1=nn.Conv2d(a,b,3,padding=dilation,dilation=dilation); self.c2=nn.Conv2d(b,b,3,padding=1); self.skip=nn.Conv2d(a,b,1) if a!=b else nn.Identity()\\n    def forward(self,x): return F.gelu(self.c2(F.gelu(self.c1(x)))+self.skip(x))\\ndef inputs(x,h):\\n    n=len(x); a=h*(2*np.pi/24); clock=torch.stack([a.sin(),a.cos()],1)[:,:,None,None].expand(-1,-1,17,13)\\n    return x.permute(0,1,4,2,3).reshape(n,12,17,13),clock\\nclass Reference(nn.Module):\\n    def __init__(self,kind):\\n        super().__init__(); w=64\\n        self.enc=nn.Sequential(nn.Conv2d(14,w,3,padding=1),nn.GELU(),nn.Conv2d(w,w,3,padding=1),nn.GELU())\\n        self.static=nn.Parameter(torch.zeros(1,8,70,35))\\n        self.dec=nn.Sequential(nn.Conv2d(73,w,3,padding=1),nn.GELU(),nn.Conv2d(w,w,3,padding=1),nn.GELU(),nn.Conv2d(w,4,1))\\n        if kind==\\\'ResNet\\\': self.enc=Residual(14,w); self.dec=nn.Sequential(Residual(73,w),nn.Conv2d(w,4,1))\\n        self.register_buffer(\\\'grid\\\',torch.from_numpy(grid)); self.register_buffer(\\\'dem\\\',torch.from_numpy(z(DEM)[None,None]))\\n    def forward(self,x,h):\\n        n=len(x); fields,clock=inputs(x,h); v=self.enc(torch.cat([fields,clock],1))\\n        with torch.autocast(device_type=x.device.type,enabled=False): v=F.grid_sample(v.float(),self.grid.expand(n,-1,-1,-1),align_corners=True)\\n        return self.dec(torch.cat([v,self.dem.expand(n,-1,-1,-1),self.static.expand(n,-1,-1,-1)],1)).permute(0,2,3,1)\\nclass GeoWind(nn.Module):\\n    def __init__(self,kind,st):\\n        super().__init__(); self.kind=kind; w=48\\n        self.register_buffer(\\\'grid\\\',torch.from_numpy(grid)); self.register_buffer(\\\'geo\\\',torch.from_numpy(geo)); self.register_buffer(\\\'gcl\\\',torch.from_numpy(((st[\\\'gfs_clim\\\']-st[\\\'mu\\\'])/st[\\\'sd\\\']).astype(\\\'f4\\\')))\\n        self.enc=nn.Sequential(Residual(22,w),Residual(w,w,2),Residual(w,w)); self.map=nn.Parameter(torch.zeros(1,8,70,35)); self.fuse=Residual(w+26,w)\\n        if kind==\\\'Mutalisk\\\': self.down=nn.Sequential(Residual(w,64),Residual(64,64,2)); self.up=Residual(w+64,w)\\n        else: self.wind=nn.Sequential(Residual(w,w,2),Residual(w,w,3))\\n        self.thermal=nn.Sequential(Residual(w,32),nn.Conv2d(32,2,1)); self.velocity=nn.Sequential(Residual(w,32),nn.Conv2d(32,2,1))\\n    def interp(self,x):\\n        with torch.autocast(device_type=x.device.type,enabled=False): return F.grid_sample(x.float(),self.grid.expand(len(x),-1,-1,-1),align_corners=True,padding_mode=\\\'border\\\')\\n    def forward(self,x,h):\\n        n=len(x); fields,clock=inputs(x,h); current=x[:,-1]; anomaly=current-self.gcl[h.long()]; trend=current-x[:,0]\\n        av=anomaly.permute(0,3,1,2); cv=current.permute(0,3,1,2); v=self.enc(torch.cat([fields,av,trend.permute(0,3,1,2),clock],1))\\n        fine=self.fuse(torch.cat([self.interp(v),self.interp(torch.cat([cv,av],1)),self.geo.expand(n,-1,-1,-1),self.map.expand(n,-1,-1,-1)],1))\\n        if self.kind==\\\'Mutalisk\\\':\\n            low=self.down(F.avg_pool2d(fine,2)); fine=self.up(torch.cat([fine,F.interpolate(low,size=fine.shape[-2:],mode=\\\'bilinear\\\',align_corners=False)],1)); wind=fine\\n        else: wind=self.wind(fine)\\n        return torch.cat([self.thermal(fine),self.velocity(wind)],1).permute(0,2,3,1)\\ndef make(kind,st): return Reference(kind) if kind in (\\\'CNN\\\',\\\'ResNet\\\') else GeoWind(kind,st)\\n# Benchmark de inferencia, mismo modelo y lote para todos los dispositivos.\\ndef benchmark():\\n    st=stats(np.arange(128)); torch.manual_seed(0); base=Reference(\\\'ResNet\\\').eval(); results=[]\\n    x=(W[:8]-st[\\\'mu\\\'])/st[\\\'sd\\\']; h=H[:8]\\n    for d,amp in [(torch.device(\\\'cpu\\\'),False)]+[(d,a) for d in DEVICES if d.type==\\\'cuda\\\' for a in (False,True)]:\\n        m=copy.deepcopy(base).to(d,memory_format=torch.channels_last); xb,hb=tensor(x,d),tensor(h,d)\\n        with torch.inference_mode():\\n            for _ in range(3):\\n                with torch.autocast(device_type=d.type,dtype=torch.float16,enabled=amp): m(xb,hb)\\n            sync(d); t=time.perf_counter()\\n            for _ in range(8):\\n                with torch.autocast(device_type=d.type,dtype=torch.float16,enabled=amp): m(xb,hb)\\n            sync(d); sec=time.perf_counter()-t\\n        r=dict(device=str(d),amp=amp,batch=8,ms_per_batch=sec/8*1000,samples_per_second=64/sec); results.append(r); log(f\\\'BENCHMARK {r}\\\')\\n    if len(DEVICES)==2:\\n        barrier=threading.Barrier(2)\\n        def both(d):\\n            m=copy.deepcopy(base).to(d,memory_format=torch.channels_last); xb,hb=tensor(x,d),tensor(h,d)\\n            with torch.inference_mode(), torch.autocast(device_type=\\\'cuda\\\',dtype=torch.float16):\\n                for _ in range(3): m(xb,hb)\\n                sync(d); barrier.wait(timeout=60); t=time.perf_counter()\\n                for _ in range(8): m(xb,hb)\\n                sync(d); return time.perf_counter()-t\\n        with ThreadPoolExecutor(max_workers=2) as pool:\\n            sec=max(pool.map(both,DEVICES))\\n        r=dict(device=\\\'T4x2_parallel\\\',amp=True,batch=16,ms_per_batch=sec/8*1000,samples_per_second=128/sec)\\n        results.append(r); log(f\\\'BENCHMARK {r}\\\')\\n    return results\\nBENCH=benchmark() if os.environ.get(\\\'ULTRALISK_BENCH\\\',\\\'0\\\')==\\\'1\\\' else []\\nlog(\\\'Benchmark opcional; iniciando experimentos v2.4\\\')\\n@torch.inference_mode()\\ndef infer(model,x,h,d,amp):\\n    model.eval(); out=[]\\n    for i in range(0,len(x),BATCH):\\n        with torch.autocast(device_type=d.type,dtype=torch.float16,enabled=amp): p=model(x[i:i+BATCH],h[i:i+BATCH])\\n        out.append(p.float().cpu().numpy())\\n    return np.concatenate(out)\\ndef fit_neural(job,d,st,tr,we,he,tag):\\n    kind,seed,model=job; ref=kind in (\\\'CNN\\\',\\\'ResNet\\\'); epochs=1 if SMOKE else (EPOCHS_REF if ref else EPOCHS_NEW)\\n    amp=not ref and d.type==\\\'cuda\\\'; lr=.002 if ref else .0012; rng=np.random.RandomState(seed)\\n    model=model.to(d,memory_format=torch.channels_last); ema=None if ref else copy.deepcopy(model).eval().requires_grad_(False)\\n    x=tensor((W[tr]-st[\\\'mu\\\'])/st[\\\'sd\\\'],d); h=tensor(H[tr],d); y=tensor((Y[tr]-st[\\\'clim\\\'][H[tr]])/st[\\\'scale\\\'],d)\\n    xe=tensor((we-st[\\\'mu\\\'])/st[\\\'sd\\\'],d); he_gpu=tensor(he,d)\\n    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4 if ref else 5e-4)\\n    steps=(len(tr)+BATCH-1)//BATCH; sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=lr,total_steps=steps*epochs)\\n    scaler=torch.amp.GradScaler(\\\'cuda\\\',enabled=amp); ratio=tensor(st[\\\'scale\\\']/NORM,d)\\n    order=np.arange(len(tr)); started=time.perf_counter(); hist=[]\\n    if d.type==\\\'cuda\\\': torch.cuda.reset_peak_memory_stats(d)\\n    log(f\\\'{tag} {kind}:{seed} {d} START params={sum(p.numel() for p in model.parameters()):,} epochs={epochs} amp={amp}\\\')\\n    for ep in range(1,epochs+1):\\n        model.train(); rng.shuffle(order); accum=torch.zeros((),device=d); t0=time.perf_counter()\\n        for k in range(0,len(order),BATCH):\\n            ix=torch.as_tensor(order[k:k+BATCH],device=d,dtype=torch.long); opt.zero_grad(set_to_none=True)\\n            with torch.autocast(device_type=d.type,dtype=torch.float16,enabled=amp): p=model(x[ix],h[ix])\\n            mse=(p.float()-y[ix]).square(); loss=mse.mean() if ref else .5*mse.mean()+.5*((mse.mean((0,1,2))+1e-8).sqrt()*ratio).mean()\\n            scaler.scale(loss).backward()\\n            if not ref: scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),1.)\\n            old=scaler.get_scale(); scaler.step(opt); scaler.update()\\n            if scaler.get_scale()>=old:\\n                sched.step()\\n                if ema is not None:\\n                    with torch.no_grad():\\n                        for a,b in zip(ema.parameters(),model.parameters()): a.lerp_(b,.015)\\n            accum+=loss.detach()*len(ix)\\n        sync(d); sec=time.perf_counter()-t0; value=float(accum.item()/len(tr)); assert np.isfinite(value)\\n        mem=torch.cuda.max_memory_allocated(d)/2**30 if d.type==\\\'cuda\\\' else None\\n        hist.append(dict(epoch=ep,loss=value,seconds=sec,samples_per_second=len(tr)/sec,peak_gib=mem,lr=opt.param_groups[0][\\\'lr\\\']))\\n        log(f\\\'{tag} {kind}:{seed} {d} ep={ep:02d}/{epochs} loss={value:.5f} lr={opt.param_groups[0]["lr"]:.6f} {sec:.2f}s {len(tr)/sec:.1f} samples/s VRAM={mem}\\\')\\n    chosen=model if ref else ema; p=clip(infer(chosen,xe,he_gpu,d,amp)*st[\\\'scale\\\']+st[\\\'clim\\\'][he]); elapsed=time.perf_counter()-started\\n    pd.DataFrame(hist).to_csv(OUT/f\\\'{tag}_{kind}_{seed}_history.csv\\\',index=False)\\n    if tag==\\\'final\\\': torch.save({\\\'kind\\\':kind,\\\'seed\\\':seed,\\\'epochs\\\':epochs,\\\'state_dict\\\':{k:v.detach().cpu() for k,v in chosen.state_dict().items()}},OUT/f\\\'{kind}_{seed}.pt\\\')\\n    log(f\\\'{tag} {kind}:{seed} DONE {elapsed:.1f}s\\\'); model.to(\\\'cpu\\\')\\n    return kind,p,dict(kind=kind,seed=seed,device=str(d),seconds=elapsed,epochs=epochs,peak_gib=mem)\\n# Kernel Ridge no lineal: anomalias del GFS, tendencia y promedio temporal.\\ndef kernel_features(w,h,st):\\n    a=(w-st[\\\'gfs_clim\\\'][(h[:,None]-np.array([2,1,0]))%24])/st[\\\'sd\\\']; a=a.reshape(len(w),3,-1)\\n    return np.concatenate([a[:,2],a.mean(1),.5*(a[:,2]-a[:,0])],axis=1).astype(\\\'f4\\\')\\ndef kernel(a,b):\\n    dot=(a@b.T)/a.shape[1]; dist=(a.square().mean(1)[:,None]+b.square().mean(1)[None,:]-2*dot).clamp_min(0)\\n    return .5*dot+.25*dot.square()+.25*torch.exp(-dist/2)\\ndef fit_kernel(d,st,tr,we,he,tag):\\n    t=time.perf_counter(); a=kernel_features(W[tr],H[tr],st); b=kernel_features(we,he,st)\\n    mu=a.mean(0); sd=np.maximum(a.std(0),.15); a=(a-mu)/sd; b=(b-mu)/sd\\n    ag,bg=tensor(a,d),tensor(b,d); K=kernel(ag,ag); K=(K+K.T)*.5; K.diagonal().add_(.05)\\n    L=torch.linalg.cholesky(K); yy=tensor(((Y[tr]-st[\\\'clim\\\'][H[tr]])/st[\\\'scale\\\']).reshape(len(tr),-1),d)\\n    dual=torch.cholesky_solve(yy,L); p=(kernel(bg,ag)@dual).cpu().numpy().reshape(len(we),70,35,4)\\n    if tag==\\\'final\\\': np.savez_compressed(OUT/\\\'kernel_model.npz\\\',features=a,mu=mu,sd=sd,dual=dual.cpu().numpy())\\n    log(f\\\'{tag} Kernel DONE {time.perf_counter()-t:.2f}s alpha=.05\\\'); return clip(p*st[\\\'scale\\\']+st[\\\'clim\\\'][he])\\nALL_RUNS=[]\\ndef run_stage(tr,we,he,tag,families):\\n    st=stats(tr); jobs=[]\\n    for kind in families:\\n        if kind==\\\'Kernel\\\': continue\\n        for seed in (SEEDS[kind][:1] if SMOKE else SEEDS[kind]):\\n            torch.manual_seed(seed); jobs.append((kind,seed,make(kind,st)))\\n    def worker(group,d): return [fit_neural(j,d,st,tr,we,he,tag) for j in group]\\n    results=[]\\n    if len(DEVICES)==2:\\n        with ThreadPoolExecutor(max_workers=2) as pool:\\n            futures=[pool.submit(worker,jobs[i::2],d) for i,d in enumerate(DEVICES)]\\n            for f in futures: results.extend(f.result())\\n    else: results=worker(jobs,DEVICES[0])\\n    ALL_RUNS.extend([dict(tag=tag,**r) for _,_,r in results]); results.sort(key=lambda x:x[2][\\\'seed\\\'])\\n    pred={k:np.mean([p for kind,p,_ in results if kind==k],axis=0,dtype=np.float32) for k in families if k!=\\\'Kernel\\\'}\\n    if \\\'Kernel\\\' in families: pred[\\\'Kernel\\\']=fit_kernel(DEVICES[0],st,tr,we,he,tag)\\n    if \\\'Kernel\\\' in families: pred[\\\'Clock\\\']=fit_clock(DEVICES[0],st,tr,we,he,tag)\\n    if tag==\\\'final\\\': np.savez_compressed(OUT/\\\'preprocessing.npz\\\',**st,grid=grid,geo=geo,normalizers=NORM)\\n    return pred\\n# Calibracion por variable, separada de evaluacion.\\nCANDIDATES={\\\'Reference\\\':{},\\\'Mutalisk\\\':{\\\'Mutalisk\\\':1.},\\\'Viper\\\':{\\\'Viper\\\':1.},\\\'Neural\\\':{\\\'Mutalisk\\\':.5,\\\'Viper\\\':.5},\\\'Kernel\\\':{\\\'Kernel\\\':1.},\\\'Hybrid\\\':{\\\'Mutalisk\\\':.375,\\\'Viper\\\':.375,\\\'Kernel\\\':.25}}\\ndef fit_alpha(a,b,y,prior=.5,shrink=.2,cap=1.):\\n    d=b.astype(\\\'f8\\\')-a; r=y.astype(\\\'f8\\\')-a; den=(d*d).mean((0,1,2)); num=(r*d).mean((0,1,2))\\n    raw=np.divide(num,den,out=np.full(4,prior),where=den>1e-12); return np.clip((1-shrink)*raw+shrink*prior,0,cap).astype(\\\'f4\\\')\\ndef mix(a,b,w): return clip(a*(1-w)+b*w)\\ndef candidate(p,name): return sum(p[k]*np.float32(w) for k,w in CANDIDATES[name].items())\\ndef calibrate(p,y,name):\\n    rw=fit_alpha(p[\\\'CNN\\\'],p[\\\'ResNet\\\'],y); ref=mix(p[\\\'CNN\\\'],p[\\\'ResNet\\\'],rw)\\n    alpha=np.zeros(4,dtype=\\\'f4\\\') if name==\\\'Reference\\\' else fit_alpha(ref,candidate(p,name),y,prior=.25,shrink=.35,cap=.65)\\n    return rw,alpha\\ndef combine(p,name,rw,alpha):\\n    ref=mix(p[\\\'CNN\\\'],p[\\\'ResNet\\\'],rw); return ref if name==\\\'Reference\\\' else mix(ref,candidate(p,name),alpha)\\n\\n\\n# ================================================================\\n# ULTRALISK v2.4 | CHRONOS | extension incremental de v2.3\\n# No usa WRF oculto, datos externos ni horas futuras de GFS.\\n# ================================================================\\nfrom scipy.optimize import minimize\\nPUBLIC_REFERENCE=.49056\\nOLD_FOLD={1:.439850572420165,2:.4396332592736687,3:.4496978946404211}\\nSTRESS=(\\\'2024040900\\\',\\\'2024041600\\\',\\\'2024050100\\\')\\n# Dos variantes conservadoras, probadas individualmente en CPU en folds 1 y 2.\\nCLOCK_SETTINGS=[(.6,90),(1.2,90)]\\nFAMILIES=[\\\'CNN\\\',\\\'ResNet\\\',\\\'Mutalisk\\\',\\\'Viper\\\',\\\'Kernel\\\']\\nKEYS=FAMILIES+[\\\'Clock\\\']\\nMETHODS=[\\\'ClockReplace\\\',\\\'ClockBlend\\\',\\\'Stack\\\']\\nMIN_DEV=.002; MIN_AUDIT=.002; STRESS_TOL=.002; REPRO_TOL=.0015\\n\\ndef fit_clock(d,st,tr,we,he,tag):\\n    t=time.perf_counter()\\n    a=kernel_features(W[tr],H[tr],st); b=kernel_features(we,he,st)\\n    mu=a.mean(0); sd=np.maximum(a.std(0),.15)\\n    a=(a-mu)/sd; b=(b-mu)/sd\\n    ag,bg=tensor(a,d),tensor(b,d)\\n    h,eh=tensor(H[tr],d),tensor(he,d)\\n    def clock(a,b): return .4+.6*(1+torch.cos((a[:,None]-b[None,:])*(np.pi/12)))/2\\n    K=kernel(ag,ag)*clock(h,h); Q=kernel(bg,ag)*clock(eh,h)\\n    K=(K+K.T)*.5\\n    stamps=pd.to_datetime(TT[tr],format=\\\'%Y%m%d%H\\\')\\n    age=tensor(np.asarray((stamps.max()-stamps).total_seconds()/86400,dtype=\\\'f4\\\'),d)\\n    # Edad calculada solo desde fechas del entrenamiento. Un dia mas aproxima\\n    # el comienzo del siguiente periodo de calibracion (igual en todos los folds).\\n    age=age+1.\\n    target=tensor(((Y[tr]-st[\\\'clim\\\'][H[tr]])/st[\\\'scale\\\']).reshape(len(tr),-1),d)\\n    preds=[]\\n    for alpha,days in CLOCK_SETTINGS:\\n        Kreg=K.clone(); Kreg.diagonal().add_(alpha*torch.exp(age/days))\\n        L=torch.linalg.cholesky(Kreg); dual=torch.cholesky_solve(target,L)\\n        p=(Q@dual).cpu().numpy().reshape(len(we),70,35,4)\\n        preds.append(clip(p*st[\\\'scale\\\']+st[\\\'clim\\\'][he]))\\n        if tag==\\\'final\\\':\\n            np.savez_compressed(OUT/f\\\'clock_alpha_{alpha}.npz\\\',features=a,mu=mu,sd=sd,dual=dual.cpu().numpy(),hours=H[tr],alpha=alpha,days=days)\\n        del Kreg,L,dual,p\\n        log(f\\\'{tag} Clock alpha={alpha} days={days} done elapsed={time.perf_counter()-t:.2f}s\\\')\\n    return clip(np.mean(preds,axis=0,dtype=np.float32))\\n\\ndef old_weights(p,y):\\n    rw,alpha=calibrate(p,y,\\\'Hybrid\\\')\\n    # Convertir mezcla jerarquica v2.3 en pesos efectivos por variable.\\n    return np.stack([(1-alpha)*(1-rw),(1-alpha)*rw,.375*alpha,.375*alpha,.25*alpha,np.zeros(4,dtype=\\\'f4\\\')])\\n\\ndef weighted(p,w):\\n    return clip(sum(p[k]*w[i].astype(\\\'f4\\\') for i,k in enumerate(KEYS)))\\n\\ndef fit_method(p,y,method):\\n    w=old_weights(p,y)\\n    if method==\\\'Baseline\\\': return w\\n    if method==\\\'ClockReplace\\\':\\n        # Misma arquitectura y calibracion que v2.3, sustituyendo 75% del\\n        # componente kernel por la variante horaria/reciente.\\n        ref=mix(p[\\\'CNN\\\'],p[\\\'ResNet\\\'],fit_alpha(p[\\\'CNN\\\'],p[\\\'ResNet\\\'],y))\\n        fresh=.375*p[\\\'Mutalisk\\\']+.375*p[\\\'Viper\\\']+.0625*p[\\\'Kernel\\\']+.1875*p[\\\'Clock\\\']\\n        alpha=fit_alpha(ref,fresh,y,prior=.25,shrink=.35,cap=.65)\\n        rw=fit_alpha(p[\\\'CNN\\\'],p[\\\'ResNet\\\'],y)\\n        return np.stack([(1-alpha)*(1-rw),(1-alpha)*rw,.375*alpha,.375*alpha,.0625*alpha,.1875*alpha])\\n    if method==\\\'ClockBlend\\\':\\n        a=fit_alpha(weighted(p,w),p[\\\'Clock\\\'],y,prior=.15,shrink=.4,cap=.55)\\n        w=w*(1-a); w[-1]+=a; return w\\n    if method!=\\\'Stack\\\': raise ValueError(method)\\n    # Regresion convexa multisalida. Se ajusta solo en la semana de calibracion.\\n    # Penalizacion en escala de RMSE oficial y limites para evitar pesos extremos.\\n    out=[]\\n    for v in range(4):\\n        prior=w[:,v].astype(\\\'f8\\\')*.85; prior[-1]+=.15\\n        # Residuos normalizados, matrices de solo 6x6.\\n        a=np.stack([((p[k][...,v]-y[...,v])/NORM[v]).reshape(-1) for k in KEYS],axis=0).astype(\\\'f8\\\')\\n        gram=(a@a.T)/a.shape[1]; reg=.004\\n        def fun(z): return float(z@gram@z+reg*np.square(z-prior).sum())\\n        def jac(z): return 2*(gram@z+reg*(z-prior))\\n        bounds=[(max(0,float(t)-.25),min(1,float(t)+.25)) for t in prior]; bounds[-1]=(0,.55)\\n        opt=minimize(fun,prior,jac=jac,bounds=bounds,constraints=[dict(type=\\\'eq\\\',fun=lambda z:z.sum()-1,jac=lambda z:np.ones(6))],method=\\\'SLSQP\\\',options=dict(maxiter=100,ftol=1e-11))\\n        z=opt.x if opt.success and np.isfinite(opt.x).all() and abs(opt.x.sum()-1)<1e-6 else prior\\n        z=np.maximum(z,0); out.append(z/z.sum())\\n    return np.asarray(out,dtype=\\\'f4\\\').T\\n\\n# Cache de predicciones de calibracion y validacion para futuras versiones.\\n# Las etiquetas se obtienen exclusivamente de train.npz.\\ndef regional(d,st,tr,we,he,alpha=1.2,sigma=.65,local=.7,days=90,save_model=False,model_prefix="regional"):\\n    a=kernel_features(W[tr],H[tr],st); b=kernel_features(we,he,st); mu=a.mean(0); sd=np.maximum(a.std(0),.15)\\n    ag,bg=tensor((a-mu)/sd,d),tensor((b-mu)/sd,d)\\n    G=kernel(ag,ag); Q=kernel(bg,ag)\\n    h,eh=tensor(H[tr],d),tensor(he,d)\\n    clock=lambda a,b: .4+.6*(1+torch.cos((a[:,None]-b[None,:])*np.pi/12))/2\\n    C,D=clock(h,h),clock(eh,h)\\n    stamps=pd.to_datetime(TT[tr],format=\\\'%Y%m%d%H\\\'); age=tensor(np.asarray((stamps.max()-stamps).total_seconds()/86400,dtype=\\\'f4\\\')+1,d)\\n    yy=((Y[tr]-st[\\\'clim\\\'][H[tr]])/st[\\\'scale\\\']).reshape(len(tr),-1,4); pred=np.zeros((len(we),2450,4),np.float32); total=np.zeros(2450,np.float32)\\n    gy,gx=np.meshgrid(glat,glon,indexing=\\\'ij\\\'); fy,fx=lat.ravel(),lon.ravel()\\n    if save_model: np.savez(OUT/f"{model_prefix}_preprocessing.npz",**st,features=(a-mu)/sd,feature_mu=mu,feature_sd=sd,train_hours=H[tr],train_times=TT[tr],alpha=alpha,sigma=sigma,local=local,days=days)\\n    anchor=0\\n    for cy in np.linspace(wlat[0],wlat[-1],3):\\n        for cx in np.linspace(wlon[0],wlon[-1],2):\\n            ww=np.exp(-.5*((gy-cy)**2+(gx-cx)**2)/sigma**2); ww=(ww/max(ww.mean(),1e-6))**.5; w=tensor(np.tile(np.repeat(ww.ravel(),4),3),d)\\n            k=((1-local)*G+local*kernel(ag*w,ag*w))*C; q=((1-local)*Q+local*kernel(bg*w,ag*w))*D\\n            k=(k+k.T)*.5; k.diagonal().add_(alpha*torch.exp(age/days)); L=torch.linalg.cholesky(k)\\n            blend=np.maximum(1-np.abs(fy-cy)/((wlat[-1]-wlat[0])/2),0)*np.maximum(1-np.abs(fx-cx)/(wlon[-1]-wlon[0]),0)\\n            ids=np.flatnonzero(blend>1e-6); target=tensor(yy[:,ids].reshape(len(tr),-1),d); dual=torch.cholesky_solve(target,L)\\n            if save_model: np.savez(OUT/f"{model_prefix}_sector_{anchor}.npz",indices=ids,blend=blend[ids],feature_weights=w.cpu().numpy(),dual=dual.cpu().numpy(),center=[cy,cx])\\n            anchor+=1\\n            pred[:,ids]+= (q@dual).cpu().numpy().reshape(len(we),len(ids),4)*blend[ids,None]; total[ids]+=blend[ids]\\n    assert total.min()>.99\\n    p=pred.reshape(len(we),70,35,4)/total.reshape(1,70,35,1)\\n    return clip(p*st[\\\'scale\\\']+st[\\\'clim\\\'][he])\\n\\n\\n\\nfrom scipy.ndimage import gaussian_filter\\nANCHOR="Stack"\\nKEYS27=KEYS+["Regional","Narrow"]\\ndef predict27(p,w):\\n    assert w.shape==(len(KEYS27),4) and np.isfinite(w).all() and (w>=-1e-7).all() and np.allclose(w.sum(0),1,atol=1e-5)\\n    return clip(sum(p[k]*w[i].astype(\\\'f4\\\') for i,k in enumerate(KEYS27)))\\n\\ndef padded(w): return np.vstack([w,np.zeros((2,4),dtype=\\\'f4\\\')])\\ndef add_component(w,j,a):\\n    w=w*(1-a); w[j]+=a; return w\\n\\ndef calibration(p,y,method,h):\\n    w=padded(fit_method(p,y,\\\'Baseline\\\' if method==\\\'V23\\\' else ANCHOR))\\n    if method in (\\\'V23\\\',\\\'Anchor\\\'): return w,None\\n    j=6 if method.startswith(\\\'Regional\\\') else 7\\n    if method.endswith(\\\'Replace\\\'):\\n        a=.75*(w[4]+w[5]); w[4:6]*=.25; w[j]+=a\\n    elif method.endswith(\\\'Fixed\\\'): w=add_component(w,j,np.full(4,.20,dtype=\\\'f4\\\'))\\n    elif method in (\\\'NarrowAdaptive\\\',\\\'RegionalAdaptive\\\',\\\'NarrowSpatial\\\',\\\'NarrowCorrected\\\',\\\'NarrowStack\\\'):\\n        a=fit_alpha(predict27(p,w),p[KEYS27[j]],y,prior=.20,shrink=.50,cap=.50)\\n        w=add_component(w,j,a)\\n    else: raise ValueError(method)\\n    if method==\\\'NarrowStack\\\':\\n        result=[]\\n        for v in range(4):\\n            prior=w[:,v].astype(\\\'f8\\\'); prior=prior*.95; prior[6]+=.05\\n            # Regularizar frente a la mezcla jerarquica original, en unidades oficiales.\\n            e=np.stack([((p[k][...,v]-y[...,v])/NORM[v]).ravel() for k in KEYS27]).astype(\\\'f8\\\'); gram=e@e.T/e.shape[1]; reg=.012\\n            bounds=[(max(0,float(x)-.18),min(1,float(x)+.18)) for x in prior]; bounds[7]=(0,.55)\\n            opt=minimize(lambda z:float(z@gram@z+reg*np.square(z-prior).sum()),prior,jac=lambda z:2*(gram@z+reg*(z-prior)),bounds=bounds,constraints=[{\\\'type\\\':\\\'eq\\\',\\\'fun\\\':lambda z:z.sum()-1,\\\'jac\\\':lambda z:np.ones(len(KEYS27))}],method=\\\'SLSQP\\\',options={\\\'maxiter\\\':100,\\\'ftol\\\':1e-11})\\n            q=opt.x if opt.success and np.isfinite(opt.x).all() and abs(opt.x.sum()-1)<1e-6 else prior; q=np.maximum(q,0); result.append(q/q.sum())\\n        w=np.array(result,dtype=\\\'f4\\\').T\\n    maps=None\\n    if method in (\\\'NarrowSpatial\\\',\\\'NarrowCorrected\\\'):\\n        # Dos alternativas: corregir el ensemble o solamente el kernel regional.\\n        e=y-(p[\\\'Narrow\\\'] if method==\\\'NarrowCorrected\\\' else predict27(p,w)); glob=gaussian_filter(e.mean(0),sigma=(3,3,0)); maps=[]\\n        for hour in range(24):\\n            hw=np.exp(1.5*(np.cos((h-hour)*np.pi/12)-1)); local=np.average(e,axis=0,weights=hw)\\n            maps.append(.35*(.5*glob+.5*gaussian_filter(local,sigma=(3,3,0))))\\n        maps=np.asarray(maps,dtype=\\\'f4\\\')\\n        if method==\\\'NarrowCorrected\\\': maps*=w[7]\\n    return w,maps\\n\\ndef apply27(p,params,h):\\n    w,b=params; pred=predict27(p,w); return clip(pred if b is None else pred+b[h])\\n\\n\'\n'
    '# Atlas v3.2 | ShadowBytes | paired retrospective audit, not a Kaggle score.\n'
    'import os, argparse, json, hashlib, time, platform, copy, math, threading\n'
    'from pathlib import Path\n'
    'from concurrent.futures import ThreadPoolExecutor\n'
    "for k in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS'): os.environ[k]='4'\n"
    "os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')\n"
    'import numpy as np\n'
    'import pandas as pd\n'
    'import torch\n'
    'from scipy.linalg import cho_factor, cho_solve\n'
    'from scipy.ndimage import gaussian_filter\n'
    'from sklearn.decomposition import PCA\n'
    'from torch import nn\n'
    'import sklearn, scipy\n'
    'NORM=np.array([25.360,2.0652,2.6776,2.0876],dtype=np.float64)\n'
    "VARS=['RH','T','U','V']; CUT='2024052100'; CAL='2024052800'; END='2024070100'\n"
    "SEEDS=[42,123]; LOCK=threading.Lock(); BASE_HASH='9ce6e7594d0edbf884e56902fd67e1fa02de1a94ba2f6815fc5c8b5115a801e1'\n"
    "A={'__name__':'atlas_legacy'}; exec(ATLAS_SOURCE,A)\n"
    '\n'
    'def sha(p):\n'
    '    h=hashlib.sha256()\n'
    "    with open(p,'rb') as f:\n"
    "        for b in iter(lambda:f.read(2**20),b''): h.update(b)\n"
    '    return h.hexdigest()\n'
    'def dump(p,x): Path(p).write_text(json.dumps(x,indent=2,allow_nan=False))\n'
    'def readnp(p):\n'
    '    with np.load(p,allow_pickle=False) as f:return dict(f)\n'
    'def data_root(root):\n'
    "    root=Path(root); req=['train.npz','test.npz','static.npz','sample_submission.csv']\n"
    "    candidates=[root] if all((root/n).is_file() for n in req) else [p.parent for p in root.rglob('train.npz') if all((p.parent/n).is_file() for n in req)]\n"
    "    if len(candidates)!=1: raise ValueError(f'Dataset ambiguous: {candidates}')\n"
    '    return candidates[0]\n'
    'def clip(x):\n'
    "    x=np.asarray(x,dtype='f4').copy(); x[...,0]=np.clip(x[...,0],0,100)\n"
    "    if not np.isfinite(x).all(): raise ValueError('Nonfinite predictions')\n"
    '    return x\n'
    "def blend(a,b,w):return clip(a+(b-a)*np.asarray(w,dtype='f4'))\n"
    'def sums(p,y):\n'
    "    e=p.astype('f8')-y.astype('f8'); return np.square(e).sum(axis=(0,1,2)),int(np.prod(e.shape[:3]))\n"
    'def skill(s,n):return 1-(np.sqrt(np.asarray(s)/np.asarray(n)[...,None])/NORM).mean(-1)\n'
    'def score(p,y):\n'
    '    s,n=sums(p,y);return float(skill(s,n))\n'
    "def daystr(t):return np.asarray(t).astype(str).astype('U8')\n"
    'def history(X,t,h,physics):\n'
    '    a=X[:,:15] if physics else X[:,:4]; prev=X[:,15:30] if physics else X[:,15:19]; old=X[:,30:45] if physics else X[:,30:34]\n'
    '    k=7 if physics else 4\n'
    '    b=(a-prev)*t[:,7,None,None,None]; c=(a-old)*t[:,6,None,None,None]\n'
    '    return np.concatenate([a,b[:,:k],c[:,:k]],1).transpose(0,2,3,1).copy()\n'
    'def design(z,g,t,group,physics):\n'
    '    r,c=divmod(int(group),13); k=7 if physics else 4\n'
    '    cells=sorted(set((int(np.clip(r+dr,0,16)),int(np.clip(c+dc,0,12))) for dr,dc in [(0,0),(-1,0),(1,0),(0,-1),(0,1)]))\n'
    "    return np.concatenate([z[:,r,c],*[z[:,rr,cc,:k] for rr,cc in cells if (rr,cc)!=(r,c)],g,t[:,:4],t[:,6:]],1).astype('f8')\n"
    'def fit_anchor(X,t,h,Y,tr,s,physics=True):\n'
    '    # No held-out X/Y used to fit any statistics, EOF or coefficient.\n'
    '    z=history(X[tr],t[tr],h[tr],physics); hh=h[tr]; tt=t[tr]; n=len(tr)\n'
    "    cl=A['climate'](z,hh,np.arange(n)); z-=cl[hh]; sd=np.maximum(z.std((0,1,2)),1e-4); z=np.clip(z/sd,-12,12)\n"
    "    flat=z.reshape(n,-1); pca=PCA(n_components=min(24,n-1),svd_solver='randomized',random_state=17).fit(flat)\n"
    '    g=pca.transform(flat); gs=np.maximum(g.std(0),1e-4); g/=gs\n'
    "    yc=A['climate'](Y[tr],hh,np.arange(n)); yy=((Y[tr]-yc[hh])/NORM).reshape(n,-1,4)\n"
    "    lat,lon=np.meshgrid(s['wrf_lats'],s['wrf_lons'],indexing='ij'); groups=np.abs(lat.ravel()[:,None]-s['gfs_lats']).argmin(1)*13+np.abs(lon.ravel()[:,None]-s['gfs_lons']).argmin(1)\n"
    '    st=dict(physics=np.array(int(physics)),cl=cl,sd=sd,pca_mean=pca.mean_,pca_components=pca.components_,pca_sd=gs,yc=yc,groups=groups)\n'
    '    for gr in np.unique(groups):\n'
    '        d=design(z,g,tt,gr,physics); mu=d.mean(0); ds=np.maximum(d.std(0),.05); d=np.column_stack([np.ones(n),(d-mu)/ds]); pix=np.flatnonzero(groups==gr)\n'
    '        gram=d.T@d; reg=np.full(len(gram),.08*n); reg[0]=1e-6; gram.flat[::len(gram)+1]+=reg\n'
    '        beta=cho_solve(cho_factor(gram),d.T@yy[:,pix].reshape(n,-1))\n'
    "        for key,v in [('mu',mu),('sd',ds),('beta',beta)]:st[f'{key}_{gr}']=v\n"
    '    return st\n'
    'def anchor_predict(st,X,t,h):\n'
    "    physics=bool(st['physics']); z=history(X,t,h,physics); z=np.clip((z-st['cl'][h])/st['sd'],-12,12)\n"
    "    g=((z.reshape(len(z),-1)-st['pca_mean'])@st['pca_components'].T)/st['pca_sd']; groups=st['groups']; pred=np.empty((len(z),2450,4),dtype='f4')\n"
    '    for gr in np.unique(groups):\n'
    "        d=design(z,g,t,gr,physics); d=np.column_stack([np.ones(len(d)),(d-st[f'mu_{gr}'])/st[f'sd_{gr}']]); pix=np.flatnonzero(groups==gr); pred[:,pix]=(d@st[f'beta_{gr}']).reshape(len(d),len(pix),4)\n"
    "    return clip(pred.reshape(-1,70,35,4)*NORM+st['yc'][h])\n"
    'def crossfit(X,t,h,Y,T,tr,s,physics,smoke=False):\n'
    "    dates=pd.to_datetime(T,format='%Y%m%d%H'); start=dates[tr[0]].normalize()+pd.Timedelta(days=28); ids=[]; preds=[]; rows=[]\n"
    "    for b in pd.date_range(start,dates[tr[-1]].normalize(),freq='7D'):\n"
    '        e=b+pd.Timedelta(days=7); fit=tr[dates[tr]<b-pd.Timedelta(hours=24)]; val=tr[(dates[tr]>=b)&(dates[tr]<e)]\n'
    '        if len(fit)<30 or not len(val):continue\n'
    '        assert dates[fit[-1]]<dates[val[0]]-pd.Timedelta(hours=24)\n'
    '        st=fit_anchor(X,t,h,Y,fit,s,physics); q=anchor_predict(st,X[val],t[val],h[val]); ids.append(val);preds.append(q)\n'
    '        rows.append(dict(physics=physics,fit_end=str(dates[fit[-1]]),holdout_start=str(b),holdout_end_exclusive=str(e),fit=len(fit),holdout=len(val),gap_hours=24))\n'
    '    ix=np.concatenate(ids);assert len(np.unique(ix))==len(ix);return ix,np.concatenate(preds),rows\n'
    '\n'
    'def model_new(s,mu,sd,width,physics):\n'
    "    geo,grid=A['geography'](s)\n"
    '    if not physics: geo=geo.copy();geo[:,1:8]=0 # DEM and coordinates retained; no engineered terrain\n'
    "    m=A['AtlasResidual'](geo,grid,s['wrf_lats'],s['wrf_lons'],mu,sd,width=width,physics=physics)\n"
    '    m.attn.dropout=0. # fixed across ablations; no shared RNG during parallel training\n'
    '    return m\n'
    '\n'
    'def train_net(X,t,Y,ix,anchor,s,kind,seed,epochs,device,width,checkpoint):\n'
    "    physics=kind!='raw'; wind=kind=='wind'; aux=kind=='aux'\n"
    '    mu=np.tile(X[ix,:15].mean((0,2,3)),3);sd=np.tile(np.maximum(X[ix,:15].std((0,2,3)),1e-4),3)\n'
    '    with LOCK:\n'
    '        torch.manual_seed(seed); model=model_new(s,mu,sd,width,physics)\n'
    '    model=model.to(device); ema=copy.deepcopy(model).eval().requires_grad_(False); rng=np.random.default_rng(seed);batch=32 if width==48 else 16\n'
    "    x=torch.as_tensor((X[ix]-mu[None,:,None,None])/sd[None,:,None,None],device=device); tt=torch.as_tensor(t[ix],device=device); aa=torch.as_tensor(anchor,device=device); yy=torch.as_tensor((Y[ix]-anchor)/NORM.astype('f4'),device=device)\n"
    "    opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.001);sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=math.ceil(len(ix)/batch)*epochs,eta_min=2e-5); amp=device.type=='cuda';scaler=torch.amp.GradScaler('cuda',enabled=amp); updates=0\n"
    '    start=time.perf_counter()\n'
    '    for ep in range(epochs):\n'
    '        model.train(); order=rng.permutation(len(ix))\n'
    '        for off in range(0,len(ix),batch):\n'
    '            k=torch.as_tensor(order[off:off+batch],device=device);opt.zero_grad(set_to_none=True)\n'
    '            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp):p=model(x[k],tt[k])\n'
    '            p=p.float();e=p-yy[k];mse=e.square().mean((0,1,2)); loss_v=.6*(mse+1e-6).sqrt()+.4*mse\n'
    '            if wind: loss=.35*loss_v[2]+.65*loss_v[3]+.01*(e[...,2:].diff(dim=1).square().mean()+e[...,2:].diff(dim=2).square().mean())\n'
    "            else: loss=loss_v.mean()+(A['auxiliary'](p,yy[k],aa[k]) if aux else 0)\n"
    "            if not torch.isfinite(loss):raise FloatingPointError('nonfinite loss')\n"
    '            scaler.scale(loss).backward();scaler.unscale_(opt);nn.utils.clip_grad_norm_(model.parameters(),1.);old=scaler.get_scale();scaler.step(opt);scaler.update()\n'
    '            if scaler.get_scale()>=old:\n'
    '                sched.step();updates+=1\n'
    '                with torch.no_grad():torch._foreach_lerp_(list(ema.parameters()),list(model.parameters()),1-min(.995,(1+updates)/(10+updates)))\n'
    '        print(kind,seed,ep+1,epochs,round(time.perf_counter()-start,1),flush=True)\n'
    '    # Only tensors and primitive metadata; load weights_only=True.\n'
    '    torch.save(dict(state=ema.cpu().state_dict(),mu=torch.from_numpy(mu),sd=torch.from_numpy(sd),width=width,kind=kind,seed=seed,epochs=epochs),checkpoint)\n'
    '\n'
    'def net_predict(path,X,t,s,anchor,device):\n'
    "    ck=torch.load(path,map_location='cpu',weights_only=True);mu=ck['mu'].numpy();sd=ck['sd'].numpy();kind=ck['kind'];model=model_new(s,mu,sd,ck['width'],kind!='raw');model.load_state_dict(ck['state']);model=model.to(device).eval();out=[]\n"
    '    with torch.inference_mode():\n'
    '        for j in range(0,len(X),32):\n'
    '            x=torch.as_tensor((X[j:j+32]-mu[None,:,None,None])/sd[None,:,None,None],device=device);tt=torch.as_tensor(t[j:j+32],device=device)\n'
    '            # FP32 inference fixed for both audit and deployment.\n'
    '            out.append(model(x,tt).float().cpu().numpy())\n'
    '    delta=np.concatenate(out)*NORM\n'
    "    if kind=='wind':delta[...,:2]=0\n"
    '    return clip(anchor+delta)\n'
    '\n'
    'def net_family(out,name,kind,X,t,Y,ix,at,s,devices,epochs,width,seeds):\n'
    "    paths=[out/f'{name}_{seed}.pt' for seed in seeds]\n"
    '    def run(j):train_net(X,t,Y,ix,at,s,kind,seeds[j],epochs,devices[j%len(devices)],width,paths[j])\n'
    '    with ThreadPoolExecutor(max_workers=len(devices)) as pool:list(pool.map(run,range(len(seeds))))\n'
    '    return paths\n'
    '\n'
    "def avg_predict(paths,X,t,s,anchor,device):return clip(np.mean([net_predict(p,X,t,s,anchor,device) for p in paths],axis=0,dtype='f4'))\n"
    '\n'
    'def bias(p,y,h):\n'
    "    e=y-p;g=gaussian_filter(e.mean(0),sigma=(3,3,0));return np.array([.35*(.5*g+.5*gaussian_filter(np.average(e,axis=0,weights=np.exp(1.5*(np.cos((h-i)*np.pi/12)-1))),sigma=(3,3,0))) for i in range(24)],dtype='f4')\n"
    'def optimum(a,b,y,wind=False):\n'
    "    d=b.astype('f8')-a.astype('f8');r=y.astype('f8')-a.astype('f8');w=np.clip((d*r).sum((0,1,2))/np.maximum((d*d).sum((0,1,2)),1e-12),0,.5)\n"
    '    if wind:w[:2]=0\n'
    '    return w\n'
    '\n'
    'def chain(a,others,ws):\n'
    '    for b,w in zip(others,ws):a=blend(a,b,w)\n'
    '    return a\n'
    '\n'
    'def fit_chain(a,others,y,shrink,windflags):\n'
    '    ws=[]\n'
    '    for b,wind in zip(others,windflags):\n'
    '        w=shrink*optimum(a,b,y,wind);ws.append(w);a=blend(a,b,w)\n'
    '    return ws\n'
    '\n'
    'def nested(a,others,y,dates,windflags):\n'
    '    # a is already prequential w.r.t. base calibration (no same-day labels).\n'
    '    days=np.unique(dates); grid=[0.,.5,.85,1.]; losses=[]\n'
    '    for shrink in grid:\n'
    '        preds=[];ys=[]\n'
    '        for day in days[1:]:\n'
    '            tr=dates<day;ev=dates==day;w=fit_chain(a[tr],[b[tr] for b in others],y[tr],shrink,windflags);preds.append(chain(a[ev],[b[ev] for b in others],w));ys.append(y[ev])\n'
    '        losses.append(score(np.concatenate(preds),np.concatenate(ys)))\n'
    '    best=int(np.argmax(losses));return fit_chain(a,others,y,grid[best],windflags),dict(shrink=grid[best],prequential_scores=dict(zip(map(str,grid),losses)),days=days.tolist())\n'
    '\n'
    'def boot(a,b,n,L,B=5000):\n'
    '    rng=np.random.default_rng(2026+L);D=len(n);start=rng.integers(0,D-L+1,size=(B,math.ceil(D/L)));idx=(start[...,None]+np.arange(L)).reshape(B,-1)[:,:D];cnt=n[idx].sum(1);good=cnt>0;idx=idx[good];cnt=cnt[good];d=skill(b[idx].sum(1),cnt)-skill(a[idx].sum(1),cnt)\n'
    '    return dict(block_calendar_days=L,replicates=len(d),ci95=np.quantile(d,[.025,.975]).tolist(),se=float(d.std(ddof=1)))\n'
    '\n'
    'def evaluate(pred,y,T,h,mask,Ytrain,out,base,primary,mode):\n'
    "    days=pd.date_range('2024-05-28','2024-06-30').strftime('%Y%m%d').to_numpy();dt=daystr(T); ns=np.array([np.sum(dt==d)*2450 for d in days]); allstats={};rows=[];daily=[];weekly=[];detail=[]\n"
    '    def row(name,p,yy,**kw):\n'
    "        ss,n=sums(p,yy);rm=np.sqrt(ss/n);return dict(model=name,skill=float(skill(ss,n)),pixels_hours=n,**dict(zip(['RMSE_'+v for v in VARS],rm)),**dict(zip(['normalized_'+v for v in VARS],rm/NORM)),**kw)\n"
    '    quantiles=[np.quantile(Ytrain[...,j],[.05,.95]) for j in range(4)]\n'
    '    for name,p in pred.items():\n'
    '        rows.append(row(name,p,y));ss=np.array([sums(p[dt==d],y[dt==d])[0] if n else np.zeros(4) for d,n in zip(days,ns)]);allstats[name]=ss\n'
    '        for i,(d,n) in enumerate(zip(days,ns)):\n'
    '            if n:daily.append(dict(date=d,day_since_cal=i+1,**row(name,p[dt==d],y[dt==d])))\n'
    '        for wk in range(5):\n'
    '            q=np.isin(dt,days[7*wk:7*(wk+1)])\n'
    '            if q.any():weekly.append(row(name,p[q],y[q],week=wk+1))\n'
    "        for hr in np.unique(h):detail.append(row(name,p[h==hr],y[h==hr],group=f'UTC{hr:02}'))\n"
    '        for status in [True,False]:\n'
    '            q=(mask[:,:2].sum(1)==2)==status\n'
    "            if q.any():detail.append(row(name,p[q],y[q],group='history_complete' if status else 'history_missing'))\n"
    '        # Thresholds fixed by training only. Per-variable tail errors, not a combined score.\n'
    '        for j,v in enumerate(VARS):\n'
    '            lo,hi=quantiles[j];\n'
    "            for tag,q in [('low',y[...,j]<=lo),('high',y[...,j]>=hi)]:\n"
    "                if q.any():detail.append(dict(model=name,group=f'{v}_{tag}_train_quantile',variable=v,threshold=float(lo if tag=='low' else hi),tail_rmse=float(np.sqrt(np.mean((p[...,j][q].astype('f8')-y[...,j][q])**2))),pixels_hours=int(q.sum())))\n"
    "    pd.DataFrame(rows).to_csv(out/'validation_metrics.csv',index=False);pd.DataFrame(daily).to_csv(out/'daily_scores.csv',index=False);pd.DataFrame(weekly).to_csv(out/'weekly_scores.csv',index=False);pd.DataFrame(detail).to_csv(out/'diagnostics.csv',index=False)\n"
    '    a=allstats[base];gates=[]\n'
    '    for name,b in allstats.items():\n'
    '        if name==base:continue\n'
    '        gain=float(skill(b.sum(0),ns.sum())-skill(a.sum(0),ns.sum()));b3=boot(a,b,ns,3);b7=boot(a,b,ns,7);good=ns>0;wins=float(np.mean(skill(b[good],ns[good])>skill(a[good],ns[good])));late=float(skill(b[-14:].sum(0),ns[-14:].sum())-skill(a[-14:].sum(0),ns[-14:].sum())) if ns[-14:].sum() else None\n'
    "        threshold=max(.01311,1.96*max(b3['se'],b7['se']));passed=mode=='full' and name==primary and gain>=threshold and min(b3['ci95'][0],b7['ci95'][0])>=.003 and wins>=.65 and late is not None and late>=0\n"
    '        gates.append(dict(model=name,gain=gain,threshold=threshold,win_fraction=wins,late_gain=late,bootstrap3=b3,bootstrap7=b7,review_eligible=bool(passed)))\n'
    '    ablations=[]\n'
    "    for first,second in [('AtlasAnchorRaw','AtlasAnchor'),('AtlasPhysicsInSample','AtlasPhysicsOOF'),('AtlasRawOOF','AtlasPhysicsOOF'),('AtlasPhysicsOOF','AtlasAuxOOF'),('MixAnchor','MixWind'),('MixPhysics','MixFull'),('MixRaw','MixPhysics')]:\n"
    '        if first in allstats and second in allstats:\n'
    '            a0,b0=allstats[first],allstats[second]; ablations.append(dict(reference=first,candidate=second,gain=float(skill(b0.sum(0),ns.sum())-skill(a0.sum(0),ns.sum())),bootstrap3=boot(a0,b0,ns,3),bootstrap7=boot(a0,b0,ns,7)))\n'
    "    dump(out/'paired_ablations.json',ablations)\n"
    "    import matplotlib;matplotlib.use('Agg');import matplotlib.pyplot as plt\n"
    '    fig,ax=plt.subplots(figsize=(12,5));df=pd.DataFrame(daily)\n'
    "    for name in dict.fromkeys([base,'AtlasAnchor','AtlasPhysicsOOF',primary]):\n"
    "        z=df[df.model==name];ax.plot(z.day_since_cal,z.skill,marker='.',label=name)\n"
    "    ax.set(xlabel='Días calendario desde la calibración',ylabel='Skill diario',title=f'Atlas v3.2 — {mode}, retrospectivo');ax.grid(alpha=.2);ax.legend();fig.tight_layout();fig.savefig(out/'daily_scores.png',dpi=180);plt.close(fig)\n"
    '    return rows,gates\n'
    '\n'
    "def fingerprint(data):return {n:sha(data/n) for n in ['train.npz','test.npz','static.npz','sample_submission.csv']}\n"
    'def codehash():return sha(Path(__file__))\n'
    'def manifest(out,data_hash,mode,extra=None):\n'
    "    files={str(p.relative_to(out)):sha(p) for p in sorted(out.rglob('*')) if p.is_file() and p.name!='manifest.json'}\n"
    "    dump(out/'manifest.json',dict(model='Atlas v3.2',mode=mode,data=data_hash,code_sha256=codehash(),seeds=SEEDS,cuts=[CUT,CAL,END],versions=dict(python=platform.python_version(),torch=str(torch.__version__),numpy=np.__version__,scipy=scipy.__version__,sklearn=sklearn.__version__,cuda=torch.version.cuda,gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]),tolerance=dict(skill_absolute=.0005,anchor_export_abs_per_value=.0001,checkpoint_roundtrip_abs_per_value=.0001),files=files,extra=extra or {}))\n"
    'def verify_manifest(out,expected_data=None):\n'
    "    m=json.loads((out/'manifest.json').read_text());assert m['code_sha256']==codehash(),'Code mismatch'\n"
    "    if expected_data is not None:assert m['data']==expected_data,'Data mismatch'\n"
    "    for name,h in m['files'].items():\n"
    "        if not (out/name).is_file() or sha(out/name)!=h:raise ValueError(f'Artifact mismatch: {name}')\n"
    '    return m\n'
    '\n'
    'def init_ultra(data,out,mode):\n'
    "    os.environ.update(ULTRALISK_DATA=str(data),ULTRALISK_OUT=str(out/'ultra'),ULTRALISK_SMOKE='1' if mode=='smoke' else '0');u={'__name__':'ultra_reference'};exec(ULTRA_SOURCE,u);return u\n"
    '\n'
    'def baseline(data,out,mode,tr,ca,ev,T,h,Y,dh,cache_arg):\n'
    '    both=np.r_[ca,ev];nc=len(ca);dt=daystr(T[ca]);days=np.unique(dt);base_oof=[];ii=[]\n'
    "    if mode=='proxy':\n"
    "        if not cache_arg:raise ValueError('--cache must point to previous proxy validation_predictions.npz')\n"
    '        with np.load(cache_arg,allow_pickle=False) as f:\n'
    "            assert np.array_equal(f['times'],T[both]) and np.array_equal(f['target'],Y[both]) and int(f['n_cal'])==nc\n"
    "            raw=f['NarrowSinSesgo'];old_anchor=f['AtlasAnchor']\n"
    '        def predict(ids,inds):return clip(raw[inds]+bias(raw[ids],Y[ca[ids]],h[ca[ids]])[h[both[inds]]])\n'
    "        name='NarrowProxy';extra=dict(source_sha256=sha(cache_arg),source_scope='previous proxy; not full Ultralisk',old_anchor=old_anchor)\n"
    '    else:\n'
    "        u=init_ultra(data,out,mode);st=u['stats'](tr);fp=hashlib.sha256((json.dumps(dh,sort_keys=True)+hashlib.sha256(ULTRA_SOURCE.encode()).hexdigest()+mode).encode()+tr.tobytes()+both.tobytes()).hexdigest()\n"
    "        cache=Path(cache_arg) if cache_arg else out/'ultra_family_cache.npz'; p=None\n"
    '        if cache.is_file():\n'
    '            # Exact split + source + dataset. No historical fold cache can masquerade as this split.\n'
    '            with np.load(cache,allow_pickle=False) as f:\n'
    "                assert str(f['fingerprint'])==fp and np.array_equal(f['times'],T[both]) and np.array_equal(f['target'],Y[both]);p={k:f[k] for k in u['KEYS27']}\n"
    '            assert all(v.shape==Y[both].shape and np.isfinite(v).all() for v in p.values())\n'
    '        if p is None:\n'
    "            p=u['run_stage'](tr,u['W'][both],h[both],'long',u['FAMILIES']);d=u['DEVICES'][0]\n"
    "            p['Regional']=u['regional'](d,st,tr,u['W'][both],h[both],sigma=.65,local=.7,days=90)\n"
    "            p['Narrow']=u['regional'](d,st,tr,u['W'][both],h[both],sigma=.4,local=.85,days=90)\n"
    '            np.savez(cache,fingerprint=fp,times=T[both],target=Y[both],**p)\n'
    '        def predict(ids,inds):\n'
    "            w,m=u['calibration']({k:v[ids] for k,v in p.items()},Y[ca[ids]],'NarrowSpatial',h[ca[ids]])\n"
    "            return u['apply27']({k:v[inds] for k,v in p.items()},(w,m),h[both[inds]])\n"
    "        name='Ultralisk27' if mode=='full' else 'Ultralisk27Smoke';extra=dict(cache_sha256=sha(cache),fingerprint=fp)\n"
    '        # Frozen calibration from all seven days; retained for diagnostics.\n'
    "        w,m=u['calibration']({k:v[:nc] for k,v in p.items()},Y[ca],'NarrowSpatial',h[ca]);np.savez(out/'ultra_calibration.npz',weights=w,bias=m)\n"
    '    for day in days[2:]:\n'
    '        fit=np.flatnonzero(dt<day);val=np.flatnonzero(dt==day);base_oof.append(predict(fit,val));ii.append(val)\n'
    '    ids=np.concatenate(ii);return name,predict(np.arange(nc),np.arange(len(both))),ids,np.concatenate(base_oof),extra\n'
    '\n'
    'def run_audit(args):\n'
    '    out=Path(args.out).resolve();out.mkdir(parents=True,exist_ok=True);data=data_root(args.data);dh=fingerprint(data);mode=args.mode;start=time.perf_counter()\n'
    "    train=readnp(data/'train.npz');s=readnp(data/'static.npz');Y=train['Y'];T=train['times'].astype(str);X,t,h=A['prepare'](train['X'],T,s)\n"
    '    assert len(T)==2935 and np.all(T[1:]>T[:-1]) and Y.shape==(2935,70,35,4)\n'
    '    tr=np.flatnonzero(T<CUT);ca=np.flatnonzero((T>=CUT)&(T<CAL)&(h%3==0));ev=np.flatnonzero((T>=CAL)&(T<END)&(h%3==0));assert (len(tr),len(ca),len(ev))==(1999,56,256)\n'
    "    if mode=='smoke':tr=tr[::12];ca=ca[::8];ev=ev[::8]\n"
    "    both=np.r_[ca,ev];nc=len(ca);base,bp,oid,boof,extra=baseline(data,out,mode,tr,ca,ev,T,h,Y,dh,args.cache);pred={base:bp};policies={};np.savez(out/'static.npz',**s)\n"
    "    phys=fit_anchor(X,t,h,Y,tr,s,True);np.savez(out/'anchor_physics.npz',**phys);ae=anchor_predict(phys,X[both],t[both],h[both]);pred['AtlasAnchor']=ae\n"
    '    # Reload comparison verifies prediction path does not depend on training labels.\n'
    "    reloaded=anchor_predict(readnp(out/'anchor_physics.npz'),X[both],t[both],h[both]);assert np.max(np.abs(ae-reloaded))<=1e-4\n"
    "    raw=fit_anchor(X,t,h,Y,tr,s,False);np.savez(out/'anchor_raw.npz',**raw);ar=anchor_predict(raw,X[both],t[both],h[both]);pred['AtlasAnchorRaw']=ar\n"
    "    if mode=='proxy':\n"
    "        prev=extra.pop('old_anchor');extra['anchor_vs_previous_max_abs']=float(np.max(np.abs(prev-ae)));extra['anchor_vs_previous_skill_delta']=score(ae[nc:],Y[ev])-score(prev[nc:],Y[ev])\n"
    '    else:\n'
    "        devices=[torch.device('cpu')] if mode=='smoke' else [torch.device('cuda:0'),torch.device('cuda:1')];width=16 if mode=='smoke' else 48;epochs=1 if mode=='smoke' else args.epochs;seeds=[42] if mode=='smoke' else SEEDS\n"
    "        ix,oo,man=crossfit(X,t,h,Y,T,tr,s,True,mode=='smoke');ir,orr,mr=crossfit(X,t,h,Y,T,tr,s,False,mode=='smoke');assert np.array_equal(ix,ir)\n"
    "        pd.DataFrame(man+mr).to_csv(out/'crossfit_manifest.csv',index=False);infit=anchor_predict(phys,X[ix],t[ix],h[ix]);extra['crossfit']={'rows':len(ix),'in_sample_skill':score(infit,Y[ix]),'oof_skill':score(oo,Y[ix])}\n"
    "        for name,kind,at,ap in [('AtlasPhysicsInSample','physics',infit,ae),('AtlasPhysicsOOF','physics',oo,ae),('AtlasRawOOF','raw',orr,ar),('AtlasAuxOOF','aux',oo,ae),('AtlasWindOOF','wind',oo,ae)]:\n"
    '            paths=net_family(out,name,kind,X,t,Y,ix,at,s,devices,epochs,width,seeds);pred[name]=avg_predict(paths,X[both],t[both],s,ap,devices[0]);\n'
    '            # Replay from serialized EMA weights (same FP32 path).\n'
    '            p2=avg_predict(paths,X[both][:2],t[both][:2],s,ap[:2],devices[0]);assert np.allclose(pred[name][:2],p2,rtol=0,atol=1e-4),name\n'
    '        extra.update(epochs=epochs,width=width,trained_seeds=seeds)\n'
    "    specs={'MixAnchor':(['AtlasAnchor'],[False]),'MixAnchorUV':(['AtlasAnchor'],[True]),'MixAnchorRaw':(['AtlasAnchorRaw'],[False])}\n"
    "    if mode!='proxy':specs.update({'MixInSample':(['AtlasAnchor','AtlasPhysicsInSample'],[False,False]),'MixPhysics':(['AtlasAnchor','AtlasPhysicsOOF'],[False,False]),'MixRaw':(['AtlasAnchorRaw','AtlasRawOOF'],[False,False]),'MixAux':(['AtlasAnchor','AtlasAuxOOF'],[False,False]),'MixWind':(['AtlasAnchor','AtlasWindOOF'],[False,True]),'MixFull':(['AtlasAnchor','AtlasPhysicsOOF','AtlasWindOOF'],[False,False,True])})\n"
    '    for name,(components,flags) in specs.items():\n'
    '        w,report=nested(boof,[pred[k][oid] for k in components],Y[ca[oid]],daystr(T[ca[oid]]),flags);pred[name]=chain(bp,[pred[k] for k in components],w);policies[name]=dict(components=components,weights=[x.tolist() for x in w],**report)\n'
    '    correlations=[]\n'
    "    for period,ref,ids,target in [('prequential_calibration',boof,oid,Y[ca[oid]]),('evaluation',bp[nc:],np.arange(nc,len(both)),Y[ev])]:\n"
    "        for name in ['AtlasAnchor','AtlasAnchorRaw','AtlasPhysicsOOF','AtlasWindOOF']:\n"
    '            if name not in pred:continue\n'
    "            e=ref.astype('f8')-target;f=pred[name][ids].astype('f8')-target\n"
    '            for j,v in enumerate(VARS):correlations.append(dict(period=period,reference=base,model=name,variable=v,pearson=float(np.corrcoef(e[...,j].ravel(),f[...,j].ravel())[0,1])))\n'
    "    pd.DataFrame(correlations).to_csv(out/'error_correlations.csv',index=False)\n"
    "    primary='MixAnchor' if mode=='proxy' else 'MixFull';rows,gates=evaluate({k:v[nc:] for k,v in pred.items()},Y[ev],T[ev],h[ev],t[ev,6:9],Y[tr],out,base,primary,mode)\n"
    "    np.savez(out/'validation_predictions.npz',times=T[both],n_cal=nc,target=Y[both],**pred)\n"
    "    report=dict(model='Atlas v3.2',mode=mode,base=base,primary=primary,scope='retrospective reused historical labels; no blind score',train_count=len(tr),calibration_count=nc,eval_count=len(ev),missing_days=sorted(set(pd.date_range('2024-05-28','2024-06-30').strftime('%Y%m%d'))-set(daystr(T[ev]))),missing_history={k:float(np.mean(t[ix,6:8].sum(1)<2)) for k,ix in [('train',tr),('cal',ca),('eval',ev)]},policy=policies,metrics=rows,gate=gates,review_eligible=any(g['review_eligible'] for g in gates),recommend_submit=False,seconds=time.perf_counter()-start,extra=extra)\n"
    "    families={} if mode=='proxy' else {name:[f'{name}_{seed}.pt' for seed in seeds] for name in ['AtlasPhysicsOOF','AtlasWindOOF']}\n"
    "    dump(out/'bundle.json',dict(model='Atlas v3.2 audit replay',policy=policies[primary],networks=families,static='static.npz',anchor='anchor_physics.npz',base_sha256=BASE_HASH,training='frozen historical audit; NOT a final refit'))\n"
    "    dump(out/'report.json',report);manifest(out,dh,mode,dict(primary=primary));print(pd.DataFrame(rows)[['model','skill']].to_string(index=False));print('REVIEW_ELIGIBLE',report['review_eligible'],'; no automatic submission')\n"
    '\n'
    'def refit(args):\n'
    "    audit=Path(args.audit);data=data_root(args.data);dh=fingerprint(data);m=verify_manifest(audit,dh);r=json.loads((audit/'report.json').read_text())\n"
    "    if m['mode']!='full' or not r['review_eligible'] or not args.approve:raise ValueError('Requires full audit gate AND explicit --approve; no automatic deployment')\n"
    "    out=Path(args.out);out.mkdir(parents=True,exist_ok=True);train=readnp(data/'train.npz');s=readnp(data/'static.npz');T=train['times'].astype(str);Y=train['Y'];X,t,h=A['prepare'](train['X'],T,s);tr=np.arange(len(T));devices=[torch.device('cuda:0'),torch.device('cuda:1')]\n"
    "    if torch.cuda.device_count()!=2 or any('T4' not in torch.cuda.get_device_name(i) for i in range(2)):raise RuntimeError('T4 x2 required')\n"
    "    st=fit_anchor(X,t,h,Y,tr,s,True);np.savez(out/'anchor_physics.npz',**st);np.savez(out/'static.npz',**s);ix,oo,man=crossfit(X,t,h,Y,T,tr,s,True);pd.DataFrame(man).to_csv(out/'crossfit_manifest.csv',index=False)\n"
    "    policy=r['policy']['MixFull'];epochs=r['extra']['epochs'];families={}\n"
    "    for name,kind in [('AtlasPhysicsOOF','physics'),('AtlasWindOOF','wind')]:\n"
    '        paths=net_family(out,name,kind,X,t,Y,ix,oo,s,devices,epochs,48,SEEDS);families[name]=[p.name for p in paths]\n'
    "    dump(out/'bundle.json',dict(model='Atlas v3.2',policy=policy,networks=families,static='static.npz',anchor='anchor_physics.npz',base_sha256=BASE_HASH,training='all official train rows; fixed audit blend weights; no test labels',audit_manifest_sha256=sha(audit/'manifest.json')));manifest(out,dh,'refit',dict(audit_sha256=sha(audit/'manifest.json')));print('Bundle exported. Inference requires upstream Ultralisk forecast.')\n"
    '\n'
    'def infer(args):\n'
    '    bundle=Path(args.bundle);m=verify_manifest(bundle);\n'
    "    if args.base_csv and m['mode']!='refit':raise ValueError('No Kaggle CSV from audit or smoke bundle')\n"
    "    b=json.loads((bundle/'bundle.json').read_text());s=readnp(bundle/b['static']);\n"
    '    # Read only X and times; Y is neither accessed nor required.\n'
    "    with np.load(args.input,allow_pickle=False) as f:x=f['X'];T=f['times'].astype(str)\n"
    '    assert len(T)==len(x) and np.all(T[1:]>T[:-1]) and x.shape[1:]==(17,13,4)\n'
    "    X,t,h=A['prepare'](x,T,s);keep=np.flatnonzero(h%3==0);X,t,h,T=X[keep],t[keep],h[keep],T[keep];ae=anchor_predict(readnp(bundle/b['anchor']),X,t,h);device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu');pred={'AtlasAnchor':ae}\n"
    "    for k,paths in b['networks'].items():pred[k]=avg_predict([bundle/p for p in paths],X,t,s,ae,device)\n"
    '    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)\n'
    "    np.savez(out/'atlas_predictions.npz',times=T,**pred)\n"
    '    if args.base_csv:\n'
    "        if sha(args.base_csv)!=b['base_sha256']:raise ValueError('Historical baseline CSV SHA-256 differs')\n"
    "        if not args.template:raise ValueError('--template required for Kaggle CSV')\n"
    "        template=pd.read_csv(args.template,dtype={'id':str});base=pd.read_csv(args.base_csv,dtype={'id':str});assert template.columns.tolist()==base.columns.tolist()==['id',*VARS] and template.id.is_unique and base.id.is_unique and np.array_equal(template.id,base.id)\n"
    "        ids=[f'{time}_{i:02d}_{j:02d}' for time in T for i in range(70) for j in range(35)];assert len(ids)==646800 and set(ids)==set(base.id);p=base.set_index('id').loc[ids,VARS].to_numpy(dtype='f4').reshape(len(T),70,35,4)\n"
    '    elif args.base_npz:\n'
    "        z=readnp(args.base_npz);assert np.array_equal(z['times'].astype(str),T);p=z['prediction'];assert p.shape==ae.shape and np.isfinite(p).all()\n"
    "    else:print('Atlas-only predictions exported. Upstream Ultralisk required for ensemble.');return\n"
    "    policy=b['policy'];q=chain(p,[pred[k] for k in policy['components']],policy['weights']);np.savez(out/'ensemble_predictions.npz',times=T,prediction=q)\n"
    "    if args.base_csv:A['save_csv'](template,T,q,out/'submission_atlas_v3_2_REVIEW.csv')\n"
    "    dump(out/'inference_manifest.json',dict(bundle_manifest_sha256=sha(bundle/'manifest.json'),input_sha256=sha(args.input),baseline_sha256=sha(args.base_csv or args.base_npz),outputs={p.name:sha(p) for p in out.iterdir() if p.is_file() and p.name!='inference_manifest.json'},recommend_submit=False))\n"
    '\n'
    "if __name__=='__main__':\n"
    "    p=argparse.ArgumentParser();p.add_argument('--mode',choices=['proxy','smoke','full','refit','infer'],default='full');p.add_argument('--data',default='/kaggle/input');p.add_argument('--out',default='/kaggle/working/Atlas-v3.2');p.add_argument('--cache');p.add_argument('--epochs',type=int,default=48);p.add_argument('--audit');p.add_argument('--approve',action='store_true');p.add_argument('--bundle');p.add_argument('--input');p.add_argument('--base-csv');p.add_argument('--base-npz');p.add_argument('--template');args=p.parse_args();torch.set_num_threads(2);torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True;torch.use_deterministic_algorithms(True,warn_only=True)\n"
    "    if args.mode=='refit':refit(args)\n"
    "    elif args.mode=='infer':infer(args)\n"
    '    else:run_audit(args)\n'
)

# ----- Atlas_v3_2_5.py -----
SOURCES['Atlas_v3_2_5.py']=(
    '"""Atlas v3.2.5 patches. Requires unmodified Atlas_v3_2.py beside this file."""\n'
    'import os\n'
    "for k in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS'): os.environ[k]='4'\n"
    'import argparse, json, math, time, hashlib, copy, platform\n'
    'from pathlib import Path\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    'import torch\n'
    'from torch import nn\n'
    'from scipy.linalg import cho_factor, cho_solve\n'
    'from sklearn.utils.extmath import randomized_svd\n'
    'import Atlas_v3_2 as v\n'
    'NORM=v.NORM; VARS=v.VARS\n'
    "ORIGINS={'O1':('2024031700','2024032400','2024042100'),'O2':('2024041400','2024042100','2024052100'),'O3':('2024052100','2024052800','2024070100')}\n"
    "PREREG='bcb9d59dc0e13859bd7c6a9d7f6ab63fc4eea2f5175f83892b8520926aff5f37'\n"
    "CONFIG={'Raw':('raw',False,False),'Placebo':('placebo',False,False),'Physics':('physics',False,False),'Recent':('physics',True,False),'Drift':('physics',False,True)}\n"
    '\n'
    'def check_protocol():\n'
    "    p=Path(__file__).with_name('Atlas_v3_2_5_preregistro.md')\n"
    "    if v.sha(p)!=PREREG: raise ValueError('Preregistration mismatch')\n"
    '\n'
    'def split(T,h,origin):\n'
    '    cut,cal,end=ORIGINS[origin]; tr=np.flatnonzero(T<cut); ca=np.flatnonzero((T>=cut)&(T<cal)&(h%3==0)); ev=np.flatnonzero((T>=cal)&(T<end)&(h%3==0)); assert len(tr) and len(ca) and len(ev) and max(T[tr])<cut\n'
    '    return tr,ca,ev\n'
    '\n'
    'def zhistory(X,t,h,kind):\n'
    "    if kind!='placebo': return v.history(X,t,h,kind=='physics')\n"
    '    r=v.history(X,t,h,False); # generic fixed nonlinear basis, no additional observations\n'
    '    # Scale in fixed physical units, not fitted to heldout data. 29 channels total.\n'
    "    q=r/np.tile(np.array([25.36,2.0652,2.6776,2.0876],dtype='f4'),3); q[...,1]-=290/2.0652\n"
    '    ext=[np.tanh(q[...,j]) for j in range(12)]+[np.tanh(q[...,j])*np.tanh(q[...,(j+1)%12]) for j in range(5)]\n'
    "    return np.concatenate([r,np.stack(ext,axis=-1)],axis=-1).astype('f4')\n"
    '\n'
    'def wavg(a,w): return np.tensordot(w,a,axes=(0,0))/w.sum()\n'
    'def climate(a,h,w,day,drift):\n'
    '    # day relative to last training timestamp, in units of 30 days.\n'
    '    base=[]; slopes=[]\n'
    '    for hour in range(24):\n'
    '        ix=np.flatnonzero(h==hour); ix=ix if len(ix) else np.arange(len(h)); wi=w[ix]; d=day[ix]; m=wavg(a[ix],wi); dm=np.average(d,weights=wi)\n'
    '        if drift:\n'
    '            b=np.tensordot(wi*(d-dm),a[ix],axes=(0,0))/wi.sum()\n'
    '            b=b/(np.average((d-dm)**2,weights=wi)+.25)\n'
    '        else:b=np.zeros_like(m)\n'
    '        base.append(m-b*dm);slopes.append(b)\n'
    "    return np.asarray(base,dtype='f4'),np.asarray(slopes,dtype='f4')\n"
    '\n'
    "def dated(T,ref): return (pd.to_datetime(T,format='%Y%m%d%H')-pd.Timestamp(ref)).total_seconds().to_numpy()/86400/30\n"
    '\n'
    'def seasonal(cl,sl,h,d): return cl[h]+sl[h]*np.clip(d,-100,34/30).reshape((-1,)+(1,)*(cl.ndim-1))\n'
    '\n'
    "def fit_anchor(X,t,h,Y,T,tr,s,kind='physics',recent=False,drift=False):\n"
    "    if not recent and not drift and kind!='placebo':\n"
    "        st=v.fit_anchor(X,t,h,Y,tr,s,kind=='physics');st['legacy']=np.array(1);return st\n"
    "    z=zhistory(X[tr],t[tr],h[tr],kind); hh=h[tr];tt=t[tr]; n=len(tr);ref=str(pd.to_datetime(T[tr[-1]],format='%Y%m%d%H')); d=dated(T[tr],ref); w=np.exp2(d) if recent else np.ones(n);w=w/w.mean();wf=w.astype('f4')\n"
    "    cl,sl=climate(z,hh,wf,d,drift);z-=seasonal(cl,sl,hh,d).astype('f4'); sd=np.maximum(np.sqrt(np.average(np.mean(z*z,axis=(1,2)),axis=0,weights=w)),1e-4);z=np.clip(z/sd.astype('f4'),-12,12)\n"
    '    flat=z.reshape(n,-1);pm=wavg(flat,wf); centered=flat-pm; _,_,pc=randomized_svd(centered*np.sqrt(wf[:,None]),n_components=min(24,n-1),random_state=17); g=centered@pc.T;gs=np.maximum(np.sqrt(np.average(g*g,axis=0,weights=w)),1e-4);g/=gs\n'
    '    yc,ys=climate(Y[tr],hh,wf,d,drift); yy=((Y[tr]-seasonal(yc,ys,hh,d))/NORM).reshape(n,-1,4)\n'
    "    lat,lon=np.meshgrid(s['wrf_lats'],s['wrf_lons'],indexing='ij');groups=np.abs(lat.ravel()[:,None]-s['gfs_lats']).argmin(1)*13+np.abs(lon.ravel()[:,None]-s['gfs_lons']).argmin(1)\n"
    '    st=dict(legacy=np.array(0),kind=np.array(kind),ref=np.array(ref),cl=cl,sl=sl,sd=sd,pca_mean=pm,pca_components=pc,pca_sd=gs,yc=yc,ys=ys,groups=groups)\n'
    '    for gr in np.unique(groups):\n'
    "        a=v.design(z,g,tt,gr,kind!='raw');mu=np.average(a,axis=0,weights=w);ds=np.maximum(np.sqrt(np.average((a-mu)**2,axis=0,weights=w)),.05);a=np.column_stack([np.ones(n),(a-mu)/ds]);pix=np.flatnonzero(groups==gr); gram=a.T@(w[:,None]*a);reg=np.full(len(gram),.08*w.sum());reg[0]=1e-6;gram.flat[::len(gram)+1]+=reg;beta=cho_solve(cho_factor(gram),a.T@(w[:,None]*yy[:,pix].reshape(n,-1)))\n"
    "        for k,x in [('mu',mu),('sd',ds),('beta',beta)]:st[f'{k}_{gr}']=x\n"
    '    return st\n'
    '\n'
    'def predict_anchor(st,X,t,h,T):\n'
    "    if int(st['legacy']):return v.anchor_predict(st,X,t,h)\n"
    "    kind=str(st['kind']);d=dated(T,str(st['ref']));z=np.clip((zhistory(X,t,h,kind)-seasonal(st['cl'],st['sl'],h,d))/st['sd'],-12,12).astype('f4');g=((z.reshape(len(z),-1)-st['pca_mean'])@st['pca_components'].T)/st['pca_sd'];groups=st['groups'];p=np.empty((len(z),2450,4),dtype='f4')\n"
    '    for gr in np.unique(groups):\n'
    "        a=v.design(z,g,t,gr,kind!='raw');a=np.column_stack([np.ones(len(a)),(a-st[f'mu_{gr}'])/st[f'sd_{gr}']]);pix=np.flatnonzero(groups==gr);p[:,pix]=(a@st[f'beta_{gr}']).reshape(len(a),len(pix),4)\n"
    "    return v.clip(p.reshape(-1,70,35,4)*NORM+seasonal(st['yc'],st['ys'],h,d))\n"
    '\n'
    'def optimal(a,b,y):\n'
    "    d=(b.astype('f8')-a)/NORM;r=(y.astype('f8')-a)/NORM;energy=np.sum(d*d,axis=(0,1,2));return np.clip(np.sum(d*r,axis=(0,1,2))/np.maximum(1.25*energy,1e-12),0,.5)\n"
    '\n'
    'def choose_weights(a,b,y,dates,variable):\n'
    '    grid=np.array([0,.5,.85,1.]);days=np.unique(dates); losses=np.zeros((4,4)); counts=0\n'
    '    for day in days[1:]:\n'
    '        tr=dates<day;ev=dates==day;w=optimal(a[tr],b[tr],y[tr]);\n'
    '        for i,f in enumerate(grid): losses[i]+=v.sums(v.blend(a[ev],b[ev],w*f),y[ev])[0]\n'
    '        counts+=int(ev.sum())*2450\n'
    '    if len(days)<4: factors=np.full(4,.5);common=.5\n'
    '    else:\n'
    '        rm=np.sqrt(losses/counts)/NORM;common=float(grid[np.argmin(rm.mean(1))]);factors=.5*grid[np.argmin(rm,axis=0)]+.5*common if variable else np.full(4,common)\n'
    '    return optimal(a,b,y)*factors,dict(common=common,factors=factors.tolist(),prequential_days=len(days))\n'
    '\n'
    'def cache_meta(data_hash,tr,T,both,origin,mode):\n'
    "    return dict(origin=origin,mode=mode,cut=ORIGINS[origin][0],train_max=str(max(T[tr])),train_times=T[tr].tolist(),query_times=T[both].tolist(),data=data_hash,ultra_sha256=hashlib.sha256(v.ULTRA_SOURCE.encode()).hexdigest(),patch_sha256=v.sha(__file__),epochs=[25,36] if mode=='full' else None)\n"
    '\n'
    'def reference(u,data_hash,tr,ca,ev,T,h,Y,origin,mode,out):\n'
    "    both=np.r_[ca,ev];nc=len(ca);meta=cache_meta(data_hash,tr,T,both,origin,mode);path=out/'reference_cache.npz';start=time.perf_counter();hit=path.exists()\n"
    '    if hit:\n'
    "        z=v.readnp(path);stored=json.loads(str(z.pop('metadata')));assert stored==meta and stored['train_max']<stored['cut'] and np.array_equal(z.pop('times'),T[both]);p=z\n"
    '    else:\n'
    "        st=u['stats'](tr);d=u['DEVICES'][0]\n"
    "        if mode=='proxy':p={'Narrow':u['regional'](d,st,tr,u['W'][both],h[both],sigma=.4,local=.85,days=90)}\n"
    '        else:\n'
    "            p=u['run_stage'](tr,u['W'][both],h[both],origin,u['FAMILIES']);p['Regional']=u['regional'](d,st,tr,u['W'][both],h[both],sigma=.65,local=.7,days=90);p['Narrow']=u['regional'](d,st,tr,u['W'][both],h[both],sigma=.4,local=.85,days=90)\n"
    '        np.savez(path,metadata=json.dumps(meta,sort_keys=True),times=T[both],**p)\n'
    "    assert set(p)==({'Narrow'} if mode=='proxy' else set(u['KEYS27'])) and all(x.shape==(len(both),70,35,4) and np.isfinite(x).all() for x in p.values())\n"
    '    def f(ids,inds):\n'
    "        if mode=='proxy':return v.clip(p['Narrow'][inds]+v.bias(p['Narrow'][ids],Y[ca[ids]],h[ca[ids]])[h[both[inds]]])\n"
    "        w=u['calibration']({k:x[ids] for k,x in p.items()},Y[ca[ids]],'NarrowSpatial',h[ca[ids]]);return u['apply27']({k:x[inds] for k,x in p.items()},w,h[both[inds]])\n"
    '    dt=v.daystr(T[ca]);idx=[];pre=[]\n'
    '    for day in np.unique(dt)[2:]:\n'
    '        fit=np.flatnonzero(dt<day);val=np.flatnonzero(dt==day);idx.append(val);pre.append(f(fit,val))\n'
    "    if mode=='full':\n"
    "        ww,mm=u['calibration']({k:x[:nc] for k,x in p.items()},Y[ca],'NarrowSpatial',h[ca]);np.savez(out/'ultra_calibration.npz',weights=ww,bias=mm)\n"
    "    return f(np.arange(nc),np.arange(len(both))),np.concatenate(idx),np.concatenate(pre),dict(cache_hit=hit,seconds=time.perf_counter()-start,sha256=v.sha(path),train_max=meta['train_max'],cut=meta['cut'])\n"
    '\n'
    'def corrected(a,ca,both,h,Y,T,oid):\n'
    '    nc=len(ca);dates=v.daystr(T[ca]);frozen=v.clip(a+v.bias(a[:nc],Y[ca],h[ca])[h[both]]);pre=[]\n'
    '    for day in np.unique(dates[oid]):\n'
    '        tr=dates<day;val=oid[dates[oid]==day];pre.append(v.clip(a[val]+v.bias(a[:nc][tr],Y[ca][tr],h[ca][tr])[h[ca[val]]]))\n'
    '    return frozen,np.concatenate(pre)\n'
    '\n'
    'def metrics(pred,y,T,origin,out):\n'
    "    start=pd.to_datetime(ORIGINS[origin][1],format='%Y%m%d%H');end=pd.to_datetime(ORIGINS[origin][2],format='%Y%m%d%H');days=pd.date_range(start,end-pd.Timedelta(days=1)).strftime('%Y%m%d').to_numpy(dtype='U8');dt=v.daystr(T);ns=np.array([(dt==d).sum()*2450 for d in days]);rows=[];weekly=[];daily=[];allss={}\n"
    '    for name,p in pred.items():\n'
    '        ss=np.array([v.sums(p[dt==d],y[dt==d])[0] if n else np.zeros(4) for d,n in zip(days,ns)]);allss[name]=ss\n'
    "        def row(s,n):return dict(origin=origin,model=name,skill=float(v.skill(s,n)),pixels_hours=int(n),**dict(zip(['RMSE_'+j for j in VARS],np.sqrt(s/n))))\n"
    '        rows.append(row(ss.sum(0),ns.sum()))\n'
    '        for i,n in enumerate(ns):\n'
    '            if n:daily.append(dict(date=days[i],day=i+1,**row(ss[i],n)))\n'
    '        for i in range(0,len(ns),7):\n'
    '            n=ns[i:i+7].sum()\n'
    '            if n:weekly.append(dict(week=i//7+1,**row(ss[i:i+7].sum(0),n)))\n'
    "    for name,r in [('validation_metrics',rows),('weekly_scores',weekly),('daily_scores',daily)]:pd.DataFrame(r).to_csv(out/f'{name}.csv',index=False)\n"
    "    np.savez(out/'daily_sse.npz',days=days,counts=ns,**allss);return rows\n"
    '\n'
    'def pooled_gate(records,candidate,full):\n'
    '    def gain(items):\n'
    "        a=sum(x['Reference'].sum(0) for x in items);b=sum(x[candidate].sum(0) for x in items);n=sum(x['counts'].sum() for x in items);return float(v.skill(b,n)-v.skill(a,n))\n"
    "    r=records['O3'];n=r['counts'];a=r['Reference'];b=r[candidate];good=n>0\n"
    '    def partial(q):return float(v.skill(b[q].sum(0),n[q].sum())-v.skill(a[q].sum(0),n[q].sum()))\n'
    "    def wins(x):q=x['counts']>0;return np.sum(v.skill(x[candidate][q],x['counts'][q])>v.skill(x['Reference'][q],x['counts'][q])),q.sum()\n"
    "    w,nn=map(sum,zip(*(wins(x) for x in records.values())));out=dict(last_gain=gain([r]),late_weeks_gain=partial(slice(14,None)),last14_gain=partial(slice(-14,None)),last_win_fraction=float(wins(r)[0]/wins(r)[1]),pooled_gain=gain(list(records.values())),pooled_win_fraction=float(w/nn),origins={k:gain([x]) for k,x in records.items()});out['leave_one_out']={k:gain([x for j,x in records.items() if j!=k]) for k in records} if len(records)>1 else {}\n"
    '    for L in (3,7):\n'
    '        rng=np.random.default_rng(2026+L); B=10000;aa=np.zeros((B,4));bb=aa.copy();nn=np.zeros(B)\n'
    '        for x in records.values():\n'
    "            D=len(x['counts']);idx=(rng.integers(0,D-L+1,size=(B,math.ceil(D/L)))[...,None]+np.arange(L)).reshape(B,-1)[:,:D];aa+=x['Reference'][idx].sum(1);bb+=x[candidate][idx].sum(1);nn+=x['counts'][idx].sum(1)\n"
    "        z=v.skill(bb[nn>0],nn[nn>0])-v.skill(aa[nn>0],nn[nn>0]);out[f'bootstrap{L}']=dict(ci95=np.quantile(z,[.025,.975]).tolist(),se=float(z.std(ddof=1)),replicates=len(z),approx_nonoverlap_blocks=sum(len(x['counts'])//L for x in records.values()))\n"
    "    out['cpu_signal']=bool(out['last_gain']>0 and out['late_weeks_gain']>0 and out['last_win_fraction']>=.65)\n"
    "    out['review_eligible']=bool(full and set(records)==set(ORIGINS) and out['cpu_signal'] and out['last14_gain']>=0 and out['pooled_gain']>0 and out['pooled_win_fraction']>=.65 and min(out['leave_one_out'].values())>0 and all(out[f'bootstrap{L}']['ci95'][0]>0 for L in (3,7)))\n"
    '    return out\n'
    '\n'
    'class Conditioned(nn.Module):\n'
    '    def __init__(self,s,mu,sd,width,condition):\n'
    '        super().__init__();self.net=v.model_new(s,mu,sd,width,True);self.condition=condition;old=self.net.fuse[0];new=nn.Conv2d(old.in_channels+4,old.out_channels,1,bias=old.bias is not None)\n'
    '        with torch.no_grad():\n'
    '            new.weight.zero_();new.weight[:,:old.in_channels].copy_(old.weight)\n'
    '            if old.bias is not None:new.bias.copy_(old.bias)\n'
    '        self.net.fuse[0]=new\n'
    '    def forward(self,x,t,a):\n'
    "        q=((a-a.new_tensor([70,290,0,0]))/a.new_tensor(NORM.astype('f4'))).permute(0,3,1,2).to(x.dtype);q=q if self.condition else torch.zeros_like(q)\n"
    '        def inject(module,args):return (torch.cat([args[0],q.to(args[0].dtype)],1),)\n'
    '        handle=self.net.fuse[0].register_forward_pre_hook(inject)\n'
    '        try:return self.net(x,t)\n'
    '        finally:handle.remove()\n'
    '\n'
    'def train_conditioned(X,t,Y,ix,anchor,s,kind,condition,seed,epochs,device,width,path):\n'
    '    start=time.perf_counter();mu=np.tile(X[ix,:15].mean((0,2,3)),3);sd=np.tile(np.maximum(X[ix,:15].std((0,2,3)),1e-4),3)\n'
    '    with v.LOCK:torch.manual_seed(seed);model=Conditioned(s,mu,sd,width,condition)\n'
    '    model=model.to(device);ema=copy.deepcopy(model).eval().requires_grad_(False);rng=np.random.default_rng(seed);batch=32 if width==48 else 16\n'
    "    xa=torch.as_tensor((X[ix]-mu[None,:,None,None])/sd[None,:,None,None],device=device);ta=torch.as_tensor(t[ix],device=device);aa=torch.as_tensor(anchor,device=device);yy=torch.as_tensor((Y[ix]-anchor)/NORM.astype('f4'),device=device)\n"
    "    opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.001);sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=math.ceil(len(ix)/batch)*epochs,eta_min=2e-5);amp=device.type=='cuda';scaler=torch.amp.GradScaler('cuda',enabled=amp);updates=0;times=[]\n"
    '    for ep in range(epochs):\n'
    '        es=time.perf_counter();model.train();order=rng.permutation(len(ix))\n'
    '        for j in range(0,len(ix),batch):\n'
    '            k=torch.as_tensor(order[j:j+batch],device=device);opt.zero_grad(set_to_none=True)\n'
    '            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp):p=model(xa[k],ta[k],aa[k])\n'
    '            e=p.float()-yy[k];m=e.square().mean((0,1,2));lv=.6*(m+1e-6).sqrt()+.4*m;loss=lv.mean()\n'
    "            if kind=='wind':loss=.35*lv[2]+.65*lv[3]+.01*(e[...,2:].diff(dim=1).square().mean()+e[...,2:].diff(dim=2).square().mean())\n"
    "            if not torch.isfinite(loss):raise FloatingPointError('nonfinite loss')\n"
    '            scaler.scale(loss).backward();scaler.unscale_(opt);nn.utils.clip_grad_norm_(model.parameters(),1.);old=scaler.get_scale();scaler.step(opt);scaler.update()\n'
    '            if scaler.get_scale()>=old:\n'
    '                sched.step();updates+=1\n'
    '                with torch.no_grad():torch._foreach_lerp_(list(ema.parameters()),list(model.parameters()),1-min(.995,(1+updates)/(10+updates)))\n'
    '        if amp:torch.cuda.synchronize(device)\n'
    '        times.append(time.perf_counter()-es);print(kind,condition,seed,ep+1,times[-1],flush=True)\n'
    '    torch.save(dict(state=ema.cpu().state_dict(),mu=torch.from_numpy(mu),sd=torch.from_numpy(sd),width=width,kind=kind,condition=condition,seed=seed,epochs=epochs),path)\n'
    '    return dict(seed=seed,kind=kind,condition=condition,seconds=time.perf_counter()-start,epoch_seconds=times,device=str(device),updates=updates)\n'
    '\n'
    'def predict_net(path,X,t,s,anchor,device):\n'
    "    ck=torch.load(path,map_location='cpu',weights_only=True);mu=ck['mu'].numpy();sd=ck['sd'].numpy();model=Conditioned(s,mu,sd,ck['width'],ck['condition']);model.load_state_dict(ck['state']);model=model.to(device).eval();p=[]\n"
    '    with torch.inference_mode():\n'
    '        for j in range(0,len(X),32):\n'
    '            sl=slice(j,j+32);p.append(model(torch.as_tensor((X[sl]-mu[None,:,None,None])/sd[None,:,None,None],device=device),torch.as_tensor(t[sl],device=device),torch.as_tensor(anchor[sl],device=device)).cpu().numpy())\n'
    '    d=np.concatenate(p)*NORM\n'
    "    if ck['kind']=='wind':d[...,:2]=0\n"
    '    return v.clip(anchor+d)\n'
    '\n'
    'def load_or_fit(out,name,X,t,h,Y,T,tr,s,dh,origin):\n'
    "    path=out/f'anchor_{name}.npz'; meta=dict(config=list(CONFIG[name]),data=dh,train_times=T[tr].tolist(),cut=ORIGINS[origin][0],code=v.sha(__file__),legacy_code=v.sha(v.__file__))\n"
    '    if path.exists():\n'
    '        st=v.readnp(path)\n'
    "        if 'fit_metadata' not in st or json.loads(str(st['fit_metadata']))!=meta:raise ValueError(f'Anchor cache provenance mismatch: {path}')\n"
    '        return st\n'
    "    st=fit_anchor(X,t,h,Y,T,tr,s,*CONFIG[name]);st['fit_metadata']=np.array(json.dumps(meta,sort_keys=True));np.savez(path,**st);return st\n"
    '\n'
    'def run_origin(args,data,dh,u,X,t,h,Y,T,s,origin):\n'
    "    start=time.perf_counter();out=Path(args.out)/origin;out.mkdir(parents=True,exist_ok=True);tr,ca,ev=split(T,h,origin);both=np.r_[ca,ev];nc=len(ca);mode='proxy' if args.mode=='proxy' else 'full';bp,oid,boof,meta=reference(u,dh,tr,ca,ev,T,h,Y,origin,mode,out);pred={'Reference':bp};policies={};pre={};stphys=None\n"
    '    for name,(kind,rec,drift) in CONFIG.items():\n'
    '        st=load_or_fit(out,name,X,t,h,Y,T,tr,s,dh,origin);p=predict_anchor(st,X[both],t[both],h[both],T[both]);pred[name]=p;pre[name]=p[oid];print(origin,name,v.score(p[nc:],Y[ev]),flush=True)\n'
    "        if name=='Physics':stphys=st;pred['Bias'],pre['Bias']=corrected(p,ca,both,h,Y,T,oid);np.savez(out/'anchor_bias.npz',bias=v.bias(p[:nc],Y[ca],h[ca]))\n"
    '    timings=[];cross_seconds=None\n'
    "    if args.neural and origin=='O3':\n"
    "        cs=time.perf_counter();ix,oo,man=v.crossfit(X,t,h,Y,T,tr,s,True);cross_seconds=time.perf_counter()-cs;pd.DataFrame(man).to_csv(out/'crossfit_manifest.csv',index=False);inside=predict_anchor(stphys,X[ix],t[ix],h[ix],T[ix]);dev=[torch.device('cuda:0'),torch.device('cuda:1')];epochs=args.epochs\n"
    '        from concurrent.futures import ThreadPoolExecutor\n'
    "        for label,kind,cond,aa in [('InNoInput','physics',False,inside),('InInput','physics',True,inside),('OOFNoInput','physics',False,oo),('OOFInput','physics',True,oo),('WindInput','wind',True,oo)]:\n"
    "            paths=[out/f'{label}_{seed}.pt' for seed in v.SEEDS]\n"
    '            with ThreadPoolExecutor(max_workers=2) as pool:\n'
    '                fs=[pool.submit(train_conditioned,X,t,Y,ix,aa,s,kind,cond,seed,epochs,dev[j],48,paths[j]) for j,seed in enumerate(v.SEEDS)];rr=[f.result() for f in fs]\n'
    "            pp=[predict_net(p,X[both],t[both],s,pred['Physics'],dev[0]) for p in paths];pred[label]=v.clip(np.mean(pp,axis=0));pre[label]=pred[label][oid]\n"
    '            for j,r in enumerate(rr):\n'
    "                replay=predict_net(paths[j],X[both][:32],t[both][:32],s,pred['Physics'][:32],dev[0]);r['replay_max_abs']=float(np.max(np.abs(replay-pp[j][:32])));assert r['replay_max_abs']<=1e-4\n"
    "                r['seed_skill']=v.score(pp[j][nc:],Y[ev]);timings.append(dict(variant=label,**r))\n"
    '    for name in list(pre):\n'
    "        for variable in ([False,True] if name=='Bias' else [True]):\n"
    "            label='Mix'+name+('Var' if variable else 'Common');w,info=choose_weights(boof,pre[name],Y[ca[oid]],v.daystr(T[ca[oid]]),variable)\n"
    "            if name=='WindInput':w[:2]=0\n"
    '            pred[label]=v.blend(bp,pred[name],w);policies[label]=dict(weights=w.tolist(),**info)\n'
    "    if args.neural and origin=='O3':\n"
    '        # Stage 1 bias blend uses only previous prequential days for each day.\n'
    '        dates=v.daystr(T[ca[oid]]);use=[];pstage=[]\n'
    '        for day in np.unique(dates)[1:]:\n'
    "            q=dates<day;r=dates==day;w0,_=choose_weights(boof[q],pre['Bias'][q],Y[ca[oid[q]]],dates[q],True);pstage.append(v.blend(boof[r],pre['Bias'][r],w0));use.extend(np.flatnonzero(r))\n"
    '        use=np.asarray(use);pstage=np.concatenate(pstage)\n'
    "        for label in ('InNoInput','InInput','OOFNoInput','OOFInput','WindInput'):\n"
    '            w,info=choose_weights(pstage,pre[label][use],Y[ca[oid[use]]],dates[use],True)\n'
    "            if label=='WindInput':w[:2]=0\n"
    "            pred['MixBiasPlus'+label]=v.blend(pred['MixBiasVar'],pred[label],w);policies['MixBiasPlus'+label]=dict(weights=w.tolist(),base='MixBiasVar',**info)\n"
    '    rows=metrics({k:p[nc:] for k,p in pred.items()},Y[ev],T[ev],origin,out);corr=[]\n'
    '    for name in pre:\n'
    "        for scope,a,b,y in [('calibration',boof,pre[name],Y[ca[oid]]),('evaluation',bp[nc:],pred[name][nc:],Y[ev])]:\n"
    "            e=a.astype('f8')-y;f=b.astype('f8')-y\n"
    '            for j,var in enumerate(VARS):corr.append(dict(origin=origin,model=name,scope=scope,variable=var,pearson=float(np.corrcoef(e[...,j].ravel(),f[...,j].ravel())[0,1])))\n'
    "    pd.DataFrame(corr).to_csv(out/'correlations.csv',index=False);np.savez(out/'static.npz',**s);np.savez(out/'validation_predictions.npz',times=T[both],n_cal=nc,target=Y[both],**pred);report=dict(origin=origin,mode=args.mode,reference='Narrow proxy' if mode=='proxy' else 'reconstructed v2.7',cache=meta,counts=[len(tr),len(ca),len(ev)],policies=policies,neural_timings=timings,crossfit_seconds=cross_seconds,wall_seconds=time.perf_counter()-start,code_sha256=v.sha(__file__),data=dh,seed_list=v.SEEDS);v.dump(out/'report.json',report);print(origin,pd.DataFrame(rows)[['model','skill']].to_string(index=False),flush=True)\n"
    '    return report\n'
    '\n'
    'def main():\n'
    "    p=argparse.ArgumentParser();p.add_argument('--mode',choices=['proxy','full','benchmark','infer'],default='proxy');p.add_argument('--data',default='/kaggle/input');p.add_argument('--out',required=True);p.add_argument('--origin',choices=['all',*ORIGINS],default='all');p.add_argument('--neural',action='store_true');p.add_argument('--epochs',type=int,default=48);p.add_argument('--cpu-report');p.add_argument('--benchmark-report');p.add_argument('--bundle');p.add_argument('--input');p.add_argument('--fit-only',action='store_true');p.add_argument('--component',default='Physics',choices=list(CONFIG)+['Bias','InNoInput','InInput','OOFNoInput','OOFInput','WindInput']);args=p.parse_args();check_protocol();out=Path(args.out);out.mkdir(parents=True,exist_ok=True)\n"
    "    if args.mode=='infer':\n"
    "        src=Path(args.bundle);is_net=args.component in ['InNoInput','InInput','OOFNoInput','OOFInput','WindInput'];anchor_name='Physics' if args.component=='Bias' or is_net else args.component;manifest=json.loads((src.parent/'manifest.json').read_text());assert manifest['code'][Path(__file__).name]==v.sha(__file__) and manifest['code']['Atlas_v3_2.py']==v.sha(v.__file__)\n"
    "        for name in ['static.npz',f'anchor_{anchor_name}.npz']+(['anchor_bias.npz'] if args.component=='Bias' else [])+([f'{args.component}_{seed}.pt' for seed in v.SEEDS] if is_net else []):assert v.sha(src/name)==manifest['files'][str(Path(src.name)/name)]\n"
    "        s=v.readnp(src/'static.npz')\n"
    "        with np.load(args.input,allow_pickle=False) as f:X0=f['X'];T=f['times'].astype(str)\n"
    "        X,t,h=v.A['prepare'](X0,T,s);st=v.readnp(src/f'anchor_{anchor_name}.npz');q=predict_anchor(st,X,t,h,T)\n"
    "        if args.component=='Bias':q=v.clip(q+v.readnp(src/'anchor_bias.npz')['bias'][h])\n"
    '        if is_net:\n'
    "            device=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu');q=v.clip(np.mean([predict_net(src/f'{args.component}_{seed}.pt',X,t,s,q,device) for seed in v.SEEDS],axis=0))\n"
    "        np.savez(out/'atlas_predictions.npz',times=T,prediction=q);return\n"
    "    if args.mode=='proxy' and args.neural:raise ValueError('Neural training requires GPU phase')\n"
    "    if args.mode=='benchmark':args.neural=True;assert args.epochs==2 and args.origin=='O3','Benchmark requires --epochs 2 --origin O3'\n"
    '    if args.neural:\n'
    "        if not args.cpu_report or not json.loads(Path(args.cpu_report).read_text())['gate']['cpu_signal']:raise ValueError('CPU signal required')\n"
    "        if args.mode=='full':\n"
    "            if not args.benchmark_report:raise ValueError('Benchmark required')\n"
    "            br=json.loads(Path(args.benchmark_report).read_text());assert br['mode']=='benchmark' and br['epochs']==2 and br['reports'][0]['neural_timings'] and br['preregister']==PREREG and br['reports'][0]['code_sha256']==v.sha(__file__)\n"
    "    torch.use_deterministic_algorithms(True,warn_only=True);data=v.data_root(args.data);dh=v.fingerprint(data);train=v.readnp(data/'train.npz');s=v.readnp(data/'static.npz');T=train['times'].astype(str);Y=train['Y'];X,t,h=v.A['prepare'](train['X'],T,s)\n"
    "    origins=list(ORIGINS) if args.origin=='all' else [args.origin]\n"
    '    if args.fit_only:\n'
    '        for origin in origins:\n'
    '            tr,_,_=split(T,h,origin);dest=out/origin;dest.mkdir(parents=True,exist_ok=True);load_or_fit(dest,args.component,X,t,h,Y,T,tr,s,dh,origin)\n'
    '        return\n'
    "    u=v.init_ultra(data,out,'smoke' if args.mode=='proxy' else 'full');reports=[]\n"
    '    for origin in origins:reports.append(run_origin(args,data,dh,u,X,t,h,Y,T,s,origin))\n'
    "    records={k:v.readnp(out/k/'daily_sse.npz') for k in origins};gate=pooled_gate(records,'MixBiasVar',args.mode=='full' and (not args.neural or args.epochs>=48)) if 'O3' in records else None\n"
    "    summary=dict(mode=args.mode,epochs=args.epochs,reports=reports,gate=gate,scope='retrospective; no automatic submission',preregister=PREREG,reference_scope='Narrow only' if args.mode=='proxy' else 'reconstructed Ultralisk 2.7')\n"
    "    if args.neural and args.mode=='benchmark':\n"
    "        rr=reports[0]['neural_timings'];summary['projected_atlas_epoch_seconds_48']=sum(max(x['epoch_seconds'][-1] for x in rr if x['variant']==k) for k in set(x['variant'] for x in rr))*48;summary['benchmark_note']='Training-only estimate for O3; excludes anchor, inference and v2.7 reconstruction.'\n"
    "    v.dump(out/'report.json',summary);v.dump(out/'manifest.json',dict(code={str(Path(__file__).name):v.sha(__file__),'Atlas_v3_2.py':v.sha(v.__file__)},preregister=PREREG,data=dh,origins=ORIGINS,seeds=v.SEEDS,torch=str(torch.__version__),cuda=torch.version.cuda,python=platform.python_version(),replay_atol=1e-4,files={str(p.relative_to(out)):v.sha(p) for p in sorted(out.rglob('*')) if p.is_file() and p.name!='manifest.json'}));print('GATE',gate,flush=True)\n"
    "if __name__=='__main__':main()\n"
)

# ----- Atlas_v3_2_5_preregistro.md -----
SOURCES['Atlas_v3_2_5_preregistro.md']=(
    '# Atlas v3.2.5 — protocolo congelado antes de nuevas métricas\n'
    '\n'
    'Registro local, previo a la ejecución v3.2.5. No es preregistro externo ni evaluación ciega. Código y archivo se incluyen en manifiesto. Hipótesis H1–H9 del revisor.\n'
    '\n'
    '## T1. Orígenes y gate\n'
    'Intervalos [inicio, fin), UTC; entrenamiento estrictamente anterior al inicio de calibración. Orígenes con evaluación no solapada: 24/03 (cal 17–24/03, eval 24/03–21/04), 21/04 (cal 14–21/04, eval 21/04–21/05), 28/05 (cal 21–28/05, eval 28/05–01/07). Ventanas de 28, 30 y 34 días calendario. Los cortes previos son retrospectivos, y los conjuntos train se solapan; no son experimentos independientes. No añadir 04/06 a la inferencia conjunta porque duplicaría días. Evaluar solo UTC múltiplos de 3; días sin Y tienen n=0, no se imputan.\n'
    '\n'
    'Gate de revisión (solo referencia v2.7 reconstruida con 10 redes, no proxy ni benchmark): candidato primario predefinido `MixBiasVar`, y candidato neuronal primario `MixOOFInput` solo si se habilita fase neuronal. Para evitar elegir por evaluación, el neuronal solo se evalúa como contraste secundario en esta versión; no sustituye al primario automáticamente. Exigir ganancia global >0 en el último origen; ganancia agregada semanas 3–5 >0; últimos 14 días >=0; >=65% de días observados ganados en último origen y en conjunto de orígenes. Ganancia pooled >0; límites inferiores de IC percentil 95% >0 para bootstrap pareado estratificado por origen, bloques calendario móviles de 3 y 7 días, 10000 réplicas, semillas 2029/2033. Reportar sensibilidad por origen y leave-one-origin-out; exigir ganancia pooled positiva al excluir cualquiera. El 65% es juicio de ingeniería para evitar ganancias concentradas, no nivel de significancia. Horizonte tardío protege el test largo; IC positivo exige evidencia de mejora; no elimina selección retrospectiva ni incertidumbre estacional. Los bloques de 7 días siguen siendo pocos; no interpretarlos como inferencia confirmatoria. No usar 0.01311 como ruido; 0.00563/0.01874 es una transferencia histórica única, no estimador confiable. No emitir score esperado.\n'
    '\n'
    '## T2–T4. Configuraciones fijas\n'
    'Anchor original, bias espacial/horario 35% (sigma 3 píxeles), recencia con semivida 30 días (PCA ponderado, Ridge ponderado y escalas/climatologías ponderadas), deriva lineal horaria regularizada (ridge 0.25 en días/30, extrapolación máximo 34 días), placebo Raw 29 canales (transformaciones algebraicas genéricas sin meteorología derivada; vecinos 7 canales, igual dimensiones, alpha .08). Cambios aislados. Primario bias sin recencia ni deriva; demás exploratorios. Raw placebo no es prueba causal perfecta.\n'
    '\n'
    'T3: pesos convexos [0,.5] con prior 0 (sin Atlas), ridge equivalente al 25% de la energía de diferencia; contracción única vs por variable, rejilla [0,.5,.85,1], penalización adicional hacia factor común (50% de contracción individual, fijada). Prequential: base y Anchor bias se calibran con días anteriores; pesos se ajustan con predicciones prequential anteriores, selección de contracción solo durante calibración, luego congelar. No pesos por semana del test. Si menos de 4 días prequential, fallback factor 0.5. Selección de factor es retrospectiva dentro de calibración; no se usa evaluación para ajustar.\n'
    '\n'
    'T6: cuatro contrastes factoriales (OOF/in-sample × Anchor input sí/no), rama U/V OOF con input, semillas 42/123, igual muestras y epochs. OOF expanding bloques 7d, warmup 28d y gap 24h. Para primario neuronal, usar Anchor original sin bias como input/residual; añadir bias al componente Anchor por separado en mezcla. T6 solo con señal CPU (proxy último origen global >0, tarde >0, wins >=65%; no autorización de envío), benchmark 2 épocas y decisión explícita. Red benchmark no autoriza gate. T7 registrar tiempos por variante y caché, extrapolación lineal del entrenamiento neuronal, no de overhead. CUDA warn_only, tolerancia de replay FP32 atol 1e-4, repetición de entrenamiento no garantizada; comparar score delta entre seeds/repeticiones y reportar, criterio operativo sugerido |delta| <= 0.001 no probado.\n'
    '\n'
    'T8 reconstruir v2.7 en los tres cortes (25/36 épocas originales, independientemente de --epochs Atlas). Si solo último origen, gate incompleto. Cache con hashes de datos, código, índices, tiempos train exactos y max<CUT. No cargar cachés legacy sin procedencia. No despliegue automático. Refit definitivo queda separado del audit.\n'
)

# ----- Atlas_v3_3.py -----
SOURCES['Atlas_v3_3.py']=(
    '"""Atlas v3.3 AdaptiveFlow: retrospective experiments, no submission."""\n'
    'import os\n'
    "for k in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS'):os.environ[k]='4'\n"
    'import argparse,json,time\n'
    'from pathlib import Path\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    'from scipy.ndimage import gaussian_filter\n'
    'from scipy.fft import dctn,idctn\n'
    'import Atlas_v3_2_5 as a\n'
    'v=a.v\n'
    'm325=a\n'
    "CANDIDATES=['PhysicsNewBias','RecentBias','DriftBias','MultiScale','AdaptiveFlow','WindFlow','Taper']\n"
    "COUNTS={'O1':[666,32,208],'O2':[1215,56,206],'O3':[1999,56,256]}\n"
    "EXPECTED={'Atlas_v3_2.py':'cd60f877d245cd2955187788e562b6fe8fcb4c448829514cbdf376b81455829e','Atlas_v3_2_5.py':'4411658eb868a353f72815ece14101b4c7fc135d8e7d20973e865ec98971f55e'}\n"
    "OFFICIAL_HASHES={'sample_submission.csv': '507d20c815b09f253b43696dc611312355672f1552d2b313e940b32f7f0ca6a7', 'static.npz': '0e181ff372a9a33fdbbcede81da6ddd2da8208f15b94594d78d3f10464852165', 'test.npz': '97894492e5caae72e45b09f0d4d8bd87e03c12e5029f33d276b353870aa5c0bd', 'train.npz': 'f5120b7a7cf5661e8e4143a726f2d22d13905d11d878afb60c8ff5cd02444844'}\n"
    "FULL_HISTORY={'O1': [0.5027231229629487, 0.5034352685502492], 'O2': [0.4739757924262317, 0.476131105418833], 'O3': [0.4309236143627557, 0.4388617738091176]}\n"
    'FULL_AUDIT_SHA="724ca9c3b31bd82852e2019e3db4dc3d2439d0188092c9868f7a1ba8ecfb17c7"\n'
    '# Lock selection on O1/O2: positive in both, highest pooled score. O3 is audit only.\n'
    '# Use identical incumbent weights for all variants; no evaluation labels in fit.\n'
    "def fit_new(X,t,h,Y,T,tr,s,kind='physics',recent=False,drift=False):\n"
    '    """New v3.2.5 anchor route, without legacy dispatch."""\n'
    "    z=m325.zhistory(X[tr],t[tr],h[tr],kind); hh=h[tr];tt=t[tr]; n=len(tr);ref=str(pd.to_datetime(T[tr[-1]],format='%Y%m%d%H')); d=m325.dated(T[tr],ref); w=np.exp2(d) if recent else np.ones(n);w=w/w.mean();wf=w.astype('f4')\n"
    "    cl,sl=m325.climate(z,hh,wf,d,drift);z-=m325.seasonal(cl,sl,hh,d).astype('f4'); sd=np.maximum(np.sqrt(np.average(np.mean(z*z,axis=(1,2)),axis=0,weights=w)),1e-4);z=np.clip(z/sd.astype('f4'),-12,12)\n"
    '    flat=z.reshape(n,-1);pm=m325.wavg(flat,wf); centered=flat-pm; _,_,pc=m325.randomized_svd(centered*np.sqrt(wf[:,None]),n_components=min(24,n-1),random_state=17); g=centered@pc.T;gs=np.maximum(np.sqrt(np.average(g*g,axis=0,weights=w)),1e-4);g/=gs\n'
    '    yc,ys=m325.climate(Y[tr],hh,wf,d,drift); yy=((Y[tr]-m325.seasonal(yc,ys,hh,d))/v.NORM).reshape(n,-1,4)\n'
    "    lat,lon=np.meshgrid(s['wrf_lats'],s['wrf_lons'],indexing='ij');groups=np.abs(lat.ravel()[:,None]-s['gfs_lats']).argmin(1)*13+np.abs(lon.ravel()[:,None]-s['gfs_lons']).argmin(1)\n"
    '    st=dict(legacy=np.array(0),kind=np.array(kind),ref=np.array(ref),cl=cl,sl=sl,sd=sd,pca_mean=pm,pca_components=pc,pca_sd=gs,yc=yc,ys=ys,groups=groups)\n'
    '    for gr in np.unique(groups):\n'
    "        a=v.design(z,g,tt,gr,kind!='raw');mu=np.average(a,axis=0,weights=w);ds=np.maximum(np.sqrt(np.average((a-mu)**2,axis=0,weights=w)),.05);a=np.column_stack([np.ones(n),(a-mu)/ds]);pix=np.flatnonzero(groups==gr); gram=a.T@(w[:,None]*a);reg=np.full(len(gram),.08*w.sum());reg[0]=1e-6;gram.flat[::len(gram)+1]+=reg;beta=m325.cho_solve(m325.cho_factor(gram),a.T@(w[:,None]*yy[:,pix].reshape(n,-1)))\n"
    "        for k,x in [('mu',mu),('sd',ds),('beta',beta)]:st[f'{k}_{gr}']=x\n"
    '    return st\n'
    '\n'
    'def spectral_smooth(x):\n'
    "    z=dctn(x,axes=(0,1),norm='ortho');z[6:]=0;z[:,3:]=0\n"
    "    return idctn(z,axes=(0,1),norm='ortho').astype('f4')\n"
    'def features(p):\n'
    '    u,w=p[...,2],p[...,3];return np.stack([u,w,np.hypot(u,w)],axis=-1)\n'
    'def fit_correction(p,y,h,variant):\n'
    '    assert len(p)==len(y)==len(h)>0\n'
    "    st={'bias':v.bias(p,y,h)}\n"
    "    if variant=='MultiScale':st['bias']=np.stack([.6*b+.4*spectral_smooth(b) for b in st['bias']])\n"
    "    if variant in ('AdaptiveFlow','WindFlow'):\n"
    "        f=features(p);mu=f.mean(0);sd=np.maximum(f.std(0),.5);z=np.clip((f-mu)/sd,-3,3).astype('f8')\n"
    "        e=(y-p-st['bias'][h])/v.NORM;e=e-e.mean(0)\n"
    "        gram=np.einsum('nhwi,nhwj->hwij',z,z)/len(z)+.75*np.eye(3)\n"
    "        rhs=np.einsum('nhwi,nhwv->hwiv',z,e)/len(z)\n"
    "        beta=gaussian_filter(np.linalg.solve(gram,rhs).astype('f4'),sigma=(3,3,0,0))\n"
    "        if variant=='WindFlow':beta[...,:2]=0\n"
    '        st.update(mu=mu,sd=sd,beta=beta)\n'
    '    return st\n'
    'def apply_correction(st,p,h,T,last,variant):\n'
    "    b=st['bias'][h]\n"
    "    if variant=='Taper':b=b*np.exp2(-np.maximum(a.dated(T,last)*30,0)/21.)[:,None,None,None]\n"
    "    if 'beta' in st:\n"
    "        z=np.clip((features(p)-st['mu'])/st['sd'],-3,3)\n"
    "        extra=.35*np.einsum('nhwi,hwiv->nhwv',z,st['beta'])*v.NORM\n"
    '        b=b+np.clip(extra,-.3*v.NORM,.3*v.NORM)\n'
    '    return v.clip(p+b)\n'
    'def fit_apply(p,y,h,T,nc,variant):\n'
    '    assert len(p)==len(h)==len(T) and len(y)==nc and T[nc-1]<T[nc]\n'
    "    st=fit_correction(p[:nc],y,h[:nc],variant);last=pd.to_datetime(T[nc-1],format='%Y%m%d%H')\n"
    '    q=apply_correction(st,p,h,T,last,variant)\n'
    '    assert np.max(np.abs(apply_correction(st,p[nc:nc+1],h[nc:nc+1],T[nc:nc+1],last,variant)-q[nc:nc+1]))<=1e-5\n'
    '    return q\n'
    'def run_origin(args,u,dh,X,t,h,Y,T,s,o):\n'
    '    dest=Path(args.out)/o;dest.mkdir(parents=True,exist_ok=True)\n'
    '    tr,ca,ev=a.split(T,h,o);both=np.r_[ca,ev];nc=len(ca)\n'
    '    assert [len(tr),nc,len(ev)]==COUNTS[o] and max(T[tr])<min(T[ca])<=max(T[ca])<min(T[ev])\n'
    '    refdir=Path(args.cache)/o if args.cache else dest;refdir.mkdir(parents=True,exist_ok=True)\n'
    '    bp,oid,boof,cache=a.reference(u,dh,tr,ca,ev,T,h,Y,o,args.mode,refdir)\n'
    '    def anchor(name,kind,rec=False,drift=False):\n'
    "        st=fit_new(X,t,h,Y,T,tr,s,kind,rec,drift) if name=='PhysicsNew' else a.load_or_fit(refdir,name,X,t,h,Y,T,tr,s,dh,o)\n"
    '        return a.predict_anchor(st,X[both],t[both],h[both],T[both])\n'
    "    phys=anchor('Physics','physics');old,pre=a.corrected(phys,ca,both,h,Y,T,oid)\n"
    '    w,info=a.choose_weights(boof,pre,Y[ca[oid]],v.daystr(T[ca[oid]]),True)\n'
    "    pred={'Ultralisk':bp,'PhysicsLegacy':phys,'Bias325':old,'Reference':v.blend(bp,old,w)}\n"
    "    if (refdir/'validation_metrics.csv').exists() and refdir!=dest:\n"
    "        m=pd.read_csv(refdir/'validation_metrics.csv').set_index('model');assert abs(v.score(pred['Reference'][nc:],Y[ev])-m.loc['MixBiasVar','skill'])<1e-6\n"
    "    for name,kind,rec,drift in [('PhysicsNew','physics',False,False),('Placebo','placebo',False,False),('Recent','physics',True,False),('Drift','physics',False,True)]:\n"
    '        p=anchor(name,kind,rec,drift);pred[name]=p\n'
    "        if name!='Placebo':pred[name+'Bias']=fit_apply(p,Y[ca],h[both],T[both],nc,'Bias')\n"
    "    for name in ['MultiScale','AdaptiveFlow','WindFlow','Taper']:pred[name]=fit_apply(phys,Y[ca],h[both],T[both],nc,name)\n"
    "    for name in CANDIDATES:pred['Mix'+name]=v.blend(bp,pred[name],w)\n"
    '    for p in pred.values():assert p.shape==Y[both].shape and np.isfinite(p).all() and np.all((p[...,0]>=0)&(p[...,0]<=100))\n'
    "    rows=a.metrics({k:p[nc:] for k,p in pred.items()},Y[ev],T[ev],o,dest);base=next(r['skill'] for r in rows if r['model']=='Reference')\n"
    "    if args.mode=='full':\n"
    '        current=[v.score(bp[nc:],Y[ev]),base]\n'
    "        if not np.allclose(current,FULL_HISTORY[o],rtol=0,atol=.001):raise ValueError(f'Historical baseline drift {o}: {current} vs {FULL_HISTORY[o]}')\n"
    "    print(o,'baseline',base,flush=True)\n"
    '    for r in rows:\n'
    "        if r['model'].startswith('Mix'):print(r['model'],r['skill']-base,flush=True)\n"
    "    report=dict(origin=o,mode=args.mode,reference='MixBiasVar / '+('Narrow proxy' if args.mode=='proxy' else 'reconstructed Ultralisk v2.7'),counts=COUNTS[o],cache=cache,weights=w.tolist(),policy=info,data=dh,code_sha256=v.sha(__file__))\n"
    "    v.dump(dest/'report.json',report);return report\n"
    'def gain(rs,name):\n'
    "    n=sum(x['counts'].sum() for x in rs);b=sum(x['Mix'+name].sum(0) for x in rs);r=sum(x['Reference'].sum(0) for x in rs);return float(v.skill(b,n)-v.skill(r,n))\n"
    'def select_dev(records):\n'
    "    rs=[records[o] for o in ('O1','O2')];ok=[n for n in CANDIDATES if min(gain([r],n) for r in rs)>0];return max(ok,key=lambda n:gain(rs,n)) if ok else None\n"
    'def tests():\n'
    "    rng=np.random.default_rng(33);p=rng.normal(size=(8,70,35,4)).astype('f4');p[...,0]+=50;p[...,1]+=280;y=p.copy();h=np.arange(8)*3;T=np.array([f'20240501{x:02}' for x in h]);last=pd.Timestamp('2024-05-01')\n"
    "    for n in ['Bias','MultiScale','AdaptiveFlow','WindFlow','Taper']:assert np.array_equal(p,apply_correction(fit_correction(p,y,h,n),p,h,T,last,n)),n\n"
    "    assert np.max(np.abs(spectral_smooth(np.ones((70,35,4),'f4'))-1))<1e-5\n"
    "    assert np.all(fit_correction(p,y+1,h,'WindFlow')['beta'][...,:2]==0)\n"
    "    fit_apply(np.concatenate([p,p]),y,np.tile(h,2),np.r_[T,np.char.replace(T,'20240501','20240502')],8,'AdaptiveFlow')\n"
    "    print('TESTS OK: zero residual, constant DCT, wind isolation, replay',flush=True)\n"
    'def main():\n'
    "    p=argparse.ArgumentParser();p.add_argument('--mode',choices=['proxy','full'],default='proxy');p.add_argument('--data',required=True);p.add_argument('--out',required=True);p.add_argument('--cache');p.add_argument('--locked');args=p.parse_args()\n"
    '    start=time.perf_counter();a.check_protocol();tests();out=Path(args.out);out.mkdir(parents=True,exist_ok=True)\n'
    '    for name,digest in EXPECTED.items():assert v.sha(Path(__file__).with_name(name))==digest\n'
    "    if args.mode=='full':\n"
    '        import torch\n'
    "        assert args.locked=='AdaptiveFlow', 'Full follow-up is locked to exploratory AdaptiveFlow'\n"
    "        assert torch.cuda.device_count()==2 and all('T4' in torch.cuda.get_device_name(i) for i in range(2));torch.use_deterministic_algorithms(True,warn_only=True)\n"
    "    data=v.data_root(args.data);dh=v.fingerprint(data);train=v.readnp(data/'train.npz');s=v.readnp(data/'static.npz');T=train['times'].astype(str);Y=train['Y'];X,t,h=v.A['prepare'](train['X'],T,s)\n"
    "    assert dh==OFFICIAL_HASHES,'Official data hashes differ'\n"
    "    u=v.init_ultra(data,out,'smoke' if args.mode=='proxy' else 'full');records={};reports=[]\n"
    "    for o in ('O1','O2','O3'):\n"
    "        if o=='O3':\n"
    "            selected=select_dev(records) if args.mode=='proxy' else args.locked\n"
    "            v.dump(out/'selection_before_O3.json',dict(selected=selected,mode=args.mode,dev_gains={n:{k:gain([r],n) for k,r in records.items()} for n in CANDIDATES}));print('LOCKED BEFORE O3:',selected,flush=True)\n"
    "        reports.append(run_origin(args,u,dh,X,t,h,Y,T,s,o));records[o]=v.readnp(out/o/'daily_sse.npz')\n"
    "    gate=a.pooled_gate(records,'Mix'+selected,args.mode=='full') if selected else None\n"
    '    if gate:\n'
    "        gate['statistical_gate_passed']=bool(gate['review_eligible'] and min(gate['origins'].values())>0)\n"
    '        # Full follow-up chosen after proxy O3, hence not independent confirmation.\n'
    "        gate['review_eligible']=False\n"
    "    exploratory={n:a.pooled_gate(records,'Mix'+n,False) for n in CANDIDATES}\n"
    "    summary=dict(version='3.3',mode=args.mode,selected=selected,gate=gate,exploratory_gates=exploratory,selection_provenance=('CPU O1/O2 selection before current O3' if args.mode=='proxy' else 'AdaptiveFlow locked before full run, selected after retrospective proxy O3'),historical_audit_sha256=FULL_AUDIT_SHA,source_sha256=v.sha(__file__),scope='retrospective; no submission; Reference=MixBiasVar',data=dh,reports=reports,seconds=time.perf_counter()-start)\n"
    "    v.dump(out/'report.json',summary);frames=[]\n"
    "    def add(section,df):df=df.copy();df.insert(0,'section',section);frames.append(df)\n"
    "    add('metadata',pd.DataFrame([{'field':k,'value':json.dumps(x,sort_keys=True)} for k,x in summary.items()]))\n"
    '    for o,r in records.items():\n'
    "        for section,file in [('global','validation_metrics.csv'),('weekly','weekly_scores.csv'),('daily','daily_scores.csv')]:add(section,pd.read_csv(out/o/file))\n"
    "        for name in sorted(set(r)-{'days','counts'}):\n"
    "            d=pd.DataFrame(r[name],columns=['SSE_'+x for x in v.VARS]);d['origin']=o;d['model']=name;d['date']=r['days'];d['pixels_hours']=r['counts'];add('daily_sse',d)\n"
    "    dest=out/'Atlas33_AUDIT.csv';tmp=dest.with_suffix('.tmp');pd.concat(frames,ignore_index=True).to_csv(tmp,index=False,float_format='%.17g');assert len(pd.read_csv(tmp,low_memory=False))==sum(map(len,frames));tmp.replace(dest)\n"
    "    print('GATE',gate,flush=True);print('CSV',dest,v.sha(dest),flush=True)\n"
    "if __name__=='__main__':main()\n"
)

# ----- Atlas_v3_4.py -----
SOURCES['Atlas_v3_4.py']=(
    '"""Atlas v3.4 RegimeFlow: retrospective variable routing; no automatic submission."""\n'
    'import os\n'
    "for k in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS'): os.environ[k]='4'\n"
    'import argparse,json\n'
    'from pathlib import Path\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    'import Atlas_v3_3 as m\n'
    'import Atlas_v3_2_5 as a\n'
    "v=a.v; VARS=v.VARS; SSE=['SSE_'+x for x in VARS]\n"
    "AUDIT_SHA='78568dd2c40e4e5d4317fc113fe992eff4e8f4143b9d8c0bbec6d6cbcd9d83a7'\n"
    "BASE_SHA='9ce6e7594d0edbf884e56902fd67e1fa02de1a94ba2f6815fc5c8b5115a801e1'\n"
    "CODE_SHA='58015cb53e74b0f55c4fc1136a98851e7e91f184ea78634359ddca420df09377'\n"
    "ROUTE=('MixDriftBias','MixDriftBias','MixAdaptiveFlow','MixDriftBias') # post-hoc, NOT confirmatory\n"
    "NAMES=['Ultralisk','Reference','MixAdaptiveFlow','MixDriftBias','MixRecentBias','RegimeFlow']\n"
    'def audit(path):\n'
    '    assert v.sha(path)==AUDIT_SHA\n'
    "    d=pd.read_csv(path,low_memory=False);meta={r.field:json.loads(r.value) for r in d[d.section=='metadata'].itertuples()}\n"
    "    assert meta['mode']=='full' and meta['source_sha256']==CODE_SHA and meta['data']==m.OFFICIAL_HASHES and meta['historical_audit_sha256']==m.FULL_AUDIT_SHA\n"
    "    assert {r['origin'] for r in meta['reports']}==set(a.ORIGINS)\n"
    '    records={};rows=[]\n'
    "    for r in meta['reports']:\n"
    "        o=r['origin'];assert r['counts']==m.COUNTS[o] and r['mode']=='full' and r['data']==meta['data'] and r['code_sha256']==CODE_SHA and r['reference']=='MixBiasVar / reconstructed Ultralisk v2.7'\n"
    "        assert r['cache']['train_max']<r['cache']['cut']==a.ORIGINS[o][0]\n"
    "        w=np.asarray(r['weights']);assert w.shape==(4,) and np.all((w>=0)&(w<=.5))\n"
    "        q=d[(d.section=='daily_sse')&(d.origin==o)];rec={}\n"
    "        days=pd.date_range(pd.to_datetime(a.ORIGINS[o][1],format='%Y%m%d%H'),pd.to_datetime(a.ORIGINS[o][2],format='%Y%m%d%H')-pd.Timedelta(days=1)).strftime('%Y%m%d').astype(int).to_numpy()\n"
    '        for n in NAMES[:-1]:\n'
    "            x=q[q.model==n].sort_values('date');assert np.array_equal(x.date.to_numpy(),days)\n"
    "            ns=x.pixels_hours.to_numpy(dtype='int64');ss=x[SSE].to_numpy(dtype='f8');assert np.isfinite(ss).all() and (ss>=0).all() and (ns>=0).all() and (ss[ns==0]==0).all()\n"
    "            if 'counts' in rec:assert np.array_equal(rec['counts'],ns)\n"
    "            rec['counts']=ns;rec[n]=ss\n"
    "            g=d[(d.section=='global')&(d.origin==o)&(d.model==n)];assert len(g)==1 and abs(float(v.skill(ss.sum(0),ns.sum()))-g.skill.iloc[0])<1e-9\n"
    "        assert rec['counts'].sum()==m.COUNTS[o][2]*2450\n"
    "        rec['RegimeFlow']=np.stack([rec[n][:,j] for j,n in enumerate(ROUTE)],axis=1);records[o]=rec\n"
    "    def row(o,n,ss,counts,section='global',**extra):\n"
    "        rm=np.sqrt(ss/counts);return dict(section=section,origin=o,model=n,skill=float(v.skill(ss,counts)),pixels_hours=int(counts),**dict(zip(['NRMSE_'+j for j in VARS],rm/v.NORM)),**extra)\n"
    '    for o,r in records.items():\n'
    '        for n in NAMES:\n'
    "            rows.append(row(o,n,r[n].sum(0),r['counts'].sum()))\n"
    "            for i in range(0,len(r['counts']),7):\n"
    "                ns=r['counts'][i:i+7].sum()\n"
    "                if ns:rows.append(row(o,n,r[n][i:i+7].sum(0),ns,'weekly',week=i//7+1))\n"
    "            for i,ns in enumerate(r['counts']):\n"
    "                if ns:rows.append(row(o,n,r[n][i],ns,'daily',day=i+1))\n"
    "                rows.append(dict(section='daily_sse',origin=o,model=n,day=i+1,pixels_hours=int(ns),**dict(zip(SSE,r[n][i]))))\n"
    "    for n in NAMES:rows.append(row('Pooled',n,sum(r[n].sum(0) for r in records.values()),sum(r['counts'].sum() for r in records.values())))\n"
    "    gate=a.pooled_gate(records,'RegimeFlow',True);gate['statistical_gate_passed_retrospective']=bool(gate['review_eligible'] and min(gate['origins'].values())>0);gate['review_eligible']=False\n"
    '    return meta,rows,gate\n'
    '\n'
    'def components(X,t,h,Y,T,s,tr,ca,query,XX=None,tt=None,hh=None,TT=None):\n'
    '    assert max(T[tr])<min(T[ca]); assert not np.intersect1d(tr,ca).size\n'
    '    if XX is None: XX,tt,hh,TT=X,t,h,T\n'
    '    assert max(T[ca])<min(TT[query]);pred={};states={}\n'
    "    for name,drift in [('Physics',False),('Drift',True)]:\n"
    "        st=a.fit_anchor(X,t,h,Y,T,tr,s,'physics',False,drift)\n"
    '        p=a.predict_anchor(st,X[ca],t[ca],h[ca],T[ca]);q=a.predict_anchor(st,XX[query],tt[query],hh[query],TT[query]);states[name]=st\n'
    "        variant='AdaptiveFlow' if not drift else 'Bias';corr=m.fit_correction(p,Y[ca],h[ca],variant);last=pd.to_datetime(T[ca[-1]],format='%Y%m%d%H')\n"
    "        pred['AdaptiveFlow' if not drift else 'DriftBias']=m.apply_correction(corr,q,hh[query],TT[query],last,variant);states[variant]=corr\n"
    "        if not drift:pred['Bias325']=v.clip(q+v.bias(p,Y[ca],h[ca])[hh[query]])\n"
    "    pred['RegimeFlow']=pred['DriftBias'].copy();pred['RegimeFlow'][...,2]=pred['AdaptiveFlow'][...,2]\n"
    '    return pred,states\n'
    '\n'
    'def diagnostics(X,t,h,Y,T,s,train,test,TT):\n'
    '    from sklearn.preprocessing import StandardScaler\n'
    '    from sklearn.pipeline import make_pipeline\n'
    '    from sklearn.linear_model import LogisticRegression\n'
    '    from sklearn.metrics import roc_auc_score\n'
    '    from sklearn.decomposition import PCA\n'
    '    from scipy.stats import ks_2samp\n'
    '    def feat(x):\n'
    '        q=np.concatenate([x,np.hypot(x[...,2],x[...,3])[...,None]],axis=-1)\n'
    '        return np.concatenate([q.mean((1,2)),q.std((1,2)),np.quantile(q,.1,axis=(1,2)),np.quantile(q,.9,axis=(1,2))],axis=1)\n'
    "    t0=pd.to_datetime(T,format='%Y%m%d%H');t1=pd.to_datetime(TT,format='%Y%m%d%H');it=(t0>=t0.max().normalize()-pd.Timedelta(days=33))&(t0.hour%3==0);ie=t1.hour%3==0\n"
    '    f0,f1=feat(train[it]),feat(test[ie]);F=np.r_[f0,f1];y=np.r_[np.zeros(len(f0),int),np.ones(len(f1),int)]\n'
    '    groups=np.r_[np.asarray((t0[it]-t0[it].min()).days)//7,100+np.asarray((t1[ie]-t1[ie].min()).days)//7]\n'
    '    blocks0=np.unique(groups[y==0]);blocks1=np.unique(groups[y==1]);assert len(blocks0)==len(blocks1)\n'
    '    prob=np.full(len(y),np.nan);auc=[]\n'
    '    for b0,b1 in zip(blocks0,blocks1):\n'
    '        ev=np.isin(groups,[b0,b1]);tr=~ev;clf=make_pipeline(StandardScaler(),LogisticRegression(C=.1,max_iter=1000));clf.fit(F[tr],y[tr]);prob[ev]=clf.predict_proba(F[ev])[:,1];auc.append(float(roc_auc_score(y[ev],prob[ev])))\n'
    "    rows=[dict(section='shift',metric=k,value=x) for k,x in [('adversarial_auc_oof',float(roc_auc_score(y,prob))),('auc_fold_min',min(auc)),('auc_fold_max',max(auc)),('n_train',len(f0)),('n_test',len(f1)),('blocks',len(np.unique(groups)))]]\n"
    "    for j,n in enumerate([*VARS,'speed']):\n"
    "        for hour in range(0,24,3):rows.append(dict(section='shift',metric='KS_spatial_mean',variable=n,hour=hour,value=float(ks_2samp(f0[t0[it].hour==hour,j],f1[t1[ie].hour==hour,j]).statistic)))\n"
    "    x=train[it].reshape(len(f0),-1);z=test[ie].reshape(len(f1),-1);sc=StandardScaler().fit(x);xp=sc.transform(x);zp=sc.transform(z);pca=PCA(n_components=12,random_state=34,svd_solver='randomized').fit(xp)\n"
    "    for n,arr in [('train',xp),('test',zp)]:rows.append(dict(section='shift',metric='EOF_reconstruction_MSE',period=n,value=float(np.mean((arr-pca.inverse_transform(pca.transform(arr)))**2))))\n"
    "    dem=s['dem'];sl=np.hypot(*np.gradient(dem))\n"
    '    for o in a.ORIGINS:\n'
    "        tr,ca,ev=a.split(T,h,o);pred,_=components(X,t,h,Y,T,s,tr,ca,ev);clim=v.clip(v.A['climate'](Y[tr],h[tr],np.arange(len(tr)))[h[ev]])\n"
    "        for n,q in [('Climatology',clim),*pred.items()]:rows.append(dict(section='anchor_only',origin=o,model=n,skill=v.score(q,Y[ev])))\n"
    "        p=pred['Bias325'];y=Y[ev];speed=np.hypot(p[...,2],p[...,3]);ts=np.hypot(y[...,2],y[...,3]);ang=np.arctan2(p[...,3],p[...,2])-np.arctan2(y[...,3],y[...,2]);ang=np.abs(np.arctan2(np.sin(ang),np.cos(ang)))*180/np.pi\n"
    "        masks={**{f'hour_{i:02}':np.broadcast_to((h[ev]==i)[:,None,None],speed.shape) for i in range(0,24,3)},'wind_lt2':speed<2,'wind_2_5':(speed>=2)&(speed<5),'wind_ge5':speed>=5}\n"
    "        for name,field in [('slope',sl),('altitude',dem)]:\n"
    '            l,u=np.quantile(field,[.33,.67]);\n'
    "            for i,mask in enumerate([field<=l,(field>l)&(field<=u),field>u]):masks[f'{name}_{i+1}']=np.broadcast_to(mask,speed.shape)\n"
    '        for label,mask in masks.items():\n'
    '            if not mask.any():continue\n'
    "            keep=mask&(ts>=1)&(speed>=1);rows.append(dict(section='wind_anchor_only',origin=o,regime=label,pixels_hours=int(mask.sum()),RMSE_U=float(np.sqrt(np.mean((p[...,2][mask]-y[...,2][mask])**2))),RMSE_V=float(np.sqrt(np.mean((p[...,3][mask]-y[...,3][mask])**2))),speed_RMSE=float(np.sqrt(np.mean((speed[mask]-ts[mask])**2))),direction_MAE_deg=float(ang[keep].mean()) if keep.any() else np.nan,direction_count=int(keep.sum())))\n"
    "        print('diagnostics',o,flush=True)\n"
    '    return rows\n'
    '\n'
    'def main():\n'
    "    p=argparse.ArgumentParser();p.add_argument('--data',required=True);p.add_argument('--audit',required=True);p.add_argument('--base',required=True);p.add_argument('--out',required=True);p.add_argument('--diagnostics',action='store_true');args=p.parse_args();out=Path(args.out);out.mkdir(parents=True,exist_ok=True);data=v.data_root(args.data);dh=v.fingerprint(data);assert dh==m.OFFICIAL_HASHES\n"
    '    a.check_protocol();m.tests();assert v.sha(m.__file__)==CODE_SHA\n'
    '    for name,digest in m.EXPECTED.items():assert v.sha(Path(__file__).with_name(name))==digest\n'
    "    meta,rows,gate=audit(Path(args.audit));print('RETROSPECTIVE GATE:',json.dumps(gate),flush=True)\n"
    "    assert gate['statistical_gate_passed_retrospective'],'No competitive file: retrospective gate failed'\n"
    "    train=v.readnp(data/'train.npz');s=v.readnp(data/'static.npz');test=v.readnp(data/'test.npz');assert 'Y' not in test;T=train['times'].astype(str);TT=test['times'].astype(str);Y=train['Y'];X,t,h=v.A['prepare'](train['X'],T,s)\n"
    "    if args.diagnostics:rows+=diagnostics(X,t,h,Y,T,s,train['X'],test['X'],TT)\n"
    '    base=Path(args.base);assert v.sha(base)==BASE_SHA\n'
    "    tr=np.flatnonzero(T<'2024062400');ca=np.flatnonzero((T>='2024062400')&(T<'2024070100')&(h%3==0));assert len(tr)==2767 and len(ca)==56 and max(T[tr])<min(T[ca]);assert len(TT)==792\n"
    "    XX,tt,hh=v.A['prepare'](test['X'],TT,s);ix=np.flatnonzero(hh%3==0);assert len(ix)==264 and min(TT)>='2024070100'\n"
    '    pred,states=components(X,t,h,Y,T,s,tr,ca,ix,XX,tt,hh,TT)\n'
    "    w=np.asarray(next(r['weights'] for r in meta['reports'] if r['origin']=='O3'),dtype='f8')\n"
    "    tpl=pd.read_csv(data/'sample_submission.csv',dtype={'id':str});b=pd.read_csv(base,dtype={'id':str});ids=np.array([f'{stamp}_{i:02d}_{j:02d}' for stamp in TT[ix] for i in range(70) for j in range(35)]);assert len(ids)==646800 and np.array_equal(ids,tpl.id.to_numpy()) and np.array_equal(ids,b.id.to_numpy()) and list(b.columns)==['id',*VARS]\n"
    "    bv=b[VARS].to_numpy();assert np.isfinite(bv).all() and b.RH.between(0,100).all();av=pred['RegimeFlow'].reshape(-1,4);inc=pred['Bias325'].reshape(-1,4)\n"
    '    mix=bv+(av-bv)*w;mix[:,0]=np.clip(mix[:,0],0,100);prev=bv+(inc-bv)*w;prev[:,0]=np.clip(prev[:,0],0,100)\n'
    "    dd=bv+(pred['DriftBias'].reshape(-1,4)-bv)*w;aa=bv+(pred['AdaptiveFlow'].reshape(-1,4)-bv)*w;dd[:,2]=aa[:,2];dd[:,0]=np.clip(dd[:,0],0,100);assert np.array_equal(dd,mix)\n"
    "    for name,st in states.items():np.savez(out/f'{name}.npz',**st)\n"
    "    last=pd.to_datetime(T[ca[-1]],format='%Y%m%d%H');ii=ix[:8];p=a.predict_anchor(v.readnp(out/'Physics.npz'),XX[ii],tt[ii],hh[ii],TT[ii]);q=a.predict_anchor(v.readnp(out/'Drift.npz'),XX[ii],tt[ii],hh[ii],TT[ii]);u=m.apply_correction(v.readnp(out/'AdaptiveFlow.npz'),p,hh[ii],TT[ii],last,'AdaptiveFlow');q=m.apply_correction(v.readnp(out/'Bias.npz'),q,hh[ii],TT[ii],last,'Bias');q[...,2]=u[...,2];err=float(np.max(np.abs(q-pred['RegimeFlow'][:8])));assert err<=1e-4\n"
    "    result=pd.DataFrame(mix,columns=VARS);result.insert(0,'id',ids);dest=out/'Atlas34_REVIEW.csv';tmp=dest.with_suffix('.tmp');result.to_csv(tmp,index=False,float_format='%.6f');check=pd.read_csv(tmp,dtype={'id':str});assert len(check)==646800 and np.array_equal(check.id,ids) and list(check.columns)==['id',*VARS] and np.isfinite(check[VARS]).all().all() and check.RH.between(0,100).all();tmp.replace(dest)\n"
    "    report=dict(train_count=len(tr),cal_count=len(ca),train_max=str(max(T[tr])),cal_min=str(min(T[ca])),cal_max=str(max(T[ca])),test_count=len(ix),weights=dict(zip(VARS,w.tolist())),mean_abs_vs_v27=dict(zip(VARS,np.abs(check[VARS].to_numpy()-bv).mean(0).tolist())),mean_abs_vs_same_weights_bias325=dict(zip(VARS,np.abs(check[VARS].to_numpy()-prev).mean(0).tolist())),replay_max_abs=err,csv_sha256=v.sha(dest),rows=len(check),scope='manual experimental review; no test labels; no automatic submission')\n"
    "    rows+=[dict(section='metadata',field=k,value=json.dumps(val,sort_keys=True)) for k,val in dict(route=dict(zip(VARS,ROUTE)),gate=gate,deployment=report,source_sha256=v.sha(__file__),audit_sha256=AUDIT_SHA,data=dh).items()]\n"
    "    pd.DataFrame(rows).to_csv(out/'Atlas34_AUDIT.csv',index=False,float_format='%.17g');manifest=dict(version='3.4',route=dict(zip(VARS,ROUTE)),gate=gate,deploy=report,source_sha256=v.sha(__file__),audit_sha256=AUDIT_SHA,base_sha256=BASE_SHA,data=dh,files={p.name:v.sha(p) for p in sorted(out.iterdir()) if p.is_file() and p.name!='manifest.json'})\n"
    "    v.dump(out/'manifest.json',manifest);print('DEPLOY:',json.dumps(report),flush=True)\n"
    "if __name__=='__main__':main()\n"
)

EXPECTED={'Atlas_v3_2.py': 'cd60f877d245cd2955187788e562b6fe8fcb4c448829514cbdf376b81455829e', 'Atlas_v3_2_5.py': '4411658eb868a353f72815ece14101b4c7fc135d8e7d20973e865ec98971f55e', 'Atlas_v3_2_5_preregistro.md': 'bcb9d59dc0e13859bd7c6a9d7f6ab63fc4eea2f5175f83892b8520926aff5f37', 'Atlas_v3_3.py': '58015cb53e74b0f55c4fc1136a98851e7e91f184ea78634359ddca420df09377', 'Atlas_v3_4.py': 'd2cb790e59832ad52b3173de496681d39ad4dcd84864cae3940941a4f60bf6bf'}
for name,src in SOURCES.items():
    raw=src.encode('utf-8'); assert hashlib.sha256(raw).hexdigest()==EXPECTED[name],name
    (CODE/name).write_bytes(raw)
    if name.endswith('.py'): compile(src,name,'exec')

# --- Atlas33_FULL_AUDIT.csv: texto original o interfaz de despliegue ---
text=(
    'section,field,value,origin,model,skill,pixels_hours,RMSE_RH,RMSE_T,RMSE_U,RMSE_V,week,date,day,SSE_RH,SSE_T,SSE_U,SSE_V\n'
    'metadata,version,"""3.3""",,,,,,,,,,,,,,,\n'
    'metadata,mode,"""full""",,,,,,,,,,,,,,,\n'
    'metadata,selected,"""AdaptiveFlow""",,,,,,,,,,,,,,,\n'
    'metadata,gate,"{""bootstrap3"": {""approx_nonoverlap_blocks"": 30, ""ci95"": [0.00016421861027064721, 0.0002456283569604478], ""replicates"": 10000, ""se"": 2.0724894235878983e-05}, ""bootstrap7"": {""approx_nonoverlap_blocks"": 12, ""ci95"": [0.00016371468969498537, 0.00024761247960524693], ""replicates"": 10000, ""se"": 2.1278111782102246e-05}, ""cpu_signal"": true, ""last14_gain"": 0.0003390865583513536, ""last_gain"": 0.00032012449781793073, ""last_win_fraction"": 0.9375, ""late_weeks_gain"": 0.00033552040288320484, ""leave_one_out"": {""O1"": 0.00026645592927287964, ""O2"": 0.00021403240007444868, ""O3"": 0.00012802921983656734}, ""origins"": {""O1"": 6.421333549444519e-05, ""O2"": 0.00019247792443266576, ""O3"": 0.00032012449781793073}, ""pooled_gain"": 0.00020724831792451326, ""pooled_win_fraction"": 0.8452380952380952, ""review_eligible"": false, ""statistical_gate_passed"": true}",,,,,,,,,,,,,,,\n'
    'metadata,exploratory_gates,"{""AdaptiveFlow"": {""bootstrap3"": {""approx_nonoverlap_blocks"": 30, ""ci95"": [0.00016421861027064721, 0.0002456283569604478], ""replicates"": 10000, ""se"": 2.0724894235878983e-05}, ""bootstrap7"": {""approx_nonoverlap_blocks"": 12, ""ci95"": [0.00016371468969498537, 0.00024761247960524693], ""replicates"": 10000, ""se"": 2.1278111782102246e-05}, ""cpu_signal"": true, ""last14_gain"": 0.0003390865583513536, ""last_gain"": 0.00032012449781793073, ""last_win_fraction"": 0.9375, ""late_weeks_gain"": 0.00033552040288320484, ""leave_one_out"": {""O1"": 0.00026645592927287964, ""O2"": 0.00021403240007444868, ""O3"": 0.00012802921983656734}, ""origins"": {""O1"": 6.421333549444519e-05, ""O2"": 0.00019247792443266576, ""O3"": 0.00032012449781793073}, ""pooled_gain"": 0.00020724831792451326, ""pooled_win_fraction"": 0.8452380952380952, ""review_eligible"": false}, ""DriftBias"": {""bootstrap3"": {""approx_nonoverlap_blocks"": 30, ""ci95"": [0.0008929261303149388, 0.00156920912232113], ""replicates"": 10000, ""se"": 0.00017429320307431318}, ""bootstrap7"": {""approx_nonoverlap_blocks"": 12, ""ci95"": [0.0008441353771074661, 0.001592779531872257], ""replicates"": 10000, ""se"": 0.00019155292722719446}, ""cpu_signal"": true, ""last14_gain"": 0.002982707940663798, ""last_gain"": 0.002571298379083542, ""last_win_fraction"": 0.9375, ""late_weeks_gain"": 0.0031170440112578968, ""leave_one_out"": {""O1"": 0.0017452268753870293, ""O2"": 0.0014771738350083385, ""O3"": 0.0002935375538063445}, ""origins"": {""O1"": -5.991826958362534e-05, ""O2"": 0.0006287601063005299, ""O3"": 0.002571298379083542}, ""pooled_gain"": 0.0012224051053759677, ""pooled_win_fraction"": 0.7619047619047619, ""review_eligible"": false}, ""MultiScale"": {""bootstrap3"": {""approx_nonoverlap_blocks"": 30, ""ci95"": [-4.9402532294029464e-05, -2.2157428286154193e-05], ""replicates"": 10000, ""se"": 6.923420938784955e-06}, ""bootstrap7"": {""approx_nonoverlap_blocks"": 12, ""ci95"": [-5.1012654736465326e-05, -2.25527777492629e-05], ""replicates"": 10000, ""se"": 7.3607322022212105e-06}, ""cpu_signal"": false, ""last14_gain"": -6.590017095542411e-05, ""last_gain"": -4.692765563951706e-05, ""last_win_fraction"": 0.25, ""late_weeks_gain"": -6.20975300361648e-05, ""leave_one_out"": {""O1"": -4.474647810004395e-05, ""O2"": -3.3418990734279674e-05, ""O3"": -2.8738264797878443e-05}, ""origins"": {""O1"": -1.4434371341853769e-05, ""O2"": -4.2515075767690647e-05, ""O3"": -4.692765563951706e-05}, ""pooled_gain"": -3.599843099333366e-05, ""pooled_win_fraction"": 0.17857142857142858, ""review_eligible"": false}, ""PhysicsNewBias"": {""bootstrap3"": {""approx_nonoverlap_blocks"": 30, ""ci95"": [1.4021263466945656e-08, 4.72421007563506e-08], ""replicates"": 10000, ""se"": 8.555661649044985e-09}, ""bootstrap7"": {""approx_nonoverlap_blocks"": 12, ""ci95"": [1.8048337815934624e-08, 4.717816663168949e-08], ""replicates"": 10000, ""se"": 7.374741253620746e-09}, ""cpu_signal"": true, ""last14_gain"": 2.4877928717081943e-08, ""last_gain"": 5.502685329616952e-08, ""last_win_fraction"": 0.75, ""late_weeks_gain"": 4.0434196546534906e-08, ""leave_one_out"": {""O1"": 3.6740348652308796e-08, ""O2"": 3.3811447419651586e-08, ""O3"": 7.0376841021513314e-09}, ""origins"": {""O1"": 1.165411211978551e-09, ""O2"": 1.26913848319532e-08, ""O3"": 5.502685329616952e-08}, ""pooled_gain"": 2.7075198305581694e-08, ""pooled_win_fraction"": 0.6666666666666666, ""review_eligible"": false}, ""RecentBias"": {""bootstrap3"": {""approx_nonoverlap_blocks"": 30, ""ci95"": [-9.222766746261878e-05, 0.0005205305020759581], ""replicates"": 10000, ""se"": 0.0001575553600923623}, ""bootstrap7"": {""approx_nonoverlap_blocks"": 12, ""ci95"": [-0.00015337365275596703, 0.000572570123796015], ""replicates"": 10000, ""se"": 0.00018797199619714617}, ""cpu_signal"": false, ""last14_gain"": 0.0008416836364992797, ""last_gain"": 6.637954476973462e-05, ""last_win_fraction"": 0.625, ""late_weeks_gain"": 0.0007774332571233167, ""leave_one_out"": {""O1"": 0.0003287265215521362, ""O2"": 4.7249794129844425e-05, ""O3"": 0.0003502484239087389}, ""origins"": {""O1"": 1.0855005896037717e-05, ""O2"": 0.0006778459046713481, ""O3"": 6.637954476973462e-05}, ""pooled_gain"": 0.0002375776627074888, ""pooled_win_fraction"": 0.6666666666666666, ""review_eligible"": false}, ""Taper"": {""bootstrap3"": {""approx_nonoverlap_blocks"": 30, ""ci95"": [-0.00039613996632449633, -0.00020576993146177448], ""replicates"": 10000, ""se"": 4.881508387236856e-05}, ""bootstrap7"": {""approx_nonoverlap_blocks"": 12, ""ci95"": [-0.0003782391271439312, -0.00022441292257450454], ""replicates"": 10000, ""se"": 3.901766431457758e-05}, ""cpu_signal"": false, ""last14_gain"": -0.0005786204719631094, ""last_gain"": -0.0004629921979234508, ""last_win_fraction"": 0.09375, ""late_weeks_gain"": -0.0005024667888818835, ""leave_one_out"": {""O1"": -0.0003936101214784271, ""O2"": -0.00028130529477887745, ""O3"": -0.0001646772025750698}, ""origins"": {""O1"": -2.4990189320228673e-05, ""O2"": -0.00030048162875617734, ""O3"": -0.0004629921979234508}, ""pooled_gain"": -0.0002866200914121153, ""pooled_win_fraction"": 0.20238095238095238, ""review_eligible"": false}, ""WindFlow"": {""bootstrap3"": {""approx_nonoverlap_blocks"": 30, ""ci95"": [2.6792293574162684e-05, 8.416308786719228e-05], ""replicates"": 10000, ""se"": 1.4614131348354325e-05}, ""bootstrap7"": {""approx_nonoverlap_blocks"": 12, ""ci95"": [2.2299657962832067e-05, 8.774394952281438e-05], ""replicates"": 10000, ""se"": 1.686827490733544e-05}, ""cpu_signal"": false, ""last14_gain"": 0.00012681049358331808, ""last_gain"": 6.841223319831169e-05, ""last_win_fraction"": 0.625, ""late_weeks_gain"": 0.00010426611384117823, ""leave_one_out"": {""O1"": 5.2226170359270796e-05, ""O2"": 6.731568859430492e-05, ""O3"": 4.6916647736372497e-05}, ""origins"": {""O1"": 6.44251325415679e-05, ""O2"": 3.069182584813923e-05, ""O3"": 6.841223319831169e-05}, ""pooled_gain"": 5.5960519327658e-05, ""pooled_win_fraction"": 0.7261904761904762, ""review_eligible"": false}}",,,,,,,,,,,,,,,\n'
    'metadata,selection_provenance,"""AdaptiveFlow locked before full run, selected after retrospective proxy O3""",,,,,,,,,,,,,,,\n'
    'metadata,historical_audit_sha256,"""724ca9c3b31bd82852e2019e3db4dc3d2439d0188092c9868f7a1ba8ecfb17c7""",,,,,,,,,,,,,,,\n'
    'metadata,source_sha256,"""58015cb53e74b0f55c4fc1136a98851e7e91f184ea78634359ddca420df09377""",,,,,,,,,,,,,,,\n'
    'metadata,scope,"""retrospective; no submission; Reference=MixBiasVar""",,,,,,,,,,,,,,,\n'
    'metadata,data,"{""sample_submission.csv"": ""507d20c815b09f253b43696dc611312355672f1552d2b313e940b32f7f0ca6a7"", ""static.npz"": ""0e181ff372a9a33fdbbcede81da6ddd2da8208f15b94594d78d3f10464852165"", ""test.npz"": ""97894492e5caae72e45b09f0d4d8bd87e03c12e5029f33d276b353870aa5c0bd"", ""train.npz"": ""f5120b7a7cf5661e8e4143a726f2d22d13905d11d878afb60c8ff5cd02444844""}",,,,,,,,,,,,,,,\n'
    'metadata,reports,"[{""cache"": {""cache_hit"": false, ""cut"": ""2024031700"", ""seconds"": 111.61179115899998, ""sha256"": ""6a3d7bf9446a0e66f1b5b7f398eba452984f3c9ddfc7a782bac0d12c39eb7594"", ""train_max"": ""2024031623""}, ""code_sha256"": ""58015cb53e74b0f55c4fc1136a98851e7e91f184ea78634359ddca420df09377"", ""counts"": [666, 32, 208], ""data"": {""sample_submission.csv"": ""507d20c815b09f253b43696dc611312355672f1552d2b313e940b32f7f0ca6a7"", ""static.npz"": ""0e181ff372a9a33fdbbcede81da6ddd2da8208f15b94594d78d3f10464852165"", ""test.npz"": ""97894492e5caae72e45b09f0d4d8bd87e03c12e5029f33d276b353870aa5c0bd"", ""train.npz"": ""f5120b7a7cf5661e8e4143a726f2d22d13905d11d878afb60c8ff5cd02444844""}, ""mode"": ""full"", ""origin"": ""O1"", ""policy"": {""common"": 0.5, ""factors"": [0.5, 0.5, 0.5, 0.5], ""prequential_days"": 2}, ""reference"": ""MixBiasVar / reconstructed Ultralisk v2.7"", ""weights"": [0.03952687046992803, 0.0005696570216325534, 0.0889546934474862, 0.10351098446660346]}, {""cache"": {""cache_hit"": false, ""cut"": ""2024041400"", ""seconds"": 187.89921829900004, ""sha256"": ""2a054c741e192618a52a501f2a510adff8a9341ba82d5dfc8dc9e4c2c2efa586"", ""train_max"": ""2024041323""}, ""code_sha256"": ""58015cb53e74b0f55c4fc1136a98851e7e91f184ea78634359ddca420df09377"", ""counts"": [1215, 56, 206], ""data"": {""sample_submission.csv"": ""507d20c815b09f253b43696dc611312355672f1552d2b313e940b32f7f0ca6a7"", ""static.npz"": ""0e181ff372a9a33fdbbcede81da6ddd2da8208f15b94594d78d3f10464852165"", ""test.npz"": ""97894492e5caae72e45b09f0d4d8bd87e03c12e5029f33d276b353870aa5c0bd"", ""train.npz"": ""f5120b7a7cf5661e8e4143a726f2d22d13905d11d878afb60c8ff5cd02444844""}, ""mode"": ""full"", ""origin"": ""O2"", ""policy"": {""common"": 0.85, ""factors"": [0.675, 0.925, 0.425, 0.85], ""prequential_days"": 5}, ""reference"": ""MixBiasVar / reconstructed Ultralisk v2.7"", ""weights"": [0.12194809903384705, 0.3041453030346121, 0.022018879873766897, 0.06668981192635731]}, {""cache"": {""cache_hit"": false, ""cut"": ""2024052100"", ""seconds"": 308.90242458600005, ""sha256"": ""cb8549fba9901dd574eed0e881631d28f5f455dd19ace32ce4214e7963bda8e2"", ""train_max"": ""2024052023""}, ""code_sha256"": ""58015cb53e74b0f55c4fc1136a98851e7e91f184ea78634359ddca420df09377"", ""counts"": [1999, 56, 256], ""data"": {""sample_submission.csv"": ""507d20c815b09f253b43696dc611312355672f1552d2b313e940b32f7f0ca6a7"", ""static.npz"": ""0e181ff372a9a33fdbbcede81da6ddd2da8208f15b94594d78d3f10464852165"", ""test.npz"": ""97894492e5caae72e45b09f0d4d8bd87e03c12e5029f33d276b353870aa5c0bd"", ""train.npz"": ""f5120b7a7cf5661e8e4143a726f2d22d13905d11d878afb60c8ff5cd02444844""}, ""mode"": ""full"", ""origin"": ""O3"", ""policy"": {""common"": 0.85, ""factors"": [0.85, 0.85, 0.925, 0.925], ""prequential_days"": 5}, ""reference"": ""MixBiasVar / reconstructed Ultralisk v2.7"", ""weights"": [0.16763285710499315, 0.20518818054638593, 0.20938052051138795, 0.186255211148316]}]",,,,,,,,,,,,,,,\n'
    'metadata,seconds,675.0878721500001,,,,,,,,,,,,,,,\n'
    'global,,,O1,Ultralisk,0.50268596483755124,509600,9.1639470615691927,0.85642796359667683,1.5103936325368803,1.3551070241962151,,,,,,,\n'
    'global,,,O1,PhysicsLegacy,0.47080313671278018,509600,9.6196177536124594,0.89263706939163945,1.6350263665169713,1.4500590313659072,,,,,,,\n'
    'global,,,O1,Bias325,0.4719158374691354,509600,9.5694802840482236,0.89834472898996343,1.6266095636585254,1.4456874038173031,,,,,,,\n'
    'global,,,O1,Reference,0.50339748910585913,509600,9.1489194922794557,0.85640180435464308,1.5076989566438048,1.3525299168458866,,,,,,,\n'
    'global,,,O1,PhysicsNew,0.4708031351774607,509600,9.6196243808204915,0.89263678967525195,1.6350260997967825,1.4500589893434852,,,,,,,\n'
    'global,,,O1,PhysicsNewBias,0.47191585661325519,509600,9.5694852643838608,0.89834443598294533,1.6266093129231536,1.4456873256537115,,,,,,,\n'
    'global,,,O1,Placebo,0.47156739297081018,509600,9.5090859529166192,0.87933522144805099,1.6567611968112079,1.4492765007313455,,,,,,,\n'
    'global,,,O1,Recent,0.47096603004636051,509600,9.5824699466101517,0.89123936265258485,1.6399028525423174,1.4493676606073811,,,,,,,\n'
    'global,,,O1,RecentBias,0.47175735318474848,509600,9.5409558863735722,0.89819247534901592,1.6318056449762748,1.4454616622808825,,,,,,,\n'
    'global,,,O1,Drift,0.46177480948945382,509600,9.6670285188935043,0.90298166973556482,1.6843293078945387,1.4726503776204471,,,,,,,\n'
    'global,,,O1,DriftBias,0.46305764113214443,509600,9.6283845345547796,0.90850145707454721,1.6735694370477827,1.4679286523389421,,,,,,,\n'
    'global,,,O1,MultiScale,0.47161134123500509,509600,9.5674488598265288,0.89880057752376563,1.6281767945411889,1.4467146034213605,,,,,,,\n'
    'global,,,O1,AdaptiveFlow,0.47358272653497502,509600,9.5706769390357778,0.89978943526840272,1.6148458909553554,1.4393809178996442,,,,,,,\n'
    'global,,,O1,WindFlow,0.47376941018228741,509600,9.5694802840482236,0.89834472898996343,1.6148458909553554,1.4393809178996442,,,,,,,\n'
    'global,,,O1,Taper,0.47185208484122859,509600,9.588899689385606,0.89318302546911121,1.6300206329302265,1.4471794229002335,,,,,,,\n'
    'global,,,O1,MixPhysicsNewBias,0.50339749027127034,509600,9.1489196338113636,0.85640180402889321,1.5076989406336494,1.3525299082751816,,,,,,,\n'
    'global,,,O1,MixRecentBias,0.50340834411175517,509600,9.1480034886154016,0.85640087897500139,1.5077209130843701,1.3524984943575402,,,,,,,\n'
    'global,,,O1,MixDriftBias,0.50333757083627551,509600,9.1473598093757236,0.85640174143995518,1.5086295608861782,1.3524331641119405,,,,,,,\n'
    'global,,,O1,MixMultiScale,0.50338305473451728,509600,9.1487257102590522,0.85640209279067114,1.5078050663480334,1.352583381128865,,,,,,,\n'
    'global,,,O1,MixAdaptiveFlow,0.50346170244135358,509600,9.1489380714290753,0.85640204096848693,1.5070983986861624,1.3524601682852813,,,,,,,\n'
    'global,,,O1,MixWindFlow,0.5034619142384007,509600,9.1489194922794557,0.85640180435464308,1.5070983986861624,1.3524601682852813,,,,,,,\n'
    'global,,,O1,MixTaper,0.5033724989165389,509600,9.1493320663961679,0.85639732911963085,1.5079738814494861,1.3524948101041301,,,,,,,\n'
    'weekly,,,O1,Ultralisk,0.42829760140711998,98000,14.104740630515677,0.878768750956331,1.5488681337167649,1.5169804643737574,1,,,,,,\n'
    'weekly,,,O1,Ultralisk,0.53408195136528547,137200,6.7585456295938906,0.84764263299048326,1.4808943821923479,1.3228270967736724,2,,,,,,\n'
    'weekly,,,O1,Ultralisk,0.53617624954908794,137200,6.6543017386565317,0.81321688779319823,1.5097098175135344,1.3262531697501243,3,,,,,,\n'
    'weekly,,,O1,Ultralisk,0.50823319825203883,137200,8.9363130265785244,0.89044768117076245,1.5124552530359014,1.2915278653373223,4,,,,,,\n'
    'weekly,,,O1,PhysicsLegacy,0.4051792283741229,98000,13.938851328904921,0.92595856935588561,1.6743943688853389,1.5781152865472869,1,,,,,,\n'
    'weekly,,,O1,PhysicsLegacy,0.50568902469880805,137200,7.1495552734942489,0.8856098741109476,1.5880382538521769,1.4058178850987122,2,,,,,,\n'
    'weekly,,,O1,PhysicsLegacy,0.49596357518352391,137200,7.4821931434915196,0.86093071970451984,1.639964349012639,1.4441094678115358,3,,,,,,\n'
    'weekly,,,O1,PhysicsLegacy,0.4708582686442796,137200,9.8909036356111262,0.90642800875274077,1.647884759080172,1.4033011691730015,4,,,,,,\n'
    'weekly,,,O1,Bias325,0.4075265239776189,98000,13.8771346175352,0.91703679105946323,1.6711608696564957,1.5751344196278478,1,,,,,,\n'
    'weekly,,,O1,Bias325,0.50588462653321931,137200,7.1505246884451319,0.89139908247035227,1.5824752673194429,1.402589931658309,2,,,,,,\n'
    'weekly,,,O1,Bias325,0.49732805117574952,137200,7.4267204404437006,0.87083820054324912,1.6273513457608495,1.4371008131286382,3,,,,,,\n'
    'weekly,,,O1,Bias325,0.47162667385559359,137200,9.813046642582794,0.91862954632285976,1.6371111144012549,1.3993595864121366,4,,,,,,\n'
    'weekly,,,O1,Reference,0.42997586745141958,98000,14.068088035420329,0.87872021443294968,1.5442244550361297,1.5096529852909579,1,,,,,,\n'
    'weekly,,,O1,Reference,0.53487306959598635,137200,6.7420267256161424,0.84762328980707635,1.4773162315239237,1.3203900267438753,2,,,,,,\n'
    'weekly,,,O1,Reference,0.53598130768319452,137200,6.6552940168646844,0.8132107477033188,1.5097615852556217,1.3277651751328603,3,,,,,,\n'
    'weekly,,,O1,Reference,0.50892782278600923,137200,8.9321280473591731,0.89041246506153549,1.5093049893985646,1.2885636862086935,4,,,,,,\n'
    'weekly,,,O1,PhysicsNew,0.40517921693036829,98000,13.93885757632372,0.92595839752856224,1.6743937833480222,1.578115498035576,1,,,,,,\n'
    'weekly,,,O1,PhysicsNew,0.50568913760479417,137200,7.1495540779368243,0.88560940253053078,1.5880379166590686,1.4058177802943237,2,,,,,,\n'
    'weekly,,,O1,PhysicsNew,0.49596364674093468,137200,7.4821947530816537,0.86093043531933577,1.6399639829138517,1.4441093106792209,3,,,,,,\n'
    'weekly,,,O1,PhysicsNew,0.47085811378404469,137200,9.8909209336505857,0.90642784185186076,1.6478848903908985,1.4033011047016055,4,,,,,,\n'
    'weekly,,,O1,PhysicsNewBias,0.40752654858094067,98000,13.877138521997839,0.91703656420795521,1.671160294966636,1.5751345701412718,1,,,,,,\n'
    'weekly,,,O1,PhysicsNewBias,0.50588473917701215,137200,7.1505233741512422,0.89139861825760469,1.5824749508039313,1.4025898152486187,2,,,,,,\n'
    'weekly,,,O1,PhysicsNewBias,0.49732813648565699,137200,7.4267209049551086,0.8708379087907836,1.6273509926020713,1.4371006327772331,3,,,,,,\n'
    'weekly,,,O1,PhysicsNewBias,0.47162655035877832,137200,9.8130613440346526,0.91862937082505558,1.6371112651085782,1.3993594673586769,4,,,,,,\n'
    'weekly,,,O1,Placebo,0.40825670040621531,98000,13.892907268698671,0.91341402138387517,1.6750723449122491,1.5683512402735738,1,,,,,,\n'
    'weekly,,,O1,Placebo,0.50102093894246813,137200,7.3972014281423659,0.86987646832670062,1.614650246279542,1.4195683064577649,2,,,,,,\n'
    'weekly,,,O1,Placebo,0.49617507450187209,137200,7.3189077083176377,0.85159455544055418,1.6674250647857254,1.4438123813925718,3,,,,,,\n'
    'weekly,,,O1,Placebo,0.47541693046284639,137200,9.471244089316416,0.8911530934951154,1.6743932841432874,1.3945534418383931,4,,,,,,\n'
    'weekly,,,O1,Recent,0.40400310883117108,98000,13.922942086334023,0.9235813403088412,1.6865479745223908,1.5821733951739869,1,,,,,,\n'
    'weekly,,,O1,Recent,0.5053204880939095,137200,7.1224244557798331,0.88826455550998229,1.5893300577448746,1.4074380499421382,2,,,,,,\n'
    'weekly,,,O1,Recent,0.49648833966059791,137200,7.4496429046639552,0.85895910459367453,1.6429479087361487,1.4420738225082412,3,,,,,,\n'
    'weekly,,,O1,Recent,0.47228041562957618,137200,9.8168414965281237,0.90231547632146736,1.652805412255028,1.3978431000282547,4,,,,,,\n'
    'weekly,,,O1,RecentBias,0.4060373217629038,98000,13.872370575413823,0.91568064688932804,1.6831864039846514,1.5799571276204272,1,,,,,,\n'
    'weekly,,,O1,RecentBias,0.50518444833037601,137200,7.1267301720916327,0.8956303400712684,1.5840145287167544,1.4049181893247575,2,,,,,,\n'
    'weekly,,,O1,RecentBias,0.4975782934419094,137200,7.4012453724366631,0.86966981744077754,1.6310799536088314,1.4353822963450522,3,,,,,,\n'
    'weekly,,,O1,RecentBias,0.47270546213659281,137200,9.7510547480709473,0.91603044823923063,1.6423103060210638,1.3940280808880898,4,,,,,,\n'
    'weekly,,,O1,Drift,0.39910815786749509,98000,13.994574970350383,0.92984421938511419,1.7165062454851414,1.5874635968610868,1,,,,,,\n'
    'weekly,,,O1,Drift,0.5011523224970138,137200,7.1841746262031805,0.89007640024621071,1.6096615488380313,1.4194776982970494,2,,,,,,\n'
    'weekly,,,O1,Drift,0.48580433723924837,137200,7.5176008227901718,0.88006329578426956,1.6864054625409233,1.4704803885023741,3,,,,,,\n'
    'weekly,,,O1,Drift,0.45636292096975101,137200,9.9542524773129877,0.91881057516040199,1.7314036996457445,1.4414956311822018,4,,,,,,\n'
    'weekly,,,O1,DriftBias,0.40158155734674539,98000,13.940868248519514,0.92110510758247277,1.7108987887111493,1.584436557269058,1,,,,,,\n'
    'weekly,,,O1,DriftBias,0.50154724510366022,137200,7.1922490801752854,0.89582500916704799,1.6018216994110031,1.4158166634710638,2,,,,,,\n'
    'weekly,,,O1,DriftBias,0.48741050214313281,137200,7.4767971630189072,0.88926199034441333,1.6713387589109991,1.4628755094068395,3,,,,,,\n'
    'weekly,,,O1,DriftBias,0.4572203396172258,137200,9.8936839247950008,0.93078683285051345,1.7184455527294784,1.4373184681975604,4,,,,,,\n'
    'weekly,,,O1,MultiScale,0.4073132049464423,98000,13.873150211924631,0.91754609847412005,1.6724446108986604,1.5757280047839135,1,,,,,,\n'
    'weekly,,,O1,MultiScale,0.5056272134437545,137200,7.1497238332426241,0.89157231688018868,1.5843385654199815,1.4031775190900957,2,,,,,,\n'
    'weekly,,,O1,MultiScale,0.496943787354656,137200,7.4258658997660554,0.87135371560543551,1.6290670250084791,1.4385211725996281,3,,,,,,\n'
    'weekly,,,O1,MultiScale,0.47127559536635449,137200,9.8109435183119444,0.91926529362719778,1.6384522407178717,1.4007761019003526,4,,,,,,\n'
    'weekly,,,O1,AdaptiveFlow,0.40865375380650693,98000,13.890984618241388,0.91889385792672618,1.6597761226665151,1.5715804343976376,1,,,,,,\n'
    'weekly,,,O1,AdaptiveFlow,0.50712015287622181,137200,7.1530947954288351,0.89331266071332682,1.5738925508987116,1.396818435371147,2,,,,,,\n'
    'weekly,,,O1,AdaptiveFlow,0.49973468013449079,137200,7.4135260955132694,0.87096140977052094,1.6125974954653368,1.4294689830830425,3,,,,,,\n'
    'weekly,,,O1,AdaptiveFlow,0.47346608681918811,137200,9.8114879844740237,0.92057808724098222,1.6249556277002115,1.3916354501768613,4,,,,,,\n'
    'weekly,,,O1,WindFlow,0.40901509244953299,98000,13.8771346175352,0.91703679105946323,1.6597761226665151,1.5715804343976376,1,,,,,,\n'
    'weekly,,,O1,WindFlow,0.50737713473711787,137200,7.1505246884451319,0.89139908247035227,1.5738925508987116,1.396818435371147,2,,,,,,\n'
    'weekly,,,O1,WindFlow,0.49961952462624659,137200,7.4267204404437006,0.87083820054324912,1.6125974954653368,1.4294689830830425,3,,,,,,\n'
    'weekly,,,O1,WindFlow,0.47368659949093411,137200,9.813046642582794,0.91862954632285976,1.6249556277002115,1.3916354501768613,4,,,,,,\n'
    'weekly,,,O1,Taper,0.4072960130204783,98000,13.888462263765026,0.91757731427300715,1.6713049057705474,1.5754681180121872,1,,,,,,\n'
    'weekly,,,O1,Taper,0.50626169239472718,137200,7.1452765058372671,0.88708271040961639,1.5838574014941766,1.4031589075479538,2,,,,,,\n'
    'weekly,,,O1,Taper,0.49699617654807871,137200,7.4522751678847747,0.86413931251414577,1.6330161424480578,1.4401234381065409,3,,,,,,\n'
    'weekly,,,O1,Taper,0.4715149961757939,137200,9.8563636996704034,0.91001301384078037,1.6426210105292265,1.4011405212307304,4,,,,,,\n'
    'weekly,,,O1,MixPhysicsNewBias,0.42997586437214541,98000,14.068088204773415,0.87872021338866069,1.5442244300631385,1.5096530175891116,1,,,,,,\n'
    'weekly,,,O1,MixPhysicsNewBias,0.53487307191818834,137200,6.7420267287118731,0.84762328893240413,1.4773162098779893,1.3203900248582097,2,,,,,,\n'
    'weekly,,,O1,MixPhysicsNewBias,0.53598131086156842,137200,6.6552940310605564,0.81321074746591493,1.5097615626872551,1.3277651652590652,3,,,,,,\n'
    'weekly,,,O1,MixPhysicsNewBias,0.50892782495298616,137200,8.9321283823721132,0.89041246568340793,1.509304992008456,1.2885636378723413,4,,,,,,\n'
    'weekly,,,O1,MixRecentBias,0.42986145681014482,98000,14.06816300570612,0.87871832556870977,1.5449026792327731,1.5100753179965214,1,,,,,,\n'
    'weekly,,,O1,MixRecentBias,0.53489596948539087,137200,6.7405587773537228,0.84762501276500635,1.4771810132125278,1.3204233247440482,2,,,,,,\n'
    'weekly,,,O1,MixRecentBias,0.53602147678230638,137200,6.6537517352980746,0.81320887541661435,1.5096373494102331,1.3276554590616465,3,,,,,,\n'
    'weekly,,,O1,MixRecentBias,0.50900326851057387,137200,8.9308158181459376,0.89041056048303768,1.509147299273832,1.2881665739992667,4,,,,,,\n'
    'weekly,,,O1,MixDriftBias,0.4299854794473324,98000,14.067649310111396,0.878719106311174,1.545441886821826,1.5086607818231572,1,,,,,,\n'
    'weekly,,,O1,MixDriftBias,0.53499860658824372,137200,6.7399747632897089,0.8476232434959825,1.4766994700156326,1.3199915643675586,2,,,,,,\n'
    'weekly,,,O1,MixDriftBias,0.53584117592121405,137200,6.6528266274589027,0.81321489847485318,1.5106525965924249,1.3284395676398608,3,,,,,,\n'
    'weekly,,,O1,MixDriftBias,0.50871623853165748,137200,8.9300748626016784,0.89040927461009556,1.5115791495463082,1.2887296830029686,4,,,,,,\n'
    'weekly,,,O1,MixMultiScale,0.4299669734127109,98000,14.067869827667236,0.87872055628061518,1.5442709798223218,1.5097085978669511,1,,,,,,\n'
    'weekly,,,O1,MixMultiScale,0.53485707021614504,137200,6.7418134133251622,0.84762336044403475,1.477463838652487,1.3204260337311555,2,,,,,,\n'
    'weekly,,,O1,MixMultiScale,0.53595943259773693,137200,6.6551818013125628,0.81321100435272042,1.5098953471138887,1.3278525309514402,3,,,,,,\n'
    'weekly,,,O1,MixMultiScale,0.50891840787104714,137200,8.9318809198062166,0.89041295286401645,1.5093864079147423,1.2885986763936836,4,,,,,,\n'
    'weekly,,,O1,MixAdaptiveFlow,0.4299251160393841,98000,14.068662192351976,0.87872035642772561,1.5439786062636118,1.5102210493819701,1,,,,,,\n'
    'weekly,,,O1,MixAdaptiveFlow,0.53492453965390407,137200,6.7419131087063917,0.84762342852925465,1.4769369060323905,1.3202651861488697,2,,,,,,\n'
    'weekly,,,O1,MixAdaptiveFlow,0.53614993948354583,137200,6.6547320414881215,0.81321056897557764,1.5086758435195735,1.3272499761052754,3,,,,,,\n'
    'weekly,,,O1,MixAdaptiveFlow,0.50899426388063329,137200,8.9320572572527794,0.890413241426257,1.5087136198404003,1.2884749823490398,4,,,,,,\n'
    'weekly,,,O1,MixWindFlow,0.42993079329275941,98000,14.068088035420329,0.87872021443294968,1.5439786062636118,1.5102210493819701,1,,,,,,\n'
    'weekly,,,O1,MixWindFlow,0.53492343640621609,137200,6.7420267256161424,0.84762328980707635,1.4769369060323905,1.3202651861488697,2,,,,,,\n'
    'weekly,,,O1,MixWindFlow,0.53614437786981894,137200,6.6552940168646844,0.8132107477033188,1.5086758435195735,1.3272499761052754,3,,,,,,\n'
    'weekly,,,O1,MixWindFlow,0.50899366001042212,137200,8.9321280473591731,0.89041246506153549,1.5087136198404003,1.2884749823490398,4,,,,,,\n'
    'weekly,,,O1,MixTaper,0.42997267967346758,98000,14.06872396927468,0.87871804347708082,1.5442107850192852,1.5096401076980994,1,,,,,,\n'
    'weekly,,,O1,MixTaper,0.53486770605620093,137200,6.7417481292204497,0.84761964111961563,1.4774449825961653,1.3203610551637806,2,,,,,,\n'
    'weekly,,,O1,MixTaper,0.5359147240509935,137200,6.6556304663814787,0.81320664192406211,1.5102527613953254,1.3279146821925547,3,,,,,,\n'
    'weekly,,,O1,MixTaper,0.50891214551843322,137200,8.9329418044552114,0.89040523113075831,1.5097176464172095,1.2883131932458805,4,,,,,,\n'
    'daily,,,O1,Ultralisk,0.44892049605149159,19600,9.9702419874219395,0.88217765368724588,1.6335839722227139,1.6156223641189478,,20240326,3,,,,\n'
    'daily,,,O1,Ultralisk,0.3524993077837385,19600,18.869866936348561,0.97048716123469436,1.6393372774827146,1.5944201490786645,,20240327,4,,,,\n'
    'daily,,,O1,Ultralisk,0.39173568424113769,19600,16.135166910750545,0.82911408391195618,1.5153703245959531,1.7314551476000395,,20240328,5,,,,\n'
    'daily,,,O1,Ultralisk,0.44135124354670618,19600,12.85742642715433,0.91147470324469138,1.612547364697543,1.427946080342674,,20240329,6,,,,\n'
    'daily,,,O1,Ultralisk,0.53873891365693516,19600,10.657575344321089,0.78922338200542419,1.3199556623272819,1.1475061981148926,,20240330,7,,,,\n'
    'daily,,,O1,Ultralisk,0.57589762598284666,19600,6.8206487483635465,0.69459415603020358,1.3298425212172769,1.2410138481153443,,20240331,8,,,,\n'
    'daily,,,O1,Ultralisk,0.42750357003191308,19600,8.883583129056845,1.1065174600427363,1.7796299785112617,1.5432761564246289,,20240401,9,,,,\n'
    'daily,,,O1,Ultralisk,0.56525471290237506,19600,6.4560357717017558,0.80698639196160071,1.3605579964629304,1.2223421504214893,,20240402,10,,,,\n'
    'daily,,,O1,Ultralisk,0.54821570535361419,19600,5.8374992384517226,0.78730325343572782,1.4442653568631605,1.3701756479004228,,20240403,11,,,,\n'
    'daily,,,O1,Ultralisk,0.57402136090607825,19600,5.7910164938442881,0.84241652125659316,1.3260351850676606,1.1949820059334957,,20240404,12,,,,\n'
    'daily,,,O1,Ultralisk,0.50929509998786504,19600,6.8924558649233933,0.90420556296149479,1.6081455061193857,1.3623957071821446,,20240405,13,,,,\n'
    'daily,,,O1,Ultralisk,0.56073537734268819,19600,6.1243497210227638,0.72409775363018214,1.4609077462813247,1.2929340018957898,,20240406,14,,,,\n'
    'daily,,,O1,Ultralisk,0.55053397138235505,19600,6.3847441437236085,0.77210820375954481,1.5326932246290288,1.2521855299930764,,20240407,15,,,,\n'
    'daily,,,O1,Ultralisk,0.56828066373849784,19600,6.2291547164722143,0.76789772604021089,1.4596457032538701,1.1780093520422821,,20240408,16,,,,\n'
    'daily,,,O1,Ultralisk,0.60149491112845355,19600,5.7362436410129582,0.72786910020554796,1.2195180252953941,1.1689120144003051,,20240409,17,,,,\n'
    'daily,,,O1,Ultralisk,0.55336770780997968,19600,6.5355942437281351,0.82632283451213284,1.403579889392871,1.261965915277538,,20240410,18,,,,\n'
    'daily,,,O1,Ultralisk,0.55033345175282489,19600,6.0164608235217392,0.78761460983263365,1.5390517012103702,1.2635445179593576,,20240411,19,,,,\n'
    'daily,,,O1,Ultralisk,0.48527855406791082,19600,8.0104644669568401,0.85907333487021065,1.6711566013972317,1.4674054905418308,,20240412,20,,,,\n'
    'daily,,,O1,Ultralisk,0.4615403885666412,19600,7.3761080588604759,0.93407030142106084,1.6900138421046922,1.6273354803523259,,20240413,21,,,,\n'
    'daily,,,O1,Ultralisk,0.51672598318681096,19600,7.8396217092586857,0.93532435795125479,1.5467948078108684,1.2387514421959676,,20240414,22,,,,\n'
    'daily,,,O1,Ultralisk,0.49909419217512291,19600,9.4848352531909192,0.90477645684609098,1.503497072283869,1.315188718202122,,20240415,23,,,,\n'
    'daily,,,O1,Ultralisk,0.55011342300956101,19600,7.9376234458632551,0.8423856317657894,1.3388596287021486,1.2079497489509876,,20240416,24,,,,\n'
    'daily,,,O1,Ultralisk,0.48727626012007169,19600,10.607335799032125,0.87149310000675717,1.5422597511391714,1.3248933976484301,,20240417,25,,,,\n'
    'daily,,,O1,Ultralisk,0.47013712163438459,19600,10.870070444767505,0.94762610145653059,1.5772831794752671,1.3421192016055581,,20240418,26,,,,\n'
    'daily,,,O1,Ultralisk,0.51336309382471002,19600,8.3756297255435097,0.8591829296536535,1.5157933725767689,1.3238471418741451,,20240419,27,,,,\n'
    'daily,,,O1,Ultralisk,0.53158226591183155,19600,6.6144555137337928,0.86690432467605549,1.5501634111469813,1.2820853212953098,,20240420,28,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.42598271897915568,19600,10.190780462919705,0.97383948045399238,1.7352412732426703,1.617094065825877,,20240326,3,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.32779611788409979,19600,18.239391049619503,1.0274103031088868,1.7859133346550062,1.6807824880504538,,20240327,4,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.37367853857821942,19600,15.675127150910528,0.89912456606994751,1.5843394039917926,1.7955675481896369,,20240328,5,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.41650931724427909,19600,12.850515043088141,0.90971495444208239,1.7658487968402381,1.5182118870260004,,20240329,6,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.50882608490694714,19600,11.139351034614164,0.80439717338312577,1.4795541481027541,1.2178612992808309,,20240330,7,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.55529042777872295,19600,7.0651703150815903,0.71257029837373687,1.4102552788767626,1.3120983195247025,,20240331,8,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.37805058844973138,19600,8.628311925228207,1.2365043835224954,1.9115005578142377,1.7430317421555217,,20240401,9,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.54236120843229829,19600,7.0905714893125813,0.78970549113568922,1.4479543763329361,1.310607420583686,,20240402,10,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.50875951364622196,19600,6.6106923866364831,0.86592869142598505,1.6008041079749149,1.4344782300565413,,20240403,11,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.55552457065314531,19600,6.4733934126395587,0.79031748966156012,1.4277927181985712,1.2665937925414128,,20240404,12,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.48288791600255038,19600,7.3136298726202496,0.95521629299939104,1.698209392207856,1.4264531665354929,,20240405,13,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.54555163557120201,19600,6.6390604709421153,0.73683597972944337,1.5584451487897395,1.2884322850518406,,20240406,14,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.52519063393886978,19600,7.0632718488422919,0.77495578913467411,1.6303687307275196,1.328925562180308,,20240407,15,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.55339034521708297,19600,6.8839721974695456,0.76601469393997812,1.4837282155310003,1.2315732448332433,,20240408,16,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.55636647374761405,19600,6.2560882770406048,0.79831729070658042,1.3615348111098808,1.3210234558657763,,20240409,17,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.51519831994295373,19600,7.1154895099156503,0.87117503444268451,1.5176889655035215,1.398655181606322,,20240410,18,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.50777844053719878,19600,6.4343887479422417,0.85556362402496888,1.6892672331696457,1.3986905667255924,,20240411,19,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.42895870333390701,19600,9.2184870350421395,0.93426643192082082,1.8661435184443675,1.6102251421414591,,20240412,20,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.40796727678074007,19600,8.857365520213591,0.99998881225001024,1.8637482641608407,1.7506712084631075,,20240413,21,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.47512275138118382,19600,9.0614300940441357,0.92337223733974239,1.6969977562958043,1.380553014266634,,20240414,22,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.44977355983326539,19600,9.9434451053282729,0.9673363301348008,1.7118722313543944,1.4635847712782946,,20240415,23,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.51550502672157383,19600,8.4105072596443389,0.85931392382957827,1.4568699801358851,1.3488974847152917,,20240416,24,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.47572365243037229,19600,10.858305148362176,0.85862626555110322,1.6068139884922681,1.3633793058021642,,20240417,25,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.41978548347322531,19600,12.575950450887651,0.99901206201247961,1.6832825142181893,1.4875637890275928,,20240418,26,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.47221646152110958,19600,9.9228090406487439,0.86966398191718852,1.6916664876208394,1.3923615167221033,,20240419,27,,,,\n'
    'daily,,,O1,PhysicsLegacy,0.49934591100349679,19600,7.6532421319890167,0.85611112362802333,1.6729807149091851,1.3809156442238362,,20240420,28,,,,\n'
    'daily,,,O1,Bias325,0.42766286768696699,19600,10.107528162804885,0.96982489456961496,1.7337984938453064,1.6151003634565724,,20240326,3,,,,\n'
    'daily,,,O1,Bias325,0.33546793153390619,19600,18.14005079668193,0.98955419888062846,1.771319113404314,1.6745424685569463,,20240327,4,,,,\n'
    'daily,,,O1,Bias325,0.37636011413240261,19600,15.608386890696693,0.88242204960517467,1.5848628173387174,1.7951448830183576,,20240328,5,,,,\n'
    'daily,,,O1,Bias325,0.4152371840379051,19600,12.838850789701189,0.91730553476630161,1.7753177711554218,1.51473946423449,,20240329,6,,,,\n'
    'daily,,,O1,Bias325,0.50928626976867608,19600,11.099230899841301,0.81540568086790643,1.4685779871087907,1.2147508966479754,,20240330,7,,,,\n'
    'daily,,,O1,Bias325,0.55406389034959846,19600,7.0922183128199459,0.72394022478612841,1.4029527282903995,1.3143140533252264,,20240331,8,,,,\n'
    'daily,,,O1,Bias325,0.38742933020279158,19600,8.5426775109258699,1.1838882405462592,1.9071061082593737,1.7283777898576771,,20240401,9,,,,\n'
    'daily,,,O1,Bias325,0.53884723176819627,19600,7.0968232102946169,0.81706084083639474,1.4476928974316403,1.3119877042850596,,20240402,10,,,,\n'
    'daily,,,O1,Bias325,0.51020119684626386,19600,6.5454114418792333,0.86360126407596305,1.6072498620636562,1.4251406527731287,,20240403,11,,,,\n'
    'daily,,,O1,Bias325,0.55097652154677634,19600,6.5121431305960602,0.82622220949513492,1.4283185051744378,1.2646779101450227,,20240404,12,,,,\n'
    'daily,,,O1,Bias325,0.48165984750969049,19600,7.3526145339553359,0.985154610104925,1.6714021541152062,1.4241361810725901,,20240405,13,,,,\n'
    'daily,,,O1,Bias325,0.54248248511384978,19600,6.7049880184908321,0.7549153251987315,1.5529069445347166,1.2946762933076046,,20240406,14,,,,\n'
    'daily,,,O1,Bias325,0.52451659271548412,19600,7.1305587018936238,0.78855959790112096,1.6219850315932862,1.3218001341594907,,20240407,15,,,,\n'
    'daily,,,O1,Bias325,0.55259572600791829,19600,6.811074558930132,0.7753651411185577,1.4916828322293965,1.2285557554422295,,20240408,16,,,,\n'
    'daily,,,O1,Bias325,0.55808055804170287,19600,6.2704421466589535,0.79411447238503174,1.3519850441661529,1.3172224880201169,,20240409,17,,,,\n'
    'daily,,,O1,Bias325,0.51662816074054119,19600,7.073874340933564,0.88109091077460011,1.5016677051731135,1.3926087444196522,,20240410,18,,,,\n'
    'daily,,,O1,Bias325,0.51116572192180876,19600,6.3748262924688772,0.86346498243601322,1.6659136314261049,1.3855291666428569,,20240411,19,,,,\n'
    'daily,,,O1,Bias325,0.4309045338852957,19600,9.1095520597368758,0.94754776089742643,1.8488740895275495,1.6029828403245348,,20240412,20,,,,\n'
    'daily,,,O1,Bias325,0.40998393024954199,19600,8.7107485075020801,1.0174179749729844,1.845394434927252,1.7425920735495517,,20240413,21,,,,\n'
    'daily,,,O1,Bias325,0.47722808608609979,19600,8.9202201777242713,0.94717423421846836,1.667047454791049,1.373887523657072,,20240414,22,,,,\n'
    'daily,,,O1,Bias325,0.44908077500812021,19600,9.8567337336002101,1.0031918900654888,1.691351852774075,1.456262080874704,,20240415,23,,,,\n'
    'daily,,,O1,Bias325,0.51545792478649088,19600,8.3836565416902662,0.8799784470755545,1.4459614524758695,1.3391173303765729,,20240416,24,,,,\n'
    'daily,,,O1,Bias325,0.47419391244880138,19600,10.81932543934753,0.86028292405222373,1.6187455574417629,1.3683848862432759,,20240417,25,,,,\n'
    'daily,,,O1,Bias325,0.42555108815938081,19600,12.465481535916028,0.97069282361445897,1.6778276440659721,1.4813916399669125,,20240418,26,,,,\n'
    'daily,,,O1,Bias325,0.4719019380050346,19600,9.8370610486842036,0.88572907262146938,1.6782810642682748,1.3962432214593723,,20240419,27,,,,\n'
    'daily,,,O1,Bias325,0.49908354181200709,19600,7.6056665048517935,0.87316629615672003,1.6668343241630901,1.3745747862090785,,20240420,28,,,,\n'
    'daily,,,O1,Reference,0.45131174464041629,19600,9.9458648811156696,0.88214466768654154,1.6275782823024589,1.6023768719293252,,20240326,3,,,,\n'
    'daily,,,O1,Reference,0.3549395818449535,19600,18.815108340507081,0.97041548011105916,1.631570595775526,1.5846783147308543,,20240327,4,,,,\n'
    'daily,,,O1,Reference,0.39430980108231412,19600,16.076633602597649,0.8290639426039188,1.502560961979299,1.7248161715573045,,20240328,5,,,,\n'
    'daily,,,O1,Reference,0.44165932548433101,19600,12.832181427357961,0.91142186442417761,1.61486769697832,1.4256959632049593,,20240329,6,,,,\n'
    'daily,,,O1,Reference,0.53904048133624116,19600,10.653823097200911,0.78919187883687103,1.3215994251012706,1.1440471467351163,,20240330,7,,,,\n'
    'daily,,,O1,Reference,0.57674256544360525,19600,6.8088956509074352,0.69457878925784977,1.3249203621843166,1.2387788758866134,,20240331,8,,,,\n'
    'daily,,,O1,Reference,0.42838097539744702,19600,8.8294918072897115,1.1065136645396068,1.7712760213571954,1.5469192211332468,,20240401,9,,,,\n'
    'daily,,,O1,Reference,0.56598151956348663,19600,6.4455892052488517,0.80694327753153128,1.3557419236036865,1.2209314206760813,,20240402,10,,,,\n'
    'daily,,,O1,Reference,0.54874973973300356,19600,5.8254673178957894,0.78728114460378518,1.4456964055461723,1.3656133248505209,,20240403,11,,,,\n'
    'daily,,,O1,Reference,0.57463918667917069,19600,5.7913905863035389,0.84237160449268877,1.3240444752345295,1.1913895865068431,,20240404,12,,,,\n'
    'daily,,,O1,Reference,0.51004130743225629,19600,6.8834151631260072,0.90421548493927939,1.6040516889638656,1.3600905235164005,,20240405,13,,,,\n'
    'daily,,,O1,Reference,0.561834776481556,19600,6.1204727037005178,0.72407174271772767,1.4595410826044182,1.2851645470596278,,20240406,14,,,,\n'
    'daily,,,O1,Reference,0.55049962050275492,19600,6.3909461742209439,0.77208612814575195,1.5326138092678785,1.252046062548932,,20240407,15,,,,\n'
    'daily,,,O1,Reference,0.56932215649724893,19600,6.2258174903350403,0.76787093363347958,1.4529322801909408,1.1748484129986487,,20240408,16,,,,\n'
    'daily,,,O1,Reference,0.6011482776377578,19600,5.7286459754888046,0.72786930441905473,1.2180711440145169,1.1735598314931934,,20240409,17,,,,\n'
    'daily,,,O1,Reference,0.55322645785877922,19600,6.5257370293466863,0.82631903146649055,1.4014411530521231,1.2656281584756346,,20240410,18,,,,\n'
    'daily,,,O1,Reference,0.55032505072202409,19600,5.9985832513465178,0.78760948473879211,1.5396645874905479,1.2646136686518124,,20240411,19,,,,\n'
    'daily,,,O1,Reference,0.48418234745050548,19600,8.0259869874882064,0.85908216377298785,1.6768574740251705,1.4708278324235651,,20240412,20,,,,\n'
    'daily,,,O1,Reference,0.46101219745973232,19600,7.3921225704338474,0.93407256175551157,1.6928054712518266,1.6282490069414706,,20240413,21,,,,\n'
    'daily,,,O1,Reference,0.51691678241604344,19600,7.8357110328103419,0.93527651178967519,1.5437712261130887,1.2398858247041402,,20240414,22,,,,\n'
    'daily,,,O1,Reference,0.50033447812985754,19600,9.4524340575323684,0.90477098571540882,1.5002541817926254,1.3100329169571854,,20240415,23,,,,\n'
    'daily,,,O1,Reference,0.5508159633641978,19600,7.9265165264236384,0.84235189793815846,1.3327482592861135,1.2077964119558606,,20240416,24,,,,\n'
    'daily,,,O1,Reference,0.48871426028197629,19600,10.595225955133374,0.8714419614820742,1.5383378825141316,1.3169917781216565,,20240417,25,,,,\n'
    'daily,,,O1,Reference,0.47035726284639179,19600,10.89572673819557,0.94759280235790722,1.5716724527664632,1.3425770265119463,,20240418,26,,,,\n'
    'daily,,,O1,Reference,0.51355481778728951,19600,8.3866754169960771,0.85914946360094613,1.5187091270843371,1.319097455170799,,20240419,27,,,,\n'
    'daily,,,O1,Reference,0.53249632618388554,19600,6.6024116951129423,0.86686220553371651,1.5468308161270872,1.2780848279231334,,20240420,28,,,,\n'
    'daily,,,O1,PhysicsNew,0.42598285769256172,19600,10.190782291624716,0.9738388622683456,1.735240553212261,1.6170939432419051,,20240326,3,,,,\n'
    'daily,,,O1,PhysicsNew,0.32779609632421169,19600,18.239400091541221,1.0274100416633811,1.7859124732827389,1.680782859618783,,20240327,4,,,,\n'
    'daily,,,O1,PhysicsNew,0.37367832626837583,19600,15.675138031885981,0.89912475088172994,1.5843390853508488,1.7955684869681341,,20240328,5,,,,\n'
    'daily,,,O1,PhysicsNew,0.41650924441090731,19600,12.850518743680784,0.90971486119119083,1.7658490379109522,1.5182120968968598,,20240329,6,,,,\n'
    'daily,,,O1,PhysicsNew,0.50882625932793357,19600,11.139354063419052,0.80439716562977581,1.479552772553874,1.2178606737576474,,20240330,7,,,,\n'
    'daily,,,O1,PhysicsNew,0.55529056358248652,19600,7.0651717686645004,0.71256982151574455,1.4102547073034497,1.3120979935113048,,20240331,8,,,,\n'
    'daily,,,O1,PhysicsNew,0.37805075951494499,19600,8.6283169877660075,1.2365033964588812,1.9115001650448999,1.7430312009451421,,20240401,9,,,,\n'
    'daily,,,O1,PhysicsNew,0.54236119573684627,19600,7.0905676503378094,0.78970599502179772,1.4479541076406803,1.3106075427502375,,20240402,10,,,,\n'
    'daily,,,O1,PhysicsNew,0.50875966351554147,19600,6.6106901682822095,0.86592836793950545,1.6008034822925781,1.4344779760101405,,20240403,11,,,,\n'
    'daily,,,O1,PhysicsNew,0.55552469359590406,19600,6.4733889729673244,0.79031648266146459,1.4277928858385172,1.2665940186092268,,20240404,12,,,,\n'
    'daily,,,O1,PhysicsNew,0.48288802539842091,19600,7.313629889154023,0.95521566383139878,1.6982090034218993,1.4264531907856846,,20240405,13,,,,\n'
    'daily,,,O1,PhysicsNew,0.54555171567906768,19600,6.6390539518363649,0.7368358655718823,1.5584449049876816,1.2884324582397244,,20240406,14,,,,\n'
    'daily,,,O1,PhysicsNew,0.52519070579690619,19600,7.0632674700086664,0.77495567486547412,1.6303685918629225,1.328925546371335,,20240407,15,,,,\n'
    'daily,,,O1,PhysicsNew,0.55339055771068335,19600,6.8839687696018448,0.76601422649336559,1.4837272765263649,1.2315729572189,,20240408,16,,,,\n'
    'daily,,,O1,PhysicsNew,0.55636660902684143,19600,6.2560862991802964,0.79831699449190241,1.3615339087215568,1.321023492022519,,20240409,17,,,,\n'
    'daily,,,O1,PhysicsNew,0.51519823435003964,19600,7.1154942614548489,0.87117513056391394,1.5176886865942203,1.3986556254900708,,20240410,18,,,,\n'
    'daily,,,O1,PhysicsNew,0.50777837815935145,19600,6.4343912988229341,0.85556347030054003,1.6892677343530884,1.3986906422630117,,20240411,19,,,,\n'
    'daily,,,O1,PhysicsNew,0.42895893696607229,19600,9.2184934733813968,0.93426569822821237,1.866142781670419,1.6102239773031413,,20240412,20,,,,\n'
    'daily,,,O1,PhysicsNew,0.40796729429402312,19600,8.857370220038808,0.99998851473972739,1.8637480479988424,1.7506711446057428,,20240413,21,,,,\n'
    'daily,,,O1,PhysicsNew,0.47512262527992449,19600,9.0614496309581192,0.92337244337998203,1.6969969403138541,1.3805528869190316,,20240414,22,,,,\n'
    'daily,,,O1,PhysicsNew,0.44977399635089449,19600,9.9434526882951726,0.96733423312399802,1.7118707909036412,1.4635837447703666,,20240415,23,,,,\n'
    'daily,,,O1,PhysicsNew,0.51550476243492604,19600,8.4105134400240491,0.85931433240166455,1.4568706389315105,1.3488982562183476,,20240416,24,,,,\n'
    'daily,,,O1,PhysicsNew,0.47572333255798038,19600,10.858323167330164,0.85862654022392149,1.6068148596300855,1.3633795367305399,,20240417,25,,,,\n'
    'daily,,,O1,PhysicsNew,0.41978520663189611,19600,12.575979213070935,0.9990119624545688,1.6832827683753879,1.4875636355836503,,20240418,26,,,,\n'
    'daily,,,O1,PhysicsNew,0.47221615177692089,19600,9.9228340708349307,0.86966419311662302,1.6916670772644375,1.3923613695519546,,20240419,27,,,,\n'
    'daily,,,O1,PhysicsNew,0.49934568417466108,19600,7.6532535653418101,0.85611124993081822,1.6729816595970339,1.3809157329557742,,20240420,28,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.4276630232875589,19600,10.10752930133274,0.96982426054651005,1.7337977301666925,1.6151002067119786,,20240326,3,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.33546794612474617,19600,18.140057022006697,0.98955396273493845,1.7713182519588897,1.6745427445933954,,20240327,4,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.3763599660363699,19600,15.608394960731877,0.88242205960311659,1.5848624358756851,1.7951457426679485,,20240328,5,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.41523714514867838,19600,12.838852065932183,0.91730544092439681,1.7753179828952053,1.5147396136936295,,20240329,6,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.50928647084067569,19600,11.0992312721889,0.81540554065994819,1.4685768136090422,1.2147502436170159,,20240330,7,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.55406404213352367,19600,7.0922188183499992,0.72393957273639975,1.4029522823832736,1.3143137510293468,,20240331,8,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.3874295168994471,19600,8.542680489797192,1.183887375156611,1.9071056722060091,1.7283772003958864,,20240401,9,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.53884722202951418,19600,7.0968191969676413,0.81706137635346443,1.4476926434283273,1.3119877726875024,,20240402,10,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.51020134594866628,19600,6.545409245861479,0.86360100498120718,1.6072491942240039,1.4251403710700341,,20240403,11,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.55097665041932764,19600,6.5121390719718386,0.82622116809675428,1.42831866908141,1.2646780930111627,,20240404,12,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.4816599458806885,19600,7.3526156322516627,0.98515387788281539,1.6714018684764298,1.4241362320885174,,20240405,13,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.5424825355892493,19600,6.7049830061881837,0.75491528275702191,1.5529067097818094,1.2946765103515816,,20240406,14,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.52451664312544166,19600,7.1305550728655485,0.78855953714421367,1.6219849536255695,1.3218001341564334,,20240407,15,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.55259592289723658,19600,6.8110706617204197,0.77536485812006006,1.4916818893152497,1.2285554533648146,,20240408,16,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.55808067979086229,19600,6.2704399843442324,0.79411433170664569,1.3519841570395901,1.3172224832201829,,20240409,17,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.51662809411595378,19600,7.0738777637186159,0.88109097837534323,1.5016674397626038,1.3926091575969357,,20240410,18,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.51116568136126328,19600,6.3748277269411497,0.86346477071629624,1.6659141437534746,1.385529201834198,,20240411,19,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.43090480353233618,19600,9.1095562428946657,0.94754693187139438,1.848873372797214,1.6029816411313893,,20240412,20,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.40998399830602739,19600,8.7107506502384169,1.0174174927323334,1.8453942071102669,1.7425919939531158,,20240413,21,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.4772279827914595,19600,8.9202376001098571,0.94717439741493759,1.667046690610029,1.373887382851819,,20240414,22,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.44908122933484068,19600,9.8567395476556108,1.0031897248391219,1.6913505439999423,1.4562610175614099,,20240415,23,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.51545769189097845,19600,8.3836609280484584,0.87997883462911075,1.4459620863631113,1.3391180280988646,,20240416,24,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.47419364178073881,19600,10.819340318688626,0.86028312113638017,1.6187464147535524,1.3683850539552778,,20240417,25,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.4255508345694049,19600,12.465506912281056,0.9706928854245398,1.677827861528566,1.4813914365715479,,20240418,26,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.47190165875510098,19600,9.8370828252314286,0.88572931670517829,1.6782817100958856,1.3962430104400312,,20240419,27,,,,\n'
    'daily,,,O1,PhysicsNewBias,0.49908335586116598,19600,7.6056755558662186,0.87316641327052757,1.6668352009764422,1.3745747919114348,,20240420,28,,,,\n'
    'daily,,,O1,Placebo,0.42370701346842021,19600,10.476014120543493,0.97709826157027357,1.7422071953783946,1.6038919461286845,,20240326,3,,,,\n'
    'daily,,,O1,Placebo,0.34828354017575203,19600,17.648694364242441,0.97362593787094598,1.7542858848626728,1.6373558299797173,,20240327,4,,,,\n'
    'daily,,,O1,Placebo,0.37070122200011191,19600,16.112237320712513,0.88817603084513197,1.6167396156285845,1.7702533817322068,,20240328,5,,,,\n'
    'daily,,,O1,Placebo,0.41150417142068529,19600,12.883799830790574,0.91531578625046239,1.7705697523249744,1.5479246093830106,,20240329,6,,,,\n'
    'daily,,,O1,Placebo,0.51090110185213811,19600,10.874032725175701,0.80143480228741548,1.4723340970722174,1.2309983524137635,,20240330,7,,,,\n'
    'daily,,,O1,Placebo,0.54733351486360671,19600,7.2476706443927146,0.70799035412943057,1.447602614789032,1.3390301933108923,,20240331,8,,,,\n'
    'daily,,,O1,Placebo,0.37596967223523381,19600,8.7077633310948777,1.2114083089693732,1.9531308079429544,1.7467790053278442,,20240401,9,,,,\n'
    'daily,,,O1,Placebo,0.53680843397964084,19600,7.4849619001819843,0.76019958951348665,1.4849000626032285,1.3255307411222537,,20240402,10,,,,\n'
    'daily,,,O1,Placebo,0.497792448729723,19600,6.9902015664764585,0.87866726099427717,1.6298285753529351,1.4593111774511016,,20240403,11,,,,\n'
    'daily,,,O1,Placebo,0.55130259248808544,19600,6.8601637934716466,0.77533974297431707,1.4518532483546442,1.266391924882037,,20240404,12,,,,\n'
    'daily,,,O1,Placebo,0.48681924698861812,19600,7.6011591374455474,0.91081625998194282,1.6971401953923539,1.4156711870901997,,20240405,13,,,,\n'
    'daily,,,O1,Placebo,0.53780934417075255,19600,6.7044172518161993,0.73982262099556539,1.5779615213508158,1.3294683934511511,,20240406,14,,,,\n'
    'daily,,,O1,Placebo,0.5258776274188155,19600,7.0770560580320714,0.77411949530775559,1.6298140196793147,1.3233320418341521,,20240407,15,,,,\n'
    'daily,,,O1,Placebo,0.55033222570278473,19600,6.901767219114185,0.75722151991363307,1.4949407116306757,1.2557915919733778,,20240408,16,,,,\n'
    'daily,,,O1,Placebo,0.54759498926377204,19600,6.5928851552521381,0.81633186595310292,1.3970276011437843,1.3206621645298713,,20240409,17,,,,\n'
    'daily,,,O1,Placebo,0.51342767434331749,19600,7.1762065570973412,0.86139966072925989,1.5492425068279545,1.3937232125326615,,20240410,18,,,,\n'
    'daily,,,O1,Placebo,0.50730290994816363,19600,6.595428186008002,0.827154970392872,1.7237092028356371,1.3912688803196354,,20240411,19,,,,\n'
    'daily,,,O1,Placebo,0.42861917852149117,19600,8.6172990347464768,0.94374793198826656,1.8999063007650856,1.626641662847558,,20240412,20,,,,\n'
    'daily,,,O1,Placebo,0.41929865472123179,19600,8.0326935876597165,0.95947641435493236,1.9074451828947607,1.7308188786210528,,20240413,21,,,,\n'
    'daily,,,O1,Placebo,0.47091847216724941,19600,8.6665567145078946,0.91367389624888595,1.7561690257849865,1.4118363064950357,,20240414,22,,,,\n'
    'daily,,,O1,Placebo,0.44562857445536991,19600,10.151191920953492,0.95784276081007402,1.7274831812650882,1.4785210376109963,,20240415,23,,,,\n'
    'daily,,,O1,Placebo,0.52184725521917374,19600,8.3908633756783271,0.83945079367376119,1.4656560008882646,1.3107829197470318,,20240416,24,,,,\n'
    'daily,,,O1,Placebo,0.48684082235338838,19600,10.184401258592832,0.82599214364200091,1.633684622754588,1.3380596258509423,,20240417,25,,,,\n'
    'daily,,,O1,Placebo,0.42696398598636581,19600,11.60965963072872,0.98989382537633563,1.7269688853607994,1.4823210487652068,,20240418,26,,,,\n'
    'daily,,,O1,Placebo,0.4841773690586012,19600,9.1784151545934005,0.83000758054546786,1.7120606685597743,1.3779467638022149,,20240419,27,,,,\n'
    'daily,,,O1,Placebo,0.50196761326888484,19600,7.5245751253672601,0.86664357354079513,1.6811234570400899,1.3526198651095676,,20240420,28,,,,\n'
    'daily,,,O1,Recent,0.42516689747085129,19600,10.231067582465332,0.97143178541398678,1.7359945597259716,1.6224366294838253,,20240326,3,,,,\n'
    'daily,,,O1,Recent,0.32804947014809421,19600,18.199792266886597,1.0271885167268835,1.7955851292340861,1.6746101563730189,,20240327,4,,,,\n'
    'daily,,,O1,Recent,0.37048835364412208,19600,15.633460100347577,0.89592807062607438,1.6136988328056421,1.805977829053552,,20240328,5,,,,\n'
    'daily,,,O1,Recent,0.41654288303831438,19600,12.804317720137851,0.90141605102932365,1.7764424054056052,1.5218640722617265,,20240329,6,,,,\n'
    'daily,,,O1,Recent,0.50609111427009212,19600,11.17939051446287,0.80683238249101974,1.4915246365318924,1.225608949049156,,20240330,7,,,,\n'
    'daily,,,O1,Recent,0.55353810245877122,19600,7.0839261794942345,0.70936198431344899,1.4187195760207654,1.3218308746676712,,20240331,8,,,,\n'
    'daily,,,O1,Recent,0.3764324533081348,19600,8.5350426989715764,1.2505696110846412,1.9238035659390345,1.740411743969096,,20240401,9,,,,\n'
    'daily,,,O1,Recent,0.54291902136393311,19600,7.0960008467543423,0.78885806463879893,1.4448833220452857,1.3087534983218232,,20240402,10,,,,\n'
    'daily,,,O1,Recent,0.50989984891191864,19600,6.5749100397948697,0.86404095550332016,1.6001638572226553,1.4303089123438315,,20240403,11,,,,\n'
    'daily,,,O1,Recent,0.55811132734362146,19600,6.4233529659234971,0.79121185616532164,1.4193934992116637,1.2547570122398244,,20240404,12,,,,\n'
    'daily,,,O1,Recent,0.47953512621464378,19600,7.3380414667704397,0.95998046335597642,1.7060184665090274,1.4415365578521389,,20240405,13,,,,\n'
    'daily,,,O1,Recent,0.54650254095610507,19600,6.5866957548400871,0.73453124969291117,1.5474545982013648,1.2957009839138207,,20240406,14,,,,\n'
    'daily,,,O1,Recent,0.52746595885357495,19600,7.0644430399238889,0.77264251506626203,1.6199564297206186,1.3202856290693423,,20240407,15,,,,\n'
    'daily,,,O1,Recent,0.55636527080088771,19600,6.8165711860891687,0.76404037577245731,1.4733658682611912,1.2223545558660118,,20240408,16,,,,\n'
    'daily,,,O1,Recent,0.55829991473738561,19600,6.1838337797705165,0.79770832066058528,1.3641153156061685,1.3094300161745809,,20240409,17,,,,\n'
    'daily,,,O1,Recent,0.51885682505239228,19600,7.0360239532367848,0.86609873310480134,1.5130669361274716,1.3833816368508509,,20240410,18,,,,\n'
    'daily,,,O1,Recent,0.50733609613237762,19600,6.4292490692940687,0.85310927236738443,1.7019076472539745,1.3954332411275112,,20240411,19,,,,\n'
    'daily,,,O1,Recent,0.42772453221279161,19600,9.1504051676940907,0.92883824956485161,1.8772635475452584,1.6229526593947461,,20240412,20,,,,\n'
    'daily,,,O1,Recent,0.40414787787235301,19600,8.9049236962135634,1.0034637745635631,1.8785869119090304,1.7635681384216717,,20240413,21,,,,\n'
    'daily,,,O1,Recent,0.4739039093165478,19600,9.0831942220628896,0.92383395747221675,1.7078654178308743,1.3799995026027361,,20240414,22,,,,\n'
    'daily,,,O1,Recent,0.44963452133051668,19600,10.027043488019173,0.9650078984881556,1.7192547818908226,1.454461951547156,,20240415,23,,,,\n'
    'daily,,,O1,Recent,0.51660203807131255,19600,8.4276468628042647,0.85379774866565483,1.464254186950295,1.3381449765397979,,20240416,24,,,,\n'
    'daily,,,O1,Recent,0.48099209014896083,19600,10.744477434912833,0.84436717942034489,1.6024431176056579,1.3465773932772924,,20240417,25,,,,\n'
    'daily,,,O1,Recent,0.42344403103456468,19600,12.299697649144283,0.9902203911537284,1.6903611724190499,1.483122329486535,,20240418,26,,,,\n'
    'daily,,,O1,Recent,0.47453637589077807,19600,9.7698146967186048,0.86660387044375131,1.6889797979976839,1.3907715752479961,,20240419,27,,,,\n'
    'daily,,,O1,Recent,0.49754220638677288,19600,7.6384391028063936,0.86083557425783086,1.6816304304305665,1.3856763898293596,,20240420,28,,,,\n'
    'daily,,,O1,RecentBias,0.42667452885270041,19600,10.143978139073004,0.96790335835030938,1.7345305549127481,1.6217245006727254,,20240326,3,,,,\n'
    'daily,,,O1,RecentBias,0.33539540835837423,19600,18.114150281130502,0.98937129530249757,1.7810445299650848,1.6698825937046604,,20240327,4,,,,\n'
    'daily,,,O1,RecentBias,0.3729255888441208,19600,15.580458760390874,0.88117800403844126,1.6124494491402299,1.8058730750420389,,20240328,5,,,,\n'
    'daily,,,O1,RecentBias,0.41510485868041758,19600,12.808855629186001,0.91040499633716843,1.7843607306866272,1.5182386040237039,,20240329,6,,,,\n'
    'daily,,,O1,RecentBias,0.50585610693804406,19600,11.1522695583049,0.81936001379557832,1.4835531827003334,1.2233553738972631,,20240330,7,,,,\n'
    'daily,,,O1,RecentBias,0.55197404296510366,19600,7.1110556522219461,0.72232272182933499,1.4125849654472655,1.3243396916930124,,20240331,8,,,,\n'
    'daily,,,O1,RecentBias,0.38564638415569669,19600,8.4599225356919003,1.1984194220031867,1.9181045989005789,1.7268145738046274,,20240401,9,,,,\n'
    'daily,,,O1,RecentBias,0.53891828673580022,19600,7.1102069049588321,0.81840969153555676,1.4444854724322895,1.3114298387058203,,20240402,10,,,,\n'
    'daily,,,O1,RecentBias,0.51122180959216967,19600,6.5132199194577476,0.86325076731730943,1.6036707359680324,1.422412865836959,,20240403,11,,,,\n'
    'daily,,,O1,RecentBias,0.55303044124929179,19600,6.4645783699203641,0.8301824546524682,1.4194785301125774,1.2543312413491423,,20240404,12,,,,\n'
    'daily,,,O1,RecentBias,0.4777947589799787,19600,7.3736151012461022,0.99155421517494702,1.6835714638806429,1.4387256169991891,,20240405,13,,,,\n'
    'daily,,,O1,RecentBias,0.54307468363691214,19600,6.6525581327028656,0.75480973547682106,1.5432311468011433,1.3016976579800024,,20240406,14,,,,\n'
    'daily,,,O1,RecentBias,0.52643202491943808,19600,7.1371833905392581,0.78690307338596377,1.6125192674853557,1.3143146811327833,,20240407,15,,,,\n'
    'daily,,,O1,RecentBias,0.55544265146058092,19600,6.7541508961960304,0.7746571179238716,1.4792237023497099,1.2198981719833932,,20240408,16,,,,\n'
    'daily,,,O1,RecentBias,0.55982046870467439,19600,6.2070662398857213,0.79361793858884433,1.3550006394603027,1.3060613549155899,,20240409,17,,,,\n'
    'daily,,,O1,RecentBias,0.51969608314690852,19600,7.0050992736720046,0.87795737417984698,1.4981532499093353,1.378559414511507,,20240410,18,,,,\n'
    'daily,,,O1,RecentBias,0.51035930519694461,19600,6.3712032964422773,0.86292223108054433,1.6795669760784031,1.3824650702580452,,20240411,19,,,,\n'
    'daily,,,O1,RecentBias,0.4294400089785792,19600,9.0480856021722005,0.94296993673435203,1.8611714981998475,1.6153118029433191,,20240412,20,,,,\n'
    'daily,,,O1,RecentBias,0.40603561099245122,19600,8.7653818349897819,1.0200506754634011,1.8625888139498401,1.754997866728657,,20240413,21,,,,\n'
    'daily,,,O1,RecentBias,0.47541299998501391,19600,8.944275238743705,0.94957511255623905,1.6819589662930143,1.3730113092051004,,20240414,22,,,,\n'
    'daily,,,O1,RecentBias,0.44840618755152117,19600,9.9321966321590693,1.0036618310003358,1.7001128910867911,1.448377543309034,,20240415,23,,,,\n'
    'daily,,,O1,RecentBias,0.51623395175036335,19600,8.4137326243760047,0.87529212123417932,1.4538715687548716,1.3287313798589016,,20240416,24,,,,\n'
    'daily,,,O1,RecentBias,0.479398158524734,19600,10.728414926076905,0.84674916553238599,1.6118334754994756,1.3514805565853747,,20240417,25,,,,\n'
    'daily,,,O1,RecentBias,0.42905199404117661,19600,12.209033988034356,0.96226776557973659,1.6841356994627472,1.4768664243552023,,20240418,26,,,,\n'
    'daily,,,O1,RecentBias,0.47367631513786829,19600,9.7031239654842274,0.88421090463687124,1.6771324566575594,1.3948821279377168,,20240419,27,,,,\n'
    'daily,,,O1,RecentBias,0.49717723637390421,19600,7.6047342032297047,0.87968396935457782,1.673342809933003,1.3789072151841359,,20240420,28,,,,\n'
    'daily,,,O1,Drift,0.4229947687486888,19600,10.16755459329311,0.97548877215662955,1.7520783813613958,1.6291622772442389,,20240326,3,,,,\n'
    'daily,,,O1,Drift,0.31793935504075421,19600,18.290739571830741,1.0387310874698454,1.8586373854966445,1.6907203321985818,,20240327,4,,,,\n'
    'daily,,,O1,Drift,0.37011351728795783,19600,15.733630686212431,0.89536686669678867,1.6347892108020694,1.7849860645927431,,20240328,5,,,,\n'
    'daily,,,O1,Drift,0.413041998930467,19600,12.91202936172628,0.90569681238894517,1.7849714932032621,1.5312542573660786,,20240329,6,,,,\n'
    'daily,,,O1,Drift,0.4972516198646561,19600,11.271321382315802,0.81886641339702282,1.5326440791796951,1.2476311706583596,,20240330,7,,,,\n'
    'daily,,,O1,Drift,0.54667389843093661,19600,7.115193960519302,0.71731772539610605,1.4549235155967746,1.3403072460151786,,20240331,8,,,,\n'
    'daily,,,O1,Drift,0.37134648621648769,19600,8.7452223189633447,1.2401493383956426,1.9424604706486155,1.7615672846127823,,20240401,9,,,,\n'
    'daily,,,O1,Drift,0.53712892758473862,19600,7.1419217078227586,0.79802185552041516,1.4871044570264671,1.3111419184573569,,20240402,10,,,,\n'
    'daily,,,O1,Drift,0.50651325562094218,19600,6.5987646905601753,0.87165518219824378,1.6147130088881856,1.4375845296544394,,20240403,11,,,,\n'
    'daily,,,O1,Drift,0.56192731621700664,19600,6.3963181424341924,0.78061692715944164,1.3981938543694812,1.25235566944048,,20240404,12,,,,\n'
    'daily,,,O1,Drift,0.46954793844943421,19600,7.4230741032101104,0.96881163946018378,1.7543341136918027,1.4713373797131295,,20240405,13,,,,\n'
    'daily,,,O1,Drift,0.54523041450245946,19600,6.6042411276036246,0.74064598086636302,1.5478065099825993,1.298424014658291,,20240406,14,,,,\n'
    'daily,,,O1,Drift,0.52418169730939568,19600,7.0958715550625957,0.78890342089833254,1.6248694578548502,1.3248556374956293,,20240407,15,,,,\n'
    'daily,,,O1,Drift,0.56533821577433696,19600,6.657002711421967,0.76062847217257201,1.4163921394754275,1.2084310045601447,,20240408,16,,,,\n'
    'daily,,,O1,Drift,0.55229935284634146,19600,6.2693247874789551,0.81713120486458413,1.3815910978522352,1.3192410009715838,,20240409,17,,,,\n'
    'daily,,,O1,Drift,0.50598994285820287,19600,7.1859423331250172,0.89302875568907858,1.566238364812734,1.4098067741356906,,20240410,18,,,,\n'
    'daily,,,O1,Drift,0.49470991969825001,19600,6.5541711231206472,0.86022622250214087,1.7715675948135816,1.4290786948522911,,20240411,19,,,,\n'
    'daily,,,O1,Drift,0.40285464603263732,19600,9.3366409448741443,0.97428728621971084,1.9664099086699476,1.6998502254454291,,20240412,20,,,,\n'
    'daily,,,O1,Drift,0.38640233439115379,19600,8.9364955625422393,1.032582644392845,1.9733879476152241,1.8058049157462903,,20240413,21,,,,\n'
    'daily,,,O1,Drift,0.45536854853212277,19600,9.142375947175438,0.92645200295298158,1.8333365088888876,1.4294350145860659,,20240414,22,,,,\n'
    'daily,,,O1,Drift,0.42678212314959119,19600,10.203957541918571,0.98524354935891723,1.8409619852866657,1.5153807254957503,,20240415,23,,,,\n'
    'daily,,,O1,Drift,0.50113648295285118,19600,8.5409024295571569,0.87222195115382273,1.5326675467388711,1.3860027580572103,,20240416,24,,,,\n'
    'daily,,,O1,Drift,0.48533643933812831,19600,10.71098992565226,0.84609172368871999,1.5584319742643136,1.3456271741787502,,20240417,25,,,,\n'
    'daily,,,O1,Drift,0.40250247157511332,19600,12.549408629316501,1.019864695392229,1.7897099113902593,1.5300134436713089,,20240418,26,,,,\n'
    'daily,,,O1,Drift,0.45361922632584872,19600,10.02269957272628,0.89916339305966453,1.7884393527180351,1.4341643504490684,,20240419,27,,,,\n'
    'daily,,,O1,Drift,0.48251555632384341,19600,7.7647746927095636,0.86902865577723576,1.7477673470511841,1.4409093377962776,,20240420,28,,,,\n'
    'daily,,,O1,DriftBias,0.42470041456195728,19600,10.097244670209779,0.97274903379186395,1.7481066092248578,1.6265733281383561,,20240326,3,,,,\n'
    'daily,,,O1,DriftBias,0.32593993568994878,19600,18.196773253632848,0.99902631260221242,1.8417294819123609,1.6849651943249069,,20240327,4,,,,\n'
    'daily,,,O1,DriftBias,0.37295288204901822,19600,15.676636505891372,0.87802560833625343,1.6329939582480284,1.7848969384114619,,20240328,5,,,,\n'
    'daily,,,O1,DriftBias,0.41168723529387569,19600,12.907617727303579,0.91462681521038125,1.7929021890070846,1.5277201798774991,,20240329,6,,,,\n'
    'daily,,,O1,DriftBias,0.4978427253976811,19600,11.238265137762257,0.83090773258038897,1.518835605746925,1.2440102465012577,,20240330,7,,,,\n'
    'daily,,,O1,DriftBias,0.54545890108459361,19600,7.1514691975236557,0.73015879190029687,1.4454304000535301,1.341887826765914,,20240331,8,,,,\n'
    'daily,,,O1,DriftBias,0.3811537283009252,19600,8.6690468388317221,1.1854597516240055,1.9358170866333355,1.7464058594725631,,20240401,9,,,,\n'
    'daily,,,O1,DriftBias,0.5337782452659966,19600,7.1541140072794693,0.82492658474865743,1.4846112154069817,1.3128651184064062,,20240402,10,,,,\n'
    'daily,,,O1,DriftBias,0.5081047891891487,19600,6.5473867591302071,0.86888593432255357,1.6190507117892494,1.4279413271081889,,20240403,11,,,,\n'
    'daily,,,O1,DriftBias,0.557319928046074,19600,6.4432521494886679,0.81723974422537937,1.3977636865741545,1.2502810000505296,,20240404,12,,,,\n'
    'daily,,,O1,DriftBias,0.46843595799990678,19600,7.4636578206552606,1.0004518520439412,1.7247528608736999,1.4683617957147757,,20240405,13,,,,\n'
    'daily,,,O1,DriftBias,0.54258490006409765,19600,6.6732253156538306,0.75816825332235127,1.5395652112763334,1.30354946638778,,20240406,14,,,,\n'
    'daily,,,O1,DriftBias,0.52397718979721919,19600,7.1552986950261408,0.80176751342861596,1.6142969647751475,1.3169106560317492,,20240407,15,,,,\n'
    'daily,,,O1,DriftBias,0.56483297539161481,19600,6.6031670648765886,0.76948106728851295,1.4217636106694158,1.2039451396923253,,20240408,16,,,,\n'
    'daily,,,O1,DriftBias,0.55463322788803382,19600,6.2870119507978437,0.81084619766219201,1.3685798861868861,1.3147936407242469,,20240409,17,,,,\n'
    'daily,,,O1,DriftBias,0.50765577801931361,19600,7.1593217860832361,0.90179258050301725,1.5480615968823823,1.403400449162459,,20240410,18,,,,\n'
    'daily,,,O1,DriftBias,0.49805141871614012,19600,6.519531569562119,0.86920736221279515,1.7455869020411321,1.415204704612991,,20240411,19,,,,\n'
    'daily,,,O1,DriftBias,0.40485520995567031,19600,9.2470516444794093,0.98713230323705481,1.9472935016610939,1.6924394161396576,,20240412,20,,,,\n'
    'daily,,,O1,DriftBias,0.38844696066562379,19600,8.8160474311582462,1.0487720994866494,1.9538973993929925,1.7974774115791541,,20240413,21,,,,\n'
    'daily,,,O1,DriftBias,0.45733821902571747,19600,9.0264425443195009,0.95075403239349798,1.8019422955222333,1.4224419416066714,,20240414,22,,,,\n'
    'daily,,,O1,DriftBias,0.42617519341852961,19600,10.130919459786266,1.0208039886253149,1.8183657617254545,1.5081323057976317,,20240415,23,,,,\n'
    'daily,,,O1,DriftBias,0.50136717855676227,19600,8.5258367167322824,0.89115351527822428,1.5196635984386888,1.376318215256604,,20240416,24,,,,\n'
    'daily,,,O1,DriftBias,0.48342227236337881,19600,10.691781807621004,0.8487596058358543,1.5708959960749511,1.3507779833730913,,20240417,25,,,,\n'
    'daily,,,O1,DriftBias,0.40844218983247149,19600,12.458011521467116,0.99053033502279897,1.7817743128761443,1.5237776527067186,,20240418,26,,,,\n'
    'daily,,,O1,DriftBias,0.45345994777667359,19600,9.9550348867417515,0.91510166618178856,1.7727196475417106,1.4372092288730156,,20240419,27,,,,\n'
    'daily,,,O1,DriftBias,0.48238916735246179,19600,7.7313528254920119,0.88626521076308973,1.7388471497958442,1.4342471306094589,,20240420,28,,,,\n'
    'daily,,,O1,MultiScale,0.42735716776535571,19600,10.125768176345682,0.97054259206487381,1.7348369495465368,1.614616470010205,,20240326,3,,,,\n'
    'daily,,,O1,MultiScale,0.33505726996214302,19600,18.137145958732901,0.9899830342521988,1.7739767984567647,1.6757052192361941,,20240327,4,,,,\n'
    'daily,,,O1,MultiScale,0.37623466930102822,19600,15.600155631084174,0.88269191140764547,1.5850491313559201,1.7964519342101408,,20240328,5,,,,\n'
    'daily,,,O1,MultiScale,0.41515207093741308,19600,12.825624099384886,0.91801226208329123,1.7756259696577792,1.5155843146709622,,20240329,6,,,,\n'
    'daily,,,O1,MultiScale,0.50913248125500066,19600,11.089307916677068,0.81580840839405322,1.4708751227448669,1.2146538731234595,,20240330,7,,,,\n'
    'daily,,,O1,MultiScale,0.55391873857860952,19600,7.0861229084045885,0.72371211735393692,1.404448659318263,1.3150921677395209,,20240331,8,,,,\n'
    'daily,,,O1,MultiScale,0.38683218937518399,19600,8.5407067099612686,1.1847261078224618,1.9106564770746619,1.7299113759060831,,20240401,9,,,,\n'
    'daily,,,O1,MultiScale,0.53858510171763141,19600,7.1054498594762379,0.81704648029006166,1.4484150779820322,1.3129179270425542,,20240402,10,,,,\n'
    'daily,,,O1,MultiScale,0.50986748372977353,19600,6.5580580792682124,0.86402610143021019,1.6082188198645886,1.4257013405674706,,20240403,11,,,,\n'
    'daily,,,O1,MultiScale,0.55094111621424302,19600,6.5121863214221802,0.82583848229257284,1.4286184706894007,1.2651240235490633,,20240404,12,,,,\n'
    'daily,,,O1,MultiScale,0.48134483686676433,19600,7.3419151019627229,0.98531233391083817,1.6752816620681561,1.4244633021462276,,20240405,13,,,,\n'
    'daily,,,O1,MultiScale,0.54249327426381444,19600,6.698150172766888,0.75499506298351782,1.5543562096064016,1.2939385544344164,,20240406,14,,,,\n'
    'daily,,,O1,MultiScale,0.52415261884828435,19600,7.1252447690261747,0.78919653000529866,1.6241757023110062,1.3229250926696083,,20240407,15,,,,\n'
    'daily,,,O1,MultiScale,0.55271280979399107,19600,6.8086485296780292,0.77532727633494647,1.491734204720534,1.2277759890976925,,20240408,16,,,,\n'
    'daily,,,O1,MultiScale,0.55773932346943134,19600,6.2610556232257304,0.79435942911901591,1.3541349228177411,1.3189208448929839,,20240409,17,,,,\n'
    'daily,,,O1,MultiScale,0.51603634270576304,19600,7.073983400872379,0.88230236160137332,1.5041989391089829,1.3943436081614349,,20240410,18,,,,\n'
    'daily,,,O1,MultiScale,0.51047401337175302,19600,6.3732862635758218,0.86394558771040009,1.6693754369241254,1.3882471561459999,,20240411,19,,,,\n'
    'daily,,,O1,MultiScale,0.43048945699258689,19600,9.1143302841856233,0.94805889690160805,1.8498174084196839,1.6048034198103225,,20240412,20,,,,\n'
    'daily,,,O1,MultiScale,0.40963289925369079,19600,8.7146847166311634,1.0179172486298695,1.8463107329942745,1.7439802156004871,,20240413,21,,,,\n'
    'daily,,,O1,MultiScale,0.47664074385576682,19600,8.9287958650531856,0.94769838714163557,1.6692358925334581,1.375850066467863,,20240414,22,,,,\n'
    'daily,,,O1,MultiScale,0.44858506753197303,19600,9.8660212373000178,1.0033055220049711,1.6936235657797192,1.4577508889160464,,20240415,23,,,,\n'
    'daily,,,O1,MultiScale,0.51497306117005803,19600,8.3766951481806817,0.88071586657865175,1.4474049046177755,1.3418683775147398,,20240416,24,,,,\n'
    'daily,,,O1,MultiScale,0.47446817876899572,19600,10.80409518893541,0.86029570150437362,1.6173162791306888,1.3684498115204258,,20240417,25,,,,\n'
    'daily,,,O1,MultiScale,0.42516931719939732,19600,12.45935299175809,0.97224653415209961,1.6786617797818679,1.4828631739605982,,20240418,26,,,,\n'
    'daily,,,O1,MultiScale,0.47158542455063313,19600,9.8375725284078595,0.88637591042456743,1.6807089980286425,1.3962973305766715,,20240419,27,,,,\n'
    'daily,,,O1,MultiScale,0.498598500985176,19600,7.6032699346962902,0.87400906607078388,1.6684094091978501,1.3767424220828377,,20240420,28,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.4288084342955083,19600,10.111027970622729,0.97099476928915518,1.7269488522034258,1.609404107608323,,20240326,3,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.33586824153159311,19600,18.21096722723729,0.99570787699834962,1.7592484311163938,1.6685525031451922,,20240327,4,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.37789924091688598,19600,15.590003283457833,0.88450632605231483,1.5684286521872746,1.7945119414715542,,20240328,5,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.41677725221352341,19600,12.838898538167085,0.91618993287345918,1.7655430810098556,1.5106239211294856,,20240329,6,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.51060070753790576,19600,11.092318797045708,0.81597129756420628,1.4559758577852406,1.2135973518412433,,20240330,7,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.55443872858959353,19600,7.0959857542565965,0.72654162462313798,1.3932115139162031,1.3158390289687023,,20240331,8,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.39026542676704018,19600,8.5348119419838007,1.1850049562712364,1.8926195272055599,1.7155084162238252,,20240401,9,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.54027873828538298,19600,7.1108183532511449,0.81892236382654582,1.438561055505571,1.3041199494922715,,20240402,10,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.5108178567968007,19600,6.5302026850321226,0.86583762563269784,1.6059662235794845,1.4199834343666886,,20240403,11,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.5508739254462095,19600,6.5228047737706278,0.82951132139426853,1.4299820081665895,1.2600352342110486,,20240404,12,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.48477865042801388,19600,7.3594069828179993,0.98542914691985439,1.6502059481274836,1.4137819641780545,,20240405,13,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.54221479328496791,19600,6.7123907321524783,0.75794776995559365,1.5492913536652064,1.2960558179449018,,20240406,14,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.5270791074062422,19600,7.1452000707466023,0.78658032033239611,1.6073820859656025,1.312582836458186,,20240407,15,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.55217543869529662,19600,6.7904891995415921,0.77865648798312503,1.4949367220617944,1.2278959283377122,,20240408,16,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.55927976800295576,19600,6.2528547775038188,0.79426540828832148,1.3470901677288891,1.3123201071210422,,20240409,17,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.51801547744510035,19600,7.0581552786596298,0.88233618886418175,1.491310898416526,1.3891340013334463,,20240410,18,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.51354919563174106,19600,6.3812580864347614,0.86637832298476847,1.6450803743150524,1.378394531773792,,20240411,19,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.43614124748052041,19600,9.0836859898401396,0.94691604283450481,1.817102463451592,1.586792872984687,,20240412,20,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.4135175904514492,19600,8.6838011163300575,1.0140837415399691,1.827011463305416,1.7330056138364287,,20240413,21,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.47979286608606347,19600,8.9087417949020171,0.947643549120048,1.6500404462457261,1.3662006363979844,,20240414,22,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.45240233055167928,19600,9.8735956128134408,1.0044960675075052,1.6682348044042092,1.4438426797343047,,20240415,23,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.51766655476784684,19600,8.3749380124555461,0.88143946284675734,1.4343846856736904,1.3289410871034912,,20240416,24,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.47317378855027847,19600,10.825656071139305,0.86713010590826711,1.6259576768193018,1.3638377973000979,,20240417,25,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.4281832100389229,19600,12.451378938360175,0.97198646325709404,1.6610099381941468,1.4723775866112263,,20240418,26,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.4733438781062036,19600,9.8352687856689656,0.88448819840130888,1.6647576252228751,1.3961479102045284,,20240419,27,,,,\n'
    'daily,,,O1,AdaptiveFlow,0.50067441356510645,19600,7.609187798752413,0.87708622838845463,1.6571526629340807,1.3645863935809741,,20240420,28,,,,\n'
    'daily,,,O1,WindFlow,0.42898455316152961,19600,10.107528162804885,0.96982489456961496,1.7269488522034258,1.609404107608323,,20240326,3,,,,\n'
    'daily,,,O1,WindFlow,0.33731226403905812,19600,18.14005079668193,0.98955419888062846,1.7592484311163938,1.6685525031451922,,20240327,4,,,,\n'
    'daily,,,O1,WindFlow,0.377970323778405,19600,15.608386890696693,0.88242204960517467,1.5684286521872746,1.7945119414715542,,20240328,5,,,,\n'
    'daily,,,O1,WindFlow,0.41664267523784221,19600,12.838850789701189,0.91730553476630161,1.7655430810098556,1.5106239211294856,,20240329,6,,,,\n'
    'daily,,,O1,WindFlow,0.51060103768942178,19600,11.099230899841301,0.81540568086790643,1.4559758577852406,1.2135973518412433,,20240330,7,,,,\n'
    'daily,,,O1,WindFlow,0.55479077714158531,19600,7.0922183128199459,0.72394022478612841,1.3932115139162031,1.3158390289687023,,20240331,8,,,,\n'
    'daily,,,O1,WindFlow,0.39032307015664708,19600,8.5426775109258699,1.1838882405462592,1.8926195272055599,1.7155084162238252,,20240401,9,,,,\n'
    'daily,,,O1,WindFlow,0.54064204717706787,19600,7.0968232102946169,0.81706084083639474,1.438561055505571,1.3041199494922715,,20240402,10,,,,\n'
    'daily,,,O1,WindFlow,0.51093864793147159,19600,6.5454114418792333,0.86360126407596305,1.6059662235794845,1.4199834343666886,,20240403,11,,,,\n'
    'daily,,,O1,WindFlow,0.55137718739935127,19600,6.5121431305960602,0.82622220949513492,1.4299820081665895,1.2600352342110486,,20240404,12,,,,\n'
    'daily,,,O1,WindFlow,0.4848788443726591,19600,7.3526145339553359,0.985154610104925,1.6502059481274836,1.4137819641780545,,20240405,13,,,,\n'
    'daily,,,O1,WindFlow,0.54265485807238578,19600,6.7049880184908321,0.7549153251987315,1.5492913536652064,1.2960558179449018,,20240406,14,,,,\n'
    'daily,,,O1,WindFlow,0.52698384389102371,19600,7.1305587018936238,0.78855959790112096,1.6073820859656025,1.312582836458186,,20240407,15,,,,\n'
    'daily,,,O1,WindFlow,0.55237093686790295,19600,6.811074558930132,0.7753651411185577,1.4949367220617944,1.2278959283377122,,20240408,16,,,,\n'
    'daily,,,O1,WindFlow,0.55912466228325197,19600,6.2704421466589535,0.79411447238503174,1.3470901677288891,1.3123201071210422,,20240409,17,,,,\n'
    'daily,,,O1,WindFlow,0.51801126369421358,19600,7.073874340933564,0.88109091077460011,1.491310898416526,1.3891340013334463,,20240410,18,,,,\n'
    'daily,,,O1,WindFlow,0.51396527105068013,19600,6.3748262924688772,0.86346498243601322,1.6450803743150524,1.378394531773792,,20240411,19,,,,\n'
    'daily,,,O1,WindFlow,0.43580978684322291,19600,9.1095520597368758,0.94754776089742643,1.817102463451592,1.586792872984687,,20240412,20,,,,\n'
    'daily,,,O1,WindFlow,0.41284832074889,19600,8.7107485075020801,1.0174179749729844,1.827011463305416,1.7330056138364287,,20240413,21,,,,\n'
    'daily,,,O1,WindFlow,0.4797365239635647,19600,8.9202201777242713,0.94717423421846836,1.6500404462457261,1.3662006363979844,,20240414,22,,,,\n'
    'daily,,,O1,WindFlow,0.45272643114253142,19600,9.8567337336002101,1.0031918900654888,1.6682348044042092,1.4438426797343047,,20240415,23,,,,\n'
    'daily,,,O1,WindFlow,0.51775746841484627,19600,8.3836565416902662,0.8799784470755545,1.4343846856736904,1.3289410871034912,,20240416,24,,,,\n'
    'daily,,,O1,WindFlow,0.47406507256065311,19600,10.81932543934753,0.86028292405222373,1.6259576768193018,1.3638377973000979,,20240417,25,,,,\n'
    'daily,,,O1,WindFlow,0.4282007858111806,19600,12.465481535916028,0.97069282361445897,1.6610099381941468,1.4723775866112263,,20240418,26,,,,\n'
    'daily,,,O1,WindFlow,0.47317599754349071,19600,9.8370610486842036,0.88572907262146938,1.6647576252228751,1.3961479102045284,,20240419,27,,,,\n'
    'daily,,,O1,WindFlow,0.50118364874413157,19600,7.6056665048517935,0.87316629615672003,1.6571526629340807,1.3645863935809741,,20240420,28,,,,\n'
    'daily,,,O1,Taper,0.42752979073522301,19600,10.120685345229983,0.96978153078164242,1.7337158685102549,1.6152367814939204,,20240326,3,,,,\n'
    'daily,,,O1,Taper,0.33414469883483139,19600,18.15838499119857,0.99617899559755985,1.7737583062562776,1.675484368432073,,20240327,4,,,,\n'
    'daily,,,O1,Taper,0.37599867456891228,19600,15.621875230244056,0.88502142903461756,1.5841584619092584,1.794974286322919,,20240328,5,,,,\n'
    'daily,,,O1,Taper,0.4158227071006188,19600,12.839731882320397,0.91422991537788745,1.7723567163066709,1.5151951574811491,,20240329,6,,,,\n'
    'daily,,,O1,Taper,0.50946582590741918,19600,11.108105256757325,0.8110564478536868,1.470885731880629,1.2151181717834931,,20240330,7,,,,\n'
    'daily,,,O1,Taper,0.55478857285881877,19600,7.080966706636989,0.71886730494516959,1.4043910883933799,1.3131954015238407,,20240331,8,,,,\n'
    'daily,,,O1,Taper,0.38464467621977189,19600,8.5671336560954003,1.1999533193800771,1.9079136905526808,1.7327486070354861,,20240401,9,,,,\n'
    'daily,,,O1,Taper,0.54041376506172101,19600,7.0910023545580216,0.8061625697669158,1.4469191361628542,1.3110054332103378,,20240402,10,,,,\n'
    'daily,,,O1,Taper,0.51002714968891272,19600,6.5651129775480239,0.86289901692692916,1.6041826690268184,1.4280734246370737,,20240403,11,,,,\n'
    'daily,,,O1,Taper,0.55310201437234185,19600,6.4934269706785352,0.81088285083362976,1.4272187343749143,1.2648330588173251,,20240404,12,,,,\n'
    'daily,,,O1,Taper,0.48248455253484351,19600,7.3329267132795604,0.97169114302682336,1.6815597077404771,1.4245603641923488,,20240405,13,,,,\n'
    'daily,,,O1,Taper,0.54420387602579279,19600,6.6730454086574928,0.74534857444272051,1.5544154880439354,1.2914258364631994,,20240406,14,,,,\n'
    'daily,,,O1,Taper,0.52522726375879236,19600,7.0967722982901842,0.78065633623063313,1.6248593228597841,1.3243950291735318,,20240407,15,,,,\n'
    'daily,,,O1,Taper,0.55337940898569626,19600,6.8407171029270479,0.76922285597195905,1.4871058544901583,1.2293489192511029,,20240408,16,,,,\n'
    'daily,,,O1,Taper,0.55768111973645373,19600,6.2588760951586364,0.79431382112649318,1.3555897554857477,1.3184981224789325,,20240409,17,,,,\n'
    'daily,,,O1,Taper,0.51631111323491075,19600,7.0899967127218613,0.87450132362434074,1.5087224040399545,1.3950898870329311,,20240410,18,,,,\n'
    'daily,,,O1,Taper,0.50981549087773814,19600,6.4006960949012521,0.85772031573933605,1.6771046569005159,1.3917564241197935,,20240411,19,,,,\n'
    'daily,,,O1,Taper,0.43023169919944371,19600,9.1637181906606315,0.93893101803362444,1.8572351150758248,1.6063338997643557,,20240412,20,,,,\n'
    'daily,,,O1,Taper,0.40921851698668371,19600,8.7869803556718544,1.0065120637475304,1.854635066106044,1.7465279919973531,,20240413,21,,,,\n'
    'daily,,,O1,Taper,0.47638328274455322,19600,8.9959388141971388,0.93248735717901221,1.6830543722004592,1.3770752453940631,,20240414,22,,,,\n'
    'daily,,,O1,Taper,0.44978660670938331,19600,9.9032134602571045,0.98140540229276763,1.7024009698075078,1.4599502682476559,,20240415,23,,,,\n'
    'daily,,,O1,Taper,0.51585669155505709,19600,8.3959560332860175,0.86629195967720996,1.4514762161758756,1.3443103235042655,,20240416,24,,,,\n'
    'daily,,,O1,Taper,0.47547482378305789,19600,10.8400177514576,0.85767812728861914,1.6107637782781514,1.3648414727061453,,20240417,25,,,,\n'
    'daily,,,O1,Taper,0.42234650759768311,19600,12.530812079450831,0.98660455900184796,1.6804292518556996,1.4846605768643448,,20240418,26,,,,\n'
    'daily,,,O1,Taper,0.47244904571340368,19600,9.8876772989027248,0.87408921555969388,1.6859525666501545,1.3932929884037448,,20240419,27,,,,\n'
    'daily,,,O1,Taper,0.499622281039861,19600,7.6321235164619958,0.86062914227746645,1.6700190791772671,1.3780883255004031,,20240420,28,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.45131175086758879,19600,9.9458649039807856,0.88214466284302762,1.6275782196234547,1.6023768718116691,,20240326,3,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.35493958103883311,19600,18.815108572551441,0.97041548070733141,1.6315705428046787,1.5846783430568754,,20240327,4,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.39430978666050231,19600,16.076633888192848,0.82906394225771729,1.5025609403569049,1.7248162856833325,,20240328,5,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.44165931458276908,19600,12.832181560743273,0.91142186517261958,1.6148677481179128,1.4256960026295502,,20240329,6,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.53904049009043409,19600,10.653823192564211,0.78919187720322015,1.3215993838833555,1.1440470995709886,,20240330,7,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.5767425678715572,19600,6.8088957878034231,0.69457878862652389,1.3249203475012268,1.2387788564290512,,20240331,8,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.42838098259997909,19600,8.8294919947558803,1.1065136642107316,1.7712759951036303,1.5469191663583917,,20240401,9,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.56598151752357495,19600,6.4455891101388252,0.8069432779774931,1.3557419219690019,1.2209314463631733,,20240402,10,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.5487497516279326,19600,5.8254672018684142,0.78728113944864198,1.4456963370069844,1.3656132937221888,,20240403,11,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.57463918799995739,19600,5.7913904959733591,0.84237160484865115,1.3240444538487586,1.1913895992272681,,20240404,12,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.51004130533091396,19600,6.8834152087577198,0.90421548493927939,1.6040516691555615,1.3600905527507223,,20240405,13,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.56183477350295419,19600,6.1204725495863004,0.72407174135251962,1.4595410850047104,1.285164584127233,,20240406,14,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.55049962290437826,19600,6.3909460531705751,0.77208612509528329,1.5326138032243868,1.2520460602544987,,20240407,15,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.56932216608690656,19600,6.2258173634626965,0.76787093643438975,1.4529322093287504,1.1748483957817857,,20240408,16,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.60114828370409779,19600,5.7286459064008888,0.72786930359640889,1.2180710780152819,1.1735598388121475,,20240409,17,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.55322645147147675,19600,6.5257371336959702,0.82631903027253983,1.4014411496373589,1.2656282070915099,,20240410,18,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.55032504501869539,19600,5.9985833388595511,0.78760948475366344,1.5396646350959382,1.2646136719421992,,20240411,19,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.48418236145003968,19600,8.0259871396628739,0.85908216377298785,1.676857433810969,1.4708277343481868,,20240412,20,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.4610121996417621,19600,7.3921225965930963,0.93407256221235124,1.6928054435019355,1.6282490077407583,,20240413,21,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.51691678931599239,19600,7.8357113308636244,0.93527651411587043,1.5437711611756997,1.2398857908287051,,20240414,22,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.50033450173611738,19600,9.4524338773105008,0.90477098577094639,1.5002541535071656,1.3100327566678027,,20240415,23,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.55081595550260209,19600,7.9265166560597393,0.84235189811584765,1.3327482797856611,1.2077964507697063,,20240416,24,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.48871425065248569,19600,10.595226518761343,0.87144196141879648,1.5383379282918961,1.3169917765078556,,20240417,25,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.47035725721970478,19600,10.895727482996277,0.94759280233804999,1.5716724766784953,1.3425769935630478,,20240418,26,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.5135548204278827,19600,8.3866758460567326,0.85914946341041554,1.518709135546167,1.3190973913964152,,20240419,27,,,,\n'
    'daily,,,O1,MixPhysicsNewBias,0.53249633162368515,19600,6.6024119377302233,0.86686220753882404,1.5468308304029597,1.2780847493696237,,20240420,28,,,,\n'
    'daily,,,O1,MixRecentBias,0.45130386797126337,19600,9.9478814029688394,0.88214262862565096,1.6273960380997678,1.6024207964808808,,20240326,3,,,,\n'
    'daily,,,O1,MixRecentBias,0.35493103952908678,19600,18.813970693017719,0.9704139377693356,1.6321500429994422,1.5843930871077128,,20240327,4,,,,\n'
    'daily,,,O1,MixRecentBias,0.3939539646312562,19600,16.076232494593103,0.82906156723822799,1.5045838822010029,1.7262457913710758,,20240328,5,,,,\n'
    'daily,,,O1,MixRecentBias,0.4416066907275728,19600,12.83149450582035,0.91141690299132161,1.6151489313661798,1.4259777807992997,,20240329,6,,,,\n'
    'daily,,,O1,MixRecentBias,0.53886943134491583,19600,10.655876860081454,0.78919376408206943,1.3224262114555769,1.1446599071701171,,20240330,7,,,,\n'
    'daily,,,O1,MixRecentBias,0.57670060811772739,19600,6.8089632620488159,0.69457695638747907,1.3250008534768865,1.2390627681241291,,20240331,8,,,,\n'
    'daily,,,O1,MixRecentBias,0.4284925692365939,19600,8.822877701915953,1.1065192718532719,1.7719172476526961,1.5460262295049636,,20240401,9,,,,\n'
    'daily,,,O1,MixRecentBias,0.56605874759600405,19600,6.4462067144975377,0.80694205584138257,1.3550373773875837,1.2207862400178224,,20240402,10,,,,\n'
    'daily,,,O1,MixRecentBias,0.54889055362106776,19600,5.8253044950973951,0.78728149255026314,1.4447525257879343,1.3651864232684443,,20240403,11,,,,\n'
    'daily,,,O1,MixRecentBias,0.57476555363803772,19600,5.7896576606147727,0.84237562986295789,1.3236165132593034,1.1908066168778553,,20240404,12,,,,\n'
    'daily,,,O1,MixRecentBias,0.50979804727793254,19600,6.8842985685243461,0.90421857612978218,1.6049363068854141,1.3613563022952482,,20240405,13,,,,\n'
    'daily,,,O1,MixRecentBias,0.56182718138108112,19600,6.1187679066532086,0.72407149047872255,1.4587360446995898,1.285996211244693,,20240406,14,,,,\n'
    'daily,,,O1,MixRecentBias,0.55063065061266692,19600,6.3921988032442858,0.77208543253990791,1.5317519636042856,1.2515214381072264,,20240407,15,,,,\n'
    'daily,,,O1,MixRecentBias,0.56948141338166314,19600,6.2242392554475447,0.76787051347596891,1.4521073081240494,1.1742920894252344,,20240408,16,,,,\n'
    'daily,,,O1,MixRecentBias,0.60126744245714692,19600,5.7262828108084234,0.72786692629215854,1.2180123315484339,1.1728075472831596,,20240409,17,,,,\n'
    'daily,,,O1,MixRecentBias,0.55350076446188812,19600,6.521611422836548,0.82631614792437413,1.4007009729464035,1.2642572015883038,,20240410,18,,,,\n'
    'daily,,,O1,MixRecentBias,0.55033755845068111,19600,5.9964851895001887,0.78760776221649231,1.5402686035339257,1.2642127516844237,,20240411,19,,,,\n'
    'daily,,,O1,MixRecentBias,0.48401067160137812,19600,8.0234931625458614,0.85907833320412907,1.6775734904395831,1.4719123101117089,,20240412,20,,,,\n'
    'daily,,,O1,MixRecentBias,0.46086448467438701,19600,7.3925312105175207,0.93407145143895243,1.6929138585741967,1.6293654469125562,,20240413,21,,,,\n'
    'daily,,,O1,MixRecentBias,0.51690939367932209,19600,7.8376985880001167,0.93527548811365724,1.5441839251704712,1.2394631833740912,,20240414,22,,,,\n'
    'daily,,,O1,MixRecentBias,0.5003821596510718,19600,9.4580393217577008,0.90477121248013803,1.4998178624103109,1.3095132882863338,,20240415,23,,,,\n'
    'daily,,,O1,MixRecentBias,0.55108158556858988,19600,7.9260219290376961,0.84234784176681454,1.3319904357872097,1.2062140147202531,,20240416,24,,,,\n'
    'daily,,,O1,MixRecentBias,0.4889342891762809,19600,10.591679282484582,0.87143402245127621,1.5377347448755851,1.3159246694762463,,20240417,25,,,,\n'
    'daily,,,O1,MixRecentBias,0.4704190888200725,19600,10.887809306042676,0.94758855519977436,1.5718005303864988,1.3426169439965818,,20240418,26,,,,\n'
    'daily,,,O1,MixRecentBias,0.51365821384110277,19600,8.3830622587626102,0.85914863130378771,1.5181953518479381,1.3189328948789705,,20240419,27,,,,\n'
    'daily,,,O1,MixRecentBias,0.53232919622868391,19600,6.6035333961165721,0.86686676906397397,1.5473915732148869,1.2789462838213617,,20240420,28,,,,\n'
    'daily,,,O1,MixDriftBias,0.45159520960050908,19600,9.9410960808855684,0.8821440074546546,1.626814789988978,1.6009983139294766,,20240326,3,,,,\n'
    'daily,,,O1,MixDriftBias,0.35471422411861031,19600,18.814784779166157,0.97041586411625602,1.6352968335116751,1.5836812148407635,,20240327,4,,,,\n'
    'daily,,,O1,MixDriftBias,0.39447044489507921,19600,16.075445253652944,0.82905479879454813,1.5041396253277803,1.7223509874055158,,20240328,5,,,,\n'
    'daily,,,O1,MixDriftBias,0.44179637544771322,19600,12.832315860734097,0.9114187959111868,1.614394681020495,1.4249123651268305,,20240329,6,,,,\n'
    'daily,,,O1,MixDriftBias,0.53862001990502928,19600,10.657579346775066,0.78919912494909494,1.3238296093184996,1.1455028631582145,,20240330,7,,,,\n'
    'daily,,,O1,MixDriftBias,0.57659676377342939,19600,6.8075463310466606,0.69457762290793879,1.3256792387546092,1.2395169705778073,,20240331,8,,,,\n'
    'daily,,,O1,MixDriftBias,0.42832315540919758,19600,8.8289008645947114,1.106512764483025,1.7717377255601088,1.5470916273032234,,20240401,9,,,,\n'
    'daily,,,O1,MixDriftBias,0.56612449665125708,19600,6.4444562107314383,0.8069419635455124,1.3561894361884472,1.2194831948588334,,20240402,10,,,,\n'
    'daily,,,O1,MixDriftBias,0.5491623857078094,19600,5.8232861498718069,0.78728142910331345,1.4435994111800967,1.3639817578726965,,20240403,11,,,,\n'
    'daily,,,O1,MixDriftBias,0.57551189157784499,19600,5.784697364007255,0.8423648406228319,1.3197043832511479,1.1880437330858356,,20240404,12,,,,\n'
    'daily,,,O1,MixDriftBias,0.50950858921633779,19600,6.8845711529924074,0.90422297537519003,1.6064930734374685,1.3625327687883471,,20240405,13,,,,\n'
    'daily,,,O1,MixDriftBias,0.56215900066118407,19600,6.1153021309822098,0.7240735274395167,1.4568255077426835,1.2849981829906669,,20240406,14,,,,\n'
    'daily,,,O1,MixDriftBias,0.55097852016917448,19600,6.3890393868143169,0.77209037693672755,1.530324013765874,1.249984974888507,,20240407,15,,,,\n'
    'daily,,,O1,MixDriftBias,0.570661306691014,19600,6.2144503326879095,0.76786090074363256,1.4451624058007131,1.1706696527199201,,20240408,16,,,,\n'
    'daily,,,O1,MixDriftBias,0.60146059621868042,19600,5.7239122862586811,0.72787436947626249,1.2169644959191013,1.1721991991205858,,20240409,17,,,,\n'
    'daily,,,O1,MixDriftBias,0.55325435665175693,19600,6.5239221607403444,0.82632354004574704,1.4028748815485808,1.264422221051213,,20240410,18,,,,\n'
    'daily,,,O1,MixDriftBias,0.54993116276014342,19600,5.997827460291175,0.78760758016504451,1.5431487041157488,1.2652505281449482,,20240411,19,,,,\n'
    'daily,,,O1,MixDriftBias,0.48295798676671658,19600,8.026996580696439,0.85909796286063544,1.6819061772589174,1.4770164163672437,,20240412,20,,,,\n'
    'daily,,,O1,MixDriftBias,0.45996403576695422,19600,7.3925744603187171,0.9340817330202994,1.6985099579244305,1.6325075841034948,,20240413,21,,,,\n'
    'daily,,,O1,MixDriftBias,0.51604610728793454,19600,7.8352723168347627,0.93526204749918163,1.5511462181864508,1.2414571081604917,,20240414,22,,,,\n'
    'daily,,,O1,MixDriftBias,0.49955071611229701,19600,9.4587493567213556,0.90477237163145219,1.505800400893698,1.3117322470012245,,20240415,23,,,,\n'
    'daily,,,O1,MixDriftBias,0.55053046006109063,19600,7.9265595677389431,0.84235162924870655,1.3358413853736906,1.2077656409048767,,20240416,24,,,,\n'
    'daily,,,O1,MixDriftBias,0.49001577708812638,19600,10.587093613328411,0.87142827684614754,1.5313939357649178,1.3122207401763171,,20240417,25,,,,\n'
    'daily,,,O1,MixDriftBias,0.4698164610047576,19600,10.890400247329646,0.94759658915999123,1.5764079733225156,1.343835514868275,,20240418,26,,,,\n'
    'daily,,,O1,MixDriftBias,0.51328007994100855,19600,8.3849432622843185,0.85915224672427715,1.5218338374957547,1.3190952101819688,,20240419,27,,,,\n'
    'daily,,,O1,MixDriftBias,0.53243709578012111,19600,6.5984269998538947,0.8668605449538892,1.5459402702561671,1.2796034357955934,,20240420,28,,,,\n'
    'daily,,,O1,MixMultiScale,0.45131454795794079,19600,9.9464767506255836,0.88214496566469192,1.6275837348233029,1.6022985425712242,,20240326,3,,,,\n'
    'daily,,,O1,MixMultiScale,0.35492338866849521,19600,18.814908241954733,0.97041592104859964,1.6316425836038382,1.5847734347899014,,20240327,4,,,,\n'
    'daily,,,O1,MixMultiScale,0.39429830847631958,19600,16.076247251558772,0.82906435720360017,1.5025774729763337,1.7249306513437399,,20240328,5,,,,\n'
    'daily,,,O1,MixMultiScale,0.44165285407540561,19600,12.831668067473442,0.91142225194438442,1.6148349449028938,1.4258174046754348,,20240329,6,,,,\n'
    'daily,,,O1,MixMultiScale,0.53902402829652463,19600,10.653365874218853,0.78919202362518537,1.3217968828608695,1.1440680791827005,,20240330,7,,,,\n'
    'daily,,,O1,MixMultiScale,0.5767346042589806,19600,6.8083759070691654,0.69457853384760582,1.3250163809216895,1.2388135363975159,,20240331,8,,,,\n'
    'daily,,,O1,MixMultiScale,0.42833750518388441,19600,8.8292014886620152,1.1065142817539386,1.7716072954477882,1.547047210544146,,20240401,9,,,,\n'
    'daily,,,O1,MixMultiScale,0.56597537829086653,19600,6.4456176717717177,0.80694312413958158,1.3557131292278488,1.221002964123646,,20240402,10,,,,\n'
    'daily,,,O1,MixMultiScale,0.54873940131183063,19600,5.8256789037168319,0.78728135931623477,1.445774129899541,1.3656214222522736,,20240403,11,,,,\n'
    'daily,,,O1,MixMultiScale,0.57463592579312506,19600,5.7913757071043852,0.84237148141458762,1.3240280789943373,1.1914309488425143,,20240404,12,,,,\n'
    'daily,,,O1,MixMultiScale,0.51001206804960719,19600,6.8829063165978361,0.90421538130254042,1.6043761891517785,1.3601236786763993,,20240405,13,,,,\n'
    'daily,,,O1,MixMultiScale,0.5618316867022729,19600,6.1201798566314762,0.72407183343786463,1.459705692581692,1.2850860242807689,,20240406,14,,,,\n'
    'daily,,,O1,MixMultiScale,0.5504703931941215,19600,6.3908174626969263,0.77208648766745747,1.5327968339384801,1.252157658389166,,20240407,15,,,,\n'
    'daily,,,O1,MixMultiScale,0.56933648595770758,19600,6.2257262433858509,0.76787069924569862,1.4529114521674822,1.1747527431640277,,20240408,16,,,,\n'
    'daily,,,O1,MixMultiScale,0.60111860608416279,19600,5.728170980963661,0.72786952280415174,1.2182484100998692,1.1737082748606864,,20240409,17,,,,\n'
    'daily,,,O1,MixMultiScale,0.55319884393689311,19600,6.525604604788672,0.82631968052989635,1.4016316372420312,1.2657204790210468,,20240410,18,,,,\n'
    'daily,,,O1,MixMultiScale,0.55027923737562356,19600,5.9983162588631744,0.78760969099722511,1.5399393460021966,1.2648037819610909,,20240411,19,,,,\n'
    'daily,,,O1,MixMultiScale,0.48416410977299379,19600,8.026142465121664,0.85908240297726868,1.6768944667705197,1.4709382423036259,,20240412,20,,,,\n'
    'daily,,,O1,MixMultiScale,0.46099457363348889,19600,7.3921363288019313,0.93407288307051117,1.6929209123293532,1.6283047115392415,,20240413,21,,,,\n'
    'daily,,,O1,MixMultiScale,0.51688353266800791,19600,7.8359098945560088,0.9352768348987498,1.5439717382167539,1.2399904468160388,,20240414,22,,,,\n'
    'daily,,,O1,MixMultiScale,0.50031493040913733,19600,9.4526647772385672,0.90477122182712422,1.5004086614215957,1.3100564764744391,,20240415,23,,,,\n'
    'daily,,,O1,MixMultiScale,0.55077949712725383,19600,7.9261045887177559,0.84235258243532507,1.332910617818436,1.20800755445436,,20240416,24,,,,\n'
    'daily,,,O1,MixMultiScale,0.48875454359741721,19600,10.594477205548335,0.87144217017514491,1.5381143842877811,1.3168910725510798,,20240417,25,,,,\n'
    'daily,,,O1,MixMultiScale,0.47035224171246531,19600,10.895201236663729,0.94759387762512304,1.5717459220443148,1.3426038460415697,,20240418,26,,,,\n'
    'daily,,,O1,MixMultiScale,0.51355439899205746,19600,8.3865616031560446,0.85914993490182179,1.518855436421688,1.3189957742876592,,20240419,27,,,,\n'
    'daily,,,O1,MixMultiScale,0.53247943519276342,19600,6.60221291474571,0.86686260041710195,1.5469010196669399,1.2781871041828576,,20240420,28,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.45127511596260289,19600,9.9460600460000936,0.88214487476704018,1.6274304255551111,1.602781738030449,,20240326,3,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.3548324669613015,19600,18.81797875000828,0.97041728815074801,1.6316686224718593,1.5852582242297617,,20240327,4,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.39424398455250559,19600,16.075861362535356,0.82906429669835002,1.5021153701863956,1.7257763848684775,,20240328,5,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.44164337546461208,19600,12.832075004979076,0.91142058851583585,1.6146173501538754,1.4260343862976064,,20240329,6,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.53903389290325232,19600,10.653655581878979,0.78919131619161553,1.3210365727900135,1.1445553508926438,,20240330,7,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.57670570915978603,19600,6.8088412169082728,0.69457877594387341,1.3245375133859316,1.2393896243640354,,20240331,8,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.42849341046710437,19600,8.8295245792859358,1.1065140607955986,1.7706796705344596,1.5464421919874398,,20240401,9,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.56605272989519806,19600,6.445817491529537,0.80694309228614747,1.3554407479790884,1.2205529935730579,,20240402,10,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.54872101849663513,19600,5.8245339633864921,0.78728089817618441,1.4461642091944988,1.3655655154960826,,20240403,11,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.5746075068249813,19600,5.7910363415806279,0.84237271135924774,1.3244886620153535,1.1913358562798555,,20240404,12,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.51027219774400101,19600,6.8833556885252394,0.90421534655896763,1.6024867286711415,1.3593876594509433,,20240405,13,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.56184326105345228,19600,6.1206598561866619,0.72407164617519093,1.459057153843359,1.2854556856613195,,20240406,14,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.55070995454457561,19600,6.3914805992023611,0.77208413527467035,1.5314016165476527,1.2511928007312074,,20240407,15,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.56929865382074307,19600,6.2247571920197728,0.76787234756045675,1.4529883709822622,1.1750867913938052,,20240408,16,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.60127851166800239,19600,5.7275954491755119,0.72786951215656592,1.2175271405691961,1.1729827272714544,,20240409,17,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.55332408703366043,19600,6.525010315495912,0.8263192461396156,1.4005554842128869,1.2655630356280181,,20240410,18,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.55047911354871126,19600,5.9988213235168546,0.78761078850746413,1.5382988271292126,1.2643710864812969,,20240411,19,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.48454289436716008,19600,8.0252159463688546,0.85908141209594246,1.674577665340486,1.4696588129556836,,20240412,20,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.46121481986475549,19600,7.3911112800715584,0.93407119766875835,1.6917008740248169,1.6275028587185336,,20240413,21,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.5170060651693893,19600,7.834920717855101,0.93527667921830482,1.5431022371810568,1.2397267458955736,,20240414,22,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.50041015871608363,19600,9.4533328375148535,0.90477155564379019,1.4996763247012916,1.3097769196173583,,20240415,23,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.55101813896498808,19600,7.9259588478276513,0.84235263168331453,1.3311140380376916,1.2074274566302838,,20240416,24,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.4886562714492963,19600,10.595650386517992,0.87144577412817503,1.5390800268210696,1.3168586001202274,,20240417,25,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.47049631923101581,19600,10.895269164437307,0.94759233649224039,1.5705341181436014,1.3423414943160064,,20240418,26,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.51355512962308336,19600,8.3863405373003665,0.8591486748013113,1.5178482920256551,1.3197943684025937,,20240419,27,,,,\n'
    'daily,,,O1,MixAdaptiveFlow,0.53253119498245316,19600,6.60256124485797,0.86686375747141486,1.5466903854688765,1.2778892672525779,,20240420,28,,,,\n'
    'daily,,,O1,MixWindFlow,0.45127706497450321,19600,9.9458648811156696,0.88214466768654154,1.6274304255551111,1.602781738030449,,20240326,3,,,,\n'
    'daily,,,O1,MixWindFlow,0.35486098245473863,19600,18.815108340507081,0.97041548011105916,1.6316686224718593,1.5852582242297617,,20240327,4,,,,\n'
    'daily,,,O1,MixWindFlow,0.39423641464029002,19600,16.076633602597649,0.8290639426039188,1.5021153701863956,1.7257763848684775,,20240328,5,,,,\n'
    'daily,,,O1,MixWindFlow,0.44164217189473798,19600,12.832181427357961,0.91142186442417761,1.6146173501538754,1.4260343862976064,,20240329,6,,,,\n'
    'daily,,,O1,MixWindFlow,0.53903217341954823,19600,10.653823097200911,0.78919187883687103,1.3210365727900135,1.1445553508926438,,20240330,7,,,,\n'
    'daily,,,O1,MixWindFlow,0.57670517093531282,19600,6.8088956509074352,0.69457878925784977,1.3245375133859316,1.2393896243640354,,20240331,8,,,,\n'
    'daily,,,O1,MixWindFlow,0.428493781503125,19600,8.8294918072897115,1.1065136645396068,1.7706796705344596,1.5464421919874398,,20240401,9,,,,\n'
    'daily,,,O1,MixWindFlow,0.56605495792680516,19600,6.4455892052488517,0.80694327753153128,1.3554407479790884,1.2205529935730579,,20240402,10,,,,\n'
    'daily,,,O1,MixWindFlow,0.54871178761570094,19600,5.8254673178957894,0.78728114460378518,1.4461642091944988,1.3655655154960826,,20240403,11,,,,\n'
    'daily,,,O1,MixWindFlow,0.57460414865509613,19600,5.7913905863035389,0.84237160449268877,1.3244886620153535,1.1913358562798555,,20240404,12,,,,\n'
    'daily,,,O1,MixWindFlow,0.51027159468931804,19600,6.8834151631260072,0.90421548493927939,1.6024867286711415,1.3593876594509433,,20240405,13,,,,\n'
    'daily,,,O1,MixWindFlow,0.56184509432409957,19600,6.1204727037005178,0.72407174271772767,1.459057153843359,1.2854556856613195,,20240406,14,,,,\n'
    'daily,,,O1,MixWindFlow,0.5507149816853244,19600,6.3909461742209439,0.77208612814575195,1.5314016165476527,1.2511928007312074,,20240407,15,,,,\n'
    'daily,,,O1,MixWindFlow,0.56928837251414688,19600,6.2258174903350403,0.76787093363347958,1.4529883709822622,1.1750867913938052,,20240408,16,,,,\n'
    'daily,,,O1,MixWindFlow,0.60126818068059462,19600,5.7286459754888046,0.72786930441905473,1.2175271405691961,1.1729827272714544,,20240409,17,,,,\n'
    'daily,,,O1,MixWindFlow,0.55331694904339046,19600,6.5257370293466863,0.82631903146649055,1.4005554842128869,1.2655630356280181,,20240410,18,,,,\n'
    'daily,,,O1,MixWindFlow,0.55048161830063835,19600,5.9985832513465178,0.78760948473879211,1.5382988271292126,1.2643710864812969,,20240411,19,,,,\n'
    'daily,,,O1,MixWindFlow,0.4845352024165025,19600,8.0259869874882064,0.85908216377298785,1.674577665340486,1.4696588129556836,,20240412,20,,,,\n'
    'daily,,,O1,MixWindFlow,0.4612046853920197,19600,7.3921225704338474,0.93407256175551157,1.6917008740248169,1.6275028587185336,,20240413,21,,,,\n'
    'daily,,,O1,MixWindFlow,0.51699829447750389,19600,7.8357110328103419,0.93527651178967519,1.5431022371810568,1.2397267458955736,,20240414,22,,,,\n'
    'daily,,,O1,MixWindFlow,0.50041908792075618,19600,9.4524340575323684,0.90477098571540882,1.4996763247012916,1.3097769196173583,,20240415,23,,,,\n'
    'daily,,,O1,MixWindFlow,0.55101273016728958,19600,7.9265165264236384,0.84235189793815846,1.3311140380376916,1.2074274566302838,,20240416,24,,,,\n'
    'daily,,,O1,MixWindFlow,0.48866091704736098,19600,10.595225955133374,0.8714419614820742,1.5390800268210696,1.3168586001202274,,20240417,25,,,,\n'
    'daily,,,O1,MixWindFlow,0.47049175205395871,19600,10.89572673819557,0.94759280235790722,1.5705341181436014,1.3423414943160064,,20240418,26,,,,\n'
    'daily,,,O1,MixWindFlow,0.5135517328771777,19600,8.3866754169960771,0.85914946360094613,1.5178482920256551,1.3197943684025937,,20240419,27,,,,\n'
    'daily,,,O1,MixWindFlow,0.53253285711816867,19600,6.6024116951129423,0.86686220553371651,1.5466903854688765,1.2778892672525779,,20240420,28,,,,\n'
    'daily,,,O1,MixTaper,0.45131875499828311,19600,9.9463813193481236,0.88214313704990033,1.6275259660363537,1.6023181559529125,,20240326,3,,,,\n'
    'daily,,,O1,MixTaper,0.35490856831769829,19600,18.816010063319105,0.97041592189260995,1.6317850342138631,1.5846954272832312,,20240327,4,,,,\n'
    'daily,,,O1,MixTaper,0.39430767281501278,19600,16.077442770363639,0.82906294260826452,1.5024654516970577,1.7248428096335189,,20240328,5,,,,\n'
    'daily,,,O1,MixTaper,0.4416868600871211,19600,12.832559923176968,0.91141733124292523,1.6145414121263864,1.425693852458362,,20240329,6,,,,\n'
    'daily,,,O1,MixTaper,0.53902309716552255,19600,10.654270240104232,0.78918724613222857,1.3218264522350021,1.1439831838025001,,20240330,7,,,,\n'
    'daily,,,O1,MixTaper,0.57676620944250512,19600,6.8083882733921346,0.69457555938408466,1.3249649764390894,1.2385916869009466,,20240331,8,,,,\n'
    'daily,,,O1,MixTaper,0.42829595229580381,19600,8.8304777175389741,1.1065219052174931,1.7714760587185625,1.547383749405199,,20240401,9,,,,\n'
    'daily,,,O1,MixTaper,0.56603789779460412,19600,6.4450912409625705,0.80693586749751889,1.3555288397063583,1.2206752536195882,,20240402,10,,,,\n'
    'daily,,,O1,MixTaper,0.54875039364753864,19600,5.8258527917690266,0.78728072576929131,1.4453689252236606,1.3658318772594762,,20240403,11,,,,\n'
    'daily,,,O1,MixTaper,0.57466571086945184,19600,5.7910745849287135,0.84236112442820188,1.3239639768875675,1.1912674662500069,,20240404,12,,,,\n'
    'daily,,,O1,MixTaper,0.50998073164613755,19600,6.8824049758425625,0.90420630108052524,1.6048898191138536,1.360035345207792,,20240405,13,,,,\n'
    'daily,,,O1,MixTaper,0.56186972946967895,19600,6.119058939764435,0.72406472319737525,1.4598439683465092,1.2847600044644445,,20240406,14,,,,\n'
    'daily,,,O1,MixTaper,0.55047395117858211,19600,6.3894934820897458,0.77207869271085283,1.532919896970224,1.252148869031791,,20240407,15,,,,\n'
    'daily,,,O1,MixTaper,0.56937135114402193,19600,6.2267481668148168,0.76786622039592634,1.452553472228894,1.1746611093069594,,20240408,16,,,,\n'
    'daily,,,O1,MixTaper,0.60110048679917816,19600,5.7280610503093339,0.72786861731642927,1.2186088214332249,1.1735885468569032,,20240409,17,,,,\n'
    'daily,,,O1,MixTaper,0.55316680932188667,19600,6.5254644272143478,0.82631498214838373,1.4020250676634998,1.2656975300616049,,20240410,18,,,,\n'
    'daily,,,O1,MixTaper,0.55019058001129051,19600,5.9982702938630954,0.78760592160752452,1.5405694535116554,1.2650604349899284,,20240411,19,,,,\n'
    'daily,,,O1,MixTaper,0.4840838713062886,19600,8.027783623593022,0.85907995901137446,1.677471070585677,1.4710260870456973,,20240412,20,,,,\n'
    'daily,,,O1,MixTaper,0.4608843559915885,19600,7.393711708756225,0.93406671046526757,1.6936185046010948,1.6285577491997014,,20240413,21,,,,\n'
    'daily,,,O1,MixTaper,0.51677169661475642,19600,7.8369563167724214,0.93527021879879924,1.5450215576820781,1.2400263750940217,,20240414,22,,,,\n'
    'daily,,,O1,MixTaper,0.5002771536337316,19600,9.4533215416755194,0.90475774568651079,1.5009855978580466,1.309881675564891,,20240415,23,,,,\n'
    'daily,,,O1,MixTaper,0.55071495473639753,19600,7.9265993090414772,0.84234114823223494,1.3335354235579575,1.2080302108354464,,20240416,24,,,,\n'
    'daily,,,O1,MixTaper,0.48889149433362888,19600,10.595641040982285,0.87143381513992502,1.5374291957859696,1.3161943290226961,,20240417,25,,,,\n'
    'daily,,,O1,MixTaper,0.47030309223421018,19600,10.897045553291548,0.94759965646750199,1.5720807300955102,1.3425955664953837,,20240418,26,,,,\n'
    'daily,,,O1,MixTaper,0.51361203939239108,19600,8.387815378800406,0.85913937623517056,1.5191849754170716,1.3181649918030789,,20240419,27,,,,\n'
    'daily,,,O1,MixTaper,0.53249760302263938,19600,6.6029794938635424,0.86685194117209696,1.5470346328493758,1.2778788946920461,,20240420,28,,,,\n'
    'daily_sse,,,O1,AdaptiveFlow,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,AdaptiveFlow,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240326,,2003764.5778052176,18479.484502943233,58454.1058272839,50767.55899909624\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240327,,6500130.8160896096,19432.109855804592,60661.118830755673,54567.722132740899\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240328,,4763744.7666132301,15334.088240200654,48215.381365239024,63117.35291844653\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240329,,3230811.3871976375,16452.318264734,61095.990469674754,44726.898769336993\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240330,,2411574.9113877532,13049.859505592845,41549.367689687955,28867.243234963138\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240331,,986919.07096240623,10346.109553276561,38044.351121170104,33936.074063083026\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240401,,1427723.091746574,27523.040229192935,70207.370025292024,57682.194872241635\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240402,,991049.25799749303,13144.423224315047,40561.375044179127,33334.285316209018\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240403,,835813.52330897446,14693.62596164085,50550.899221051761,39520.517895965997\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240404,,833920.84948780783,13486.545033496805,40079.031456130979,31118.700312484521\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240405,,1061553.0743195065,19032.983830544166,53374.32155621241,39176.077067809092\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240406,,883101.31108529516,11259.902510820888,47045.952491418648,32923.309391285031\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240407,,1000656.1273995461,12126.688566550612,50640.072537549357,33768.324570266494\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240408,,903770.57395418384,11883.596155053005,43802.781738189711,29551.476852235312\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240409,,766324.58022362576,12364.807760546915,35567.17763184019,33754.807645661997\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240410,,976424.0963783476,15258.936143515632,43590.560636423761,37821.988163749156\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240411,,798120.91340750502,14711.983411342837,53043.272983948431,37239.441110388274\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240412,,1617261.6827755542,17574.339846676216,64716.482708564137,49351.067786397944\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240413,,1478004.675828313,20155.970363171771,65424.229386168168,58864.845768736115\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240414,,1555567.3352173869,17601.354605301283,53363.616095237172,36583.481906327448\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240415,,1910762.6503772282,19776.642052905634,54546.944307460777,40859.761002917818\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240416,,1374735.8995644678,15227.936322606169,40326.204759306151,34615.254494639456\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240417,,2297018.6556636668,14737.526563220657,51817.271989429377,36457.049331949922\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240418,,3038722.0143461321,18517.250621198677,54075.498689682579,42490.756848079945\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240419,,1895957.2368925444,15333.459712979384,54319.791834459051,38204.888148502047\n'
    'daily_sse,,,O1,AdaptiveFlow,,19600,,,,,,20240420,,1134834.883550979,15077.892939762212,53824.636986082485,36497.082100708045\n'
    'daily_sse,,,O1,Bias325,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,Bias325,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240326,,2002377.6610131201,18434.98239208851,58918.721458300948,51127.564007136018\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240327,,6449604.2809615321,19192.663245436735,61496.399469624332,54960.21258841551\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240328,,4774986.130061578,15261.906003136188,49231.086935743202,63161.884960129064\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240329,,3230787.3561641318,16492.409104612656,61774.362496172995,44970.938632384037\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240330,,2414581.3607326411,13031.773918076418,42271.737562721944,28922.146921778378\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240331,,985871.38769529271,10272.153201642446,38578.216613222525,33857.460043056453\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240401,,1430355.845491301,27471.190775632858,71286.252679940211,58550.879775676862\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240402,,987152.03369225725,13084.732985514216,41077.968615370766,33737.710029425551\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240403,,839711.25449227821,14617.820008946583,50631.741534431378,39808.107251657748\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240404,,831196.95980604133,13379.805533475243,39985.837543585316,31348.440241612134\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240405,,1059594.4335046522,19022.380273895338,54754.269151306653,39752.011699904411\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240406,,881154.54083087016,11169.984105110168,47265.79157632932,32853.25940727329\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240407,,996559.40106255724,12187.794293062761,51564.374677168402,34244.249655415333\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240408,,909258.43828718364,11783.345600411296,43612.30637057111,29583.245186912449\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240409,,770641.51640610024,12360.108786926605,35826.125769119506,34007.471625739774\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240410,,980778.08454983798,15215.89538377244,44198.11557649375,38011.438654667974\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240411,,796512.84107938875,14613.206807507202,54395.257256477715,37625.945003713772\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240412,,1626485.1990895043,17597.796479961835,66999.373818955995,50363.258132948293\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240413,,1487191.935394614,20288.730981643312,66747.42016102522,59517.891842035424\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240414,,1559578.4291739529,17583.924987359904,54469.325443896174,36996.311782147008\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240415,,1904241.9179438099,19725.321778546087,56069.153361692996,41565.705264593053\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240416,,1377599.6613769094,15177.49651942309,40979.768632104082,35147.41040049163\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240417,,2294332.9380652476,14505.699504550546,51358.608722853896,36700.553059220852\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240418,,3045609.306476362,18467.993333205581,55176.069822562611,43012.61534289163\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240419,,1896648.2934805839,15376.513405703008,55205.895681356487,38210.104616036413\n'
    'daily_sse,,,O1,Bias325,,19600,,,,,,20240420,,1133784.7944672802,14943.41986258328,54455.398618481202,37033.334520481978\n'
    'daily_sse,,,O1,Drift,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,Drift,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240326,,2026231.6615888781,18650.935554231517,60167.661626905785,52021.726621674439\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240327,,6557202.6200568862,21147.660532695241,67708.845443009719,56027.290737509778\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240328,,4851923.8375744587,15712.963789177127,52381.70096959512,62449.034915489661\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240329,,3267721.8438663986,16077.619633041322,62448.015338346326,45956.896173754118\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240330,,2490036.6397876088,13142.627178598195,46040.358319513682,30509.037344767632\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240331,,992269.30787788285,10085.076495681889,41489.327750234967,35209.900868927536\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240401,,1498986.7027986548,30144.219477853738,73953.792528635837,60821.138245073802\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240402,,999738.09534113109,12482.042085009627,43345.001455716476,33694.22535458635\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240403,,853456.43065112107,14891.742030399852,51103.042781025659,40506.325886074796\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240404,,801892.56127296295,11943.510624569841,38316.94266617309,30740.536566482369\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240405,,1079999.7711782723,18396.481457969174,60322.688376270933,42430.740224845526\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240406,,854873.61708201142,10751.706791880541,46955.81784995248,33043.736468090538\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240407,,986887.30526855076,12198.424707099795,51747.934799362498,34402.752219997426\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240408,,868587.42795763654,11339.69118451979,39320.867178248467,28621.987658531914\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240409,,770366.89250160172,13086.986756883562,37412.361648625068,34111.777645432332\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240410,,1012100.2374139652,15631.007026356645,48080.81126206083,38956.080751818088\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240411,,841960.31857851194,14503.787416053936,61513.654162674131,40028.411955182273\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240412,,1708588.1370166072,18605.020035351627,75788.651406740872,56634.019463358774\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240413,,1565274.6776110081,20898.04758302588,76327.495839143288,63914.255317175943\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240414,,1638227.5440060433,16822.94095000159,65878.00599456791,40048.37543412346\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240415,,2040766.6905386269,19025.815090445802,66427.164212904187,45009.023366798938\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240416,,1429761.4804998208,14911.114188661799,46041.768253000373,37651.671448707006\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240417,,2248615.9816735149,14031.075615933165,47602.720280823567,35489.964841010464\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240418,,3086758.076133043,20386.430339386687,62780.006711799178,45882.446301172778\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240419,,1968908.3318125005,15846.49822540395,62690.900239669878,40313.81672834036\n'
    'daily_sse,,,O1,Drift,,19600,,,,,,20240420,,1181717.8301594409,14802.131769414991,59871.937708599347,40693.906507070744\n'
    'daily_sse,,,O1,DriftBias,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,DriftBias,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240326,,1998305.258629564,18546.317381762899,59895.183657426343,51856.51951949732\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240327,,6490002.1141449055,19561.850036122836,66482.562697085392,55646.511039292862\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240328,,4816835.8699017828,15110.207790327258,52266.717646421443,62442.79878271195\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240329,,3265489.2697263528,16396.227337596938,63004.165883189358,45745.007380896786\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240330,,2475452.6248101937,13531.99013721291,45214.487306778719,30332.205270642335\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240331,,1002412.8289893409,10449.3844832303,40949.67321141858,35292.993616601932\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240401,,1472986.5126396222,27544.170525320806,73448.800740870851,59778.695149602005\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240402,,1003154.405691385,13337.875856411643,43199.781033879022,33782.850454914034\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240403,,840218.15812321776,14797.270230526105,51377.974063967384,39964.722099804509\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240404,,813703.76593305077,13090.463671014644,38293.369140705276,30638.770550112102\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240405,,1091841.2860510373,19617.716601859778,58305.539649403443,42259.292717048513\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240406,,872825.94782427,11266.454366778955,46457.116379537889,33305.127741869372\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240407,,1003486.6685348388,12599.490453554317,51076.711933452192,33991.372049011457\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240408,,854595.57961874967,11605.181813143194,39619.670586625194,28409.884428019948\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240409,,774719.7776817081,12886.442502759397,36711.013735556058,33882.17342670283\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240410,,1004615.4133590469,15939.305221705697,46971.296271743828,38602.843285904077\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240411,,833084.12097573513,14808.220195088536,59722.643198520105,39254.96537679137\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240412,,1675956.0966671191,19098.831608244218,74322.258839585891,56141.283075141655\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240413,,1523364.7692452655,21558.489166568033,74827.214928152098,63326.130884690996\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240414,,1596942.6341156613,17717.091310204938,63641.12231328195,39657.485113938441\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240415,,2011656.3703732505,20423.9993505897,64806.49925094167,44579.475815093436\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240416,,1424721.877719454,15565.429920737632,45263.798067424636,37127.335861083629\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240417,,2240558.2851467994,14119.700222573243,48367.198917492547,35762.182743163321\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240418,,3041960.2009525453,19230.546754167415,62224.506159695011,45509.207363812588\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240419,,1942413.3040864095,16413.256765194237,61593.684996095981,40485.179204132255\n'
    'daily_sse,,,O1,DriftBias,,19600,,,,,,20240420,,1171566.8036399691,15395.134066655301,59262.352442921372,40318.470700564729\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240326,,1938912.5645973319,15252.319769524038,51911.183884440805,50350.622275372574\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240327,,6940679.9550209548,18457.5103775803,52181.912873760615,49255.655294766693\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240328,,5065293.0435301587,13472.013117974624,44224.471472864156,58374.760959166364\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240329,,3227378.1190948114,16281.474787742831,51096.988073391272,39858.051789702535\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240330,,2224607.3942431021,12207.329497624189,34204.697482316195,25676.136244636869\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240331,,908662.24685455114,9455.8176494371146,34386.232637585097,30107.298163232004\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240401,,1528025.8842058042,23997.717988071963,61452.007314622831,46873.075681914794\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240402,,814351.83742851927,12762.680222091265,36009.504577129366,29199.092358354999\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240403,,664933.83945659804,12148.299767608754,40991.262031120947,36549.475871396804\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240404,,657307.5974263479,13907.998982917517,34383.696229821537,27817.910000177922\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240405,,928659.47648115433,16025.065701873973,50332.088825115876,36219.522249883274\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240406,,734264.55067264452,10275.883076379076,41725.416452357073,32386.967868060674\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240407,,800680.07529961155,11683.832674079575,45965.741858829366,30683.475122191419\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240408,,759453.0011521969,11556.707666100934,41379.034041709892,27064.247759240494\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240409,,642984.85273655434,10383.962923849933,29054.497825243019,26967.414178152707\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240410,,834484.88849962992,13382.948532198556,38446.491021432361,31392.336024100885\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240411,,705322.80252101785,12158.482781797647,46380.72031832338,31333.231188865946\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240412,,1262320.1833227181,14465.209103131667,54962.523002240981,42333.98171936685\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240413,,1070719.1087062603,17100.784445361234,56092.296204656064,51915.804880685188\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240414,,1203165.2600394718,17144.952347116545,46670.824482106371,30123.679127981031\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240415,,1751563.834041998,16044.786730880849,44080.969945843652,33624.105351581791\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240416,,1231288.1436861656,13907.335939634591,34728.545812315773,28574.468835283653\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240417,,2200449.0194210494,14884.587650018744,46427.839647607063,33988.684825127857\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240418,,2326655.0472445535,17599.452229104936,48344.917358560757,35316.861472305565\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240419,,1378481.8691083211,14467.474330091849,45155.723377061884,34140.400627397146\n'
    'daily_sse,,,O1,MixAdaptiveFlow,,19600,,,,,,20240420,,854438.7738451683,14728.474370742217,46888.12251063648,32006.819195442877\n'
    'daily_sse,,,O1,MixDriftBias,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,MixDriftBias,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240326,,1936977.6692722081,15252.289777807891,51871.916674166896,50238.633783618519\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240327,,6938324.075212311,18457.4562068386,52414.236380388917,49157.70532869453\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240328,,5065030.8260225831,13471.704444323666,44343.745844631434,58143.261306808403\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240329,,3227499.2748530982,16281.410742187873,51082.895647702411,39795.354866510213\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240330,,2226246.3516430059,12207.571072880179,34349.48675636408,25718.665466271865\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240331,,908316.66616719821,9455.7862551910803,34445.538703674007,30113.485478867544\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240401,,1527810.0133460881,23997.661760091782,61525.46953618899,46912.453064126028\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240402,,814007.91069988627,12762.644517601468,36049.295821851112,29147.929545844905\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240403,,664648.96703246073,12148.316152774729,40845.993495206625,36464.746221866029\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240404,,655869.38242578879,13907.739084463567,34135.74531977695,27664.379069800747\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240405,,928987.47122805798,16025.336108248681,50584.071902050222,36387.312702031966\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240406,,732979.63500263286,10275.93647351861,41597.874976190666,32363.91847367059\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240407,,800068.55601078738,11684.021583102643,45901.075107326491,30624.263773961618\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240408,,756940.10157391941,11556.363112660125,40934.48983113822,26861.161741667809\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240409,,642158.16847134847,10384.101515713148,29027.650652821594,26931.398863411279\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240410,,834206.58304421976,13383.087619541213,38573.93549228691,31335.765640526424\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240411,,705089.11117108841,12158.383726535365,46673.635291076644,31376.834419832983\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240412,,1262880.4124876414,14465.766471909359,55444.644426393417,42758.918886679356\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240413,,1071143.0801665888,17101.170205659233,56544.747112501609,52235.587838246414\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240414,,1203273.248667561,17144.415910850279,47158.669967804919,30207.828727483335\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240415,,1753571.6121082234,16044.815671565011,44441.72300769978,33724.573161328459\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240416,,1231474.7929859101,13907.302839039825,34975.655254791171,28590.477729667207\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240417,,2196896.4030766329,14883.989937066101,45965.280775352272,33749.696110598081\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240418,,2324576.0239219377,17599.610197437927,48707.217127754098,35395.520264017126\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240419,,1378022.5608298532,14467.594627806917,45393.173287362282,34104.238601090248\n'
    'daily_sse,,,O1,MixDriftBias,,19600,,,,,,20240420,,853369.08189905714,14728.365206195973,46842.653856314333,32092.745076837797\n'
    'daily_sse,,,O1,MixMultiScale,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,MixMultiScale,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240326,,1939075.0351144117,15252.322912772186,51920.964751682877,50320.268142707035\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240327,,6938415.1342022521,18457.458372562192,52180.247404336391,49225.534056469318\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240328,,5065536.2235877197,13472.015084354207,44251.685621000513,58317.560738124739\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240329,,3227173.4257188258,16281.534218251705,51110.761225878829,39845.923320917696\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240330,,2224486.4072194281,12207.351383010857,34244.081190997938,25654.278688173959\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240331,,908538.0568424135,9455.8110577650368,34411.10083033191,30079.315968049697\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240401,,1527914.058976874,23997.727572218515,61516.411221963004,46909.759404387485\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240402,,814301.3485448549,12762.681229684502,36023.97853971101,29220.625472615102\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240403,,665195.27990854229,12148.31399904564,40969.151559860788,36552.468630720716\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240404,,657384.63858444081,13907.958368931897,34359.786937722522,27822.351034851581\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240405,,928538.22751602298,16025.066933368333,50450.849943816793,36258.753857405929\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240406,,734149.38895934657,10275.888391570188,41762.51789552578,32368.343360114359\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240407,,800513.93769363477,11683.90387099795,46049.536228983838,30730.816508667736\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240408,,759689.47824863275,11556.658050897531,41374.653081652672,27048.862548399593\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240409,,643114.07862822223,10383.963227652013,29088.932098732836,27000.785843738391\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240410,,834636.9029775667,13382.962602848187,38505.596431748425,31400.147287860058\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240411,,705204.03965030913,12158.448896913789,46479.698511567185,31354.680694516337\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240412,,1262611.6722604136,14465.242472061887,55114.711032637475,42407.722528357117\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240413,,1071016.1182704286,17100.846157398075,56173.231821880167,51966.974178969496\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240414,,1203469.0839617751,17144.958054805174,46723.435076876463,30136.495640622776\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240415,,1751316.2792605942,16044.774891392328,44124.032564871304,33638.4602424314\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240416,,1231333.4254453362,13907.334313455038,34822.354015825251,28601.93213172855\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240417,,2199961.7662741123,14884.4645368075,46369.598839398415,33990.361100508781\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240418,,2326626.0357530168,17599.509475487284,48419.550771873488,35330.66771315004\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240419,,1378554.5442612132,14467.516768579371,45215.668000255238,34099.097110738556\n'
    'daily_sse,,,O1,MixMultiScale,,19600,,,,,,20240420,,854348.62128404691,14728.435052837245,46900.894187073725,32021.740556667446\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240326,,1938836.4822894393,15252.312441175804,51920.612875456027,50325.188130612245\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240327,,6938562.8876992129,18457.441621850245,52175.639748499889,49219.62683868828\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240328,,5065779.880629817,13472.00159889739,44250.711837930023,58309.827899425567\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240329,,3227431.7187144407,16281.520399768837,51112.837740663803,39839.138201512018\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240330,,2224677.3929210142,12207.346853253432,34233.848657024959,25653.337814321119\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240331,,908676.81224367698,9455.8179947538301,34406.11297356632,30077.631880659086\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240401,,1528014.6061549801,23997.700786067173,61493.405556274993,46901.994582040061\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240402,,814294.13194410806,12762.686095912009,36025.508716090393,29217.202495682017\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240403,,665146.93515287433,12148.30721361842,40964.742817174076,36552.033492618997\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240404,,657387.99598627072,13907.962444843724,34360.636829046089,27820.419872079448\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240405,,928675.53674867889,16025.070606797934,50430.442443486572,36256.987708962602\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240406,,734219.6109126925,10275.885777855292,41753.099504807855,32372.300962580506\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240407,,800546.15250891633,11683.892897467129,46038.539368744823,30725.339005176713\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240408,,759711.71612659446,11556.665190404281,41375.835296136516,27053.267560192289\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240409,,643220.72485010372,10383.956973111257,29080.464161507294,26993.956827346759\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240410,,834670.80470673065,13382.941539894789,38495.130999578883,31395.569268279145\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240411,,705266.8406355707,12158.442529289052,46463.116895876119,31345.255689557394\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240412,,1262562.8034942632,14465.234416612424,55112.276725211326,42401.350792905287\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240413,,1071012.1390680203,17100.834408967756,56165.569283175791,51963.418691691775\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240414,,1203408.092388242,17144.94629399199,46711.296202324243,30131.408776258908\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240415,,1751230.7216165736,16044.76651918143,44114.94549226384,33637.242141435796\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240416,,1231461.4594563325,13907.311717083678,34813.872354524123,28591.936423241226\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240417,,2200272.9696837272,14884.457405580208,46383.078199779498,33995.559852030623\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240418,,2326850.7967209187,17599.469533240423,48415.025729394823,35329.254479437921\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240419,,1378592.1022378958,14467.500889376737,45206.957792471163,34104.351388581024\n'
    'daily_sse,,,O1,MixPhysicsNewBias,,19600,,,,,,20240420,,854400.1305514581,14728.421702438034,46896.638110548141,32016.61228079578\n'
    'daily_sse,,,O1,MixRecentBias,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,MixRecentBias,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240326,,1939622.7503876528,15252.242097876035,51908.990150527294,50327.947216290631\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240327,,6937723.667459501,18457.382928092964,52212.709752116389,49201.908507704284\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240328,,5065526.9239161387,13471.924412521534,44369.944108149211,58406.520831636488\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240329,,3227086.1245567976,16281.343112742528,51130.638981668919,39854.88757413259\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240330,,2225535.1484815003,12207.405226414092,34276.697260997098,25680.827540420942\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240331,,908694.8217970368,9455.7681075520813,34410.294329603756,30091.42024968801\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240401,,1525726.1504821284,23997.944020060822,61537.938357570471,46847.863205419751\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240402,,814450.18771824962,12762.647437116131,35988.07536470145,29210.253258810309\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240403,,665109.78022779757,12148.318110838532,40911.273271104074,36529.185817419166\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240404,,656994.66221146029,13908.095355025493,34338.429213785224,27793.199816481625\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240405,,928913.90889949258,16025.180174996145,50486.08276351625,36324.503243260231\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240406,,733810.68563158996,10275.878657151945,41707.052622873758,32414.210604579817\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240407,,800860.02858787449,11683.871936750598,45986.775928909454,30699.595836822806\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240408,,759326.62445746246,11556.652459130622,41328.866432422546,27027.653461218986\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240409,,642690.17065545637,10383.889142843895,29077.659180159415,26959.359842101076\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240410,,833615.74478925392,13382.848175883293,38454.479026014822,31327.586926650623\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240411,,704773.55870674411,12158.389347231947,46499.576472234046,31325.384077821429\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240412,,1261778.2735766361,14465.105418583378,55159.355190182658,42463.906633704384\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240413,,1071130.5468901226,17100.793737308122,56172.763718030481,52034.702488014067\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240414,,1204018.5754642449,17144.908677858301,46736.278297195713,30110.872065620784\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240415,,1753308.3531135311,16044.774559882469,44089.290959938662,33610.57102309035\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240416,,1231307.7429438944,13907.177775972523,34774.291012160567,28517.064086428007\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240417,,2198799.9324510572,14884.186207517982,46346.711653712584,33940.491620428911\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240418,,2323470.0731010893,17599.31177093368,48422.915383536267,35331.357062811723\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240419,,1377404.3635520856,14467.472865154967,45176.375676904601,34095.846031398891\n'
    'daily_sse,,,O1,MixRecentBias,,19600,,,,,,20240420,,854690.4049470867,14728.576708025299,46930.645344786273,32059.790499249226\n'
    'daily_sse,,,O1,MixTaper,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,MixTaper,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240326,,1939037.8264575955,15252.259679187089,51917.279094402293,50321.500068768313\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240327,,6939227.8001773451,18457.458404668607,52189.35899853304,49220.688106146707\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240328,,5066289.6542726345,13471.969111002982,44245.087697447576,58311.621271711243\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240329,,3227622.0459657824,16281.358413123526,51092.181840832891,39839.018034376291\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240330,,2224864.0972437453,12207.203585372306,34245.613328632178,25650.471486528928\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240331,,908541.35727276513,9455.7300707967952,34408.430900288637,30068.543590458601\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240401,,1528355.7997110654,23998.058243832551,61507.297561615815,46930.170771296514\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240402,,814168.34164491598,12762.451687377878,36014.185331403052,29204.942266065224\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240403,,665234.99072671402,12148.294446889311,40946.190068043114,36563.735651987641\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240404,,657316.27902486245,13907.616373379715,34356.459997080499,27814.716252455928\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240405,,928402.96572944825,16024.745084309019,50483.158097307838,36254.044348203774\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240406,,733880.49324289663,10275.686578226276,41770.430473586595,32351.922073800994\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240407,,800182.28837027983,11683.66795166675,46056.930846333169,30730.385088264826\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240408,,759938.897565461,11556.523235532455,41354.267157810646,27044.642945677831\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240409,,643089.39456298901,10383.93739185296,29106.146209627521,26995.277515348473\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240410,,834601.04542046157,13382.810414568521,38527.216090994087,31399.008657039314\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240411,,705193.23175751301,12158.332519924268,46517.743125426736,31367.406921867372\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240412,,1263128.0741816752,14465.160169109702,55152.620175976408,42412.787875871916\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240413,,1071472.6675110983,17100.620144148357,56219.535326894067,51983.126712576588\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240414,,1203790.5324955836,17144.715490570292,46786.995628566146,30138.242054204846\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240415,,1751559.6481380062,16044.296936242841,44157.972193554633,33629.484078021458\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240416,,1231487.1414794978,13906.956756101921,34855.007827324458,28603.005009706205\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240417,,2200445.137759224,14884.179125718772,46328.295228279829,33954.403230329503\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240418,,2327413.7950940174,17599.724135171622,48440.18130997759,35330.231961391975\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240419,,1378966.7578452758,14467.161168836057,45235.290594846185,34056.155334058138\n'
    'daily_sse,,,O1,MixTaper,,19600,,,,,,20240420,,854547.02864909591,14728.07284311112,46908.996642613893,32006.299602187551\n'
    'daily_sse,,,O1,MixWindFlow,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,MixWindFlow,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240326,,1938836.4733748361,15252.312608664855,51911.183884440805,50350.622275372574\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240327,,6938562.7165543744,18457.441599167883,52181.912873760615,49255.655294766693\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240328,,5065779.7006465681,13472.001610148698,44224.471472864156,58374.760959166364\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240329,,3227431.6516187587,16281.520373028703,51096.988073391272,39858.051789702535\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240330,,2224677.3530944525,12207.346903792582,34204.697482316195,25676.136244636869\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240331,,908676.7757049452,9455.8180119432509,34386.232637585097,30107.298163232004\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240401,,1528014.5412699243,23997.700800332241,61452.007314622831,46873.075681914794\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240402,,814294.15597528219,12762.686081805266,36009.504577129366,29199.092358354999\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240403,,665146.96164869051,12148.307372713462,40991.262031120947,36549.475871396804\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240404,,657388.01649325481,13907.962433089502,34383.696229821537,27817.910000177922\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240405,,928675.52443587943,16025.070606797934,50332.088825115876,36219.522249883274\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240406,,734219.64788816543,10275.88581660483,41725.416452357073,32386.967868060674\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240407,,800546.18283507065,11683.89298979193,45965.741858829366,30683.475122191419\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240408,,759711.74709004932,11556.665106095374,41379.034041709892,27064.247759240494\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240409,,643220.74036468798,10383.95699658338,29054.497825243019,26967.414178152707\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240410,,834670.77801325568,13382.941578568891,38446.491021432361,31392.336024100885\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240411,,705266.82005736511,12158.44252882991,46380.72031832338,31333.231188865946\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240412,,1262562.7556172682,14465.234416612424,54962.523002240981,42333.98171936685\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240413,,1071012.1314878233,17100.834392240271,56092.296204656064,51915.804880685188\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240414,,1203408.000838232,17144.946208707057,46670.824482106371,30123.679127981031\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240415,,1751230.7883951566,16044.766517211683,44080.969945843652,33624.105351581791\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240416,,1231461.4191758744,13907.311711216345,34728.545812315773,28574.468835283653\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240417,,2200272.7355905054,14884.457407741807,46427.839647607063,33988.684825127857\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240418,,2326850.4786072257,17599.46953397803,48344.917358560757,35316.861472305565\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240419,,1378591.9611809077,14467.50089579355,45155.723377061884,34140.400627397146\n'
    'daily_sse,,,O1,MixWindFlow,,19600,,,,,,20240420,,854400.06775857741,14728.421634302475,46888.12251063648,32006.819195442877\n'
    'daily_sse,,,O1,MultiScale,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,MultiScale,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240326,,2009611.1507574613,18462.27729103528,58989.321133633886,51096.932366473018\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240327,,6447538.8451562729,19209.301598900929,61681.076156673094,55036.564442798255\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240328,,4769951.1719953269,15271.242205103859,49242.662676718275,63253.895217776182\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240329,,3224134.0173589578,16517.831661371514,61795.812648813328,45021.117971582338\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240330,,2410265.9013891825,13044.649840446189,42404.083083510661,28917.527017278895\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240331,,984177.50231111958,10265.680884576403,38660.530318552985,33897.561229136707\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240401,,1429695.9536693175,27510.088630900718,71551.92019839221,58654.830022389862\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240402,,989553.38702840544,13084.273038705811,41118.962267263647,33785.56826973446\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240403,,842959.26511267922,14632.205637472682,50692.808342306344,39839.4365249193\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240404,,831207.98542439565,13367.380297171883,40002.634401980467,31370.560381235024\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240405,,1056512.8603427946,19028.471748993732,55008.745486332133,39770.275703562162\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240406,,879358.22844396695,11172.343884537928,47354.055236302811,32815.828859975816\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240407,,995074.61516328319,12207.490794259124,51703.75555475802,34302.563695971883\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240408,,908610.81809346098,11782.194754407741,43615.310375650799,29545.704036334399\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240409,,768336.02333568013,12367.735291553661,35940.155228214346,34095.223023827166\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240410,,980808.32665403164,15257.766162832268,44347.243188965156,38106.204313364709\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240411,,796128.04483069154,14629.478779075667,54621.561248350044,37773.711264330137\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240412,,1628191.9239727743,17616.787171088159,67067.759112053303,50477.722718204168\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240413,,1488536.3023215819,20308.648291140795,66813.721126289165,59612.753051156054\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240414,,1562578.5537559013,17603.3917666208,54612.429912470623,37102.082745832609\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240415,,1907832.1510751576,19729.790621519089,56219.871338262456,41650.738021056255\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240416,,1375312.8234688542,15202.944577810355,41061.626775067212,35291.970554451182\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240417,,2287878.0678909146,14506.130402927287,51267.954156126274,36704.035778345657\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240418,,3042615.3486753325,18527.161134147085,55230.9452696484,43098.110576694584\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240419,,1896845.5317330267,15398.980189787224,55365.741626667084,38213.066213360558\n'
    'daily_sse,,,O1,MultiScale,,19600,,,,,,20240420,,1133070.3885171879,14972.28021244891,54558.363151318401,37150.226056545347\n'
    'daily_sse,,,O1,PhysicsLegacy,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,PhysicsLegacy,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240326,,2035499.326291149,18587.92134034168,59016.820616750934,51253.867067493615\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240327,,6520437.5628744252,20689.20984631218,62513.93420241186,55370.583533886645\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240328,,4815908.3794653621,15845.12971208524,49198.574402004946,63191.631274190178\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240329,,3236660.4427036373,16220.593447376974,61117.150676721474,45177.359744578163\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240330,,2432068.7728582486,12682.274325916544,42905.977352494097,29070.448428005478\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240331,,978365.97898975783,9952.0260304389521,38980.8710513556,33743.39920195115\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240401,,1459176.2269090915,29967.284573218785,71615.153897473181,59547.929221569568\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240402,,985413.5992830277,12223.241349505261,41092.808768457355,33666.759493424848\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240403,,856544.57508237776,14696.716973240487,50226.44632534351,40331.464733120498\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240404,,821334.51658618241,12242.193995513022,39956.404104360954,31443.49277196702\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240405,,1048387.9655081923,17883.788061665371,56524.736739746331,39881.465271855042\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240406,,863911.62916179234,10641.374316067435,47603.525123010964,32537.131962012587\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240407,,977840.26052870089,11770.906912221573,52098.803083427651,34614.445736398455\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240408,,928825.83502446022,11500.858822106384,43148.408584231023,29728.744084826045\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240409,,767117.35439044773,12491.28573416546,36334.030020534774,34204.018230572154\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240410,,992351.74292810541,14875.340436469764,45146.244001818523,38342.231813870581\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240411,,811466.62776905624,14346.98664919287,55931.026187188349,38344.171908364318\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240412,,1665617.8630187046,17107.933809955604,68256.835976065689,50819.370164335851\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240413,,1537677.3095899043,19599.561442653649,68081.728806385945,60071.053730776875\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240414,,1609346.5008452749,16711.279258320108,56444.107143510693,37356.161853933503\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240415,,1937893.1710284639,18340.495681733824,57437.928115052549,41984.775501267657\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240416,,1386437.9943447907,14473.04022587277,41600.414724814225,35662.678715716313\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240417,,2310894.6976209912,14449.885652326979,50604.283394842838,36432.541377195979\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240418,,3099828.7829663549,19561.291960909963,55535.424444385091,43371.782117952112\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240419,,1929857.9294407368,14823.782652303576,56090.015904652777,37997.943627674053\n'
    'daily_sse,,,O1,PhysicsLegacy,,19600,,,,,,20240420,,1148013.4565646951,14365.354617592879,54857.743660177744,37375.789122657799\n'
    'daily_sse,,,O1,PhysicsNew,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,PhysicsNew,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240326,,2035500.0568197209,18587.897741416469,59016.771639242921,51253.859296885217\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240327,,6520444.0277065514,20689.199316730723,62513.873899646642,55370.608015290556\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240328,,4815915.0654460946,15845.136225903407,49198.554612459135,63191.697351303454\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240329,,3236662.3068411467,16220.590121973306,61117.167363944289,45177.372234813614\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240330,,2432070.0954241268,12682.274081435055,42905.897572728361,29070.418565434451\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240331,,978366.38156638364,9952.0127104856074,38980.83945364214,33743.38243369729\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240401,,1459177.9392108808,29967.236729305238,71615.124466946902,59547.892242377988\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240402,,985412.53223873419,12223.256948038004,41092.793517536964,33666.765769830803\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240403,,856544.0002200522,14696.705992686562,50226.407062832855,40331.45044769971\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240404,,821333.38998856512,12242.162798221223,39956.413487081183,31443.503996342744\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240405,,1048387.9702483309,17883.764502805658,56524.710858342725,39881.466627852147\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240406,,863909.93255772348,10641.371018744074,47603.510228888459,32537.140709134917\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240407,,977839.04811649956,11770.903440921567,52098.794208528532,34614.444912847677\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240408,,928824.91000872944,11500.844785728492,43148.353969723554,29728.730199476973\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240409,,767116.86934271175,12491.276464413851,36333.981858132574,34204.02010291726\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240410,,992353.06826201896,14875.343719015829,45146.227408555365,38342.256150814414\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240411,,811467.27117281849,14346.981493568979,55931.059375197758,38344.17604998067\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240412,,1665620.1896079974,17107.906939762644,68256.782078979944,50819.296638806161\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240413,,1537678.941410674,19599.549780382775,68081.713013822126,60071.049348481567\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240414,,1609353.4405220645,16711.286716192961,56444.052862517819,37356.154962177257\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240415,,1937896.1267415441,18340.416164042428,57437.831453081541,41984.716607938572\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240416,,1386440.0319665733,14473.053988669999,41600.452348181861,35662.719510326417\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240417,,2310902.3673211075,14449.894897307269,50604.338265309809,36432.553719041411\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240418,,3099842.9620848084,19561.288062095642,55535.441214866412,43371.7731702526\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240419,,1929867.6655475211,14823.789852268063,56090.055005887938,37997.935595043658\n'
    'daily_sse,,,O1,PhysicsNew,,19600,,,,,,20240420,,1148016.8866541758,14365.358856258914,54857.805613621698,37375.793925885751\n'
    'daily_sse,,,O1,PhysicsNewBias,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,PhysicsNewBias,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240326,,2002378.1121150777,18434.958288353868,58918.669554971035,51127.554083333089\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240327,,6449608.7077284269,19192.654085222632,61496.339654564843,54960.230708019473\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240328,,4774991.0677039241,15261.906348974444,49231.063236736292,63161.945453409659\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240329,,3230787.9984694663,16492.405730210245,61774.377231665567,44970.947506937438\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240330,,2414581.5227373065,13031.769436483271,42271.670006413711,28922.115825604957\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240331,,985871.5282402141,10272.134697485715,38578.192090230958,33857.444468438705\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240401,,1430356.843034927,27471.150614282116,71286.220081222564,58550.839838227112\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240402,,987150.91720318934,13084.750137480907,41077.954200795357,33737.713547357671\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240403,,839710.69103785511,14617.811237769201,50631.699457740717,39808.091514171225\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240404,,831195.92373696389,13379.771804778837,39985.846720751208,31348.449307270079\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240405,,1059594.7500583776,19022.351996907964,54754.250436551411,39752.014547926679\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240406,,881153.22342013777,11169.982849146239,47265.777285993157,32853.270422540525\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240407,,996558.3866845048,12187.79241497349,51564.369719839728,34244.249655256921\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240408,,909257.39775539329,11783.336998867802,43612.251234657801,29583.230639051537\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240409,,770640.98490635282,12360.104407709092,35826.078753366593,34007.471377894595\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240410,,980779.03367424058,15215.897718618624,44198.099953006196,38011.461210127803\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240411,,796513.1995442824,14613.199641255662,54395.290713414295,37625.946915052067\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240412,,1626486.692872239,17597.765686738305,66999.321873315494,50363.182779363931\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240413,,1487192.6670563282,20288.711748547852,66747.403680868156,59517.886404834127\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240414,,1559584.5213113078,17583.931046719663,54469.275506007456,36996.304198884645\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240415,,1904244.1644022353,19725.236630846746,56069.066588702466,41565.644564872273\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240416,,1377601.1029071321,15177.509888146073,40979.804561911413,35147.447026316069\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240417,,2294339.2486593858,14505.706150838174,51358.663123438615,36700.562055408489\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240418,,3045621.7065704861,18467.995685150847,55176.084125261812,43012.603531611283\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240419,,1896656.6908050778,15376.521880432032,55205.93816947045,38210.093066371766\n'
    'daily_sse,,,O1,PhysicsNewBias,,19600,,,,,,20240420,,1133787.4929575778,14943.423871168867,54455.455909397861,37033.334827743944\n'
    'daily_sse,,,O1,Placebo,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,Placebo,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240326,,2151038.6883350015,18712.53185016755,59491.603867913727,50420.399747186588\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240327,,6104937.6901438804,18579.770351143554,60319.371730236846,52546.308633783949\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240328,,5088242.1529876385,15461.590570649132,51231.400900960121,61422.421896472566\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240329,,3253449.042365633,16420.938575762324,61444.378057823174,46962.983688137501\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240330,,2317593.9190805634,12589.035749422386,42488.246790668651,29700.996095449846\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240331,,1029563.1034840015,9824.5066941902041,41072.845274743231,35142.836428524832\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240401,,1486172.7877150669,28763.197784384713,74768.511077543619,59804.243111701013\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240402,,1098083.2310846476,11326.906951570883,43216.592640013791,34437.822214938176\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240403,,957713.19162341068,15132.300648646429,52064.28722672478,41739.946607620921\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240404,,922412.206555882,11782.573653895408,41314.405953255467,31433.470745166047\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240405,,1132441.3565623302,16259.890685170889,56453.582919201392,39280.848235164543\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240406,,881004.52945443336,10727.815206520259,48803.26623213011,34642.72970003746\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240407,,981660.55999115738,11745.515463104472,52063.357279367614,34323.670781720932\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240408,,933634.05863804743,11238.334832318127,43803.015533300291,30909.445440432195\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240409,,851936.2395387427,13061.395221261308,38253.047919808123,34185.311635286205\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240410,,1009359.6347824866,14543.383759887889,47042.985961266248,38072.302105786322\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240411,,852593.58995306049,13410.032762894407,58234.998952429247,37938.3303079787\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240412,,1455453.7160231543,17456.939118989743,70749.021453062698,51860.876746504786\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240413,,1264673.6589552986,18043.6617981866,71311.603664669165,58716.386215584367\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240414,,1472140.4236013247,16362.079778257757,60448.941083681137,39068.322424215869\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240415,,2019715.2693542163,17982.269986950792,58490.283574453497,42846.079389702616\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240416,,1379969.1285102777,13811.681645990349,42103.691253619712,33675.776508930852\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240417,,2032951.7683221342,13372.355218622833,52310.938753846087,35091.90982171434\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240418,,2641770.2561308946,19205.839796156622,58455.462007684706,43066.603555602705\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240419,,1651168.7731013678,13502.686641753651,57450.573963455092,37215.250763910735\n'
    'daily_sse,,,O1,Placebo,,19600,,,,,,20240420,,1109736.9240189961,14720.993237767369,55393.051125084297,35859.777789984888\n'
    'daily_sse,,,O1,Recent,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,Recent,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240326,,2051624.9799886714,18496.122388767079,59068.071383404131,51593.092087140365\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240327,,6492155.7957333708,20680.278478353284,63192.868744001164,54964.65584622226\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240328,,4790339.4642995298,15732.667311620899,51038.86889076652,63926.496013046431\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240329,,3213430.8246534285,15925.997582244687,61852.653346575738,45394.976987044487\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240330,,2449583.9365873029,12759.178471348248,43603.056531079237,29441.499001391785\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240331,,983567.398283866,9862.6107258666307,39450.198613536988,34245.842480004139\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240401,,1427800.2959160532,30652.917302500457,72539.995142266998,59369.047555492776\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240402,,986923.26913591148,12197.022104455158,40918.681160762557,33571.580099644365\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240403,,847297.06381534925,14632.708746626973,50186.27765124906,40097.358260711822\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240404,,808685.48116682842,12269.917546196841,39487.686949844894,30858.537131394212\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240405,,1055398.3103336324,18062.62480449304,57045.780558168342,40729.341893434175\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240406,,850337.39494983712,10574.908672798425,46934.468376493125,32905.284378418772\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240407,,978164.56710084993,11700.738539323211,51435.473150186212,34165.821189609778\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240408,,910726.59760641353,11441.650837886147,42547.816842438253,29285.35294082934\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240409,,749500.08423028584,12472.235871082172,36471.887647717791,33606.296558275673\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240410,,970310.40818222705,14702.489503520541,44871.682442762605,37509.397162252339\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240411,,810170.77446236555,14264.79043974448,56771.19693971856,38165.785036695161\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240412,,1641106.3287660691,16909.713679552078,69072.721168067306,51625.916558874982\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240413,,1554234.2542935624,19736.015118482523,69170.140197679779,60959.382545579167\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240414,,1617086.5786040467,16727.995947184041,57169.363994283485,37326.213092802456\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240415,,1970615.3817683032,18252.308785232715,57934.405299067424,41463.007542567881\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240416,,1392094.5402250376,14287.823674280196,42023.190350428806,35096.386773478444\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240417,,2262698.3888472808,13973.936300172471,50329.349325170115,35540.10525127517\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240418,,2965138.220303162,19218.513891910203,56003.489507149337,43113.176146742691\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240419,,1870805.8724810886,14719.644458054565,55911.994057668227,37911.21326054872\n'
    'daily_sse,,,O1,Recent,,19600,,,,,,20240420,,1143576.7377747223,14524.342563793063,55426.465729181793,37633.941523678339\n'
    'daily_sse,,,O1,RecentBias,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,RecentBias,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240326,,2016845.732725424,18362.003457673825,58968.486420148125,51547.810979211135\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240327,,6431199.831984655,19185.568975383416,62173.544507283572,54654.75438445295\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240328,,4757913.625611119,15218.903626102954,50959.867230239579,63919.08023797108\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240329,,3215708.9375748662,16245.210244171321,62405.287057443675,45178.949791457781\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240330,,2437713.0795014454,13158.476311258972,43138.228899645663,29333.328068528939\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240331,,991115.40478435485,10226.302243630402,39109.767178309979,34375.962132273446\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240401,,1402777.6704741905,28149.698576275259,72110.854945541592,58445.01601715952\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240402,,990878.82773395535,13127.970694706775,40896.150289331519,33709.02514822028\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240403,,831471.86089673522,14605.956990568899,50406.492656244918,39655.863873610811\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240404,,819099.16061650775,13508.376997050829,39492.418230031049,30837.598515279813\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240405,,1065655.9133619615,19270.323327971622,55554.492330266941,40570.655459790083\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240406,,867427.98229622422,11166.859640703537,46678.622500160564,33210.569138696213\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240407,,998411.98030371335,12136.6423593238,50964.280405025558,33857.492388406929\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240408,,894123.66484027868,11761.835546862334,42886.814127226404,29167.770380165122\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240409,,755142.35760404961,12344.656876020133,35986.123965581457,33433.606750957355\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240410,,961799.75034640101,15107.85935718473,43991.477940192512,37248.350763030852\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240411,,795607.73631409998,14594.841627103277,55290.526451809776,37459.709541478209\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240412,,1604609.9200590225,17428.169111061841,67893.603175944707,51140.951526268727\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240413,,1505905.606778109,20393.866258061491,67997.046961081011,60368.343239553884\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240414,,1568001.1671095141,17673.180729969405,55448.124900151925,36949.137082020039\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240415,,1933511.1868214903,19743.806591736153,56651.523311813944,41116.831156053486\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240416,,1387501.5748199869,15016.271430894732,41429.353753301519,34604.330764506114\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240417,,2255938.1817909656,14052.889326863922,50920.940193718088,35799.394018634957\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240418,,2921586.0140511701,18148.801352406852,55591.735862415582,42750.234933599335\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240419,,1845352.0479152657,15323.846908023581,55130.356232614722,38135.644556465035\n'
    'daily_sse,,,O1,RecentBias,,19600,,,,,,20240420,,1133506.853114726,15167.340164412744,54881.492727267774,37267.148118502635\n'
    'daily_sse,,,O1,Reference,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,Reference,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240326,,1938836.4733748361,15252.312608664855,51920.616874443404,50325.188138002581\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240327,,6938562.7165543744,18457.441599167883,52175.643136386374,49219.62507909313\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240328,,5065779.7006465681,13472.001610148698,44250.713111497469,58309.820183045689\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240329,,3227431.6516187587,16281.520373028703,51112.834503383638,39839.135998178761\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240330,,2224677.3530944525,12207.346903792582,34233.850792388977,25653.339929474125\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240331,,908676.7757049452,9455.8180119432509,34406.113736160172,30077.632825520865\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240401,,1528014.5412699243,23997.700800332241,61493.407379165525,46901.99790354522\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240402,,814294.15597528219,12762.686081805266,36025.508802965822,29217.201266284639\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240403,,665146.96164869051,12148.307372713462,40964.746701378812,36552.035158982166\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240404,,657388.01649325481,13907.962433089502,34360.637939021995,27820.41927800417\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240405,,928675.52443587943,16025.070606797934,50430.443689009458,36256.986150318677\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240406,,734219.64788816543,10275.88581660483,41753.099367477509,32372.299095171969\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240407,,800546.18283507065,11683.89298979193,46038.539731828496,30725.339117787975\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240408,,759711.74709004932,11556.665106095374,41375.839332088595,27053.268353098691\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240409,,643220.74036468798,10383.95699658338,29080.467312864334,26993.956490648918\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240410,,834670.77801325568,13382.941578568891,38495.131187174069,31395.566856317953\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240411,,705266.82005736511,12158.44252882991,46463.114022659822,31345.255526443518\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240412,,1262562.7556172682,14465.234416612424,55112.279368603864,42401.356447583334\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240413,,1071012.1314878233,17100.834392240271,56165.571124602335,51963.418640675343\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240414,,1203408.000838232,17144.946208707057,46711.300132064302,30131.410422724413\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240415,,1751230.7883951566,16044.766517211683,44114.947155730697,33637.250372822498\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240416,,1231461.4191758744,13907.311711216345,34813.871283551256,28591.93458557564\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240417,,2200272.7355905054,14884.457407741807,46383.075439250017,33995.559935344827\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240418,,2326850.4786072257,17599.46953397803,48415.024256185032,35329.256213506123\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240419,,1378591.9611809077,14467.50089579355,45206.957288709673,34104.354686266328\n'
    'daily_sse,,,O1,Reference,,19600,,,,,,20240420,,854400.06775857741,14728.421634302475,46896.637244919657,32016.616216399183\n'
    'daily_sse,,,O1,Taper,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,Taper,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240326,,2007594.1284001977,18433.333861925639,58913.105969395649,51136.201261700444\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240327,,6462648.1315762755,19450.502788887359,61665.883168657514,55022.058229660368\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240328,,4783242.5199025301,15351.95342506934,49187.337435798661,63149.880695785258\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240329,,3231230.8102735472,16382.000228168443,61568.467264812367,44998.000758984752\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240330,,2418444.0469459137,12893.126207458787,42404.69479050026,28939.638559409788\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240331,,982745.75420982903,10128.695961534977,38657.360851511352,33799.850386633887\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240401,,1438557.2699954896,28221.804186348803,71346.639151727664,58847.387609594989\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240402,,985533.36209000903,12738.002542306669,41034.06973724749,33687.210819777698\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240403,,844773.88479620195,14594.056382903829,50438.67989787773,39972.116640631328\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240404,,826426.03894129419,12887.607556411065,39924.28498871436,31356.132266872944\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240405,,1053927.557973648,18506.000077760778,55421.803793646286,39775.695732065615\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240406,,872778.88650969509,10888.672149506398,47357.667185628976,32688.501545259616\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240407,,987137.87025406724,11944.716579821892,51747.289254053365,34378.834988671384\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240408,,917190.04545266088,11597.394522133283,43345.082920194516,29621.455799171541\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240409,,767801.18750114506,12366.315150078386,36017.422269487019,34073.371060017198\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240410,,985253.84637357318,14989.150274406187,44614.36853206114,38147.005540870508\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240411,,802990.64578596922,14419.40914462693,55128.528591868984,37964.924503942806\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240412,,1645885.1291257539,17279.192549862899,67606.716544345967,50574.048511630383\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240413,,1513336.065910876,19856.10407559853,67417.556077231333,59787.056525874024\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240414,,1586167.5369160608,17042.840357454494,55520.371587748406,37168.190136951547\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240415,,1922243.2820525826,18877.868647528812,56804.113615230242,41776.513800825494\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240416,,1381644.7231722886,14709.050484267063,41292.950840035053,35420.53681925081\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240417,,2303117.3030975517,14417.990692574531,50853.375008492949,36510.728014126122\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240418,,3077616.5268620122,19078.415694527328,55347.3124216492,43202.653758505316\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240419,,1916216.7823978306,14975.026352452114,55711.746717087168,38048.800890086743\n'
    'daily_sse,,,O1,Taper,,19600,,,,,,20240420,,1141686.4636624316,14517.377402530052,54663.689006395303,37222.897684457894\n'
    'daily_sse,,,O1,Ultralisk,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,Ultralisk,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240326,,1948352.2156399279,15253.453288236633,52304.493248337625,51160.618219449432\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240327,,6979008.8126318138,18460.168470378965,52673.563503151003,49826.641991045384\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240328,,5102734.7802604754,13473.631217166781,45008.405525054477,58759.563791753229\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240329,,3240142.9208618565,16283.408239237964,50966.056466502618,39964.988163973729\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240330,,2226244.6795096621,12208.321515399963,34148.745829993131,25808.701304357066\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240331,,911816.48723164294,9456.2364151896909,34662.230172255462,30186.261275795478\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240401,,1546793.7645329225,23997.865431836806,62074.824064153479,46681.345381779996\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240402,,816935.79855565657,12764.049921499565,36281.914010088782,29284.7585208618\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240403,,667897.38823491917,12148.989692261443,40883.687452287413,36796.673599547132\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240404,,657303.09182674112,13909.445667606778,34464.038515933527,27988.447092294879\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240405,,931116.57785837096,16024.718921774067,50688.186589498713,36379.992433787389\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240406,,735150.12630567537,10276.624113520607,41831.328285637675,32764.895331861931\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240407,,798993.17250393331,11684.5611349307,46043.311008145116,30732.184589871213\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240408,,760526.42224226147,11557.471586091444,41759.085348938715,27199.038256581916\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240409,,644928.02573760739,10383.951169867069,29149.594594799393,26780.56382922384\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240410,,837194.24552558572,13383.064765988849,38612.715515798838,31214.136237916569\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240411,,709476.8964830488,12158.600762987509,46426.13072437128,31292.277077756855\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240412,,1257683.8031370114,14464.937095826492,54738.181973313505,42204.265923977298\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240413,,1066376.6138813347,17100.751628737897,55980.477015507087,51905.127006025308\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240414,,1204609.5034678895,17146.700429707766,46894.45387842496,30076.30065663475\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240415,,1763257.1556913953,16044.960562514141,44305.867548776849,33902.538743928351\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240416,,1234914.9729790341,13908.425631066784,35133.884065221726,28599.194881418774\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240417,,2205305.2259671986,14886.204377843998,46619.876743683628,34404.713296595059\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240418,,2315905.2568944772,17600.706471969374,48761.315673809913,35305.165445839462\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240419,,1374962.9966683979,14468.628009521402,45033.539147614043,34350.396598947525\n'
    'daily_sse,,,O1,Ultralisk,,19600,,,,,,20240420,,857520.02616600203,14729.85291958414,47098.929384673371,32217.358313185599\n'
    'daily_sse,,,O1,WindFlow,,0,,,,,,20240324,,0,0,0,0\n'
    'daily_sse,,,O1,WindFlow,,0,,,,,,20240325,,0,0,0,0\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240326,,2002377.6610131201,18434.98239208851,58454.1058272839,50767.55899909624\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240327,,6449604.2809615321,19192.663245436735,60661.118830755673,54567.722132740899\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240328,,4774986.130061578,15261.906003136188,48215.381365239024,63117.35291844653\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240329,,3230787.3561641318,16492.409104612656,61095.990469674754,44726.898769336993\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240330,,2414581.3607326411,13031.773918076418,41549.367689687955,28867.243234963138\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240331,,985871.38769529271,10272.153201642446,38044.351121170104,33936.074063083026\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240401,,1430355.845491301,27471.190775632858,70207.370025292024,57682.194872241635\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240402,,987152.03369225725,13084.732985514216,40561.375044179127,33334.285316209018\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240403,,839711.25449227821,14617.820008946583,50550.899221051761,39520.517895965997\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240404,,831196.95980604133,13379.805533475243,40079.031456130979,31118.700312484521\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240405,,1059594.4335046522,19022.380273895338,53374.32155621241,39176.077067809092\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240406,,881154.54083087016,11169.984105110168,47045.952491418648,32923.309391285031\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240407,,996559.40106255724,12187.794293062761,50640.072537549357,33768.324570266494\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240408,,909258.43828718364,11783.345600411296,43802.781738189711,29551.476852235312\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240409,,770641.51640610024,12360.108786926605,35567.17763184019,33754.807645661997\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240410,,980778.08454983798,15215.89538377244,43590.560636423761,37821.988163749156\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240411,,796512.84107938875,14613.206807507202,53043.272983948431,37239.441110388274\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240412,,1626485.1990895043,17597.796479961835,64716.482708564137,49351.067786397944\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240413,,1487191.935394614,20288.730981643312,65424.229386168168,58864.845768736115\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240414,,1559578.4291739529,17583.924987359904,53363.616095237172,36583.481906327448\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240415,,1904241.9179438099,19725.321778546087,54546.944307460777,40859.761002917818\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240416,,1377599.6613769094,15177.49651942309,40326.204759306151,34615.254494639456\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240417,,2294332.9380652476,14505.699504550546,51817.271989429377,36457.049331949922\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240418,,3045609.306476362,18467.993333205581,54075.498689682579,42490.756848079945\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240419,,1896648.2934805839,15376.513405703008,54319.791834459051,38204.888148502047\n'
    'daily_sse,,,O1,WindFlow,,19600,,,,,,20240420,,1133784.7944672802,14943.41986258328,53824.636986082485,36497.082100708045\n'
    'global,,,O2,Ultralisk,0.4739908101395246,504700,8.7765020353427872,0.9221673444868832,1.6343897188209586,1.4634905909285587,,,,,,,\n'
    'global,,,O2,PhysicsLegacy,0.44483710974226509,504700,9.2389904304839483,0.96388021190353279,1.7494351481932322,1.5370033832409138,,,,,,,\n'
    'global,,,O2,Bias325,0.45018895228745959,504700,9.0650943461856954,0.94978276949130225,1.7389193757955566,1.529077248477118,,,,,,,\n'
    'global,,,O2,Reference,0.47614046166578627,504700,8.7135994134036814,0.91189151493464937,1.6337014357073223,1.461642105019334,,,,,,,\n'
    'global,,,O2,PhysicsNew,0.44483721532860732,504700,9.2389908292965437,0.96387960643000881,1.7494349088071666,1.5370032674019234,,,,,,,\n'
    'global,,,O2,PhysicsNewBias,0.45018904465459131,504700,9.0650949277040151,0.94978226270199517,1.738919175401012,1.5290770978292831,,,,,,,\n'
    'global,,,O2,Placebo,0.4047216405878154,504700,9.3237920863656871,1.0271743389245758,1.9515447381701096,1.6434467681331852,,,,,,,\n'
    'global,,,O2,Recent,0.44957274728966867,504700,9.0783629101266765,0.9499755651405728,1.7462126897813268,1.5272494129353658,,,,,,,\n'
    'global,,,O2,RecentBias,0.45448147595779148,504700,8.939062306251838,0.93749848386667201,1.7354265289815691,1.5187484773689228,,,,,,,\n'
    'global,,,O2,Drift,0.44336532243862492,504700,9.3795725052146963,0.95256688168975856,1.7515302476648955,1.5475234624079677,,,,,,,\n'
    'global,,,O2,DriftBias,0.44818874905946599,504700,9.1644463820334021,0.94257850381176156,1.7436757904553124,1.5411752749401595,,,,,,,\n'
    'global,,,O2,MultiScale,0.44968929163830018,504700,9.0700061774174952,0.95089576797801523,1.7404312468847245,1.530541473644367,,,,,,,\n'
    'global,,,O2,AdaptiveFlow,0.45240718297664839,504700,9.0163189817877551,0.94636584872840801,1.7311951965435088,1.5240454200812199,,,,,,,\n'
    'global,,,O2,WindFlow,0.45151272254684521,504700,9.0650943461856954,0.94978276949130225,1.7311951965435088,1.5240454200812199,,,,,,,\n'
    'global,,,O2,Taper,0.44791727155015371,504700,9.1378203863941767,0.95595505629674038,1.7432592262657367,1.5324371714139866,,,,,,,\n'
    'global,,,O2,MixPhysicsNewBias,0.47614047435717111,504700,8.7135994413560134,0.91189141184742817,1.6337014350592232,1.461642101450837,,,,,,,\n'
    'global,,,O2,MixRecentBias,0.47681830757045762,504700,8.7028436061924488,0.907660564969068,1.6336301995254865,1.4611996038017092,,,,,,,\n'
    'global,,,O2,MixDriftBias,0.4767692217720868,504700,8.7100235002790445,0.90820100407107518,1.6330714650280227,1.4609077692748285,,,,,,,\n'
    'global,,,O2,MixMultiScale,0.47609794659001858,504700,8.7136250627430911,0.91216615194339601,1.6337207230729529,1.4617023581958546,,,,,,,\n'
    'global,,,O2,MixAdaptiveFlow,0.47633293959021888,504700,8.7071594144387578,0.91107947578029325,1.6335846805813008,1.4614768445301911,,,,,,,\n'
    'global,,,O2,MixWindFlow,0.47617115349163441,504700,8.7135994134036814,0.91189151493464937,1.6335846805813008,1.4614768445301911,,,,,,,\n'
    'global,,,O2,MixTaper,0.4758399800370301,504700,8.7201265503213126,0.91366206400988925,1.6337696670634052,1.4617709919792521,,,,,,,\n'
    'weekly,,,O2,Ultralisk,0.56599146321781413,58800,6.8088931507836365,0.80152753921089215,1.3802323415079636,1.1773221521822299,1,,,,,,\n'
    'weekly,,,O2,Ultralisk,0.53997204848601021,134750,7.5179640754232366,0.77468645501196698,1.4769707292314005,1.2879347372395515,2,,,,,,\n'
    'weekly,,,O2,Ultralisk,0.46888305857762808,137200,9.1896199253805673,0.91143963361068403,1.6454325501076037,1.4743695550819915,3,,,,,,\n'
    'weekly,,,O2,Ultralisk,0.4186049150119896,134750,9.8475208588961198,1.015783433080917,1.794888674330545,1.6180551401038319,4,,,,,,\n'
    'weekly,,,O2,Ultralisk,0.3671818136299117,39200,9.9486327449463339,1.2115122556811246,1.8669242779809156,1.7851206338549734,5,,,,,,\n'
    'weekly,,,O2,PhysicsLegacy,0.53891587430952392,58800,7.3548844182839295,0.83125676590377284,1.4818014854360333,1.2492284939860594,1,,,,,,\n'
    'weekly,,,O2,PhysicsLegacy,0.50977636651079139,134750,8.0617324887993931,0.82942319415383203,1.5788744513013857,1.3605384773524274,2,,,,,,\n'
    'weekly,,,O2,PhysicsLegacy,0.43770568890966832,137200,9.8956507915770402,0.96517132653948401,1.758969163891811,1.5337598837431563,3,,,,,,\n'
    'weekly,,,O2,PhysicsLegacy,0.38872982982971083,134750,9.9814342250962849,1.0583286505609095,1.9254664584221772,1.7116883926774156,4,,,,,,\n'
    'weekly,,,O2,PhysicsLegacy,0.34417831470611171,39200,10.451702171586875,1.2040672391554634,1.9920636632444348,1.8457574927206191,5,,,,,,\n'
    'weekly,,,O2,Bias325,0.54149002074436692,58800,7.2683615527369998,0.82512179288792342,1.4753200627131446,1.2461105626231159,1,,,,,,\n'
    'weekly,,,O2,Bias325,0.5138749929036931,134750,7.9619778605690854,0.82071489512210494,1.5674730158474199,1.3522168910004904,2,,,,,,\n'
    'weekly,,,O2,Bias325,0.44354175548859731,137200,9.6433244672610137,0.9510286397168296,1.7497788535602261,1.5272588872196851,3,,,,,,\n'
    'weekly,,,O2,Bias325,0.39475543153172521,134750,9.8387482792235996,1.0399254164245335,1.9132861946437172,1.7012171446687874,4,,,,,,\n'
    'weekly,,,O2,Bias325,0.35225035914276992,39200,10.130473733383177,1.1809603145315295,1.9803231625488933,1.8373068452876511,5,,,,,,\n'
    'weekly,,,O2,Reference,0.56728230246985778,58800,6.7897089458490836,0.7929434868923757,1.3801085926089935,1.1768959849114515,1,,,,,,\n'
    'weekly,,,O2,Reference,0.54099622013305748,134750,7.4784861406785872,0.77122894558673716,1.4764036900004054,1.2865693681667312,2,,,,,,\n'
    'weekly,,,O2,Reference,0.47048537149191028,137200,9.1625652352066584,0.90422462127522318,1.6445212632518349,1.4712204630767052,3,,,,,,\n'
    'weekly,,,O2,Reference,0.42165621761791738,134750,9.7280850644588437,1.0018594725142189,1.7942835738321086,1.6169540858284519,4,,,,,,\n'
    'weekly,,,O2,Reference,0.37175421397102282,39200,9.8494540801571784,1.1859339849146977,1.865656347386818,1.7819477616007908,5,,,,,,\n'
    'weekly,,,O2,PhysicsNew,0.53891597734984009,58800,7.3548804494690998,0.83125613035326595,1.4818014604134178,1.2492286222184452,1,,,,,,\n'
    'weekly,,,O2,PhysicsNew,0.50977632811364249,134750,8.0617343685763672,0.82942332768663118,1.5788741347661526,1.360538755049975,2,,,,,,\n'
    'weekly,,,O2,PhysicsNew,0.43770577684399031,137200,9.8956518916923155,0.96517061598681175,1.7589690978052022,1.5337598286806968,3,,,,,,\n'
    'weekly,,,O2,PhysicsNew,0.3887300897801973,134750,9.9814350929212114,1.0583275659904905,1.9254659168122701,1.7116876691508589,4,,,,,,\n'
    'weekly,,,O2,PhysicsNew,0.34417835354168619,39200,10.451699421191176,1.2040666111387286,1.9920638506411803,1.84575788356074,5,,,,,,\n'
    'weekly,,,O2,PhysicsNewBias,0.54149012765174975,58800,7.2683573420586987,0.8251212892028067,1.4753200495087393,1.2461105359640503,1,,,,,,\n'
    'weekly,,,O2,PhysicsNewBias,0.51387495536686967,134750,7.9619797229876941,0.82071504183781252,1.56747272423767,1.3522171301837591,2,,,,,,\n'
    'weekly,,,O2,PhysicsNewBias,0.4435418372652587,137200,9.6433250153765364,0.95102804207040603,1.7497788349639487,1.5272587778591329,3,,,,,,\n'
    'weekly,,,O2,PhysicsNewBias,0.39475566300416881,134750,9.8387496617097092,1.0399244498920597,1.913285706430448,1.7012164556299811,4,,,,,,\n'
    'weekly,,,O2,PhysicsNewBias,0.35225035832234208,39200,10.130473491375378,1.1809598544555391,1.9803233841018368,1.837307164391949,5,,,,,,\n'
    'weekly,,,O2,Placebo,0.54033117673778552,58800,7.4315346791737396,0.81713499533046396,1.4776392884402851,1.2486204216769969,1,,,,,,\n'
    'weekly,,,O2,Placebo,0.49387529419128889,134750,8.1319947393773031,0.86166512924650662,1.6536491947063954,1.396644871961076,2,,,,,,\n'
    'weekly,,,O2,Placebo,0.40604651445413831,137200,9.8321381858197938,1.016705836948101,1.9331945822759709,1.6154260055127809,3,,,,,,\n'
    'weekly,,,O2,Placebo,0.31248171986982098,134750,10.357351901972516,1.179612401854593,2.2940894035517627,1.9074482042465319,4,,,,,,\n'
    'weekly,,,O2,Placebo,0.29106321489784331,39200,10.099370160262064,1.2764418951291805,2.2777715427386074,2.0223805176855727,5,,,,,,\n'
    'weekly,,,O2,Recent,0.53935639893638165,58800,7.2985866500239363,0.82641122180381965,1.4868314490913417,1.2511607637558764,1,,,,,,\n'
    'weekly,,,O2,Recent,0.51285378689711514,134750,7.9788264407766256,0.81733895796915568,1.573928667272237,1.3577368011249495,2,,,,,,\n'
    'weekly,,,O2,Recent,0.44279822904898908,137200,9.6050753366608959,0.95292037484596903,1.7608675406714356,1.5260586592428222,3,,,,,,\n'
    'weekly,,,O2,Recent,0.39509569999521221,134750,9.889051854986862,1.038167174754844,1.915903557312272,1.6939715228726515,4,,,,,,\n'
    'weekly,,,O2,Recent,0.35150106172559631,39200,10.16404638241008,1.1896014466896283,1.9893158717239925,1.8250540707036631,5,,,,,,\n'
    'weekly,,,O2,RecentBias,0.54195391985176911,58800,7.2148494289479137,0.82142660200825424,1.4806732755384482,1.2462034830033946,1,,,,,,\n'
    'weekly,,,O2,RecentBias,0.51664823804607574,134750,7.90373374375858,0.80933662025731379,1.5629301202198154,1.3488973319993842,2,,,,,,\n'
    'weekly,,,O2,RecentBias,0.44811027547316762,137200,9.4100225533827846,0.94029266789360499,1.7506227121015232,1.5185095090109739,3,,,,,,\n'
    'weekly,,,O2,RecentBias,0.40054698146263412,134750,9.7739960332729368,1.0222768610439694,1.903382266477802,1.6837473114085235,4,,,,,,\n'
    'weekly,,,O2,RecentBias,0.35904369921583068,39200,9.8830443909863224,1.168431617555449,1.9774187964725165,1.8158767656440944,5,,,,,,\n'
    'weekly,,,O2,Drift,0.53843721838300396,58800,7.2656194673031411,0.83151264867244656,1.4904004414629903,1.2536107656381501,1,,,,,,\n'
    'weekly,,,O2,Drift,0.50985732370294401,134750,8.1338688608860377,0.8187539923451147,1.576484772878008,1.3665723316641734,2,,,,,,\n'
    'weekly,,,O2,Drift,0.43720245291253329,137200,10.139124214108689,0.94911255369313563,1.753892580034085,1.5381106402211568,3,,,,,,\n'
    'weekly,,,O2,Drift,0.38428352594875348,134750,10.17271716507604,1.0481420434562108,1.933405193910452,1.7371782953999275,4,,,,,,\n'
    'weekly,,,O2,Drift,0.34470307415713419,39200,10.514484218064467,1.188189399114165,2.0018610053319223,1.8446189367607693,5,,,,,,\n'
    'weekly,,,O2,DriftBias,0.54054507842510091,58800,7.1766435074286967,0.82752861983655512,1.4842531487187896,1.2521536677121323,1,,,,,,\n'
    'weekly,,,O2,DriftBias,0.51321788746106889,134750,8.0091223354273549,0.81450991938982908,1.5678239832291962,1.3598217610608054,2,,,,,,\n'
    'weekly,,,O2,DriftBias,0.4427416902697176,137200,9.8241453937085605,0.93874485362579141,1.7473080393715681,1.5333982239108117,3,,,,,,\n'
    'weekly,,,O2,DriftBias,0.38970126147466938,134750,9.9877462119122846,1.0344339881113656,1.9245246801655345,1.7279450502020179,4,,,,,,\n'
    'weekly,,,O2,DriftBias,0.35213915784178967,39200,10.142426109197933,1.1698112754704342,1.9933612297745216,1.838356310006817,5,,,,,,\n'
    'weekly,,,O2,MultiScale,0.54125462145944447,58800,7.2742473635714093,0.82538034601330923,1.4763772584932799,1.2465061252291545,1,,,,,,\n'
    'weekly,,,O2,MultiScale,0.51351709314801808,134750,7.9675831798237455,0.82098901470223384,1.5689663595597425,1.3533026916927904,2,,,,,,\n'
    'weekly,,,O2,MultiScale,0.4430478786979265,137200,9.6481778849109716,0.95237779773724962,1.7509860815550982,1.5286784183944759,3,,,,,,\n'
    'weekly,,,O2,MultiScale,0.39410063042457388,134750,9.8429099918393597,1.0415983622027221,1.9151666258178988,1.7031852336008664,4,,,,,,\n'
    'weekly,,,O2,MultiScale,0.35155966544563061,39200,10.135519878191676,1.1826911403618692,1.982192337426276,1.8394521146402063,5,,,,,,\n'
    'weekly,,,O2,AdaptiveFlow,0.54264223396724953,58800,7.239017851553526,0.82426958697399366,1.4713254078613691,1.2428805496857791,1,,,,,,\n'
    'weekly,,,O2,AdaptiveFlow,0.51575849917806349,134750,7.9241291288201596,0.81766175069373193,1.5613947644781294,1.3474297048195121,2,,,,,,\n'
    'weekly,,,O2,AdaptiveFlow,0.4458586214645528,137200,9.5931224885272712,0.94659053439342655,1.7419965153072583,1.5225984562451671,3,,,,,,\n'
    'weekly,,,O2,AdaptiveFlow,0.3974462637109808,134750,9.7778158578461696,1.0362210996704428,1.9033506775456637,1.6952542471683312,4,,,,,,\n'
    'weekly,,,O2,AdaptiveFlow,0.35481891303602259,39200,10.073026456060523,1.1774833263485116,1.9710586911213044,1.8313251508435535,5,,,,,,\n'
    'weekly,,,O2,WindFlow,0.54224979982778021,58800,7.2683615527369998,0.82512179288792342,1.4713254078613691,1.2428805496857791,1,,,,,,\n'
    'weekly,,,O2,WindFlow,0.51501579042891177,134750,7.9619778605690854,0.82071489512210494,1.5613947644781294,1.3474297048195121,2,,,,,,\n'
    'weekly,,,O2,WindFlow,0.44482647928825009,137200,9.6433244672610137,0.9510286397168296,1.7419965153072583,1.5225984562451671,3,,,,,,\n'
    'weekly,,,O2,WindFlow,0.39639716814255821,134750,9.8387482792235996,1.0399254164245335,1.9033506775456637,1.6952542471683312,4,,,,,,\n'
    'weekly,,,O2,WindFlow,0.35383169312510998,39200,10.130473733383177,1.1809603145315295,1.9710586911213044,1.8313251508435535,5,,,,,,\n'
    'weekly,,,O2,Taper,0.54143542111773413,58800,7.2701143987163164,0.82535366868021964,1.4754452092982178,1.2460902377225871,1,,,,,,\n'
    'weekly,,,O2,Taper,0.51280169684833288,134750,7.9900385147221131,0.82259202337676673,1.5706760523517549,1.3544746815597828,2,,,,,,\n'
    'weekly,,,O2,Taper,0.4411720524544826,137200,9.7448781635064154,0.95677190966339198,1.7535592173018659,1.5299341588043034,3,,,,,,\n'
    'weekly,,,O2,Taper,0.39164539839604828,134750,9.9076904414200175,1.0495407840565389,1.9197721450082663,1.7067354890515141,4,,,,,,\n'
    'weekly,,,O2,Taper,0.34745401672332349,39200,10.320990179451051,1.1947898508129575,1.987291837146522,1.8422624855739005,5,,,,,,\n'
    'weekly,,,O2,MixPhysicsNewBias,0.56728232351880725,58800,6.7897085808138637,0.79294335492408397,1.3801085820579653,1.1768959808193149,1,,,,,,\n'
    'weekly,,,O2,MixPhysicsNewBias,0.54099620619582112,134750,7.4784864551432024,0.77122901480393335,1.4764036898928246,1.2865693887778604,2,,,,,,\n'
    'weekly,,,O2,MixPhysicsNewBias,0.47048538214781788,137200,9.1625654140235024,0.90422450298919699,1.6445212701383067,1.4712204735756034,3,,,,,,\n'
    'weekly,,,O2,MixPhysicsNewBias,0.42165624984342748,134750,9.7280850087593969,1.0018592625004712,1.7942835655879652,1.61695404003687,4,,,,,,\n'
    'weekly,,,O2,MixPhysicsNewBias,0.37175422897182347,39200,9.8494535621311083,1.1859338675151021,1.8656563580895336,1.7819477893098421,5,,,,,,\n'
    'weekly,,,O2,MixRecentBias,0.56732215993405921,58800,6.7879910802969929,0.79268458495753047,1.3802896186575555,1.1768251439628024,1,,,,,,\n'
    'weekly,,,O2,MixRecentBias,0.54143326215691512,134750,7.4775499445306766,0.76789473335580882,1.4763223467946966,1.2864307547809934,2,,,,,,\n'
    'weekly,,,O2,MixRecentBias,0.47129904890700769,137200,9.1333373256421275,0.90003657611856647,1.6445410459618657,1.4710499788782645,3,,,,,,\n'
    'weekly,,,O2,MixRecentBias,0.4224934161583872,134750,9.7332906419436025,0.99562576833259442,1.7940511450175685,1.6160171586184555,4,,,,,,\n'
    'weekly,,,O2,MixRecentBias,0.3728600426885737,39200,9.8084861359273727,1.1810284846647208,1.865580942264589,1.7811035709470855,5,,,,,,\n'
    'weekly,,,O2,MixDriftBias,0.56739877518454573,58800,6.7755424254320946,0.7939886056988853,1.3798485438431645,1.1762358527714554,1,,,,,,\n'
    'weekly,,,O2,MixDriftBias,0.54154499277190815,134750,7.4713416624929296,0.76876399302416543,1.4757287799262966,1.2855929050406341,2,,,,,,\n'
    'weekly,,,O2,MixDriftBias,0.47130869898059269,137200,9.1710193526461961,0.89832072498405668,1.6438450657119408,1.4701445500812234,3,,,,,,\n'
    'weekly,,,O2,MixDriftBias,0.42221017438688713,134750,9.7196934658468077,0.99854354127924261,1.7935675047068609,1.616929292893257,4,,,,,,\n'
    'weekly,,,O2,MixDriftBias,0.37285847784982601,39200,9.8429411619061806,1.1792229951842921,1.8651318371186856,1.7804555669797788,5,,,,,,\n'
    'weekly,,,O2,MixMultiScale,0.56728071665783786,58800,6.7899558143969188,0.79292675771554955,1.3801230313538255,1.1768945586114272,1,,,,,,\n'
    'weekly,,,O2,MixMultiScale,0.54098986298611029,134750,7.4785112760896677,0.77123056495018361,1.4764184079602811,1.2866072719389581,2,,,,,,\n'
    'weekly,,,O2,MixMultiScale,0.47043554895356238,137200,9.1626348915463645,0.90456599376233215,1.6445369782841761,1.4712734397569132,3,,,,,,\n'
    'weekly,,,O2,MixMultiScale,0.42158977591296998,134750,9.727892123878366,1.0023144202437086,1.7943088340731754,1.6170452067310777,4,,,,,,\n'
    'weekly,,,O2,MixMultiScale,0.37167430120267891,39200,9.8498536061314343,1.1864337159774567,1.865685746142185,1.7820541045684013,5,,,,,,\n'
    'weekly,,,O2,MixAdaptiveFlow,0.56730106794924262,58800,6.7872078266321241,0.79310087094957082,1.3800370851417849,1.1768418342474305,1,,,,,,\n'
    'weekly,,,O2,MixAdaptiveFlow,0.54116185020378682,134750,7.4736836910731537,0.77044466852209814,1.4763223906890113,1.2864377907490061,2,,,,,,\n'
    'weekly,,,O2,MixAdaptiveFlow,0.4707281504481638,137200,9.1556159888619284,0.90303343906693045,1.6443944313267849,1.4710682011936034,3,,,,,,\n'
    'weekly,,,O2,MixAdaptiveFlow,0.42187436234082959,134750,9.7200076746640836,1.0010422545596689,1.7941393800751388,1.6167359126397329,4,,,,,,\n'
    'weekly,,,O2,MixAdaptiveFlow,0.37198601137309451,39200,9.8412709720056597,1.1850414894079615,1.865508520821594,1.7817032120429563,5,,,,,,\n'
    'weekly,,,O2,MixWindFlow,0.56729546372027162,58800,6.7897089458490836,0.7929434868923757,1.3800370851417849,1.1768418342474305,1,,,,,,\n'
    'weekly,,,O2,MixWindFlow,0.54101956784142791,134750,7.4784861406785872,0.77122894558673716,1.4763223906890113,1.2864377907490061,2,,,,,,\n'
    'weekly,,,O2,MixWindFlow,0.47051544751549668,137200,9.1625652352066584,0.90422462127522318,1.6443944313267849,1.4710682011936034,3,,,,,,\n'
    'weekly,,,O2,MixWindFlow,0.42169580785845961,134750,9.7280850644588437,1.0018594725142189,1.7941393800751388,1.6167359126397329,4,,,,,,\n'
    'weekly,,,O2,MixWindFlow,0.37179730209196582,39200,9.8494540801571784,1.1859339849146977,1.865508520821594,1.7817032120429563,5,,,,,,\n'
    'weekly,,,O2,MixTaper,0.56728129891313572,58800,6.7894955962089858,0.79297765306753709,1.3801089264686437,1.1768871306075186,1,,,,,,\n'
    'weekly,,,O2,MixTaper,0.54092164940471799,134750,7.479166099796517,0.77166612672689561,1.4764491747681687,1.2866587049462792,2,,,,,,\n'
    'weekly,,,O2,MixTaper,0.4701532388871732,137200,9.174448878665098,0.90586480447941864,1.6445894972495392,1.4713044859757212,3,,,,,,\n'
    'weekly,,,O2,MixTaper,0.42127579970217532,134750,9.7308926188103193,1.0044673090694218,1.7943837703402381,1.6171853728522363,4,,,,,,\n'
    'weekly,,,O2,MixTaper,0.3709285366617166,39200,9.8739921593406006,1.1904947644364117,1.8657597399923651,1.7821316987933609,5,,,,,,\n'
    'daily,,,O2,Ultralisk,0.56943969713789622,19600,6.4059847526465736,0.77035641918698217,1.3865369364339724,1.2082886954290752,,20240421,1,,,,\n'
    'daily,,,O2,Ultralisk,0.60368212613923422,19600,6.931629055071487,0.72269993855278525,1.2076980002486877,1.066686177547244,,20240422,2,,,,\n'
    'daily,,,O2,Ultralisk,0.52899730882227802,19600,7.0709939759635869,0.90088566120990965,1.5277777370197807,1.249192289915747,,20240423,3,,,,\n'
    'daily,,,O2,Ultralisk,0.58004120183309027,17150,8.5628640000823548,0.69966331655074665,1.2359073595246592,1.131109318494782,,20240428,8,,,,\n'
    'daily,,,O2,Ultralisk,0.59033783619568792,19600,7.1286815741765572,0.74931821467214788,1.2788055211548166,1.0795490357198532,,20240429,9,,,,\n'
    'daily,,,O2,Ultralisk,0.58027147278176205,19600,6.0915911445486719,0.70448038054335538,1.36311827700761,1.2285684764439626,,20240430,10,,,,\n'
    'daily,,,O2,Ultralisk,0.57074699764916836,19600,6.9805157008066372,0.6804572878989551,1.3777226323297411,1.2478239392581849,,20240501,11,,,,\n'
    'daily,,,O2,Ultralisk,0.50807326595167601,19600,8.0148189159198413,0.81657710291192609,1.5944022344399269,1.3795010443974869,,20240502,12,,,,\n'
    'daily,,,O2,Ultralisk,0.5032891801951056,19600,7.2899142775739803,0.84957398152859054,1.6664063459089025,1.3896301897239347,,20240503,13,,,,\n'
    'daily,,,O2,Ultralisk,0.47107340605840509,19600,8.3894486386307658,0.88783559626182895,1.7176438971605601,1.4895084997407309,,20240504,14,,,,\n'
    'daily,,,O2,Ultralisk,0.51748775873721786,19600,7.5305205960319546,0.77429941350231268,1.6152326767658065,1.3672487541436815,,20240505,15,,,,\n'
    'daily,,,O2,Ultralisk,0.51997911098958061,19600,8.9836854329612699,0.82028162147884187,1.4707159244790924,1.2930144845141809,,20240506,16,,,,\n'
    'daily,,,O2,Ultralisk,0.51236156416128531,19600,7.9922891394817173,0.88064144944925871,1.4947569776125251,1.358476209799812,,20240507,17,,,,\n'
    'daily,,,O2,Ultralisk,0.43455584823286098,19600,9.3801519483981739,1.0361484964894248,1.7143725429147787,1.5655203534589917,,20240508,18,,,,\n'
    'daily,,,O2,Ultralisk,0.44103448334563339,19600,10.205173032246735,0.97970346117640361,1.661459215268541,1.5418178863914251,,20240509,19,,,,\n'
    'daily,,,O2,Ultralisk,0.472095851530484,19600,10.286976724136968,0.8091155170929164,1.6253078192463,1.4763328410866119,,20240510,20,,,,\n'
    'daily,,,O2,Ultralisk,0.40130898928849068,19600,9.5862961147594117,1.0387656418180735,1.898607947613598,1.6798901272535485,,20240511,21,,,,\n'
    'daily,,,O2,Ultralisk,0.34656343137918411,19600,13.323321214037083,1.1064857107237951,1.81546588556594,1.8257781196433811,,20240512,22,,,,\n'
    'daily,,,O2,Ultralisk,0.43491197752472038,19600,8.6888236941734647,0.97019181455394243,1.845920278490627,1.5835661466331346,,20240513,23,,,,\n'
    'daily,,,O2,Ultralisk,0.45023046753564738,19600,8.5977939424129275,1.0152041101606746,1.7079271695706968,1.5252303526100033,,20240514,24,,,,\n'
    'daily,,,O2,Ultralisk,0.39897449778743388,19600,10.934767984225363,1.0395348441619581,1.83472315000624,1.6374101880626439,,20240515,25,,,,\n'
    'daily,,,O2,Ultralisk,0.45243913292330279,19600,9.4642114671017996,0.98166071340482675,1.6997335988397209,1.4757601926668986,,20240516,26,,,,\n'
    'daily,,,O2,Ultralisk,0.4215934168647632,17150,8.7705831906533458,0.98963513568530037,1.9475949483593815,1.5891259118031915,,20240517,27,,,,\n'
    'daily,,,O2,Ultralisk,0.44118505535438129,19600,7.9572085033267701,0.99813164197240201,1.7187298694274082,1.6623305479232229,,20240518,28,,,,\n'
    'daily,,,O2,Ultralisk,0.41655121436701831,19600,9.1574112588877359,1.136211234758522,1.7019038646683795,1.6427751406690945,,20240519,29,,,,\n'
    'daily,,,O2,Ultralisk,0.32144074264736788,19600,10.681404683993254,1.2823992831172546,2.0184983910957066,1.916924931245144,,20240520,30,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.54496474579410303,19600,6.8751055185204359,0.79743802525446661,1.4609958418862179,1.2886194742787058,,20240421,1,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.57861748164086579,19600,7.6067739274696766,0.7599754222241385,1.2642288515725171,1.1386547358118724,,20240422,2,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.49823333455699531,19600,7.5599529723751981,0.92708866138891943,1.6895039175935995,1.3132557761848611,,20240423,3,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.54250019049630738,17150,8.9593401839864288,0.76654656860190817,1.3749639820994239,1.2359297438839818,,20240428,8,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.56732737115174559,19600,7.2499026615220865,0.78943599133500231,1.3596220740232998,1.1581550590520682,,20240429,9,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.56926702504909521,19600,6.1661064892976247,0.73656824329010628,1.3969904673005731,1.2554815472315799,,20240430,10,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.54894110981679844,19600,6.9543349918655935,0.72106942501642124,1.4817193491816778,1.3099329556798358,,20240501,11,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.46973912785168309,19600,9.2664321932652527,0.8889726762064456,1.6967331517384112,1.4436119085517327,,20240502,12,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.47211415251931599,19600,8.1247256040151203,0.890625121975546,1.7622477182339242,1.465014179530526,,20240503,13,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.42585565636046968,19600,9.2605644800880977,0.97277641313601204,1.87190511721456,1.5892533423749025,,20240504,14,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.49492148330020302,19600,7.7405190146303902,0.80234265050201226,1.6910477162642918,1.4509425419427131,,20240505,15,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.47684479898714582,19600,10.032588794561365,0.8987399166643355,1.6091138876296518,1.3796472433997089,,20240506,16,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.4718242603053901,19600,9.4242171435994155,0.96699943153507262,1.6208362469944382,1.3935117965474979,,20240507,17,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.40983632621245503,19600,10.091178164654728,1.0369549119414931,1.9246071634345951,1.5486822556714881,,20240508,18,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.40391489356277333,19600,10.986090481783972,1.0600419307956062,1.7638298962010901,1.6264737634263844,,20240509,19,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.44514478795216528,19600,10.687104731105229,0.87682652525127003,1.6748942161694294,1.5613413869253452,,20240510,20,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.37654939421734229,19600,9.9670819429353816,1.0800581558322049,1.99030860506736,1.7420617683739106,,20240511,21,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.29707722012058968,19600,14.551720080147444,1.163840088351902,2.0591486538417176,1.8899231180350291,,20240512,22,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.39264893396802669,19600,9.3177448925189879,1.0324159706985263,1.9950618487689356,1.7055297452994862,,20240513,23,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.40033266457039057,19600,8.6735327893569849,1.1311226962842265,1.8935717450348315,1.6737479350889359,,20240514,24,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.39048259168441118,19600,9.9770723799186793,1.0180508349617337,1.9249590986557876,1.738521441089558,,20240515,25,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.43876656922325408,19600,8.4173288781516238,1.0112202415221765,1.7713635838662536,1.5903827845598559,,20240516,26,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.40247642540438849,17150,8.7766671831244398,1.0118741424289652,2.0068663728812242,1.6795682225189641,,20240517,27,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.41994112698584107,19600,8.5645739972648247,1.0225787232393728,1.8207885635353436,1.6854457086770052,,20240518,28,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.39965925298255089,19600,9.5788992833714364,1.116904141961347,1.7940092025358745,1.6968397294968951,,20240519,29,,,,\n'
    'daily,,,O2,PhysicsLegacy,0.29309065815755397,19600,11.257035359548883,1.2853330192928627,2.1721340331697467,1.9835262479406508,,20240520,30,,,,\n'
    'daily,,,O2,Bias325,0.54826080453520376,19600,6.8036283542545757,0.79294533809836099,1.4474152787382693,1.2821095155053277,,20240421,1,,,,\n'
    'daily,,,O2,Bias325,0.58227111918603314,19600,7.3891584393950991,0.75529527759118886,1.259601161729875,1.134398107384041,,20240422,2,,,,\n'
    'daily,,,O2,Bias325,0.49904430230941149,19600,7.5893489680351118,0.91828353667713603,1.6876320056740799,1.3144241047154064,,20240423,3,,,,\n'
    'daily,,,O2,Bias325,0.54400122672713169,17150,9.0520740033296363,0.75449919411610733,1.3686735116438762,1.2328442036739815,,20240428,8,,,,\n'
    'daily,,,O2,Bias325,0.57085979233414785,19600,7.3276436186814422,0.77490745473362244,1.3435683332341299,1.1494608744726409,,20240429,9,,,,\n'
    'daily,,,O2,Bias325,0.57041360267844432,19600,6.3083424914223443,0.73424582530841309,1.3886007841535448,1.2430871481650332,,20240430,10,,,,\n'
    'daily,,,O2,Bias325,0.55130822250754297,19600,6.9903213372904878,0.70885778331518878,1.4753277765620141,1.3045315774583128,,20240501,11,,,,\n'
    'daily,,,O2,Bias325,0.47558160319953852,19600,8.9498477026636785,0.88021124960597452,1.6872662092276898,1.4371230889651017,,20240502,12,,,,\n'
    'daily,,,O2,Bias325,0.47887380545204888,19600,7.8322109308611596,0.88438324636170274,1.7450164983212617,1.4523917326514628,,20240503,13,,,,\n'
    'daily,,,O2,Bias325,0.43196484916086619,19600,8.9628413511477287,0.96629194798037477,1.8575902279080596,1.5804627442687809,,20240504,14,,,,\n'
    'daily,,,O2,Bias325,0.49804837445977912,19600,7.5366958366444257,0.7962074264189003,1.6842659390569965,1.4530993961277383,,20240505,15,,,,\n'
    'daily,,,O2,Bias325,0.4832319970351735,19600,9.8319124348579034,0.88153505906912411,1.5986257740623988,1.3683995463164629,,20240506,16,,,,\n'
    'daily,,,O2,Bias325,0.47814757071553382,19600,9.0330159379752288,0.95445120002886963,1.6158944116525271,1.3894500191174699,,20240507,17,,,,\n'
    'daily,,,O2,Bias325,0.41513516081088581,19600,9.6515039985495008,1.0313467307978097,1.9217213197180132,1.548547202006886,,20240508,18,,,,\n'
    'daily,,,O2,Bias325,0.41234907300498408,19600,10.64153592518463,1.0388591962134712,1.7500086953489966,1.6165964819371264,,20240509,19,,,,\n'
    'daily,,,O2,Bias325,0.44960007683417891,19600,10.695884678047049,0.86495435876231097,1.6577040939195304,1.5488184640586855,,20240510,20,,,,\n'
    'daily,,,O2,Bias325,0.38336725646576081,19600,9.7517444051056437,1.0574073690001378,1.981182792813007,1.7328676159087222,,20240511,21,,,,\n'
    'daily,,,O2,Bias325,0.30427886905505269,19600,14.301544641977069,1.1424662603463616,2.049214297073997,1.879731576081304,,20240512,22,,,,\n'
    'daily,,,O2,Bias325,0.39817245335569018,19600,9.2672997271169724,1.0106917096135866,1.9874079695157101,1.6914859894108221,,20240513,23,,,,\n'
    'daily,,,O2,Bias325,0.40589479169813819,19600,8.5112096656001945,1.1168922480444574,1.8815560867350816,1.6644170095361484,,20240514,24,,,,\n'
    'daily,,,O2,Bias325,0.39473893838008262,19600,9.9969630829953449,1.0006050732466247,1.9120989429194248,1.7290033185367348,,20240515,25,,,,\n'
    'daily,,,O2,Bias325,0.44525913727063449,19600,8.2455330071796844,0.99251542461891418,1.7546273565494841,1.5822653958249446,,20240516,26,,,,\n'
    'daily,,,O2,Bias325,0.40903593828118179,17150,8.5415750615763706,0.99663696766688636,1.9921947637426711,1.6709873369466848,,20240517,27,,,,\n'
    'daily,,,O2,Bias325,0.42653308982946792,19600,8.4191545109356856,1.0026646204452705,1.8083605677110819,1.6721905411389415,,20240518,28,,,,\n'
    'daily,,,O2,Bias325,0.4090737714058203,19600,9.1927096132984083,1.090974134671707,1.7816133979373252,1.685891018626013,,20240519,29,,,,\n'
    'daily,,,O2,Bias325,0.30006980699109709,19600,10.988497899672367,1.2645591984854401,2.1608362632428952,1.9771606815830287,,20240520,30,,,,\n'
    'daily,,,O2,Reference,0.57094364807783649,19600,6.375084322480288,0.76074133072635419,1.3857913160382225,1.2085744871382349,,20240421,1,,,,\n'
    'daily,,,O2,Reference,0.60479565781615885,19600,6.9161352711920712,0.71622012623617237,1.2066624502586021,1.066020634507107,,20240422,2,,,,\n'
    'daily,,,O2,Reference,0.53026854001823076,19600,7.0587403621232756,0.89139187718648993,1.5289366702739815,1.2482788926140285,,20240423,3,,,,\n'
    'daily,,,O2,Reference,0.58134001418193426,17150,8.5047435271871645,0.69336021034571194,1.2359662286921504,1.1313736872270559,,20240428,8,,,,\n'
    'daily,,,O2,Reference,0.5922942485533107,19600,7.0715774705444856,0.7396323209202238,1.2779380484281182,1.0783802194964602,,20240429,9,,,,\n'
    'daily,,,O2,Reference,0.58193357329282591,19600,6.0375574853230658,0.69941903504226643,1.3615967529046231,1.2254397525514211,,20240430,10,,,,\n'
    'daily,,,O2,Reference,0.57281487004120013,19600,6.8914608550493437,0.67264169046966815,1.3772700978592307,1.2461404375582461,,20240501,11,,,,\n'
    'daily,,,O2,Reference,0.50758735883189188,19600,8.0506576409593436,0.81903120986768485,1.5942701156429824,1.3782306507829594,,20240502,12,,,,\n'
    'daily,,,O2,Reference,0.50478402633110719,19600,7.2532946866197641,0.84360296409118285,1.6650320033960246,1.3872693928140201,,20240503,13,,,,\n'
    'daily,,,O2,Reference,0.47075555390126539,19600,8.3569774667238335,0.8929650759602541,1.7179260710882338,1.4894305602030831,,20240504,14,,,,\n'
    'daily,,,O2,Reference,0.52046422810607929,19600,7.4364164570213367,0.76005744963632027,1.6137279606916084,1.3657101614707856,,20240505,15,,,,\n'
    'daily,,,O2,Reference,0.51939259178856667,19600,9.0160581604983001,0.82348156228530522,1.470154981039107,1.292449969257955,,20240506,16,,,,\n'
    'daily,,,O2,Reference,0.5116314054063491,19600,8.0424759888332602,0.88683421094452464,1.4946997870658243,1.3542266737860862,,20240507,17,,,,\n'
    'daily,,,O2,Reference,0.43886638975011688,19600,9.3134217282257019,1.014118362343881,1.7155003081096332,1.5564085626328468,,20240508,18,,,,\n'
    'daily,,,O2,Reference,0.44232942871142522,19600,10.179781386935661,0.97383146315734481,1.6599481938404668,1.5402085407327917,,20240509,19,,,,\n'
    'daily,,,O2,Reference,0.47275761885353579,19600,10.273644554441454,0.80682431914353048,1.6233117426098436,1.4757766021296088,,20240510,20,,,,\n'
    'daily,,,O2,Reference,0.40416381641328208,19600,9.5121843713536833,1.0263146587326428,1.8968499985796927,1.6761085771751103,,20240511,21,,,,\n'
    'daily,,,O2,Reference,0.34833240659955289,19600,13.344196898217829,1.0937137699062134,1.8159702674350084,1.8218052390549704,,20240512,22,,,,\n'
    'daily,,,O2,Reference,0.43628521476158938,19600,8.6538179436147846,0.96138879406411237,1.845986864086556,1.5838272791543131,,20240513,23,,,,\n'
    'daily,,,O2,Reference,0.45130143134378081,19600,8.415973769109689,1.0194428358987211,1.7081320772697994,1.526810103052501,,20240514,24,,,,\n'
    'daily,,,O2,Reference,0.40582357133202029,19600,10.682286421738816,1.0065635496526979,1.8328315873310721,1.6358052940159424,,20240515,25,,,,\n'
    'daily,,,O2,Reference,0.456738497581963,19600,9.1933104714225955,0.96758158291638263,1.6985115254521139,1.4773436041385877,,20240516,26,,,,\n'
    'daily,,,O2,Reference,0.42481092341146909,17150,8.6449309125044849,0.97418328134559062,1.9461878370963761,1.5893184768184938,,20240517,27,,,,\n'
    'daily,,,O2,Reference,0.44426377672465062,19600,7.9255328299125587,0.98033737422237555,1.7181991044835665,1.6576305747889306,,20240518,28,,,,\n'
    'daily,,,O2,Reference,0.4216030225077278,19600,9.0516311440077484,1.1074229144358223,1.7003614817389776,1.6396012884965012,,20240519,29,,,,\n'
    'daily,,,O2,Reference,0.3256123944560958,19600,10.587325676821258,1.2595607653995011,2.0174533562161665,1.9137354218670748,,20240520,30,,,,\n'
    'daily,,,O2,PhysicsNew,0.54496497141630318,19600,6.8751044635661094,0.79743757744320998,1.4609946592142156,1.2886190518281342,,20240421,1,,,,\n'
    'daily,,,O2,PhysicsNew,0.57861746898720445,19600,7.6067684680178278,0.75997428124901922,1.2642296668298207,1.1386558086219594,,20240422,2,,,,\n'
    'daily,,,O2,PhysicsNew,0.4982334177812584,19600,7.5599478415509873,0.92708827231887625,1.6895042644234171,1.313255626474304,,20240423,3,,,,\n'
    'daily,,,O2,PhysicsNew,0.54250021807005766,17150,8.9593466622934717,0.76654669569984613,1.3749632330627055,1.2359294358594397,,20240428,8,,,,\n'
    'daily,,,O2,PhysicsNew,0.56732738453563469,19600,7.2499073285316618,0.78943526991508395,1.3596219655689643,1.1581553769110162,,20240429,9,,,,\n'
    'daily,,,O2,PhysicsNew,0.56926704767926173,19600,6.1661054692209563,0.73656853031076452,1.3969902058409744,1.2554813559460021,,20240430,10,,,,\n'
    'daily,,,O2,PhysicsNew,0.54894109782206779,19600,6.9543478721436474,0.72106890426355785,1.4817184584602994,1.3099332164091535,,20240501,11,,,,\n'
    'daily,,,O2,PhysicsNew,0.46973911334962393,19600,9.2664298065165056,0.88897238423107827,1.6967331755233053,1.4436125027218487,,20240502,12,,,,\n'
    'daily,,,O2,PhysicsNew,0.47211406998730687,19600,8.1247210484336794,0.8906252085269214,1.7622478124305354,1.4650150827840989,,20240503,13,,,,\n'
    'daily,,,O2,PhysicsNew,0.4258554873182735,19600,9.2605639843832037,0.97277804996524297,1.8719046318290389,1.5892535186001311,,20240504,14,,,,\n'
    'daily,,,O2,PhysicsNew,0.49492154738621008,19600,7.7405174520651689,0.80234206717086398,1.6910484090083222,1.4509421849850626,,20240505,15,,,,\n'
    'daily,,,O2,PhysicsNew,0.47684483685197332,19600,10.0326002537488,0.89873906305041085,1.6091139009759314,1.3796468363759444,,20240506,16,,,,\n'
    'daily,,,O2,PhysicsNew,0.4718246514847006,19600,9.42421012085393,0.96699800283483162,1.6208354565215122,1.3935111686378709,,20240507,17,,,,\n'
    'daily,,,O2,PhysicsNew,0.40983661632502061,19600,10.09117085035132,1.036954056814188,1.9246064002395304,1.5486818946488312,,20240508,18,,,,\n'
    'daily,,,O2,PhysicsNew,0.4039148549359719,19600,10.986093336512653,1.0600415124356457,1.7638300591726623,1.6264741468146444,,20240509,19,,,,\n'
    'daily,,,O2,PhysicsNew,0.44514474669710008,19600,10.687111793072557,0.87682574881384612,1.6748946914809291,1.5613415643710362,,20240510,20,,,,\n'
    'daily,,,O2,PhysicsNew,0.37654935288422853,19600,9.9670825944428714,1.0800579957789391,1.9903084341778972,1.7420623549145742,,20240511,21,,,,\n'
    'daily,,,O2,PhysicsNew,0.29707761530938698,19600,14.551736607641285,1.1638382336690278,2.0591470314355758,1.8899215972441621,,20240512,22,,,,\n'
    'daily,,,O2,PhysicsNew,0.39264877043664831,19600,9.3177492719978154,1.0324155661278871,1.9950631834665755,1.7055301186964855,,20240513,23,,,,\n'
    'daily,,,O2,PhysicsNew,0.40033302598000048,19600,8.6735324744025171,1.1311211283174496,1.8935700742092272,1.6737478307392255,,20240514,24,,,,\n'
    'daily,,,O2,PhysicsNew,0.39048287435393497,19600,9.9770642882740503,1.0180504522230569,1.9249590523994076,1.7385201697328816,,20240515,25,,,,\n'
    'daily,,,O2,PhysicsNew,0.43876708706791528,19600,8.4173227821641827,1.0112182065814339,1.7713629968632911,1.5903814768344271,,20240516,26,,,,\n'
    'daily,,,O2,PhysicsNew,0.40247688068285398,17150,8.7766533561611464,1.0118736397062655,2.0068653174296092,1.6795668900399887,,20240517,27,,,,\n'
    'daily,,,O2,PhysicsNew,0.41994118906760808,19600,8.5645762392361782,1.0225780885522759,1.8207883737325961,1.6854457952649795,,20240518,28,,,,\n'
    'daily,,,O2,PhysicsNew,0.39965920581164788,19600,9.5788981098290584,1.1169038170981105,1.7940095779374996,1.6968402557008573,,20240519,29,,,,\n'
    'daily,,,O2,PhysicsNew,0.29309077057815158,19600,11.257031250883566,1.2853321249661025,2.1721340668410059,1.9835265251783989,,20240520,30,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.54826104657364516,19600,6.8036266318075205,0.79294500009437996,1.4474141133683005,1.2821088864317702,,20240421,1,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.58227111598193704,19600,7.3891526095276454,0.755294224669808,1.2596019743401128,1.1343990448335646,,20240422,2,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.49904437361255671,19600,7.5893440904762004,0.91828333682631036,1.6876323640260502,1.3144238334476517,,20240423,3,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.54400124849290155,17150,9.0520801228948713,0.75449933686610404,1.3686727907566223,1.2328439359108732,,20240428,8,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.57085979096223727,19600,7.3276473069336623,0.77490692027356567,1.3435681909345938,1.1494612335181884,,20240429,9,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.57041360715237266,19600,6.3083415536638894,0.73424623919006393,1.3886004477514129,1.2430870319072096,,20240430,10,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.55130821191604618,19600,6.990334320684199,0.7088573476425859,1.4753269028298652,1.3045317187329415,,20240501,11,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.47558157191535833,19600,8.9498458355229005,0.8802110000203236,1.6872663363450144,1.4371236570862589,,20240502,12,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.47887374872332628,19600,7.8322063528991324,0.88438320764980349,1.7450166695496143,1.4523924888434812,,20240503,13,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.4319647108306211,19600,8.9628407010482345,0.96629340365664484,1.857589772969956,1.5804628361258679,,20240504,14,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.49804846467932828,19600,7.5366932848341603,0.79620679583688414,1.6842666193905269,1.4530989598170951,,20240505,15,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.48323201282003669,19600,9.8319223081716842,0.88153445389982144,1.5986258316477502,1.3683991685856538,,20240506,16,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.47814791651188648,19600,9.0330096412083041,0.95445003545762119,1.6158937023777995,1.3894493801118497,,20240507,17,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.4151354300567357,19600,9.6514975262063345,1.0313459895401089,1.9217206878392563,1.5485467284347645,,20240508,18,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.4123490387320996,19600,10.641538492653902,1.0388588319077416,1.7500088639056273,1.6165967936202983,,20240509,19,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.44960004849706348,19600,10.695888880138581,0.86495368500445724,1.6577045716384773,1.5488186633850864,,20240510,20,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.38336721671405088,19600,9.7517450447744913,1.0574072688985865,1.981182695809983,1.7328680720107816,,20240511,21,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.30427922077245068,19600,14.301560463318795,1.1424646313216842,2.0492127697218079,1.8797301742085557,,20240512,22,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.39817225495348241,19600,9.2673051408471672,1.0106914272783301,1.9874093763290552,1.691486389068604,,20240513,23,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.40589515500814399,19600,8.5112093721973476,1.1168906559027765,1.8815544198173355,1.6644169089336127,,20240514,24,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.3947392198107636,19600,9.9969547983545564,1.0006047440362595,1.9120988668985739,1.7290020425091379,,20240515,25,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.44525959937435089,19600,8.2455279312327221,0.9925135963208066,1.7546268722729892,1.5822641806153788,,20240516,26,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.4090363528361568,17150,8.5415625020763635,0.99663656115280497,1.992193850719004,1.6709860318927536,,20240517,27,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.42653311532683291,19600,8.4191590379063506,1.0026641347640624,1.8083603659346279,1.6721906038366092,,20240518,28,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.40907369202028498,19600,9.1927108970313682,1.0909739732912638,1.7816137916085939,1.6858914320555614,,20240519,29,,,,\n'
    'daily,,,O2,PhysicsNewBias,0.30006987273092939,19600,10.988496379510677,1.2645584783917148,2.1608363447496779,1.977160922123846,,20240520,30,,,,\n'
    'daily,,,O2,Placebo,0.54626817101645864,19600,7.0142796844585238,0.77759556506635086,1.4578537920134669,1.2887861176593136,,20240421,1,,,,\n'
    'daily,,,O2,Placebo,0.5743009056803563,19600,7.6736303038441518,0.75764164861428518,1.3044834490961654,1.1401707996061838,,20240422,2,,,,\n'
    'daily,,,O2,Placebo,0.50430771054756851,19600,7.5893610753081813,0.90799398194884662,1.6502238794700137,1.3100380769048487,,20240423,3,,,,\n'
    'daily,,,O2,Placebo,0.50831700693074877,17150,9.1221626574350552,0.82837838142523401,1.5498097407566398,1.3091481201966371,,20240428,8,,,,\n'
    'daily,,,O2,Placebo,0.56614125913585167,19600,7.5277817455036251,0.76606277634198727,1.3849272446376488,1.1490824158968069,,20240429,9,,,,\n'
    'daily,,,O2,Placebo,0.5630638863119144,19600,6.3811389834187882,0.75159299125710677,1.4190667127411236,1.2571795277138669,,20240430,10,,,,\n'
    'daily,,,O2,Placebo,0.53592296474073597,19600,7.3328885665103938,0.72397640024369847,1.5518698457010556,1.3298460894369066,,20240501,11,,,,\n'
    'daily,,,O2,Placebo,0.448084982003986,19600,9.1998819231248046,0.92531671378239255,1.8134254796697984,1.5021932014264039,,20240502,12,,,,\n'
    'daily,,,O2,Placebo,0.46137374154931221,19600,8.097990517921474,0.93151549854210158,1.7875194680427555,1.4958646060421834,,20240503,13,,,,\n'
    'daily,,,O2,Placebo,0.40209617435337919,19600,8.969621864285056,1.0508123717591811,1.968397078363064,1.6574918572208821,,20240504,14,,,,\n'
    'daily,,,O2,Placebo,0.46784853968179568,19600,8.0230810178535581,0.86032198652854186,1.7872039621418416,1.5201755888716435,,20240505,15,,,,\n'
    'daily,,,O2,Placebo,0.4461042954819292,19600,10.071476513911696,0.94690594945991724,1.8060744637239032,1.4308921126434957,,20240506,16,,,,\n'
    'daily,,,O2,Placebo,0.46639626563639491,19600,8.9732928406819124,0.95106616572090397,1.7217119484765886,1.4134151847073733,,20240507,17,,,,\n'
    'daily,,,O2,Placebo,0.35601483582986571,19600,9.6787584995628961,1.1335430233447776,2.1936990654139072,1.7246288760310948,,20240508,18,,,,\n'
    'daily,,,O2,Placebo,0.3892860574952402,19600,10.615324656057856,1.0725107422031466,1.8930415506992984,1.6658069625991387,,20240509,19,,,,\n'
    'daily,,,O2,Placebo,0.39455347345930958,19600,10.972855466530969,0.9655335569097504,1.9173458189881307,1.6815790897547029,,20240510,20,,,,\n'
    'daily,,,O2,Placebo,0.33980893286572522,19600,10.17865884120746,1.15198504137456,2.1606149944834931,1.8259556073017111,,20240511,21,,,,\n'
    'daily,,,O2,Placebo,0.2277700366529701,19600,14.383630102536411,1.2506037347371208,2.4312308157946978,2.1047029058411337,,20240512,22,,,,\n'
    'daily,,,O2,Placebo,0.31583118302937918,19600,9.7993040124436117,1.1361279361768406,2.3674117749045451,1.9122066530045612,,20240513,23,,,,\n'
    'daily,,,O2,Placebo,0.29901171817760641,19600,8.8221901456903069,1.3292772401324513,2.3565722994424987,1.9462973237503756,,20240514,24,,,,\n'
    'daily,,,O2,Placebo,0.29824388747667407,19600,10.8825929244887,1.1726910102802104,2.3261264066735952,1.9651213926534965,,20240515,25,,,,\n'
    'daily,,,O2,Placebo,0.34569151095490208,19600,9.3934724199997817,1.1438572156298965,2.2130593998321206,1.8087967069817521,,20240516,26,,,,\n'
    'daily,,,O2,Placebo,0.3542544075598415,17150,9.1197274443721437,1.0904338048605904,2.251105638388168,1.7841673843912949,,20240517,27,,,,\n'
    'daily,,,O2,Placebo,0.36946580868630657,19600,8.7935323210974552,1.10483446738797,2.0905919698885604,1.7945862029926325,,20240518,28,,,,\n'
    'daily,,,O2,Placebo,0.3722560946400944,19600,9.1909927369057272,1.1335516349336348,1.9799163096133288,1.7958278209305811,,20240519,29,,,,\n'
    'daily,,,O2,Placebo,0.2183470982791193,19600,10.932529797724303,1.4048731310055125,2.5409482105597445,2.2259937902429221,,20240520,30,,,,\n'
    'daily,,,O2,Recent,0.54247133434106065,19600,6.8359560671497404,0.79329689455260877,1.4771827612099271,1.3042290458387531,,20240421,1,,,,\n'
    'daily,,,O2,Recent,0.58029126280257415,19600,7.5547069982694124,0.75281039797630911,1.2577595037040092,1.1412506591564089,,20240422,2,,,,\n'
    'daily,,,O2,Recent,0.5005095206305612,19600,7.4835959325728352,0.92348419821986361,1.6935097858730066,1.3010546872642064,,20240423,3,,,,\n'
    'daily,,,O2,Recent,0.54371031678079906,17150,8.862425648680377,0.75231477067194785,1.3817208903402829,1.2429206905650318,,20240428,8,,,,\n'
    'daily,,,O2,Recent,0.57232287853810682,19600,7.1719213396235775,0.76384416822915135,1.3509376172481571,1.1555001572250601,,20240429,9,,,,\n'
    'daily,,,O2,Recent,0.56849120021971256,19600,6.1723713874162396,0.74106704312036054,1.3959522422708457,1.2577061372208584,,20240430,10,,,,\n'
    'daily,,,O2,Recent,0.55380293816278481,19600,6.8498612549992357,0.70878512351030087,1.4643325120625492,1.3039081263488654,,20240501,11,,,,\n'
    'daily,,,O2,Recent,0.47965235801129208,19600,9.0532246808541714,0.86309341770587722,1.674901990168514,1.4215641131901291,,20240502,12,,,,\n'
    'daily,,,O2,Recent,0.47251045739394482,19600,8.1270816023868164,0.89684844053618162,1.7620489012441976,1.4553751219283813,,20240503,13,,,,\n'
    'daily,,,O2,Recent,0.42639531276728748,19600,9.1914998721153047,0.95565846882996996,1.8794939089134,1.6018192832712983,,20240504,14,,,,\n'
    'daily,,,O2,Recent,0.49515721672953611,19600,7.7310615176131252,0.79372906485301098,1.6981859791106242,1.4528942425995699,,20240505,15,,,,\n'
    'daily,,,O2,Recent,0.48386703290276351,19600,9.6707627726690273,0.89087783132087228,1.6003572601573739,1.3655682925007548,,20240506,16,,,,\n'
    'daily,,,O2,Recent,0.47587540385691129,19600,9.1228376989941804,0.94691328895162197,1.6385684126072917,1.3909713264785031,,20240507,17,,,,\n'
    'daily,,,O2,Recent,0.41381803270110562,19600,9.612009162334898,1.0248189340499261,1.9379586985978297,1.5567359805704075,,20240508,18,,,,\n'
    'daily,,,O2,Recent,0.40967060914093012,19600,10.677048737351662,1.0505488477761258,1.758107503058711,1.6179086557627915,,20240509,19,,,,\n'
    'daily,,,O2,Recent,0.45304408991933331,19600,10.512454068522166,0.85815852173213114,1.6627294997833451,1.5381108029363766,,20240510,20,,,,\n'
    'daily,,,O2,Recent,0.38290540134855622,19600,9.611523951205136,1.0703943163521721,1.9908291611082769,1.7276184109050163,,20240511,21,,,,\n'
    'daily,,,O2,Recent,0.30538188804687261,19600,14.071860250430133,1.1391317507374503,2.079583525836965,1.869121430559632,,20240512,22,,,,\n'
    'daily,,,O2,Recent,0.39835412657309938,19600,9.2039808816797208,1.0166339799532249,1.9954413565601428,1.682911284531001,,20240513,23,,,,\n'
    'daily,,,O2,Recent,0.4141918249497008,19600,8.5424827648270529,1.1035336404205986,1.843472417206856,1.6357546687031319,,20240514,24,,,,\n'
    'daily,,,O2,Recent,0.39177733893137401,19600,10.241447494921596,1.0097085697952211,1.8929684053122,1.7393211934430088,,20240515,25,,,,\n'
    'daily,,,O2,Recent,0.44463808532789773,19600,8.6396153099240642,0.98745051864239874,1.7563579580155244,1.5587816924523252,,20240516,26,,,,\n'
    'daily,,,O2,Recent,0.40221130195315891,17150,8.8546091860130129,1.0033716894767564,2.0160973252357879,1.6767637623292366,,20240517,27,,,,\n'
    'daily,,,O2,Recent,0.42767738952773809,19600,8.2759253677803102,0.9926636273331314,1.8183604835172871,1.6767386045865516,,20240518,28,,,,\n'
    'daily,,,O2,Recent,0.4031476926224698,19600,9.2166959118750285,1.113508611022733,1.8132592798140621,1.6859499135720859,,20240519,29,,,,\n'
    'daily,,,O2,Recent,0.30373204850891189,19600,11.030330647609484,1.2611113262160236,2.1510104739759321,1.9542818657783985,,20240520,30,,,,\n'
    'daily,,,O2,RecentBias,0.54577955664398869,19600,6.7721056822977843,0.7899138028516256,1.4662420311976661,1.29380990504919,,20240421,1,,,,\n'
    'daily,,,O2,RecentBias,0.58370802992037529,19600,7.3636785773983666,0.74825726048185215,1.2555706040533492,1.1347535872504038,,20240422,2,,,,\n'
    'daily,,,O2,RecentBias,0.50153635296925581,19600,7.4884564019378903,0.9167181920640376,1.6884480480215325,1.3028659136566287,,20240423,3,,,,\n'
    'daily,,,O2,RecentBias,0.54480395913974078,17150,8.9505893610992526,0.74351632583804672,1.3766359495943807,1.2393891927736012,,20240428,8,,,,\n'
    'daily,,,O2,RecentBias,0.57505201747059609,19600,7.2749509243410388,0.75340438073706029,1.3364274462401291,1.1460954292185077,,20240429,9,,,,\n'
    'daily,,,O2,RecentBias,0.56936222545784931,19600,6.3272412244715888,0.73981854753973453,1.389139796207441,1.2442574421977379,,20240430,10,,,,\n'
    'daily,,,O2,RecentBias,0.5562166138560507,19600,6.9155785870644531,0.69856729164262854,1.4558876766846265,1.2952559127047254,,20240501,11,,,,\n'
    'daily,,,O2,RecentBias,0.48517740748068389,19600,8.7851672279879871,0.85535900861253567,1.6630376822807829,1.4145622054291551,,20240502,12,,,,\n'
    'daily,,,O2,RecentBias,0.4791096025119676,19600,7.8590947631197396,0.88941357802728627,1.7453058687485234,1.4428991917915299,,20240503,13,,,,\n'
    'daily,,,O2,RecentBias,0.43221634858147068,19600,8.9245084855931669,0.94548729295102119,1.8677609205885068,1.5946188272388808,,20240504,14,,,,\n'
    'daily,,,O2,RecentBias,0.49773750342447332,19600,7.5705017964777204,0.78831453499209059,1.6944974568131606,1.4529138970508677,,20240505,15,,,,\n'
    'daily,,,O2,RecentBias,0.49002100502645402,19600,9.5341064622036544,0.87497150324971495,1.5878210311101484,1.3512822899200212,,20240506,16,,,,\n'
    'daily,,,O2,RecentBias,0.48151264205085481,19600,8.789804549710647,0.93904356426134217,1.6306061331879416,1.3854758605749025,,20240507,17,,,,\n'
    'daily,,,O2,RecentBias,0.41872302904534842,19600,9.2538834712948379,1.0203485804578361,1.932111878095776,1.5543350433742829,,20240508,18,,,,\n'
    'daily,,,O2,RecentBias,0.41722112025305469,19600,10.394507448213131,1.0304032465551172,1.7450425151297777,1.6086675456705759,,20240509,19,,,,\n'
    'daily,,,O2,RecentBias,0.4566304339610765,19600,10.580363750027438,0.84676311777454705,1.6457501087248378,1.5273302084012981,,20240510,20,,,,\n'
    'daily,,,O2,RecentBias,0.3897975110827776,19600,9.4203546752142913,1.0489921363503576,1.9785505886508652,1.7170106750152043,,20240511,21,,,,\n'
    'daily,,,O2,RecentBias,0.3119204245401978,19600,13.885214759735744,1.1210137591655316,2.0661401030371458,1.858682150413757,,20240512,22,,,,\n'
    'daily,,,O2,RecentBias,0.40372096833077609,19600,9.1647009487077966,0.9974432223519758,1.9844113043742595,1.6693280049704304,,20240513,23,,,,\n'
    'daily,,,O2,RecentBias,0.41872656699481609,19600,8.4219177744749292,1.0919024845667429,1.8337065361606399,1.6271838160617789,,20240514,24,,,,\n'
    'daily,,,O2,RecentBias,0.395500990346838,19600,10.256462659655384,0.99574471945204557,1.881958187280206,1.7296906492470512,,20240515,25,,,,\n'
    'daily,,,O2,RecentBias,0.45061288602981769,19600,8.4824092832384075,0.97120004906985324,1.740103998316832,1.55092988264883,,20240516,26,,,,\n'
    'daily,,,O2,RecentBias,0.40851881052732758,17150,8.6369906934202305,0.98802225904149676,2.001729648825973,1.6687253196558189,,20240517,27,,,,\n'
    'daily,,,O2,RecentBias,0.43354907420929623,19600,8.1631355650124959,0.9757054796400324,1.806129180469364,1.6636706487632391,,20240518,28,,,,\n'
    'daily,,,O2,RecentBias,0.41213032050948117,19600,8.8786668458862881,1.0888034072155801,1.8006593238607609,1.6735642264569011,,20240519,29,,,,\n'
    'daily,,,O2,RecentBias,0.31009129807176128,19600,10.794369268659104,1.2429690383280794,2.1396252458500618,1.9478192000121133,,20240520,30,,,,\n'
    'daily,,,O2,Drift,0.54102655562947166,19600,6.7593404223085249,0.81130911737622557,1.4749732298765448,1.3061154974690377,,20240421,1,,,,\n'
    'daily,,,O2,Drift,0.58326295732106703,19600,7.5521347077489178,0.75002679385391602,1.2510745119076154,1.1246733415836889,,20240422,2,,,,\n'
    'daily,,,O2,Drift,0.49685857026002828,19600,7.45950783512715,0.92383828631975839,1.7097213119517185,1.3205271815353872,,20240423,3,,,,\n'
    'daily,,,O2,Drift,0.54291072715829936,17150,9.0812733692603391,0.76544759015424091,1.3668606353313086,1.2298929284191693,,20240428,8,,,,\n'
    'daily,,,O2,Drift,0.57328827951680084,19600,7.2601758999639383,0.76880459006427337,1.3324898609017075,1.1495423117387331,,20240429,9,,,,\n'
    'daily,,,O2,Drift,0.57217575525159714,19600,5.9492874655133399,0.73064100616186067,1.3924934342699915,1.2585383811807955,,20240430,10,,,,\n'
    'daily,,,O2,Drift,0.55678646326956271,19600,7.0068682477471551,0.69119798621392514,1.4542618969167775,1.2916993846793212,,20240501,11,,,,\n'
    'daily,,,O2,Drift,0.47203189227760939,19600,9.283798962454032,0.87613206164168655,1.6789021364894527,1.4499187012915391,,20240502,12,,,,\n'
    'daily,,,O2,Drift,0.46285643457644671,19600,8.3466007270184885,0.89180642150645084,1.7937317197321068,1.4983146534199281,,20240503,13,,,,\n'
    'daily,,,O2,Drift,0.42019442800124068,19600,9.4674653670874367,0.96460518303814236,1.8936912161335064,1.6107693218634744,,20240504,14,,,,\n'
    'daily,,,O2,Drift,0.48728352341433051,19600,7.8574049459767448,0.8001313765477942,1.7180007074596702,1.4863219410852555,,20240505,15,,,,\n'
    'daily,,,O2,Drift,0.47974619356273479,19600,10.515479819982961,0.86582262022252687,1.5880033260524711,1.3654017715747298,,20240506,16,,,,\n'
    'daily,,,O2,Drift,0.47563794917374802,19600,9.5690864613705902,0.9445336552980016,1.6175501646782042,1.3750119823747111,,20240507,17,,,,\n'
    'daily,,,O2,Drift,0.41012557418241319,19600,10.330913176762367,1.0402562656079353,1.8989366677687105,1.5432091942532935,,20240508,18,,,,\n'
    'daily,,,O2,Drift,0.40592525121814871,19600,11.143327888559416,1.0297386445009191,1.7324578236167465,1.6518342195712614,,20240509,19,,,,\n'
    'daily,,,O2,Drift,0.44060010793867221,19600,11.14642699980779,0.86358458256561621,1.6816012595719811,1.5696369121492908,,20240510,20,,,,\n'
    'daily,,,O2,Drift,0.3770182802228787,19600,10.029305878742001,1.0655639428346273,2.0020168259664386,1.7385472633005441,,20240511,21,,,,\n'
    'daily,,,O2,Drift,0.2935381974397383,19600,14.999694479162129,1.1639924803701935,2.0452398089069961,1.8932887759074288,,20240512,22,,,,\n'
    'daily,,,O2,Drift,0.39545586025068208,19600,9.4217382759648398,1.0140442818034709,1.9782219738799176,1.7052304188660969,,20240513,23,,,,\n'
    'daily,,,O2,Drift,0.40442448117131341,19600,8.9047682998583966,1.0953077995717311,1.8610662568784175,1.682091018182972,,20240514,24,,,,\n'
    'daily,,,O2,Drift,0.37926234082030957,19600,10.232298712289634,1.0314115039712863,1.9434609647854659,1.7832745189942096,,20240515,25,,,,\n'
    'daily,,,O2,Drift,0.4260460831889275,19600,8.3830009765877662,1.0098793721381318,1.8384779766088903,1.648459203450972,,20240516,26,,,,\n'
    'daily,,,O2,Drift,0.38683578878629848,17150,8.9410286452108849,1.0011500721900064,2.0771953316937752,1.7526519616505802,,20240517,27,,,,\n'
    'daily,,,O2,Drift,0.42486307086923519,19600,8.5717517669979344,1.0046609820418528,1.7907420815541215,1.6852925712097644,,20240518,28,,,,\n'
    'daily,,,O2,Drift,0.394700083144712,19600,9.7649504430969003,1.1131541125254831,1.832903304494883,1.6964020922173553,,20240519,29,,,,\n'
    'daily,,,O2,Drift,0.29829467604743781,19600,11.214031370908286,1.2587597142036937,2.1576284308723803,1.9817815180371028,,20240520,30,,,,\n'
    'daily,,,O2,DriftBias,0.54417824752488564,19600,6.6785105221238092,0.80694239512387855,1.4615513158700839,1.3013299413267232,,20240421,1,,,,\n'
    'daily,,,O2,DriftBias,0.58656625967248144,19600,7.3194170303748516,0.74796551631789709,1.2457251217279357,1.122500747159888,,20240422,2,,,,\n'
    'daily,,,O2,DriftBias,0.49680517050954037,19600,7.5057489725873161,0.9185871800198504,1.7091105583370754,1.3229508226411582,,20240423,3,,,,\n'
    'daily,,,O2,DriftBias,0.54369302287152776,17150,9.1603274789728406,0.75865349768341561,1.3611275035043036,1.2281904615176424,,20240428,8,,,,\n'
    'daily,,,O2,DriftBias,0.57591105969352774,19600,7.3273296127525134,0.75879785378130493,1.3199876957471033,1.1419756769755194,,20240429,9,,,,\n'
    'daily,,,O2,DriftBias,0.57276715566312153,19600,6.1078249728599872,0.73117923258848716,1.3845927053330689,1.2461651297259695,,20240430,10,,,,\n'
    'daily,,,O2,DriftBias,0.55833583327349945,19600,7.0190534565105027,0.68383664048981574,1.4481454863233338,1.289968326246373,,20240501,11,,,,\n'
    'daily,,,O2,DriftBias,0.47694656683888792,19600,8.9252551222374787,0.87210625600421365,1.674421020288162,1.4459572228388438,,20240502,12,,,,\n'
    'daily,,,O2,DriftBias,0.46895824997762747,19600,8.0129971353217808,0.88990573752871405,1.7803408276174324,1.4871853931334678,,20240503,13,,,,\n'
    'daily,,,O2,DriftBias,0.42557314557377568,19600,9.1351344142312403,0.96276397660207758,1.8836856635111308,1.6028739416467921,,20240504,14,,,,\n'
    'daily,,,O2,DriftBias,0.49023662192904988,19600,7.6182922706837584,0.79688152952428248,1.7123671199626285,1.4890231527385074,,20240505,15,,,,\n'
    'daily,,,O2,DriftBias,0.48576572940850921,19600,10.230559510350298,0.85413905198632201,1.580004338217698,1.3566372150678596,,20240506,16,,,,\n'
    'daily,,,O2,DriftBias,0.48118987136439317,19600,9.1165830396007763,0.93565929758552724,1.6180808187605793,1.3744575512521224,,20240507,17,,,,\n'
    'daily,,,O2,DriftBias,0.4151410389049634,19600,9.8273786497039683,1.035806560227043,1.9002346572817213,1.546264307722125,,20240508,18,,,,\n'
    'daily,,,O2,DriftBias,0.41398308114320581,19600,10.737854713932832,1.0140499498073969,1.7213843449540125,1.6424184416833949,,20240509,19,,,,\n'
    'daily,,,O2,DriftBias,0.44532871327937701,19600,11.071296856474834,0.85508054248125442,1.6657824908090431,1.5572652107768179,,20240510,20,,,,\n'
    'daily,,,O2,DriftBias,0.38344809894296311,19600,9.7636831480944757,1.0470463328025486,1.9939684476785264,1.731714804794535,,20240511,21,,,,\n'
    'daily,,,O2,DriftBias,0.30037191603942531,19600,14.677516100841562,1.1478766127820783,2.0389178835458459,1.8839653466124504,,20240512,22,,,,\n'
    'daily,,,O2,DriftBias,0.40020638932619329,19600,9.3251735468409862,0.99800799536612461,1.9751376045894609,1.6921256369400732,,20240513,23,,,,\n'
    'daily,,,O2,DriftBias,0.40938599266247477,19600,8.693955367909183,1.086437477709691,1.8515954256426836,1.6743647395309451,,20240514,24,,,,\n'
    'daily,,,O2,DriftBias,0.38306487324993271,19600,10.23137346112917,1.0173843946638188,1.9316593298374689,1.774978455691286,,20240515,25,,,,\n'
    'daily,,,O2,DriftBias,0.4318421511348271,19600,8.195544285657169,0.99521775947190039,1.825202690260106,1.6406616531832832,,20240516,26,,,,\n'
    'daily,,,O2,DriftBias,0.39283753200327598,17150,8.6658660334271449,0.99007195099712342,2.0659484491781623,1.7451529587985333,,20240517,27,,,,\n'
    'daily,,,O2,DriftBias,0.4305364177616271,19600,8.3836094292678673,0.98964154924136638,1.783124854242069,1.6745266084331822,,20240518,28,,,,\n'
    'daily,,,O2,DriftBias,0.40340252964689549,19600,9.3257899299500142,1.0929310786410791,1.8231151641840375,1.6879580050490348,,20240519,29,,,,\n'
    'daily,,,O2,DriftBias,0.30467299158707278,19600,10.898039132959775,1.2419414228440793,2.1501695474679652,1.9773481277394336,,20240520,30,,,,\n'
    'daily,,,O2,MultiScale,0.54769917876574592,19600,6.8118115169955189,0.79388196411241818,1.4506830901008108,1.2826311433830719,,20240421,1,,,,\n'
    'daily,,,O2,MultiScale,0.58211475174999228,19600,7.3937385480686784,0.7551654539744691,1.2604218771877786,1.1348181679553164,,20240422,2,,,,\n'
    'daily,,,O2,MultiScale,0.49902439404439081,19600,7.5944636584163892,0.91827812286675603,1.6869868676994495,1.3146777690383955,,20240423,3,,,,\n'
    'daily,,,O2,MultiScale,0.54385989281630842,17150,9.0532899701374632,0.75443817122281875,1.3699360749512304,1.2330016246337347,,20240428,8,,,,\n'
    'daily,,,O2,MultiScale,0.57057638186650639,19600,7.3296481533205329,0.77530734664830769,1.344692665636624,1.1503816358233487,,20240429,9,,,,\n'
    'daily,,,O2,MultiScale,0.57018374969239749,19600,6.3099943311464841,0.73429001219923473,1.3900115833312208,1.24372593482139,,20240430,10,,,,\n'
    'daily,,,O2,MultiScale,0.55096739151454432,19600,6.9996800568113926,0.70922257763395691,1.476948846654011,1.3049746319119768,,20240501,11,,,,\n'
    'daily,,,O2,MultiScale,0.47509507747554619,19600,8.9599415677248189,0.88060047888548776,1.6887509446866511,1.4388038303107402,,20240502,12,,,,\n'
    'daily,,,O2,MultiScale,0.47833415815906161,19600,7.8377518665108168,0.88468996214766282,1.7474077523616434,1.4542674893317844,,20240503,13,,,,\n'
    'daily,,,O2,MultiScale,0.43155716509279568,19600,8.9709739647733926,0.96667705552572658,1.8586890058379772,1.5819516532947744,,20240504,14,,,,\n'
    'daily,,,O2,MultiScale,0.4978271354960348,19600,7.5356601055892076,0.79766750135805331,1.6848378975381559,1.4531102490205758,,20240505,15,,,,\n'
    'daily,,,O2,MultiScale,0.48272968632269059,19600,9.8372944184464206,0.88258837177495686,1.6003766133139126,1.3697212190949424,,20240506,16,,,,\n'
    'daily,,,O2,MultiScale,0.47778275444787671,19600,9.0467371938431462,0.95531615571863637,1.6162944929396366,1.3901806040456546,,20240507,17,,,,\n'
    'daily,,,O2,MultiScale,0.41480104539004448,19600,9.6642883168300617,1.0322155819615459,1.9223173052019091,1.5489418748824568,,20240508,18,,,,\n'
    'daily,,,O2,MultiScale,0.41182148986641698,19600,10.642167144218153,1.0399905932954037,1.7512301840235207,1.6188540446036377,,20240509,19,,,,\n'
    'daily,,,O2,MultiScale,0.44894187195827567,19600,10.693608791897709,0.86684593384262087,1.6594823751113901,1.5512035509548523,,20240510,20,,,,\n'
    'daily,,,O2,MultiScale,0.38254622995140258,19600,9.7571464960365848,1.0596359461914968,1.9832753562349119,1.7353945996095701,,20240511,21,,,,\n'
    'daily,,,O2,MultiScale,0.30353005504059621,19600,14.31303405818344,1.1444373046558374,2.0507690571189872,1.8818340827865745,,20240512,22,,,,\n'
    'daily,,,O2,MultiScale,0.39748995770107648,19600,9.2734025249998808,1.0119257517999221,1.988770449237488,1.6943730381877542,,20240513,23,,,,\n'
    'daily,,,O2,MultiScale,0.40528603431016541,19600,8.5129049425830789,1.1184864998329669,1.8838157268949371,1.6659875445072387,,20240514,24,,,,\n'
    'daily,,,O2,MultiScale,0.39421165117266499,19600,9.9943167514961502,1.0019856721810434,1.9144251775557912,1.7304149897046546,,20240515,25,,,,\n'
    'daily,,,O2,MultiScale,0.44452298425141112,19600,8.2504030099417136,0.99456528660118759,1.7569772246231421,1.5841074978265697,,20240516,26,,,,\n'
    'daily,,,O2,MultiScale,0.4084363222836519,17150,8.5415629009199652,0.99837575809216839,1.9939669712436052,1.6728560137258908,,20240517,27,,,,\n'
    'daily,,,O2,MultiScale,0.42588343335339668,19600,8.4230199075872125,1.0044084528280424,1.8099798999545349,1.6742719738675318,,20240518,28,,,,\n'
    'daily,,,O2,MultiScale,0.40838890115419652,19600,9.1983742122035839,1.0927170164303346,1.7834329167365144,1.6879632774357609,,20240519,29,,,,\n'
    'daily,,,O2,MultiScale,0.29936980747123459,19600,10.99306318797019,1.2662883514367904,2.1627621128093528,1.9793807461266719,,20240520,30,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.54934629820558056,19600,6.7870445004964024,0.79330646393317128,1.4415580471693303,1.2786119368605098,,20240421,1,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.58369717296776991,19600,7.3361197639468081,0.75225651224233792,1.2588172947919185,1.1305389276043347,,20240422,2,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.4999816490677238,19600,7.5714933878755026,0.91816994619823822,1.6827610592206417,1.3119792035130466,,20240423,3,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.545710683779983,17150,9.0352968874205537,0.75274559443496725,1.3633357535732396,1.2258848436719632,,20240428,8,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.57217615459881266,19600,7.3174716790243748,0.77073117482118925,1.3398774215022391,1.1464052724744982,,20240429,9,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.57155566529872137,19600,6.2879241216454114,0.73216855161910099,1.3864546411457086,1.2390043327031746,,20240430,10,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.55359258558214086,19600,6.9363306806048479,0.70412442254647256,1.4702061032777132,1.2986784987827515,,20240501,11,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.47791787659858742,19600,8.8617099685573493,0.87835892536059945,1.6797611579016072,1.4325934017582675,,20240502,12,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.48095454723290948,19600,7.794914945238995,0.88248787990522071,1.7342018139124986,1.4484344903224542,,20240503,13,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.4341083794072822,19600,8.931690535276644,0.9614771680221158,1.8510793610532887,1.5750709238626348,,20240504,14,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.49903524031514612,19600,7.5031360816034578,0.79498454541094177,1.6836846474222622,1.4493106148275343,,20240505,15,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.48596795870028009,19600,9.770028807530668,0.87653577041917308,1.5905345124263111,1.3620092391190477,,20240506,16,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.4805031750956274,19600,8.9513085003835577,0.95187951848889996,1.6083469444239185,1.3849898068429571,,20240507,17,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.4181087832484755,19600,9.6003190727195893,1.0283242889564237,1.9043765090788061,1.5445079026801534,,20240508,18,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.41466646735448309,19600,10.596751950732688,1.0339872506647871,1.742312459346379,1.6118570521589053,,20240509,19,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.45093721982111828,19600,10.689545253903946,0.85838607312075321,1.6567450489980815,1.5455618884524382,,20240510,20,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.38673441494644512,19600,9.6744973922418875,1.049970641644169,1.9706299355463903,1.7268543230082689,,20240511,21,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.30732464996537678,19600,14.234826115471233,1.1382614423543134,2.0345574125890247,1.8754679789529556,,20240512,22,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.4003488757761472,19600,9.2327821834930983,1.0102379838883624,1.9772669346627691,1.6845185674458454,,20240513,23,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.40891341481004179,19600,8.4403635772641685,1.1109909746532107,1.8748301540349783,1.6562514282029617,,20240514,24,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.39698470263931518,19600,9.9654341458982927,0.99716633115979036,1.90352249810983,1.7230084016248164,,20240515,25,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.44813065826754889,19600,8.1593576560529986,0.9874374145344712,1.7463705782383083,1.5769513958903345,,20240516,26,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.41209362428341312,17150,8.4516736214831578,0.99339069205569197,1.9812530025994359,1.6646672650627183,,20240517,27,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.42909957629746409,19600,8.3539127284623031,0.99934667987981796,1.7984621233578355,1.6672012498153574,,20240518,28,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.41138436753509522,19600,9.1424115690776926,1.0888929837957158,1.771669688578362,1.6805934527986188,,20240519,29,,,,\n'
    'daily,,,O2,AdaptiveFlow,0.30287858163356601,19600,10.924652610979669,1.2598595308596805,2.1520527972757337,1.9705606467791712,,20240520,30,,,,\n'
    'daily,,,O2,WindFlow,0.54922652944358874,19600,6.8036283542545757,0.79294533809836099,1.4415580471693303,1.2786119368605098,,20240421,1,,,,\n'
    'daily,,,O2,WindFlow,0.58280646172285433,19600,7.3891584393950991,0.75529527759118886,1.2588172947919185,1.1305389276043347,,20240422,2,,,,\n'
    'daily,,,O2,WindFlow,0.499791877427749,19600,7.5893489680351118,0.91828353667713603,1.6827610592206417,1.3119792035130466,,20240423,3,,,,\n'
    'daily,,,O2,WindFlow,0.54533301458664996,17150,9.0520740033296363,0.75449919411610733,1.3633357535732396,1.2258848436719632,,20240428,8,,,,\n'
    'daily,,,O2,WindFlow,0.57157032523753271,19600,7.3276436186814422,0.77490745473362244,1.3398774215022391,1.1464052724744982,,20240429,9,,,,\n'
    'daily,,,O2,WindFlow,0.57110291854353068,19600,6.3083424914223443,0.73424582530841309,1.3864546411457086,1.2390043327031746,,20240430,10,,,,\n'
    'daily,,,O2,WindFlow,0.55248735270163873,19600,6.9903213372904878,0.70885778331518878,1.4702061032777132,1.2986784987827515,,20240501,11,,,,\n'
    'daily,,,O2,WindFlow,0.4768247803105512,19600,8.9498477026636785,0.88021124960597452,1.6797611579016072,1.4325934017582675,,20240502,12,,,,\n'
    'daily,,,O2,WindFlow,0.4803574407300063,19600,7.8322109308611596,0.88438324636170274,1.7342018139124986,1.4484344903224542,,20240503,13,,,,\n'
    'daily,,,O2,WindFlow,0.43321844659607578,19600,8.9628413511477287,0.96629194798037477,1.8510793610532887,1.5750709238626348,,20240504,14,,,,\n'
    'daily,,,O2,WindFlow,0.49855637255709268,19600,7.5366958366444257,0.7962074264189003,1.6836846474222622,1.4493106148275343,,20240505,15,,,,\n'
    'daily,,,O2,WindFlow,0.48475272502568728,19600,9.8319124348579034,0.88153505906912411,1.5905345124263111,1.3620092391190477,,20240506,16,,,,\n'
    'daily,,,O2,WindFlow,0.47938638812775197,19600,9.0330159379752288,0.95445120002886963,1.6083469444239185,1.3849898068429571,,20240507,17,,,,\n'
    'daily,,,O2,WindFlow,0.4172383223693425,19600,9.6515039985495008,1.0313467307978097,1.9043765090788061,1.5445079026801534,,20240508,18,,,,\n'
    'daily,,,O2,WindFlow,0.41363521816045989,19600,10.64153592518463,1.0388591962134712,1.742312459346379,1.6118570521589053,,20240509,19,,,,\n'
    'daily,,,O2,WindFlow,0.4500796105403635,19600,10.695884678047049,0.86495435876231097,1.6567450489980815,1.5455618884524382,,20240510,20,,,,\n'
    'daily,,,O2,WindFlow,0.38507266748432528,19600,9.7517444051056437,1.0574073690001378,1.9706299355463903,1.7268543230082689,,20240511,21,,,,\n'
    'daily,,,O2,WindFlow,0.30615792721488688,19600,14.301544641977069,1.1424662603463616,2.0345574125890247,1.8754679789529556,,20240512,22,,,,\n'
    'daily,,,O2,WindFlow,0.39995367515108782,19600,9.2672997271169724,1.0106917096135866,1.9772669346627691,1.6845185674458454,,20240513,23,,,,\n'
    'daily,,,O2,WindFlow,0.40750064025294558,19600,8.5112096656001945,1.1168922480444574,1.8748301540349783,1.6562514282029617,,20240514,24,,,,\n'
    'daily,,,O2,WindFlow,0.3962576167005839,19600,9.9969630829953449,1.0006050732466247,1.90352249810983,1.7230084016248164,,20240515,25,,,,\n'
    'daily,,,O2,WindFlow,0.44666642618986507,19600,8.2455330071796844,0.99251542461891418,1.7463705782383083,1.5769513958903345,,20240516,26,,,,\n'
    'daily,,,O2,WindFlow,0.41081439840174738,17150,8.5415750615763706,0.99663696766688636,1.9812530025994359,1.6646672650627183,,20240517,27,,,,\n'
    'daily,,,O2,WindFlow,0.42805477110532453,19600,8.4191545109356856,1.0026646204452705,1.7984621233578355,1.6672012498153574,,20240518,28,,,,\n'
    'daily,,,O2,WindFlow,0.41063659628202748,19600,9.1927096132984083,1.090974134671707,1.771669688578362,1.6805934527986188,,20240519,29,,,,\n'
    'daily,,,O2,WindFlow,0.30168028003348579,19600,10.988497899672367,1.2645591984854401,2.1520527972757337,1.9705606467791712,,20240520,30,,,,\n'
    'daily,,,O2,Taper,0.54820891019629614,19600,6.8042137025119045,0.7930585090976493,1.4476590426304454,1.282190219086403,,20240421,1,,,,\n'
    'daily,,,O2,Taper,0.58213502521365701,19600,7.3992777729197288,0.75531509763956683,1.2597883696172469,1.1345355447064716,,20240422,2,,,,\n'
    'daily,,,O2,Taper,0.4990676607973914,19600,7.5839998972173133,0.91879450348662761,1.6876114058012901,1.3141689318802796,,20240423,3,,,,\n'
    'daily,,,O2,Taper,0.54387411655127571,17150,9.024138447903848,0.75656867904435954,1.3697569608378299,1.23326859435935,,20240428,8,,,,\n'
    'daily,,,O2,Taper,0.57022446290429163,19600,7.298537299679797,0.77782870480437272,1.3472688683029193,1.1513240511598355,,20240429,9,,,,\n'
    'daily,,,O2,Taper,0.57036373185022093,19600,6.2580520748639303,0.734189491134005,1.3905576393073098,1.2461747031992012,,20240430,10,,,,\n'
    'daily,,,O2,Taper,0.5508929977388789,19600,6.9681348104780563,0.7116770577188648,1.4768458657615746,1.3057917976191411,,20240501,11,,,,\n'
    'daily,,,O2,Taper,0.47393092571607998,19600,9.043317259419684,0.88241426325729999,1.6899455608872198,1.4388967479261039,,20240502,12,,,,\n'
    'daily,,,O2,Taper,0.4767997436804392,19600,7.9223112624754792,0.88592449779467175,1.7505823954566324,1.4563966046151411,,20240503,13,,,,\n'
    'daily,,,O2,Taper,0.42997670215661532,19600,9.0618905549670554,0.96805037230189239,1.8624627974943819,1.5833345615816925,,20240504,14,,,,\n'
    'daily,,,O2,Taper,0.49712894081788378,19600,7.6029896284489036,0.79783400927371217,1.6865001902970471,1.4519336548289927,,20240505,15,,,,\n'
    'daily,,,O2,Taper,0.48089965033420512,19600,9.9042249770995205,0.88775467282029386,1.6024939714584376,1.3726199785547033,,20240506,16,,,,\n'
    'daily,,,O2,Taper,0.47570848198947058,19600,9.1904767586604823,0.9591273121260524,1.6175851055542387,1.3908104430173325,,20240507,17,,,,\n'
    'daily,,,O2,Taper,0.41301289187418078,19600,9.8380601450708554,1.0332588591292236,1.922642458084844,1.5482609167639281,,20240508,18,,,,\n'
    'daily,,,O2,Taper,0.4086853046057034,19600,10.792030163920108,1.0480199695303625,1.7560194183648576,1.620855521177945,,20240509,19,,,,\n'
    'daily,,,O2,Taper,0.44773227362164703,19600,10.682089347200812,0.86984829042993639,1.6655391718702739,1.5544953219209388,,20240510,20,,,,\n'
    'daily,,,O2,Taper,0.38021454542623417,19600,9.8494766822269959,1.0680397785874027,1.9853863990688316,1.7371237397460122,,20240511,21,,,,\n'
    'daily,,,O2,Taper,0.30078353706065059,19600,14.423156065929156,1.1528382659242205,2.053980559493914,1.8847075753717792,,20240512,22,,,,\n'
    'daily,,,O2,Taper,0.395501047975447,19600,9.2832183748955366,1.0215411618368369,1.991123405312156,1.698619009879873,,20240513,23,,,,\n'
    'daily,,,O2,Taper,0.4031190562941887,19600,8.5886346896130448,1.1240462099329405,1.887744317267936,1.669165758167044,,20240514,24,,,,\n'
    'daily,,,O2,Taper,0.39259461607468221,19600,9.9763619051196262,1.0097648490465403,1.918935005522572,1.7340162418318086,,20240515,25,,,,\n'
    'daily,,,O2,Taper,0.4417948670522886,19600,8.3329302079670171,1.0026394243648222,1.7638289063728607,1.5865911930288537,,20240516,26,,,,\n'
    'daily,,,O2,Taper,0.40542805297015572,17150,8.6690869480790411,1.005017150838281,2.0004681938671744,1.6756965347894508,,20240517,27,,,,\n'
    'daily,,,O2,Taper,0.42282206695638702,19600,8.4953716901578318,1.0140433027666078,1.815476517914584,1.679854899073143,,20240518,28,,,,\n'
    'daily,,,O2,Taper,0.4035124527688787,19600,9.4210712441711966,1.1063740212226554,1.7888679198121626,1.6923089084998613,,20240519,29,,,,\n'
    'daily,,,O2,Taper,0.29589296130173048,19600,11.148501835713208,1.2770990957536972,2.1676275642843499,1.980896940710914,,20240520,30,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.57094365466629871,19600,6.3750845245800356,0.76074133174711867,1.385791275801542,1.2085744458241985,,20240421,1,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.60479568694571761,19600,6.9161346215184789,0.71621988614132837,1.2066624603330007,1.0660206795883389,,20240422,2,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.53026856652324694,19600,7.0587397627781847,0.89139171704818321,1.5289366702207079,1.248278882540564,,20240423,3,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.58133996541579447,17150,8.5047446011333108,0.6933605568708554,1.2359662254302843,1.1313736582975029,,20240428,8,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.59229425531193336,19600,7.0715778106575415,0.73963224381191439,1.2779380490900747,1.0783802124901829,,20240429,9,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.58193356377818295,19600,6.0375577104479357,0.69941909085325471,1.3615967467291561,1.2254397618689159,,20240430,10,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.57281485775696805,19600,6.8914628107804994,0.67264159177478422,1.3772700983092192,1.2461404785579575,,20240501,11,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.50758737071007842,19600,8.0506574427174069,0.81903107240440487,1.5942701214029409,1.3782307023778424,,20240502,12,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.5047840096935966,19600,7.2532944543938926,0.84360305984013662,1.6650320126513642,1.387269446856956,,20240503,13,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.47075552375192609,19600,8.3569767744078938,0.89296538164441286,1.7179260622316672,1.4894305668579333,,20240504,14,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.52046422743848453,19600,7.4364163515971713,0.76005742115141861,1.6137279792487464,1.3657101900495641,,20240505,15,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.51939256733312633,19600,9.0160591974716802,0.82348167801475503,1.4701549781609489,1.292449973367723,,20240506,16,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.51163144724208731,19600,8.0424754702468721,0.88683392130474337,1.4946997829689876,1.3542266631056952,,20240507,17,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.43886640848597303,19600,9.3134212892386099,1.0141182537779505,1.7155003007424734,1.5564085578050739,,20240508,18,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.44232943602248809,19600,10.179782075381883,0.97383131631292363,1.6599482099391971,1.5402085588962788,,20240509,19,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.47275764769159778,19600,10.273644938092898,0.80682402389554297,1.6233117591126247,1.4757766153225202,,20240510,20,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.40416382127818729,19600,9.5121843937430341,1.0263145864596954,1.8968500084958693,1.6761086000338143,,20240511,21,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.3483324419963017,19600,13.344198579647172,1.0937134715072474,1.815970242642166,1.821805126030376,,20240512,22,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.43628520059938469,19600,8.6538186672026267,0.96138883057115399,1.845986884828464,1.5838272847751291,,20240513,23,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.45130146278919669,19600,8.4159747160739347,1.0194424811049514,1.7081320677675484,1.5268101285683882,,20240514,24,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.40582361966254349,19600,10.682284915430202,1.0065633404129668,1.8328315863288589,1.6358052267245846,,20240515,25,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.4567385921245033,19600,9.1933089768980913,0.96758101581164724,1.6985115054198654,1.4773435265717385,,20240516,26,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.42481095824282261,17150,8.6449292772738691,0.97418324670899981,1.9461878046874488,1.5893183808526772,,20240517,27,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.44426377387620952,19600,7.925533057768714,0.98033736941371685,1.7181991116533504,1.6576305790886201,,20240518,28,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.4216030270821251,19600,9.0516306641951552,1.1074228837719504,1.700361486961748,1.639601316720475,,20240519,29,,,,\n'
    'daily,,,O2,MixPhysicsNewBias,0.32561241869105739,19600,10.587325123191237,1.2595605712854274,2.0174533716091281,1.9137354492878456,,20240520,30,,,,\n'
    'daily,,,O2,MixRecentBias,0.57069426000867007,19600,6.3747591398956729,0.7616956274928659,1.386390502686339,1.2092519404722413,,20240421,1,,,,\n'
    'daily,,,O2,MixRecentBias,0.60502913197247088,19600,6.9226849439315128,0.71371454493663811,1.2066505570334296,1.0660739023914489,,20240422,2,,,,\n'
    'daily,,,O2,MixRecentBias,0.53042901962267874,19600,7.047648387508783,0.89189566267137632,1.5288930967839105,1.2473766221527476,,20240423,3,,,,\n'
    'daily,,,O2,MixRecentBias,0.58166916704529803,17150,8.5029223427851726,0.6904909762507635,1.2361477712322075,1.1315338612426475,,20240428,8,,,,\n'
    'daily,,,O2,MixRecentBias,0.5931922985792305,19600,7.0720351535247223,0.73242463618435982,1.2778141165628849,1.0782259528540097,,20240429,9,,,,\n'
    'daily,,,O2,MixRecentBias,0.58164859970149463,19600,6.0426307739569758,0.70134214960775754,1.361693325515704,1.2253825033342962,,20240430,10,,,,\n'
    'daily,,,O2,MixRecentBias,0.57348849139773617,19600,6.8866334716927655,0.66865589857536545,1.3767293667216838,1.2453634195772023,,20240501,11,,,,\n'
    'daily,,,O2,MixRecentBias,0.50857370808958047,19600,8.0385403183754391,0.81302435574292931,1.5938902271148476,1.3773599094183646,,20240502,12,,,,\n'
    'daily,,,O2,MixRecentBias,0.50459904567991609,19600,7.2614005022635455,0.84492990003043889,1.6650284626771712,1.3868082278577851,,20240503,13,,,,\n'
    'daily,,,O2,MixRecentBias,0.47134541959374981,19600,8.3573901178303913,0.886795008707707,1.7181361640532029,1.4905441674733471,,20240504,14,,,,\n'
    'daily,,,O2,MixRecentBias,0.52069153201094553,19600,7.4415324287145008,0.75662801657267853,1.6140288064466422,1.3666230179236127,,20240505,15,,,,\n'
    'daily,,,O2,MixRecentBias,0.52003283126887334,19600,8.9786829452195089,0.82194055119334208,1.4700864219236012,1.2917915672206799,,20240506,16,,,,\n'
    'daily,,,O2,MixRecentBias,0.51252976070417355,19600,8.0134013277506408,0.88146659795435212,1.4949126232083838,1.3543783273307726,,20240507,17,,,,\n'
    'daily,,,O2,MixRecentBias,0.43987268944582519,19600,9.2649738259797871,1.0083988386704184,1.7156887989196368,1.5576283241713482,,20240508,18,,,,\n'
    'daily,,,O2,MixRecentBias,0.4432134217564565,19600,10.147398842199742,0.96998248371887841,1.6597849723899427,1.5395105145436008,,20240509,19,,,,\n'
    'daily,,,O2,MixRecentBias,0.47385757496472242,19600,10.252266970339196,0.80091884713561512,1.623034879877095,1.4745366838080201,,20240510,20,,,,\n'
    'daily,,,O2,MixRecentBias,0.40503957014817721,19600,9.4762218462801755,1.0228392071779169,1.8968087687941839,1.6753013609954139,,20240511,21,,,,\n'
    'daily,,,O2,MixRecentBias,0.3499579884248018,19600,13.297721983100452,1.0850531415865641,1.8163224784248837,1.82053669360333,,20240512,22,,,,\n'
    'daily,,,O2,MixRecentBias,0.43718479396353421,19600,8.6414022116989955,0.95636821399849081,1.8460335823020448,1.5823760901090105,,20240513,23,,,,\n'
    'daily,,,O2,MixRecentBias,0.45258657898218541,19600,8.4227933481418376,1.0111333854827771,1.7071471138040253,1.5246847361662896,,20240514,24,,,,\n'
    'daily,,,O2,MixRecentBias,0.40529449286765629,19600,10.741080209803748,1.0061223420378702,1.8321185400536932,1.6363854106791529,,20240515,25,,,,\n'
    'daily,,,O2,MixRecentBias,0.45745461293845169,19600,9.2484907970964567,0.95972109303680242,1.6979830141381047,1.475179190254309,,20240516,26,,,,\n'
    'daily,,,O2,MixRecentBias,0.42504719627000898,17150,8.6662892197622572,0.97046117038753044,1.9463819308644028,1.5891984743513363,,20240517,27,,,,\n'
    'daily,,,O2,MixRecentBias,0.44573756458822489,19600,7.8897360514250527,0.97118349950407601,1.7181774013793587,1.657540680001383,,20240518,28,,,,\n'
    'daily,,,O2,MixRecentBias,0.42217143306996019,19600,9.0075339699861487,1.106243449240722,1.7008789739560053,1.6392736446731266,,20240519,29,,,,\n'
    'daily,,,O2,MixRecentBias,0.32719431819520067,19600,10.548797672574093,1.2513520664428863,2.0168775917829098,1.9124439282658732,,20240520,30,,,,\n'
    'daily,,,O2,MixDriftBias,0.5705698622254094,19600,6.3584673678133887,0.76504509558365008,1.3857165955076456,1.2087714440381394,,20240421,1,,,,\n'
    'daily,,,O2,MixDriftBias,0.60557354192247992,19600,6.903992661053846,0.7135037199319173,1.2059437472190671,1.0638307627145589,,20240422,2,,,,\n'
    'daily,,,O2,MixDriftBias,0.53027987948655597,19600,7.0447642646903104,0.8926769635796028,1.5288672955218314,1.2480897597589244,,20240423,3,,,,\n'
    'daily,,,O2,MixDriftBias,0.58149871390015517,17150,8.5063506111571296,0.69476451777344073,1.2350396333139784,1.1292190723168989,,20240428,8,,,,\n'
    'daily,,,O2,MixDriftBias,0.59351896703663765,19600,7.062306210009595,0.7327708743972009,1.2771614834273723,1.0764578476968276,,20240429,9,,,,\n'
    'daily,,,O2,MixDriftBias,0.58268458048494631,19600,6.0030994548754633,0.69808384391150147,1.3608924207103232,1.2239038874947543,,20240430,10,,,,\n'
    'daily,,,O2,MixDriftBias,0.57427252011000596,19600,6.8876720468479729,0.66389170104492012,1.3761562826968663,1.2439936510299949,,20240501,11,,,,\n'
    'daily,,,O2,MixDriftBias,0.50807272507209977,19600,8.0388471768196599,0.8168447961482812,1.5936256003245279,1.3778624965712898,,20240502,12,,,,\n'
    'daily,,,O2,MixDriftBias,0.50447407980512982,19600,7.2541124781211694,0.84562059538217005,1.6646260345257844,1.3880672505003395,,20240503,13,,,,\n'
    'daily,,,O2,MixDriftBias,0.47108868316688091,19600,8.3580586671051851,0.89073859822572476,1.7175419708767361,1.4891098867312049,,20240504,14,,,,\n'
    'daily,,,O2,MixDriftBias,0.52062714643931463,19600,7.4238191039839414,0.75869377876815258,1.6135197298458894,1.3669275345265,,20240505,15,,,,\n'
    'daily,,,O2,MixDriftBias,0.52068546276222538,19600,9.0547706821221023,0.81285089454053383,1.4692354391665423,1.2899301146947566,,20240506,16,,,,\n'
    'daily,,,O2,MixDriftBias,0.51277758482184066,19600,8.0367861776283238,0.88063693721248604,1.4941980717664918,1.351779650491324,,20240507,17,,,,\n'
    'daily,,,O2,MixDriftBias,0.43921514342336332,19600,9.3174947089425793,1.0138154883075383,1.7142947099828278,1.5544071556758383,,20240508,18,,,,\n'
    'daily,,,O2,MixDriftBias,0.4435733261817138,19600,10.185909482979287,0.96371815360568336,1.6587077026508534,1.5405071987605918,,20240509,19,,,,\n'
    'daily,,,O2,MixDriftBias,0.47335962334961351,19600,10.307921310448609,0.800704551472046,1.6225473946287177,1.4747100804890985,,20240510,20,,,,\n'
    'daily,,,O2,MixDriftBias,0.40525763191034531,19600,9.4994513587808722,1.0194578608269615,1.8968610874700571,1.6749454682058968,,20240511,21,,,,\n'
    'daily,,,O2,MixDriftBias,0.34811829171674108,19600,13.384303914593596,1.0933090909168104,1.8154401474592703,1.8211140082084829,,20240512,22,,,,\n'
    'daily,,,O2,MixDriftBias,0.437021333169681,19600,8.6402971695458408,0.95795542141225076,1.8452658291598236,1.5828261771104692,,20240513,23,,,,\n'
    'daily,,,O2,MixDriftBias,0.45322370311208382,19600,8.393784961181229,1.0075542220002391,1.7066665194367974,1.5257451065719323,,20240514,24,,,,\n'
    'daily,,,O2,MixDriftBias,0.4053725837945491,19600,10.682767291957784,1.0108357707013889,1.8314092755457247,1.636321986731484,,20240515,25,,,,\n'
    'daily,,,O2,MixDriftBias,0.4572022287091444,19600,9.1500643411827838,0.9656839317974476,1.6987071790217969,1.478796916989968,,20240516,26,,,,\n'
    'daily,,,O2,MixDriftBias,0.42511864789265269,17150,8.621471144996649,0.97035756561776243,1.9465887955142249,1.5922346322056322,,20240517,27,,,,\n'
    'daily,,,O2,MixDriftBias,0.4456560034554462,19600,7.8971096831490719,0.97395606543659075,1.7167753870026019,1.6559052079493972,,20240518,28,,,,\n'
    'daily,,,O2,MixDriftBias,0.42203053146276409,19600,9.0570085010848107,1.1051783748139516,1.7003324264990252,1.6378802989772716,,20240519,29,,,,\n'
    'daily,,,O2,MixDriftBias,0.3273140947683108,19600,10.570599720203742,1.2488853048164266,2.0165076689820407,1.9124309603749607,,20240520,30,,,,\n'
    'daily,,,O2,MixMultiScale,0.57090974398377492,19600,6.3756211029904097,0.76092305604803712,1.3858520322794521,1.2085823788450611,,20240421,1,,,,\n'
    'daily,,,O2,MixMultiScale,0.60481064341011725,19600,6.916218272353456,0.71607131204869823,1.2066727463884319,1.0660310671368118,,20240422,2,,,,\n'
    'daily,,,O2,MixMultiScale,0.53028151802028889,19600,7.0588866150870508,0.89131167844882031,1.5289126112273808,1.2482583080884826,,20240423,3,,,,\n'
    'daily,,,O2,MixMultiScale,0.58136297203571985,17150,8.504091149203969,0.69323837051336734,1.2359806248124567,1.1313476201871984,,20240428,8,,,,\n'
    'daily,,,O2,MixMultiScale,0.59229313163062991,19600,7.0710298167773411,0.73964499661192407,1.2779460227430373,1.0784155979662542,,20240429,9,,,,\n'
    'daily,,,O2,MixMultiScale,0.58193848710577212,19600,6.0373954668853562,0.69934711404200145,1.3616165031835403,1.2254693600928954,,20240430,10,,,,\n'
    'daily,,,O2,MixMultiScale,0.57280448594783762,19600,6.8920045827831089,0.67267332731871621,1.3772860199368693,1.246137996281625,,20240501,11,,,,\n'
    'daily,,,O2,MixMultiScale,0.50756044455391436,19600,8.0515350755605368,0.81909489145793379,1.5942916907482083,1.3783019731600028,,20240502,12,,,,\n'
    'daily,,,O2,MixMultiScale,0.504777305830862,19600,7.2529749791830866,0.84359831458260381,1.6650528225004495,1.387340297804752,,20240503,13,,,,\n'
    'daily,,,O2,MixMultiScale,0.47073750053901497,19600,8.3572772186335342,0.89302544129459283,1.7179294179109701,1.4894930083949456,,20240504,14,,,,\n'
    'daily,,,O2,MixMultiScale,0.52042534681905517,19600,7.4358966407024161,0.76042514320068133,1.6137376363777178,1.3656984009275797,,20240505,15,,,,\n'
    'daily,,,O2,MixMultiScale,0.51934442601787822,19600,9.0162185294105424,0.82378678252089665,1.4701872774152642,1.2925052606045648,,20240506,16,,,,\n'
    'daily,,,O2,MixMultiScale,0.51160908777497238,19600,8.0432975524795953,0.8869881302918704,1.4946840605950824,1.3542020773335717,,20240507,17,,,,\n'
    'daily,,,O2,MixMultiScale,0.43884042099002651,19600,9.3145376179237314,1.0142670454284335,1.7155010400957231,1.5563826872187534,,20240508,18,,,,\n'
    'daily,,,O2,MixMultiScale,0.44228145762232629,19600,10.179506056593601,0.97411067212653557,1.6599692879072958,1.5403330998820892,,20240509,19,,,,\n'
    'daily,,,O2,MixMultiScale,0.47268433225447087,19600,10.272748699711158,0.80736486759182302,1.6233435532180716,1.4758911073580181,,20240510,20,,,,\n'
    'daily,,,O2,MixMultiScale,0.4040698879474125,19600,9.5123832824429968,1.0269329015751594,1.896878154425331,1.6762296429942647,,20240511,21,,,,\n'
    'daily,,,O2,MixMultiScale,0.34823517446531399,19600,13.345144533043635,1.0943080254012318,1.8160006313581141,1.8219147839166081,,20240512,22,,,,\n'
    'daily,,,O2,MixMultiScale,0.4362293653535087,19600,8.6535889674698936,0.96170672553158176,1.8460019236927383,1.5839793718948931,,20240513,23,,,,\n'
    'daily,,,O2,MixMultiScale,0.45124110561178971,19600,8.415356975581485,1.0198938199371974,1.708162009202014,1.5268854085037311,,20240514,24,,,,\n'
    'daily,,,O2,MixMultiScale,0.40578432176560059,19600,10.681099160009188,1.0069047081526676,1.8328577661081675,1.6358655081159472,,20240515,25,,,,\n'
    'daily,,,O2,MixMultiScale,0.45666197189845747,19600,9.1930653558313047,0.96813573465354763,1.6985458535413529,1.4774158754766566,,20240516,26,,,,\n'
    'daily,,,O2,MixMultiScale,0.4247496705218376,17150,8.6444931115281332,0.97463313110315675,1.9462074566139609,1.5893959766815862,,20240517,27,,,,\n'
    'daily,,,O2,MixMultiScale,0.44419524188686138,19600,7.9255163556945822,0.98080166471508357,1.7182211256566666,1.657717728973805,,20240518,28,,,,\n'
    'daily,,,O2,MixMultiScale,0.42152861392351593,19600,9.0520527839679872,1.107889728440077,1.7003831889693599,1.6396991197463944,,20240519,29,,,,\n'
    'daily,,,O2,MixMultiScale,0.32552734055847948,19600,10.587708558545527,1.2600913754300225,2.0174894342688967,1.913849644021461,,20240520,30,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.57094471577867001,19600,6.3723239302238186,0.76110706608662226,1.3856716135437743,1.2085164272526969,,20240421,1,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.60493484439678502,19600,6.9114619809525637,0.71548612923336818,1.206661772071991,1.0659855566338152,,20240422,2,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.53020152687711231,19600,7.0585941418835798,0.89208890102420246,1.5288520583558689,1.2482118997241776,,20240423,3,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.58143050114015871,17150,8.5018410052982478,0.69312322391928716,1.2358919360630185,1.1311544958844033,,20240428,8,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.59244170116449624,19600,7.0699817665330329,0.73856832236840264,1.2778919731461229,1.078391749215909,,20240429,9,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.58209164761729493,19600,6.0339829458104504,0.69852855112306422,1.3615948225462811,1.2253156673243637,,20240430,10,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.57309816140446945,19600,6.8849946926364058,0.67109065943052282,1.3771953009205589,1.2459332967154977,,20240501,11,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.50777564530355757,19600,8.0393338585677903,0.81868945359018852,1.5941327372149521,1.3780431121454844,,20240502,12,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.50487715196730432,19600,7.2498582264693878,0.84322923716105613,1.6648964483294304,1.3872581076272144,,20240503,13,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.47095369108712409,19600,8.3531686508935739,0.89188200718469735,1.7178471251701148,1.4892459385143286,,20240504,14,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.5205371596900682,19600,7.4294903722422969,0.76014086572754791,1.6137708863313971,1.36555351127934,,20240505,15,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.51972136402121338,19600,9.0073882755619152,0.82187225860245694,1.4700196949315001,1.2921505176229069,,20240506,16,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.51185417337267469,19600,8.0321454257824527,0.8861044867151846,1.4945610504293132,1.3540626753801648,,20240507,17,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.43906723382023388,19600,9.3058308111730561,1.0133938984800104,1.7151812882177802,1.556337354709157,,20240508,18,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.44257062639294742,19600,10.173808380857006,0.97255413102225718,1.6598351645303324,1.5400654434705732,,20240509,19,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.47299739711700772,19600,10.272719801314038,0.80497787989301928,1.6233204963394898,1.4757101237192871,,20240510,20,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.4045567545043926,19600,9.5025134515430327,1.0242061899652772,1.8966378545289984,1.6759202205474506,,20240511,21,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.34858877475752509,19600,13.334781786329788,1.0926465507216556,1.8157074032465423,1.8217232389730853,,20240512,22,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.436369612064822,19600,8.6492109306246352,0.96148332989945839,1.845805499637444,1.5835476110318456,,20240513,23,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.45161181569147069,19600,8.4060241551709858,1.0179839859348041,1.7080979591391643,1.5265385815666268,,20240514,24,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.40593234625675562,19600,10.67880492513585,1.0061234399338528,1.8327752005353488,1.6356724173386474,,20240515,25,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.4570153242431293,19600,9.1834063584656143,0.96639934813143802,1.6984066649043263,1.4771240962883208,,20240516,26,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.42508023598135619,17150,8.6335977021162158,0.97337909989445681,1.9459979888335639,1.5889634630615559,,20240517,27,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.4445017825494163,19600,7.9158112985923257,0.97952784790519143,1.7180224244130775,1.6573994499050564,,20240518,28,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.42176371586323708,19600,9.0443204040427432,1.1070252116294592,1.7001944085188545,1.639393520327715,,20240519,29,,,,\n'
    'daily,,,O2,MixAdaptiveFlow,0.3259076112574717,19600,10.578350396819848,1.2582296467867979,2.0173207617424569,1.9134580102829215,,20240520,30,,,,\n'
    'daily,,,O2,MixWindFlow,0.57096177731054787,19600,6.375084322480288,0.76074133072635419,1.3856716135437743,1.2085164272526969,,20240421,1,,,,\n'
    'daily,,,O2,MixWindFlow,0.60479992187821618,19600,6.9161352711920712,0.71622012623617237,1.206661772071991,1.0659855566338152,,20240422,2,,,,\n'
    'daily,,,O2,MixWindFlow,0.53028446271190566,19600,7.0587403621232756,0.89139187718648993,1.5288520583558689,1.2482118997241776,,20240423,3,,,,\n'
    'daily,,,O2,MixWindFlow,0.58137319987903968,17150,8.5047435271871645,0.69336021034571194,1.2358919360630185,1.1311544958844033,,20240428,8,,,,\n'
    'daily,,,O2,MixWindFlow,0.59229716973454527,19600,7.0715774705444856,0.7396323209202238,1.2778919731461229,1.078391749215909,,20240429,9,,,,\n'
    'daily,,,O2,MixWindFlow,0.58194861331935677,19600,6.0375574853230658,0.69941903504226643,1.3615948225462811,1.2253156673243637,,20240430,10,,,,\n'
    'daily,,,O2,MixWindFlow,0.5728466597197619,19600,6.8914608550493437,0.67264169046966815,1.3771953009205589,1.2459332967154977,,20240501,11,,,,\n'
    'daily,,,O2,MixWindFlow,0.50762264411150204,19600,8.0506576409593436,0.81903120986768485,1.5941327372149521,1.3780431121454844,,20240502,12,,,,\n'
    'daily,,,O2,MixWindFlow,0.5047980341820133,19600,7.2532946866197641,0.84360296409118285,1.6648964483294304,1.3872581076272144,,20240503,13,,,,\n'
    'daily,,,O2,MixWindFlow,0.47078503418291517,19600,8.3569774667238335,0.8929650759602541,1.7178471251701148,1.4892459385143286,,20240504,14,,,,\n'
    'daily,,,O2,MixWindFlow,0.52047897986282599,19600,7.4364164570213367,0.76005744963632027,1.6137708863313971,1.36555351127934,,20240505,15,,,,\n'
    'daily,,,O2,MixWindFlow,0.51944108382627208,19600,9.0160581604983001,0.82348156228530522,1.4700196949315001,1.2921505176229069,,20240506,16,,,,\n'
    'daily,,,O2,MixWindFlow,0.51166399844371169,19600,8.0424759888332602,0.88683421094452464,1.4945610504293132,1.3540626753801648,,20240507,17,,,,\n'
    'daily,,,O2,MixWindFlow,0.43890470322896008,19600,9.3134217282257019,1.014118362343881,1.7151812882177802,1.556337354709157,,20240508,18,,,,\n'
    'daily,,,O2,MixWindFlow,0.4423571185167679,19600,10.179781386935661,0.97383146315734481,1.6598351645303324,1.5400654434705732,,20240509,19,,,,\n'
    'daily,,,O2,MixWindFlow,0.47276476264725309,19600,10.273644554441454,0.80682431914353048,1.6233204963394898,1.4757101237192871,,20240510,20,,,,\n'
    'daily,,,O2,MixWindFlow,0.40420618030752947,19600,9.5121843713536833,1.0263146587326428,1.8966378545289984,1.6759202205474506,,20240511,21,,,,\n'
    'daily,,,O2,MixWindFlow,0.34836676939003891,19600,13.344196898217829,1.0937137699062134,1.8157074032465423,1.8217232389730853,,20240512,22,,,,\n'
    'daily,,,O2,MixWindFlow,0.4363356398353177,19600,8.6538179436147846,0.96138879406411237,1.845805499637444,1.5835476110318456,,20240513,23,,,,\n'
    'daily,,,O2,MixWindFlow,0.45133713284348631,19600,8.415973769109689,1.0194428358987211,1.7080979591391643,1.5265385815666268,,20240514,24,,,,\n'
    'daily,,,O2,MixWindFlow,0.40584474862118941,19600,10.682286421738816,1.0065635496526979,1.8327752005353488,1.6356724173386474,,20240515,25,,,,\n'
    'daily,,,O2,MixWindFlow,0.45677457522316289,19600,9.1933104714225955,0.96758158291638263,1.6984066649043263,1.4771240962883208,,20240516,26,,,,\n'
    'daily,,,O2,MixWindFlow,0.42487116359211968,17150,8.6449309125044849,0.97418328134559062,1.9459979888335639,1.5889634630615559,,20240517,27,,,,\n'
    'daily,,,O2,MixWindFlow,0.44430795114806138,19600,7.9255328299125587,0.98033737422237555,1.7180224244130775,1.6573994499050564,,20240518,28,,,,\n'
    'daily,,,O2,MixWindFlow,0.42164350288907582,19600,9.0516311440077484,1.1074229144358223,1.7001944085188545,1.639393520327715,,20240519,29,,,,\n'
    'daily,,,O2,MixWindFlow,0.32565799578288052,19600,10.587325676821258,1.2595607653995011,2.0173207617424569,1.9134580102829215,,20240520,30,,,,\n'
    'daily,,,O2,MixTaper,0.57093842376354798,19600,6.3749574855930859,0.76078677475817402,1.3857955154623147,1.2085793422571225,,20240421,1,,,,\n'
    'daily,,,O2,MixTaper,0.60478461631105518,19600,6.9173825918738014,0.71620381611806594,1.2066658967387045,1.0660239577483868,,20240422,2,,,,\n'
    'daily,,,O2,MixTaper,0.53028202882293074,19600,7.0570168263470636,0.89145737527067037,1.5289310480618838,1.248246309645898,,20240423,3,,,,\n'
    'daily,,,O2,MixTaper,0.5813604832173439,17150,8.4992713330880587,0.69364826368627552,1.2359743845925313,1.1313556895565329,,20240428,8,,,,\n'
    'daily,,,O2,MixTaper,0.592234390578259,19600,7.0647852273784935,0.74053434068933022,1.2780097306946656,1.0784714947939398,,20240429,9,,,,\n'
    'daily,,,O2,MixTaper,0.58202521600816992,19600,6.030882417562009,0.69901902897252477,1.3616256735617711,1.2256057781433702,,20240430,10,,,,\n'
    'daily,,,O2,MixTaper,0.57273917019216636,19600,6.8870049235691324,0.67361886537576399,1.3772746176780419,1.2461480700370191,,20240501,11,,,,\n'
    'daily,,,O2,MixTaper,0.50745345582196655,19600,8.0590441425111212,0.81937585027810234,1.5942934380687162,1.3782918673802369,,20240502,12,,,,\n'
    'daily,,,O2,MixTaper,0.50467025003264365,19600,7.2595315550680661,0.84380080348138675,1.6651173285563377,1.3874395507729886,,20240503,13,,,,\n'
    'daily,,,O2,MixTaper,0.47054954004547112,19600,8.366765554855883,0.89369156239734104,1.7180054688148436,1.4895488475615437,,20240504,14,,,,\n'
    'daily,,,O2,MixTaper,0.52033385742985738,19600,7.4473906374414147,0.76030078424792724,1.6137966454746111,1.3655959052217197,,20240505,15,,,,\n'
    'daily,,,O2,MixTaper,0.51905278473344341,19600,9.0269531659337883,0.82511792755908564,1.4702397023847693,1.2926704651212886,,20240506,16,,,,\n'
    'daily,,,O2,MixTaper,0.51140187863746278,19600,8.055617713546722,0.88770036103762318,1.494700782315538,1.3541851849199771,,20240507,17,,,,\n'
    'daily,,,O2,MixTaper,0.4386480911447157,19600,9.3360283034738192,1.0142186759389775,1.7154815585720395,1.5562837179151134,,20240508,18,,,,\n'
    'daily,,,O2,MixTaper,0.44172862340482411,19600,10.199222039921716,0.97692294273520563,1.6600689440389278,1.5404060234657899,,20240509,19,,,,\n'
    'daily,,,O2,MixTaper,0.47253701321035191,19600,10.272500824252038,0.80832039613169915,1.6234737275397051,1.4760743016211442,,20240510,20,,,,\n'
    'daily,,,O2,MixTaper,0.40361487473074631,19600,9.5205708962281754,1.0299750863975654,1.89691187365182,1.6762537214485975,,20240511,21,,,,\n'
    'daily,,,O2,MixTaper,0.34776666463026168,19600,13.361138001031261,1.0967300046806561,1.8160619606033963,1.8220144037489905,,20240512,22,,,,\n'
    'daily,,,O2,MixTaper,0.4359455222853596,19600,8.6500008292516242,0.96411286754128322,1.8460202235832723,1.584198438031108,,20240513,23,,,,\n'
    'daily,,,O2,MixTaper,0.45101466743354551,19600,8.4175046024039641,1.0214657238473877,1.7082300466241953,1.526957469179324,,20240514,24,,,,\n'
    'daily,,,O2,MixTaper,0.40564988951318348,19600,10.670976925798866,1.0086288122142031,1.8329352326416333,1.6360181180229914,,20240515,25,,,,\n'
    'daily,,,O2,MixTaper,0.45634983846552002,19600,9.1919919381955904,0.97062018954450024,1.6986664571653023,1.4775052454167279,,20240516,26,,,,\n'
    'daily,,,O2,MixTaper,0.42438239021721658,17150,8.6531004697165699,0.97679355829204639,1.9463171279420881,1.5894850026704372,,20240517,27,,,,\n'
    'daily,,,O2,MixTaper,0.44381065950432341,19600,7.9321884648754581,0.9831299872189676,1.7182964521972297,1.6579676019014269,,20240518,28,,,,\n'
    'daily,,,O2,MixTaper,0.42068786473188829,19600,9.0795562783254606,1.1123774771279509,1.7004507916190887,1.6398665310414942,,20240519,29,,,,\n'
    'daily,,,O2,MixTaper,0.32486077663032598,19600,10.609104585880941,1.2637926715563237,2.017569309846571,1.9138507110164464,,20240520,30,,,,\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240421,,902853.87181368168,12334.968856075779,40730.556225829598,32043.030307610818\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240422,,1054845.6025430311,11091.441260135733,31058.571240677978,25051.118029843681\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240423,,1123619.2376037925,16523.506581992842,55501.021735615752,33737.27283683428\n'
    'daily_sse,,,O2,AdaptiveFlow,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,AdaptiveFlow,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,AdaptiveFlow,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,AdaptiveFlow,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,AdaptiveFlow,,17150,,,,,,20240428,,1400067.5158217107,9717.6346984924749,31876.437065054582,25772.911096550466\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240429,,1049489.6787571465,11642.920259288512,35187.32149116919,25759.202955643636\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240430,,774944.59928757627,10506.987444408238,37676.226850307721,30088.582034561881\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240501,,943008.5928897222,9717.5075675575063,42365.517327854745,33056.690526732171\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240502,,1539186.1099098427,15121.68227450829,55303.311932860983,40225.547553321972\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240503,,1190909.7004688007,15264.183220320381,58946.136254997051,41120.064466011085\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240504,,1563591.8780318275,18118.991554705426,67159.298097982057,48624.628937870883\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240505,,1103422.2007575699,12387.208377867937,55561.962242522372,41169.824661734609\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240506,,1870887.872839591,15058.973153756931,49584.080690296374,36359.35568193467\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240507,,1570468.1078331631,17759.062507285737,50700.885915300001,37596.65659515428\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240508,,1806456.0754412608,20726.036527851596,71082.33781129113,46755.891364252348\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240509,,2200906.5773449973,20954.940836931579,59498.793037475101,50922.429869250154\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240510,,2239621.0036111032,14441.802350342274,53798.161484641241,46819.726400318679\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240511,,1834479.6359329035,21607.79162696749,76114.293920282798,58447.706716689994\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240512,,3971553.3809389556,25394.526578550227,81132.707756370539,68940.650745526509\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240513,,1670787.6302174204,20003.383368180133,76627.856805849806,55617.014959768145\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240514,,1396298.8514015889,24192.298536913469,68893.7668869849,53766.108351117269\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240515,,1946473.6032382182,19489.077563174069,71018.598855881646,58187.655860566207\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240516,,1304872.3002440571,19110.639893401414,59776.279852113497,48740.803818009292\n'
    'daily_sse,,,O2,AdaptiveFlow,,17150,,,,,,20240517,,1225037.9971198731,16924.049900128506,67319.983344304143,47524.658322819356\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240518,,1367842.014345384,19574.398217101581,63395.533779394391,54479.376144763439\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240519,,1638240.3102487507,23239.483431126922,61520.744314376068,55358.129330355834\n'
    'daily_sse,,,O2,AdaptiveFlow,,19600,,,,,,20240520,,2339221.4795434633,31110.022334960289,90774.092348341277,76108.941547641007\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240421,,907271.43214320973,12323.74126055371,41062.215386849646,32218.574271086385\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240422,,1070153.3838726827,11181.2307444904,31097.26369797251,25222.437694315282\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240423,,1128925.0680688662,16527.595213152468,55822.795016880147,33863.130250315196\n'
    'daily_sse,,,O2,Bias325,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,Bias325,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,Bias325,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,Bias325,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,Bias325,,17150,,,,,,20240428,,1405271.7505141189,9762.9639317598194,32126.532162306186,26066.367843632954\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240429,,1052409.4756470961,11769.438642674126,35381.44697496295,25896.701918090799\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240430,,779985.62578605954,10566.691866863519,37792.957899936053,30287.206895488274\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240501,,957746.01101214206,9848.595396543853,42661.204146590142,33355.331677083137\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240502,,1569955.5684571369,15185.528141085058,55798.598311710928,40480.326347597278\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240503,,1202333.1500838201,15329.82103832718,59683.618556502581,41345.05820345664\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240504,,1574517.4916825357,18300.914523141459,67632.572514462518,48958.104726023477\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240505,,1113314.9690282312,12425.346811338328,55600.3343679639,41385.357958525252\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240506,,1894663.4416835841,15231.239583212882,50089.845563733426,36701.339439838383\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240507,,1599269.3879400045,17855.151027436368,51177.849092353405,37839.198570260545\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240508,,1825769.9769067327,20848.051150895655,72383.051480911367,47000.769362129759\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240509,,2219548.8222011039,21152.877219323069,60025.596502423105,51222.330034065279\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240510,,2242278.2013031999,14663.662437741645,53860.464114752394,47017.237238338479\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240511,,1863891.7712731815,21914.962742709555,76931.671067351592,58855.471415597502\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240512,,4008869.9112706743,25582.491458184086,82305.873012516531,69254.459643095222\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240513,,1683303.7469515572,20021.355544880033,77415.89257097333,56078.047106512917\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240514,,1419837.5234474009,24449.986557339318,69388.964827584758,54297.566040011799\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240515,,1958809.7093023269,19623.726047094911,71659.998403266203,58593.268520016427\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240516,,1332580.7656207969,19307.702614886686,60342.856342895866,49069.850143371339\n'
    'daily_sse,,,O2,Bias325,,17150,,,,,,20240517,,1251238.3527331192,17034.841957242228,68065.605600125753,47886.207366050374\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240518,,1389290.3885085708,19704.592285416089,64095.291679906004,54805.935635141097\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240519,,1656315.8366748074,23328.401425444521,62213.267474311724,55707.679123003567\n'
    'daily_sse,,,O2,Bias325,,19600,,,,,,20240520,,2366642.8873856389,31342.555342893116,91516.581788292169,76619.62147163818\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240421,,895498.18571521807,12901.160685180686,42640.702165507959,33436.378777484315\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240422,,1117880.8774221223,11025.787753376178,30677.673713159613,24791.846455272815\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240423,,1090627.4399895375,16728.152713696472,57293.680505021337,34178.323928606354\n'
    'daily_sse,,,O2,Drift,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,Drift,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,Drift,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,Drift,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,Drift,,17150,,,,,,20240428,,1414352.3710241159,10048.356727630831,32041.482138573989,25941.71795368948\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240429,,1033119.0203289767,11584.785754996352,34800.372896354711,25900.371518961565\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240430,,693722.8184073572,10463.191085750237,38005.144103906707,31044.809595341481\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240501,,962285.57176923077,9363.9912604652345,41451.602230550088,32702.351087466363\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240502,,1689302.8942351465,15045.104832955636,55246.762724613443,41204.379110956899\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240503,,1365448.5764468047,15588.246391426772,63062.48025451279,44000.957292796433\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240504,,1756804.8493492005,18237.077919223346,70286.901872399496,50853.725041827733\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240505,,1210080.7247071723,12548.120306830853,57849.91804430577,43299.39708600827\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240506,,2167276.1905515892,14693.116669904441,49426.389445652734,36540.711157260441\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240507,,1794721.3478216438,17486.018989415839,51282.783290909516,37056.895852811045\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240508,,2091864.2344897261,21209.808723476715,70676.825176652055,46677.294497666844\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240509,,2433805.6260661432,20783.088849180378,58827.638167973106,53479.703263353316\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240510,,2435159.5632960638,14617.25529240258,55424.542805403842,48289.696705238588\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240511,,1971504.7376236287,22254.359718877822,78558.198880473559,59241.913099904297\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240512,,4409820.3555768533,26555.618489423767,81986.915168383348,70257.030823950161\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240513,,1739875.3819593058,20154.401786983013,76701.89868765061,56993.091315954392\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240514,,1554180.0100935949,23514.103845732287,67885.925204832311,55456.831791655823\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240515,,2052118.7639754724,20850.669934276491,74029.994224239141,62329.332997842976\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240516,,1377384.22532004,19989.184386894107,66248.024901328041,53261.387810667373\n'
    'daily_sse,,,O2,Drift,,17150,,,,,,20240517,,1371005.1839713592,17189.470159839839,73997.798649078584,52681.179612321299\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240518,,1440108.5957586314,19783.136301211081,62852.44117191626,55668.136591266448\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240519,,1868943.4402603107,24286.596733354963,65846.87666311393,56404.489146196633\n'
    'daily_sse,,,O2,Drift,,19600,,,,,,20240520,,2464788.1919192174,31055.729954802431,91245.064735892665,76978.176510575489\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240421,,874209.05476472131,12762.65816934593,41868.192078862849,33191.808477390885\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240422,,1050047.7670250121,10965.267306573689,30415.88914651997,24696.155376540333\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240423,,1104190.8457341129,16538.527183017693,57252.754452137691,34303.898030887576\n'
    'daily_sse,,,O2,DriftBias,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,DriftBias,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,DriftBias,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,DriftBias,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,DriftBias,,17150,,,,,,20240428,,1439083.9318027268,9870.7704717358574,31773.25758564896,25869.948537434069\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240429,,1052319.2813768301,11285.173984901048,34150.403331705449,25560.525557352437\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240430,,731188.30762220523,10478.612175306305,37575.100409366307,30437.379598680876\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240501,,965635.38393689995,9165.5979971773922,41103.656851349449,32614.758341289838\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240502,,1561339.5083417168,14907.158706529066,54952.240762383852,40979.5288894845\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240503,,1258479.2125772312,15521.871545055881,62124.423864637865,43349.719713571118\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240504,,1635633.3430150091,18167.523702995852,69546.124906780431,50356.415507082347\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240505,,1137552.1915825738,12446.395373100415,57471.142609170493,43456.923008069949\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240506,,2051421.2187384511,14299.248994510621,48929.708692220222,36073.104852818724\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240507,,1628996.8918315556,17158.983094701543,51316.436506401762,37027.017779802016\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240508,,1892916.4740432841,21028.746512103826,70773.478353597777,46862.292862973503\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240509,,2259909.8676075791,20154.627093805932,58078.015635833981,52871.751416601532\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240510,,2402442.8360501151,14330.789588948712,54386.693611045208,47531.468759237032\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240511,,1868458.3688811276,21487.598051491193,77927.839338615231,58777.188836833659\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240512,,4222417.7862530798,25825.366076172329,81480.848262524189,69566.778373836773\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240513,,1704393.6889025769,19521.991192768328,76462.903718442496,56120.467755321028\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240514,,1481463.254808259,23134.789302255027,67196.750157113842,54948.54670729678\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240515,,2051747.6568615262,20287.391727507114,73133.632224343513,61750.750956097152\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240516,,1316472.1443080904,19412.984419858083,65294.751266441475,52758.704940431642\n'
    'daily_sse,,,O2,DriftBias,,17150,,,,,,20240517,,1287917.5649746035,16811.158328793943,73198.652358447376,52231.334270696178\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240518,,1377584.1784251782,19196.051761303097,62318.871217989683,54959.171502074423\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240519,,1704619.0132241189,23412.167516127229,65145.478476804703,55844.363645458696\n'
    'daily_sse,,,O2,DriftBias,,19600,,,,,,20240520,,2327838.2360930452,30231.402556409128,90615.290024028451,76634.15011818301\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240421,,795887.64052538155,11353.965734520927,37633.682083387743,28626.034316816618\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240422,,936258.81159739359,10033.639862056822,28538.239590726389,22271.974056257332\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240423,,976545.52469273051,15598.123103679158,45812.816880243954,30537.44575361559\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,17150,,,,,,20240428,,1239624.3032212078,8239.1996306469664,26195.405251278979,21943.604964545404\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240429,,979698.98671054724,10691.470069399104,32006.954742613299,22793.403789628177\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240430,,713615.42373049469,9563.6658799881116,36337.233031382842,29427.41029797478\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240501,,929101.77758557699,8827.108394227922,37174.671178802309,30426.055685341234\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240502,,1266765.4222344838,13136.947459828109,49808.680003664522,37220.455251059597\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240503,,1030184.7083565549,13936.296709503047,54328.851599735062,37719.907120677824\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240504,,1367598.3596013151,15590.888888900168,57839.575410922473,43469.927921475901\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240505,,1081867.6129483234,11325.157060680911,51043.426881988133,36548.833286479756\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240506,,1590207.6535159138,13239.290585421957,42354.774908335414,32725.198019785654\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240507,,1264501.0587619925,15389.550762984902,43780.769575823295,35936.320285610716\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240508,,1697330.3468890844,20128.556992139667,57660.198288467116,47474.844848596695\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240509,,2028724.9886197662,18538.886140261777,53999.034358864134,46487.310775375379\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240510,,2068363.9334796667,12700.591987494379,51649.320903183398,42683.31923725285\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240511,,1769836.1331764231,20560.36706343852,70505.808964154319,55050.688278540387\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240512,,3485201.5436650398,23399.979102157056,64617.150134404335,65046.040964525942\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240513,,1466253.4545597588,18119.223796021193,66777.159672839945,49149.41151353145\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240514,,1384960.3451074343,20311.311354146339,57184.933305101433,45674.272803824395\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240515,,2235122.7427304718,19840.773777132854,65837.672739668793,52438.315434112104\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240516,,1652965.0659562505,18304.982921349816,56537.869908072156,42765.15367837753\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,17150,,,,,,20240517,,1278344.0091860667,16249.056856709532,64945.475159134345,43300.403811099422\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240518,,1228137.3428924696,18805.706174506806,57851.380595409319,53840.66955629344\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240519,,1603274.7387912848,24019.894455991685,56656.956124472024,52677.17784405296\n'
    'daily_sse,,,O2,MixAdaptiveFlow,,19600,,,,,,20240520,,2193269.3435108126,31029.580143443309,79763.827892840462,71761.902519471201\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240421,,792430.10244393535,11471.762366221286,37636.125468079881,28638.116716872155\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240422,,934236.24741215305,9978.116143791005,28504.286300552591,22182.023477277664\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240423,,972722.58948312909,15618.694361591712,45813.730063398281,30531.469748935757\n'
    'daily_sse,,,O2,MixDriftBias,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,MixDriftBias,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,MixDriftBias,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,MixDriftBias,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,MixDriftBias,,17150,,,,,,20240428,,1240939.7123468558,8278.2661579418927,26159.287663935993,21868.577482824669\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240429,,977572.91247722576,10524.281825550832,31970.372513107966,22711.725358214495\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240430,,706329.1800764713,9551.4926413530484,36299.752342637345,29359.638226165553\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240501,,929824.5140086466,8638.7429380398244,37118.599842358737,30331.395994537565\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240502,,1266612.0530723236,13077.814251492731,49776.994058590361,37210.699165370286\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240503,,1031394.0977665717,14015.454150156118,54311.204762492343,37763.921561466814\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240504,,1369200.0357823335,15550.93890723493,57819.028265773602,43461.985793308188\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240505,,1080216.5657380677,11282.078498853371,51027.540004598261,36622.421339075234\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240506,,1606981.8932740318,12950.240904404782,42309.594403776908,32612.826135609994\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240507,,1265962.6684723867,15200.219737586565,43759.506402345753,35815.241180255944\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240508,,1701587.8699629903,20145.308148912154,57600.60451243211,47357.159470078514\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240509,,2033553.9391107685,18203.55251994729,53925.700359532129,46513.98361689083\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240510,,2082563.5381510514,12566.104463461787,51600.136937202216,42625.488501324799\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240511,,1768695.6919097379,20370.168868036941,70522.406909098514,54986.669500684024\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240512,,3511135.9890528275,23428.365458314307,64598.129408535977,65002.542125506057\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240513,,1463232.8094900127,17986.500352497213,66738.117213191857,49104.638656144394\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240514,,1380930.2691012225,19897.244001301937,57089.127927903639,45626.803352472671\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240515,,2236781.7334768111,20027.063524457626,65739.574717276351,52479.973027513071\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240516,,1640984.0779765807,18277.890940182842,56557.879169179738,42862.070305301073\n'
    'daily_sse,,,O2,MixDriftBias,,17150,,,,,,20240517,,1274755.4646737687,16148.333758350462,64984.916150789082,43478.870776514334\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240518,,1222341.0904146619,18592.372181054205,57767.427496591503,53743.632331193163\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240519,,1607776.2985789618,23939.817107065581,56666.155067839638,52579.976726046385\n'
    'daily_sse,,,O2,MixDriftBias,,19600,,,,,,20240520,,2190056.5375175197,30570.404289893806,79699.542309642333,71684.88669273362\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240421,,796711.47119837417,11348.476385619491,37643.48276531251,28629.158782513827\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240422,,937547.87374038203,10050.059229207225,28538.758690781393,22273.875827576656\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240423,,976626.45279524033,15570.955559529364,45816.445946294516,30539.71655275366\n'
    'daily_sse,,,O2,MixMultiScale,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,MixMultiScale,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,MixMultiScale,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,MixMultiScale,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,MixMultiScale,,17150,,,,,,20240428,,1240280.5615985731,8241.9373677372932,26199.164999237219,21951.098556610519\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240429,,979989.46832718235,10722.664531855844,32009.662326077047,22794.411957963508\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240430,,714422.82286192989,9586.0931640099734,36338.390234138744,29434.792989519214\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240501,,930994.65251442627,8868.792343606241,37179.568901985433,30436.054153224955\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240502,,1270613.4546304399,13149.962247764692,49818.613505700101,37234.440052648439\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240503,,1031070.6625536557,13948.49908078555,54339.057673647585,37724.376797494369\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240504,,1368944.0171781846,15630.891000468284,57845.117104508929,43484.352672325542\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240505,,1083734.1534837058,11333.629408870824,51041.323517614095,36556.589597004502\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240506,,1593327.0527736857,13301.043395900168,42364.432361204657,32743.169034333288\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240507,,1268014.856147395,15420.259688261896,43787.976643541282,35943.720018589382\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240508,,1700507.9763000398,20163.257733065635,57681.698843962353,47477.610553855651\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240509,,2030997.9337020253,18598.275390395895,54007.761521190907,46503.470748410386\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240510,,2068375.5706093789,12776.025376662612,51650.78811878361,42693.789391258135\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240511,,1773514.5399571795,20669.987213016488,70523.675961626577,55071.01799463253\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240512,,3490620.49911336,23471.197067367844,64638.022544624153,65059.720205136458\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240513,,1467738.1995511663,18127.644588280469,66791.372804634899,49176.216751535387\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240514,,1388037.3673187722,20387.594717343338,57189.222013748804,45695.02939375148\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240515,,2236083.2336125998,19871.598989480175,65843.604779347239,52450.696828611544\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240516,,1656444.0324770811,18370.821293978021,56547.137125017434,42782.050314564927\n'
    'daily_sse,,,O2,MixMultiScale,,17150,,,,,,20240517,,1281572.5288126636,16290.952045183629,64959.457410683259,43323.979637361161\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240518,,1231150.6662858936,18854.64934795443,57864.763198396104,53861.350151499755\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240519,,1606017.3282333554,24057.42514750734,56669.538590860364,52696.818784623174\n'
    'daily_sse,,,O2,MixMultiScale,,19600,,,,,,20240520,,2197151.6214056853,31121.473378889263,79777.166900778,71791.281014453038\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240421,,796577.37283258117,11343.056527036242,37640.18221771824,28628.782945545656\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240422,,937525.19481814455,10054.23013596423,28538.272146267056,22273.441750475686\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240423,,976585.817957066,15573.752187153324,45817.887894293497,30540.723304495688\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,17150,,,,,,20240428,,1240471.1745281813,8244.8429802861065,26198.554553435191,21952.108982924499\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240429,,980141.36955080857,10722.294779290445,32009.262883318068,22792.916100731294\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240430,,714461.22089699074,9588.0664671398699,36337.335733787048,29433.371155399163\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240501,,930848.28957846505,8867.9555353121832,37178.709304455049,30436.175409092855\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240502,,1270336.4710956311,13147.913192252629,49817.265511963698,37230.589431947643\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240503,,1031161.4966659676,13948.656002404168,54337.699421815509,37720.523756382965\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240504,,1368845.5918366623,15628.788587180898,57844.891123778129,43480.706904418592\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240505,,1083885.6478243149,11322.67075556796,51040.712623800762,36557.220734822236\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240506,,1593270.7396653481,13291.192650909536,42362.570932303832,32740.367899701276\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240507,,1267755.6691146439,15414.898317944258,43788.897847667766,35945.025159301171\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240508,,1700100.3957725225,20157.342319854535,57681.649124211326,47479.188936653845\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240509,,2031108.0768044195,18587.609679582529,54006.389969734781,46495.9511359881\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240510,,2068736.4941544358,12758.914108485915,51648.764918557157,42687.165719322751\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240511,,1773440.3780351435,20645.103955446742,70521.583112723296,55063.064766503339\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240512,,3490125.6603679289,23445.69949202612,64635.859274372204,65051.888777718857\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240513,,1467816.1194865631,18115.662277520634,66790.284547590461,49166.773812763902\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240514,,1388241.1562632765,20369.554256715812,57187.217154342485,45690.523706500746\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240515,,2236579.7358827819,19858.127261960879,65841.723827357331,52446.8312996776\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240516,,1656531.8269164148,18349.775234318338,56544.850147255689,42777.860351867057\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,17150,,,,,,20240517,,1281701.8578854974,16275.91591858957,64958.145554607727,43319.74950453241\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240518,,1231155.8552957801,18836.802614232525,57863.280470812708,53855.687079901596\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240519,,1605867.5465475612,24037.15469263494,56668.092052318229,52690.5325647137\n'
    'daily_sse,,,O2,MixPhysicsNewBias,,19600,,,,,,20240520,,2196992.483977464,31095.2595216427,79774.314889693909,71782.714049274669\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240421,,796496.06019699504,11371.532487258315,37672.741068402043,28660.889008503265\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240422,,939301.9099255465,9984.0136524206027,28537.709109066087,22275.665881058605\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240423,,973519.21676152106,15591.366312603466,45815.276387312508,30496.589374866686\n'
    'daily_sse,,,O2,MixRecentBias,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,MixRecentBias,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,MixRecentBias,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,MixRecentBias,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,MixRecentBias,,17150,,,,,,20240428,,1239939.6555015151,8176.7390690660104,26206.251506328372,21958.326277228622\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240429,,980268.15176871303,10514.33861471992,32003.054763152759,22786.395625995567\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240430,,715662.37873928878,9640.8638920020312,36342.490769978729,29430.620677765393\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240501,,929544.12323940732,8763.1739297127351,37149.52148420021,30398.228917692064\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240502,,1266515.3568228914,12955.768619411625,49793.526699407528,37183.558273430128\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240503,,1033467.5701837561,13992.568104922771,54337.467717892039,37695.446392739388\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240504,,1368981.0037995381,15413.545594390482,57859.040813258114,43545.749537700736\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240505,,1085377.5357971468,11220.724727069028,51059.744165575619,36606.106073125404\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240506,,1580088.2496432036,13241.490886041895,42358.620123313071,32707.018881592234\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240507,,1258606.1764560766,15228.8739208607,43801.369520144312,35953.076809448707\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240508,,1682458.9039233751,19930.617069504224,57694.325872870795,47553.637526712519\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240509,,2018206.1839484628,18440.973966940306,53995.768629601072,46453.815438049933\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240510,,2060135.9694097175,12572.831594062038,51631.147537434066,42615.465265152845\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240511,,1760056.0974063862,20505.520857310854,70518.516705340371,55010.039143000482\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240512,,3465856.4348207274,23075.870273310691,64660.935974377622,64961.335514020444\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240513,,1463607.1108133832,17926.947150634602,66793.663744943638,49076.716174754096\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240514,,1390491.5765958466,20038.858175462112,57121.28485612051,45563.405476089982\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240515,,2261267.7598393792,19840.730476096272,65790.503558246157,52484.041360758158\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240516,,1676477.8076699651,18052.86569782719,56509.667799509858,42652.611409843434\n'
    'daily_sse,,,O2,MixRecentBias,,17150,,,,,,20240517,,1288043.355615733,16151.782247393392,64971.104946641819,43313.213213602547\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240518,,1220059.5252386611,18486.668838296086,57861.818219170229,53849.845674845157\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240519,,1590259.0971209065,23985.981552165002,56702.589967294414,52669.474409550363\n'
    'daily_sse,,,O2,MixRecentBias,,19600,,,,,,20240520,,2181031.7938033296,30691.287086141296,79728.786316626181,71685.858863715708\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240421,,796545.62568513863,11344.411726280116,37640.412529239067,28629.014920000907\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240422,,937863.56567823072,10053.778961952776,28538.43469250355,22273.578738473268\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240423,,976109.13515197346,15576.046537719667,45817.550934661165,30539.129451074223\n'
    'daily_sse,,,O2,MixTaper,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,MixTaper,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,MixTaper,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,MixTaper,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,MixTaper,,17150,,,,,,20240428,,1238875.0662677099,8251.6867202119902,26198.900451176403,21951.411691406738\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240429,,978259.33005611366,10748.465750907548,32012.853886304929,22796.774995628264\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240430,,712882.2375953882,9577.1010161675513,36338.879708089917,29441.346659000952\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240501,,929644.40161840315,8893.7425654865801,37178.95330100385,30436.546244156954\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240502,,1272984.5728224791,13158.984966771677,49818.722706711756,37233.894045055407\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240503,,1032935.6486209678,13955.196000734344,54343.268070028455,37729.774738163389\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240504,,1372054.2106584881,15654.21833052393,57850.238701203132,43487.613077729686\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240505,,1087087.0952103408,11329.922737549059,51045.056413724124,36551.102656623232\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240506,,1597123.3158152562,13344.064049836248,42367.453736381722,32751.539855379018\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240507,,1271902.3442374312,15445.033847332001,43788.956401631753,35942.82329511235\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240508,,1708363.9198719799,20161.334643418901,57680.388764894786,47471.572608694696\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240509,,2038872.9523046482,18705.817346431315,54014.246419665404,46507.874055741893\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240510,,2068275.7544114725,12806.284510929137,51659.072102628626,42704.388740564427\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240511,,1776568.8957260968,20792.63410055358,70526.183265464657,55072.600157937362\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240512,,3498992.1701789922,23575.207382069901,64642.388477112792,65066.835154388267\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240513,,1466521.2811826544,18218.466978630051,66792.697051217328,49189.819944779963\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240514,,1388745.9211372416,20450.487609903328,57193.777886918127,45699.342608577506\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240515,,2231847.0715982639,19939.708784241229,65849.170714357169,52460.48353699001\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240516,,1656057.2295203137,18465.229626087472,56555.167560891008,42787.226304585347\n'
    'daily_sse,,,O2,MixTaper,,17150,,,,,,20240517,,1284125.9337240064,16363.254992182367,64966.778717230634,43328.833139199218\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240518,,1233224.4313091442,18944.273606675677,57869.836873618289,53877.588751513467\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240519,,1615791.5073410745,24252.719571782276,56674.044736472519,52707.579896749245\n'
    'daily_sse,,,O2,MixTaper,,19600,,,,,,20240520,,2206040.7622375363,31304.569566917606,79783.484032681488,71791.36106353988\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240421,,796577.32232718542,11343.056496595964,37633.682083387743,28626.034316816618\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240422,,937525.37095276965,10054.236876824871,28538.239590726389,22271.974056257332\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240423,,976585.98379741726,15573.757782795466,45812.816880243954,30537.44575361559\n'
    'daily_sse,,,O2,MixWindFlow,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,MixWindFlow,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,MixWindFlow,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,MixWindFlow,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,MixWindFlow,,17150,,,,,,20240428,,1240470.8612444284,8244.834739134647,26195.405251278979,21943.604964545404\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240429,,980141.27526948182,10722.297014936805,32006.954742613299,22793.403789628177\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240430,,714461.16761617945,9588.0649369573221,36337.233031382842,29427.41029797478\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240501,,930847.76124687772,8867.9581376546994,37174.671178802309,30426.055685341234\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240502,,1270336.5336579664,13147.917605651543,49808.680003664522,37220.455251059597\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240503,,1031161.5626945514,13948.65283605922,54328.851599735062,37719.907120677824\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240504,,1368845.818634866,15628.777886940166,57839.575410922473,43469.927921475901\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240505,,1083885.6785562523,11322.671604254283,51043.426881988133,36548.833286479756\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240506,,1593270.3731683649,13291.188915107399,42354.774908335414,32725.198019785654\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240507,,1267755.8326068064,15414.908386951312,43780.769575823295,35936.320285610716\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240508,,1700100.5560406181,20157.346635723487,57660.198288467116,47474.844848596695\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240509,,2031107.8020817139,18587.615285249427,53999.034358864134,46487.310775375379\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240510,,2068736.3396476889,12758.923446443863,51649.320903183398,42683.31923725285\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240511,,1773440.3696866552,20645.106863098219,70505.808964154319,55050.688278540387\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240512,,3490124.7808247642,23445.712285456248,64617.150134404335,65046.040964525942\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240513,,1467815.8740240925,18115.660901700146,66777.159672839945,49149.41151353145\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240514,,1388240.8438539098,20369.56843503844,57184.933305101433,45674.272803824395\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240515,,2236580.3666428835,19858.135517993011,65837.672739668793,52438.315434112104\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240516,,1656532.3655097799,18349.796744139865,56537.869908072156,42765.15367837753\n'
    'daily_sse,,,O2,MixWindFlow,,17150,,,,,,20240517,,1281702.3427658821,16275.917075953446,64945.475159134345,43300.403811099422\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240518,,1231155.7845052266,18836.80279902555,57851.380595409319,53840.66955629344\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240519,,1605867.7167965521,24037.156023783609,56656.956124472024,52677.17784405296\n'
    'daily_sse,,,O2,MixWindFlow,,19600,,,,,,20240520,,2196992.7137467428,31095.269105982035,79763.827892840462,71761.902519471201\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240421,,909455.21240422688,12352.872029682621,41247.83598692697,32244.795939532862\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240422,,1071480.4464570559,11177.387312378734,31137.800846473845,25241.120576700629\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240423,,1130447.2138765419,16527.400334339589,55780.123959091849,33876.201673513962\n'
    'daily_sse,,,O2,MultiScale,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,MultiScale,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,MultiScale,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,MultiScale,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,MultiScale,,17150,,,,,,20240428,,1405649.3167101662,9761.384764496237,32185.831168115226,26073.025058892708\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240429,,1052985.344208912,11781.589042630047,35440.687954331814,25938.206997576235\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240430,,780394.157798375,10567.963711504824,37869.791155181359,30318.342338567876\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240501,,960312.2095953736,9858.7346266647801,42755.006754399226,33377.992282702471\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240502,,1573498.8367820447,15198.961186901666,55896.843162329118,40575.06665749041\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240503,,1204034.9446914783,15340.456050846726,59847.303519065972,41451.921038336484\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240504,,1577376.1279821843,18315.504781725816,67712.606480290196,49050.392253896578\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240505,,1113008.9952485878,12470.95947736688,55638.103323223557,41385.976157849342\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240506,,1896738.2849138624,15267.659786250442,50199.623967065316,36772.269873563127\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240507,,1604131.695547513,17887.527564589866,51203.194602977135,37879.001392548918\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240508,,1830609.9859480334,20883.192549826577,72427.954908823114,47024.730262581856\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240509,,2219812.1418993366,21198.97650920134,60109.420285727057,51365.49298749931\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240510,,2241324.0722853788,14727.868711181916,53976.082364784663,47162.156147300891\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240511,,1865957.3918043326,22007.435433838516,77094.270317514834,59027.250560539571\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240512,,4015313.7014340954,25670.840188043192,82430.813022479313,69409.470496688868\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240513,,1685521.4900572128,20070.277052254416,77522.074835301508,56269.639853736982\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240514,,1420403.1910045259,24519.836186048575,69555.72918077535,54400.084169683869\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240515,,1957772.7996530363,19677.915630219504,71834.465705006645,58688.986317253366\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240516,,1334155.3365985234,19387.538142517209,60504.591769751009,49184.172667538965\n'
    'daily_sse,,,O2,MultiScale,,17150,,,,,,20240517,,1251234.7899548847,17094.333747035824,68186.758443338316,47993.370211599744\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240518,,1390566.3815267656,19773.192266203463,64210.133869492776,54942.458192574319\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240519,,1658357.7276955449,23402.997368729673,62340.406182586485,55844.712509044846\n'
    'daily_sse,,,O2,MultiScale,,19600,,,,,,20240520,,2368609.7897922238,31428.329304096289,91679.783149430092,76791.783507484797\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240421,,926434.687459487,12463.785120786168,41836.373460172836,32546.586930010431\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240422,,1134114.9878391961,11320.22779074125,31326.181947307959,25412.078304781371\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240423,,1120196.6233126821,16846.07036708761,55946.700356256777,33802.958380184886\n'
    'daily_sse,,,O2,PhysicsLegacy,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,PhysicsLegacy,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,PhysicsLegacy,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,PhysicsLegacy,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,PhysicsLegacy,,17150,,,,,,20240428,,1376626.6675305564,10077.230957476422,32422.520078012592,26197.007990663693\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240429,,1030197.3365902826,12214.900014535524,36232.014809759821,26289.933559834826\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240430,,745209.03705222229,10633.642429660074,38251.014368281998,30894.184742604437\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240501,,947910.39351008995,10190.845867592841,43031.647702891751,33632.117228171774\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240502,,1682988.6056106996,15489.339413216338,56426.506408879985,40846.700713242572\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240503,,1293818.8563545616,15546.976914721541,60868.133600245186,42066.824306019815\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240504,,1680857.8680014389,18547.361419093795,68678.963849939508,49504.233250496058\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240505,,1174346.4384707506,12617.573084766045,56048.990622180463,41262.591496375964\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240506,,1972795.6232468614,15831.575380994007,50749.251065907192,37307.159717920142\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240507,,1740791.0278863779,18327.722851547413,51491.358735591908,38060.752491493884\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240508,,1995904.7843157649,21075.399592231959,72600.609577457668,47008.967889021871\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240509,,2365606.0078493096,22024.302342879586,60977.479693561865,51850.17130104197\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240510,,2238598.4676587908,15068.965205530636,54983.304453013028,47780.623759908827\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240511,,1947117.3601608956,22863.902151601389,77642.035530741516,59481.672414668719\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240512,,4150350.1229029396,26548.66552459728,83105.826300915898,70007.464084831663\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240513,,1701679.2496884479,20891.301636446267,78013.32689610112,57013.101557186077\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240514,,1474511.352541795,25076.99565936625,70278.033490447444,54908.070144203739\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240515,,1951022.676173069,20313.979050299153,72627.163617354949,59240.153302111023\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240516,,1388687.9386821196,20042.30098653771,61499.487346450966,49574.621067917476\n'
    'daily_sse,,,O2,PhysicsLegacy,,17150,,,,,,20240517,,1321062.5593631694,17559.701153995469,69071.84175201469,48379.282451738036\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240518,,1437697.7839906458,20495.078006348573,64979.311464781538,55678.253843195555\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240519,,1798404.1050270863,24450.507301676087,63081.992768154712,56433.595324942304\n'
    'daily_sse,,,O2,PhysicsLegacy,,19600,,,,,,20240520,,2483728.5636882233,32380.787021496333,92476.058657863687,77113.776974882509\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240421,,926434.40314495424,12463.771122402512,41836.305727348255,32546.565590395014\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240422,,1134113.3599101936,11320.193799935281,31326.222349655734,25412.126189967155\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240423,,1120195.10279264,16846.056227555498,55946.723326296276,33802.950673141771\n'
    'daily_sse,,,O2,PhysicsNew,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,PhysicsNew,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,PhysicsNew,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,PhysicsNew,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,PhysicsNew,,17150,,,,,,20240428,,1376628.6583498083,10077.234299205244,32422.484752503355,26196.994932768725\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240429,,1030198.662937023,12214.877689563669,36232.00902944924,26289.947990530804\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240430,,745208.79048810923,10633.650716945529,38251.000050225914,30894.175328509031\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240501,,947913.90480506187,10190.831148038618,43031.59596678253,33632.130616459806\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240502,,1682987.7386383114,15489.329238539562,56426.507990859427,40846.734337090864\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240503,,1293817.405451281,15546.979936446995,60868.140107357671,42066.876178584047\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240504,,1680857.6880535642,18547.423836085945,68678.928232994993,49504.244229104792\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240505,,1174345.9643442188,12617.554737939499,56049.036543547722,41262.571193716853\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240506,,1972800.1298898016,15831.545307673514,50749.251907754006,37307.137705194182\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240507,,1740788.4334793105,18327.668694736436,51491.308511499054,38060.718191522297\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240508,,1995901.8909672117,21075.364832490683,72600.551998522089,47008.945971936635\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240509,,2365607.2372519267,22024.284958502278,60977.490961764335,51850.195745025936\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240510,,2238601.4261615574,15068.938518146053,54983.335659999539,47780.634620398858\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240511,,1947117.6147111428,22863.895375221968,77642.022197929604,59481.712468846032\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240512,,4150359.5506440829,26548.580909336917,83105.695342570878,70007.351417106518\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240513,,1701680.8493179909,20891.285263190046,78013.431278066229,57013.126521304628\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240514,,1474511.2454564953,25076.92613575235,70277.909468438462,54908.063297723551\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240515,,1951019.5115221262,20313.963776122779,72627.16012692271,59240.066659133736\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240516,,1388685.9272551476,20042.220321906731,61499.446586467406,49574.539540417863\n'
    'daily_sse,,,O2,PhysicsNew,,17150,,,,,,20240517,,1321058.3969017821,17559.683705860749,69071.769099476645,48379.205688733986\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240518,,1437698.5366907024,20495.052564869635,64979.297917627897,55678.259564017448\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240519,,1798403.6643703396,24450.493078307249,63082.019168337116,56433.630325992235\n'
    'daily_sse,,,O2,PhysicsNew,,19600,,,,,,20240520,,2483726.750634037,32380.741960809566,92476.061524892444,77113.798531291352\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240421,,907270.9727627947,12323.730754223652,41062.149265323758,32218.542654679357\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240422,,1070151.6952230285,11181.199570063502,31097.303821725593,25222.479381218349\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240423,,1128923.6169834621,16527.588019180112,55822.818723719829,33863.116273130312\n'
    'daily_sse,,,O2,PhysicsNewBias,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,PhysicsNewBias,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,PhysicsNewBias,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,PhysicsNewBias,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,PhysicsNewBias,,17150,,,,,,20240428,,1405273.6505549396,9762.9676260333508,32126.498319901497,26066.356520854453\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240429,,1052410.5350743181,11769.422407722101,35381.439480348636,25896.718096278648\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240430,,779985.3938905776,10566.70377938915,37792.939588510308,30287.201230359165\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240501,,957749.5687327343,9848.5832904148847,42661.153616185824,33355.338901534335\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240502,,1569954.9134006812,15185.519529312849,55798.606719357296,40480.358352836862\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240503,,1202331.7445461131,15329.819696269929,59683.63026931815,41345.101256319649\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240504,,1574517.2632743893,18300.969662226737,67632.539386994438,48958.110416950447\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240505,,1113314.2151254208,12425.327130042017,55600.379285786606,41385.333105621874\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240506,,1894667.2469689108,15231.218670884147,50089.849172384638,36701.319177883808\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240507,,1599267.1582919788,17855.107455627061,51177.804164730958,37839.163765906786\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240508,,1825767.5281679931,20848.021182753146,72383.003880559947,47000.740614861832\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240509,,2219549.89321644,21152.862383601256,60025.608065465975,51222.349785611223\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240510,,2242279.9631509339,14663.639593174681,53860.495157889709,47017.249340179325\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240511,,1863892.0157983638,21914.958593463525,76931.663533867468,58855.502397889526\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240512,,4008878.7810448804,25582.418502891436,82305.750321582076,69254.356345470485\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240513,,1683305.7136420181,20021.344359008595,77416.002170764637,56078.073606325153\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240514,,1419837.4255566471,24449.916849961504,69388.841880789347,54297.55947618479\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240515,,1958806.4627107461,19623.713134242222,71659.992705178302,58593.182034815109\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240516,,1332579.1249488839,19307.631482080556,60342.823033688852,49069.774770265736\n'
    'daily_sse,,,O2,PhysicsNewBias,,17150,,,,,,20240517,,1251234.6731034413,17034.828060704283,68065.543211150827,47886.132567088855\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240518,,1389291.8825489793,19704.57319598645,64095.277376431113,54805.939744965101\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240519,,1656316.2992734257,23328.394523818977,62213.294968019043,55707.706445295691\n'
    'daily_sse,,,O2,PhysicsNewBias,,19600,,,,,,20240520,,2366642.2325773765,31342.519647338428,91516.588692306526,76619.640114682887\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240421,,964322.34203942842,11851.235311092809,41656.618506205683,32555.005278598779\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240422,,1154138.1999854757,11250.809007213451,33352.87055173023,25479.793264582248\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240423,,1128928.6700154978,16159.280196604319,53375.48150651202,33637.515353634866\n'
    'daily_sse,,,O2,Placebo,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,Placebo,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,Placebo,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,Placebo,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,Placebo,,17150,,,,,,20240428,,1427117.5540602498,11768.514239237644,41192.760488132393,29392.849930536759\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240429,,1110682.9609555772,11502.302675017156,37593.260069616743,25879.651811055544\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240430,,798091.12062385655,11071.883680333383,39469.506570114012,30977.807152096084\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240501,,1053916.5926856336,10273.179830952547,47202.680352725874,34662.416183176407\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240502,,1658901.4170289959,16781.736007778905,64454.834618186491,44229.054522469487\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240503,,1285318.0283955836,17007.334030873142,62626.426633184376,43857.174024350839\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240504,,1576900.681209906,21642.450156586245,75941.906338921646,53846.673432369149\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240505,,1261648.6487732246,14507.016841886565,62604.320844991751,45294.302891624466\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240506,,1988118.9277373878,17573.965191602707,63933.337382905462,40130.063865297176\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240507,,1578191.6943308122,17728.72629094962,58100.123857130209,39155.752693483009\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240508,,1836095.9754200755,25184.427801162936,94321.38555611785,58297.157296789446\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240509,,2208628.3040487925,22545.47412596643,70238.683728410484,54388.29159821786\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240510,,2359909.7189518204,18272.198970568366,72053.813795988812,55423.081407965001\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240511,,2030659.8777915393,26010.56289679464,91497.840225983324,65348.632044796592\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240512,,4055020.7725611967,30654.590146233328,115853.31228152713,86823.576708379784\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240513,,1882116.6389145528,25299.419072284363,109850.9148343511,71668.071962380171\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240514,,1525488.3637476172,34632.768430229276,108847.28684899423,74246.236139782312\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240515,,2321244.2436985783,26954.002429603599,106052.93557255456,75689.360922142587\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240516,,1729451.5524638125,25644.822863072157,95993.585380831995,64126.2123328854\n'
    'daily_sse,,,O2,Placebo,,17150,,,,,,20240517,,1426355.7015127309,20392.136889724061,86907.223607388427,54592.793332265319\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240518,,1515593.7293708376,23924.920326437801,85663.265777425608,63122.576943441665\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240519,,1655697.2108011351,25184.810457590036,76833.344424228111,63209.952223594271\n'
    'daily_sse,,,O2,Placebo,,19600,,,,,,20240520,,2342596.0724513442,38683.902878736146,126545.78905143666,97118.947742320976\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240421,,915913.78889922635,12334.67127297353,42768.550636309366,33339.8627185855\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240422,,1118642.5174621362,11107.780507904477,31006.395795492012,25528.080113688771\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240423,,1097682.4784076056,16715.332061490975,56212.317739013684,33177.768665342373\n'
    'daily_sse,,,O2,Recent,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,Recent,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,Recent,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,Recent,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,Recent,,17150,,,,,,20240428,,1347005.3906893507,9706.5143680358306,32741.96741246706,26494.209108044342\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240429,,1008154.5317542618,11435.775101418607,35770.635935644117,26169.54002160392\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240430,,746724.10346621554,10763.935103023425,38194.18018893963,31003.76466101905\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240501,,919643.74456969812,9846.5764856664464,42027.686235314955,33323.457478388736\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240502,,1606433.1915917317,14600.632854669355,54983.814862740801,39608.55274703676\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240503,,1294569.3252883435,15765.007655726746,60854.400075367317,41515.088212349794\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240504,,1655879.9300222744,17900.348937310278,69236.948131394398,50290.170318691606\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240505,,1171478.5189067237,12348.114236491732,56523.178145099024,41373.672931507979\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240506,,1833063.5910627267,15555.80088264402,50198.40985871297,36549.624525075167\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240507,,1631232.886570913,17574.237625146285,52624.16627876983,37922.104129273182\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240508,,1810858.1146814763,20584.975412709638,73611.404782431477,47499.167498769173\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240509,,2234387.6468997407,21631.596478649415,60582.463049302183,51305.51700048639\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240510,,2166029.1346386494,14434.146549060941,54187.520033215551,46369.382905347862\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240511,,1810675.2962259639,22456.582252589054,77682.654674894089,58499.441324480293\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240512,,3881138.1177896578,25433.374452548102,84763.485762277051,68474.852474674786\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240513,,1660379.9757783855,20257.475124232471,78043.00966642343,55510.931675394968\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240514,,1430290.6310323984,23868.615312582813,66608.45483884876,52443.589389208239\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240515,,2055790.0371082223,19982.423359991051,70233.255916800146,59294.668993616251\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240516,,1463001.8729880957,19111.147124635987,60461.948223015461,47624.087148600891\n'
    'daily_sse,,,O2,Recent,,17150,,,,,,20240517,,1344630.3808049967,17265.843915225007,69708.720485712693,48217.854656427517\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240518,,1342422.4375841655,19313.469109791331,64806.123021137551,55104.466022972782\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240519,,1664970.6772266869,24302.06796570681,64443.020630303494,55711.571377044995\n'
    'daily_sse,,,O2,Recent,,19600,,,,,,20240520,,2384696.6062336247,31171.874831362627,90686.182759421601,74856.665173841844\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240421,,898883.74129531463,12229.690792336129,42137.367603392886,32809.303779906528\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240422,,1062785.7389482185,10973.822986130603,30898.567818552689,25238.247794042127\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240423,,1099108.7939610016,16471.295975758694,55876.793493007353,33270.207943779111\n'
    'daily_sse,,,O2,RecentBias,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,RecentBias,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,RecentBias,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,RecentBias,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,RecentBias,,17150,,,,,,20240428,,1373938.8059740467,9480.8034344092011,32501.420121822928,26343.867545462574\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240429,,1037328.2546507826,11125.315953910351,35006.35105365271,25745.280764360883\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240430,,784666.03764799354,10727.697072362527,37822.303718782117,30344.261016303364\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240501,,937374.45299974177,9564.7267146771774,41544.334971594319,32882.682436172407\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240502,,1512711.5991847969,14340.125058846548,54207.608920642422,39219.330167360473\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240503,,1210601.2617156438,15504.707650474273,59703.414479565552,40806.378322383956\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240504,,1561078.2935047192,17521.345934184268,68375.204786961491,49839.060402020201\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240505,,1123324.9500292588,12180.220199164003,56277.903970466861,41374.792327973359\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240506,,1781624.0462396161,15005.272577381693,49415.042285979616,35788.891010209321\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240507,,1514309.0148334343,17283.335185380653,52113.976687166571,37623.049860621024\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240508,,1678433.4422859545,20405.78002258949,73167.903665784237,47352.765570402145\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240509,,2117697.3877827823,20809.924670021981,59685.398240365023,50721.100940878372\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240510,,2194104.3028247356,14053.352441416122,53086.471039209195,45721.656283704979\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240511,,1739364.4112539033,21567.536241647787,76727.383664273439,57783.262899076894\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240512,,3778864.1029100735,24630.768225473352,83671.124537415613,67712.10699082748\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240513,,1646238.1721932138,19499.902443591505,77182.609208595648,54618.457368299722\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240514,,1390202.5004003281,23368.120301739313,65904.601350861762,51895.45255656611\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240515,,2061822.5152625425,19433.547907806002,69418.625725951584,58639.862945016626\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240516,,1410244.8380680347,18487.298892140388,59348.053729181222,47145.516617505033\n'
    'daily_sse,,,O2,RecentBias,,17150,,,,,,20240517,,1279348.9812856046,16741.623931799084,68718.705216860559,47756.647900696116\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240518,,1306080.932154329,18659.223186791874,63937.211284241544,54248.880540103426\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240519,,1545082.2092207111,23235.660047459416,63550.530411891159,54896.017513495208\n'
    'daily_sse,,,O2,RecentBias,,19600,,,,,,20240520,,2283760.7950001722,30281.451792747714,89728.725376507165,74362.392864342241\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240421,,796577.32232718542,11343.056496595964,37640.18440349619,28628.78490284436\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240422,,937525.37095276965,10054.236876824871,28538.271669736216,22273.439866620727\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240423,,976585.98379741726,15573.757782795466,45817.887897486391,30540.723797415823\n'
    'daily_sse,,,O2,Reference,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,Reference,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,Reference,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,Reference,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,Reference,,17150,,,,,,20240428,,1240470.8612444284,8244.834739134647,26198.554691717574,21952.110105568117\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240429,,980141.27526948182,10722.297014936805,32009.262850157244,22792.91639690418\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240430,,714461.16761617945,9588.0649369573221,36337.336063400115,29433.370707812464\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240501,,930847.76124687772,8867.9581376546994,37178.709280160634,30436.173406310001\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240502,,1270336.5336579664,13147.917605651543,49817.265151992862,37230.586644449344\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240503,,1031161.5626945514,13948.65283605922,54337.698817726392,37720.520817476172\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240504,,1368845.818634866,15628.777886940166,57844.891720203246,43480.706515870654\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240505,,1083885.6785562523,11322.671604254283,51040.711449910785,36557.219204833367\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240506,,1593270.3731683649,13291.188915107399,42362.571098172302,32740.367691483825\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240507,,1267755.8326068064,15414.908386951312,43788.898087710571,35945.025726277039\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240508,,1700100.5560406181,20157.346635723487,57681.649619635224,47479.189231202137\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240509,,2031107.8020817139,18587.615285249427,54006.388922190861,46495.950039346215\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240510,,2068736.3396476889,12758.923446443863,51648.763868422138,42687.164956106993\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240511,,1773440.3696866552,20645.106863098219,70521.582375390921,55063.063264607481\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240512,,3490124.7808247642,23445.712285456248,64635.861039276329,65051.89684934351\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240513,,1467815.8740240925,18115.660901700146,66790.283046650278,49166.773463789759\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240514,,1388240.8438539098,20369.56843503844,57187.217790601586,45690.522179350497\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240515,,2236580.3666428835,19858.135517993011,65841.723899363322,52446.835614639451\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240516,,1656532.3655097799,18349.796744139865,56544.851481035868,42777.864843908166\n'
    'daily_sse,,,O2,Reference,,17150,,,,,,20240517,,1281702.3427658821,16275.917075953446,64958.14771804108,43319.754735976669\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240518,,1231155.7845052266,18836.80279902555,57863.279987903356,53855.686800511576\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240519,,1605867.7167965521,24037.156023783609,56668.091704198807,52690.530750691978\n'
    'daily_sse,,,O2,Reference,,19600,,,,,,20240520,,2196992.7137467428,31095.269105982035,79774.313672354343,71782.711992211523\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240421,,907427.55254523479,12327.259257502854,41076.047392710083,32222.630455248418\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240422,,1073086.5065921494,11181.81757575646,31106.508029968467,25228.549683167239\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240423,,1127334.2670434474,16545.993456889875,55821.432237015892,33849.983637779354\n'
    'daily_sse,,,O2,Taper,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,Taper,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,Taper,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,Taper,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,Taper,,17150,,,,,,20240428,,1396611.5315669603,9816.5942488024011,32177.415359747254,26084.31695303744\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240429,,1044065.4756104182,11858.342882745899,35576.614708565285,25980.72258727028\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240430,,767599.02912549139,10565.07049427554,37899.550745424,30437.847261514904\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240501,,951676.09364512283,9927.0909958742559,42749.044739854267,33419.80748709679\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240502,,1602919.1062685517,15261.636667198502,55975.953575743428,40580.307483369441\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240503,,1230155.1084950981,15383.299429537728,60064.958976340546,41573.384970716419\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240504,,1609510.0644319428,18367.581856951118,67987.846372191765,49136.187344422156\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240505,,1132986.8452899111,12476.166484533809,55747.944680690751,41318.982225291511\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240506,,1922635.9789812395,15446.923838639632,50332.543799788473,36928.077868339271\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240507,,1655511.3158089784,18030.533936976455,51284.998844733986,37913.332292758932\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240508,,1897033.5773934186,20925.427851392888,72452.458823958412,46983.392581022104\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240509,,2282771.1351556452,21527.578788074665,60438.642274419268,51492.583362447338\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240510,,2236497.8433029884,14830.066547932103,54370.806367472622,47362.531835132031\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240511,,1901438.9419091728,22357.895785442553,77258.479410707019,59144.938188907749\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240512,,4077337.6456821207,26049.106920631602,82689.188320067085,69621.603835409915\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240513,,1689091.6105615634,20453.508368408307,77705.619337564756,56552.008198215495\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240514,,1445787.0582998416,24764.205688466318,69846.140704596779,54607.840833452305\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240515,,1950744.8184936731,19984.650987251662,72173.306486230285,58933.521607955583\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240516,,1360979.4266766449,19703.601979696192,60977.411254747043,49338.523630415737\n'
    'daily_sse,,,O2,Taper,,17150,,,,,,20240517,,1288875.1250040277,17322.519970166497,68632.121858662445,48156.494735497145\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240518,,1414558.2670171289,20154.362869761884,64600.717747145551,55309.484646024779\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240519,,1739629.0343998889,23991.644106793217,62720.949316848644,56132.625059044643\n'
    'daily_sse,,,O2,Taper,,19600,,,,,,20240520,,2436066.2263456546,31967.249167348258,92092.741445927924,76909.472718470031\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240421,,804318.15676235151,11631.600646618754,37680.699651475828,28615.246801433255\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240422,,941730.6345993802,10236.985943210311,28587.275412171704,22301.260266858873\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240423,,979979.53383902134,15907.261501642875,45748.454349172331,30585.434992824976\n'
    'daily_sse,,,O2,Ultralisk,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,Ultralisk,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,Ultralisk,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,Ultralisk,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,Ultralisk,,17150,,,,,,20240428,,1257483.2740089947,8395.4181744344532,26196.059072761738,21941.852180115275\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240429,,996034.77932568546,11004.964622053318,32052.733794346426,22842.351962263827\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240430,,727306.66037793853,9727.3350887820125,36418.592167399031,29583.857825712057\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240501,,955060.94920447632,9075.2335648341104,37203.145172018398,30518.465834361967\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240502,,1259051.5161977352,13069.244034000672,49825.522309665801,37299.253377277637\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240503,,1041599.8634177884,14146.808621770702,54427.438149835034,37849.012458166704\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240504,,1379503.8298203826,15449.740101396106,57825.890926077147,43485.2571876777\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240505,,1111491.3127663247,11750.975802300498,51135.941361803816,36639.635451866008\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240506,,1581849.4375846502,13188.093995304778,42394.904478121272,32768.974560404065\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240507,,1251983.0395059171,15200.375504766591,43792.249073578088,36170.969206804424\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240508,,1724550.1112707458,21042.632652834989,57605.835031641669,48036.737951049581\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240509,,2041252.9097146832,18812.449888084084,54104.755790414813,46593.166938011826\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240510,,2074109.0464095408,12831.491232010536,51775.859943142001,42719.349690349001\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240511,,1801182.6347170877,21149.067548985593,70652.357919333765,55311.804457021288\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240512,,3479213.4081792482,23996.488309504464,64599.961080412984,65335.928546503026\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240513,,1479714.8808932323,18448.934277738445,66785.464821041111,49150.56211894528\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240514,,1448872.3892533719,20200.531951627694,57173.498244532326,45596.021519047572\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240515,,2343555.3570292653,21180.400767645799,65977.69712850888,52549.797629838315\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240516,,1755597.4544028158,18887.692022352479,56626.248417682727,42686.215666700744\n'
    'daily_sse,,,O2,Ultralisk,,17150,,,,,,20240517,,1319231.670996533,16796.327585576102,65052.112321305955,43309.257955128167\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240518,,1241016.4764421508,19526.82878424786,57899.034335614328,54161.519870950964\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240519,,1643620.3469023155,25303.129011833109,56770.944585634068,52894.719190887117\n'
    'daily_sse,,,O2,Ultralisk,,19600,,,,,,20240520,,2236211.1580553669,32233.139258257113,79856.980795176758,72022.183363772332\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240421,,907271.43214320973,12323.74126055371,40730.556225829598,32043.030307610818\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240422,,1070153.3838726827,11181.2307444904,31058.571240677978,25051.118029843681\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240423,,1128925.0680688662,16527.595213152468,55501.021735615752,33737.27283683428\n'
    'daily_sse,,,O2,WindFlow,,0,,,,,,20240424,,0,0,0,0\n'
    'daily_sse,,,O2,WindFlow,,0,,,,,,20240425,,0,0,0,0\n'
    'daily_sse,,,O2,WindFlow,,0,,,,,,20240426,,0,0,0,0\n'
    'daily_sse,,,O2,WindFlow,,0,,,,,,20240427,,0,0,0,0\n'
    'daily_sse,,,O2,WindFlow,,17150,,,,,,20240428,,1405271.7505141189,9762.9639317598194,31876.437065054582,25772.911096550466\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240429,,1052409.4756470961,11769.438642674126,35187.32149116919,25759.202955643636\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240430,,779985.62578605954,10566.691866863519,37676.226850307721,30088.582034561881\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240501,,957746.01101214206,9848.595396543853,42365.517327854745,33056.690526732171\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240502,,1569955.5684571369,15185.528141085058,55303.311932860983,40225.547553321972\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240503,,1202333.1500838201,15329.82103832718,58946.136254997051,41120.064466011085\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240504,,1574517.4916825357,18300.914523141459,67159.298097982057,48624.628937870883\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240505,,1113314.9690282312,12425.346811338328,55561.962242522372,41169.824661734609\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240506,,1894663.4416835841,15231.239583212882,49584.080690296374,36359.35568193467\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240507,,1599269.3879400045,17855.151027436368,50700.885915300001,37596.65659515428\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240508,,1825769.9769067327,20848.051150895655,71082.33781129113,46755.891364252348\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240509,,2219548.8222011039,21152.877219323069,59498.793037475101,50922.429869250154\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240510,,2242278.2013031999,14663.662437741645,53798.161484641241,46819.726400318679\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240511,,1863891.7712731815,21914.962742709555,76114.293920282798,58447.706716689994\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240512,,4008869.9112706743,25582.491458184086,81132.707756370539,68940.650745526509\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240513,,1683303.7469515572,20021.355544880033,76627.856805849806,55617.014959768145\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240514,,1419837.5234474009,24449.986557339318,68893.7668869849,53766.108351117269\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240515,,1958809.7093023269,19623.726047094911,71018.598855881646,58187.655860566207\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240516,,1332580.7656207969,19307.702614886686,59776.279852113497,48740.803818009292\n'
    'daily_sse,,,O2,WindFlow,,17150,,,,,,20240517,,1251238.3527331192,17034.841957242228,67319.983344304143,47524.658322819356\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240518,,1389290.3885085708,19704.592285416089,63395.533779394391,54479.376144763439\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240519,,1656315.8366748074,23328.401425444521,61520.744314376068,55358.129330355834\n'
    'daily_sse,,,O2,WindFlow,,19600,,,,,,20240520,,2366642.8873856389,31342.555342893116,90774.092348341277,76108.941547641007\n'
    'global,,,O3,Ultralisk,0.43091365302558421,627200,10.38317977090354,1.039481033679162,1.6544313133986055,1.5567309835892744,,,,,,,\n'
    'global,,,O3,PhysicsLegacy,0.40297204008998833,627200,10.837965901062612,1.0678300517361472,1.7517184824722205,1.6481104880272477,,,,,,,\n'
    'global,,,O3,Bias325,0.40989511143143681,627200,10.707172838381069,1.0472096058399585,1.7416892235441723,1.6297302261557576,,,,,,,\n'
    'global,,,O3,Reference,0.43884737567232052,627200,10.099090293348212,1.0195774034043326,1.6387280078410402,1.5462297265232074,,,,,,,\n'
    'global,,,O3,PhysicsNew,0.40297250262883499,627200,10.837977733944184,1.0678255830262426,1.7517188237428294,1.6481099026830297,,,,,,,\n'
    'global,,,O3,PhysicsNewBias,0.4098954628588824,627200,10.7071805959543,1.0472065420220278,1.7416893824531632,1.6297296261587635,,,,,,,\n'
    'global,,,O3,Placebo,0.39098658371487249,627200,11.000879982230732,1.0646962308432264,1.7926414644071913,1.7060450506351881,,,,,,,\n'
    'global,,,O3,Recent,0.40756433168089767,627200,10.711766108950524,1.0416722532875715,1.7480750545493864,1.6494337356705029,,,,,,,\n'
    'global,,,O3,RecentBias,0.4147063435881389,627200,10.537492699322712,1.0251466132771778,1.7376033067386401,1.6290102424613615,,,,,,,\n'
    'global,,,O3,Drift,0.40867068391207162,627200,10.779256431885514,1.0279439157068599,1.7587327642664661,1.6402074631713479,,,,,,,\n'
    'global,,,O3,DriftBias,0.4148794016763126,627200,10.632409892509372,1.0139219737390921,1.7490322801281819,1.622187434952012,,,,,,,\n'
    'global,,,O3,MultiScale,0.40926370151189101,627200,10.715801242172084,1.0485493753825863,1.7432342820138669,1.6317335619741438,,,,,,,\n'
    'global,,,O3,AdaptiveFlow,0.41215718318832328,627200,10.647572472960118,1.0425649147792118,1.7361298234655118,1.624776716073312,,,,,,,\n'
    'global,,,O3,WindFlow,0.41100738334324188,627200,10.707172838381069,1.0472096058399585,1.7361298234655118,1.624776716073312,,,,,,,\n'
    'global,,,O3,Taper,0.40778459037101528,627200,10.684513541038624,1.0557236953317417,1.7457665573280758,1.6374338591187352,,,,,,,\n'
    'global,,,O3,MixPhysicsNewBias,0.43884743069917381,627200,10.099095233238531,1.019576645885208,1.6387280278612515,1.5462296105087163,,,,,,,\n'
    'global,,,O3,MixRecentBias,0.43891375521709031,627200,10.090915140092331,1.0168760838380511,1.6404408511995106,1.5477435930354253,,,,,,,\n'
    'global,,,O3,MixDriftBias,0.44141867405140411,627200,10.059985155481504,1.0088094407163075,1.63558719397022,1.5413109398835727,,,,,,,\n'
    'global,,,O3,MixMultiScale,0.438800448016681,627200,10.099820379224177,1.0196540947108488,1.6387954095882626,1.5464314184284671,,,,,,,\n'
    'global,,,O3,MixAdaptiveFlow,0.4391675001701385,627200,10.085712216677338,1.01858750683368,1.6387176354702859,1.5456665438654498,,,,,,,\n'
    'global,,,O3,MixWindFlow,0.43891578790551888,627200,10.099090293348212,1.0195774034043326,1.6387176354702859,1.5456665438654498,,,,,,,\n'
    'global,,,O3,MixTaper,0.43838438347439712,627200,10.113641819127128,1.0212577601883071,1.6389308033352188,1.5470413423497609,,,,,,,\n'
    'weekly,,,O3,Ultralisk,0.45012563532933791,117600,11.522131552339811,0.89448417237534572,1.6692413300604825,1.4375691929302532,1,,,,,,\n'
    'weekly,,,O3,Ultralisk,0.41396345389689632,137200,11.931142475474481,1.0734285920646862,1.6661512689852334,1.5273925142534166,2,,,,,,\n'
    'weekly,,,O3,Ultralisk,0.48063803636483948,137200,8.6833286674221526,0.96624019822484075,1.5085485357303421,1.4692151184212447,3,,,,,,\n'
    'weekly,,,O3,Ultralisk,0.37077731407942549,117600,10.025413148256362,1.1336611878954563,1.8535986653666865,1.8378611422964928,4,,,,,,\n'
    'weekly,,,O3,Ultralisk,0.44633037085834698,117600,9.3635504352876335,1.1153359014367392,1.5754870216416876,1.4968012612468524,5,,,,,,\n'
    'weekly,,,O3,PhysicsLegacy,0.42269619506283779,117600,11.406592044004558,0.97827117203804259,1.7636050973877762,1.5178602213017371,1,,,,,,\n'
    'weekly,,,O3,PhysicsLegacy,0.40027650920044838,137200,12.439015314573137,1.0573359686206474,1.7073646668073783,1.5840115940536099,2,,,,,,\n'
    'weekly,,,O3,PhysicsLegacy,0.43876939002987642,137200,9.1877813780217163,1.0490307956092362,1.634979835509857,1.5950480425256353,3,,,,,,\n'
    'weekly,,,O3,PhysicsLegacy,0.34694164841711211,117600,10.822000614670092,1.118266718471308,1.9601385868227512,1.9038217401548636,4,,,,,,\n'
    'weekly,,,O3,PhysicsLegacy,0.40980892305134797,117600,10.011667520314395,1.133884786746441,1.7009857459871269,1.6318222763343764,5,,,,,,\n'
    'weekly,,,O3,Bias325,0.43236373325373129,117600,10.91134113517853,0.96053206843318639,1.7561091975724989,1.5016764908714408,1,,,,,,\n'
    'weekly,,,O3,Bias325,0.40918133999500589,137200,12.02120023947144,1.0370705564058793,1.6998807815432211,1.5703667079111483,2,,,,,,\n'
    'weekly,,,O3,Bias325,0.44470397882127949,137200,9.0897708255097154,1.0300136219704681,1.6259491028309625,1.5798242281982804,3,,,,,,\n'
    'weekly,,,O3,Bias325,0.35028835976457579,117600,11.184769290904288,1.0959341495100201,1.9441572384696264,1.8810474578306431,4,,,,,,\n'
    'weekly,,,O3,Bias325,0.4155483289952564,117600,10.113709403186608,1.1102154283340395,1.6909694389000574,1.6072313287738966,5,,,,,,\n'
    'weekly,,,O3,Reference,0.4617089586867929,117600,10.950749153675924,0.88122221434820514,1.647050518818147,1.4185861600606808,1,,,,,,\n'
    'weekly,,,O3,Reference,0.42874084172195892,137200,11.544800262680084,1.0383956692072769,1.6228419750428025,1.5049777130521602,2,,,,,,\n'
    'weekly,,,O3,Reference,0.48194453207839549,137200,8.535345698843015,0.96432511115524722,1.5102188141267121,1.4711207298038995,3,,,,,,\n'
    'weekly,,,O3,Reference,0.37767392005523981,117600,9.9222698161854392,1.1048450004926396,1.8447611972623887,1.824781241431402,4,,,,,,\n'
    'weekly,,,O3,Reference,0.45011158749151048,117600,9.2246783591926071,1.09730646062453,1.5725557309434708,1.4971687327095089,5,,,,,,\n'
    'weekly,,,O3,PhysicsNew,0.42269629773869999,117600,11.406623977367191,0.97826831535256875,1.7636051190699611,1.5178596059727876,1,,,,,,\n'
    'weekly,,,O3,PhysicsNew,0.40027731336300681,137200,12.439033975943316,1.0573284254396202,1.7073643600082495,1.5840112069909209,2,,,,,,\n'
    'weekly,,,O3,PhysicsNew,0.4387701204645425,137200,9.1877726811871767,1.0490258018744121,1.6349800182711476,1.5950475644237871,3,,,,,,\n'
    'weekly,,,O3,PhysicsNew,0.34694204468106099,117600,10.822001989432552,1.1182636481464845,1.9601388746287285,1.9038211972617491,4,,,,,,\n'
    'weekly,,,O3,PhysicsNew,0.40980916304047471,117600,10.011680230062655,1.1338814309906735,1.7009874205741673,1.6318213126364829,5,,,,,,\n'
    'weekly,,,O3,PhysicsNewBias,0.43236379836955419,117600,10.911364219099582,0.96053029298937698,1.756109080420571,1.5016759329308311,1,,,,,,\n'
    'weekly,,,O3,PhysicsNewBias,0.40918204842680661,137200,12.021210511516474,1.0370646099075378,1.6998803277133501,1.5703663114682456,2,,,,,,\n'
    'weekly,,,O3,PhysicsNewBias,0.44470461772745667,137200,9.0897591007600766,1.0300099384504773,1.625949053870291,1.5798236198865574,3,,,,,,\n'
    'weekly,,,O3,PhysicsNewBias,0.35028858303592331,117600,11.184772424892776,1.095932486335633,1.9441573097102929,1.8810469611108247,4,,,,,,\n'
    'weekly,,,O3,PhysicsNewBias,0.41554838891896601,117600,10.113722883990542,1.1102136606510398,1.6909709387749949,1.6072303361392657,5,,,,,,\n'
    'weekly,,,O3,Placebo,0.40265410526791079,117600,11.835433774547324,0.98377755390022581,1.8239814660691505,1.5972792831946665,1,,,,,,\n'
    'weekly,,,O3,Placebo,0.38119460167706021,137200,12.512592909520777,1.077706714194437,1.7568125234323493,1.6781524635961844,2,,,,,,\n'
    'weekly,,,O3,Placebo,0.42754724679355188,137200,9.5270961452943794,1.036387511472036,1.6722755911700544,1.6445281615936311,3,,,,,,\n'
    'weekly,,,O3,Placebo,0.33859632721584071,117600,10.908660972200098,1.1136528368282099,1.989029569608056,1.9485137220857336,4,,,,,,\n'
    'weekly,,,O3,Placebo,0.40873563575275701,117600,9.8897882963029407,1.1088580234946941,1.7298784189438292,1.6535895255465385,5,,,,,,\n'
    'weekly,,,O3,Recent,0.42595987834856469,117600,11.429313843349124,0.93500668088746564,1.7676795985670304,1.5292937900613006,1,,,,,,\n'
    'weekly,,,O3,Recent,0.39932517867921508,137200,12.422312472328356,1.0265741769788923,1.7281291518499078,1.6082368839876562,2,,,,,,\n'
    'weekly,,,O3,Recent,0.44433311302181339,137200,9.0953008388930563,1.0309071474507736,1.6263521394345557,1.5812484370559408,3,,,,,,\n'
    'weekly,,,O3,Recent,0.35409994764861907,117600,10.410685086303392,1.0996336688117463,1.9510222375540525,1.9038487458324251,4,,,,,,\n'
    'weekly,,,O3,Recent,0.41904732216870011,117600,9.8183839181482284,1.110978237016691,1.6721886629339962,1.6161955340055336,5,,,,,,\n'
    'weekly,,,O3,RecentBias,0.43532325527985821,117600,10.913715025317035,0.92322497357548416,1.7612536316645433,1.5104687499616991,1,,,,,,\n'
    'weekly,,,O3,RecentBias,0.40837666882837209,137200,11.97505254305357,1.0108453359261147,1.7209763068612534,1.5909473197531938,2,,,,,,\n'
    'weekly,,,O3,RecentBias,0.4508964317123571,137200,8.9638865369457257,1.0143881077038672,1.6137458078550984,1.5637867251132345,3,,,,,,\n'
    'weekly,,,O3,RecentBias,0.35750216645458049,117600,10.727628449131204,1.0820794713813542,1.9336795178968229,1.8806144283207991,4,,,,,,\n'
    'weekly,,,O3,RecentBias,0.42514747579523821,117600,9.8473425590645753,1.0904613572569988,1.6637386706303212,1.5902004556345335,5,,,,,,\n'
    'weekly,,,O3,Drift,0.42450479602943869,117600,11.500038363737076,0.96152813343638766,1.7671700676439579,1.5092105076135029,1,,,,,,\n'
    'weekly,,,O3,Drift,0.40707557043945869,137200,12.472445987847248,1.0072922848556125,1.709961311160713,1.5730467417418368,2,,,,,,\n'
    'weekly,,,O3,Drift,0.4486305031648477,137200,8.9082370865201668,1.0042990671465413,1.6428526980107463,1.5747943042882224,3,,,,,,\n'
    'weekly,,,O3,Drift,0.34812659913613131,117600,10.722019927950608,1.090999726508401,1.9863363582768936,1.9092947577498247,4,,,,,,\n'
    'weekly,,,O3,Drift,0.41892991476059571,117600,9.9200980724011263,1.0771658129280963,1.6935883102163742,1.6262978240234884,5,,,,,,\n'
    'weekly,,,O3,DriftBias,0.4348747192002429,117600,10.900302298222201,0.94776442942256278,1.7594924466754744,1.4918858158441406,1,,,,,,\n'
    'weekly,,,O3,DriftBias,0.41589278028174093,137200,11.995776017881038,0.99305006543318164,1.7030668373178459,1.5584303194286244,2,,,,,,\n'
    'weekly,,,O3,DriftBias,0.45388425712491243,137200,8.7956380392765627,0.99211007582163424,1.6333457951896799,1.5599256418948972,3,,,,,,\n'
    'weekly,,,O3,DriftBias,0.3494826118785393,117600,11.158955493517906,1.0765412659718216,1.9707516352911847,1.8887695451369368,4,,,,,,\n'
    'weekly,,,O3,DriftBias,0.42293703730958859,117600,10.072023803560834,1.0615241943743214,1.6850225172114406,1.6028200563125377,5,,,,,,\n'
    'weekly,,,O3,MultiScale,0.43158523261404369,117600,10.931556948537596,0.96219773673193276,1.757305959792689,1.5038963503428571,1,,,,,,\n'
    'weekly,,,O3,MultiScale,0.40875717726260818,137200,12.029118918584246,1.0379337761295948,1.7003967911673072,1.5719818904934693,2,,,,,,\n'
    'weekly,,,O3,MultiScale,0.44401570144446723,137200,9.1024553467693483,1.0313630374335687,1.6275871110560265,1.581886316932944,3,,,,,,\n'
    'weekly,,,O3,MultiScale,0.34959097246335691,117600,11.18280638724907,1.0976901757105268,1.9467907686072388,1.883204190698472,4,,,,,,\n'
    'weekly,,,O3,MultiScale,0.41495164358258457,117600,10.118492453679911,1.1113792073247075,1.6927428346163904,1.6092611207178094,5,,,,,,\n'
    'weekly,,,O3,AdaptiveFlow,0.43515639797970551,117600,10.812725567551452,0.95464805119176599,1.7517930970161577,1.4957874202430617,1,,,,,,\n'
    'weekly,,,O3,AdaptiveFlow,0.41173093622025519,137200,11.961733551949289,1.0328743661676409,1.6918347302692816,1.5644866099951431,2,,,,,,\n'
    'weekly,,,O3,AdaptiveFlow,0.44658309082300279,137200,9.0301532551325092,1.0251622462255514,1.6225941782213229,1.5765602011729924,3,,,,,,\n'
    'weekly,,,O3,AdaptiveFlow,0.35202012851584841,117600,11.153121345417144,1.0924331986332416,1.9403100074049675,1.8757301414709315,4,,,,,,\n'
    'weekly,,,O3,AdaptiveFlow,0.41799032676278558,117600,10.063428216569374,1.1052181132085801,1.6825272764459169,1.6026122269849741,5,,,,,,\n'
    'weekly,,,O3,WindFlow,0.43347195958928397,117600,10.91134113517853,0.96053206843318639,1.7517930970161577,1.4957874202430617,1,,,,,,\n'
    'weekly,,,O3,WindFlow,0.41063674683983542,137200,12.02120023947144,1.0370705564058793,1.6918347302692816,1.5644866099951431,2,,,,,,\n'
    'weekly,,,O3,WindFlow,0.4454081014350828,137200,9.0897708255097154,1.0300136219704681,1.6225941782213229,1.5765602011729924,3,,,,,,\n'
    'weekly,,,O3,WindFlow,0.35128433878600601,117600,11.184769290904288,1.0959341495100201,1.9403100074049675,1.8757301414709315,4,,,,,,\n'
    'weekly,,,O3,WindFlow,0.41688970936346759,117600,10.113709403186608,1.1102154283340395,1.6825272764459169,1.6026122269849741,5,,,,,,\n'
    'weekly,,,O3,Taper,0.43138492029117509,117600,10.954338692166107,0.96251304933123805,1.7573702331453165,1.5033248333022038,1,,,,,,\n'
    'weekly,,,O3,Taper,0.40680226913537221,137200,12.132154441317082,1.0427163785487581,1.7017020152855624,1.5739723151813121,2,,,,,,\n'
    'weekly,,,O3,Taper,0.44259107084693872,137200,9.0927642243920328,1.0380522141035344,1.6297189633965627,1.5861564758434588,3,,,,,,\n'
    'weekly,,,O3,Taper,0.3488131958437744,117600,10.970728180924786,1.107740651014457,1.9521783494211611,1.8927969874967709,4,,,,,,\n'
    'weekly,,,O3,Taper,0.41244576205456251,117600,10.01568221227245,1.1247879378625374,1.696462482186142,1.6221952271592719,5,,,,,,\n'
    'weekly,,,O3,MixPhysicsNewBias,0.46170896104696851,117600,10.950758243370904,0.88122159032931069,1.6470505012214125,1.4185860366078082,1,,,,,,\n'
    'weekly,,,O3,MixPhysicsNewBias,0.42874098855668541,137200,11.544805871485188,1.038394371041613,1.6228416464953854,1.5049775936136232,2,,,,,,\n'
    'weekly,,,O3,MixPhysicsNewBias,0.48194461849418779,137200,8.5353470071184141,0.96432437240311364,1.5102188599875883,1.4711206115114042,3,,,,,,\n'
    'weekly,,,O3,MixPhysicsNewBias,0.37767396908623319,117600,9.922273090241708,1.104844293060071,1.8447613366374029,1.824781168928691,4,,,,,,\n'
    'weekly,,,O3,MixPhysicsNewBias,0.45011157118738587,117600,9.2246832888064105,1.0973061107802795,1.5725560413197188,1.4971685747094046,5,,,,,,\n'
    'weekly,,,O3,MixRecentBias,0.46111771862739909,117600,10.95620945255291,0.87633141778964718,1.652218646962839,1.4239882621692233,1,,,,,,\n'
    'weekly,,,O3,MixRecentBias,0.42750211895911783,137200,11.56523890506897,1.034416206882083,1.6316985768667065,1.5107566082221391,2,,,,,,\n'
    'weekly,,,O3,MixRecentBias,0.48291680628570649,137200,8.524932861546656,0.96174689035396876,1.5080766363171785,1.4681353630675482,3,,,,,,\n'
    'weekly,,,O3,MixRecentBias,0.378295144105741,117600,9.863695741464058,1.1042151866229133,1.8432346348492801,1.8262423433446888,4,,,,,,\n'
    'weekly,,,O3,MixRecentBias,0.45088423203371591,117600,9.21465728788219,1.0955118782266202,1.5701532831188585,1.4952288862395684,5,,,,,,\n'
    'weekly,,,O3,MixDriftBias,0.46250003813086082,117600,10.967526811718029,0.87578661220222631,1.6475872450727302,1.415675315187837,1,,,,,,\n'
    'weekly,,,O3,MixDriftBias,0.43142924034319741,137200,11.528934119890186,1.0249075803567904,1.6200675914638598,1.4996320318593777,2,,,,,,\n'
    'weekly,,,O3,MixDriftBias,0.48547561470596301,137200,8.4441745052165267,0.95350248918195479,1.506180167612208,1.4632286166517667,3,,,,,,\n'
    'weekly,,,O3,MixDriftBias,0.37957648865552468,117600,9.8693710673368855,1.0969540039420225,1.8426824693469379,1.8228458570150687,4,,,,,,\n'
    'weekly,,,O3,MixDriftBias,0.45415111919302442,117600,9.1543799136606498,1.0822205665822535,1.5648288854915238,1.4904976809311397,5,,,,,,\n'
    'weekly,,,O3,MixMultiScale,0.46165393243994818,117600,10.953023613764202,0.8813059341963414,1.6470040283356704,1.4188100393908574,1,,,,,,\n'
    'weekly,,,O3,MixMultiScale,0.42874486040409387,137200,11.545604354607168,1.0383293390325188,1.6226150128110803,1.5051219652167449,2,,,,,,\n'
    'weekly,,,O3,MixMultiScale,0.48188409459276382,137200,8.5365775053519126,0.96444225136406436,1.5103224915396691,1.4713247631729669,3,,,,,,\n'
    'weekly,,,O3,MixMultiScale,0.37759582229692751,117600,9.9214390034548856,1.105047427725143,1.8451407372235229,1.8250012478334383,4,,,,,,\n'
    'weekly,,,O3,MixMultiScale,0.45006467091888203,117600,9.2246307707427153,1.0973685662321275,1.5726908155956962,1.4973963238311967,5,,,,,,\n'
    'weekly,,,O3,MixAdaptiveFlow,0.46206675470377889,117600,10.931397193384983,0.8799659073173357,1.6475462970799752,1.4180748449413509,1,,,,,,\n'
    'weekly,,,O3,MixAdaptiveFlow,0.4289885486350416,137200,11.532518363364868,1.0374854434528491,1.6234454666970122,1.5043698742501552,2,,,,,,\n'
    'weekly,,,O3,MixAdaptiveFlow,0.4823032676986837,137200,8.5219265627298739,0.96313051856162435,1.5098982777551655,1.4706872454490163,3,,,,,,\n'
    'weekly,,,O3,MixAdaptiveFlow,0.37791902317136178,117600,9.9126710006833605,1.1041202226609446,1.844967910458948,1.8240961679556689,4,,,,,,\n'
    'weekly,,,O3,MixAdaptiveFlow,0.45052676351017751,117600,9.2122787496447511,1.0963691909797415,1.5713681588678048,1.4965958958147871,5,,,,,,\n'
    'weekly,,,O3,MixWindFlow,0.46172390167173383,117600,10.950749153675924,0.88122221434820514,1.6475462970799752,1.4180748449413509,1,,,,,,\n'
    'weekly,,,O3,MixWindFlow,0.42875728697838622,137200,11.544800262680084,1.0383956692072769,1.6234454666970122,1.5043698742501552,2,,,,,,\n'
    'weekly,,,O3,MixWindFlow,0.48202637146734761,137200,8.535345698843015,0.96432511115524722,1.5098982777551655,1.4706872454490163,3,,,,,,\n'
    'weekly,,,O3,MixWindFlow,0.3777366606215451,117600,9.9222698161854392,1.1048450004926396,1.844967910458948,1.8240961679556689,4,,,,,,\n'
    'weekly,,,O3,MixWindFlow,0.45029106769918531,117600,9.2246783591926071,1.09730646062453,1.5713681588678048,1.4965958958147871,5,,,,,,\n'
    'weekly,,,O3,MixTaper,0.46148047045062729,117600,10.968220386024738,0.88156732169562424,1.6469365092111139,1.41879595808923,1,,,,,,\n'
    'weekly,,,O3,MixTaper,0.42827216070430169,137200,11.580933184582737,1.0394707216702519,1.6223089247720257,1.5052458565914193,2,,,,,,\n'
    'weekly,,,O3,MixTaper,0.4815410110960453,137200,8.5501763492335101,0.96588049981713142,1.51047773348651,1.4714953264547257,3,,,,,,\n'
    'weekly,,,O3,MixTaper,0.37717654085846058,117600,9.9103262932805496,1.1073975489747589,1.8462106627069643,1.82620741574934,4,,,,,,\n'
    'weekly,,,O3,MixTaper,0.44949926936143259,117600,9.232919092425,1.1000033268841167,1.5724528093505157,1.498957593904495,5,,,,,,\n'
    'daily,,,O3,Ultralisk,0.53194480740272498,19600,8.7117896059300772,0.81458461503992496,1.4018665681396554,1.2749156549642908,,20240528,1,,,,\n'
    'daily,,,O3,Ultralisk,0.52316594429924657,19600,10.519890267652013,0.83575098648760149,1.3995738125306163,1.1797739408413921,,20240529,2,,,,\n'
    'daily,,,O3,Ultralisk,0.47795469664368828,19600,11.423641646826528,0.78810908575842242,1.646615176266728,1.3384623839709171,,20240530,3,,,,\n'
    'daily,,,O3,Ultralisk,0.35226641984972867,19600,14.223074589418207,1.0128387185983601,1.9927965247313773,1.6604957217860752,,20240601,5,,,,\n'
    'daily,,,O3,Ultralisk,0.47808731943328969,19600,11.536111560701594,0.85099723491627077,1.4194384163718403,1.4416454418188156,,20240602,6,,,,\n'
    'daily,,,O3,Ultralisk,0.36882591530338049,19600,12.004849623502189,1.03316470445095,2.0240499081637866,1.6599052541393575,,20240603,7,,,,\n'
    'daily,,,O3,Ultralisk,0.3236294446288942,19600,13.809461767268065,1.2023357615451826,2.0483486139805334,1.6988101716402506,,20240604,8,,,,\n'
    'daily,,,O3,Ultralisk,0.41914636173990211,19600,10.139226297319109,1.1663826094902152,1.6122857525696157,1.579655412787957,,20240605,9,,,,\n'
    'daily,,,O3,Ultralisk,0.42344356906242892,19600,10.163210228006784,1.0550474681942976,1.6573674786257604,1.6191922945006574,,20240606,10,,,,\n'
    'daily,,,O3,Ultralisk,0.3744796536243159,19600,13.32532412636078,1.0823231423939601,1.8510968397515848,1.5891464705049976,,20240607,11,,,,\n'
    'daily,,,O3,Ultralisk,0.42868838094139128,19600,14.851467116519974,0.99663638537605537,1.5067611536095695,1.3659305874063361,,20240608,12,,,,\n'
    'daily,,,O3,Ultralisk,0.484688780949805,19600,9.9063650231808325,0.96447202341331195,1.4240776721587565,1.4023564765344114,,20240609,13,,,,\n'
    'daily,,,O3,Ultralisk,0.4701509184657251,19600,10.168713413397198,1.0254969562680487,1.4715536367867199,1.4034556557240878,,20240610,14,,,,\n'
    'daily,,,O3,Ultralisk,0.4457671882803273,19600,10.388065771538226,0.8659708503793645,1.7867441050562618,1.5045298387417969,,20240611,15,,,,\n'
    'daily,,,O3,Ultralisk,0.48147817087255829,19600,9.5720649291432824,0.92153328233058263,1.6142134311458534,1.3518499518581559,,20240612,16,,,,\n'
    'daily,,,O3,Ultralisk,0.51338824961963236,19600,8.3783782441208938,0.94079375802489262,1.3789740184348651,1.3475864915603262,,20240613,17,,,,\n'
    'daily,,,O3,Ultralisk,0.46408522862215468,19600,8.3837858035061785,1.0813516722229219,1.4859355253708679,1.5333659279592051,,20240614,18,,,,\n'
    'daily,,,O3,Ultralisk,0.48527767315308001,19600,8.1529578740438247,1.0665345634747232,1.3587643813750749,1.4895292125170636,,20240615,19,,,,\n'
    'daily,,,O3,Ultralisk,0.50500421761857384,19600,7.1416664767405864,0.94193073197823562,1.3612879546291794,1.5320406026151072,,20240616,20,,,,\n'
    'daily,,,O3,Ultralisk,0.48190561780396468,19600,8.384857254731612,0.92577963221283877,1.5241299241680362,1.5119513937671811,,20240617,21,,,,\n'
    'daily,,,O3,Ultralisk,0.34426399142859959,19600,9.1175440253748281,1.2937004587496284,1.909384438568086,1.92872382696134,,20240618,22,,,,\n'
    'daily,,,O3,Ultralisk,0.26766459406957732,19600,11.763658130528034,1.2814788996491184,2.2024567789223584,2.134394260727857,,20240619,23,,,,\n'
    'daily,,,O3,Ultralisk,0.30722600324659588,19600,11.252736230798757,1.2082783997190647,1.9698553226633055,2.1014420346058982,,20240620,24,,,,\n'
    'daily,,,O3,Ultralisk,0.38468103771652518,19600,8.8129723202052208,1.0567603165211483,1.8918532186613739,1.8694756107089243,,20240621,25,,,,\n'
    'daily,,,O3,Ultralisk,0.51674271627990753,19600,7.9064902847982736,0.96582636718585324,1.2817132806066918,1.4089460589281646,,20240623,27,,,,\n'
    'daily,,,O3,Ultralisk,0.44500309510419422,19600,10.711444751879124,0.941157509174392,1.7367140666672869,1.4472941402667625,,20240624,28,,,,\n'
    'daily,,,O3,Ultralisk,0.34942842891769532,19600,12.658783825540162,1.123572071980639,1.8472774811607904,1.8144843334130505,,20240625,29,,,,\n'
    'daily,,,O3,Ultralisk,0.4277102864409229,19600,9.6957972588049124,1.1599895559461535,1.5868164221066972,1.5709653247608619,,20240626,30,,,,\n'
    'daily,,,O3,Ultralisk,0.47775759282766972,19600,7.952656356633236,1.2289636707758422,1.4581253451940457,1.3271554392775544,,20240627,31,,,,\n'
    'daily,,,O3,Ultralisk,0.47474627375329359,19600,8.5051134016854366,1.1289683339018397,1.4481596742281493,1.4156733574143432,,20240628,32,,,,\n'
    'daily,,,O3,Ultralisk,0.50600874602247803,19600,8.2755273796919173,0.9772731705895884,1.4071272647414068,1.3588499808799308,,20240629,33,,,,\n'
    'daily,,,O3,Ultralisk,0.46359173686225219,19600,8.2302562284042313,1.0563479608391562,1.6610937898748062,1.4388372947572703,,20240630,34,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.51244986479459576,19600,8.9661720305012942,0.87694441304464044,1.4375668422182311,1.3258957970371821,,20240528,1,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.50702466599700402,19600,10.38843774649407,0.85472664736171222,1.4665504937821181,1.2539809829797159,,20240529,2,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.46347365682424951,19600,11.772502571842075,0.84983043606657782,1.6132893202953686,1.3942589241681369,,20240530,3,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.37254833700722551,19600,12.981063709483616,1.0467206198489178,1.9358950053510624,1.6034882785612581,,20240601,5,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.38795756148437338,19600,11.354552895722977,1.0303551879360404,1.8436284278668045,1.697186093885914,,20240602,6,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.32234671211682259,19600,12.502978419612266,1.1674888143254247,2.1668159076797702,1.7599308775630218,,20240603,7,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.31574497509428739,19600,14.537287063713704,1.1384799595964059,2.0023926179636056,1.8051132353523616,,20240604,8,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.41566800809804128,19600,11.309244367419554,1.0822623301784802,1.6591229409028705,1.5609027015041432,,20240605,9,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.4223836447516689,19600,10.388873264316718,1.0189183942213218,1.7157591418371605,1.6004625031786388,,20240606,10,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.38278495786514499,19600,13.526274390992556,1.0619921168612263,1.7570989670719008,1.5970892182213714,,20240607,11,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.4003751074809106,19600,15.074486347160216,1.038034491944539,1.6194839026716894,1.4542673083834201,,20240608,12,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.45243896326012711,19600,10.441669163124551,0.96984450574714443,1.5576919175916442,1.517986302371908,,20240609,13,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.42976179663170111,19600,10.80345363729324,1.0835825588732615,1.600948211125071,1.5288714992455461,,20240610,14,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.42929193539959137,19600,10.340331473481635,0.98662141792303837,1.746130174001054,1.5557398091242676,,20240611,15,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.46294541564953201,19600,9.2030275745366392,0.98629930546629785,1.6700900424336271,1.4279516209415968,,20240612,16,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.48459380056768919,19600,8.8836076504061161,0.986535663264412,1.4730492372268389,1.4268577687364823,,20240613,17,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.41970551742993528,19600,8.9906481846370543,1.1906186236216834,1.6342987818174923,1.6278741737419802,,20240614,18,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.43609960983789109,19600,9.0888371509753139,1.1282540927682554,1.5471322021156728,1.6138948490816607,,20240615,19,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.43664351569796239,19600,8.2007428842576964,1.0472225001192674,1.5982487937597742,1.7245169736870714,,20240616,20,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.41145117309313528,19600,9.4699382657004598,0.99829531916980641,1.7564504781380523,1.7565200898775282,,20240617,21,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.2898445601333543,19600,10.504618699620066,1.3573270090493637,2.0554621548090388,2.0907591231786959,,20240618,22,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.24017320994568131,19600,12.80705096465798,1.2933216451679022,2.2921060070258394,2.1962011407161333,,20240619,23,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.32037221073515942,19600,11.31974983289629,1.0759340427790676,2.0462727033188632,2.0603502223508903,,20240620,24,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.38334640093163252,19600,9.7578390144788205,0.99225469458446403,1.9625096806706408,1.8129579707865708,,20240621,25,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.48088766625952839,19600,8.6990593871484858,0.92642187588185843,1.4543578598496438,1.5483358318487339,,20240623,27,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.4009576507267979,19600,11.371051959022427,0.99380294779224143,1.8494193825177925,1.6197057317035477,,20240624,28,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.28869520966276579,19600,12.573292820117356,1.2331363030899021,2.0829122178942305,2.0342024704221888,,20240625,29,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.37961176768755811,19600,9.9098322598639488,1.1832337460218052,1.7886472822207322,1.7741337049844643,,20240626,30,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.42602269561732281,19600,10.325507800119448,1.2158417279094058,1.5602117462928904,1.4975049446122264,,20240627,31,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.4576674917627509,19600,8.8504623620128449,1.0935642681316906,1.505791868849514,1.520714306469412,,20240628,32,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.49240168129044631,19600,8.4137996475874584,0.97242426649654956,1.5027852590416817,1.3914134478300135,,20240629,33,,,,\n'
    'daily,,,O3,PhysicsLegacy,0.44005470815190839,19600,9.4535512694107648,1.0833919641734584,1.6918331065158361,1.483377602491011,,20240630,34,,,,\n'
    'daily,,,O3,Bias325,0.52033604437516556,19600,8.5577906158448958,0.86156380531659638,1.4383178392953757,1.3086223489045457,,20240528,1,,,,\n'
    'daily,,,O3,Bias325,0.51818892716227549,19600,9.6830905138281285,0.83789837653438859,1.4687106417659306,1.2341447720460157,,20240529,2,,,,\n'
    'daily,,,O3,Bias325,0.47268029011516088,19600,11.190100443964679,0.83115193033126689,1.6241586278276827,1.3757291959329403,,20240530,3,,,,\n'
    'daily,,,O3,Bias325,0.37949985354163029,19600,12.401087168485958,1.0353077440516378,1.9342796897347669,1.6059792495734062,,20240601,5,,,,\n'
    'daily,,,O3,Bias325,0.40258285137557992,19600,10.833880391628044,1.0070977858287304,1.804682133363688,1.6717943841183851,,20240602,6,,,,\n'
    'daily,,,O3,Bias325,0.330554577212824,19600,12.286899317537159,1.1464918841809648,2.1544004172323472,1.7400837036253165,,20240603,7,,,,\n'
    'daily,,,O3,Bias325,0.31967318889901608,19600,14.582808489900716,1.1183046415579461,2.0034581697083298,1.78812720486679,,20240604,8,,,,\n'
    'daily,,,O3,Bias325,0.42531942040904769,19600,10.784180677799815,1.0674647748416459,1.6428029722482913,1.5512140373150278,,20240605,9,,,,\n'
    'daily,,,O3,Bias325,0.42662394677799609,19600,10.265536089377964,1.0030893913359424,1.7093188962347627,1.5962290803963195,,20240606,10,,,,\n'
    'daily,,,O3,Bias325,0.39106390293676568,19600,12.986441457244831,1.0444541009938324,1.7629259062477167,1.5855802603719698,,20240607,11,,,,\n'
    'daily,,,O3,Bias325,0.41286746012828079,19600,14.244653007343851,1.0114717017660344,1.6136344658494828,1.4496733299957878,,20240608,12,,,,\n'
    'daily,,,O3,Bias325,0.4626663177496021,19600,9.9648894650102804,0.94607819837398077,1.546974923895297,1.5042112716322942,,20240609,13,,,,\n'
    'daily,,,O3,Bias325,0.44382809632935982,19600,10.313577253477792,1.0598629151379189,1.5773046669360689,1.4941489009704261,,20240610,14,,,,\n'
    'daily,,,O3,Bias325,0.43579844427445841,19600,9.7807162272934391,0.97031624570614117,1.7631673791140785,1.5506735172316091,,20240611,15,,,,\n'
    'daily,,,O3,Bias325,0.4676726258503392,19600,8.9675302682263602,0.97849025617718643,1.666210547928999,1.4187817468421706,,20240612,16,,,,\n'
    'daily,,,O3,Bias325,0.49096774128823478,19600,8.7951437295036357,0.96642484839445442,1.4572058093167448,1.4135963690490569,,20240613,17,,,,\n'
    'daily,,,O3,Bias325,0.42818312530976532,19600,8.7000561760505573,1.1623873717680573,1.6323619702678931,1.6110513889590934,,20240614,18,,,,\n'
    'daily,,,O3,Bias325,0.43878394783262342,19600,9.3020987329504621,1.1114019533225259,1.5296056835699527,1.6046236904250843,,20240615,19,,,,\n'
    'daily,,,O3,Bias325,0.44002530020549702,19600,8.3975192138838768,1.0251495627759462,1.5844458548733935,1.7131532239390672,,20240616,20,,,,\n'
    'daily,,,O3,Bias325,0.42003319395893712,19600,9.6018553476630863,0.97789177031467245,1.7270610907716877,1.7175359422085286,,20240617,21,,,,\n'
    'daily,,,O3,Bias325,0.29558980971708682,19600,11.062770992410016,1.3260055406448117,2.0209519187980298,2.0554048864019601,,20240618,22,,,,\n'
    'daily,,,O3,Bias325,0.24322935185390579,19600,13.509063740321992,1.2685750545894423,2.2622500888013226,2.161184686990584,,20240619,23,,,,\n'
    'daily,,,O3,Bias325,0.31677993115223479,19600,12.086328611422202,1.0586923322373949,2.0487474924460938,2.042742736097459,,20240620,24,,,,\n'
    'daily,,,O3,Bias325,0.38443405940938641,19600,10.001676260535096,0.98189811156239559,1.9542103368271952,1.800742770345114,,20240621,25,,,,\n'
    'daily,,,O3,Bias325,0.48498299034462738,19600,8.7480700849291146,0.90182720504700598,1.4454864233212348,1.5418818254809137,,20240623,27,,,,\n'
    'daily,,,O3,Bias325,0.41089101393499172,19600,11.093830800912592,0.96985098537621195,1.8359539091751744,1.5942887940274877,,20240624,28,,,,\n'
    'daily,,,O3,Bias325,0.30435272661119672,19600,12.099188750219524,1.1980376007939495,2.0534722937451537,2.0009158645543184,,20240625,29,,,,\n'
    'daily,,,O3,Bias325,0.39225101608926771,19600,9.7904018592927553,1.1474130462184129,1.7558527070931291,1.7401998983196278,,20240626,30,,,,\n'
    'daily,,,O3,Bias325,0.42827263447425568,19600,10.749398618541402,1.189768250770507,1.5556556558844452,1.4737314035349964,,20240627,31,,,,\n'
    'daily,,,O3,Bias325,0.4609802253148178,19600,9.0734567746254395,1.0658609369485983,1.5150138218328222,1.49550895128942,,20240628,32,,,,\n'
    'daily,,,O3,Bias325,0.4910989860196831,19600,8.858555197519312,0.96727982037953797,1.4987123740647663,1.3740554984544475,,20240629,33,,,,\n'
    'daily,,,O3,Bias325,0.43837324508077802,19600,9.7534189740806756,1.0753514374144582,1.7014783246853271,1.4733416087357738,,20240630,34,,,,\n'
    'daily,,,O3,Reference,0.53799587245257074,19600,8.490517358186402,0.80617517603723732,1.3879994273093512,1.2619138806669468,,20240528,1,,,,\n'
    'daily,,,O3,Reference,0.5308259372750358,19600,10.204749143452821,0.81868831348686055,1.3926834586879495,1.1643717450645317,,20240529,2,,,,\n'
    'daily,,,O3,Reference,0.48934368253739219,19600,10.985474023987411,0.77361839232643947,1.6140166048736746,1.3194925997920255,,20240530,3,,,,\n'
    'daily,,,O3,Reference,0.37719934600160188,19600,13.312100088960614,0.98385069994631957,1.9241872506295503,1.6100798614492386,,20240601,5,,,,\n'
    'daily,,,O3,Reference,0.48021940217574288,19600,10.839522719603044,0.84885015447057499,1.4596074906401371,1.4520363369950755,,20240602,6,,,,\n'
    'daily,,,O3,Reference,0.38204577053666311,19600,11.309277208157043,1.0257447907694028,1.9931238395781119,1.6383847390235646,,20240603,7,,,,\n'
    'daily,,,O3,Reference,0.34590064056021108,19600,13.311484527719845,1.1500407982376752,1.959537868226048,1.6759333498133018,,20240604,8,,,,\n'
    'daily,,,O3,Reference,0.43459079563608261,19600,9.8759280534053353,1.1216709220672298,1.573436650283504,1.547848042046263,,20240605,9,,,,\n'
    'daily,,,O3,Reference,0.44013292039860608,19600,9.7343023873538765,1.0168886103017325,1.614399429883973,1.5872095638110029,,20240606,10,,,,\n'
    'daily,,,O3,Reference,0.39651469630663583,19600,12.740306916182249,1.0420366524254303,1.7748344208076441,1.5534845642830486,,20240607,11,,,,\n'
    'daily,,,O3,Reference,0.4408114336623547,19600,14.45411058196877,0.96480325070582218,1.4822355649057404,1.3487079552969312,,20240608,12,,,,\n'
    'daily,,,O3,Reference,0.4920576815575643,19600,9.6352075127975514,0.93812291382845681,1.4174881388007388,1.3949169737641471,,20240609,13,,,,\n'
    'daily,,,O3,Reference,0.4739972729247362,19600,9.9707931816414206,1.0179866015792145,1.4687420972563523,1.3974134176784352,,20240610,14,,,,\n'
    'daily,,,O3,Reference,0.45099255113383191,19600,10.06125531847403,0.87204043699386613,1.76545843698127,1.4982585757594695,,20240611,15,,,,\n'
    'daily,,,O3,Reference,0.4870306496232662,19600,9.2550750991066106,0.91721608190372639,1.6035333789662676,1.3442694618793114,,20240612,16,,,,\n'
    'daily,,,O3,Reference,0.51677230339480451,19600,8.2696427962720325,0.93207671588327201,1.3747055615504129,1.3404187477987537,,20240613,17,,,,\n'
    'daily,,,O3,Reference,0.46505242351428522,19600,8.2539231461211298,1.085262296745646,1.4925102795534559,1.5269005070979504,,20240614,18,,,,\n'
    'daily,,,O3,Reference,0.48349006418566631,19600,8.1814544941063385,1.0635416955338772,1.3736483837588327,1.4935316266574064,,20240615,19,,,,\n'
    'daily,,,O3,Reference,0.50276875043090763,19600,7.0711416661215898,0.9414323833142908,1.3747323998761469,1.5465348985663729,,20240616,20,,,,\n'
    'daily,,,O3,Reference,0.47895458118328571,19600,8.3443470886689894,0.91855508700104882,1.5436845230246576,1.5319855662822823,,20240617,21,,,,\n'
    'daily,,,O3,Reference,0.34366623132605828,19600,9.1970095703834804,1.2888663498200232,1.9082250727669119,1.9329643147118685,,20240618,22,,,,\n'
    'daily,,,O3,Reference,0.27212807625583812,19600,11.815930710880474,1.2659013050868009,2.1875283904607365,2.1202049174488349,,20240619,23,,,,\n'
    'daily,,,O3,Reference,0.31817451185172591,19600,11.19423282038492,1.156101118373313,1.9628184942818407,2.0730630325938737,,20240620,24,,,,\n'
    'daily,,,O3,Reference,0.39722167405642278,19600,8.7349637149124284,1.0071547598143851,1.8758491006781497,1.8337991052329079,,20240621,25,,,,\n'
    'daily,,,O3,Reference,0.52113660311978449,19600,7.7409618400884606,0.93078020175980236,1.2900107795352369,1.4148385350527537,,20240623,27,,,,\n'
    'daily,,,O3,Reference,0.4543719373237341,19600,10.245706246107634,0.91822109464649804,1.7194957867976242,1.4440089905624831,,20240624,28,,,,\n'
    'daily,,,O3,Reference,0.35256936200334649,19600,12.252552305882302,1.1124509460601095,1.8558039052231088,1.8262907951095075,,20240625,29,,,,\n'
    'daily,,,O3,Reference,0.4313302268653354,19600,9.3982469583889205,1.1407002077996122,1.5884651809970756,1.5834444097280411,,20240626,30,,,,\n'
    'daily,,,O3,Reference,0.47899215903876918,19600,8.1738051823421198,1.2050749366365561,1.447111406344386,1.3313765493914542,,20240627,31,,,,\n'
    'daily,,,O3,Reference,0.47949587714385111,19600,8.4162931232075167,1.1035781954154289,1.4404036155982671,1.4150363992119912,,20240628,32,,,,\n'
    'daily,,,O3,Reference,0.50987690738242197,19600,8.1896006951215554,0.96389659389411564,1.409042576698581,1.345651036639008,,20240629,33,,,,\n'
    'daily,,,O3,Reference,0.46877294514216239,19600,8.2098368098228303,1.0423807652097361,1.6480486687314624,1.4215423941779952,,20240630,34,,,,\n'
    'daily,,,O3,PhysicsNew,0.51245028666478709,19600,8.966160335717074,0.87694138423016754,1.4375679948286919,1.3258953999802794,,20240528,1,,,,\n'
    'daily,,,O3,PhysicsNew,0.50702506663461799,19600,10.388447881178028,0.8547226573399811,1.4665513855243948,1.2539801412714826,,20240529,2,,,,\n'
    'daily,,,O3,PhysicsNew,0.46347433732507931,19600,11.772531688242635,0.84982496215052472,1.613286627149134,1.3942584779013765,,20240530,3,,,,\n'
    'daily,,,O3,PhysicsNew,0.37254828099979681,19600,12.981117595264646,1.0467189276456801,1.9358937998351089,1.6034869608851905,,20240601,5,,,,\n'
    'daily,,,O3,PhysicsNew,0.38795709035334908,19600,11.35460904302515,1.0303555228610175,1.8436267301159528,1.697186391149986,,20240602,6,,,,\n'
    'daily,,,O3,PhysicsNew,0.3223466101944214,19600,12.503018832579411,1.1674848544262195,2.1668191720452894,1.7599298596926194,,20240603,7,,,,\n'
    'daily,,,O3,PhysicsNew,0.31574598657868841,19600,14.537302942874865,1.1384742779047481,2.0023909371370654,1.8051105356822732,,20240604,8,,,,\n'
    'daily,,,O3,PhysicsNew,0.41566871753762891,19600,11.309280132080874,1.0822496172278515,1.6591268891115218,1.5609036059086177,,20240605,9,,,,\n'
    'daily,,,O3,PhysicsNew,0.42238489967708948,19600,10.388840675389218,1.0189099970389053,1.7157608627591558,1.600461853263639,,20240606,10,,,,\n'
    'daily,,,O3,PhysicsNew,0.38278622174509003,19600,13.526279038506305,1.0619864573441822,1.7570951424237775,1.5970869845438611,,20240607,11,,,,\n'
    'daily,,,O3,PhysicsNew,0.4003757026044748,19600,15.074536975333549,1.0380257632743088,1.6194829357003344,1.4542677484701187,,20240608,12,,,,\n'
    'daily,,,O3,PhysicsNew,0.45243925414940789,19600,10.441700142202151,0.96983727509561835,1.5576933038729741,1.5179875514345249,,20240609,13,,,,\n'
    'daily,,,O3,PhysicsNew,0.42976230531610482,19600,10.803470171078756,1.0835779788383677,1.6009459140840985,1.5288723110966955,,20240610,14,,,,\n'
    'daily,,,O3,PhysicsNew,0.42929270675279668,19600,10.340319900885325,0.98661482885915197,1.7461311034316611,1.5557402565543299,,20240611,15,,,,\n'
    'daily,,,O3,PhysicsNew,0.46294515179799378,19600,9.2030466641956234,0.98629378155604563,1.6700939480823851,1.4279547915456678,,20240612,16,,,,\n'
    'daily,,,O3,PhysicsNew,0.48459483517398899,19600,8.8835864225426437,0.98653085978630484,1.4730487368234313,1.4268561225281071,,20240613,17,,,,\n'
    'daily,,,O3,PhysicsNew,0.41970652643531547,19600,8.9906520673126114,1.1906119811459817,1.6342979581400181,1.6278727852326491,,20240614,18,,,,\n'
    'daily,,,O3,PhysicsNew,0.43610015912530559,19600,9.0888201136581568,1.1282533516674671,1.547129876603994,1.6138942270325398,,20240615,19,,,,\n'
    'daily,,,O3,PhysicsNew,0.4366444197306209,19600,8.2007375798856721,1.0472169672155338,1.598247968577569,1.7245160975732905,,20240616,20,,,,\n'
    'daily,,,O3,PhysicsNew,0.41145229134430777,19600,9.4699104587813157,0.99828986671949915,1.7564510166946938,1.7565181327621993,,20240617,21,,,,\n'
    'daily,,,O3,PhysicsNew,0.28984553221594561,19600,10.504604707541274,1.3573242848592719,2.0554601885001009,2.0907564444859861,,20240618,22,,,,\n'
    'daily,,,O3,PhysicsNew,0.24017306650539991,19600,12.80705656649083,1.2933233750723738,2.2921065970469399,2.1961996686850642,,20240619,23,,,,\n'
    'daily,,,O3,PhysicsNew,0.32037258809772662,19600,11.319745285163409,1.075931226773585,2.0462688180316406,2.0603533213120664,,20240620,24,,,,\n'
    'daily,,,O3,PhysicsNew,0.38334655764561448,19600,9.7578299907563792,0.99225027672267763,1.9625140458856396,1.8129584674073485,,20240621,25,,,,\n'
    'daily,,,O3,PhysicsNew,0.48088880689718128,19600,8.6990423856352077,0.92641642846918781,1.4543578061093936,1.5483332550053477,,20240623,27,,,,\n'
    'daily,,,O3,PhysicsNew,0.40095784223129738,19600,11.371091703092397,0.99379622583184724,1.8494223758004364,1.6197053220341604,,20240624,28,,,,\n'
    'daily,,,O3,PhysicsNew,0.2886957390608102,19600,12.573355938423385,1.2331298418045664,2.0829125345225856,2.034199138432101,,20240625,29,,,,\n'
    'daily,,,O3,PhysicsNew,0.37961192087172801,19600,9.9098474013148596,1.1832313885481582,1.7886487612500122,1.7741324093256774,,20240626,30,,,,\n'
    'daily,,,O3,PhysicsNew,0.4260234236961089,19600,10.325495911353748,1.2158380634862798,1.5602118457943668,1.4975034701221079,,20240627,31,,,,\n'
    'daily,,,O3,PhysicsNew,0.45766784428576751,19600,8.85046230732436,1.0935609684950969,1.5057935765232515,1.5207133712952177,,20240628,32,,,,\n'
    'daily,,,O3,PhysicsNew,0.49240130010585059,19600,8.4138349711147882,0.97242202319061055,1.5027867682618681,1.3914148140591889,,20240629,33,,,,\n'
    'daily,,,O3,PhysicsNew,0.4400546859260443,19600,9.4535138077132199,1.0833902768873582,1.6918383026749777,1.4833785262646522,,20240630,34,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.52033626193328975,19600,8.5577796588514712,0.86156227654809958,1.4383190807256108,1.3086220116364886,,20240528,1,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.5181893160730332,19600,9.6830947841139032,0.8378955033017933,1.4687110760102009,1.2341437387984411,,20240529,2,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.47268090272459717,19600,11.190122369583769,0.83114782408760135,1.6241555261415972,1.3757288445343971,,20240530,3,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.37949983108624868,19600,12.401127353338223,1.0353069739909284,1.9342782671024024,1.6059780166968538,,20240601,5,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.40258233300713769,19600,10.833924152084366,1.0070991451913578,1.8046807155508668,1.6717948416979886,,20240602,6,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.33055450782516987,19600,12.286927469078854,1.1464886861859869,2.1544035226662945,1.740082777163678,,20240603,7,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.31967414754025492,19600,14.582813110128836,1.1183003883738269,2.003456282616896,1.788124590091402,,20240604,8,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.42532008995789922,19600,10.784199498492409,1.0674537494917171,1.6428068174265569,1.551215044049981,,20240605,9,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.42662508076525379,19600,10.265503327962676,1.003082631273543,1.7093199797157064,1.5962282966685195,,20240606,10,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.39106502880377148,19600,12.986440853409729,1.0444501194828837,1.7629218970657028,1.5855780591075026,,20240607,11,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.4128679850335874,19600,14.24469056121478,1.0114644637068766,1.6136336671679634,1.449673794704859,,20240608,12,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.46266649364693541,19600,9.9649145283868155,0.94607259147906564,1.5469763232445275,1.5042123163389982,,20240609,13,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.4438284137042518,19600,10.313592136049962,1.0598600110285878,1.5773023867925644,1.4941497389806109,,20240610,14,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.43579908277989421,19600,9.7807024619460528,0.9703113422088584,1.763167956139047,1.5506738254032286,,20240611,15,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.46767243646992041,19600,8.9675403917022827,0.97848569828524201,1.6662137683860303,1.4187845913822881,,20240612,16,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.49096872343911813,19600,8.7951183078962991,0.96642114824219638,1.4572050801883925,1.4135945691207821,,20240613,17,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.42818400001604029,19600,8.700058974087117,1.1623821314899931,1.6323608803709715,1.611050001339446,,20240614,18,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.43878432687736107,19600,9.3020830714759608,1.1114024736608423,1.5296036239085855,1.6046228943206631,,20240615,19,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.44002609330353037,19600,8.3975082446372369,1.0251455365470632,1.5844448983500219,1.7131523198821277,,20240616,20,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.4200341995040362,19600,9.6018277350277632,0.9778875526565356,1.7270614190824531,1.7175338259745101,,20240617,21,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.29559049968412049,19600,11.062764049238842,1.3260045153239686,2.0209501119607309,2.0554021416022259,,20240618,22,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.2432289514561658,19600,13.509080780885682,1.2685781903619993,2.2622503109886649,2.1611832847027768,,20240619,23,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.31678013004212069,19600,12.086330798594531,1.0586908969739794,2.0487431857092964,2.0427457038353092,,20240620,24,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.38443415287484778,19600,10.001661175826399,0.98189475123116499,1.95421440819077,1.8007434541496203,,20240621,25,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.48498395418746659,19600,8.7480532310087717,0.90182315632280463,1.4454861989042709,1.5418794320047644,,20240623,27,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.4108911337461561,19600,11.093860439205722,0.96984574278708335,1.8359567259072271,1.5942884571507925,,20240624,28,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.30435319686602469,19600,12.099236728252984,1.198032816837022,2.0534726006875732,2.0009125847902478,,20240625,29,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.392250881464839,19600,9.7904202435328767,1.1474125958014452,1.7558541890593968,1.7401988090046308,,20240626,30,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.42827299969394012,19600,10.749395395225042,1.1897667393480631,1.5556555057523971,1.4737302640103831,,20240627,31,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.46098044462207471,19600,9.0734590146442162,1.0658590743739629,1.5150150666338149,1.4955078478543915,,20240628,32,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.4910984751677997,19600,8.8585969519001377,0.96727844318150158,1.498713923775451,1.3740565110077916,,20240629,33,,,,\n'
    'daily,,,O3,PhysicsNewBias,0.43837308604627989,19600,9.7533884206521488,1.0753510549708678,1.7014830324319694,1.4733421680342291,,20240630,34,,,,\n'
    'daily,,,O3,Placebo,0.50938326374718645,19600,9.0633733906799208,0.86850745129782658,1.4427627340984597,1.347979138959688,,20240528,1,,,,\n'
    'daily,,,O3,Placebo,0.50558043718326973,19600,10.750208581869568,0.82958895345019823,1.4515337855494268,1.2733785737596106,,20240529,2,,,,\n'
    'daily,,,O3,Placebo,0.43937840818040907,19600,12.146861933300158,0.88036671642449738,1.7076239918591831,1.4602313067359178,,20240530,3,,,,\n'
    'daily,,,O3,Placebo,0.31608419014253381,19600,13.731619542382536,1.0867473354276649,2.1472792300165846,1.8079344806245987,,20240601,5,,,,\n'
    'daily,,,O3,Placebo,0.3805785775848573,19600,12.067504632735602,1.0148299449261915,1.8136099176197691,1.7392120339584942,,20240602,6,,,,\n'
    'daily,,,O3,Placebo,0.30268013042886671,19600,12.689560113459113,1.1740704770732251,2.2264247059726494,1.8556683015796691,,20240603,7,,,,\n'
    'daily,,,O3,Placebo,0.28035006481406322,19600,14.500200536715122,1.1947069219126765,2.1104583844632216,1.9626371381995944,,20240604,8,,,,\n'
    'daily,,,O3,Placebo,0.40125591891120599,19600,11.013353726617265,1.133789034865275,1.6691224973339884,1.6457249452172769,,20240605,9,,,,\n'
    'daily,,,O3,Placebo,0.39568267589738471,19600,10.736343461049207,1.0686416692587051,1.7519946329240192,1.7163092906887512,,20240606,10,,,,\n'
    'daily,,,O3,Placebo,0.34696290387840079,19600,13.92340040103586,1.0910228011100691,1.8970824347312516,1.7250426835916126,,20240607,11,,,,\n'
    'daily,,,O3,Placebo,0.3967341007379574,19600,15.003430162294093,1.016882507980392,1.6167241651743569,1.5140534633756386,,20240608,12,,,,\n'
    'daily,,,O3,Placebo,0.43375046645893639,19600,10.685560717176154,0.97619569634774417,1.6176314634631883,1.6008137616656202,,20240609,13,,,,\n'
    'daily,,,O3,Placebo,0.43575348197023489,19600,10.769903398206772,1.0478674942325257,1.5702839418265755,1.5416102863407557,,20240610,14,,,,\n'
    'daily,,,O3,Placebo,0.40915116473450452,19600,10.99633241498972,0.97862728048574354,1.8019356634513908,1.6344941178661376,,20240611,15,,,,\n'
    'daily,,,O3,Placebo,0.445663718125135,19600,9.6225792537573742,0.98847930582421439,1.7239664815254534,1.4935152004817189,,20240612,16,,,,\n'
    'daily,,,O3,Placebo,0.47838448848801579,19600,9.3006114663897659,0.99053384521858445,1.4540787461754436,1.4551296933837308,,20240613,17,,,,\n'
    'daily,,,O3,Placebo,0.41863985886554861,19600,9.2534564195546096,1.1242908548371782,1.6575699104750692,1.6640426116015712,,20240614,18,,,,\n'
    'daily,,,O3,Placebo,0.42254338121832019,19600,9.1534090140932456,1.1342779560523839,1.5968630183522741,1.6769173128367314,,20240615,19,,,,\n'
    'daily,,,O3,Placebo,0.4226666921465928,19600,8.673018636657714,1.0296279113089168,1.6477815332533068,1.7815190137450465,,20240616,20,,,,\n'
    'daily,,,O3,Placebo,0.40497980089414848,19600,9.5242632495932646,0.99598367620458039,1.7970722067344982,1.7767525253375185,,20240617,21,,,,\n'
    'daily,,,O3,Placebo,0.3053240562540227,19600,10.041704362492061,1.2773622266298521,2.0081089191160171,2.1173568296818623,,20240618,22,,,,\n'
    'daily,,,O3,Placebo,0.2444194408003646,19600,12.293824680968656,1.2629898549717995,2.3037586298805892,2.2245672637653877,,20240619,23,,,,\n'
    'daily,,,O3,Placebo,0.3097638764107109,19600,11.608509955357391,1.085977582742514,2.0837794037042032,2.0857690212636912,,20240620,24,,,,\n'
    'daily,,,O3,Placebo,0.36254203539700031,19600,10.433916009228138,1.0316060563446756,1.9858772737198309,1.8730322184309796,,20240621,25,,,,\n'
    'daily,,,O3,Placebo,0.47031739094295011,19600,8.7757483689255302,0.95574977211326284,1.4718506740344812,1.5870045973618787,,20240623,27,,,,\n'
    'daily,,,O3,Placebo,0.3664222049734624,19600,11.890394751326186,1.0283589944517502,1.9863124025777383,1.7236789731066038,,20240624,28,,,,\n'
    'daily,,,O3,Placebo,0.28886277338738581,19600,12.705741383367799,1.1803008671346025,2.0815200854714302,2.0763941558199432,,20240625,29,,,,\n'
    'daily,,,O3,Placebo,0.37150267640579798,19600,10.09084258096137,1.1459417165359786,1.8506276651714764,1.8163206388549615,,20240626,30,,,,\n'
    'daily,,,O3,Placebo,0.42887311727222321,19600,9.9638081258203552,1.2205941087909999,1.5634414766364464,1.4961554013190981,,20240627,31,,,,\n'
    'daily,,,O3,Placebo,0.46165829415690218,19600,8.515664560187103,1.0327089106750555,1.5480927468435544,1.5434849957012631,,20240628,32,,,,\n'
    'daily,,,O3,Placebo,0.49850169451137799,19600,7.9689544340543232,0.95229225086223435,1.5125096614466451,1.3898636500274513,,20240629,33,,,,\n'
    'daily,,,O3,Placebo,0.43141111659402132,19600,9.3961831153692845,1.0990477420476428,1.751770979603319,1.4977211616354953,,20240630,34,,,,\n'
    'daily,,,O3,Recent,0.51348647171515038,19600,8.9001550700803573,0.85576938211135301,1.4352827605827638,1.3458596351578664,,20240528,1,,,,\n'
    'daily,,,O3,Recent,0.5091864909101429,19600,10.432204364726854,0.83867754033186981,1.4600970375423767,1.2535807143738804,,20240529,2,,,,\n'
    'daily,,,O3,Recent,0.46861926372504648,19600,11.490515546543072,0.81746719998023576,1.6256062301072929,1.3976151688749938,,20240530,3,,,,\n'
    'daily,,,O3,Recent,0.36483770590847908,19600,12.91566693552935,1.014522133473641,1.9820587990167609,1.6698144687593952,,20240601,5,,,,\n'
    'daily,,,O3,Recent,0.40956814930552149,19600,11.354851693696549,0.93974774137572603,1.7759964516320423,1.6610241581629182,,20240602,6,,,,\n'
    'daily,,,O3,Recent,0.3203134050863008,19600,12.95965217507827,1.1086343437024655,2.1979333299582731,1.7745490803509556,,20240603,7,,,,\n'
    'daily,,,O3,Recent,0.31601098398428568,19600,14.7402514912512,1.0973622724771903,2.0357775142243324,1.8017192224749392,,20240604,8,,,,\n'
    'daily,,,O3,Recent,0.4021917947809146,19600,11.463332082312926,1.0706436347696284,1.7179547844624246,1.6266264547892819,,20240605,9,,,,\n'
    'daily,,,O3,Recent,0.41586276594734611,19600,10.387674777678724,1.0007531094881219,1.7403197858112851,1.6542266300140347,,20240606,10,,,,\n'
    'daily,,,O3,Recent,0.3706982774817118,19600,13.530245905571698,1.0250703021271983,1.834604559350532,1.6745856881314234,,20240607,11,,,,\n'
    'daily,,,O3,Recent,0.40472123289838358,19600,15.059537247666489,0.98192569160859322,1.6140124904235953,1.480189196322758,,20240608,12,,,,\n'
    'daily,,,O3,Recent,0.46791112640068933,19600,10.125690071938228,0.92742890046282001,1.5128940089326006,1.4926010164327583,,20240609,13,,,,\n'
    'daily,,,O3,Recent,0.43821572704542988,19600,10.546247226742285,1.0723440196327982,1.5877286864738909,1.5011177580600843,,20240610,14,,,,\n'
    'daily,,,O3,Recent,0.42827148847740287,19600,10.424070794639768,0.96244642642839795,1.7622186160078523,1.569261438862763,,20240611,15,,,,\n'
    'daily,,,O3,Recent,0.46406291496337593,19600,9.1442844406216661,0.97381620912028677,1.6712752417954373,1.4351501562731666,,20240612,16,,,,\n'
    'daily,,,O3,Recent,0.4911946529402782,19600,8.757234690739363,0.97070497341260598,1.4850849648175206,1.388759557485983,,20240613,17,,,,\n'
    'daily,,,O3,Recent,0.42000154777243481,19600,8.9201293366729431,1.1855963713843534,1.6608795801576297,1.6155601309250125,,20240614,18,,,,\n'
    'daily,,,O3,Recent,0.4433625937027218,19600,8.8787766827687573,1.118024132158818,1.5197892721910835,1.6021968425752271,,20240615,19,,,,\n'
    'daily,,,O3,Recent,0.4408551777314097,19600,8.2191829134581784,1.0181450715239491,1.5784139490021958,1.732687072228005,,20240616,20,,,,\n'
    'daily,,,O3,Recent,0.4321955997526214,19600,9.1731554891487956,0.96450061988561642,1.6886163646243082,1.6947749247387609,,20240617,21,,,,\n'
    'daily,,,O3,Recent,0.29803052635197308,19600,10.085413771560914,1.3474210650091336,2.0440014653511271,2.0758601523676963,,20240618,22,,,,\n'
    'daily,,,O3,Recent,0.24806224106859751,19600,12.219166931543548,1.2831220818936109,2.2769058871597827,2.2008793901857322,,20240619,23,,,,\n'
    'daily,,,O3,Recent,0.32988471313715451,19600,10.693967050386917,1.0455000676576014,2.0419869474856371,2.0665360709580241,,20240620,24,,,,\n'
    'daily,,,O3,Recent,0.38542410936841248,19600,9.4298221890249927,0.99124381867148204,1.9559639628167595,1.828735395638148,,20240621,25,,,,\n'
    'daily,,,O3,Recent,0.48666402107839729,19600,8.3610138303615482,0.91120416164190865,1.4519019593209319,1.5452259226053833,,20240623,27,,,,\n'
    'daily,,,O3,Recent,0.4109912073859569,19600,11.228603755128026,0.94196162677305595,1.836616237057318,1.6100332878520691,,20240624,28,,,,\n'
    'daily,,,O3,Recent,0.30795819543927999,19600,12.541029711289209,1.1617674032049607,2.0150798395836347,2.0010334301631199,,20240625,29,,,,\n'
    'daily,,,O3,Recent,0.39195079956455448,19600,9.7526604542712487,1.1734513175393275,1.7325960146397841,1.737625113282647,,20240626,30,,,,\n'
    'daily,,,O3,Recent,0.43910230584904358,19600,9.8475431939971632,1.1897071815357061,1.5123724219775925,1.4913464554516205,,20240627,31,,,,\n'
    'daily,,,O3,Recent,0.46349684293975119,19600,8.5530563583606423,1.07946976529672,1.5115928652143371,1.506243552708439,,20240628,32,,,,\n'
    'daily,,,O3,Recent,0.48968032961524049,19600,8.3300098225902861,0.98886577718823443,1.5176891049736618,1.3927956084741064,,20240629,33,,,,\n'
    'daily,,,O3,Recent,0.4445753098767069,19600,9.3002269196929692,1.0585039856714888,1.6850193721577289,1.4887204924521829,,20240630,34,,,,\n'
    'daily,,,O3,RecentBias,0.52185839321167082,19600,8.4143943438380617,0.84320051879304714,1.4389218423976682,1.3258058583763641,,20240528,1,,,,\n'
    'daily,,,O3,RecentBias,0.52157786570559439,19600,9.6439136807822319,0.82491983534438251,1.4577311684117149,1.2307502578450682,,20240529,2,,,,\n'
    'daily,,,O3,RecentBias,0.4775813009482382,19600,10.896742371709699,0.80564549499754046,1.6366518260694345,1.3749953481633783,,20240530,3,,,,\n'
    'daily,,,O3,RecentBias,0.37164307242070488,19600,12.322721749753001,1.009233487474267,1.9801962277826051,1.6685955297549673,,20240601,5,,,,\n'
    'daily,,,O3,RecentBias,0.4228216196075939,19600,10.846017011729884,0.92504447660042599,1.7408804552876711,1.6344799935549872,,20240602,6,,,,\n'
    'daily,,,O3,RecentBias,0.32762239008486421,19600,12.752388012094404,1.0951082511902619,2.1877245810470964,1.7522099182614099,,20240603,7,,,,\n'
    'daily,,,O3,RecentBias,0.3204212742618357,19600,14.715951915411654,1.0836852268380677,2.0350157343270241,1.7813111586436725,,20240604,8,,,,\n'
    'daily,,,O3,RecentBias,0.4133998216129236,19600,10.85917901370402,1.056644320194273,1.7027198328907145,1.6087971285190812,,20240605,9,,,,\n'
    'daily,,,O3,RecentBias,0.42059273382078638,19600,10.272975032399369,0.98747732475514938,1.7318121571129019,1.6442242112285581,,20240606,10,,,,\n'
    'daily,,,O3,RecentBias,0.37919526319985691,19600,12.980480589629469,1.0137286459921266,1.8382518949052564,1.6575093910301255,,20240607,11,,,,\n'
    'daily,,,O3,RecentBias,0.41709019851357809,19600,14.190696450517455,0.9617191358136864,1.6107631299639964,1.4713842536329658,,20240608,12,,,,\n'
    'daily,,,O3,RecentBias,0.47709693100793898,19600,9.6668989757821784,0.90856787549346718,1.5052996627867534,1.4786494771458008,,20240609,13,,,,\n'
    'daily,,,O3,RecentBias,0.4517224035601155,19600,10.023995227508999,1.0529153084173539,1.5661877157232851,1.4677566006404363,,20240610,14,,,,\n'
    'daily,,,O3,RecentBias,0.43712326522280581,19600,9.7719424949928246,0.94754948034949016,1.7701699630643963,1.5578870882796301,,20240611,15,,,,\n'
    'daily,,,O3,RecentBias,0.47064084151686092,19600,8.8454658754092463,0.96658688438450979,1.658384156287962,1.422178483756384,,20240612,16,,,,\n'
    'daily,,,O3,RecentBias,0.4972418330500642,19600,8.6909724789694547,0.95372316594153561,1.4666521134165775,1.3752550327190554,,20240613,17,,,,\n'
    'daily,,,O3,RecentBias,0.42867560068084642,19600,8.6113967347409481,1.1600498000571211,1.6561597604669014,1.5980462421937265,,20240614,18,,,,\n'
    'daily,,,O3,RecentBias,0.44618938422281262,19600,9.0952889012855511,1.1019558227956761,1.5014215590076951,1.5913320689168944,,20240615,19,,,,\n'
    'daily,,,O3,RecentBias,0.4447407264852829,19600,8.3847124681643415,1.0004586926923689,1.5617595749566389,1.7174778749231625,,20240616,20,,,,\n'
    'daily,,,O3,RecentBias,0.44038307313610908,19600,9.2740965342847144,0.94879648056859522,1.6607816764481882,1.6556728029330401,,20240617,21,,,,\n'
    'daily,,,O3,RecentBias,0.30297398601456987,19600,10.641374212756798,1.322365371444105,2.011824911999148,2.0392284126763349,,20240618,22,,,,\n'
    'daily,,,O3,RecentBias,0.2506461606521514,19600,12.882565517915214,1.2631455724048877,2.2508045839581272,2.1652357304473946,,20240619,23,,,,\n'
    'daily,,,O3,RecentBias,0.32690509449470972,19600,11.423408607892028,1.0319366574328273,2.0360418095829451,2.0497161335282672,,20240620,24,,,,\n'
    'daily,,,O3,RecentBias,0.38781951336078069,19600,9.5577848421263205,0.98293535912652941,1.9407863204808675,1.8184309834000325,,20240621,25,,,,\n'
    'daily,,,O3,RecentBias,0.49113965133991599,19600,8.3341923058331631,0.89069363631328335,1.4435005920442201,1.5373436759150481,,20240623,27,,,,\n'
    'daily,,,O3,RecentBias,0.42027594318793909,19600,10.953924260031272,0.92563418405183917,1.8235477412251559,1.5818066990838109,,20240624,28,,,,\n'
    'daily,,,O3,RecentBias,0.32296898348700198,19600,12.025753011828201,1.1341556702235656,1.9899271272322376,1.9656258276488443,,20240625,29,,,,\n'
    'daily,,,O3,RecentBias,0.4041752662786231,19600,9.5662712776688927,1.1430309981790705,1.7050220199230699,1.7031376545806618,,20240626,30,,,,\n'
    'daily,,,O3,RecentBias,0.44156788314473527,19600,10.215788281316076,1.1671006091280438,1.5085797446146332,1.4662532207188499,,20240627,31,,,,\n'
    'daily,,,O3,RecentBias,0.46674563251784962,19600,8.7059290318695197,1.0540832969090459,1.5251195701210449,1.4816462725257551,,20240628,32,,,,\n'
    'daily,,,O3,RecentBias,0.48989832051777998,19600,8.6972016162962831,0.98155959220566302,1.5108517580894625,1.3734647701786002,,20240629,33,,,,\n'
    'daily,,,O3,RecentBias,0.44479444899194082,19600,9.4811288738635433,1.0512526914236684,1.6901989648669844,1.4752906529736178,,20240630,34,,,,\n'
    'daily,,,O3,Drift,0.5143853832791534,19600,8.7878062464532753,0.84601046301837257,1.4616702104331072,1.3368934660092009,,20240528,1,,,,\n'
    'daily,,,O3,Drift,0.51346384004950407,19600,10.217018717155771,0.81929057933843685,1.4846017420397504,1.2360689851222169,,20240529,2,,,,\n'
    'daily,,,O3,Drift,0.46308082248981741,19600,11.741678167837351,0.84162148361623323,1.630575809837373,1.3948971895300433,,20240530,3,,,,\n'
    'daily,,,O3,Drift,0.37054514968107749,19600,13.317068559774588,1.034784866393095,1.9458464805205529,1.5968627529500921,,20240601,5,,,,\n'
    'daily,,,O3,Drift,0.39328403701338832,19600,11.647615553521128,1.0221972362822032,1.8029176076551177,1.6685701477806525,,20240602,6,,,,\n'
    'daily,,,O3,Drift,0.3226628711423043,19600,12.689129789201468,1.1559644112796763,2.1682274451615093,1.752515992259122,,20240603,7,,,,\n'
    'daily,,,O3,Drift,0.31794037278169829,19600,14.854440241470964,1.1002582943038277,1.9979186110505971,1.8027975849536679,,20240604,8,,,,\n'
    'daily,,,O3,Drift,0.41764415756898321,19600,11.716518214287619,1.0244238639716303,1.6648389529714349,1.5648841425844311,,20240605,9,,,,\n'
    'daily,,,O3,Drift,0.42653470076601852,19600,10.240200215037634,0.96774520357376081,1.7560988589638316,1.5983153215414103,,20240606,10,,,,\n'
    'daily,,,O3,Drift,0.38924253690572053,19600,13.411451571617205,1.0166003343060392,1.7771966901406104,1.5828327670811295,,20240607,11,,,,\n'
    'daily,,,O3,Drift,0.40848447671176058,19600,15.210948425392573,0.98921687734198838,1.6031385919753056,1.4374082504159786,,20240608,12,,,,\n'
    'daily,,,O3,Drift,0.4617297812837568,19600,10.199747123799192,0.93115285196535702,1.5367860771887796,1.5157295712333876,,20240609,13,,,,\n'
    'daily,,,O3,Drift,0.4474364759586002,19600,10.52927544486845,1.0133355711490255,1.590701170531168,1.4828488806383044,,20240610,14,,,,\n'
    'daily,,,O3,Drift,0.43697556196594961,19600,9.6610739392817955,0.95324388916731218,1.7742303766955383,1.5593251271025586,,20240611,15,,,,\n'
    'daily,,,O3,Drift,0.4730600260242398,19600,8.9687437784678856,0.94558693683500561,1.6733332145130064,1.401401896722255,,20240612,16,,,,\n'
    'daily,,,O3,Drift,0.4983941223491879,19600,8.4895605253919406,0.9285837997872004,1.489409320133273,1.389882212497912,,20240613,17,,,,\n'
    'daily,,,O3,Drift,0.42386888718882831,19600,8.8941129104726393,1.1178305097939392,1.7096142270316703,1.6159126795153147,,20240614,18,,,,\n'
    'daily,,,O3,Drift,0.44141243940132258,19600,9.085950679782556,1.1158154771652002,1.5129160781254689,1.6090184554107347,,20240615,19,,,,\n'
    'daily,,,O3,Drift,0.44854951648772151,19600,7.9609634848356183,0.99897581671184621,1.5867980258243259,1.7025330290077283,,20240616,20,,,,\n'
    'daily,,,O3,Drift,0.42755120056701251,19600,9.1985205094428313,0.95010418546712205,1.7311422450992322,1.712866638249811,,20240617,21,,,,\n'
    'daily,,,O3,Drift,0.29364507356559799,19600,10.442809548536353,1.3211027122569317,2.0613980022931431,2.0961006558054112,,20240618,22,,,,\n'
    'daily,,,O3,Drift,0.22860645836906021,19600,12.775944578839191,1.2921234477461214,2.3467866838114131,2.2539279944278867,,20240619,23,,,,\n'
    'daily,,,O3,Drift,0.3183361237797378,19600,11.092646637670777,1.065870433518735,2.0783625705901607,2.081200971139181,,20240620,24,,,,\n'
    'daily,,,O3,Drift,0.39379925342956712,19600,9.5521744014567815,0.95396181811768643,1.9734197655394856,1.7728046283706262,,20240621,25,,,,\n'
    'daily,,,O3,Drift,0.48597762364758951,19600,8.6257435650632566,0.88777010322512862,1.4866421638802854,1.5257683458241531,,20240623,27,,,,\n'
    'daily,,,O3,Drift,0.4063094825190296,19600,11.34896204898677,0.94473718633507642,1.8687808423742012,1.6113369265086144,,20240624,28,,,,\n'
    'daily,,,O3,Drift,0.30284569469722811,19600,12.55264188197498,1.1631506619390832,2.0400915300535494,2.0218702464258089,,20240625,29,,,,\n'
    'daily,,,O3,Drift,0.39940144253652388,19600,9.635890220166516,1.1017861707430725,1.7553836860727048,1.7396975979690179,,20240626,30,,,,\n'
    'daily,,,O3,Drift,0.44010548643248648,19600,10.214375372566906,1.152780008576981,1.5204182352867579,1.483827134775169,,20240627,31,,,,\n'
    'daily,,,O3,Drift,0.45839657214227408,19600,8.7601229285696398,1.0442595750874804,1.565357031012129,1.5254620998043291,,20240628,32,,,,\n'
    'daily,,,O3,Drift,0.49381916522756047,19600,8.4880682675439036,0.94907910623943181,1.505630661775073,1.3948431436470914,,20240629,33,,,,\n'
    'daily,,,O3,Drift,0.44175446958538689,19600,9.3225723464228327,1.0366937564196104,1.7138158933427017,1.5100316783320302,,20240630,34,,,,\n'
    'daily,,,O3,DriftBias,0.52293694333984875,19600,8.2708491703863203,0.83608492921831534,1.4629835384613148,1.3170489573278157,,20240528,1,,,,\n'
    'daily,,,O3,DriftBias,0.52637193036543928,19600,9.3798034029052957,0.80620156076657834,1.48158516826116,1.2127825398139991,,20240529,2,,,,\n'
    'daily,,,O3,DriftBias,0.47338348361993732,19600,11.064534916547563,0.8269427148281816,1.6374294881721216,1.3741018275224055,,20240530,3,,,,\n'
    'daily,,,O3,DriftBias,0.37842147974354629,19600,12.61620113653645,1.026373895112632,1.9424293732853672,1.5999530393096459,,20240601,5,,,,\n'
    'daily,,,O3,DriftBias,0.4082822789813273,19600,10.997304272303936,1.0031060986100588,1.7690257936223233,1.6425836267586591,,20240602,6,,,,\n'
    'daily,,,O3,DriftBias,0.33082861227600779,19600,12.408816139529875,1.138879293594725,2.1577319382605307,1.7328571057457169,,20240603,7,,,,\n'
    'daily,,,O3,DriftBias,0.32158485320107533,19600,14.865844012644535,1.0861485984787698,1.9981113478296699,1.7855384404865431,,20240604,8,,,,\n'
    'daily,,,O3,DriftBias,0.42762071489853293,19600,11.089465600084001,1.0154399679055046,1.6508060096065296,1.5532161704652951,,20240605,9,,,,\n'
    'daily,,,O3,DriftBias,0.4307600901600423,19600,10.098815379538784,0.95849360173919296,1.7459445293420539,1.5919390396618482,,20240606,10,,,,\n'
    'daily,,,O3,DriftBias,0.39780047564135651,19600,12.813598543761984,1.0044926605906914,1.7807503470360595,1.5700533646068977,,20240607,11,,,,\n'
    'daily,,,O3,DriftBias,0.42140726242429188,19600,14.258736157208654,0.96724406449704481,1.5989851651500635,1.4333319851881765,,20240608,12,,,,\n'
    'daily,,,O3,DriftBias,0.47154835115253141,19600,9.6820230655046018,0.9143014999347876,1.5279829712862414,1.5002564115599115,,20240609,13,,,,\n'
    'daily,,,O3,DriftBias,0.4605061258024824,19600,10.004670071616671,0.99648287389061196,1.5730758119063215,1.4476740223709921,,20240610,14,,,,\n'
    'daily,,,O3,DriftBias,0.44390780562912391,19600,9.0274296890256025,0.94388664739022154,1.7857176373253132,1.5541014869758365,,20240611,15,,,,\n'
    'daily,,,O3,DriftBias,0.47821138107664901,19600,8.6777181612913559,0.94265067972775518,1.6648968348176976,1.3918884035887089,,20240612,16,,,,\n'
    'daily,,,O3,DriftBias,0.50396493899777028,19600,8.3826647245572126,0.91499049208441796,1.476171292689294,1.3762249987435686,,20240613,17,,,,\n'
    'daily,,,O3,DriftBias,0.43172512657618989,19600,8.5568470635431364,1.0977394711166355,1.7062232147928404,1.6010259630800592,,20240614,18,,,,\n'
    'daily,,,O3,DriftBias,0.44245832116992911,19600,9.3227348411435145,1.1058047943159579,1.4979086664429999,1.6026130234355855,,20240615,19,,,,\n'
    'daily,,,O3,DriftBias,0.45091096967073119,19600,8.1682154423656428,0.98432781357617005,1.573643533329921,1.690816083288518,,20240616,20,,,,\n'
    'daily,,,O3,DriftBias,0.43502563599004879,19600,9.3607495620970944,0.93622618845596961,1.7040236995863216,1.6722692324525923,,20240617,21,,,,\n'
    'daily,,,O3,DriftBias,0.29665087958988878,19600,11.118395979615585,1.2984441618254741,2.0318212494663435,2.0613515667132987,,20240618,22,,,,\n'
    'daily,,,O3,DriftBias,0.22874394036555259,19600,13.61408051026886,1.2764907671544614,2.3209503827277844,2.2197313777673542,,20240619,23,,,,\n'
    'daily,,,O3,DriftBias,0.31264079487139762,19600,12.001155517965946,1.0557108777025801,2.0772592903598284,2.065101982351921,,20240620,24,,,,\n'
    'daily,,,O3,DriftBias,0.392928968398499,19600,9.8469656627154354,0.95083796982092961,1.963111668249244,1.7669995231844939,,20240621,25,,,,\n'
    'daily,,,O3,DriftBias,0.48844430382947679,19600,8.6981591075604516,0.87115510472390134,1.4764616248356592,1.5239419363215378,,20240623,27,,,,\n'
    'daily,,,O3,DriftBias,0.4157992477299603,19600,11.02376390116952,0.92775597297607315,1.8545380201374031,1.5871333148343023,,20240624,28,,,,\n'
    'daily,,,O3,DriftBias,0.31767019096342308,19600,11.995105533868289,1.136569678828206,2.0143196576704017,1.9909378108572084,,20240625,29,,,,\n'
    'daily,,,O3,DriftBias,0.40963636355820421,19600,9.5504746047345357,1.0758188021634689,1.7276092095972311,1.709166695205103,,20240626,30,,,,\n'
    'daily,,,O3,DriftBias,0.44025758258885173,19600,10.751127457403651,1.1351173166832791,1.5142463880447641,1.461038554882863,,20240627,31,,,,\n'
    'daily,,,O3,DriftBias,0.46027913580958951,19600,9.0487434080844444,1.023713862577708,1.5749701819546345,1.4992567365457043,,20240628,32,,,,\n'
    'daily,,,O3,DriftBias,0.4906505304751399,19600,9.0444493341700625,0.94981370130671683,1.5029570639676366,1.3768439110353976,,20240629,33,,,,\n'
    'daily,,,O3,DriftBias,0.43851531698832469,19600,9.7114003327841232,1.0359524599489294,1.7208529579286722,1.5003349834353661,,20240630,34,,,,\n'
    'daily,,,O3,MultiScale,0.51987399988338545,19600,8.5752704391147283,0.86148780577152662,1.439700556516702,1.3100404740945797,,20240528,1,,,,\n'
    'daily,,,O3,MultiScale,0.51734012765298232,19600,9.7156885160365043,0.83902003800867786,1.4694549720971617,1.2368350180126839,,20240529,2,,,,\n'
    'daily,,,O3,MultiScale,0.47199696655475931,19600,11.210868845324439,0.83263454719123764,1.6244461138083366,1.378002757664238,,20240530,3,,,,\n'
    'daily,,,O3,MultiScale,0.37913832738358799,19600,12.420937947126184,1.0365327238888289,1.9333936530636564,1.6068165841135025,,20240601,5,,,,\n'
    'daily,,,O3,MultiScale,0.40122341758489499,19600,10.851703452765296,1.0106781636120503,1.8081972604456789,1.6753192375168742,,20240602,6,,,,\n'
    'daily,,,O3,MultiScale,0.3296302462072061,19600,12.3020733980066,1.1487733964921978,2.1564545632646088,1.7426453475083377,,20240603,7,,,,\n'
    'daily,,,O3,MultiScale,0.31953070408932921,19600,14.576083417770551,1.1187536949703316,2.0034209349277785,1.7894457148203873,,20240604,8,,,,\n'
    'daily,,,O3,MultiScale,0.42505178253232251,19600,10.790610737369374,1.0676548852196808,1.6429309645832917,1.5526276450076355,,20240605,9,,,,\n'
    'daily,,,O3,MultiScale,0.42644045198459302,19600,10.280064122923706,1.0034385634618583,1.7083680574126243,1.5969537731142844,,20240606,10,,,,\n'
    'daily,,,O3,MultiScale,0.39101562409868829,19600,12.993460379247807,1.0445606199045665,1.7613677495965463,1.5865127679698099,,20240607,11,,,,\n'
    'daily,,,O3,MultiScale,0.41228552525370632,19600,14.256565007814793,1.0132313261647323,1.6145165233281635,1.4510857304956064,,20240608,12,,,,\n'
    'daily,,,O3,MultiScale,0.4618725765113682,19600,9.9842501606312961,0.94752671183731396,1.5489718018265386,1.5062244855369631,,20240609,13,,,,\n'
    'daily,,,O3,MultiScale,0.44273119614451029,19600,10.322489899449245,1.0617007357097741,1.5810161666254556,1.4978233418767095,,20240610,14,,,,\n'
    'daily,,,O3,MultiScale,0.43567513696172638,19600,9.7953339994967958,0.97066968608275817,1.7621266442732491,1.550954006441861,,20240611,15,,,,\n'
    'daily,,,O3,MultiScale,0.46726855319020472,19600,8.9751828107091836,0.97918708654207443,1.6664156033521853,1.4206617078750954,,20240612,16,,,,\n'
    'daily,,,O3,MultiScale,0.49014392898352821,19600,8.8162381686056968,0.96772705291693062,1.459859032685259,1.4153541429319849,,20240613,17,,,,\n'
    'daily,,,O3,MultiScale,0.42732803945074388,19600,8.7165269234556124,1.1645887546621532,1.6342012876252625,1.6131765587923483,,20240614,18,,,,\n'
    'daily,,,O3,MultiScale,0.43789797396882119,19600,9.3195687306098005,1.112811495644165,1.532131187928423,1.6071899717143496,,20240615,19,,,,\n'
    'daily,,,O3,MultiScale,0.43925876154724619,19600,8.4125525149840374,1.0270257723148697,1.5860250740248834,1.7151888045854062,,20240616,20,,,,\n'
    'daily,,,O3,MultiScale,0.41907922659813701,19600,9.5995354334119085,0.97931968053799001,1.7310527429079632,1.7211374202169147,,20240617,21,,,,\n'
    'daily,,,O3,MultiScale,0.2946107016426453,19600,11.057061693016983,1.3281050149587748,2.025283227731308,2.0585516461739779,,20240618,22,,,,\n'
    'daily,,,O3,MultiScale,0.2425301490371696,19600,13.498053796309399,1.2701174989561022,2.2660190145737409,2.1634320032825376,,20240619,23,,,,\n'
    'daily,,,O3,MultiScale,0.31652954211731149,19600,12.07104865083198,1.0594037637155589,2.0503540201027581,2.0441197271046625,,20240620,24,,,,\n'
    'daily,,,O3,MultiScale,0.38388236409573739,19600,9.9941738201403716,0.98313357339769258,1.9565917543233124,1.8028616949170524,,20240621,25,,,,\n'
    'daily,,,O3,MultiScale,0.48417900406972508,19600,8.76226373115672,0.90446908835154038,1.4471365995326486,1.5434699274394834,,20240623,27,,,,\n'
    'daily,,,O3,MultiScale,0.40993939623424208,19600,11.113230449543869,0.97238297480746605,1.8376463791824191,1.596759237876946,,20240624,28,,,,\n'
    'daily,,,O3,MultiScale,0.30319028904030948,19600,12.110982600410756,1.2011330640267071,2.0567982218953675,2.003929720721068,,20240625,29,,,,\n'
    'daily,,,O3,MultiScale,0.39107943999469041,19600,9.7929967637233446,1.1500384427391639,1.7602715977316381,1.7436703425653104,,20240626,30,,,,\n'
    'daily,,,O3,MultiScale,0.4280215533546739,19600,10.746628881221479,1.189926569171321,1.5566816502363929,1.4750960766546564,,20240627,31,,,,\n'
    'daily,,,O3,MultiScale,0.46055163959569889,19600,9.075139894121488,1.0666417905294936,1.5160850465736386,1.4973247543470771,,20240628,32,,,,\n'
    'daily,,,O3,MultiScale,0.4907776290216368,19600,8.863008041882793,0.96733780693223437,1.49957381631597,1.3756421640669594,,20240629,33,,,,\n'
    'daily,,,O3,MultiScale,0.4382934068546559,19600,9.7633776988426515,1.0753050802083139,1.7008237834328459,1.473745676801554,,20240630,34,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.52278625454266869,19600,8.4641734946483513,0.85835257719246549,1.4316365498045025,1.3043236940168039,,20240528,1,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.52196586287257984,19600,9.5896757963773442,0.82856893497617079,1.4594290159106329,1.2269627018003475,,20240529,2,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.4749722745220154,19600,11.087507992047447,0.822001956140896,1.6276250057333508,1.3715821227272551,,20240530,3,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.38191006618776602,19600,12.279329879278922,1.0295796849428194,1.9262378336236603,1.6079359495379533,,20240601,5,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.40720477452707349,19600,10.723204253970188,0.99931534377572362,1.7932884387712575,1.659060169620755,,20240602,6,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.33214690299834693,19600,12.21409862158152,1.1447572158466262,2.1581749905187104,1.7315906229065627,,20240603,7,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.32149443525767141,19600,14.582394933104791,1.1142741219800891,1.997517873901083,1.7816587208767447,,20240604,8,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.427488349477463,19600,10.750897524965829,1.0663160195683294,1.632771652176553,1.5448246031175072,,20240605,9,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.42944568186378929,19600,10.214393128354049,0.99973178667703444,1.6954932239132523,1.591049736629069,,20240606,10,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.39368849865819577,19600,12.901812528077391,1.0406195893373411,1.758596912116446,1.5778815896576572,,20240607,11,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.41613728297473129,19600,14.152339081835896,1.0048980541962025,1.6039549713835113,1.444159754515091,,20240608,12,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.46520654763384472,19600,9.8803111162176727,0.94191664157637922,1.5389228379990958,1.5004462393355691,,20240609,13,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.44662714633511608,19600,10.230034441130361,1.0534655606568191,1.5721949414526648,1.4881033999020381,,20240610,14,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.43670110380941818,19600,9.6797881945246118,0.96745828343529161,1.7711465550318948,1.5481121737113437,,20240611,15,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.46934129126209861,19600,8.9200949057788002,0.97338576646343877,1.6621358885038111,1.4170892136311302,,20240612,16,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.4935769444343836,19600,8.7055092318975635,0.96069103652789123,1.4504134871564021,1.4102787268597057,,20240613,17,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.43109570458308621,19600,8.629530336350312,1.1543959017572656,1.6241900560265681,1.606985185870756,,20240614,18,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.440147600151931,19600,9.2583898941961777,1.1095206952886585,1.5259527310565015,1.6015843988839549,,20240615,19,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.44101746890896742,19600,8.3851234863771644,1.0204853240870406,1.5807105726266619,1.7135156719831679,,20240616,20,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.4227776232137449,19600,9.5528277453281305,0.97224708868295562,1.7208012568211126,1.7092411514335604,,20240617,21,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.2965885983378867,19600,11.06597832997948,1.3215680323713896,2.023943151613171,2.0489540927234722,,20240618,22,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.2438841767948316,19600,13.533504968688923,1.2679775119151286,2.261309958793841,2.1550416682513731,,20240619,23,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.31778183343152638,19600,12.101572855505371,1.0585192670236441,2.039897492029505,2.0401964417406724,,20240620,24,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.3865943513222363,19600,9.9158175505300115,0.98090048723728918,1.9475383949135077,1.7959814858929417,,20240621,25,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.48801277370271501,19600,8.6875820488950932,0.89421741519526432,1.4386845293416599,1.5345566623366105,,20240623,27,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.41412138523052611,19600,10.97704314502344,0.96119672064918404,1.8316790618707068,1.58900873102203,,20240624,28,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.30775008763421008,19600,12.048089844205007,1.1896249168471422,2.040799469232276,1.9951372744117359,,20240625,29,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.39517313539991639,19600,9.7199356724950956,1.1446888436726483,1.7450974876760874,1.7327388058847106,,20240626,30,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.42977286118988961,19600,10.753602209140531,1.1853812332710143,1.5485268725225496,1.4708504578381356,,20240627,31,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.46410045002520889,19600,9.0117810236746632,1.057055690831197,1.5040577452664738,1.4919735830563736,,20240628,32,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.49395855225713098,19600,8.7939052818712593,0.96272543258402199,1.48794467746919,1.3684977258930269,,20240629,33,,,,\n'
    'daily,,,O3,AdaptiveFlow,0.439227465199518,19600,9.6859165106442688,1.0743308683641095,1.7032486493655377,1.471416637016671,,20240630,34,,,,\n'
    'daily,,,O3,WindFlow,0.5214746419388534,19600,8.5577906158448958,0.86156380531659638,1.4316365498045025,1.3043236940168039,,20240528,1,,,,\n'
    'daily,,,O3,WindFlow,0.51991561350016968,19600,9.6830905138281285,0.83789837653438859,1.4594290159106329,1.2269627018003475,,20240529,2,,,,\n'
    'daily,,,O3,WindFlow,0.4728532758180185,19600,11.190100443964679,0.83115193033126689,1.6276250057333508,1.3715821227272551,,20240530,3,,,,\n'
    'daily,,,O3,WindFlow,0.38001637498532859,19600,12.401087168485958,1.0353077440516378,1.9262378336236603,1.6079359495379533,,20240601,5,,,,\n'
    'daily,,,O3,WindFlow,0.40517163125009509,19600,10.833880391628044,1.0070977858287304,1.7932884387712575,1.659060169620755,,20240602,6,,,,\n'
    'daily,,,O3,WindFlow,0.33121924258733582,19600,12.286899317537159,1.1464918841809648,2.1581749905187104,1.7315906229065627,,20240603,7,,,,\n'
    'daily,,,O3,WindFlow,0.32100244928627458,19600,14.582808489900716,1.1183046415579461,1.997517873901083,1.7816587208767447,,20240604,8,,,,\n'
    'daily,,,O3,WindFlow,0.42702118166699171,19600,10.784180677799815,1.0674647748416459,1.632771652176553,1.5448246031175072,,20240605,9,,,,\n'
    'daily,,,O3,WindFlow,0.42853506199283381,19600,10.265536089377964,1.0030893913359424,1.6954932239132523,1.591049736629069,,20240606,10,,,,\n'
    'daily,,,O3,WindFlow,0.3923900413011715,19600,12.986441457244831,1.0444541009938324,1.758596912116446,1.5778815896576572,,20240607,11,,,,\n'
    'daily,,,O3,WindFlow,0.41443148418154663,19600,14.244653007343851,1.0114717017660344,1.6039549713835113,1.444159754515091,,20240608,12,,,,\n'
    'daily,,,O3,WindFlow,0.46386899889202338,19600,9.9648894650102804,0.94607819837398077,1.5389228379990958,1.5004462393355691,,20240609,13,,,,\n'
    'daily,,,O3,WindFlow,0.44502915448486879,19600,10.313577253477792,1.0598629151379189,1.5721949414526648,1.4881033999020381,,20240610,14,,,,\n'
    'daily,,,O3,WindFlow,0.43536018403774968,19600,9.7807162272934391,0.97031624570614117,1.7711465550318948,1.5481121737113437,,20240611,15,,,,\n'
    'daily,,,O3,WindFlow,0.46825575424792398,19600,8.9675302682263602,0.97849025617718643,1.6621358885038111,1.4170892136311302,,20240612,16,,,,\n'
    'daily,,,O3,WindFlow,0.49199922473789232,19600,8.7951437295036357,0.96642484839445442,1.4504134871564021,1.4102787268597057,,20240613,17,,,,\n'
    'daily,,,O3,WindFlow,0.42943306111912682,19600,8.7000561760505573,1.1623873717680573,1.6241900560265681,1.606985185870756,,20240614,18,,,,\n'
    'daily,,,O3,WindFlow,0.43948898333570091,19600,9.3020987329504621,1.1114019533225259,1.5259527310565015,1.6015843988839549,,20240615,19,,,,\n'
    'daily,,,O3,WindFlow,0.4403306481574355,19600,8.3975192138838768,1.0251495627759462,1.5807105726266619,1.7135156719831679,,20240616,20,,,,\n'
    'daily,,,O3,WindFlow,0.42161099762454157,19600,9.6018553476630863,0.97789177031467245,1.7208012568211126,1.7092411514335604,,20240617,21,,,,\n'
    'daily,,,O3,WindFlow,0.29608303983555218,19600,11.062770992410016,1.3260055406448117,2.023943151613171,2.0489540927234722,,20240618,22,,,,\n'
    'daily,,,O3,WindFlow,0.2440527847811553,19600,13.509063740321992,1.2685750545894423,2.261309958793841,2.1550416682513731,,20240619,23,,,,\n'
    'daily,,,O3,WindFlow,0.31791116168696387,19600,12.086328611422202,1.0586923322373949,2.039897492029505,2.0401964417406724,,20240620,24,,,,\n'
    'daily,,,O3,WindFlow,0.3856271863002802,19600,10.001676260535096,0.98189811156239559,1.9475383949135077,1.7959814858929417,,20240621,25,,,,\n'
    'daily,,,O3,WindFlow,0.48649528712736612,19600,8.7480700849291146,0.90182720504700598,1.4386845293416599,1.5345566623366105,,20240623,27,,,,\n'
    'daily,,,O3,WindFlow,0.41192245701921859,19600,11.093830800912592,0.96985098537621195,1.8316790618707068,1.58900873102203,,20240624,28,,,,\n'
    'daily,,,O3,WindFlow,0.30622796625569881,19600,12.099188750219524,1.1980376007939495,2.040799469232276,1.9951372744117359,,20240625,29,,,,\n'
    'daily,,,O3,WindFlow,0.39414870195422502,19600,9.7904018592927553,1.1474130462184129,1.7450974876760874,1.7327388058847106,,20240626,30,,,,\n'
    'daily,,,O3,WindFlow,0.4292832358864449,19600,10.749398618541402,1.189768250770507,1.5485268725225496,1.4708504578381356,,20240627,31,,,,\n'
    'daily,,,O3,WindFlow,0.46242654055821369,19600,9.0734567746254395,1.0658609369485983,1.5040577452664738,1.4919735830563736,,20240628,32,,,,\n'
    'daily,,,O3,WindFlow,0.4927699052628538,19600,8.858555197519312,0.96727982037953797,1.48794467746919,1.3684977258930269,,20240629,33,,,,\n'
    'daily,,,O3,WindFlow,0.43843847932416041,19600,9.7534189740806756,1.0753514374144582,1.7032486493655377,1.471416637016671,,20240630,34,,,,\n'
    'daily,,,O3,Taper,0.52022329135485756,19600,8.5629809885225665,0.86180405753519862,1.4382837788976468,1.3089203146644333,,20240528,1,,,,\n'
    'daily,,,O3,Taper,0.51769037709874743,19600,9.7167764931538336,0.83858784188257862,1.468462425113408,1.2350314608271984,,20240529,2,,,,\n'
    'daily,,,O3,Taper,0.47206418664010258,19600,11.231350568003716,0.83249057954428207,1.6229407800948681,1.3770745838395997,,20240530,3,,,,\n'
    'daily,,,O3,Taper,0.3787252504486277,19600,12.471989699956204,1.0366673598830489,1.9342222220001923,1.6052813394017942,,20240601,5,,,,\n'
    'daily,,,O3,Taper,0.40034874727375791,19600,10.907017776852618,1.0107842479891931,1.8110284001478281,1.6757551433355824,,20240602,6,,,,\n'
    'daily,,,O3,Taper,0.32924135631953561,19600,12.312109032437496,1.1503251030035599,2.1564049827305154,1.7435367328106703,,20240603,7,,,,\n'
    'daily,,,O3,Taper,0.31909791142863841,19600,14.556811063001698,1.1224142980420573,2.0027245989165161,1.791488773438874,,20240604,8,,,,\n'
    'daily,,,O3,Taper,0.42328353577222771,19600,10.89429743729532,1.0707226092194959,1.6463374365177379,1.553101002253259,,20240605,9,,,,\n'
    'daily,,,O3,Taper,0.42586961975792592,19600,10.275575542044658,1.0069537484955593,1.7105431795656505,1.5968407906982534,,20240606,10,,,,\n'
    'daily,,,O3,Taper,0.38896504892715728,19600,13.128087312750305,1.0491932878237789,1.7606407343526569,1.5884374927805671,,20240607,11,,,,\n'
    'daily,,,O3,Taper,0.4092055505093522,19600,14.495950005626424,1.0195230204059833,1.6148790334932324,1.450456342041557,,20240608,12,,,,\n'
    'daily,,,O3,Taper,0.45959751981779101,19600,10.104112580416247,0.95374588187399956,1.5499886408460746,1.508275803217803,,20240609,13,,,,\n'
    'daily,,,O3,Taper,0.43913781701803711,19600,10.467948601554752,1.0680452037066237,1.5852705206415625,1.5061253321331556,,20240610,14,,,,\n'
    'daily,,,O3,Taper,0.43371855207241028,19600,9.9725202331813136,0.97609787523176117,1.7560549189263477,1.5519533240208727,,20240611,15,,,,\n'
    'daily,,,O3,Taper,0.46626537064082479,19600,9.03283055789427,0.98114181577386439,1.6671552374777727,1.4217406116058169,,20240612,16,,,,\n'
    'daily,,,O3,Taper,0.48879512317217588,19600,8.8008909590599398,0.97448714522463842,1.4631800969740212,1.4184578778116821,,20240613,17,,,,\n'
    'daily,,,O3,Taper,0.42493851242573971,19600,8.797119037586878,1.174459268509181,1.6325709455214037,1.6177893626293882,,20240614,18,,,,\n'
    'daily,,,O3,Taper,0.43806266217895379,19600,9.1724345343731883,1.118741915569637,1.536968059345019,1.6081608150733535,,20240615,19,,,,\n'
    'daily,,,O3,Taper,0.43895204271977972,19600,8.2688565887367496,1.0352391437806552,1.590325204165395,1.717923810467922,,20240616,20,,,,\n'
    'daily,,,O3,Taper,0.41627248001746547,19600,9.5054095854230756,0.98751057800989983,1.7410235135268921,1.736269701319304,,20240617,21,,,,\n'
    'daily,,,O3,Taper,0.29301920913834928,19600,10.754380385743769,1.3417057837277979,2.0381190035776169,2.0730018054773387,,20240618,22,,,,\n'
    'daily,,,O3,Taper,0.24191172836240121,19600,13.123223417241611,1.2812871877829954,2.2775425186595979,2.1791764006191428,,20240619,23,,,,\n'
    'daily,,,O3,Taper,0.31907102793490771,19600,11.653498173492553,1.0675904324887389,2.046816190337013,2.0517522960266321,,20240620,24,,,,\n'
    'daily,,,O3,Taper,0.38428854944204438,19600,9.8381260509375039,0.98718825070671223,1.9582690750246616,1.8069091357529961,,20240621,25,,,,\n'
    'daily,,,O3,Taper,0.4831257436663412,19600,8.6860777171737631,0.91575235535427724,1.4499599970373154,1.5449296786021678,,20240623,27,,,,\n'
    'daily,,,O3,Taper,0.40537062152139708,19600,11.234984672699952,0.9837542790668492,1.8434246853109371,1.6088879773897884,,20240624,28,,,,\n'
    'daily,,,O3,Taper,0.29512531243275808,19600,12.368030709495336,1.2191963053996493,2.0709814527125738,2.0207984907609262,,20240625,29,,,,\n'
    'daily,,,O3,Taper,0.38479466008414848,19600,9.8368665663669166,1.16945760559037,1.7757609323559731,1.7608333637424776,,20240626,30,,,,\n'
    'daily,,,O3,Taper,0.42726995689994041,19600,10.454835778589995,1.2060085445896676,1.5578793420740591,1.488202019276718,,20240627,31,,,,\n'
    'daily,,,O3,Taper,0.45929773454580503,19600,8.8999913292443509,1.0834476034665217,1.5083712041027146,1.5112393778730351,,20240628,32,,,,\n'
    'daily,,,O3,Taper,0.49242801536686781,19600,8.5375659964050747,0.97021470085977435,1.5007575743224102,1.3848196965261337,,20240629,33,,,,\n'
    'daily,,,O3,Taper,0.43993704888478302,19600,9.5261610120223779,1.0803632832283689,1.6944537832931006,1.4794012828436836,,20240630,34,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.53799580369683642,19600,8.4905177146201236,0.80617542090700223,1.3879998538446596,1.2619138453882672,,20240528,1,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.53082598257151425,19600,10.204752942835682,0.81868779588369922,1.3926834195141178,1.1643716078202024,,20240529,2,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.48934381219281498,19600,10.9854816512585,0.77361725991959129,1.6140159723896934,1.3194925270587083,,20240530,3,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.3771993472650591,19600,13.312114542961693,0.98385003750923739,1.924186970877132,1.610079548797634,,20240601,5,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.48021932097917208,19600,10.83953339363997,0.84884990015990891,1.4596075370688959,1.4520363572176975,,20240602,6,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.38204576854541727,19600,11.309291668138975,1.0257434947479382,1.9931242309221999,1.638384570291505,,20240603,7,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.34590079362629822,19600,13.311495676091766,1.1500395720678347,1.9595373226364703,1.6759328187719047,,20240604,8,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.43459100662662031,19600,9.8759357837154553,1.121668192768482,1.5734368674282448,1.5478482334466714,,20240605,9,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.44013316434596478,19600,9.7343005446820872,1.0168873376598948,1.6143989245025356,1.587209358906958,,20240606,10,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.39651481924209908,19600,12.740309192961606,1.04203627184592,1.7748337125929063,1.5534842871709142,,20240607,11,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.44081157833434959,19600,14.454120403663998,0.96480150455838398,1.4822352264931264,1.3487079676510032,,20240608,12,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.49205781378263358,19600,9.6352110290791941,0.9381215660284512,1.4174881394970591,1.3949169420524092,,20240609,13,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.47399728764767107,19600,9.9707973546906317,1.017986282571872,1.468741803567948,1.397413502659016,,20240610,14,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.4509926541439776,19600,10.06125589594998,0.87203940698704019,1.7654585897813464,1.4982585900941283,,20240611,15,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.48703060414297811,19600,9.2550808953987165,0.91721515053012759,1.6035338080436812,1.3442699714591704,,20240612,16,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.51677246844645142,19600,8.2696405901612557,0.93207606489740158,1.3747053962851563,1.3404183380518091,,20240613,17,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.46505257073378292,19600,8.2539239239646189,1.0852612837736335,1.4925101947783721,1.5269003037795634,,20240614,18,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.48349007029813451,19600,8.1814552693928579,1.0635418054163877,1.3736481056766645,1.4935316175287146,,20240615,19,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.50276886249949382,19600,7.0711453446074826,0.94143150570300482,1.3747322551088172,1.5465346599395571,,20240616,20,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.47895471665331291,19600,8.3443468705846673,0.91855419555032403,1.5436848221399035,1.5319851209193034,,20240617,21,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.34366640071566951,19600,9.1970088686373792,1.2888659091734986,1.9082246896634325,1.9329637021215185,,20240618,22,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.27212795940498408,19600,11.815934210526985,1.26590179645547,2.1875289349124154,2.120204683932295,,20240619,23,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.31817452081590059,19600,11.194236319460364,1.156100529276229,1.9628179672890076,2.0730636760588319,,20240620,24,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.3972217273990129,19600,8.7349673473798628,1.0071532760803259,1.8758498330389159,1.8337992896209323,,20240621,25,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.52113681465804074,19600,7.7409606164000477,0.93077891461388618,1.2900107841559636,1.4148381668602898,,20240623,27,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.45437200510935438,19600,10.245715868905281,0.91821960235213895,1.7194962156195042,1.4440088065380403,,20240624,28,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.35256944987660999,19600,12.252563723827246,1.1124496700194515,1.8558041501905609,1.8262902203146325,,20240625,29,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.43133017056193701,19600,9.3982512654834593,1.1407003267625373,1.5884653623033851,1.5834442637205659,,20240626,30,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.47899216210060691,19600,8.1738073237117259,1.2050748856126126,1.4471115212947687,1.3313763095052222,,20240627,31,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.47949596134365952,19600,8.4162927065581759,1.1035776303134166,1.4404036962285445,1.4150362387755726,,20240628,32,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.50987679287303789,19600,8.18960970001954,0.96389636478958862,1.4090428450427623,1.3456512739414943,,20240629,33,,,,\n'
    'daily,,,O3,MixPhysicsNewBias,0.46877285877011232,19600,8.2098373851626754,1.0423806562935172,1.648049594249404,1.4215424565723236,,20240630,34,,,,\n'
    'daily,,,O3,MixRecentBias,0.53755156302860374,19600,8.4994873852286901,0.80350084400659849,1.390929209952924,1.2653047657690959,,20240528,1,,,,\n'
    'daily,,,O3,MixRecentBias,0.53080037256853696,19600,10.20968265690637,0.81812068356816758,1.3916142615277165,1.1655864904975228,,20240529,2,,,,\n'
    'daily,,,O3,MixRecentBias,0.48878057222695509,19600,10.95749320666831,0.77034401295553978,1.6218614065638253,1.3236918063203391,,20240530,3,,,,\n'
    'daily,,,O3,MixRecentBias,0.37328799984835448,19600,13.305263678265909,0.9840131476087568,1.9416861549671445,1.6294966399406563,,20240601,5,,,,\n'
    'daily,,,O3,MixRecentBias,0.4829757622712425,19600,10.850004801058532,0.83512251114587632,1.4501377414739931,1.4494164185237115,,20240602,6,,,,\n'
    'daily,,,O3,MixRecentBias,0.38142879339063662,19600,11.354864355750994,1.016692717611366,2.0010510759356235,1.6427538329532003,,20240603,7,,,,\n'
    'daily,,,O3,MixRecentBias,0.34493871248951607,19600,13.396707901291398,1.1417493367085736,1.9696763495622771,1.6774272565927979,,20240604,8,,,,\n'
    'daily,,,O3,MixRecentBias,0.43099756188489657,19600,9.9428734405883556,1.1208606792939244,1.5897717068312642,1.5604254780873343,,20240605,9,,,,\n'
    'daily,,,O3,MixRecentBias,0.43745823813786949,19600,9.7450535299304022,1.0165005900731738,1.6288603891384656,1.5977769039274794,,20240606,10,,,,\n'
    'daily,,,O3,MixRecentBias,0.39347626158000021,19600,12.737557144799563,1.035941380306755,1.795702456669334,1.5689746153609041,,20240607,11,,,,\n'
    'daily,,,O3,MixRecentBias,0.44008932558024783,19600,14.482604187374656,0.95830252909382097,1.486501510739537,1.3556375644004477,,20240608,12,,,,\n'
    'daily,,,O3,MixRecentBias,0.49380186717028929,19600,9.5967433352670763,0.9331988947110248,1.4119705407786085,1.3927978851361915,,20240609,13,,,,\n'
    'daily,,,O3,MixRecentBias,0.47534868836440919,19600,9.9446470355547056,1.0171263961534198,1.465947369016201,1.3913293276428502,,20240610,14,,,,\n'
    'daily,,,O3,MixRecentBias,0.45139289429688428,19600,10.085895421230449,0.86733928030185292,1.7651862893228336,1.4978515393569587,,20240611,15,,,,\n'
    'daily,,,O3,MixRecentBias,0.48782818812075718,19600,9.2467669266691921,0.9139440552323288,1.5999912188240637,1.3443627873103785,,20240612,16,,,,\n'
    'daily,,,O3,MixRecentBias,0.51799323986532586,19600,8.2528372654850806,0.9289395971084724,1.3767700971229331,1.3331683708838069,,20240613,17,,,,\n'
    'daily,,,O3,MixRecentBias,0.46534801251132829,19600,8.2343540238965414,1.084349852919392,1.4959080146589416,1.5243164084812999,,20240614,18,,,,\n'
    'daily,,,O3,MixRecentBias,0.48473204278305682,19600,8.143277760959041,1.062547811937109,1.3686494935274156,1.4912053297935479,,20240615,19,,,,\n'
    'daily,,,O3,MixRecentBias,0.50277478046122692,19600,7.08020036025954,0.93888234333520304,1.3744382269404871,1.5485458979598787,,20240616,20,,,,\n'
    'daily,,,O3,MixRecentBias,0.48194574518050859,19600,8.314881722430929,0.91535627960284038,1.5325445788791292,1.5213524970183825,,20240617,21,,,,\n'
    'daily,,,O3,MixRecentBias,0.34492884771154347,19600,9.1180160436200097,1.2887573415790723,1.9065558827699831,1.9303351806057425,,20240618,22,,,,\n'
    'daily,,,O3,MixRecentBias,0.27313453597193438,19600,11.747785093603367,1.2657944718197405,2.182272516166686,2.1216159812605557,,20240619,23,,,,\n'
    'daily,,,O3,MixRecentBias,0.31923699282263041,19600,11.104501531880896,1.1511974501998348,1.9631573434252845,2.0762701170769491,,20240620,24,,,,\n'
    'daily,,,O3,MixRecentBias,0.39659533021929477,19600,8.6869482638252862,1.0140124310441982,1.8733002153579297,1.838037086933564,,20240621,25,,,,\n'
    'daily,,,O3,MixRecentBias,0.52172988786165286,19600,7.7079159181831942,0.93088268353700443,1.2877678372224126,1.4142497888783014,,20240623,27,,,,\n'
    'daily,,,O3,MixRecentBias,0.45470373006507647,19600,10.218674018293738,0.91246730882857163,1.7222741916481408,1.4471136448026221,,20240624,28,,,,\n'
    'daily,,,O3,MixRecentBias,0.35434280309896438,19600,12.264503727172784,1.1046232137168566,1.8482950584124327,1.8242649613913999,,20240625,29,,,,\n'
    'daily,,,O3,MixRecentBias,0.43197945466307058,19600,9.4158968341851885,1.1428740685230871,1.5847648721211498,1.577257703820391,,20240626,30,,,,\n'
    'daily,,,O3,MixRecentBias,0.48053656163797881,19600,8.1089952910040548,1.202017270653583,1.4416200759225886,1.3311873954471549,,20240627,31,,,,\n'
    'daily,,,O3,MixRecentBias,0.48019976978924078,19600,8.3892089915812154,1.1020388447224694,1.4420761620755302,1.4116401827228404,,20240628,32,,,,\n'
    'daily,,,O3,MixRecentBias,0.50933456223844442,19600,8.1772527143948643,0.96718754307543842,1.4121446090872922,1.3454511498437354,,20240629,33,,,,\n'
    'daily,,,O3,MixRecentBias,0.4691639331970181,19600,8.208544002674115,1.039104034022583,1.6470000529959778,1.4225137384078923,,20240630,34,,,,\n'
    'daily,,,O3,MixDriftBias,0.53863185856671769,19600,8.4504376830615886,0.80123319472666754,1.3920009363803902,1.2617782392601415,,20240528,1,,,,\n'
    'daily,,,O3,MixDriftBias,0.53263462206497636,19600,10.168713627742951,0.81116850891224312,1.39410599011542,1.1587271831727819,,20240529,2,,,,\n'
    'daily,,,O3,MixDriftBias,0.48974762597129901,19600,10.972273784156243,0.77062963342093938,1.6162699429505931,1.318470489423448,,20240530,3,,,,\n'
    'daily,,,O3,MixDriftBias,0.37710979817620138,19600,13.384640881613372,0.97891685734539924,1.9248594879909044,1.6093194091548102,,20240601,5,,,,\n'
    'daily,,,O3,MixDriftBias,0.4819706530957536,19600,10.888658685626275,0.84317482741666561,1.4515811617650429,1.4453625255179741,,20240602,6,,,,\n'
    'daily,,,O3,MixDriftBias,0.38264523507658499,19600,11.349226441474658,1.0192900094685979,1.9953856309027691,1.6348517840356491,,20240603,7,,,,\n'
    'daily,,,O3,MixDriftBias,0.34753662399442981,19600,13.379796359276638,1.1371341546604876,1.9553179857187089,1.6729855787091628,,20240604,8,,,,\n'
    'daily,,,O3,MixDriftBias,0.43655191208105759,19600,9.8918430551200274,1.1045679287180807,1.5757549417025962,1.5456428699756193,,20240605,9,,,,\n'
    'daily,,,O3,MixDriftBias,0.44188596302624411,19600,9.6841213559531365,1.0053567275281896,1.6189682844874489,1.5847966286536983,,20240606,10,,,,\n'
    'daily,,,O3,MixDriftBias,0.39949736233253269,19600,12.692965567062618,1.0290894020914141,1.7733757960130307,1.5467000873428971,,20240607,11,,,,\n'
    'daily,,,O3,MixDriftBias,0.44437836130057101,19600,14.451073475324897,0.94910669170931117,1.4742942845584264,1.3412309498057382,,20240608,12,,,,\n'
    'daily,,,O3,MixDriftBias,0.49485223793682132,19600,9.5766722563186342,0.92855928332201165,1.4106664528599799,1.3913857635575293,,20240609,13,,,,\n'
    'daily,,,O3,MixDriftBias,0.47864773172095931,19600,9.9048970981475364,1.0035494213479306,1.4617172934730147,1.3840753847422731,,20240610,14,,,,\n'
    'daily,,,O3,MixDriftBias,0.45454862976351751,19600,9.8846172439295792,0.86280593729950794,1.7640291908546717,1.4935534753604454,,20240611,15,,,,\n'
    'daily,,,O3,MixDriftBias,0.49114454655325818,19600,9.1635693238719114,0.90573578483759964,1.5980236461936981,1.3333499044080954,,20240612,16,,,,\n'
    'daily,,,O3,MixDriftBias,0.52103191927146064,19600,8.1635288548015872,0.91887720454416577,1.3719410344314371,1.3290824553953429,,20240613,17,,,,\n'
    'daily,,,O3,MixDriftBias,0.46780671438899479,19600,8.1927656116082286,1.069656716064441,1.4987889202689899,1.5198151613812514,,20240614,18,,,,\n'
    'daily,,,O3,MixDriftBias,0.48618872841325489,19600,8.1252651618092564,1.0592707848150384,1.3598765207415251,1.4906766436044745,,20240615,19,,,,\n'
    'daily,,,O3,MixDriftBias,0.5062300793810609,19600,7.000218151543173,0.92961029179069599,1.3700224611028669,1.5390921811969007,,20240616,20,,,,\n'
    'daily,,,O3,MixDriftBias,0.4828860590080461,19600,8.2863937186577719,0.90810070110098795,1.5361497961164026,1.5203690488696455,,20240617,21,,,,\n'
    'daily,,,O3,MixDriftBias,0.34570930572425101,19600,9.1532016205367199,1.2805862407610582,1.90322305396348,1.9317797954473492,,20240618,22,,,,\n'
    'daily,,,O3,MixDriftBias,0.27113850698600372,19600,11.7860048185634,1.2650581927217648,2.1909049288037061,2.1291514020325417,,20240619,23,,,,\n'
    'daily,,,O3,MixDriftBias,0.3193485827676793,19600,11.12919517013532,1.1538998250218619,1.9591311987570601,2.0737128619581502,,20240620,24,,,,\n'
    'daily,,,O3,MixDriftBias,0.40123762451371481,19600,8.6520061519113494,0.99669987654250836,1.8685793395644208,1.8233294392887391,,20240621,25,,,,\n'
    'daily,,,O3,MixDriftBias,0.52513727394744714,19600,7.6777770662277156,0.9150627675318288,1.2868256689616466,1.4050038105298843,,20240623,27,,,,\n'
    'daily,,,O3,MixDriftBias,0.45681645978827151,19600,10.201541345941576,0.90406849905262598,1.7218726992283304,1.439684775954482,,20240624,28,,,,\n'
    'daily,,,O3,MixDriftBias,0.35707640545237068,19600,12.198629758551348,1.0938593024241965,1.8444362079898675,1.820750168952338,,20240625,29,,,,\n'
    'daily,,,O3,MixDriftBias,0.43668858969066349,19600,9.3040198188254681,1.1218491459181508,1.5793281593988866,1.5726358224517845,,20240626,30,,,,\n'
    'daily,,,O3,MixDriftBias,0.48381537254240697,19600,8.1075226436423851,1.1902694530292373,1.4316874676722462,1.3235484703963121,,20240627,31,,,,\n'
    'daily,,,O3,MixDriftBias,0.48330066185668269,19600,8.3389384931627593,1.0871333788051782,1.4387094268638709,1.4075767145509357,,20240628,32,,,,\n'
    'daily,,,O3,MixDriftBias,0.51328598248609225,19600,8.1292950738026075,0.95353960156629458,1.4023760860015342,1.337815051696402,,20240629,33,,,,\n'
    'daily,,,O3,MixDriftBias,0.47114695859568562,19600,8.1295399525196306,1.0316473829862418,1.646098957587858,1.4206982578958121,,20240630,34,,,,\n'
    'daily,,,O3,MixMultiScale,0.53798271607777581,19600,8.4932785385143479,0.80599733805466212,1.3879629872861641,1.2620046226014665,,20240528,1,,,,\n'
    'daily,,,O3,MixMultiScale,0.53075281968105215,19600,10.209226915551421,0.81872814554591067,1.3925942447510362,1.1646429940947147,,20240529,2,,,,\n'
    'daily,,,O3,MixMultiScale,0.4893186644869022,19600,10.987823435847236,0.77363501307220051,1.6137419465451861,1.3197054474818195,,20240530,3,,,,\n'
    'daily,,,O3,MixMultiScale,0.37725305554785948,19600,13.313612768742823,0.98376149380982836,1.9236423438837136,1.6100218556651855,,20240601,5,,,,\n'
    'daily,,,O3,MixMultiScale,0.48003969941272639,19600,10.840844649250768,0.84927435530440121,1.4601797851195952,1.4525531142961343,,20240602,6,,,,\n'
    'daily,,,O3,MixMultiScale,0.38194000073706008,19600,11.311047769038664,1.0260061753742002,1.993310234118044,1.6387126662146081,,20240603,7,,,,\n'
    'daily,,,O3,MixMultiScale,0.34596720367276018,19600,13.309712572676569,1.1498537427467532,1.9591318175578627,1.6760290492414098,,20240604,8,,,,\n'
    'daily,,,O3,MixMultiScale,0.43462060746567938,19600,9.8764808094826382,1.1215154734502095,1.573102029751096,1.5479716219501749,,20240605,9,,,,\n'
    'daily,,,O3,MixMultiScale,0.44020496265424702,19600,9.7354873609042407,1.0166877753154531,1.6137878868837376,1.5871902416981041,,20240606,10,,,,\n'
    'daily,,,O3,MixMultiScale,0.39660019497382187,19600,12.74110595174308,1.041782988461434,1.7741145767751263,1.5535224848834244,,20240607,11,,,,\n'
    'daily,,,O3,MixMultiScale,0.44076863248941128,19600,14.456231642162651,0.96491725308619203,1.4821771528715719,1.3488210617260403,,20240608,12,,,,\n'
    'daily,,,O3,MixMultiScale,0.49199676821037092,19600,9.6371602593576338,0.93814339528964685,1.4176598644077525,1.3951102871237284,,20240609,13,,,,\n'
    'daily,,,O3,MixMultiScale,0.47384056228822108,19600,9.9719883303305519,1.0182288368635397,1.4693217499918023,1.3979268402913725,,20240610,14,,,,\n'
    'daily,,,O3,MixMultiScale,0.45102906692393341,19600,10.062924481566466,0.87198462574894209,1.765054316586707,1.498187741561372,,20240611,15,,,,\n'
    'daily,,,O3,MixMultiScale,0.48701467604922111,19600,9.2561738307492636,0.91725436756090017,1.6033231104171499,1.3444376372228919,,20240612,16,,,,\n'
    'daily,,,O3,MixMultiScale,0.51669636644181449,19600,8.2721296649211951,0.93216646215520604,1.3749930816017712,1.3405332504674301,,20240613,17,,,,\n'
    'daily,,,O3,MixMultiScale,0.46494486462613049,19600,8.2562182771099568,1.0855888766437221,1.4927367649949519,1.5271030325743946,,20240614,18,,,,\n'
    'daily,,,O3,MixMultiScale,0.48338299551021241,19600,8.1836795860801708,1.063684321533948,1.3740036203531496,1.4938213920363368,,20240615,19,,,,\n'
    'daily,,,O3,MixMultiScale,0.50271447857318263,19600,7.0721405937850381,0.94159434746283877,1.3747380689198236,1.5467377191814495,,20240616,20,,,,\n'
    'daily,,,O3,MixMultiScale,0.47885203576258001,19600,8.3421718903990154,0.91862258727433277,1.5442787090916483,1.5324894292571702,,20240617,21,,,,\n'
    'daily,,,O3,MixMultiScale,0.34351425663686758,19600,9.1956759781294366,1.2891770277319434,1.909034897585828,1.9333977132783005,,20240618,22,,,,\n'
    'daily,,,O3,MixMultiScale,0.27203611595128918,19600,11.813830027104723,1.2660766866414492,2.1882225878762451,2.1204272309443599,,20240619,23,,,,\n'
    'daily,,,O3,MixMultiScale,0.31815723599037637,19600,11.192217845762279,1.1561785214805409,1.9630275989026049,2.0731318910748509,,20240620,24,,,,\n'
    'daily,,,O3,MixMultiScale,0.39716977094252898,19600,8.7327189729915187,1.0073077311574019,1.8761528536343357,1.8340258484704619,,20240621,25,,,,\n'
    'daily,,,O3,MixMultiScale,0.52105417574520807,19600,7.7427393729169367,0.93110882756612456,1.2901323225324304,1.4149535609370469,,20240623,27,,,,\n'
    'daily,,,O3,MixMultiScale,0.45430274122579711,19600,10.247269920638548,0.91840624578070384,1.7194956855955561,1.4442710056854624,,20240624,28,,,,\n'
    'daily,,,O3,MixMultiScale,0.35243332826739138,19600,12.252892576448414,1.1128079390306882,1.8562100883872592,1.8267211734623747,,20240625,29,,,,\n'
    'daily,,,O3,MixMultiScale,0.43117366183872641,19600,9.3975531371046941,1.1410372709161776,1.5891360261764149,1.5839451589519511,,20240626,30,,,,\n'
    'daily,,,O3,MixMultiScale,0.47902163121560692,19600,8.1721245998254926,1.2049588437539542,1.4470381119829181,1.331443284419016,,20240627,31,,,,\n'
    'daily,,,O3,MixMultiScale,0.47946408354961118,19600,8.415888935851612,1.1035937306356298,1.4405041703360772,1.4152410590165996,,20240628,32,,,,\n'
    'daily,,,O3,MixMultiScale,0.50986723457793293,19600,8.1905127989299782,0.96376616743407884,1.409010919927653,1.3458132477343885,,20240629,33,,,,\n'
    'daily,,,O3,MixMultiScale,0.46881773759255041,19600,8.21097976610654,1.0422614332339359,1.6477214182227062,1.4214500410030977,,20240630,34,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.53846997342979186,19600,8.4717030467986465,0.80523678947729815,1.3869694096246279,1.2612553373598674,,20240528,1,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.53152869452954576,19600,10.187247508152836,0.81666263465233069,1.3910663541150678,1.1632525824960749,,20240529,2,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.48960893637812197,19600,10.967783971888316,0.77182211234545672,1.6153543361840403,1.3195066410276799,,20240530,3,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.37727754522342971,19600,13.288168928905678,0.98297285907790155,1.9252179191805063,1.6114806445724841,,20240601,5,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.48105145630246787,19600,10.817908637994448,0.84710133500230445,1.4578396404737897,1.4500136917188498,,20240602,6,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.38205144301894439,19600,11.292831232894056,1.0252658951355325,1.9966404223713157,1.6374335555099784,,20240603,7,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.34579592287068811,19600,13.303813023824912,1.1495796571392209,1.9622758850137048,1.6757707306277512,,20240604,8,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.4347743416257363,19600,9.8689522716093272,1.1212177995174579,1.5738209483429055,1.5470480142939722,,20240605,9,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.4403047413895475,19600,9.7242427539087313,1.0161828844311442,1.615043674327389,1.5868139779470001,,20240606,10,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.39662580540172587,19600,12.725950749669858,1.0412285715104552,1.777084929797164,1.5528007672531272,,20240607,11,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.44130889153404362,19600,14.439106931523328,0.96309367562690396,1.4815328461962554,1.3480650576366648,,20240608,12,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.49250533903788318,19600,9.6170466038888165,0.93720271743645556,1.416259737128498,1.3945617403370882,,20240609,13,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.4745685862945766,19600,9.9558568047122566,1.0165070689520397,1.4678075902370566,1.3960964361667749,,20240610,14,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.45109154034038218,19600,10.045181403732162,0.87115923766080505,1.7675244467824065,1.498035144210413,,20240611,15,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.4873712393277973,19600,9.2441627005153801,0.91570625241499481,1.6034832666579182,1.3438889710627753,,20240612,16,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.51729743973970932,19600,8.2498735029007211,0.93088184448975841,1.3734393598828241,1.33985605997145,,20240613,17,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.46561392988838168,19600,8.2394530781255337,1.0833643742597916,1.4914124587074984,1.526177288088455,,20240614,18,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.48377857557685872,19600,8.1709516262407913,1.0628465977390176,1.3728478259422414,1.4933138173899008,,20240615,19,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.50297140005353236,19600,7.0640572681249543,0.94033504178251837,1.3741494036264656,1.5469896497327198,,20240616,20,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.4794924402956755,19600,8.3299494302323982,0.91750430815027495,1.5425964714746732,1.5305899019476832,,20240617,21,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.34384907092495942,19600,9.1861798671511448,1.2879988938581362,1.9098289452705215,1.9319554171216944,,20240618,22,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.27210100255242509,19600,11.815640357263067,1.266039829157493,2.1888257586590001,2.1193033710953868,,20240619,23,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.31828916680115982,19600,11.194635741303358,1.1560644855115287,1.9620076833108224,2.0727416317574097,,20240620,24,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.39758166423539032,19600,8.7200493032821154,1.0063862202640297,1.875286363888516,1.8332363920128325,,20240621,25,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.52173189132697606,19600,7.7276881927428471,0.92907380205064405,1.2888230448610554,1.4136112380499832,,20240623,27,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.45478165563554218,19600,10.222266214812723,0.91663206810858,1.7198246436353457,1.4438670970208449,,20240624,28,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.35309970891057868,19600,12.238125092598215,1.1111879338016577,1.8537920869564684,1.8258950468653281,,20240625,29,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.43180700800418698,19600,9.3810686135473045,1.140336495248458,1.5871583628615435,1.5822637167652975,,20240626,30,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.47930864918316379,19600,8.1684163981707982,1.2040959683507189,1.4462323404116133,1.3308522806591627,,20240627,31,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.48004580206934327,19600,8.4072766311380462,1.1018328715926169,1.4385433842756146,1.4144011212563872,,20240628,32,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.5104053149422052,19600,8.1775328242527809,0.9626466707817668,1.4071503926678786,1.3449707605713064,,20240629,33,,,,\n'
    'daily,,,O3,MixAdaptiveFlow,0.46886749308756331,19600,8.1940804581512516,1.0423411534942877,1.6487884325397415,1.4215132039907119,,20240630,34,,,,\n'
    'daily,,,O3,MixWindFlow,0.53817090599810602,19600,8.490517358186402,0.80617517603723732,1.3869694096246279,1.2612553373598674,,20240528,1,,,,\n'
    'daily,,,O3,MixWindFlow,0.53111094681718674,19600,10.204749143452821,0.81868831348686055,1.3910663541150678,1.1632525824960749,,20240529,2,,,,\n'
    'daily,,,O3,MixWindFlow,0.48921710081338809,19600,10.985474023987411,0.77361839232643947,1.6153543361840403,1.3195066410276799,,20240530,3,,,,\n'
    'daily,,,O3,MixWindFlow,0.37693536494861168,19600,13.312100088960614,0.98385069994631957,1.9252179191805063,1.6114806445724841,,20240601,5,,,,\n'
    'daily,,,O3,MixWindFlow,0.48062668274986953,19600,10.839522719603044,0.84885015447057499,1.4578396404737897,1.4500136917188498,,20240602,6,,,,\n'
    'daily,,,O3,MixWindFlow,0.3818313458035637,19600,11.309277208157043,1.0257447907694028,1.9966404223713157,1.6374335555099784,,20240603,7,,,,\n'
    'daily,,,O3,MixWindFlow,0.34566447403290329,19600,13.311484527719845,1.1500407982376752,1.9622758850137048,1.6757707306277512,,20240604,8,,,,\n'
    'daily,,,O3,MixWindFlow,0.43465072192172222,19600,9.8759280534053353,1.1216709220672298,1.5738209483429055,1.5470480142939722,,20240605,9,,,,\n'
    'daily,,,O3,MixWindFlow,0.44012014238554681,19600,9.7343023873538765,1.0168886103017325,1.615043674327389,1.5868139779470001,,20240606,10,,,,\n'
    'daily,,,O3,MixWindFlow,0.39638646053317689,19600,12.740306916182249,1.0420366524254303,1.777084929797164,1.5528007672531272,,20240607,11,,,,\n'
    'daily,,,O3,MixWindFlow,0.44095403458040638,19600,14.45411058196877,0.96480325070582218,1.4815328461962554,1.3480650576366648,,20240608,12,,,,\n'
    'daily,,,O3,MixWindFlow,0.49221491486293312,19600,9.6352075127975514,0.93812291382845681,1.416259737128498,1.3945617403370882,,20240609,13,,,,\n'
    'daily,,,O3,MixWindFlow,0.47424224000489751,19600,9.9707931816414206,1.0179866015792145,1.4678075902370566,1.3960964361667749,,20240610,14,,,,\n'
    'daily,,,O3,MixWindFlow,0.45082641058246131,19600,10.06125531847403,0.87204043699386613,1.7675244467824065,1.498035144210413,,20240611,15,,,,\n'
    'daily,,,O3,MixWindFlow,0.48708089404858401,19600,9.2550750991066106,0.91721608190372639,1.6034832666579182,1.3438889710627753,,20240612,16,,,,\n'
    'daily,,,O3,MixWindFlow,0.51695790962792898,19600,8.2696427962720325,0.93207671588327201,1.3734393598828241,1.33985605997145,,20240613,17,,,,\n'
    'daily,,,O3,MixWindFlow,0.46524153287283843,19600,8.2539231461211298,1.085262296745646,1.4914124587074984,1.526177288088455,,20240614,18,,,,\n'
    'daily,,,O3,MixWindFlow,0.48359089371573649,19600,8.1814544941063385,1.0635416955338772,1.3728478259422414,1.4933138173899008,,20240615,19,,,,\n'
    'daily,,,O3,MixWindFlow,0.50276872454641497,19600,7.0711416661215898,0.9414323833142908,1.3741494036264656,1.5469896497327198,,20240616,20,,,,\n'
    'daily,,,O3,MixWindFlow,0.47922330692669202,19600,8.3443470886689894,0.91855508700104882,1.5425964714746732,1.5305899019476832,,20240617,21,,,,\n'
    'daily,,,O3,MixWindFlow,0.34363730252096247,19600,9.1970095703834804,1.2888663498200232,1.9098289452705215,1.9319554171216944,,20240618,22,,,,\n'
    'daily,,,O3,MixWindFlow,0.27211490907814712,19600,11.815930710880474,1.2659013050868009,2.1888257586590001,2.1193033710953868,,20240619,23,,,,\n'
    'daily,,,O3,MixWindFlow,0.31828870427170358,19600,11.19423282038492,1.156101118373313,1.9620076833108224,2.0727416317574097,,20240620,24,,,,\n'
    'daily,,,O3,MixWindFlow,0.39734160278836639,19600,8.7349637149124284,1.0071547598143851,1.875286363888516,1.8332363920128325,,20240621,25,,,,\n'
    'daily,,,O3,MixWindFlow,0.52139447321040355,19600,7.7409618400884606,0.93078020175980236,1.2888230448610554,1.4136112380499832,,20240623,27,,,,\n'
    'daily,,,O3,MixWindFlow,0.45435822530791431,19600,10.245706246107634,0.91822109464649804,1.7198246436353457,1.4438670970208449,,20240624,28,,,,\n'
    'daily,,,O3,MixWindFlow,0.3528045925597445,19600,12.252552305882302,1.1124509460601095,1.8537920869564684,1.8258950468653281,,20240625,29,,,,\n'
    'daily,,,O3,MixWindFlow,0.43159363438984949,19600,9.3982469583889205,1.1407002077996122,1.5871583628615435,1.5822637167652975,,20240626,30,,,,\n'
    'daily,,,O3,MixWindFlow,0.47913701862562008,19600,8.1738051823421198,1.2050749366365561,1.4462323404116133,1.3308522806591627,,20240627,31,,,,\n'
    'daily,,,O3,MixWindFlow,0.47974563927142733,19600,8.4162931232075167,1.1035781954154289,1.4385433842756146,1.4144011212563872,,20240628,32,,,,\n'
    'daily,,,O3,MixWindFlow,0.51013504158202339,19600,8.1896006951215554,0.96389659389411564,1.4071503926678786,1.3449707605713064,,20240629,33,,,,\n'
    'daily,,,O3,MixWindFlow,0.46870737113414429,19600,8.2098368098228303,1.0423807652097361,1.6487884325397415,1.4215132039907119,,20240630,34,,,,\n'
    'daily,,,O3,MixTaper,0.53797473411105678,19600,8.4925155663619734,0.80620103893318606,1.3879524873246258,1.2619363580546736,,20240528,1,,,,\n'
    'daily,,,O3,MixTaper,0.53074123524102479,19600,10.212225449078398,0.81877564659706548,1.3925873061535317,1.1644502871191671,,20240529,2,,,,\n'
    'daily,,,O3,MixTaper,0.48922843325497328,19600,10.997020679136492,0.77382996658701797,1.6135872371281854,1.3196253617254472,,20240530,3,,,,\n'
    'daily,,,O3,MixTaper,0.3770265511531436,19600,13.33645295965564,0.98396035083664157,1.9236367786531408,1.609836410189319,,20240601,5,,,,\n'
    'daily,,,O3,MixTaper,0.47968250072189178,19600,10.867166574766678,0.84957849287158849,1.4606738374161337,1.4526764559405747,,20240602,6,,,,\n'
    'daily,,,O3,MixTaper,0.38164624376352368,19600,11.336138258885052,1.0265659144427084,1.9927562300949899,1.6389663610536649,,20240603,7,,,,\n'
    'daily,,,O3,MixTaper,0.345662751332662,19600,13.329719371453651,1.1509108728081801,1.9584904031404384,1.6763559009768738,,20240604,8,,,,\n'
    'daily,,,O3,MixTaper,0.4341945834747083,19600,9.9098244861253306,1.1223966287549594,1.5730908091322231,1.5479023230245601,,20240605,9,,,,\n'
    'daily,,,O3,MixTaper,0.44004557826149748,19600,9.7490806794595599,1.0175234756190843,1.6132738081196198,1.5869582206332755,,20240606,10,,,,\n'
    'daily,,,O3,MixTaper,0.39606741543093821,19600,12.788079732364016,1.0428700986581989,1.7731789058835568,1.5537351883327719,,20240607,11,,,,\n'
    'daily,,,O3,MixTaper,0.44012507914247029,19600,14.510594433337063,0.9664048715093192,1.4819125096903598,1.3484224963044735,,20240608,12,,,,\n'
    'daily,,,O3,MixTaper,0.4915221481322265,19600,9.6706421896107759,0.93947297199466595,1.4175188796454741,1.3950832900161152,,20240609,13,,,,\n'
    'daily,,,O3,MixTaper,0.47307359168132179,19600,10.014081366190698,1.0196301938361776,1.4699197617665936,1.3989835127739896,,20240610,14,,,,\n'
    'daily,,,O3,MixTaper,0.45066304654653078,19600,10.105539584577564,0.87302765247314962,1.7635624465952942,1.497844943717012,,20240611,15,,,,\n'
    'daily,,,O3,MixTaper,0.48680969606559288,19600,9.2795285042121005,0.91774723457507101,1.6030820938669526,1.343916474302709,,20240612,16,,,,\n'
    'daily,,,O3,MixTaper,0.51646319585856237,19600,8.2796816035019081,0.933519382731134,1.3751994529753158,1.3403301599533588,,20240613,17,,,,\n'
    'daily,,,O3,MixTaper,0.46448135724243239,19600,8.2784327451859543,1.0878201029451409,1.4922301578872079,1.5272843913275369,,20240614,18,,,,\n'
    'daily,,,O3,MixTaper,0.48336676820941887,19600,8.1648801744085073,1.0650910595858831,1.3748724217017261,1.493405077914741,,20240615,19,,,,\n'
    'daily,,,O3,MixTaper,0.50250212776692682,19600,7.0674088147296397,0.94334099141356242,1.3754029953567546,1.5466164459047094,,20240616,20,,,,\n'
    'daily,,,O3,MixTaper,0.47804624866953621,19600,8.3551132772747856,0.92028066661142904,1.5462355013476408,1.534951073476962,,20240617,21,,,,\n'
    'daily,,,O3,MixTaper,0.34290715812902878,19600,9.1690992759929415,1.2918599534049298,1.911913457638684,1.9356986809194183,,20240618,22,,,,\n'
    'daily,,,O3,MixTaper,0.27164593876960508,19600,11.76898850010584,1.2684183805179658,2.1911602563352681,2.1227192026388022,,20240619,23,,,,\n'
    'daily,,,O3,MixTaper,0.31839699637716129,19600,11.134095315553054,1.1582785210563116,1.9629288484045555,2.0738685760905486,,20240620,24,,,,\n'
    'daily,,,O3,MixTaper,0.39691709148011167,19600,8.7317740659838918,1.0091327367628775,1.876574095671137,1.8340403833324239,,20240621,25,,,,\n'
    'daily,,,O3,MixTaper,0.52061926480410636,19600,7.7586499485700902,0.93388902247953276,1.2899407698914054,1.4146144993884739,,20240623,27,,,,\n'
    'daily,,,O3,MixTaper,0.4531697850479528,19600,10.310059443448084,0.92091364554441624,1.7192438273390371,1.446224662696818,,20240624,28,,,,\n'
    'daily,,,O3,MixTaper,0.3507594049141351,19600,12.328601336130911,1.1165304405663916,1.8578233959115733,1.8294461649857483,,20240625,29,,,,\n'
    'daily,,,O3,MixTaper,0.42980968670106068,19600,9.4442667299897121,1.1448205093691586,1.5905035261262337,1.5865990498027154,,20240626,30,,,,\n'
    'daily,,,O3,MixTaper,0.47894019277780048,19600,8.1376656986024241,1.2079326136221074,1.4459187971330667,1.3328265899343543,,20240627,31,,,,\n'
    'daily,,,O3,MixTaper,0.47923272259650418,19600,8.3913024376582293,1.1073032216752103,1.4385652863958596,1.4169588737315644,,20240628,32,,,,\n'
    'daily,,,O3,MixTaper,0.51015185746715852,19600,8.1485125118257802,0.96417839321097698,1.4088723825145226,1.3465852523202655,,20240629,33,,,,\n'
    'daily,,,O3,MixTaper,0.46901977983994841,19600,8.201119987740471,1.0430379158180072,1.6460149292918018,1.4211201159925515,,20240630,34,,,,\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240528,,1404187.7657711508,14440.675276749767,40171.830930428332,33344.701855963373\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240529,,1802452.8848406589,13455.919008147903,41746.687828644805,29506.574443540499\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240530,,2409483.536084835,13243.469431629404,51923.597922054403,36872.255379946058\n'
    'daily_sse,,,O3,AdaptiveFlow,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240601,,2955326.0687693814,20776.672821880318,72723.686956990161,50674.977149203783\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240602,,2253747.3456583414,19573.170663589612,63031.315122760818,53948.620669872151\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240603,,2924010.420699303,25685.194031365216,91291.298078128559,58768.759272623589\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240604,,4167866.3429067899,24335.49365072418,78205.522068464343,62216.432834452695\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240605,,2265403.2328094011,22285.785130325705,52252.488055766466,46775.067866184407\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240606,,2044943.0088191014,19589.487447732128,56344.066537780789,49616.209582777628\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240607,,3262552.623589233,21224.626942367293,60616.196746387781,48798.322095219279\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240608,,3925658.54915029,19792.473946815357,50424.362384427259,40877.708972597335\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240609,,1913362.7359637888,17389.256409699097,46418.356625777757,44126.242775870538\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240610,,2051210.6514675813,21751.877874803729,48447.219905015227,43403.253884480095\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240611,,1836486.6700208178,18345.120391676202,61484.418340266428,46974.365526907932\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240612,,1559534.6253107779,18570.605066930875,54148.835952306144,39359.580052038014\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240613,,1485403.4633384123,18089.374446234666,41232.505961013812,38982.167313687431\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240614,,1459588.3589894103,26119.546000681818,51704.669426673485,50615.067197118136\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240615,,1680068.5552858969,24128.308996167034,45639.222053408397,50275.422700270181\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240616,,1378081.7992831611,20411.249814869836,48973.459922508715,57548.26477938579\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240617,,1788627.7514654547,18527.182268468663,58038.676523351569,57261.504149576896\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240618,,2400135.1774316793,34232.224458045326,80288.379266852382,82284.972332130012\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240619,,3589852.8320555445,31512.232626160607,100225.04550290793,91026.410001233366\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240620,,2870382.0853112461,21961.075557741337,81559.162848569991,81583.069809469511\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240621,,1927139.3788298212,18858.449010902084,74340.953673380704,63220.770154336293\n'
    'daily_sse,,,O3,AdaptiveFlow,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240623,,1479292.0043831714,15672.645798514597,40568.338229353853,46155.337338464895\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240624,,2361711.3336710399,18108.42306142021,65758.944439632833,49488.995446379144\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240625,,2845066.7903231038,27738.065878557973,81631.304482927284,78019.225777442945\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240626,,1851752.1297578767,25682.125957041048,59689.15873327048,58846.721880607947\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240627,,2266543.2252596705,27540.521896545775,46999.535308519582,42402.660958724002\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240628,,1591759.0654857904,21900.387976964936,44338.918141482951,43629.309381746243\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240629,,1515722.2940878556,18166.06906746421,43393.995518894284,36706.606105177969\n'
    'daily_sse,,,O3,AdaptiveFlow,,19600,,,,,,20240630,,1838812.7815629554,22622.06156851165,56860.696846684361,42435.311625913215\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240528,,1435421.2924030018,14548.926936379634,40547.660853972222,33564.852060228026\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240529,,1837739.9412201701,13760.644312219694,42279.374605035242,29853.021040022835\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240530,,2454279.6197419628,13539.945213350467,51702.66846760318,37095.564082628989\n'
    'daily_sse,,,O3,Bias325,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240601,,3014224.4740235871,21008.497647908516,73332.183195160353,50551.719261183076\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240602,,2300510.1010660115,19879.220624334179,63834.801008649411,54779.970670287512\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240603,,2958970.7388501838,25763.095353659242,90972.246692309884,59346.669394199002\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240604,,4168102.7476811605,24511.863318068907,78671.354900312581,62669.018455380501\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240605,,2279451.636672046,22333.828492343426,52896.511470305282,47162.793795438498\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240606,,2065472.1315654998,19721.291209409945,57266.713344894466,49939.766631216466\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240607,,3305494.1697599669,21381.333634023555,60914.991918018932,49275.669336792373\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240608,,3977038.7302727532,20052.270068080164,51034.79731179597,41190.434168541098\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240609,,1946260.8321775079,17543.25356579572,46905.375737152855,44347.970374230616\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240610,,2084849.5649637084,22016.86421813909,48762.644241830742,43756.626390114187\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240611,,1874983.2344092876,18453.666886952706,60931.680452731351,47129.931798051606\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240612,,1576165.3425864961,18765.88635610044,54414.648764585159,39453.656245381404\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240613,,1516149.2431634918,18305.948956847191,41619.595905846807,39165.792013938088\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240614,,1483543.1583421347,26482.430280098692,52226.26979874682,50871.536926193672\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240615,,1695969.2004161484,24210.200316242874,45857.99352530625,50466.416882318968\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240616,,1382159.2473719581,20598.259870770387,49205.185873699389,57523.921786379273\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240617,,1807034.2719019454,18742.937363203615,58461.70422064684,57818.622370451507\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240618,,2398744.0797979659,34462.49759888649,80051.234498631398,82803.909242083071\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240619,,3576898.1415456152,31541.940314881504,100308.39909991922,91546.097325138719\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240620,,2863155.0503443452,21968.25730502978,82268.379240961469,81786.838563227095\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240621,,1960657.1492028059,18896.82846920006,74851.185595020288,63556.420689023835\n'
    'daily_sse,,,O3,Bias325,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240623,,1499963.1121323009,15940.529232152738,40952.847600117915,46597.031449467737\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240624,,2412232.4040498277,18435.974303170107,66066.24442966585,49818.43247172777\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240625,,2869251.2209033989,28131.764221156016,82648.269839108063,78471.820221693037\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240626,,1878698.5839022845,25804.511293191463,60427.167088522889,59354.595447788204\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240627,,2264771.5849418761,27744.750414613634,47433.264585831159,42568.931295398521\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240628,,1613621.3096835276,22266.766923493706,44987.230854752088,43836.321658380904\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240629,,1538090.4036749299,18338.352917904034,44024.320091427042,37005.358851524848\n'
    'daily_sse,,,O3,Bias325,,19600,,,,,,20240630,,1864531.9610055566,22665.061993407086,56742.55839173017,42546.415722229518\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240528,,1513620.5570539827,14028.380589316599,41875.004159724253,35030.769133378642\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240529,,2045994.4407475428,13156.246246497147,43199.229716362242,29946.184105228916\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240530,,2702193.321462539,13883.203745014034,52112.03844388538,38136.468119432742\n'
    'daily_sse,,,O3,Drift,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240601,,3475948.5745044407,20987.282506437041,74211.843104782762,49979.424774483203\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240602,,2659072.1824194943,20479.788921314292,63710.03323985992,54568.876226069049\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240603,,3155874.6902211225,26190.57291484531,92143.720977451463,60197.721141229908\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240604,,4324826.1397936661,23727.138958013617,78236.904017093999,63701.550993369652\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240605,,2690625.2616883782,20569.107360261492,54325.099290888065,47997.702642359312\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240606,,2055289.3287035134,18356.003269184381,60444.110768099788,50070.392594650839\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240607,,3525393.8518535378,20256.134298338555,61905.190278756127,49105.047543495835\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240608,,4534909.859199075,19179.580596197397,50373.045563579079,40496.392575932921\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240609,,2039082.8912332181,16994.094420975074,46289.544362009045,45029.748208982441\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240610,,2172966.5713206297,20126.24000321608,49594.472193012865,43097.279735081334\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240611,,1829392.4533412899,17810.00867980253,61698.711219948171,47657.299099462842\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240612,,1576591.9532906057,17525.039240214974,54880.863317131647,38492.974612279984\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240613,,1412623.7031201445,16900.450315253809,43479.46640883723,37862.742266514579\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240614,,1550462.7914990271,24491.082953073084,57286.503783274275,51179.006241239927\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240615,,1618068.1952066456,24402.865909995511,44862.73516523077,50743.231641105995\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240616,,1242188.0162949669,19559.872574551962,49351.388305295564,56812.926811299731\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240617,,1658410.4794277523,17692.880079546012,58738.328066237314,57504.6775604125\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240618,,2137424.5168332397,34208.122576119378,83287.489787619939,86115.304001650351\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240619,,3199205.2936787819,32723.826882620342,107945.19169056558,99571.751519688027\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240620,,2411717.4647932844,22267.163708568551,84663.983106670756,84895.390652505128\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240621,,1788383.1015985857,17836.845748357475,76329.957192029615,61599.590507297362\n'
    'daily_sse,,,O3,Drift,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240623,,1458307.6601845117,15447.46082113497,43318.056499162478,45628.193284331865\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240624,,2524459.2159511028,17493.555684388615,68449.900001766669,50889.571138312458\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240625,,3088348.8370554019,26517.221462438814,81574.679639526148,80124.00215028644\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240626,,1819867.4545679737,23793.082214397378,60394.888952471847,59320.335554631747\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240627,,2044935.8993333459,26046.474264224991,45308.763559772997,43154.162131543715\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240628,,1504099.17298357,21373.369979172945,48026.71563696651,45609.8785116129\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240629,,1412127.1371239212,17654.722538044676,44431.704317673983,38133.512949434284\n'
    'daily_sse,,,O3,Drift,,19600,,,,,,20240630,,1703442.9610240385,21064.785314148292,57568.432358971237,44691.835123498458\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240528,,1340776.1415858895,13701.144973773509,41950.288342652275,33998.511937566407\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240529,,1724421.9527922135,12739.234749016352,43023.854371904417,28828.493182002807\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240530,,2399509.0852222033,13403.151370705105,52551.036443218087,37007.854315048127\n'
    'daily_sse,,,O3,DriftBias,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240601,,3119703.2099038549,20647.490102346055,73951.424655954979,50173.054668724995\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240602,,2370437.7446496347,19721.948163346387,61337.26426662138,52882.38702955436\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240603,,3017982.8724992801,25422.102489424869,91253.819500835132,58854.757479095089\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240604,,4331469.0368822766,23122.488048356958,78251.999583183744,62487.691440120281\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240605,,2410334.4469907489,20209.91923703067,53413.145434519451,47284.617255019592\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240606,,1998927.0125721777,18006.715697669424,59747.117070973152,49671.690157589954\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240607,,3218090.8297577049,19776.507901539095,62153.00724999333,48315.324327183414\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240608,,3984906.513297433,18336.997173973359,50112.369744051524,40267.035363364186\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240609,,1837334.7845628774,16384.565762546845,45760.746426598322,44115.078288361816\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240610,,1961831.0955412854,19462.371111962944,48501.523196092756,41076.89747093703\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240611,,1597291.9410898972,17462.071261182427,62500.234612992099,47338.53606368191\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240612,,1475934.7327413943,17416.3699582275,54328.876823485363,37972.125229676596\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240613,,1377273.7305329826,16409.268971855752,42710.00103305564,37122.30684446801\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240614,,1435104.7807097896,23618.626150369644,57059.474110481104,50240.369035346048\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240615,,1703502.3443981155,23966.963165390305,43977.115310898924,50340.022656552843\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240616,,1307706.972852851,18990.464393761009,48536.537811825132,56033.636939139644\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240617,,1717423.1943402987,17179.781728635542,56912.456667536353,54811.093961828628\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240618,,2422927.0915268264,33044.761931021698,80914.632759746266,83283.73751919088\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240619,,3632726.487545616,31936.802101159468,105581.48931005129,96573.264833121124\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240620,,2822943.5818215278,21844.69896307122,84574.120723969871,83587.065471271126\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240621,,1900469.5621488588,17720.219759122469,75534.625471520048,61196.831372710883\n'
    'daily_sse,,,O3,DriftBias,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240623,,1482896.2484645618,14874.659843135625,42726.803020402156,45519.020895476984\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240624,,2381858.0627550734,16870.330449698493,67410.500855449107,49372.246317515659\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240625,,2820098.1126652975,25319.096442700364,79526.680192237109,77691.133987337511\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240626,,1787746.6774433141,22684.767463733442,58498.818189269325,57256.515523167342\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240627,,2265500.1354646394,25254.429923631251,44941.665624649628,41838.81971354242\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240628,,1604843.2424008928,20540.605419701897,48618.409051305796,44056.306936722445\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240629,,1603320.4496640381,17682.062916923314,44274.046748152286,37155.703444962899\n'
    'daily_sse,,,O3,DriftBias,,19600,,,,,,20240630,,1848501.4099025521,21034.670985775068,58042.164095112472,44119.699225392003\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240528,,1406687.1492574946,12708.76322770305,37704.209207395994,31178.994509967557\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240529,,2034088.2311303765,13071.982033210807,37927.285790399357,26521.868785402232\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240530,,2357728.7909786003,11675.903712865897,51143.644776000205,34125.516404036549\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240601,,3460878.4962694324,18938.218577002175,72646.695112140907,50898.649409502264\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240602,,2293732.0870798994,14064.581166548654,41655.809779800271,41209.778240973697\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240603,,2499549.5301906983,20602.93505227007,78136.83033444325,52551.297514716964\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240604,,3469032.2430687095,25902.054406923242,75470.322318563878,55040.867815921876\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240605,,1908965.8912103369,24639.735337513499,48547.482597882059,46909.80814720608\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240606,,1853393.5838841512,20239.502030371688,51123.974971704301,49352.380571916437\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240607,,3174216.520667247,21249.475987342186,61897.404615158957,47259.328366525253\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240608,,4086361.0560073145,18179.968789437786,43020.815657424188,35618.676232570513\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240609,,1812756.6734748401,17215.63909797743,39313.516203021129,38118.127973195362\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240610,,1942734.0604323321,20252.417776097544,42227.398790367304,38202.071077920373\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240611,,1977751.1209041739,14874.800980290398,61233.196331479659,43984.542148474451\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240612,,1674909.0630585563,16434.95163795352,50394.708294458193,35398.336304265635\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240613,,1333984.0911517232,16984.203764652833,36972.179235396696,35186.199524267467\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240614,,1330616.3057219936,23004.096001340076,43596.497990963733,45652.655447669807\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240615,,1308583.2293759941,22141.000650377944,36940.338602609045,43707.72868126892\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240616,,978057.73971204017,17330.907819760963,37010.417036344639,46906.268737051192\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240617,,1360005.9272004897,16499.557447296567,46640.235926599787,45917.026779706677\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240618,,1653963.6508123996,32515.246551363729,71489.953363785025,73156.053981418867\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240619,,2736343.3982228017,31415.994240657426,93902.780754675274,88032.356863230889\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240620,,2456269.4398571453,26195.107855356298,75449.693327665751,84206.654291599247\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240621,,1490369.4930927495,19851.139197011478,68927.299353089707,65870.811112404466\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240623,,1170456.4301634415,16918.291341274045,32556.87088291247,39166.615953887645\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240624,,2048096.640703432,16468.201226386242,57972.817375166691,40861.142999644282\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240625,,2935525.4333288088,24200.87703483738,67356.283992583398,65344.297354479808\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240626,,1724887.1873088132,25487.199518952519,49373.80470850617,49069.746000081905\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240627,,1307771.3184965495,28417.003179569729,40995.124456068072,34714.888741539617\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240628,,1385373.0869086059,23795.099267671816,40560.378541485516,39210.398423501974\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240629,,1310692.0445979405,18163.096810237505,38809.415660673149,35455.348397118461\n'
    'daily_sse,,,O3,MixAdaptiveFlow,,19600,,,,,,20240630,,1316001.9092712619,21294.911573248915,53282.664587426414,39605.715866750805\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240528,,1399633.9818920232,12582.702793705277,37978.265494924111,31204.852771380291\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240529,,2026693.6421236861,12896.689257077873,38093.217628843631,26315.914226461195\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240530,,2359659.5230918555,11639.852625367232,51201.639158316066,34071.94285701793\n'
    'daily_sse,,,O3,MixDriftBias,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240601,,3511312.7859832174,18782.252986461855,72619.647350768704,50762.21562937479\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240602,,2323832.6042524655,13934.498275946826,41298.922236146565,40945.827471365243\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240603,,2524576.8400694029,20363.461618688889,78038.650793859575,52385.710972985093\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240604,,3508771.4320679624,25344.252079630271,74936.061135391355,54858.062632749119\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240605,,1917831.7569316761,23913.378059390001,48666.871271482982,46824.632877526798\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240606,,1838131.2461618206,19810.546131889336,51372.742801054177,49226.974942165718\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240607,,3157782.9477780899,20756.889950938523,61639.289592143068,46888.7107396559\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240608,,4093137.0819490794,17655.748840048909,42601.455294640182,35258.449030049327\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240609,,1797567.9694968835,16899.557915812358,39003.60488800134,37944.705123399151\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240610,,1922896.9358878725,19739.384245320223,41877.701942346284,37547.027544729652\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240611,,1915030.8979562041,14590.908074609935,60991.260129272778,43721.75888172078\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240612,,1645831.653966764,16079.003313933499,50052.119646366358,34845.310564667518\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240613,,1306206.7859182907,16548.972213805653,36891.555158353323,34622.61939549839\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240614,,1315579.6039883071,22425.643608346581,44028.817259413256,45272.827245380664\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240615,,1293990.7054143394,21992.270073028281,36245.57737261588,43553.490373442874\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240616,,960459.86171621247,16937.835774222389,36788.446260956589,46428.572947540095\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240617,,1345820.688867975,16163.078913466074,46251.221443725924,45305.83207731149\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240618,,1642109.5581614422,32142.061952520162,70996.256665506284,73142.754290732628\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240619,,2722634.2278307127,31367.295727060176,94081.26237830492,88852.399578431927\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240620,,2427636.1086452813,26097.102201235481,75228.62305728819,84285.586663473005\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240621,,1467201.3248731522,19470.848620437086,68435.139465641318,65160.792785868958\n'
    'daily_sse,,,O3,MixDriftBias,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240623,,1155385.9093023685,16411.861423050985,32456.037925052347,38691.09986902852\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240624,,2039800.3383259268,16019.861079193652,58110.973610017987,40624.768180656101\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240625,,2916608.7325298036,23451.952200599015,66678.120536743198,64976.571083703355\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240626,,1696669.7818663032,24667.491921466775,48887.837727377388,48474.395229148577\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240627,,1288345.6989766106,27768.130867964588,40174.688499759475,34334.898848372977\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240628,,1362942.7457779313,23164.436072922312,40569.742372960565,38832.935263981708\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240629,,1295274.5925802463,17821.060326402076,38546.510257144051,35079.082605890755\n'
    'daily_sse,,,O3,MixDriftBias,,19600,,,,,,20240630,,1295352.6288564124,20860.20792731829,53108.978852167915,39560.317383768626\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240528,,1413861.2945226531,12732.781495443545,37758.248579896106,31216.051082362414\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240529,,2042874.958579103,13138.189215657301,38010.647118068693,26585.308752400473\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240530,,2366352.3716041334,11730.818215643987,51041.596172773148,34135.800375018523\n'
    'daily_sse,,,O3,MixMultiScale,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240601,,3474144.7851382256,18968.61886337772,72527.837396775576,50806.539364103533\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240602,,2303468.6891001044,14136.831839323044,41789.650095489385,41354.246777087421\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240603,,2507620.1120161014,20632.697969357483,77876.39951301874,52633.432367278903\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240604,,3472109.5958383875,25914.407142290846,75228.670579924466,55057.838128460855\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240605,,1911879.514329545,24652.820360889658,48503.139921737551,46965.83639031589\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240606,,1857682.397424794,20259.619036527351,51044.502339512517,49375.788121497026\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240607,,3181781.3051214414,21272.111182933673,61690.65761790931,47303.269376352044\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240608,,4096059.6125205471,18248.879983946681,43058.242604891806,35658.637828492952\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240609,,1820347.2141450252,17250.215390461497,39391.286026591108,38148.121179473659\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240610,,1949034.8047008747,20321.083298721351,42314.565537979855,38302.3092358176\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240611,,1984744.0027855106,14903.000875833444,61062.168113826803,43993.503575709379\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240612,,1679264.378106931,16490.569266267121,50384.641929395417,35427.246183476869\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240613,,1341191.3321880773,17031.112538072281,37055.87709927361,35221.776153931947\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240614,,1336036.7486899763,23098.662898214534,43673.955771524867,45708.055973119088\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240615,,1312663.186725456,22175.916983191855,37002.56459537382,43737.446085585427\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240616,,980297.38253393909,17377.358337409794,37042.133259493159,46890.992409997198\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240617,,1363999.9042396843,16539.802173871547,46742.015934533869,46031.06747538536\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240618,,1657384.951217036,32574.757213102654,71430.51910793045,73265.323667111326\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240619,,2735504.9662226951,31417.823458557017,93851.034644199492,88125.748177915171\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240620,,2455208.5100171929,26200.27596124541,75528.1561394452,84238.366420715116\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240621,,1494703.4609604224,19887.509758889675,68991.010791925102,65927.555932012852\n'
    'daily_sse,,,O3,MixMultiScale,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240623,,1175020.2547395958,16992.487515922636,32623.051629001293,39241.034160325216\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240624,,2058128.2001979013,16532.012632864527,57950.642090521942,40884.007262128456\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240625,,2942614.1792036989,24271.493579726666,67532.111487720482,65403.440813284804\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240626,,1730954.4973082428,25518.53465094883,49496.924869958639,49174.092424719507\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240627,,1308958.9613114302,28457.745976760983,41040.818231609323,34745.727904640145\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240628,,1388212.8569795557,23871.214797046036,40671.024389210354,39256.982200477956\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240629,,1314856.1982249413,18205.366419615224,38912.110740517273,35499.780636436699\n'
    'daily_sse,,,O3,MixMultiScale,,19600,,,,,,20240630,,1321435.6989004558,21291.65434605442,53213.72309256898,39602.196293727073\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240528,,1412942.2648206516,12738.40866178181,37760.254447746811,31211.560442379036\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240529,,2041084.8594365453,13136.894259726629,38015.515296994905,26572.920325516949\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240530,,2365343.8193586888,11730.279830971733,51058.932158929339,34124.786367690009\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240601,,3473362.9146566698,18972.033567615785,72568.911778308917,50810.180607745307\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240602,,2302911.4901599865,14122.70459882915,41756.901580459242,41324.827820568011\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240603,,2506841.5294867381,20622.134453547187,77861.86631782855,52612.358403317885\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240604,,3473039.975838352,25922.783939510584,75259.815768584027,55051.515935520139\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240605,,1911668.5090398001,24659.53487950284,48523.790085335197,46958.349414166041\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240606,,1857229.4990462807,20267.573206863366,51083.164193724893,49376.977560435997\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240607,,3181383.3753123358,21282.45600011386,61740.680264183859,47300.943237543703\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240608,,4094863.2942148717,18244.502086683176,43061.616826479745,35652.658367303869\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240609,,1819610.9148678309,17249.412623894401,39381.743462050763,38137.548194406947\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240610,,1948569.277810792,20311.403001488186,42281.168716741362,38274.184149309302\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240611,,1984085.8559942492,14904.87345583085,61090.143031761712,43997.664534700569\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240612,,1678867.8386553363,16489.159194295295,50397.885201365752,35418.410420870045\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240613,,1340384.3276126753,17027.809498790652,37040.372560880358,35215.737891317251\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240614,,1335294.2987948717,23084.724259536713,43660.698957740518,45695.920938569609\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240615,,1311949.722371493,22169.974968619645,36983.418717290144,43720.479174135588\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240616,,980021.49109745584,17371.348286632448,37041.819955436855,46878.681306129482\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240617,,1364711.2440540683,16537.339479196817,46706.071470060066,46000.776850075395\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240618,,1657865.4537439747,32559.036503860727,71369.90073832561,73232.434004898794\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240619,,2736479.504843038,31409.144222076051,93791.543685149314,88107.250874661491\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240620,,2456094.1648081401,26196.741302338429,75512.025705165986,84232.822897893304\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240621,,1495473.2293719312,19881.411341778934,68968.726883797717,65911.268758439022\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240623,,1174480.4367872693,16980.448002640158,32616.905335478212,39234.633952729877\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240624,,2057503.9958602141,16525.293867616914,57950.677816384014,40869.164093844556\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240625,,2942456.2290063687,24255.867659197189,67502.577259744401,65372.58498881061\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240626,,1731211.6862435653,25503.465815332718,49455.155261857442,49142.996431667489\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240627,,1309498.0728372037,28463.227406711318,41044.982399255568,34742.232399230204\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240628,,1388346.0652803013,23870.518288112246,40665.351038933506,39245.620118143139\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240629,,1314566.2579576212,18210.28556026984,38913.874087657561,35491.236080783201\n'
    'daily_sse,,,O3,MixPhysicsNewBias,,19600,,,,,,20240630,,1321068.0258599687,21296.52567925211,53234.92231607026,39607.345934418459\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240528,,1415929.2019085682,12654.026683858596,37919.807715165196,31379.524545448548\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240529,,2043057.3511128004,13118.700476488099,37957.168956591697,26628.400589874509\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240530,,2353306.4845339707,11631.226006610319,51556.514673183316,34342.335963144207\n'
    'daily_sse,,,O3,MixRecentBias,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240601,,3469788.8143443689,18978.324743471108,73894.844438065425,52043.082271726627\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240602,,2307363.0419866671,13669.62032900285,41216.829597246964,41175.835904007625\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240603,,2527085.7129350873,20259.816008062102,78482.426006661073,52893.347051375647\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240604,,3517646.9388134582,25550.394338339567,76040.648471689565,55149.739142744642\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240605,,1937670.352208924,24624.041782789864,49536.531964887356,47724.58238421609\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240606,,1861334.9387036962,20252.159612534568,52002.448879164549,50036.664280591998\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240607,,3180009.0955338757,21034.221051264554,63201.127332610318,48248.954335479168\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240608,,4111018.1513439738,17999.537250445224,43309.860132046153,36019.962837866129\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240609,,1805110.6598026636,17068.859470961615,39075.751837322066,38021.764597261019\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240610,,1938361.6913706327,20277.10367274005,42120.433099020236,37941.627039998471\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240611,,1993815.6143807475,14744.637572228909,61071.299665860963,43973.760985498608\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240612,,1675852.8924844067,16371.757227452472,50175.449246156597,35423.301556536622\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240613,,1334942.7294393536,16913.403991490602,37151.719646505051,34835.822940449674\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240614,,1328969.8893408789,23045.966229115613,43859.719451088786,45541.39405804046\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240615,,1299734.2647657502,22128.553911985829,36714.748148203886,43584.38937785181\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240616,,982533.0479718186,17277.401070681401,37025.976617636086,47000.690202531994\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240617,,1355090.2579371124,16422.391524723731,46034.380570535432,45364.463035607725\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240618,,1629509.0448055535,32553.551515289582,71245.124548846667,73033.400625890412\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240619,,2705004.9102675947,31403.818639832549,93341.341362402993,88224.585690027714\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240620,,2416875.103722285,25975.009159193374,75538.140398874544,84493.592941707902\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240621,,1479076.174712192,20153.13572211843,68781.372458457292,66216.254525687109\n'
    'daily_sse,,,O3,MixRecentBias,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240623,,1164474.5689149247,16984.234381977469,32503.581650656004,39202.008320709479\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240624,,2046657.4543661692,16318.89315774478,58138.076467858227,41045.10285908902\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240625,,2948193.8128071688,23915.771907930262,66957.414609855638,65227.675927463184\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240626,,1737718.6185635652,25600.758275449276,49225.002118219621,48759.740539511142\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240627,,1288813.7707387081,28318.972171409987,40734.06148873979,34732.373683228652\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240628,,1379425.0190867714,23803.996459433809,40759.83968163922,39057.468907364382\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240629,,1310602.2543195325,18334.854172213934,39085.386980696305,35480.680413670263\n'
    'daily_sse,,,O3,MixRecentBias,,19600,,,,,,20240630,,1320651.8150192087,21162.848993031308,53167.139821547578,39661.488584800267\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240528,,1413607.2846400486,12739.218257468194,37757.677298584065,31212.674086893749\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240529,,2044075.1530069681,13139.713765424676,38010.268343095027,26576.511634969454\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240530,,2370315.4908201708,11736.731216886081,51031.809927730232,34131.657468056743\n'
    'daily_sse,,,O3,MixTaper,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240601,,3486075.1598841106,18976.288251563907,72527.417741265832,50794.836044396165\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240602,,2314668.0635290323,14146.958864779212,41817.933962514682,41361.270158623738\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240603,,2518757.4002413256,20655.216503233649,77833.11689461497,52649.730360243637\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240604,,3482555.8030254515,25962.078408102505,75179.419320186644,55079.314492103731\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240605,,1924810.5783778594,24691.574167913757,48502.447998014955,46961.631391846648\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240606,,1862873.6522543887,20292.93885934446,51011.986647309692,49361.353323096439\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240607,,3205285.6715294663,21316.529636437073,61625.60327249999,47316.223495084108\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240608,,4126924.075852335,18305.192163268104,43042.867852984898,35637.567279383773\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240609,,1833017.8790458064,17299.145516126417,39383.451573366663,38146.644767210884\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240610,,1965523.7819302799,20377.056350775063,42349.016478226396,38360.235432663649\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240611,,2001589.8337910965,14938.67472686246,60958.98905960712,43973.37371820493\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240612,,1687749.1255055028,16508.295736775734,50369.495113666322,35399.785202083593\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240613,,1343641.2981252111,17080.585383520462,37067.001295086709,35211.104778539688\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240614,,1343235.9948447144,23193.710496882908,43644.316544518406,45718.913195057416\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240615,,1306639.2579440023,22234.611718111672,37049.373848736985,43713.071044134056\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240616,,978986.04014855693,17441.887631190009,37077.974632872123,46883.639642561131\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240617,,1368235.1903714309,16599.563504640013,46860.546822304677,46179.066040174323\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240618,,1647818.6780470801,32710.481928543188,71646.096162198155,73439.815912938255\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240619,,2714778.1701862207,31534.149685502052,94103.192071287442,88316.361539729682\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240620,,2429774.3385180812,26295.538993871771,75520.55741241714,84298.245069558441\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240621,,1494380.0154520215,19959.638055968098,69021.994596261342,65928.600902805221\n'
    'daily_sse,,,O3,MixTaper,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240623,,1179854.3208791546,17094.114643632434,32613.364920629425,39222.229964850019\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240624,,2083427.5842576865,16622.406073978171,57933.667021730267,40994.689189853489\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240625,,2979090.4537428776,24434.148404343054,67649.552299769653,65598.716103388826\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240626,,1748205.8117169354,25688.03437397629,49582.148745751663,49339.012278763628\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240627,,1297943.4192353254,28598.383501417935,40977.350890893591,34817.963689186392\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240628,,1380113.5493648795,24031.960324753076,40561.613631174754,39352.340016993818\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240629,,1301405.8206454732,18220.943489124067,38904.459248158084,35540.520098622088\n'
    'daily_sse,,,O3,MixTaper,,19600,,,,,,20240630,,1318264.033444999,21323.390639145859,53103.556890049302,39583.814727942183\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240528,,1412942.1461894261,12738.400923389941,37704.209207395994,31178.994509967557\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240529,,2041083.3395837001,13136.910870943218,37927.285790399357,26521.868785402232\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240530,,2365340.5348213627,11730.314172136597,51143.644776000205,34125.516404036549\n'
    'daily_sse,,,O3,MixWindFlow,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240601,,3473355.3720587017,18972.059115783311,72646.695112140907,50898.649409502264\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240602,,2302906.9546602955,14122.713060996495,41655.809779800271,41209.778240973697\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240603,,2506835.1190304318,20622.186565495096,78136.83033444325,52551.297514716964\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240604,,3473034.1585018057,25922.839217178524,75470.322318563878,55040.867815921876\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240605,,1911665.5163543548,24659.654885258526,48547.482597882059,46909.80814720608\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240606,,1857230.2021814899,20267.623936923221,51123.974971704301,49352.380571916437\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240607,,3181382.2382430155,21282.471545960754,61897.404615158957,47259.328366525253\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240608,,4094857.7292293189,18244.568126421422,43020.815657424188,35618.676232570513\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240609,,1819609.5867675398,17249.462188419886,39313.516203021129,38118.127973195362\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240610,,1948567.6467529135,20311.41573149804,42227.398790367304,38202.071077920373\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240611,,1984085.6282370307,14904.908665548079,61233.196331479659,43984.542148474451\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240612,,1678865.7357660236,16489.192681695335,50394.708294458193,35398.336304265635\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240613,,1340385.0427675049,17027.833284118213,36972.179235396696,35186.199524267467\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240614,,1335294.0471206531,23084.767353657633,43596.497990963733,45652.655447669807\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240615,,1311949.4737270032,22169.970387525856,36940.338602609045,43707.72868126892\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240616,,980020.47146227187,17371.380674115382,37010.417036344639,46906.268737051192\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240617,,1364711.3153891016,16537.37157796789,46640.235926599787,45917.026779706677\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240618,,1657865.7067394166,32559.058766888455,71489.953363785025,73156.053981418867\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240619,,2736477.8838608353,31409.119838721119,93902.780754675274,88032.356863230889\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240620,,2456092.6293648649,26196.767999718897,75449.693327665751,84206.654291599247\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240621,,1495471.9855763998,19881.469920248725,68927.299353089707,65870.811112404466\n'
    'daily_sse,,,O3,MixWindFlow,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240623,,1174480.8081102322,16980.494966165163,32556.87088291247,39166.615953887645\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240624,,2057500.1310379684,16525.34758161474,57972.817375166691,40861.142999644282\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240625,,2942450.744964282,24255.923304844648,67356.283992583398,65344.297354479808\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240626,,1731210.0994609846,25503.460495851934,49373.80470850617,49069.746000081905\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240627,,1309497.386714105,28463.229817028157,40995.124456068072,34714.888741539617\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240628,,1388346.2027407025,23870.542734568939,40560.378541485516,39210.398423501974\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240629,,1314563.3670924953,18210.294216925278,38809.415660673149,35455.348397118461\n'
    'daily_sse,,,O3,MixWindFlow,,19600,,,,,,20240630,,1321067.8407008697,21296.530129713006,53282.664587426414,39605.715866750805\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240528,,1441291.1568375165,14546.360294063576,40625.658771716226,33637.638457812638\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240529,,1850134.2254766142,13797.510633929633,42322.239134412972,29983.312890935762\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240530,,2463398.1732344963,13588.293667856604,51720.973462673348,37218.275362552798\n'
    'daily_sse,,,O3,MultiScale,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240601,,3023882.1099326406,21058.241718770936,73265.015947053878,50604.446885650803\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240602,,2308085.5694044619,20020.818867883645,64083.715720591856,55011.213132844772\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240603,,2966283.7938428563,25865.734203169122,91145.807155125207,59521.531020972114\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240604,,4164259.2729153903,24531.552668191493,78668.430673135139,62761.47293926531\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240605,,2282170.6896744519,22341.784297095612,52904.754225977049,47248.791039222342\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240606,,2071322.480079894,19735.023432590999,57203.019823920549,49985.122527893414\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240607,,3309068.2474908186,21385.695017646067,60807.360446648534,49333.646153452086\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240608,,3983693.0581121645,20122.099318302236,51090.606640353348,41270.736026060171\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240609,,1953830.9248932945,17597.014645046555,47026.547399933574,44466.759136289351\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240610,,2088454.4353949409,22093.285663250834,48992.397534968586,43972.105364026007\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240611,,1880591.9359692789,18467.112933808006,60859.770084970951,47146.98326992197\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240612,,1578856.5671187323,18792.624068834819,54428.042876674299,39558.281889162616\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240613,,1523430.6867333667,18355.314719366841,41771.292548129699,39263.256058328778\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240614,,1489165.6955036006,26582.832562716678,52344.031430130897,51005.836752807605\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240615,,1702345.4819613756,24271.648726820946,46009.549149650033,50627.968261511771\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240616,,1387112.3804203391,20673.725965179503,49303.320494538486,57660.703653352255\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240617,,1806161.1785316819,18797.713919105008,58732.254535091983,58061.354777710359\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240618,,2396268.8203503983,34571.713442869484,80394.734189583003,83057.643647325633\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240619,,3571070.1432460272,31618.689838628285,100642.90661843105,91736.585443411052\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240620,,2855920.2244027504,21997.792157663964,82397.451511930209,81897.138991273401\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240621,,1957716.8028047122,18944.411813577637,75033.725344485152,63706.081703584074\n'
    'daily_sse,,,O3,MultiScale,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240623,,1504834.4076091521,16034.060902955942,41046.405019055579,46693.068571436859\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240624,,2420676.2640835121,18532.361534030177,66188.1066124762,49972.945249409167\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240625,,2874847.6311300606,28277.324494964443,82916.210941602112,78708.392781548668\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240626,,1879694.5980401994,25922.733027647249,60731.699516501583,59591.570765419805\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240627,,2263604.6332897916,27752.134704388678,47495.852099580909,42647.805333094417\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240628,,1614220.0163183573,22299.404302357696,45050.87182150616,43942.835831618489\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240629,,1539637.0663893893,18340.551681321114,44074.94395937668,37090.870725753011\n'
    'daily_sse,,,O3,MultiScale,,19600,,,,,,20240630,,1868341.4641690578,22663.10790422745,56698.91022890008,42569.755869868895\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240528,,1575687.9212586565,15073.017469976097,40505.329146567805,34456.793426176941\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240529,,2115224.9207305424,14318.929777519777,42155.098875926444,30820.378791225583\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240530,,2716399.6093589542,14155.350693276152,51012.967647190206,38101.5757734008\n'
    'daily_sse,,,O3,PhysicsLegacy,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240601,,3302757.0945815826,21474.231497931294,73454.713646166521,50395.02332587359\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240602,,2526947.080650711,20807.983540811576,66619.729288757051,56456.636490682649\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240603,,3063959.5994812846,26715.39057886973,92023.787084372481,60708.19119847507\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240604,,4142121.2173872287,25404.277720691636,78587.693450912804,63865.302331907696\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240605,,2506820.5599754136,22957.318325937726,53952.70308739171,47753.777973833479\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240606,,2115402.2789598801,20348.616004018113,57699.056882836674,50205.012391984354\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240607,,3586017.9384482517,22105.414222997613,60512.976889668753,49993.601830795444\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240608,,4453906.7171621062,21119.305886744522,51405.470975849428,41451.910722962049\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240609,,2136957.7162780655,18435.727960427292,47557.520558554541,45163.935318099262\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240610,,2287606.3656656337,23013.362773128785,50235.489424209467,45813.981999624331\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240611,,2095680.1176369076,19079.067717167549,59759.823457316241,47438.396532402643\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240612,,1660036.0441385619,19066.611871280707,54668.33469678471,39965.298302294781\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240613,,1546802.3037725401,19075.751251894049,42529.531483773593,39904.092607191604\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240614,,1584302.3936863842,27784.425055529922,52350.277161702732,51939.496780508241\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240615,,1619096.4308362014,24949.963037827052,46914.913796136512,51051.269044289395\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240616,,1318142.8035325834,21494.829309218563,50066.224452389637,58289.592333682376\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240617,,1757722.7228210852,19533.233467816375,60468.318330167305,60473.111392409846\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240618,,2162801.4748783885,36109.797546099871,82808.523529103477,85676.96473863702\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240619,,3214802.8664623983,32784.545206052251,102973.49896990116,94536.669229463761\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240620,,2511480.03107537,22689.6276624538,82069.746736418616,83202.843559330737\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240621,,1866222.2757567063,19297.559826928191,75488.307235829197,64421.605435235797\n'
    'daily_sse,,,O3,PhysicsLegacy,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240623,,1483203.2307342691,16821.846845404245,41457.072976326148,46987.939424459539\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240624,,2534296.1240338408,19357.828261194751,67038.900227676859,51419.55448334117\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240625,,3098518.7698721276,29804.252783165313,85035.05682608005,81104.40193716604\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240626,,1924813.5982053475,27440.825115405954,62705.478363833987,61691.987901973283\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240627,,2089675.7820744205,28974.113703584298,47711.509588098088,43953.412759106119\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240628,,1535281.4068195575,23439.303047274239,44441.219384948912,45326.411237254593\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240629,,1387523.6803909594,18533.935499798506,44263.925281942298,37946.215102923248\n'
    'daily_sse,,,O3,PhysicsLegacy,,19600,,,,,,20240630,,1751644.7794262066,23005.267701498233,56101.06550193928,43128.018586810809\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240528,,1575683.8108494077,15072.91335096024,40505.394099213416,34456.772789101749\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240529,,2115229.0478470642,14318.796091018245,42155.150141117032,30820.337416183644\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240530,,2716413.0460732696,14155.168339365162,51012.797330229281,38101.55138271732\n'
    'daily_sse,,,O3,PhysicsNew,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240601,,3302784.5148329521,21474.162064437754,73454.622163104315,50394.940501084959\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240602,,2526972.0717909909,20807.997068407945,66619.606591961623,56456.656267572376\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240603,,3063979.4065855737,26715.209352166392,92024.064357123454,60708.120976338585\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240604,,4142130.2663171957,25404.024156434461,78587.56151652163,63865.111302210433\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240605,,2506836.4152752319,22956.77898620069,53952.95986978837,47753.833311995091\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240606,,2115389.0073409816,20348.280608490109,57699.172628250446,50204.97161754081\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240607,,3586020.4026996866,22105.17861741595,60512.713454773009,49993.461989508309\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240608,,4453936.6344056875,21118.950710335746,51405.409588881645,41451.935811110838\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240609,,2136970.3964494234,18435.453067231923,47557.60520704175,45164.009643679616\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240610,,2287613.3676528139,23013.168229979463,50235.34526852236,45814.030655307761\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240611,,2095675.4267918437,19078.812882285565,59759.887075280829,47438.423818929165\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240612,,1660042.9309058997,19066.398301308043,54668.590390259611,39965.475779285334\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240613,,1546794.9114449664,19075.565491289832,42529.502588719282,39904.000530160512\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240614,,1584303.7620712616,27784.11503710784,52350.2243932204,51939.408176061625\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240615,,1619090.3607453669,24949.930260757916,46914.772759581516,51051.229690559601\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240616,,1318141.0983413225,21494.602177912369,50066.172753621599,58289.533107472453\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240617,,1757712.4003077813,19533.020096698776,60468.355411337354,60472.976634159102\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240618,,2162795.7132092868,36109.652599667199,82808.365095573856,85676.745199129757\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240619,,3214805.6787869995,32784.632909168489,102973.55198363162,94536.542500754702\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240620,,2511478.0130911926,22689.508893031627,82069.435082712691,83203.093849376601\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240621,,1866218.8241186915,19297.387988462113,75488.643053849053,64421.640729062434\n'
    'daily_sse,,,O3,PhysicsNew,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240623,,1483197.4331707268,16821.649019177072,41457.06991255004,46987.783023686905\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240624,,2534313.8397946795,19357.566394161433,67039.117232582081,51419.528472425358\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240625,,3098549.8792640166,29803.940452279523,85035.082678837527,81104.13624203889\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240626,,1924819.4801399994,27440.715769373812,62705.582065975585,61691.897794066761\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240627,,2089670.969981506,28973.939053792506,47711.515673638467,43953.326203343997\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240628,,1535281.3878460368,23439.161599592306,44441.320183934244,45326.355489666908\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240629,,1387535.3308546224,18533.849987247959,44264.014188913818,37946.289621754004\n'
    'daily_sse,,,O3,PhysicsNew,,19600,,,,,,20240630,,1751630.8969274398,23005.196044259705,56101.41011100373,43128.072302788591\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240528,,1435417.6167132591,14548.875304866582,40547.730848395579,33564.834759054793\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240529,,1837741.5621234239,13760.549939285964,42279.399605982821,29852.971053101741\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240530,,2454289.2374666771,13539.811427516863,51702.470992687253,37095.545132205363\n'
    'daily_sse,,,O3,PhysicsNewBias,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240601,,3014244.0088207843,21008.466395727359,73332.075325859609,50551.641646225769\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240602,,2300528.6856490918,19879.274289605208,63834.700707591292,54780.00065746886\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240603,,2958984.2979559284,25762.951628028415,90972.508954148027,59346.606199080503\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240604,,4168105.3888169308,24511.676869286224,78671.206696599358,62668.835173915082\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240605,,2279459.5929363649,22333.367143156938,52896.759091910171,47162.8550125849\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240606,,2065458.9480976907,19721.02539718803,57266.785943883937,49939.717591671542\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240607,,3305493.8623665404,21381.170620921068,60914.714857013227,49275.532517853062\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240608,,3977059.7000213237,20051.983082300052,51034.746791631427,41190.460576658123\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240609,,1946270.6225378714,17543.04562761914,46905.460595711484,44348.031975468315\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240610,,2084855.5818763154,22016.743562359363,48762.503259877805,43756.675472918207\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240611,,1874977.9567227042,18453.480376055464,60931.720334488666,47129.950530697381\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240612,,1576168.9012655139,18765.711530275643,54414.859110399855,39453.814448178608\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240613,,1516140.4785778963,18305.808781087399,41619.554256246447,39165.692274616275\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240614,,1483544.1125908385,26482.191504301503,52226.200057803682,50871.449293590253\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240615,,1695963.4895853375,24210.222985805012,45857.870026975848,50466.366806369231\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240616,,1382155.636487507,20598.098073606379,49205.126463785258,57523.861073903303\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240617,,1807023.8787217089,18742.775686555542,58461.726447548703,57818.479889986105\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240618,,2398741.0688189752,34462.44430332724,80051.091358668171,82803.688088579132\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240619,,3576907.1654721005,31542.096251217648,100308.41880353888,91545.978525542203\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240620,,2863156.0865918738,21968.197740577161,82268.033363409399,81787.076206537226\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240621,,1960651.23501025,18896.699128908105,74851.497482335864,63556.468958189042\n'
    'daily_sse,,,O3,PhysicsNewBias,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240623,,1499957.3325182351,15940.386103488505,40952.834883965261,46596.88678365096\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240624,,2412245.2931136456,18435.774990123697,66066.447148318111,49818.41141824337\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240625,,2869273.9763635965,28131.539552281611,82648.294546781864,78471.562970650979\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240626,,1878705.6394815757,25804.491034074686,60427.269091453672,59354.521139278251\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240627,,2264770.2267125524,27744.679923554882,47433.255430523837,42568.865464778246\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240628,,1613622.1064123961,22266.689101936296,44987.304781698251,43836.256970683855\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240629,,1538104.9031418026,18338.300698215142,44024.4111362408,37005.413390680944\n'
    'daily_sse,,,O3,PhysicsNewBias,,19600,,,,,,20240630,,1864520.279408583,22665.045871968381,56742.87238921626,42546.448024512931\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240528,,1610036.8494901427,14784.381782012992,40798.660415303937,35614.136077781841\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240529,,2265112.8972525615,13489.069501057267,41296.226479592406,31781.262645357172\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240530,,2891906.594602467,15190.892885605805,57153.202072432607,41792.599195765048\n'
    'daily_sse,,,O3,Placebo,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240601,,3695724.5550321424,23147.987512758933,90371.838596548085,64065.090890134205\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240602,,2854243.4939974668,20185.644415530376,64467.946292460198,59287.226581694442\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240603,,3156088.7431126116,27017.453108645044,97156.552638761816,67492.694971556455\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240604,,4121013.9858570891,27975.562733614817,87299.078014001803,75498.112910309734\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240605,,2377361.6220288491,25195.360481382348,54605.010257686423,53084.84766808401\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240606,,2259273.7899068356,22383.102338610217,60161.909798373556,57736.064593568663\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240607,,3799677.1430602842,23330.482749824412,70538.866577650711,58325.136300173872\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240608,,4412017.1660427889,20267.380686715245,51230.42171467092,44930.214643211417\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240609,,2237951.6736729797,18677.97753633,51287.938411086718,50227.052110945442\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240610,,2273420.0564514333,21521.315195195377,48329.516495984797,46580.62058905189\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240611,,2370018.8009866923,18771.142540574074,63640.65385027288,52362.792018244465\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240612,,1814843.0172988998,19150.990225637332,58252.384416695721,43719.518019771007\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240613,,1695426.9235153196,19230.683051060885,41441.162001551347,41501.087521513793\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240614,,1678278.5318884896,24774.986554902047,53851.744959001648,54273.141139225241\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240615,,1642183.9729539566,25217.095039092936,49979.441387870247,55116.212812194659\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240616,,1474336.544527516,20778.619260628708,53217.60603408012,62206.675928168414\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240617,,1777947.1727720357,19442.876272013411,63297.582917864944,61874.250911347692\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240618,,1976382.1994723647,31980.423457207158,79037.028048252658,87870.718906328373\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240619,,2962307.2556133228,31264.810125729069,104023.1549650861,96994.510415925804\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240620,,2641247.0663191802,23115.207280297764,85105.877424716135,85268.475237240622\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240621,,2133785.424437494,20858.536687545478,77296.687507027396,68761.693949097331\n'
    'daily_sse,,,O3,Placebo,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240623,,1509469.6849201026,17903.76948713325,42460.350370452827,49364.238404135685\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240624,,2771077.1519103595,20727.435540808365,77330.564428429308,58232.956365664781\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240625,,3164142.9363800921,27304.958684390411,84921.426977931435,84503.688730334994\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240626,,1995772.038277365,25738.375386869535,67126.525999921389,64660.804997357729\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240627,,1945838.4584160098,29201.059576939791,47909.245316996181,43874.227303967702\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240628,,1421324.2408718811,20903.158806078136,46973.186595460575,46693.980266316583\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240629,,1244683.0015318675,17774.466408624314,44838.635329001125,37861.730927085533\n'
    'daily_sse,,,O3,Placebo,,19600,,,,,,20240630,,1730449.8398959956,23674.956410280429,60146.550673615282,43966.106089011242\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240528,,1552570.1013209508,14353.888213041238,40376.717415391162,35502.227887926398\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240529,,2133085.40298555,13786.248326479457,41784.913837184482,30800.706306022523\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240530,,2587826.1714968216,13097.751411653124,51794.874061127433,38285.231941281752\n'
    'daily_sse,,,O3,Recent,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240601,,3269563.2668347117,20173.401122434996,76999.718822091367,54650.295057533134\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240602,,2527080.0769225303,17309.266021447256,61821.602565708272,54076.424578416292\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240603,,3291870.6561806141,24089.774117517285,94685.854089652683,61720.878996058673\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240604,,4258590.2748965314,23602.39755830355,81230.045713459462,63625.366270059712\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240605,,2575596.4556159424,22467.044736385345,57846.825372563777,51859.907019038823\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240606,,2114914.230821725,19629.533008542843,59362.77395497026,53634.728571572763\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240607,,3588124.063598698,20595.074836341664,65969.168228119248,54963.049647134016\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240608,,4445077.3774315454,18897.890051283874,51058.711857170172,42942.817115448001\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240609,,2009580.1488857889,16858.43756210804,44861.426332377458,43666.012767419626\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240610,,2179977.2791243577,22538.465250267647,49409.294684301298,44165.748661841353\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240611,,2129760.5378605705,18155.541225397959,60866.123231850754,48266.596684631935\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240612,,1638911.5834475113,18587.032979249954,54745.954303232589,40369.257032597532\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240613,,1503107.5248023032,18468.455649996176,43227.356113450303,37801.600926769803\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240614,,1559546.6647062765,27550.519614459015,54067.011203777853,51156.676918035089\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240615,,1545120.437496572,24499.568017753772,45271.284864595218,50313.88055821733\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240616,,1324077.3681917044,20317.739978702739,48831.255650332248,58843.208009214693\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240617,,1649276.9199107743,18233.124336890876,55887.934446789484,56296.336092256133\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240618,,1993625.1904943741,35584.653118034825,81887.663011008088,84460.229294885503\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240619,,2926437.5938181761,32269.484630042687,101612.28821206438,94939.85376682863\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240620,,2241474.2529853159,21424.179672852159,81726.329596553507,83703.198118384316\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240621,,1742862.3117259101,19258.260437866673,74985.582467221699,65547.753686292315\n'
    'daily_sse,,,O3,Recent,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240623,,1370168.4245213429,16273.743274193257,41317.178269807242,46799.373777076493\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240624,,2471198.2288776338,17390.917443733662,66113.920363562618,50807.260884638192\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240625,,3082637.5539009976,26454.188583332114,79586.716493969594,78481.041857155462\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240626,,1864241.9643516028,26988.964694841765,58837.023418934972,59179.08427248642\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240627,,1900692.4963697414,27741.902284833603,44830.498718064024,43592.639303686985\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240628,,1433833.5521581508,22838.997494119219,44784.294607271047,44467.884945484504\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240629,,1360025.647431233,19165.968295764178,45146.252299372791,38021.640296901227\n'
    'daily_sse,,,O3,Recent,,19600,,,,,,20240630,,1695286.7268525227,21960.44147857558,55650.089577117811,43439.258611082572\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240528,,1387719.8306028335,13935.347451900132,40581.722943170353,34452.119012459712\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240529,,1822899.3932146225,13337.657600994222,41649.61112343608,29689.025464839757\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240530,,2327284.2885821052,12721.667406752706,52501.132315617702,37055.999266430226\n'
    'daily_sse,,,O3,RecentBias,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240601,,2976249.6379079791,19963.62375189364,76855.071170279392,54570.536421597884\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240602,,2305667.2663671868,16771.862760303542,59401.029288211146,52361.887046897609\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240603,,3187418.6402157778,23505.536803769879,93808.321313346809,60176.696114011691\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240604,,4244561.1192234755,23017.723948994651,81169.265163587683,62192.161100605874\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240605,,2311266.6694927276,21883.345500216819,56825.394654659904,50729.272734332335\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240606,,2068466.7139194959,19112.184751349501,58783.797611471236,52988.075833087401\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240607,,3302460.3762198496,20141.857047018595,66231.732570806213,53847.812674519926\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240608,,3946966.9687142828,18128.112445328385,50853.334072687649,42433.443788045181\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240609,,1831599.1418363792,16179.713453822769,44412.170665803904,42853.523814765576\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240610,,1969417.4142940142,21729.160675312392,48077.701633297431,42224.464998981966\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240611,,1871620.8584587525,17597.860347127542,61416.633283453964,47569.438724636319\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240612,,1533548.4244393772,18312.088019257411,53904.664992607941,39642.796137328303\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240613,,1480446.8515520077,17827.922394169495,42160.94106707054,37069.997538374162\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240614,,1453460.6129728984,26376.024556806311,53760.156982919769,50053.535126913994\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240615,,1621395.891877823,23800.410053708591,44183.627277920714,49633.819969841141\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240616,,1377946.7022062952,19617.984877360985,47806.222211387329,57814.7129166714\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240617,,1685773.7839337424,17644.209326171316,54060.637225790662,53728.547635294133\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240618,,2219481.3646641374,34273.54344165232,79329.813740191443,81505.669373702316\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240619,,3252825.6887382101,31272.520046886988,99295.976993271601,91889.617060758814\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240620,,2557687.5787680889,20871.90799309127,81251.138507247961,82346.190069702963\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240621,,1790484.5213322414,18936.773636335507,73826.370218607,64811.148331228498\n'
    'daily_sse,,,O3,RecentBias,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240623,,1361391.7232559302,15549.369013871998,40840.401600947473,46323.141326369449\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240624,,2351773.7512122327,16793.25339663215,65176.396744736398,49041.403692021871\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240625,,2834527.2158293026,25211.658052285202,77612.275445217034,75728.223928676016\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240626,,1793665.5046958746,25607.789310845546,56979.161733081899,56853.286260830144\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240627,,2045501.6720900275,26697.627103810199,44605.931778886421,42138.010742460661\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240628,,1485546.7260357912,21777.395297723822,45589.398182057543,43027.403267033515\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240629,,1482569.792708331,18883.800967798568,44740.391484471569,36973.547308466405\n'
    'daily_sse,,,O3,RecentBias,,19600,,,,,,20240630,,1761879.3725670557,21660.591536019929,55992.741800413547,42659.057210725943\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240528,,1412942.1461894261,12738.400923389941,37760.231240137306,31211.562187510299\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240529,,2041083.3395837001,13136.910870943218,38015.517435619397,26572.92658981061\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240530,,2365340.5348213627,11730.314172136597,51058.972175835697,34124.790129755995\n'
    'daily_sse,,,O3,Reference,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240601,,3473355.3720587017,18972.059115783311,72568.932879512053,50810.20034079023\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240602,,2302906.9546602955,14122.713060996495,41756.898923962843,41324.826669499911\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240603,,2506835.1190304318,20622.186565495096,77861.835741934046,52612.369240080152\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240604,,3473034.1585018057,25922.839217178524,75259.857677432927,55051.550823122132\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240605,,1911665.5163543548,24659.654885258526,48523.776692125321,46958.337800822417\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240606,,1857230.2021814899,20267.623936923221,51083.196176510057,49376.990309281035\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240607,,3181382.2382430155,21282.471545960754,61740.729537158659,47300.960112727589\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240608,,4094857.7292293189,18244.568126421422,43061.63648948021,35652.65771415209\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240609,,1819609.5867675398,17249.462188419886,39381.74342335933,38137.549928428401\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240610,,1948567.6467529135,20311.41573149804,42281.185625758575,38274.179494191412\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240611,,1984085.6282370307,14904.908665548079,61090.132457083644,43997.663692801179\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240612,,1678865.7357660236,16489.192681695335,50397.858230195925,35418.383568369347\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240613,,1340385.0427675049,17027.833284118213,37040.381466769664,35215.759421227427\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240614,,1335294.0471206531,23084.767353657633,43660.703917625615,45695.933108089172\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240615,,1311949.4737270032,22169.970387525856,36983.433691183767,43720.479708587991\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240616,,980020.47146227187,17371.380674115382,37041.827756876912,46878.695772680549\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240617,,1364711.3153891016,16537.37157796789,46706.053369866946,46000.803595826008\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240618,,1657865.7067394166,32559.058766888455,71369.929395391213,73232.480422210661\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240619,,2736477.8838608353,31409.119838721119,93791.496997806127,88107.270282694721\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240620,,2456092.6293648649,26196.767999718897,75512.066253298748,84232.770607303246\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240621,,1495471.9855763998,19881.469920248725,68968.673030894453,65911.255503719061\n'
    'daily_sse,,,O3,Reference,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240623,,1174480.8081102322,16980.494966165163,32616.905101815337,39234.654373296347\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240624,,2057500.1310379684,16525.34758161474,57950.6489119697,40869.174510575511\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240625,,2942450.744964282,24255.923304844648,67502.559438970289,65372.626138713655\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240626,,1731210.0994609846,25503.460495851934,49455.143972305406,49143.005494500096\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240627,,1309497.386714105,28463.229817028157,41044.975878491714,34742.244918882112\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240628,,1388346.2027407025,23870.542734568939,40665.346486239789,39245.629017458821\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240629,,1314563.3670924953,18210.294216925278,38913.859265807783,35491.223563189684\n'
    'daily_sse,,,O3,Reference,,19600,,,,,,20240630,,1321067.8407008697,21296.530129713006,53234.862524347889,39607.342457528008\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240528,,1437163.010832059,14557.042178248987,40545.740481345863,33580.138846768285\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240529,,1850548.6101909755,13783.299543644302,42265.085121811047,29895.933100966082\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240530,,2472407.4173953892,13583.595074587502,51625.160803620791,37168.154425357316\n'
    'daily_sse,,,O3,Taper,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240601,,3048790.3306859476,21063.712614919059,73327.825839955476,50507.79230117971\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240602,,2331675.520977749,20025.022001268342,64284.54777638321,55039.843888146905\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240603,,2971125.3650019276,25935.657714962959,91141.616011081889,59582.438637738087\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240604,,4153254.6671490008,24692.351586405188,78613.754054464254,62904.867697007423\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240605,,2326240.0463842838,22470.359355518594,53124.368315644024,47277.605374721526\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240606,,2069514.0733172207,19873.534691541456,57348.77619550786,49978.050012421336\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240607,,3377994.8592274324,21575.808482203633,60757.173591052502,49453.419902031877\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240608,,4118598.3046861659,20372.772907099687,51113.552139194311,41234.942563304066\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240609,,2001024.5843394275,17828.771660955623,47088.309820336486,44587.959612017243\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240610,,2147727.7793259248,22358.122920350172,49256.41942285734,44460.904915426887\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240611,,1949242.7321037289,18674.234415826388,60441.086014394263,47207.758750812936\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240612,,1599203.7465975187,18867.729548136704,54476.369082651545,39618.388787109914\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240613,,1518131.3607959542,18612.653845678084,41961.561525145749,39435.645922070005\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240614,,1516830.3458848798,27035.349638387561,52239.642686348729,51297.951468001324\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240615,,1649017.683632293,24531.036083587445,46300.507982757154,50689.151659892916\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240616,,1340130.1899878648,21005.713662387803,49571.031398072621,57844.739484023441\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240617,,1770915.1031783971,19113.471976956353,59410.792343209068,59086.79652410074\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240618,,2266871.2706328975,35283.41843773704,81417.009825786678,84227.795116041205\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240619,,3375492.2600332364,32177.258408506401,101669.11851632518,93076.671786301915\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240620,,2661758.7857200466,22339.086898213252,82113.347733704213,82509.874691310892\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240621,,1897058.994205049,19100.996589734219,75162.428295879727,63992.444247405729\n'
    'daily_sse,,,O3,Taper,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240623,,1478779.7437321381,16436.606576203369,41206.726262965647,46781.431151785633\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240624,,2474007.6596777355,18968.340639013797,66605.005580109064,50735.00226627232\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240625,,2998176.3991679866,29134.216769563034,84063.697878597857,80039.080189128101\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240626,,1896573.2993523418,26805.569388953969,61805.207022082395,60770.46904342551\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240627,,2142350.3866827958,28507.349548616447,47568.965671437632,43409.006903514302\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240628,,1552512.9749482428,23007.630705364048,44593.600311578957,44763.351361787929\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240629,,1428644.6496022546,18449.804688982666,44144.556618967261,37587.421600979964\n'
    'daily_sse,,,O3,Taper,,19600,,,,,,20240630,,1778655.775088714,22876.822545460425,56275.00302483952,42897.111851318921\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240528,,1487547.45150463,13005.542663170956,38518.505547406043,31858.034574551322\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240529,,2169094.5883714147,13690.202343733981,38392.614391741117,27280.5844091732\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240530,,2557791.934112147,12173.872248677537,53142.294158753401,35113.03844478017\n'
    'daily_sse,,,O3,Ultralisk,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240601,,3964998.675212617,20106.508489882573,77836.264584036515,54042.022424569237\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240602,,2608404.6508426778,14194.247359168716,39490.186190295077,40735.494966372571\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240603,,2824681.7238648525,20921.614407855086,80296.849402461521,54003.594873301125\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240604,,3737744.1923121121,28333.981156412512,82236.348070161053,56564.737585660201\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240605,,2014956.6342016486,26664.788477735594,50949.520819603851,48908.099973745695\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240606,,2024500.5059177694,21817.253138806671,53838.592400443689,51386.960256777951\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240607,,3480259.5562224248,22959.898337406106,67160.566398710798,49497.575492482429\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240608,,4323095.080056252,19468.368059246801,44498.451810926083,36569.020844399463\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240609,,1923466.9322610111,18232.043165360577,39748.745440285609,38545.432270648984\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240610,,2026693.5566825604,20612.222543374635,42443.214076428347,38605.880440645007\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240611,,2115073.4452865599,14698.148068652488,62572.108140284719,44366.756699022582\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240612,,1795838.7693516018,16644.782372682355,51071.426025316701,35818.966529842415\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240613,,1375865.5512305431,17347.820744716562,37270.759132960644,35593.391303823031\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240614,,1377642.1422218061,22918.700204778463,43276.885956960279,46083.736952913408\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240615,,1302826.1530802906,22294.92111168988,36186.316624234372,43486.466588857387\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240616,,999666.64127400587,17389.776675363071,36320.855950202502,46004.108798000714\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240617,,1377994.2911716164,16798.531377434731,45530.251704591494,44805.541535444623\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240618,,1629340.3374711049,32803.753188592382,71456.679111220918,72911.521773492554\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240619,,2712319.5911939908,32186.888136819936,95075.990915212591,89290.521660469094\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240620,,2481831.824526648,28614.759148062207,76054.46784760915,86554.749046248151\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240621,,1522302.2298873868,21888.150384848006,70150.528578804253,68500.805557095911\n'
    'daily_sse,,,O3,Ultralisk,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240623,,1225246.7370227459,18283.283202407882,32198.663100197937,38908.528340596458\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240624,,2248806.9539821525,17361.238158676773,59117.044687456473,41055.34243762993\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240625,,3140798.2356584412,24743.318338323385,66883.708211113597,64530.126565547485\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240626,,1842566.2958834374,26373.285090121441,49352.532606362991,48371.468211379601\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240627,,1239596.9652833007,29602.893400101922,41672.13863702619,34522.29457607837\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240628,,1417804.2979203728,24981.562179480679,41104.462264387366,39280.968675898846\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240629,,1342293.3268757991,18719.231859102845,38808.139927901204,36190.876102532806\n'
    'daily_sse,,,O3,Ultralisk,,19600,,,,,,20240630,,1327647.5046696581,21871.071881633252,54080.958543708672,40576.954111374638\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240528,,1435421.2924030018,14548.926936379634,40171.830930428332,33344.701855963373\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240529,,1837739.9412201701,13760.644312219694,41746.687828644805,29506.574443540499\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240530,,2454279.6197419628,13539.945213350467,51923.597922054403,36872.255379946058\n'
    'daily_sse,,,O3,WindFlow,,0,,,,,,20240531,,0,0,0,0\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240601,,3014224.4740235871,21008.497647908516,72723.686956990161,50674.977149203783\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240602,,2300510.1010660115,19879.220624334179,63031.315122760818,53948.620669872151\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240603,,2958970.7388501838,25763.095353659242,91291.298078128559,58768.759272623589\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240604,,4168102.7476811605,24511.863318068907,78205.522068464343,62216.432834452695\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240605,,2279451.636672046,22333.828492343426,52252.488055766466,46775.067866184407\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240606,,2065472.1315654998,19721.291209409945,56344.066537780789,49616.209582777628\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240607,,3305494.1697599669,21381.333634023555,60616.196746387781,48798.322095219279\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240608,,3977038.7302727532,20052.270068080164,50424.362384427259,40877.708972597335\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240609,,1946260.8321775079,17543.25356579572,46418.356625777757,44126.242775870538\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240610,,2084849.5649637084,22016.86421813909,48447.219905015227,43403.253884480095\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240611,,1874983.2344092876,18453.666886952706,61484.418340266428,46974.365526907932\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240612,,1576165.3425864961,18765.88635610044,54148.835952306144,39359.580052038014\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240613,,1516149.2431634918,18305.948956847191,41232.505961013812,38982.167313687431\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240614,,1483543.1583421347,26482.430280098692,51704.669426673485,50615.067197118136\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240615,,1695969.2004161484,24210.200316242874,45639.222053408397,50275.422700270181\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240616,,1382159.2473719581,20598.259870770387,48973.459922508715,57548.26477938579\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240617,,1807034.2719019454,18742.937363203615,58038.676523351569,57261.504149576896\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240618,,2398744.0797979659,34462.49759888649,80288.379266852382,82284.972332130012\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240619,,3576898.1415456152,31541.940314881504,100225.04550290793,91026.410001233366\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240620,,2863155.0503443452,21968.25730502978,81559.162848569991,81583.069809469511\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240621,,1960657.1492028059,18896.82846920006,74340.953673380704,63220.770154336293\n'
    'daily_sse,,,O3,WindFlow,,0,,,,,,20240622,,0,0,0,0\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240623,,1499963.1121323009,15940.529232152738,40568.338229353853,46155.337338464895\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240624,,2412232.4040498277,18435.974303170107,65758.944439632833,49488.995446379144\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240625,,2869251.2209033989,28131.764221156016,81631.304482927284,78019.225777442945\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240626,,1878698.5839022845,25804.511293191463,59689.15873327048,58846.721880607947\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240627,,2264771.5849418761,27744.750414613634,46999.535308519582,42402.660958724002\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240628,,1613621.3096835276,22266.766923493706,44338.918141482951,43629.309381746243\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240629,,1538090.4036749299,18338.352917904034,43393.995518894284,36706.606105177969\n'
    'daily_sse,,,O3,WindFlow,,19600,,,,,,20240630,,1864531.9610055566,22665.061993407086,56860.696846684361,42435.311625913215\n'
)
assert hashlib.sha256(text.encode('utf-8')).hexdigest()=='78568dd2c40e4e5d4317fc113fe992eff4e8f4143b9d8c0bbec6d6cbcd9d83a7'
(CODE/'Atlas33_FULL_AUDIT.csv').write_bytes(text.encode('utf-8'))

# --- Ultralisk_v2_7_completo.py: texto original o interfaz de despliegue ---
text=(
    '# ULTRALISK v2.7 COMPLETO | ShadowBytes | una sola celda Kaggle\n'
    'import os, json\n'
    'from pathlib import Path\n'
    "_cache27 = Path(os.environ.get('ULTRALISK_CHRONOS_DIR','/kaggle/working/Ultralisk-v2.4'))\n"
    "_cachefiles27 = ['report.json','fold1_predictions.npz','fold2_predictions.npz','fold3_predictions.npz','foldApril_predictions.npz','final_family_predictions.npz']\n"
    '_ready27 = all((_cache27/f).is_file() for f in _cachefiles27)\n'
    'if _ready27:\n'
    '    try:\n'
    "        _r27 = json.loads((_cache27/'report.json').read_text())\n"
    "        _ready27 = _r27.get('model')=='Ultralisk-v2.4 Chronos' and bool(_r27.get('smoke'))==(os.environ.get('ULTRALISK_SMOKE')=='1')\n"
    '    except (OSError, ValueError): _ready27=False\n'
    'if not _ready27:\n'
    "    print('FASE 1/2: creando cache neuronal v2.4. NO subir sus CSV intermedios.',flush=True)\n"
    "    os.environ['ULTRALISK_OUT']=str(_cache27)\n"
    "    os.environ['ULTRALISK_CACHE']='1'\n"
    '    # Ultralisk-v2.4 | ShadowBytes | ApuCast | Kaggle T4 x2\n'
    '    import os, time, json, copy, hashlib, threading, platform\n'
    '    from pathlib import Path\n'
    '    from concurrent.futures import ThreadPoolExecutor\n'
    '    import numpy as np\n'
    '    import pandas as pd\n'
    '    import torch\n'
    '    from torch import nn\n'
    '    from torch.nn import functional as F\n'
    '    from IPython.display import display, FileLink\n'
    "    SMOKE=os.environ.get('ULTRALISK_SMOKE')=='1'  # prueba tecnica, NO subir\n"
    "    OUT=Path(os.environ.get('ULTRALISK_OUT','/kaggle/working/Ultralisk-v2.4')); OUT.mkdir(parents=True,exist_ok=True)\n"
    "    LOG=OUT/'training.log'; LOG.write_text(''); LOCK=threading.Lock(); START=time.perf_counter()\n"
    '    BATCH=32; EPOCHS_REF=25; EPOCHS_NEW=36\n'
    "    SEEDS={'CNN':(0,1,2),'ResNet':(0,1,2),'Mutalisk':(42,123),'Viper':(42,123)}\n"
    "    CUTS=[('2024050900','2024051600','2024060100'),('2024052500','2024060100','2024061600'),('2024060900','2024061600','2024070100')]\n"
    "    VARS=['RH','T','U','V']; NORM=np.array([25.360,2.0652,2.6776,2.0876])\n"
    '    # Valores historicos de versions/v3/backtest.json (solo referencia, no etiquetas).\n'
    '    HISTORICAL={1:.42292221906027727,2:.4136267261457105,3:.42723285641743414}\n'
    '    MIN_DEV_GAIN=.003; MIN_AUDIT_GAIN=.002\n'
    '    def log(msg):\n'
    "        line=f'[{time.perf_counter()-START:8.1f}s] {msg}'\n"
    '        with LOCK:\n'
    '            print(line,flush=True)\n'
    "            with LOG.open('a') as f: f.write(line+'\\n')\n"
    "    if not SMOKE and (torch.cuda.device_count()!=2 or any('T4' not in torch.cuda.get_device_name(i) for i in range(2))): raise RuntimeError('Selecciona Accelerator > GPU T4 x2')\n"
    "    DEVICES=[torch.device('cpu')] if SMOKE else [torch.device('cuda:0'),torch.device('cuda:1')]\n"
    '    torch.set_num_threads(2); torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True\n'
    '    cpu_name=platform.processor()\n'
    "    if Path('/proc/cpuinfo').is_file():\n"
    "        cpu_name=next((l.split(':',1)[1].strip() for l in Path('/proc/cpuinfo').read_text().splitlines() if l.startswith('model name')),cpu_name)\n"
    "    log(f'torch={torch.__version__} CPU={cpu_name} cores={os.cpu_count()} torch_threads={torch.get_num_threads()} smoke={SMOKE}')\n"
    '    for d in DEVICES:\n'
    "        if d.type=='cuda': log(f'{d} {torch.cuda.get_device_name(d)} VRAM={torch.cuda.get_device_properties(d).total_memory/2**30:.2f} GiB')\n"
    '    # Datos oficiales\n'
    "    root=Path(os.environ.get('ULTRALISK_DATA','/kaggle/input')); required=('train.npz','test.npz','static.npz','sample_submission.csv')\n"
    "    folders=[p.parent for p in root.rglob('train.npz') if all((p.parent/k).is_file() for k in required)]\n"
    "    official=[p for p in folders if 'hackathon-ia-pucp-2026' in str(p)]\n"
    '    if len(official)==1: DATA=official[0]\n'
    '    elif len(folders)==1: DATA=folders[0]\n'
    "    else: raise RuntimeError(f'Dataset oficial no identificado: {folders}')\n"
    "    with np.load(DATA/'train.npz') as f: X,Y,TT=f['X'].astype('f4'),f['Y'].astype('f4'),f['times'].astype(str)\n"
    "    with np.load(DATA/'test.npz') as f: XT,TE=f['X'].astype('f4'),f['times'].astype(str)\n"
    "    with np.load(DATA/'static.npz') as f:\n"
    "        DEM=f['dem'].astype('f4'); glat,glon,wlat,wlon=[f[k] for k in ('gfs_lats','gfs_lons','wrf_lats','wrf_lons')]\n"
    "    TEMPLATE=pd.read_csv(DATA/'sample_submission.csv',dtype={'id':str})\n"
    '    assert X.shape==(2935,17,13,4) and Y.shape==(2935,70,35,4) and XT.shape==(792,17,13,4) and DEM.shape==(70,35)\n'
    "    assert TEMPLATE.shape==(646800,5) and TEMPLATE.columns.tolist()==['id',*VARS] and TEMPLATE.id.is_unique\n"
    '    assert all(np.isfinite(a).all() for a in (X,Y,XT,DEM)) and max(TT)<min(TE)\n'
    "    parts=TEMPLATE.id.str.split('_',expand=True); ids_t=parts[0].map({s:i for i,s in enumerate(TE)}); assert ids_t.notna().all()\n"
    '    ti=ids_t.to_numpy(dtype=int); ri=parts[1].astype(int).to_numpy(); ci=parts[2].astype(int).to_numpy()\n'
    '    assert (ri>=0).all() and (ri<70).all() and (ci>=0).all() and (ci<35).all()\n'
    '    assert len(np.unique(ti))==264 and np.all(np.bincount(ti)[np.unique(ti)]==2450)\n'
    '    HOURS=np.unique([int(s[-2:]) for s in parts[0].unique()]); H=np.array([int(s[-2:]) for s in TT]); HT=np.array([int(s[-2:]) for s in TE])\n'
    "    log(f'Data={DATA} train={X.shape} test={XT.shape} horas={HOURS.tolist()}')\n"
    '    def windows(x,times):\n'
    "        dates=pd.to_datetime(times,format='%Y%m%d%H'); lookup={t:i for i,t in enumerate(dates)}\n"
    '        return x[np.array([[lookup.get(t-pd.Timedelta(hours=k),i) for k in (2,1,0)] for i,t in enumerate(dates)])]\n'
    '    W,WT=windows(X,TT),windows(XT,TE)\n'
    "    lat,lon=np.meshgrid(wlat,wlon,indexing='ij'); grid=np.stack([(lon-glon[0])/(glon[-1]-glon[0])*2-1,(lat-glat[0])/(glat[-1]-glat[0])*2-1],-1)[None].astype('f4')\n"
    "    def z(a): return ((a-a.mean())/max(float(a.std()),1e-6)).astype('f4')\n"
    '    def smooth(a,k):\n'
    "        p=k//2; return F.avg_pool2d(F.pad(torch.from_numpy(a[None,None]),(p,p,p,p),mode='replicate'),k,stride=1)[0,0].numpy()\n"
    '    dy,dx=np.gradient(DEM)\n'
    "    geo=np.stack([z(DEM),z(dx),z(dy),z(np.hypot(dx,dy)),z(DEM-smooth(DEM,3)),z(DEM-smooth(DEM,7)),z(DEM-smooth(DEM,15)),z(lat),z(lon),z(np.gradient(dy,axis=0)+np.gradient(dx,axis=1))])[None].astype('f4')\n"
    '    def stats(ix):\n'
    '        yy,xx,hh=Y[ix],X[ix],H[ix]\n'
    '        def climate(a): return np.stack([a[hh==h].mean(0) if np.any(hh==h) else a.mean(0) for h in range(24)])\n'
    '        cl=climate(yy)\n'
    '        return dict(mu=xx.mean((0,1,2)),sd=np.maximum(xx.std((0,1,2)),1e-6),clim=cl,gfs_clim=climate(xx),scale=np.maximum((yy-cl[hh]).std((0,1,2)),1e-6))\n'
    '    def score(p,y):\n'
    "        rm=np.sqrt(np.mean(np.square(p.astype('f8')-y),axis=(0,1,2))); return float(1-(rm/NORM).mean()),rm\n"
    '    def clip(p):\n'
    "        p=p.astype('f4',copy=False); p[...,0]=p[...,0].clip(0,100); assert np.isfinite(p).all(); return p\n"
    '    def tensor(a,d): return torch.as_tensor(a,dtype=torch.float32,device=d)\n'
    '    def sync(d):\n'
    "        if d.type=='cuda': torch.cuda.synchronize(d)\n"
    '    # Arquitecturas\n'
    '    class Residual(nn.Module):\n'
    '        def __init__(self,a,b,dilation=1):\n'
    '            super().__init__(); self.c1=nn.Conv2d(a,b,3,padding=dilation,dilation=dilation); self.c2=nn.Conv2d(b,b,3,padding=1); self.skip=nn.Conv2d(a,b,1) if a!=b else nn.Identity()\n'
    '        def forward(self,x): return F.gelu(self.c2(F.gelu(self.c1(x)))+self.skip(x))\n'
    '    def inputs(x,h):\n'
    '        n=len(x); a=h*(2*np.pi/24); clock=torch.stack([a.sin(),a.cos()],1)[:,:,None,None].expand(-1,-1,17,13)\n'
    '        return x.permute(0,1,4,2,3).reshape(n,12,17,13),clock\n'
    '    class Reference(nn.Module):\n'
    '        def __init__(self,kind):\n'
    '            super().__init__(); w=64\n'
    '            self.enc=nn.Sequential(nn.Conv2d(14,w,3,padding=1),nn.GELU(),nn.Conv2d(w,w,3,padding=1),nn.GELU())\n'
    '            self.static=nn.Parameter(torch.zeros(1,8,70,35))\n'
    '            self.dec=nn.Sequential(nn.Conv2d(73,w,3,padding=1),nn.GELU(),nn.Conv2d(w,w,3,padding=1),nn.GELU(),nn.Conv2d(w,4,1))\n'
    "            if kind=='ResNet': self.enc=Residual(14,w); self.dec=nn.Sequential(Residual(73,w),nn.Conv2d(w,4,1))\n"
    "            self.register_buffer('grid',torch.from_numpy(grid)); self.register_buffer('dem',torch.from_numpy(z(DEM)[None,None]))\n"
    '        def forward(self,x,h):\n'
    '            n=len(x); fields,clock=inputs(x,h); v=self.enc(torch.cat([fields,clock],1))\n'
    '            with torch.autocast(device_type=x.device.type,enabled=False): v=F.grid_sample(v.float(),self.grid.expand(n,-1,-1,-1),align_corners=True)\n'
    '            return self.dec(torch.cat([v,self.dem.expand(n,-1,-1,-1),self.static.expand(n,-1,-1,-1)],1)).permute(0,2,3,1)\n'
    '    class GeoWind(nn.Module):\n'
    '        def __init__(self,kind,st):\n'
    '            super().__init__(); self.kind=kind; w=48\n'
    "            self.register_buffer('grid',torch.from_numpy(grid)); self.register_buffer('geo',torch.from_numpy(geo)); self.register_buffer('gcl',torch.from_numpy(((st['gfs_clim']-st['mu'])/st['sd']).astype('f4')))\n"
    '            self.enc=nn.Sequential(Residual(22,w),Residual(w,w,2),Residual(w,w)); self.map=nn.Parameter(torch.zeros(1,8,70,35)); self.fuse=Residual(w+26,w)\n'
    "            if kind=='Mutalisk': self.down=nn.Sequential(Residual(w,64),Residual(64,64,2)); self.up=Residual(w+64,w)\n"
    '            else: self.wind=nn.Sequential(Residual(w,w,2),Residual(w,w,3))\n'
    '            self.thermal=nn.Sequential(Residual(w,32),nn.Conv2d(32,2,1)); self.velocity=nn.Sequential(Residual(w,32),nn.Conv2d(32,2,1))\n'
    '        def interp(self,x):\n'
    "            with torch.autocast(device_type=x.device.type,enabled=False): return F.grid_sample(x.float(),self.grid.expand(len(x),-1,-1,-1),align_corners=True,padding_mode='border')\n"
    '        def forward(self,x,h):\n'
    '            n=len(x); fields,clock=inputs(x,h); current=x[:,-1]; anomaly=current-self.gcl[h.long()]; trend=current-x[:,0]\n'
    '            av=anomaly.permute(0,3,1,2); cv=current.permute(0,3,1,2); v=self.enc(torch.cat([fields,av,trend.permute(0,3,1,2),clock],1))\n'
    '            fine=self.fuse(torch.cat([self.interp(v),self.interp(torch.cat([cv,av],1)),self.geo.expand(n,-1,-1,-1),self.map.expand(n,-1,-1,-1)],1))\n'
    "            if self.kind=='Mutalisk':\n"
    "                low=self.down(F.avg_pool2d(fine,2)); fine=self.up(torch.cat([fine,F.interpolate(low,size=fine.shape[-2:],mode='bilinear',align_corners=False)],1)); wind=fine\n"
    '            else: wind=self.wind(fine)\n'
    '            return torch.cat([self.thermal(fine),self.velocity(wind)],1).permute(0,2,3,1)\n'
    "    def make(kind,st): return Reference(kind) if kind in ('CNN','ResNet') else GeoWind(kind,st)\n"
    '    # Benchmark de inferencia, mismo modelo y lote para todos los dispositivos.\n'
    '    def benchmark():\n'
    "        st=stats(np.arange(128)); torch.manual_seed(0); base=Reference('ResNet').eval(); results=[]\n"
    "        x=(W[:8]-st['mu'])/st['sd']; h=H[:8]\n"
    "        for d,amp in [(torch.device('cpu'),False)]+[(d,a) for d in DEVICES if d.type=='cuda' for a in (False,True)]:\n"
    '            m=copy.deepcopy(base).to(d,memory_format=torch.channels_last); xb,hb=tensor(x,d),tensor(h,d)\n'
    '            with torch.inference_mode():\n'
    '                for _ in range(3):\n'
    '                    with torch.autocast(device_type=d.type,dtype=torch.float16,enabled=amp): m(xb,hb)\n'
    '                sync(d); t=time.perf_counter()\n'
    '                for _ in range(8):\n'
    '                    with torch.autocast(device_type=d.type,dtype=torch.float16,enabled=amp): m(xb,hb)\n'
    '                sync(d); sec=time.perf_counter()-t\n'
    "            r=dict(device=str(d),amp=amp,batch=8,ms_per_batch=sec/8*1000,samples_per_second=64/sec); results.append(r); log(f'BENCHMARK {r}')\n"
    '        if len(DEVICES)==2:\n'
    '            barrier=threading.Barrier(2)\n'
    '            def both(d):\n'
    '                m=copy.deepcopy(base).to(d,memory_format=torch.channels_last); xb,hb=tensor(x,d),tensor(h,d)\n'
    "                with torch.inference_mode(), torch.autocast(device_type='cuda',dtype=torch.float16):\n"
    '                    for _ in range(3): m(xb,hb)\n'
    '                    sync(d); barrier.wait(timeout=60); t=time.perf_counter()\n'
    '                    for _ in range(8): m(xb,hb)\n'
    '                    sync(d); return time.perf_counter()-t\n'
    '            with ThreadPoolExecutor(max_workers=2) as pool:\n'
    '                sec=max(pool.map(both,DEVICES))\n'
    "            r=dict(device='T4x2_parallel',amp=True,batch=16,ms_per_batch=sec/8*1000,samples_per_second=128/sec)\n"
    "            results.append(r); log(f'BENCHMARK {r}')\n"
    '        return results\n'
    "    BENCH=benchmark() if os.environ.get('ULTRALISK_BENCH','0')=='1' else []\n"
    "    log('Benchmark opcional; iniciando experimentos v2.4')\n"
    '    @torch.inference_mode()\n'
    '    def infer(model,x,h,d,amp):\n'
    '        model.eval(); out=[]\n'
    '        for i in range(0,len(x),BATCH):\n'
    '            with torch.autocast(device_type=d.type,dtype=torch.float16,enabled=amp): p=model(x[i:i+BATCH],h[i:i+BATCH])\n'
    '            out.append(p.float().cpu().numpy())\n'
    '        return np.concatenate(out)\n'
    '    def fit_neural(job,d,st,tr,we,he,tag):\n'
    "        kind,seed,model=job; ref=kind in ('CNN','ResNet'); epochs=1 if SMOKE else (EPOCHS_REF if ref else EPOCHS_NEW)\n"
    "        amp=not ref and d.type=='cuda'; lr=.002 if ref else .0012; rng=np.random.RandomState(seed)\n"
    '        model=model.to(d,memory_format=torch.channels_last); ema=None if ref else copy.deepcopy(model).eval().requires_grad_(False)\n'
    "        x=tensor((W[tr]-st['mu'])/st['sd'],d); h=tensor(H[tr],d); y=tensor((Y[tr]-st['clim'][H[tr]])/st['scale'],d)\n"
    "        xe=tensor((we-st['mu'])/st['sd'],d); he_gpu=tensor(he,d)\n"
    '        opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4 if ref else 5e-4)\n'
    '        steps=(len(tr)+BATCH-1)//BATCH; sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=lr,total_steps=steps*epochs)\n'
    "        scaler=torch.amp.GradScaler('cuda',enabled=amp); ratio=tensor(st['scale']/NORM,d)\n"
    '        order=np.arange(len(tr)); started=time.perf_counter(); hist=[]\n'
    "        if d.type=='cuda': torch.cuda.reset_peak_memory_stats(d)\n"
    "        log(f'{tag} {kind}:{seed} {d} START params={sum(p.numel() for p in model.parameters()):,} epochs={epochs} amp={amp}')\n"
    '        for ep in range(1,epochs+1):\n'
    '            model.train(); rng.shuffle(order); accum=torch.zeros((),device=d); t0=time.perf_counter()\n'
    '            for k in range(0,len(order),BATCH):\n'
    '                ix=torch.as_tensor(order[k:k+BATCH],device=d,dtype=torch.long); opt.zero_grad(set_to_none=True)\n'
    '                with torch.autocast(device_type=d.type,dtype=torch.float16,enabled=amp): p=model(x[ix],h[ix])\n'
    '                mse=(p.float()-y[ix]).square(); loss=mse.mean() if ref else .5*mse.mean()+.5*((mse.mean((0,1,2))+1e-8).sqrt()*ratio).mean()\n'
    '                scaler.scale(loss).backward()\n'
    '                if not ref: scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(),1.)\n'
    '                old=scaler.get_scale(); scaler.step(opt); scaler.update()\n'
    '                if scaler.get_scale()>=old:\n'
    '                    sched.step()\n'
    '                    if ema is not None:\n'
    '                        with torch.no_grad():\n'
    '                            for a,b in zip(ema.parameters(),model.parameters()): a.lerp_(b,.015)\n'
    '                accum+=loss.detach()*len(ix)\n'
    '            sync(d); sec=time.perf_counter()-t0; value=float(accum.item()/len(tr)); assert np.isfinite(value)\n'
    "            mem=torch.cuda.max_memory_allocated(d)/2**30 if d.type=='cuda' else None\n"
    "            hist.append(dict(epoch=ep,loss=value,seconds=sec,samples_per_second=len(tr)/sec,peak_gib=mem,lr=opt.param_groups[0]['lr']))\n"
    '            log(f\'{tag} {kind}:{seed} {d} ep={ep:02d}/{epochs} loss={value:.5f} lr={opt.param_groups[0]["lr"]:.6f} {sec:.2f}s {len(tr)/sec:.1f} samples/s VRAM={mem}\')\n'
    "        chosen=model if ref else ema; p=clip(infer(chosen,xe,he_gpu,d,amp)*st['scale']+st['clim'][he]); elapsed=time.perf_counter()-started\n"
    "        pd.DataFrame(hist).to_csv(OUT/f'{tag}_{kind}_{seed}_history.csv',index=False)\n"
    "        if tag=='final': torch.save({'kind':kind,'seed':seed,'epochs':epochs,'state_dict':{k:v.detach().cpu() for k,v in chosen.state_dict().items()}},OUT/f'{kind}_{seed}.pt')\n"
    "        log(f'{tag} {kind}:{seed} DONE {elapsed:.1f}s'); model.to('cpu')\n"
    '        return kind,p,dict(kind=kind,seed=seed,device=str(d),seconds=elapsed,epochs=epochs,peak_gib=mem)\n'
    '    # Kernel Ridge no lineal: anomalias del GFS, tendencia y promedio temporal.\n'
    '    def kernel_features(w,h,st):\n'
    "        a=(w-st['gfs_clim'][(h[:,None]-np.array([2,1,0]))%24])/st['sd']; a=a.reshape(len(w),3,-1)\n"
    "        return np.concatenate([a[:,2],a.mean(1),.5*(a[:,2]-a[:,0])],axis=1).astype('f4')\n"
    '    def kernel(a,b):\n'
    '        dot=(a@b.T)/a.shape[1]; dist=(a.square().mean(1)[:,None]+b.square().mean(1)[None,:]-2*dot).clamp_min(0)\n'
    '        return .5*dot+.25*dot.square()+.25*torch.exp(-dist/2)\n'
    '    def fit_kernel(d,st,tr,we,he,tag):\n'
    '        t=time.perf_counter(); a=kernel_features(W[tr],H[tr],st); b=kernel_features(we,he,st)\n'
    '        mu=a.mean(0); sd=np.maximum(a.std(0),.15); a=(a-mu)/sd; b=(b-mu)/sd\n'
    '        ag,bg=tensor(a,d),tensor(b,d); K=kernel(ag,ag); K=(K+K.T)*.5; K.diagonal().add_(.05)\n'
    "        L=torch.linalg.cholesky(K); yy=tensor(((Y[tr]-st['clim'][H[tr]])/st['scale']).reshape(len(tr),-1),d)\n"
    '        dual=torch.cholesky_solve(yy,L); p=(kernel(bg,ag)@dual).cpu().numpy().reshape(len(we),70,35,4)\n'
    "        if tag=='final': np.savez_compressed(OUT/'kernel_model.npz',features=a,mu=mu,sd=sd,dual=dual.cpu().numpy())\n"
    "        log(f'{tag} Kernel DONE {time.perf_counter()-t:.2f}s alpha=.05'); return clip(p*st['scale']+st['clim'][he])\n"
    '    ALL_RUNS=[]\n'
    '    def run_stage(tr,we,he,tag,families):\n'
    '        st=stats(tr); jobs=[]\n'
    '        for kind in families:\n'
    "            if kind=='Kernel': continue\n"
    '            for seed in (SEEDS[kind][:1] if SMOKE else SEEDS[kind]):\n'
    '                torch.manual_seed(seed); jobs.append((kind,seed,make(kind,st)))\n'
    '        def worker(group,d): return [fit_neural(j,d,st,tr,we,he,tag) for j in group]\n'
    '        results=[]\n'
    '        if len(DEVICES)==2:\n'
    '            with ThreadPoolExecutor(max_workers=2) as pool:\n'
    '                futures=[pool.submit(worker,jobs[i::2],d) for i,d in enumerate(DEVICES)]\n'
    '                for f in futures: results.extend(f.result())\n'
    '        else: results=worker(jobs,DEVICES[0])\n'
    "        ALL_RUNS.extend([dict(tag=tag,**r) for _,_,r in results]); results.sort(key=lambda x:x[2]['seed'])\n"
    "        pred={k:np.mean([p for kind,p,_ in results if kind==k],axis=0,dtype=np.float32) for k in families if k!='Kernel'}\n"
    "        if 'Kernel' in families: pred['Kernel']=fit_kernel(DEVICES[0],st,tr,we,he,tag)\n"
    "        if 'Kernel' in families: pred['Clock']=fit_clock(DEVICES[0],st,tr,we,he,tag)\n"
    "        if tag=='final': np.savez_compressed(OUT/'preprocessing.npz',**st,grid=grid,geo=geo,normalizers=NORM)\n"
    '        return pred\n'
    '    # Calibracion por variable, separada de evaluacion.\n'
    "    CANDIDATES={'Reference':{},'Mutalisk':{'Mutalisk':1.},'Viper':{'Viper':1.},'Neural':{'Mutalisk':.5,'Viper':.5},'Kernel':{'Kernel':1.},'Hybrid':{'Mutalisk':.375,'Viper':.375,'Kernel':.25}}\n"
    '    def fit_alpha(a,b,y,prior=.5,shrink=.2,cap=1.):\n'
    "        d=b.astype('f8')-a; r=y.astype('f8')-a; den=(d*d).mean((0,1,2)); num=(r*d).mean((0,1,2))\n"
    "        raw=np.divide(num,den,out=np.full(4,prior),where=den>1e-12); return np.clip((1-shrink)*raw+shrink*prior,0,cap).astype('f4')\n"
    '    def mix(a,b,w): return clip(a*(1-w)+b*w)\n'
    '    def candidate(p,name): return sum(p[k]*np.float32(w) for k,w in CANDIDATES[name].items())\n'
    '    def calibrate(p,y,name):\n'
    "        rw=fit_alpha(p['CNN'],p['ResNet'],y); ref=mix(p['CNN'],p['ResNet'],rw)\n"
    "        alpha=np.zeros(4,dtype='f4') if name=='Reference' else fit_alpha(ref,candidate(p,name),y,prior=.25,shrink=.35,cap=.65)\n"
    '        return rw,alpha\n'
    '    def combine(p,name,rw,alpha):\n'
    "        ref=mix(p['CNN'],p['ResNet'],rw); return ref if name=='Reference' else mix(ref,candidate(p,name),alpha)\n"
    '\n'
    '\n'
    '    # ================================================================\n'
    '    # ULTRALISK v2.4 | CHRONOS | extension incremental de v2.3\n'
    '    # No usa WRF oculto, datos externos ni horas futuras de GFS.\n'
    '    # ================================================================\n'
    '    from scipy.optimize import minimize\n'
    '    PUBLIC_REFERENCE=.49056\n'
    '    OLD_FOLD={1:.439850572420165,2:.4396332592736687,3:.4496978946404211}\n'
    "    STRESS=('2024040900','2024041600','2024050100')\n"
    '    # Dos variantes conservadoras, probadas individualmente en CPU en folds 1 y 2.\n'
    '    CLOCK_SETTINGS=[(.6,90),(1.2,90)]\n'
    "    FAMILIES=['CNN','ResNet','Mutalisk','Viper','Kernel']\n"
    "    KEYS=FAMILIES+['Clock']\n"
    "    METHODS=['ClockReplace','ClockBlend','Stack']\n"
    '    MIN_DEV=.002; MIN_AUDIT=.002; STRESS_TOL=.002; REPRO_TOL=.0015\n'
    '\n'
    '    def fit_clock(d,st,tr,we,he,tag):\n'
    '        t=time.perf_counter()\n'
    '        a=kernel_features(W[tr],H[tr],st); b=kernel_features(we,he,st)\n'
    '        mu=a.mean(0); sd=np.maximum(a.std(0),.15)\n'
    '        a=(a-mu)/sd; b=(b-mu)/sd\n'
    '        ag,bg=tensor(a,d),tensor(b,d)\n'
    '        h,eh=tensor(H[tr],d),tensor(he,d)\n'
    '        def clock(a,b): return .4+.6*(1+torch.cos((a[:,None]-b[None,:])*(np.pi/12)))/2\n'
    '        K=kernel(ag,ag)*clock(h,h); Q=kernel(bg,ag)*clock(eh,h)\n'
    '        K=(K+K.T)*.5\n'
    "        stamps=pd.to_datetime(TT[tr],format='%Y%m%d%H')\n"
    "        age=tensor(np.asarray((stamps.max()-stamps).total_seconds()/86400,dtype='f4'),d)\n"
    '        # Edad calculada solo desde fechas del entrenamiento. Un dia mas aproxima\n'
    '        # el comienzo del siguiente periodo de calibracion (igual en todos los folds).\n'
    '        age=age+1.\n'
    "        target=tensor(((Y[tr]-st['clim'][H[tr]])/st['scale']).reshape(len(tr),-1),d)\n"
    '        preds=[]\n'
    '        for alpha,days in CLOCK_SETTINGS:\n'
    '            Kreg=K.clone(); Kreg.diagonal().add_(alpha*torch.exp(age/days))\n'
    '            L=torch.linalg.cholesky(Kreg); dual=torch.cholesky_solve(target,L)\n'
    '            p=(Q@dual).cpu().numpy().reshape(len(we),70,35,4)\n'
    "            preds.append(clip(p*st['scale']+st['clim'][he]))\n"
    "            if tag=='final':\n"
    "                np.savez_compressed(OUT/f'clock_alpha_{alpha}.npz',features=a,mu=mu,sd=sd,dual=dual.cpu().numpy(),hours=H[tr],alpha=alpha,days=days)\n"
    '            del Kreg,L,dual,p\n'
    "            log(f'{tag} Clock alpha={alpha} days={days} done elapsed={time.perf_counter()-t:.2f}s')\n"
    '        return clip(np.mean(preds,axis=0,dtype=np.float32))\n'
    '\n'
    '    def old_weights(p,y):\n'
    "        rw,alpha=calibrate(p,y,'Hybrid')\n"
    '        # Convertir mezcla jerarquica v2.3 en pesos efectivos por variable.\n'
    "        return np.stack([(1-alpha)*(1-rw),(1-alpha)*rw,.375*alpha,.375*alpha,.25*alpha,np.zeros(4,dtype='f4')])\n"
    '\n'
    '    def weighted(p,w):\n'
    "        return clip(sum(p[k]*w[i].astype('f4') for i,k in enumerate(KEYS)))\n"
    '\n'
    '    def fit_method(p,y,method):\n'
    '        w=old_weights(p,y)\n'
    "        if method=='Baseline': return w\n"
    "        if method=='ClockReplace':\n"
    '            # Misma arquitectura y calibracion que v2.3, sustituyendo 75% del\n'
    '            # componente kernel por la variante horaria/reciente.\n'
    "            ref=mix(p['CNN'],p['ResNet'],fit_alpha(p['CNN'],p['ResNet'],y))\n"
    "            fresh=.375*p['Mutalisk']+.375*p['Viper']+.0625*p['Kernel']+.1875*p['Clock']\n"
    '            alpha=fit_alpha(ref,fresh,y,prior=.25,shrink=.35,cap=.65)\n'
    "            rw=fit_alpha(p['CNN'],p['ResNet'],y)\n"
    '            return np.stack([(1-alpha)*(1-rw),(1-alpha)*rw,.375*alpha,.375*alpha,.0625*alpha,.1875*alpha])\n'
    "        if method=='ClockBlend':\n"
    "            a=fit_alpha(weighted(p,w),p['Clock'],y,prior=.15,shrink=.4,cap=.55)\n"
    '            w=w*(1-a); w[-1]+=a; return w\n'
    "        if method!='Stack': raise ValueError(method)\n"
    '        # Regresion convexa multisalida. Se ajusta solo en la semana de calibracion.\n'
    '        # Penalizacion en escala de RMSE oficial y limites para evitar pesos extremos.\n'
    '        out=[]\n'
    '        for v in range(4):\n'
    "            prior=w[:,v].astype('f8')*.85; prior[-1]+=.15\n"
    '            # Residuos normalizados, matrices de solo 6x6.\n'
    "            a=np.stack([((p[k][...,v]-y[...,v])/NORM[v]).reshape(-1) for k in KEYS],axis=0).astype('f8')\n"
    '            gram=(a@a.T)/a.shape[1]; reg=.004\n'
    '            def fun(z): return float(z@gram@z+reg*np.square(z-prior).sum())\n'
    '            def jac(z): return 2*(gram@z+reg*(z-prior))\n'
    '            bounds=[(max(0,float(t)-.25),min(1,float(t)+.25)) for t in prior]; bounds[-1]=(0,.55)\n'
    "            opt=minimize(fun,prior,jac=jac,bounds=bounds,constraints=[dict(type='eq',fun=lambda z:z.sum()-1,jac=lambda z:np.ones(6))],method='SLSQP',options=dict(maxiter=100,ftol=1e-11))\n"
    '            z=opt.x if opt.success and np.isfinite(opt.x).all() and abs(opt.x.sum()-1)<1e-6 else prior\n'
    '            z=np.maximum(z,0); out.append(z/z.sum())\n'
    "        return np.asarray(out,dtype='f4').T\n"
    '\n'
    '    # Cache de predicciones de calibracion y validacion para futuras versiones.\n'
    '    # Las etiquetas se obtienen exclusivamente de train.npz.\n'
    '    def fold_run(label,cut,methods):\n'
    '        cal,start,end=cut\n'
    '        tr=np.flatnonzero(TT<cal); ca=np.flatnonzero((TT>=cal)&(TT<start)&np.isin(H,HOURS)); ev=np.flatnonzero((TT>=start)&(TT<end)&np.isin(H,HOURS))\n'
    '        if SMOKE: tr,ca,ev=tr[:48],ca[:4],ev[:4]\n'
    '        assert len(tr) and len(ca) and len(ev) and max(TT[tr])<min(TT[ca])<min(TT[ev])\n'
    "        log(f'FOLD {label} train={len(tr)} cal={len(ca)} eval={len(ev)} {cal}/{start}/{end}')\n"
    "        ix=np.r_[ca,ev]; p=run_stage(tr,W[ix],H[ix],f'fold{label}',FAMILIES)\n"
    '        pc={k:v[:len(ca)].copy() for k,v in p.items()}; pe={k:v[len(ca):] for k,v in p.items()}\n'
    '        scores={}; rows=[]\n'
    '        for method in methods:\n'
    '            weights=fit_method(pc,Y[ca],method); pred=weighted(pe,weights); s,r=score(pred,Y[ev]); scores[method]=s\n'
    "            rows.append(dict(fold=str(label),model=method,skill=s,**dict(zip(['RMSE_'+v for v in VARS],r.tolist()))))\n"
    "            log(f'FOLD {label} {method:13s} skill={s:.6f} RMSE={np.round(r,4).tolist()} weights={weights.round(3).tolist()}')\n"
    "        if os.environ.get('ULTRALISK_CACHE','1')=='1':\n"
    "            np.savez(OUT/f'fold{label}_predictions.npz',times=TT[ix],n_cal=len(ca),target=Y[ix],**p)\n"
    '        return scores,rows,(pc,Y[ca])\n'
    '\n'
    '    ROWS=[]; CAL=[]; DEV=[]\n'
    '    for f,cut in enumerate(CUTS[:2],1):\n'
    "        sc,rows,cal=fold_run(f,cut,['Baseline']+METHODS); DEV.append(sc); ROWS+=rows; CAL.append(cal)\n"
    "        pd.DataFrame(ROWS).to_csv(OUT/'validation_metrics.csv',index=False)\n"
    "    dev_gains={m:[s[m]-s['Baseline'] for s in DEV] for m in METHODS}\n"
    "    repro=all(abs(s['Baseline']-OLD_FOLD[i])<=REPRO_TOL for i,s in enumerate(DEV,1))\n"
    '    eligible=[m for m,g in dev_gains.items() if min(g)>=0 and np.mean(g)>=MIN_DEV]\n'
    "    selected=max(eligible,key=lambda m:np.mean(dev_gains[m])) if eligible else 'Baseline'\n"
    "    log(f'FROZEN BEFORE CHECKS selected={selected} gains={dev_gains} baseline_reproduced={repro}')\n"
    "    methods=list(dict.fromkeys(['Baseline',selected]))\n"
    '    # Junio III ya fue observado durante el desarrollo de v2.3: regresion historica,\n'
    '    # no un nuevo test ciego. Abril es un corte adicional de estres cronologico.\n'
    '    audit,rows,cal=fold_run(3,CUTS[2],methods); ROWS+=rows; CAL.append(cal)\n'
    "    pd.DataFrame(ROWS).to_csv(OUT/'validation_metrics.csv',index=False)\n"
    "    repro=repro and abs(audit['Baseline']-OLD_FOLD[3])<=REPRO_TOL\n"
    "    stress,rows,_=fold_run('April',STRESS,methods); ROWS+=rows\n"
    "    pd.DataFrame(ROWS).to_csv(OUT/'validation_metrics.csv',index=False)\n"
    "    gain=audit[selected]-audit['Baseline']; stress_gain=stress[selected]-stress['Baseline']\n"
    "    PASS=selected!='Baseline' and repro and gain>=MIN_AUDIT and stress_gain>=-STRESS_TOL and not SMOKE\n"
    "    log(f'GATE selected={selected} june_gain={gain:+.6f} april_gain={stress_gain:+.6f} reproduced={repro} recommend={PASS}')\n"
    '    # La calibracion final concatena solo las tres semanas originales, no evaluaciones.\n'
    '    # No se usa el puntaje publico como etiqueta ni para optimizar pesos.\n'
    '    cal_p={k:np.concatenate([p[k] for p,y in CAL]) for k in KEYS}; cal_y=np.concatenate([y for p,y in CAL]); weights=fit_method(cal_p,cal_y,selected)\n'
    "    log(f'FINAL weights keys={KEYS} weights={weights.tolist()}')\n"
    '    tr=np.argsort(TT); eval_t=np.unique(ti)\n'
    '    if SMOKE: tr=tr[:48]\n'
    "    p=run_stage(tr,WT[eval_t],HT[eval_t],'final',FAMILIES)\n"
    '    forecast=weighted(p,weights)\n'
    '    positions=np.full(len(TE),-1,dtype=int); positions[eval_t]=np.arange(len(eval_t))\n'
    "    values=forecast[positions[ti],ri,ci]; sub=pd.DataFrame(values.round(4),columns=VARS); sub.insert(0,'id',TEMPLATE.id)\n"
    "    name='submission_ultralisk_v2_4.csv' if PASS else 'submission_v2_4_NO_SUBIR.csv'\n"
    "    if SMOKE: name='submission_SMOKE_NO_SUBIR.csv'\n"
    "    dest=OUT/name; tmp=dest.with_suffix('.tmp'); sub.to_csv(tmp,index=False)\n"
    "    check=pd.read_csv(tmp,dtype={'id':str}); assert check.shape==TEMPLATE.shape and check.columns.tolist()==TEMPLATE.columns.tolist() and check.id.equals(TEMPLATE.id) and check.id.is_unique\n"
    '    assert np.isfinite(check[VARS].to_numpy()).all() and check.RH.between(0,100).all(); tmp.replace(dest)\n'
    "    if os.environ.get('ULTRALISK_CACHE','1')=='1': np.savez(OUT/'final_family_predictions.npz',times=TE[eval_t],**p)\n"
    "    report=dict(model='Ultralisk-v2.4 Chronos',smoke=SMOKE,selected=selected,reference_public_score=PUBLIC_REFERENCE,reference_reproduced=repro,dev_gains=dev_gains,audit_gain=gain,stress_gain=stress_gain,recommend_submit=bool(PASS),keys=KEYS,weights=weights.tolist(),clock_settings=CLOCK_SETTINGS,validation=ROWS,benchmark=BENCH,runs=ALL_RUNS,sha256=hashlib.sha256(dest.read_bytes()).hexdigest(),rows=len(check),seconds=time.perf_counter()-START)\n"
    "    (OUT/'report.json').write_text(json.dumps(report,indent=2))\n"
    "    print(pd.DataFrame(ROWS).pivot(index='model',columns='fold',values='skill').round(6).to_string())\n"
    '    log(f\'CSV={dest} SHA256={report["sha256"]} rows={len(check)}\')\n'
    "    log('RECOMMEND_SUBMIT=YES' if PASS else 'RECOMMEND_SUBMIT=NO: conservar CSV v2.3 (0.49056)')\n"
    "    for f in (dest,OUT/'report.json',OUT/'validation_metrics.csv',LOG): display(FileLink(os.path.relpath(f)))\n"
    'else:\n'
    "    print('FASE 1/2: cache v2.4 existente; no se reentrenan redes.',flush=True)\n"
    "print('FASE 2/2: calibracion final Ultralisk-v2.7',flush=True)\n"
    '\n'
    '# Ultralisk-v2.7 | ShadowBytes | final calibration\n'
    'import os, time, json, copy, hashlib, threading, platform\n'
    'from pathlib import Path\n'
    'from concurrent.futures import ThreadPoolExecutor\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    'import torch\n'
    'from torch import nn\n'
    'from torch.nn import functional as F\n'
    'from IPython.display import display, FileLink\n'
    "SMOKE=os.environ.get('ULTRALISK_SMOKE')=='1'  # prueba tecnica, NO subir\n"
    "OUT=Path(os.environ.get('ULTRALISK_V27_OUT','/kaggle/working/Ultralisk-v2.7')); OUT.mkdir(parents=True,exist_ok=True)\n"
    "LOG=OUT/'training.log'; LOG.write_text(''); LOCK=threading.Lock(); START=time.perf_counter()\n"
    'BATCH=32; EPOCHS_REF=25; EPOCHS_NEW=36\n'
    "SEEDS={'CNN':(0,1,2),'ResNet':(0,1,2),'Mutalisk':(42,123),'Viper':(42,123)}\n"
    "CUTS=[('2024050900','2024051600','2024060100'),('2024052500','2024060100','2024061600'),('2024060900','2024061600','2024070100')]\n"
    "VARS=['RH','T','U','V']; NORM=np.array([25.360,2.0652,2.6776,2.0876])\n"
    '# Valores historicos de versions/v3/backtest.json (solo referencia, no etiquetas).\n'
    'HISTORICAL={1:.42292221906027727,2:.4136267261457105,3:.42723285641743414}\n'
    'MIN_DEV_GAIN=.003; MIN_AUDIT_GAIN=.002\n'
    'def log(msg):\n'
    "    line=f'[{time.perf_counter()-START:8.1f}s] {msg}'\n"
    '    with LOCK:\n'
    '        print(line,flush=True)\n'
    "        with LOG.open('a') as f: f.write(line+'\\n')\n"
    "DEVICES=[torch.device('cuda:0')] if torch.cuda.is_available() and not SMOKE else [torch.device('cpu')]\n"
    'torch.set_num_threads(2); torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True\n'
    'cpu_name=platform.processor()\n'
    "if Path('/proc/cpuinfo').is_file():\n"
    "    cpu_name=next((l.split(':',1)[1].strip() for l in Path('/proc/cpuinfo').read_text().splitlines() if l.startswith('model name')),cpu_name)\n"
    "log(f'torch={torch.__version__} CPU={cpu_name} cores={os.cpu_count()} torch_threads={torch.get_num_threads()} smoke={SMOKE}')\n"
    'for d in DEVICES:\n'
    "    if d.type=='cuda': log(f'{d} {torch.cuda.get_device_name(d)} VRAM={torch.cuda.get_device_properties(d).total_memory/2**30:.2f} GiB')\n"
    '# Datos oficiales\n'
    "root=Path(os.environ.get('ULTRALISK_DATA','/kaggle/input')); required=('train.npz','test.npz','static.npz','sample_submission.csv')\n"
    "folders=[p.parent for p in root.rglob('train.npz') if all((p.parent/k).is_file() for k in required)]\n"
    "official=[p for p in folders if 'hackathon-ia-pucp-2026' in str(p)]\n"
    'if len(official)==1: DATA=official[0]\n'
    'elif len(folders)==1: DATA=folders[0]\n'
    "else: raise RuntimeError(f'Dataset oficial no identificado: {folders}')\n"
    "with np.load(DATA/'train.npz') as f: X,Y,TT=f['X'].astype('f4'),f['Y'].astype('f4'),f['times'].astype(str)\n"
    "with np.load(DATA/'test.npz') as f: XT,TE=f['X'].astype('f4'),f['times'].astype(str)\n"
    "with np.load(DATA/'static.npz') as f:\n"
    "    DEM=f['dem'].astype('f4'); glat,glon,wlat,wlon=[f[k] for k in ('gfs_lats','gfs_lons','wrf_lats','wrf_lons')]\n"
    "TEMPLATE=pd.read_csv(DATA/'sample_submission.csv',dtype={'id':str})\n"
    'assert X.shape==(2935,17,13,4) and Y.shape==(2935,70,35,4) and XT.shape==(792,17,13,4) and DEM.shape==(70,35)\n'
    "assert TEMPLATE.shape==(646800,5) and TEMPLATE.columns.tolist()==['id',*VARS] and TEMPLATE.id.is_unique\n"
    'assert all(np.isfinite(a).all() for a in (X,Y,XT,DEM)) and max(TT)<min(TE)\n'
    "parts=TEMPLATE.id.str.split('_',expand=True); ids_t=parts[0].map({s:i for i,s in enumerate(TE)}); assert ids_t.notna().all()\n"
    'ti=ids_t.to_numpy(dtype=int); ri=parts[1].astype(int).to_numpy(); ci=parts[2].astype(int).to_numpy()\n'
    'assert (ri>=0).all() and (ri<70).all() and (ci>=0).all() and (ci<35).all()\n'
    'assert len(np.unique(ti))==264 and np.all(np.bincount(ti)[np.unique(ti)]==2450)\n'
    'HOURS=np.unique([int(s[-2:]) for s in parts[0].unique()]); H=np.array([int(s[-2:]) for s in TT]); HT=np.array([int(s[-2:]) for s in TE])\n'
    "log(f'Data={DATA} train={X.shape} test={XT.shape} horas={HOURS.tolist()}')\n"
    'def windows(x,times):\n'
    "    dates=pd.to_datetime(times,format='%Y%m%d%H'); lookup={t:i for i,t in enumerate(dates)}\n"
    '    return x[np.array([[lookup.get(t-pd.Timedelta(hours=k),i) for k in (2,1,0)] for i,t in enumerate(dates)])]\n'
    'W,WT=windows(X,TT),windows(XT,TE)\n'
    "lat,lon=np.meshgrid(wlat,wlon,indexing='ij'); grid=np.stack([(lon-glon[0])/(glon[-1]-glon[0])*2-1,(lat-glat[0])/(glat[-1]-glat[0])*2-1],-1)[None].astype('f4')\n"
    "def z(a): return ((a-a.mean())/max(float(a.std()),1e-6)).astype('f4')\n"
    'def smooth(a,k):\n'
    "    p=k//2; return F.avg_pool2d(F.pad(torch.from_numpy(a[None,None]),(p,p,p,p),mode='replicate'),k,stride=1)[0,0].numpy()\n"
    'dy,dx=np.gradient(DEM)\n'
    "geo=np.stack([z(DEM),z(dx),z(dy),z(np.hypot(dx,dy)),z(DEM-smooth(DEM,3)),z(DEM-smooth(DEM,7)),z(DEM-smooth(DEM,15)),z(lat),z(lon),z(np.gradient(dy,axis=0)+np.gradient(dx,axis=1))])[None].astype('f4')\n"
    'def stats(ix):\n'
    '    yy,xx,hh=Y[ix],X[ix],H[ix]\n'
    '    def climate(a): return np.stack([a[hh==h].mean(0) if np.any(hh==h) else a.mean(0) for h in range(24)])\n'
    '    cl=climate(yy)\n'
    '    return dict(mu=xx.mean((0,1,2)),sd=np.maximum(xx.std((0,1,2)),1e-6),clim=cl,gfs_clim=climate(xx),scale=np.maximum((yy-cl[hh]).std((0,1,2)),1e-6))\n'
    'def score(p,y):\n'
    "    rm=np.sqrt(np.mean(np.square(p.astype('f8')-y),axis=(0,1,2))); return float(1-(rm/NORM).mean()),rm\n"
    'def clip(p):\n'
    "    p=p.astype('f4',copy=False); p[...,0]=p[...,0].clip(0,100); assert np.isfinite(p).all(); return p\n"
    'def tensor(a,d): return torch.as_tensor(a,dtype=torch.float32,device=d)\n'
    'def sync(d):\n'
    "    if d.type=='cuda': torch.cuda.synchronize(d)\n"
    'def kernel_features(w,h,st):\n'
    "    a=(w-st['gfs_clim'][(h[:,None]-np.array([2,1,0]))%24])/st['sd']; a=a.reshape(len(w),3,-1)\n"
    "    return np.concatenate([a[:,2],a.mean(1),.5*(a[:,2]-a[:,0])],axis=1).astype('f4')\n"
    'def kernel(a,b):\n'
    '    dot=(a@b.T)/a.shape[1]; dist=(a.square().mean(1)[:,None]+b.square().mean(1)[None,:]-2*dot).clamp_min(0)\n'
    '    return .5*dot+.25*dot.square()+.25*torch.exp(-dist/2)\n'
    "CANDIDATES={'Reference':{},'Mutalisk':{'Mutalisk':1.},'Viper':{'Viper':1.},'Neural':{'Mutalisk':.5,'Viper':.5},'Kernel':{'Kernel':1.},'Hybrid':{'Mutalisk':.375,'Viper':.375,'Kernel':.25}}\n"
    'def fit_alpha(a,b,y,prior=.5,shrink=.2,cap=1.):\n'
    "    d=b.astype('f8')-a; r=y.astype('f8')-a; den=(d*d).mean((0,1,2)); num=(r*d).mean((0,1,2))\n"
    "    raw=np.divide(num,den,out=np.full(4,prior),where=den>1e-12); return np.clip((1-shrink)*raw+shrink*prior,0,cap).astype('f4')\n"
    'def mix(a,b,w): return clip(a*(1-w)+b*w)\n'
    'def candidate(p,name): return sum(p[k]*np.float32(w) for k,w in CANDIDATES[name].items())\n'
    'def calibrate(p,y,name):\n'
    "    rw=fit_alpha(p['CNN'],p['ResNet'],y); ref=mix(p['CNN'],p['ResNet'],rw)\n"
    "    alpha=np.zeros(4,dtype='f4') if name=='Reference' else fit_alpha(ref,candidate(p,name),y,prior=.25,shrink=.35,cap=.65)\n"
    '    return rw,alpha\n'
    'def combine(p,name,rw,alpha):\n'
    "    ref=mix(p['CNN'],p['ResNet'],rw); return ref if name=='Reference' else mix(ref,candidate(p,name),alpha)\n"
    '\n'
    '\n'
    'from scipy.optimize import minimize\n'
    'PUBLIC_REFERENCE=.49056\n'
    'OLD_FOLD={1:.439850572420165,2:.4396332592736687,3:.4496978946404211}\n'
    "STRESS=('2024040900','2024041600','2024050100')\n"
    '# Dos variantes conservadoras, probadas individualmente en CPU en folds 1 y 2.\n'
    'CLOCK_SETTINGS=[(.6,90),(1.2,90)]\n'
    "FAMILIES=['CNN','ResNet','Mutalisk','Viper','Kernel']\n"
    "KEYS=FAMILIES+['Clock']\n"
    "METHODS=['ClockReplace','ClockBlend','Stack']\n"
    'MIN_DEV=.002; MIN_AUDIT=.002; STRESS_TOL=.002; REPRO_TOL=.0015\n'
    '\n'
    'def old_weights(p,y):\n'
    "    rw,alpha=calibrate(p,y,'Hybrid')\n"
    '    # Convertir mezcla jerarquica v2.3 en pesos efectivos por variable.\n'
    "    return np.stack([(1-alpha)*(1-rw),(1-alpha)*rw,.375*alpha,.375*alpha,.25*alpha,np.zeros(4,dtype='f4')])\n"
    '\n'
    'def weighted(p,w):\n'
    "    return clip(sum(p[k]*w[i].astype('f4') for i,k in enumerate(KEYS)))\n"
    '\n'
    'def fit_method(p,y,method):\n'
    '    w=old_weights(p,y)\n'
    "    if method=='Baseline': return w\n"
    "    if method=='ClockReplace':\n"
    '        # Misma arquitectura y calibracion que v2.3, sustituyendo 75% del\n'
    '        # componente kernel por la variante horaria/reciente.\n'
    "        ref=mix(p['CNN'],p['ResNet'],fit_alpha(p['CNN'],p['ResNet'],y))\n"
    "        fresh=.375*p['Mutalisk']+.375*p['Viper']+.0625*p['Kernel']+.1875*p['Clock']\n"
    '        alpha=fit_alpha(ref,fresh,y,prior=.25,shrink=.35,cap=.65)\n'
    "        rw=fit_alpha(p['CNN'],p['ResNet'],y)\n"
    '        return np.stack([(1-alpha)*(1-rw),(1-alpha)*rw,.375*alpha,.375*alpha,.0625*alpha,.1875*alpha])\n'
    "    if method=='ClockBlend':\n"
    "        a=fit_alpha(weighted(p,w),p['Clock'],y,prior=.15,shrink=.4,cap=.55)\n"
    '        w=w*(1-a); w[-1]+=a; return w\n'
    "    if method!='Stack': raise ValueError(method)\n"
    '    # Regresion convexa multisalida. Se ajusta solo en la semana de calibracion.\n'
    '    # Penalizacion en escala de RMSE oficial y limites para evitar pesos extremos.\n'
    '    out=[]\n'
    '    for v in range(4):\n'
    "        prior=w[:,v].astype('f8')*.85; prior[-1]+=.15\n"
    '        # Residuos normalizados, matrices de solo 6x6.\n'
    "        a=np.stack([((p[k][...,v]-y[...,v])/NORM[v]).reshape(-1) for k in KEYS],axis=0).astype('f8')\n"
    '        gram=(a@a.T)/a.shape[1]; reg=.004\n'
    '        def fun(z): return float(z@gram@z+reg*np.square(z-prior).sum())\n'
    '        def jac(z): return 2*(gram@z+reg*(z-prior))\n'
    '        bounds=[(max(0,float(t)-.25),min(1,float(t)+.25)) for t in prior]; bounds[-1]=(0,.55)\n'
    "        opt=minimize(fun,prior,jac=jac,bounds=bounds,constraints=[dict(type='eq',fun=lambda z:z.sum()-1,jac=lambda z:np.ones(6))],method='SLSQP',options=dict(maxiter=100,ftol=1e-11))\n"
    '        z=opt.x if opt.success and np.isfinite(opt.x).all() and abs(opt.x.sum()-1)<1e-6 else prior\n'
    '        z=np.maximum(z,0); out.append(z/z.sum())\n'
    "    return np.asarray(out,dtype='f4').T\n"
    '\n'
    'def regional(d,st,tr,we,he,alpha=1.2,sigma=.65,local=.7,days=90,save_model=False,model_prefix="regional"):\n'
    '    a=kernel_features(W[tr],H[tr],st); b=kernel_features(we,he,st); mu=a.mean(0); sd=np.maximum(a.std(0),.15)\n'
    '    ag,bg=tensor((a-mu)/sd,d),tensor((b-mu)/sd,d)\n'
    '    G=kernel(ag,ag); Q=kernel(bg,ag)\n'
    '    h,eh=tensor(H[tr],d),tensor(he,d)\n'
    '    clock=lambda a,b: .4+.6*(1+torch.cos((a[:,None]-b[None,:])*np.pi/12))/2\n'
    '    C,D=clock(h,h),clock(eh,h)\n'
    "    stamps=pd.to_datetime(TT[tr],format='%Y%m%d%H'); age=tensor(np.asarray((stamps.max()-stamps).total_seconds()/86400,dtype='f4')+1,d)\n"
    "    yy=((Y[tr]-st['clim'][H[tr]])/st['scale']).reshape(len(tr),-1,4); pred=np.zeros((len(we),2450,4),np.float32); total=np.zeros(2450,np.float32)\n"
    "    gy,gx=np.meshgrid(glat,glon,indexing='ij'); fy,fx=lat.ravel(),lon.ravel()\n"
    '    if save_model: np.savez(OUT/f"{model_prefix}_preprocessing.npz",**st,features=(a-mu)/sd,feature_mu=mu,feature_sd=sd,train_hours=H[tr],train_times=TT[tr],alpha=alpha,sigma=sigma,local=local,days=days)\n'
    '    anchor=0\n'
    '    for cy in np.linspace(wlat[0],wlat[-1],3):\n'
    '        for cx in np.linspace(wlon[0],wlon[-1],2):\n'
    '            ww=np.exp(-.5*((gy-cy)**2+(gx-cx)**2)/sigma**2); ww=(ww/max(ww.mean(),1e-6))**.5; w=tensor(np.tile(np.repeat(ww.ravel(),4),3),d)\n'
    '            k=((1-local)*G+local*kernel(ag*w,ag*w))*C; q=((1-local)*Q+local*kernel(bg*w,ag*w))*D\n'
    '            k=(k+k.T)*.5; k.diagonal().add_(alpha*torch.exp(age/days)); L=torch.linalg.cholesky(k)\n'
    '            blend=np.maximum(1-np.abs(fy-cy)/((wlat[-1]-wlat[0])/2),0)*np.maximum(1-np.abs(fx-cx)/(wlon[-1]-wlon[0]),0)\n'
    '            ids=np.flatnonzero(blend>1e-6); target=tensor(yy[:,ids].reshape(len(tr),-1),d); dual=torch.cholesky_solve(target,L)\n'
    '            if save_model: np.savez(OUT/f"{model_prefix}_sector_{anchor}.npz",indices=ids,blend=blend[ids],feature_weights=w.cpu().numpy(),dual=dual.cpu().numpy(),center=[cy,cx])\n'
    '            anchor+=1\n'
    '            pred[:,ids]+= (q@dual).cpu().numpy().reshape(len(we),len(ids),4)*blend[ids,None]; total[ids]+=blend[ids]\n'
    '    assert total.min()>.99\n'
    '    p=pred.reshape(len(we),70,35,4)/total.reshape(1,70,35,1)\n'
    "    return clip(p*st['scale']+st['clim'][he])\n"
    '\n'
    '\n'
    '# ===================================================================\n'
    '# Usa las predicciones de cache del entrenamiento v2.4.\n'
    '# No vuelve a entrenar las diez redes neuronales.\n'
    '# ===================================================================\n'
    '# ULTRALISK 2.7 - calibracion final. No se realizan submissions.\n'
    'from scipy.ndimage import gaussian_filter\n'
    "CACHE=Path(os.environ.get('ULTRALISK_CHRONOS_DIR','/kaggle/working/Ultralisk-v2.4'))\n"
    "report24=json.loads((CACHE/'report.json').read_text())\n"
    "assert report24['model']=='Ultralisk-v2.4 Chronos' and bool(report24['smoke'])==SMOKE\n"
    "ANCHOR=report24['selected'] if report24['recommend_submit'] and not SMOKE else 'Baseline'\n"
    "assert ANCHOR in ['Baseline']+METHODS\n"
    "KEYS27=KEYS+['Regional','Narrow']; PREV=['RegionalFixed','RegionalAdaptive','RegionalReplace']; NEW=['NarrowFixed','NarrowAdaptive','NarrowReplace','NarrowStack','NarrowSpatial','NarrowCorrected']\n"
    'MIN_GAIN=.002; STRESS_TOL=.002\n'
    "log(f'Cache={CACHE}; anchor v2.4={ANCHOR}; smoke={SMOKE}')\n"
    "for f in ['submission_ultralisk_v2_7.csv','submission_v2_7_NO_SUBIR.csv','submission_SMOKE_NO_SUBIR.csv']:\n"
    '    (OUT/f).unlink(missing_ok=True)\n'
    '\n'
    'def predict27(p,w):\n'
    '    assert w.shape==(len(KEYS27),4) and np.isfinite(w).all() and (w>=-1e-7).all() and np.allclose(w.sum(0),1,atol=1e-5)\n'
    "    return clip(sum(p[k]*w[i].astype('f4') for i,k in enumerate(KEYS27)))\n"
    '\n'
    "def padded(w): return np.vstack([w,np.zeros((2,4),dtype='f4')])\n"
    'def add_component(w,j,a):\n'
    '    w=w*(1-a); w[j]+=a; return w\n'
    '\n'
    'def calibration(p,y,method,h):\n'
    "    w=padded(fit_method(p,y,'Baseline' if method=='V23' else ANCHOR))\n"
    "    if method in ('V23','Anchor'): return w,None\n"
    "    j=6 if method.startswith('Regional') else 7\n"
    "    if method.endswith('Replace'):\n"
    '        a=.75*(w[4]+w[5]); w[4:6]*=.25; w[j]+=a\n'
    "    elif method.endswith('Fixed'): w=add_component(w,j,np.full(4,.20,dtype='f4'))\n"
    "    elif method in ('NarrowAdaptive','RegionalAdaptive','NarrowSpatial','NarrowCorrected','NarrowStack'):\n"
    '        a=fit_alpha(predict27(p,w),p[KEYS27[j]],y,prior=.20,shrink=.50,cap=.50)\n'
    '        w=add_component(w,j,a)\n'
    '    else: raise ValueError(method)\n'
    "    if method=='NarrowStack':\n"
    '        result=[]\n'
    '        for v in range(4):\n'
    "            prior=w[:,v].astype('f8'); prior=prior*.95; prior[6]+=.05\n"
    '            # Regularizar frente a la mezcla jerarquica original, en unidades oficiales.\n'
    "            e=np.stack([((p[k][...,v]-y[...,v])/NORM[v]).ravel() for k in KEYS27]).astype('f8'); gram=e@e.T/e.shape[1]; reg=.012\n"
    '            bounds=[(max(0,float(x)-.18),min(1,float(x)+.18)) for x in prior]; bounds[7]=(0,.55)\n'
    "            opt=minimize(lambda z:float(z@gram@z+reg*np.square(z-prior).sum()),prior,jac=lambda z:2*(gram@z+reg*(z-prior)),bounds=bounds,constraints=[{'type':'eq','fun':lambda z:z.sum()-1,'jac':lambda z:np.ones(len(KEYS27))}],method='SLSQP',options={'maxiter':100,'ftol':1e-11})\n"
    '            q=opt.x if opt.success and np.isfinite(opt.x).all() and abs(opt.x.sum()-1)<1e-6 else prior; q=np.maximum(q,0); result.append(q/q.sum())\n'
    "        w=np.array(result,dtype='f4').T\n"
    '    maps=None\n'
    "    if method in ('NarrowSpatial','NarrowCorrected'):\n"
    '        # Dos alternativas: corregir el ensemble o solamente el kernel regional.\n'
    "        e=y-(p['Narrow'] if method=='NarrowCorrected' else predict27(p,w)); glob=gaussian_filter(e.mean(0),sigma=(3,3,0)); maps=[]\n"
    '        for hour in range(24):\n'
    '            hw=np.exp(1.5*(np.cos((h-hour)*np.pi/12)-1)); local=np.average(e,axis=0,weights=hw)\n'
    '            maps.append(.35*(.5*glob+.5*gaussian_filter(local,sigma=(3,3,0))))\n'
    "        maps=np.asarray(maps,dtype='f4')\n"
    "        if method=='NarrowCorrected': maps*=w[7]\n"
    '    return w,maps\n'
    '\n'
    'def apply27(p,params,h):\n'
    '    w,b=params; pred=predict27(p,w); return clip(pred if b is None else pred+b[h])\n'
    '\n'
    'ROWS=[]; CAL=[]; SCORES={}; REPRO=[]\n'
    'def run_fold(label,cut,methods):\n'
    '    cal,start,end=cut\n'
    '    tr=np.flatnonzero(TT<cal); ca=np.flatnonzero((TT>=cal)&(TT<start)&np.isin(H,HOURS)); ev=np.flatnonzero((TT>=start)&(TT<end)&np.isin(H,HOURS))\n'
    '    if SMOKE: tr,ca,ev=tr[:48],ca[:4],ev[:4]\n'
    '    ix=np.r_[ca,ev]; n=len(ca); assert len(tr) and n and len(ev) and max(TT[tr])<min(TT[ca])<min(TT[ev])\n'
    "    with np.load(CACHE/f'fold{label}_predictions.npz',allow_pickle=False) as f:\n"
    "        assert np.array_equal(f['times'],TT[ix]) and int(f['n_cal'])==n and np.array_equal(f['target'],Y[ix]), 'Cache no corresponde al dataset o corte'\n"
    "        p={k:f[k].astype('f4') for k in KEYS}\n"
    '    assert all(v.shape==(len(ix),70,35,4) and np.isfinite(v).all() for v in p.values())\n'
    '    st=stats(tr); t=time.perf_counter()\n'
    "    p['Regional']=regional(DEVICES[0],st,tr,W[ix],H[ix],alpha=1.2,sigma=.65,local=.7,days=90)\n"
    "    p['Narrow']=regional(DEVICES[0],st,tr,W[ix],H[ix],alpha=1.2,sigma=.4,local=.85,days=90)\n"
    "    log(f'FOLD {label} regional kernels done {time.perf_counter()-t:.2f}s train={len(tr)} cal={n} eval={len(ev)}')\n"
    '    pc={k:v[:n].copy() for k,v in p.items()}; pe={k:v[n:] for k,v in p.items()}; scores={}\n'
    "    for m in dict.fromkeys(['V23','Anchor',*methods,'RegionalIndividual','NarrowIndividual']):\n"
    '        params=None\n'
    "        if m.endswith('Individual'): pred=pe[m[:-10]]\n"
    '        else: params=calibration(pc,Y[ca],m,H[ca]); pred=apply27(pe,params,H[ev])\n'
    '        s,r=score(pred,Y[ev]); scores[m]=s\n'
    "        ROWS.append(dict(fold=str(label),model=m,skill=s,**dict(zip(['RMSE_'+v for v in VARS],r.tolist()))))\n"
    "        log(f'FOLD {label} {m:20s} skill={s:.6f} RMSE={r.round(4).tolist()}')\n"
    "    for m,old in [('V23','Baseline'),('Anchor',ANCHOR)]:\n"
    "        expected=[r['skill'] for r in report24['validation'] if str(r['fold'])==str(label) and r['model']==old]\n"
    "        assert len(expected)==1 and abs(scores[m]-expected[0])<2e-6, f'Cache no reproduce {m} fold {label}'\n"
    "    if label in (1,2,3): REPRO.append(abs(scores['V23']-OLD_FOLD[label])<=REPRO_TOL); CAL.append((pc,Y[ca],H[ca]))\n"
    "    SCORES[str(label)]=scores; pd.DataFrame(ROWS).to_csv(OUT/'validation_metrics.csv',index=False)\n"
    '\n'
    'for f,cut in enumerate(CUTS[:2],1): run_fold(f,cut,PREV+NEW)\n'
    'def choose(methods,base,min_gain):\n'
    '    gains={m:[SCORES[str(f)][m]-max(SCORES[str(f)][b] for b in base) for f in (1,2)] for m in methods}\n'
    '    eligible=[m for m,g in gains.items() if min(g)>=0 and np.mean(g)>=min_gain]\n'
    '    return (max(eligible,key=lambda m:np.mean(gains[m])) if eligible else None),gains\n'
    "prev,prev_gains=choose(PREV,['V23','Anchor'],.0015); prev=prev or 'Anchor'\n"
    "baselines=list(dict.fromkeys(['V23','Anchor',prev]))\n"
    'selected,dev_gains=choose(NEW,baselines,MIN_GAIN); selected=selected or prev\n'
    "log(f'FROZEN before retrospective checks: previous={prev} selected={selected} gains={dev_gains}')\n"
    "for f,cut in [(3,CUTS[2]),('April',STRESS)]: run_fold(f,cut,[prev,selected])\n"
    "repro=all(REPRO) and bool(report24['reference_reproduced'])\n"
    "audit_gain=SCORES['3'][selected]-max(SCORES['3'][b] for b in baselines); stress_gain=SCORES['April'][selected]-max(SCORES['April'][b] for b in baselines)\n"
    'PASS=bool(not SMOKE and repro and selected in NEW and audit_gain>=MIN_GAIN and stress_gain>=-STRESS_TOL)\n'
    "log(f'GATE repro={repro} audit_gain={audit_gain:+.6f} stress_gain={stress_gain:+.6f} recommend={PASS}')\n"
    '# Los pesos finales usan exclusivamente las tres semanas de calibracion; no sus evaluaciones.\n'
    'cp={k:np.concatenate([p[k] for p,y,h in CAL]) for k in KEYS27}; cy=np.concatenate([y for p,y,h in CAL]); ch=np.concatenate([h for p,y,h in CAL]); w,maps=calibration(cp,cy,selected,ch)\n'
    "log(f'FINAL weights={dict(zip(KEYS27,w.round(5).tolist()))} spatial_bias={maps is not None}')\n"
    'tr=np.argsort(TT); ev=np.unique(ti)\n'
    'if SMOKE: tr=tr[:48]\n'
    "with np.load(CACHE/'final_family_predictions.npz',allow_pickle=False) as f:\n"
    "    assert np.array_equal(f['times'],TE[ev]); p={k:f[k].astype('f4') for k in KEYS}\n"
    'assert all(v.shape==(len(ev),70,35,4) and np.isfinite(v).all() for v in p.values())\n'
    'st=stats(tr)\n'
    "p['Regional']=regional(DEVICES[0],st,tr,WT[ev],HT[ev],alpha=1.2,sigma=.65,local=.7,days=90,save_model=PASS,model_prefix='regional')\n"
    "p['Narrow']=regional(DEVICES[0],st,tr,WT[ev],HT[ev],alpha=1.2,sigma=.4,local=.85,days=90,save_model=PASS,model_prefix='narrow')\n"
    'forecast=apply27(p,(w,maps),HT[ev]); positions=np.full(len(TE),-1,dtype=int); positions[ev]=np.arange(len(ev)); assert (positions[ti]>=0).all()\n'
    "sub=pd.DataFrame(forecast[positions[ti],ri,ci].round(4),columns=VARS); sub.insert(0,'id',TEMPLATE.id)\n"
    "name='submission_SMOKE_NO_SUBIR.csv' if SMOKE else ('submission_ultralisk_v2_7.csv' if PASS else 'submission_v2_7_NO_SUBIR.csv'); dest=OUT/name; tmp=dest.with_suffix('.tmp'); sub.to_csv(tmp,index=False)\n"
    "check=pd.read_csv(tmp,dtype={'id':str}); assert check.shape==TEMPLATE.shape and check.columns.tolist()==TEMPLATE.columns.tolist() and check.id.equals(TEMPLATE.id) and check.id.is_unique\n"
    'assert np.isfinite(check[VARS].to_numpy()).all() and check.RH.between(0,100).all(); tmp.replace(dest)\n'
    "np.savez(OUT/'calibration.npz',keys=np.asarray(KEYS27),weights=w,bias=np.zeros((0,),dtype='f4') if maps is None else maps)\n"
    "report=dict(model='Ultralisk-v2.7',smoke=SMOKE,anchor=ANCHOR,previous=prev,selected=selected,reference_public_score=.49056,reference_reproduced=repro,dev_gains=dev_gains,previous_dev_gains=prev_gains,audit_gain=audit_gain,stress_gain=stress_gain,recommend_submit=PASS,keys=KEYS27,weights=w.tolist(),spatial_bias=maps is not None,validation=ROWS,sha256=hashlib.sha256(dest.read_bytes()).hexdigest(),rows=len(check),seconds=time.perf_counter()-START)\n"
    "(OUT/'report.json').write_text(json.dumps(report,indent=2)); print(pd.DataFrame(ROWS).pivot(index='model',columns='fold',values='skill').round(6).to_string())\n"
    'log(f\'CSV={dest} rows={len(check)} SHA256={report["sha256"]}\'); log(\'RECOMMEND_SUBMIT=YES\' if PASS else \'RECOMMEND_SUBMIT=NO: conservar v2.3\')\n'
    "for f in (dest,OUT/'report.json',OUT/'validation_metrics.csv',LOG): display(FileLink(os.path.relpath(f)))\n"
)
assert hashlib.sha256(text.encode('utf-8')).hexdigest()=='6049044f014417133664c249d07b0d5003dff688c460db18a975c58e68b237ac'
(CODE/'Ultralisk_v2_7_completo.py').write_bytes(text.encode('utf-8'))
compile(text,'Ultralisk_v2_7_completo.py','exec')

# --- Atlas_v3_4_Kaggle.py: texto original o interfaz de despliegue ---
text=(
    '"""Atlas v3.4 RegimeFlow: retrospective variable routing; no automatic submission."""\n'
    'import os\n'
    "for k in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS'): os.environ[k]='4'\n"
    'import argparse,json\n'
    'from pathlib import Path\n'
    'import numpy as np\n'
    'import pandas as pd\n'
    'import Atlas_v3_3 as m\n'
    'import Atlas_v3_2_5 as a\n'
    "v=a.v; VARS=v.VARS; SSE=['SSE_'+x for x in VARS]\n"
    "AUDIT_SHA='78568dd2c40e4e5d4317fc113fe992eff4e8f4143b9d8c0bbec6d6cbcd9d83a7'\n"
    "BASE_SHA='9ce6e7594d0edbf884e56902fd67e1fa02de1a94ba2f6815fc5c8b5115a801e1'\n"
    "CODE_SHA='58015cb53e74b0f55c4fc1136a98851e7e91f184ea78634359ddca420df09377'\n"
    "ROUTE=('MixDriftBias','MixDriftBias','MixAdaptiveFlow','MixDriftBias') # post-hoc, NOT confirmatory\n"
    "NAMES=['Ultralisk','Reference','MixAdaptiveFlow','MixDriftBias','MixRecentBias','RegimeFlow']\n"
    'def audit(path):\n'
    '    assert v.sha(path)==AUDIT_SHA\n'
    "    d=pd.read_csv(path,low_memory=False);meta={r.field:json.loads(r.value) for r in d[d.section=='metadata'].itertuples()}\n"
    "    assert meta['mode']=='full' and meta['source_sha256']==CODE_SHA and meta['data']==m.OFFICIAL_HASHES and meta['historical_audit_sha256']==m.FULL_AUDIT_SHA\n"
    "    assert {r['origin'] for r in meta['reports']}==set(a.ORIGINS)\n"
    '    records={};rows=[]\n'
    "    for r in meta['reports']:\n"
    "        o=r['origin'];assert r['counts']==m.COUNTS[o] and r['mode']=='full' and r['data']==meta['data'] and r['code_sha256']==CODE_SHA and r['reference']=='MixBiasVar / reconstructed Ultralisk v2.7'\n"
    "        assert r['cache']['train_max']<r['cache']['cut']==a.ORIGINS[o][0]\n"
    "        w=np.asarray(r['weights']);assert w.shape==(4,) and np.all((w>=0)&(w<=.5))\n"
    "        q=d[(d.section=='daily_sse')&(d.origin==o)];rec={}\n"
    "        days=pd.date_range(pd.to_datetime(a.ORIGINS[o][1],format='%Y%m%d%H'),pd.to_datetime(a.ORIGINS[o][2],format='%Y%m%d%H')-pd.Timedelta(days=1)).strftime('%Y%m%d').astype(int).to_numpy()\n"
    '        for n in NAMES[:-1]:\n'
    "            x=q[q.model==n].sort_values('date');assert np.array_equal(x.date.to_numpy(),days)\n"
    "            ns=x.pixels_hours.to_numpy(dtype='int64');ss=x[SSE].to_numpy(dtype='f8');assert np.isfinite(ss).all() and (ss>=0).all() and (ns>=0).all() and (ss[ns==0]==0).all()\n"
    "            if 'counts' in rec:assert np.array_equal(rec['counts'],ns)\n"
    "            rec['counts']=ns;rec[n]=ss\n"
    "            g=d[(d.section=='global')&(d.origin==o)&(d.model==n)];assert len(g)==1 and abs(float(v.skill(ss.sum(0),ns.sum()))-g.skill.iloc[0])<1e-9\n"
    "        assert rec['counts'].sum()==m.COUNTS[o][2]*2450\n"
    "        rec['RegimeFlow']=np.stack([rec[n][:,j] for j,n in enumerate(ROUTE)],axis=1);records[o]=rec\n"
    "    def row(o,n,ss,counts,section='global',**extra):\n"
    "        rm=np.sqrt(ss/counts);return dict(section=section,origin=o,model=n,skill=float(v.skill(ss,counts)),pixels_hours=int(counts),**dict(zip(['NRMSE_'+j for j in VARS],rm/v.NORM)),**extra)\n"
    '    for o,r in records.items():\n'
    '        for n in NAMES:\n'
    "            rows.append(row(o,n,r[n].sum(0),r['counts'].sum()))\n"
    "            for i in range(0,len(r['counts']),7):\n"
    "                ns=r['counts'][i:i+7].sum()\n"
    "                if ns:rows.append(row(o,n,r[n][i:i+7].sum(0),ns,'weekly',week=i//7+1))\n"
    "            for i,ns in enumerate(r['counts']):\n"
    "                if ns:rows.append(row(o,n,r[n][i],ns,'daily',day=i+1))\n"
    "                rows.append(dict(section='daily_sse',origin=o,model=n,day=i+1,pixels_hours=int(ns),**dict(zip(SSE,r[n][i]))))\n"
    "    for n in NAMES:rows.append(row('Pooled',n,sum(r[n].sum(0) for r in records.values()),sum(r['counts'].sum() for r in records.values())))\n"
    "    gate=a.pooled_gate(records,'RegimeFlow',True);gate['statistical_gate_passed_retrospective']=bool(gate['review_eligible'] and min(gate['origins'].values())>0);gate['review_eligible']=False\n"
    '    return meta,rows,gate\n'
    '\n'
    'def components(X,t,h,Y,T,s,tr,ca,query,XX=None,tt=None,hh=None,TT=None):\n'
    '    assert max(T[tr])<min(T[ca]); assert not np.intersect1d(tr,ca).size\n'
    '    if XX is None: XX,tt,hh,TT=X,t,h,T\n'
    '    assert max(T[ca])<min(TT[query]);pred={};states={}\n'
    "    for name,drift in [('Physics',False),('Drift',True)]:\n"
    "        st=a.fit_anchor(X,t,h,Y,T,tr,s,'physics',False,drift)\n"
    '        p=a.predict_anchor(st,X[ca],t[ca],h[ca],T[ca]);q=a.predict_anchor(st,XX[query],tt[query],hh[query],TT[query]);states[name]=st\n'
    "        variant='AdaptiveFlow' if not drift else 'Bias';corr=m.fit_correction(p,Y[ca],h[ca],variant);last=pd.to_datetime(T[ca[-1]],format='%Y%m%d%H')\n"
    "        pred['AdaptiveFlow' if not drift else 'DriftBias']=m.apply_correction(corr,q,hh[query],TT[query],last,variant);states[variant]=corr\n"
    "        if not drift:pred['Bias325']=v.clip(q+v.bias(p,Y[ca],h[ca])[hh[query]])\n"
    "    pred['RegimeFlow']=pred['DriftBias'].copy();pred['RegimeFlow'][...,2]=pred['AdaptiveFlow'][...,2]\n"
    '    return pred,states\n'
    '\n'
    'def diagnostics(X,t,h,Y,T,s,train,test,TT):\n'
    '    from sklearn.preprocessing import StandardScaler\n'
    '    from sklearn.pipeline import make_pipeline\n'
    '    from sklearn.linear_model import LogisticRegression\n'
    '    from sklearn.metrics import roc_auc_score\n'
    '    from sklearn.decomposition import PCA\n'
    '    from scipy.stats import ks_2samp\n'
    '    def feat(x):\n'
    '        q=np.concatenate([x,np.hypot(x[...,2],x[...,3])[...,None]],axis=-1)\n'
    '        return np.concatenate([q.mean((1,2)),q.std((1,2)),np.quantile(q,.1,axis=(1,2)),np.quantile(q,.9,axis=(1,2))],axis=1)\n'
    "    t0=pd.to_datetime(T,format='%Y%m%d%H');t1=pd.to_datetime(TT,format='%Y%m%d%H');it=(t0>=t0.max().normalize()-pd.Timedelta(days=33))&(t0.hour%3==0);ie=t1.hour%3==0\n"
    '    f0,f1=feat(train[it]),feat(test[ie]);F=np.r_[f0,f1];y=np.r_[np.zeros(len(f0),int),np.ones(len(f1),int)]\n'
    '    groups=np.r_[np.asarray((t0[it]-t0[it].min()).days)//7,100+np.asarray((t1[ie]-t1[ie].min()).days)//7]\n'
    '    blocks0=np.unique(groups[y==0]);blocks1=np.unique(groups[y==1]);assert len(blocks0)==len(blocks1)\n'
    '    prob=np.full(len(y),np.nan);auc=[]\n'
    '    for b0,b1 in zip(blocks0,blocks1):\n'
    '        ev=np.isin(groups,[b0,b1]);tr=~ev;clf=make_pipeline(StandardScaler(),LogisticRegression(C=.1,max_iter=1000));clf.fit(F[tr],y[tr]);prob[ev]=clf.predict_proba(F[ev])[:,1];auc.append(float(roc_auc_score(y[ev],prob[ev])))\n'
    "    rows=[dict(section='shift',metric=k,value=x) for k,x in [('adversarial_auc_oof',float(roc_auc_score(y,prob))),('auc_fold_min',min(auc)),('auc_fold_max',max(auc)),('n_train',len(f0)),('n_test',len(f1)),('blocks',len(np.unique(groups)))]]\n"
    "    for j,n in enumerate([*VARS,'speed']):\n"
    "        for hour in range(0,24,3):rows.append(dict(section='shift',metric='KS_spatial_mean',variable=n,hour=hour,value=float(ks_2samp(f0[t0[it].hour==hour,j],f1[t1[ie].hour==hour,j]).statistic)))\n"
    "    x=train[it].reshape(len(f0),-1);z=test[ie].reshape(len(f1),-1);sc=StandardScaler().fit(x);xp=sc.transform(x);zp=sc.transform(z);pca=PCA(n_components=12,random_state=34,svd_solver='randomized').fit(xp)\n"
    "    for n,arr in [('train',xp),('test',zp)]:rows.append(dict(section='shift',metric='EOF_reconstruction_MSE',period=n,value=float(np.mean((arr-pca.inverse_transform(pca.transform(arr)))**2))))\n"
    "    dem=s['dem'];sl=np.hypot(*np.gradient(dem))\n"
    '    for o in a.ORIGINS:\n'
    "        tr,ca,ev=a.split(T,h,o);pred,_=components(X,t,h,Y,T,s,tr,ca,ev);clim=v.clip(v.A['climate'](Y[tr],h[tr],np.arange(len(tr)))[h[ev]])\n"
    "        for n,q in [('Climatology',clim),*pred.items()]:rows.append(dict(section='anchor_only',origin=o,model=n,skill=v.score(q,Y[ev])))\n"
    "        p=pred['Bias325'];y=Y[ev];speed=np.hypot(p[...,2],p[...,3]);ts=np.hypot(y[...,2],y[...,3]);ang=np.arctan2(p[...,3],p[...,2])-np.arctan2(y[...,3],y[...,2]);ang=np.abs(np.arctan2(np.sin(ang),np.cos(ang)))*180/np.pi\n"
    "        masks={**{f'hour_{i:02}':np.broadcast_to((h[ev]==i)[:,None,None],speed.shape) for i in range(0,24,3)},'wind_lt2':speed<2,'wind_2_5':(speed>=2)&(speed<5),'wind_ge5':speed>=5}\n"
    "        for name,field in [('slope',sl),('altitude',dem)]:\n"
    '            l,u=np.quantile(field,[.33,.67]);\n'
    "            for i,mask in enumerate([field<=l,(field>l)&(field<=u),field>u]):masks[f'{name}_{i+1}']=np.broadcast_to(mask,speed.shape)\n"
    '        for label,mask in masks.items():\n'
    '            if not mask.any():continue\n'
    "            keep=mask&(ts>=1)&(speed>=1);rows.append(dict(section='wind_anchor_only',origin=o,regime=label,pixels_hours=int(mask.sum()),RMSE_U=float(np.sqrt(np.mean((p[...,2][mask]-y[...,2][mask])**2))),RMSE_V=float(np.sqrt(np.mean((p[...,3][mask]-y[...,3][mask])**2))),speed_RMSE=float(np.sqrt(np.mean((speed[mask]-ts[mask])**2))),direction_MAE_deg=float(ang[keep].mean()) if keep.any() else np.nan,direction_count=int(keep.sum())))\n"
    "        print('diagnostics',o,flush=True)\n"
    '    return rows\n'
    '\n'
    'def main():\n'
    "    p=argparse.ArgumentParser();p.add_argument('--data',required=True);p.add_argument('--audit',required=True);p.add_argument('--base',required=True);p.add_argument('--base-sha',required=True);p.add_argument('--base-mode',choices=['original','reconstructed'],required=True);p.add_argument('--out',required=True);p.add_argument('--diagnostics',action='store_true');args=p.parse_args();out=Path(args.out);out.mkdir(parents=True,exist_ok=True);data=v.data_root(args.data);dh=v.fingerprint(data);assert dh==m.OFFICIAL_HASHES\n"
    '    a.check_protocol();m.tests();assert v.sha(m.__file__)==CODE_SHA\n'
    '    for name,digest in m.EXPECTED.items():assert v.sha(Path(__file__).with_name(name))==digest\n'
    "    meta,rows,gate=audit(Path(args.audit));print('RETROSPECTIVE GATE:',json.dumps(gate),flush=True)\n"
    "    assert gate['statistical_gate_passed_retrospective'],'No competitive file: retrospective gate failed'\n"
    "    train=v.readnp(data/'train.npz');s=v.readnp(data/'static.npz');test=v.readnp(data/'test.npz');assert 'Y' not in test;T=train['times'].astype(str);TT=test['times'].astype(str);Y=train['Y'];X,t,h=v.A['prepare'](train['X'],T,s)\n"
    "    if args.diagnostics:rows+=diagnostics(X,t,h,Y,T,s,train['X'],test['X'],TT)\n"
    "    base=Path(args.base);assert v.sha(base)==args.base_sha;assert args.base_mode!='original' or args.base_sha==BASE_SHA\n"
    "    tr=np.flatnonzero(T<'2024062400');ca=np.flatnonzero((T>='2024062400')&(T<'2024070100')&(h%3==0));assert len(tr)==2767 and len(ca)==56 and max(T[tr])<min(T[ca]);assert len(TT)==792\n"
    "    XX,tt,hh=v.A['prepare'](test['X'],TT,s);ix=np.flatnonzero(hh%3==0);assert len(ix)==264 and min(TT)>='2024070100'\n"
    '    pred,states=components(X,t,h,Y,T,s,tr,ca,ix,XX,tt,hh,TT)\n'
    "    w=np.asarray(next(r['weights'] for r in meta['reports'] if r['origin']=='O3'),dtype='f8')\n"
    "    tpl=pd.read_csv(data/'sample_submission.csv',dtype={'id':str});b=pd.read_csv(base,dtype={'id':str});ids=np.array([f'{stamp}_{i:02d}_{j:02d}' for stamp in TT[ix] for i in range(70) for j in range(35)]);assert len(ids)==646800 and np.array_equal(ids,tpl.id.to_numpy()) and np.array_equal(ids,b.id.to_numpy()) and list(b.columns)==['id',*VARS]\n"
    "    bv=b[VARS].to_numpy();assert np.isfinite(bv).all() and b.RH.between(0,100).all();av=pred['RegimeFlow'].reshape(-1,4);inc=pred['Bias325'].reshape(-1,4)\n"
    '    mix=bv+(av-bv)*w;mix[:,0]=np.clip(mix[:,0],0,100);prev=bv+(inc-bv)*w;prev[:,0]=np.clip(prev[:,0],0,100)\n'
    "    dd=bv+(pred['DriftBias'].reshape(-1,4)-bv)*w;aa=bv+(pred['AdaptiveFlow'].reshape(-1,4)-bv)*w;dd[:,2]=aa[:,2];dd[:,0]=np.clip(dd[:,0],0,100);assert np.array_equal(dd,mix)\n"
    "    for name,st in states.items():np.savez(out/f'{name}.npz',**st)\n"
    "    last=pd.to_datetime(T[ca[-1]],format='%Y%m%d%H');ii=ix[:8];p=a.predict_anchor(v.readnp(out/'Physics.npz'),XX[ii],tt[ii],hh[ii],TT[ii]);q=a.predict_anchor(v.readnp(out/'Drift.npz'),XX[ii],tt[ii],hh[ii],TT[ii]);u=m.apply_correction(v.readnp(out/'AdaptiveFlow.npz'),p,hh[ii],TT[ii],last,'AdaptiveFlow');q=m.apply_correction(v.readnp(out/'Bias.npz'),q,hh[ii],TT[ii],last,'Bias');q[...,2]=u[...,2];err=float(np.max(np.abs(q-pred['RegimeFlow'][:8])));assert err<=1e-4\n"
    "    result=pd.DataFrame(mix,columns=VARS);result.insert(0,'id',ids);dest=out/'Atlas34_REVIEW.csv';tmp=dest.with_suffix('.tmp');result.to_csv(tmp,index=False,float_format='%.6f');check=pd.read_csv(tmp,dtype={'id':str});assert len(check)==646800 and np.array_equal(check.id,ids) and list(check.columns)==['id',*VARS] and np.isfinite(check[VARS]).all().all() and check.RH.between(0,100).all();tmp.replace(dest)\n"
    "    report=dict(base_mode=args.base_mode,base_actual_sha256=args.base_sha,base_original_sha256=BASE_SHA,train_count=len(tr),cal_count=len(ca),train_max=str(max(T[tr])),cal_min=str(min(T[ca])),cal_max=str(max(T[ca])),test_count=len(ix),weights=dict(zip(VARS,w.tolist())),mean_abs_vs_v27=dict(zip(VARS,np.abs(check[VARS].to_numpy()-bv).mean(0).tolist())),mean_abs_vs_same_weights_bias325=dict(zip(VARS,np.abs(check[VARS].to_numpy()-prev).mean(0).tolist())),replay_max_abs=err,csv_sha256=v.sha(dest),rows=len(check),scope='manual experimental review; no test labels; no automatic submission')\n"
    "    rows+=[dict(section='metadata',field=k,value=json.dumps(val,sort_keys=True)) for k,val in dict(route=dict(zip(VARS,ROUTE)),gate=gate,deployment=report,source_sha256=v.sha(__file__),audit_sha256=AUDIT_SHA,data=dh).items()]\n"
    "    pd.DataFrame(rows).to_csv(out/'Atlas34_AUDIT.csv',index=False,float_format='%.17g');manifest=dict(version='3.4',route=dict(zip(VARS,ROUTE)),gate=gate,deploy=report,source_sha256=v.sha(__file__),audit_sha256=AUDIT_SHA,base_sha256=args.base_sha,base_original_sha256=BASE_SHA,base_mode=args.base_mode,data=dh,files={p.name:v.sha(p) for p in sorted(out.iterdir()) if p.is_file() and p.name!='manifest.json'})\n"
    "    v.dump(out/'manifest.json',manifest);print('DEPLOY:',json.dumps(report),flush=True)\n"
    "if __name__=='__main__':main()\n"
)
assert hashlib.sha256(text.encode('utf-8')).hexdigest()=='d926aff167110f516c7f2f34273f2c35e8bb11bdc2ad8145412283c4f3d7171d'
(CODE/'Atlas_v3_4_Kaggle.py').write_bytes(text.encode('utf-8'))
compile(text,'Atlas_v3_4_Kaggle.py','exec')

import json, shutil, tempfile
import numpy as np
import pandas as pd
DEST=ROOT/'Atlas34_REVIEW.csv'; DEST.unlink(missing_ok=True)
AUDIT=CODE/'Atlas33_FULL_AUDIT.csv'
BASE_SHA='9ce6e7594d0edbf884e56902fd67e1fa02de1a94ba2f6815fc5c8b5115a801e1'
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
# Verificar datos, imports, protocolo y auditoría antes de entrenar en GPU.
subprocess.run([sys.executable,'-c',"import Atlas_v3_4_Kaggle as m; from pathlib import Path; import sys; assert m.v.fingerprint(Path(sys.argv[1]))==m.m.OFFICIAL_HASHES; m.a.check_protocol(); m.m.tests(); _,_,g=m.audit(Path('Atlas33_FULL_AUDIT.csv')); assert g['statistical_gate_passed_retrospective']; print('Gate histórico FULL:',g['pooled_gain'])",str(DATA)],cwd=CODE,check=True)
BASE=None; BASE_MODE='original'
for parent in (Path('/kaggle/input'),ROOT):
    for candidate in sorted(parent.rglob('submission_ultralisk_v2_7.csv')):
        if sha(candidate)==BASE_SHA:BASE=candidate;break
    if BASE is not None:break
if BASE is None:
    import torch
    if torch.cuda.device_count()!=2 or not all('T4' in torch.cuda.get_device_name(i) for i in range(2)):
        raise RuntimeError('Selecciona GPU T4 x2 para reconstruir Ultralisk v2.7.')
    RUN=Path(tempfile.mkdtemp(prefix='ultralisk27_',dir=ROOT))
    env=os.environ.copy();env.update(ULTRALISK_DATA=str(DATA),ULTRALISK_SMOKE='0',ULTRALISK_CACHE='1',ULTRALISK_CHRONOS_DIR=str(RUN/'v24'),ULTRALISK_V27_OUT=str(RUN/'v27'))
    print('Reconstruyendo Ultralisk v2.7 con su receta original.',flush=True)
    subprocess.run([sys.executable,str(CODE/'Ultralisk_v2_7_completo.py')],cwd=ROOT,env=env,check=True)
    BASE=RUN/'v27'/'submission_ultralisk_v2_7.csv'
    r=json.loads((RUN/'v27'/'report.json').read_text())
    assert r['model']=='Ultralisk-v2.7' and r['smoke'] is False and r['recommend_submit'] and r['reference_reproduced']
    assert r['anchor']=='Stack' and r['previous']=='RegionalFixed' and r['selected']=='NarrowSpatial' and r['spatial_bias']
    historical={'1':.45990063168146056,'2':.4604174971675282,'3':.4650797440837502,'April':.5584575773496497}
    for fold,expected in historical.items():
        rows=[x['skill'] for x in r['validation'] if str(x['fold'])==fold and x['model']=='NarrowSpatial']
        if len(rows)!=1 or abs(rows[0]-expected)>.001:raise RuntimeError(f'Fold {fold} fuera de tolerancia: {rows} vs {expected}')
        print(f'Fold {fold}: {rows[0]:.6f}; histórico {expected:.6f}',flush=True)
    assert BASE.is_file() and sha(BASE)==r['sha256'] and r['rows']==646800
    BASE_MODE='original' if sha(BASE)==BASE_SHA else 'reconstructed'
    if BASE_MODE=='reconstructed':print('AVISO: receta reconstruida; hash distinto al CSV público. No implica el mismo score público.',flush=True)
base_digest=sha(BASE);print('Referencia:',BASE_MODE,base_digest,flush=True)
OUT=Path(tempfile.mkdtemp(prefix='atlas34_run_',dir=ROOT))
subprocess.run([sys.executable,str(CODE/'Atlas_v3_4_Kaggle.py'),'--data',str(DATA),'--audit',str(AUDIT),'--base',str(BASE),'--base-sha',base_digest,'--base-mode',BASE_MODE,'--out',str(OUT)],cwd=CODE,check=True)
manifest=json.loads((OUT/'manifest.json').read_text())
assert manifest['gate']['statistical_gate_passed_retrospective'] and not manifest['gate']['review_eligible']
assert manifest['base_sha256']==base_digest and manifest['base_mode']==BASE_MODE
result=OUT/'Atlas34_REVIEW.csv';assert sha(result)==manifest['deploy']['csv_sha256']
check=pd.read_csv(result,dtype={'id':str});template=pd.read_csv(DATA/'sample_submission.csv',dtype={'id':str})
assert list(check.columns)==['id','RH','T','U','V'] and len(check)==646800 and check.id.equals(template.id) and check.id.is_unique
assert np.isfinite(check[['RH','T','U','V']]).all().all() and check.RH.between(0,100).all()
tmp=DEST.with_suffix('.tmp');shutil.copyfile(result,tmp);assert sha(tmp)==sha(result);tmp.replace(DEST)
print('CSV COMPETITIVO EXPERIMENTAL:',DEST,flush=True)
print('Filas:',len(check),'SHA256:',sha(DEST),'Base:',BASE_MODE,flush=True)
print('Sin submission automático. Descargar desde Files/Output.',flush=True)
os.chdir(ROOT);display(FileLink(DEST.name))
